In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/jmmubasshirrahman/ai-ids-research-kaggle/results/additional_baseline_model_results.csv
/kaggle/input/datasets/jmmubasshirrahman/ai-ids-research-kaggle/results/cnn_baseline_result.csv
/kaggle/input/datasets/jmmubasshirrahman/ai-ids-research-kaggle/results/final_model_comparison_with_threshold.csv
/kaggle/input/datasets/jmmubasshirrahman/ai-ids-research-kaggle/results/random_forest_results.csv
/kaggle/input/datasets/jmmubasshirrahman/ai-ids-research-kaggle/results/xgboost_threshold_results.csv
/kaggle/input/datasets/jmmubasshirrahman/ai-ids-research-kaggle/results/final_selected_xgboost_threshold_030.csv
/kaggle/input/datasets/jmmubasshirrahman/ai-ids-research-kaggle/results/all_csv_label_summary.csv
/kaggle/input/datasets/jmmubasshirrahman/ai-ids-research-kaggle/results/final_multifile_xgboost_shap_top_20_features.csv
/kaggle/input/datasets/jmmubasshirrahman/ai-ids-research-kaggle/results/processing_summary_small_files.csv
/kaggle/input/datasets/jmmubasshirrahman/

In [2]:
# ======================================================================================
# STAGE 27 — FRESH KAGGLE BOOTSTRAP / PROVENANCE PREFLIGHT
# ======================================================================================
#
# PURPOSE
# -------
# Start Stage 27 from a clean Kaggle runtime and establish provenance BEFORE
# Stage27-0A reads label supports.
#
# THIS CELL:
#   1. Forces primary Stage27 execution policy to CPU / GPU budget = 0
#   2. Prints runtime/system information
#   3. Verifies Kaggle GitHub secret without printing it
#   4. Fresh-clones or safely refreshes:
#          themubasshir/ids2018-validation-safe-ablation
#   5. Pins the exact origin/main SHA used by this notebook
#   6. Verifies clean Git state
#   7. Verifies durable Stage26 COMPLETE evidence exists
#   8. Inventories Stage22 / Stage23 / Stage24 provenance candidates
#   9. Inventories attached Kaggle input files
#  10. Creates ONLY the Stage27-0A working directory
#
# STRICT SCIENTIFIC STATE AFTER THIS CELL:
#   Stage27 model fits       = 0
#   Stage27 model inference  = 0
#   Stage27 metrics          = 0
#   Stage27 thresholds       = 0
#   Stage27 GPU budget used  = 0
#   Stage27-0 frozen         = NO
#
# IMPORTANT:
#   Do NOT manually paste a GitHub token into this notebook.
# ======================================================================================

from __future__ import annotations

import os
import sys
import re
import json
import hashlib
import platform
import socket
import subprocess
import shutil
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict


# ======================================================================================
# 0. STAGE 27 COMPUTE POLICY — FREEZE FOR THIS RUNTIME
# ======================================================================================

STAGE = "Stage27"
PRIMARY_EXECUTION_DEVICE = "CPU"
PRIMARY_GPU_BUDGET_HOURS = 0

# Prevent CUDA-aware libraries imported later from seeing GPUs.
# Stage 27 primary experiment is deliberately CPU-only.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["NVIDIA_VISIBLE_DEVICES"] = ""
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

# Avoid accidental massive thread oversubscription.
LOGICAL_CPUS = os.cpu_count() or 1
CPU_THREADS = max(1, LOGICAL_CPUS)

os.environ.setdefault("OMP_NUM_THREADS", str(CPU_THREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(CPU_THREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(CPU_THREADS))
os.environ.setdefault("NUMEXPR_NUM_THREADS", str(CPU_THREADS))


# ======================================================================================
# 1. CONSTANTS
# ======================================================================================

OWNER = "themubasshir"
REPO_NAME = "ids2018-validation-safe-ablation"
BRANCH = "main"

REPO_URL = f"https://github.com/{OWNER}/{REPO_NAME}.git"

WORK_ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")
REPO = WORK_ROOT / REPO_NAME

STAGE27_ROOT = REPO / "results" / "stage27_loao_unseen_attack"
STAGE27_0A = STAGE27_ROOT / "stage27_0a_family_day_feasibility"

FROZEN_FAMILIES = (
    "BOT",
    "DDOS",
    "DOS",
    "AUTH_BRUTE_FORCE",
    "INFILTRATION",
    "PORT_SCAN",
    "WEB_ATTACK",
)

EXPECTED_STAGE27_0A_FILES = (
    "family_day_support.csv",
    "benign_day_support.csv",
    "taxonomy_receipt.json",
    "source_artifact_receipts.json",
    "temporal_feasibility.json",
    "feasibility_audit.md",
)


# ======================================================================================
# 2. HELPERS
# ======================================================================================

def banner(title: str) -> None:
    print()
    print("=" * 100)
    print(title)
    print("=" * 100)


def run(
    cmd,
    *,
    cwd: Path | None = None,
    env: dict | None = None,
    check: bool = True,
) -> str:
    result = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    if check and result.returncode != 0:
        raise RuntimeError(
            "\nCOMMAND FAILED\n"
            f"Command : {' '.join(str(x) for x in cmd)}\n"
            f"Code    : {result.returncode}\n"
            f"STDOUT:\n{result.stdout}\n"
            f"STDERR:\n{result.stderr}"
        )

    return result.stdout.strip()


def git(*args, env=None) -> str:
    return run(["git", *args], cwd=REPO, env=env)


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def human_bytes(n: int) -> str:
    value = float(n)
    units = ["B", "KiB", "MiB", "GiB", "TiB"]

    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f"{value:,.2f} {unit}"
        value /= 1024

    return f"{n:,} B"


def recursively_find_key(obj, wanted_key: str):
    """
    Return [(path_within_json, value), ...] for exact dictionary-key matches.
    """
    found = []

    def walk(x, path="$"):
        if isinstance(x, dict):
            for key, value in x.items():
                child = f"{path}.{key}"
                if key == wanted_key:
                    found.append((child, value))
                walk(value, child)

        elif isinstance(x, list):
            for i, value in enumerate(x):
                walk(value, f"{path}[{i}]")

    walk(obj)
    return found


# ======================================================================================
# 3. RUNTIME
# ======================================================================================

banner("STAGE27 :: FRESH-RUNTIME PREFLIGHT")

print("timestamp_utc             :", datetime.now(timezone.utc).isoformat())
print("hostname                  :", socket.gethostname())
print("python                    :", sys.version.replace("\n", " "))
print("python_executable         :", sys.executable)
print("platform                  :", platform.platform())
print("machine                   :", platform.machine())
print("working_directory         :", Path.cwd())
print("kaggle_working_exists     :", WORK_ROOT.exists())
print("kaggle_input_exists       :", INPUT_ROOT.exists())
print("logical_cpus              :", LOGICAL_CPUS)

try:
    import psutil

    print("physical_cores            :", psutil.cpu_count(logical=False))
    print("ram_total_gib             :", round(psutil.virtual_memory().total / 2**30, 3))
    print("ram_available_gib         :", round(psutil.virtual_memory().available / 2**30, 3))
except Exception as exc:
    print("psutil_runtime_info       : unavailable:", repr(exc))

print()
print("PRIMARY_EXECUTION_DEVICE  :", PRIMARY_EXECUTION_DEVICE)
print("PRIMARY_GPU_BUDGET_HOURS  :", PRIMARY_GPU_BUDGET_HOURS)
print("CUDA_VISIBLE_DEVICES      :", repr(os.environ["CUDA_VISIBLE_DEVICES"]))

if PRIMARY_EXECUTION_DEVICE != "CPU":
    raise RuntimeError("Stage27 primary execution device must be CPU.")

if PRIMARY_GPU_BUDGET_HOURS != 0:
    raise RuntimeError("Stage27 primary GPU budget must be exactly zero.")


# ======================================================================================
# 4. CHECK GITHUB SECRET
# ======================================================================================

banner("STAGE27 :: GITHUB SECRET")

try:
    from kaggle_secrets import UserSecretsClient
except Exception as exc:
    raise RuntimeError(
        "Kaggle UserSecretsClient is unavailable. "
        "This cell must run inside Kaggle."
    ) from exc


secret_client = UserSecretsClient()

# We know previous notebooks have used these labels.
# Do not display the secret value.
SECRET_LABELS = (
    "GITHUB_TOKEN",
    "github_token",
    "GH_TOKEN",
    "github_pat",
    "GITHUB_PAT",
    "GH_PAT",
)

github_token = None
github_secret_label = None

for label in SECRET_LABELS:
    try:
        candidate = secret_client.get_secret(label)
    except Exception:
        candidate = None

    if candidate and len(candidate.strip()) >= 20:
        github_token = candidate.strip()
        github_secret_label = label
        break


if not github_token:
    raise RuntimeError(
        "\nNo usable GitHub token was accessible from Kaggle Secrets.\n\n"
        "Expected one of these secret labels:\n"
        + "\n".join(f"  - {x}" for x in SECRET_LABELS)
        + "\n\nEnable notebook access to the existing secret and rerun this cell.\n"
        "Do NOT paste the token into notebook source."
    )


print("GitHub secret label       :", github_secret_label)
print("GitHub secret available   : YES")
print("GitHub token length       :", len(github_token))
print("GitHub token printed      : NO")


# ======================================================================================
# 5. SAFE GIT AUTHENTICATION VIA GIT_ASKPASS
# ======================================================================================

banner("STAGE27 :: CONFIGURE TEMPORARY GIT AUTH")

ASKPASS = WORK_ROOT / ".stage27_git_askpass.sh"

ASKPASS.write_text(
    """#!/bin/sh
case "$1" in
    *Username*) echo "x-access-token" ;;
    *Password*) echo "$GITHUB_TOKEN" ;;
    *) echo "" ;;
esac
""",
    encoding="utf-8",
)

ASKPASS.chmod(0o700)

git_env = os.environ.copy()
git_env["GITHUB_TOKEN"] = github_token
git_env["GIT_ASKPASS"] = str(ASKPASS)
git_env["GIT_TERMINAL_PROMPT"] = "0"

print("Temporary askpass         :", ASKPASS)
print("Interactive prompt        : DISABLED")
print("Credential in command URL : NO")


# ======================================================================================
# 6. CLONE / SAFELY REFRESH REPOSITORY
# ======================================================================================

banner("STAGE27 :: REPOSITORY BOOTSTRAP")

if not REPO.exists():
    print("Repository does not exist in this fresh runtime.")
    print("Cloning origin/main...")

    run(
        [
            "git",
            "clone",
            "--branch",
            BRANCH,
            "--single-branch",
            REPO_URL,
            str(REPO),
        ],
        env=git_env,
    )

else:
    print("Repository directory already exists:", REPO)

    if not (REPO / ".git").exists():
        raise RuntimeError(
            f"{REPO} exists but is not a Git repository. "
            "Refusing to delete or overwrite it."
        )

    existing_status = run(
        ["git", "status", "--porcelain"],
        cwd=REPO,
    )

    if existing_status:
        raise RuntimeError(
            "\nExisting repository contains modifications.\n"
            "Refusing to reset or overwrite them.\n\n"
            + existing_status
        )

    print("Existing repository is clean.")
    print("Refreshing origin/main...")

    run(
        ["git", "fetch", "--prune", "origin", BRANCH],
        cwd=REPO,
        env=git_env,
    )

    run(
        ["git", "checkout", BRANCH],
        cwd=REPO,
        env=git_env,
    )

    run(
        ["git", "reset", "--hard", f"origin/{BRANCH}"],
        cwd=REPO,
        env=git_env,
    )


# Ensure remote is canonical and contains no embedded credential.
git("remote", "set-url", "origin", REPO_URL)

remote_url = git("remote", "get-url", "origin")

if github_token in remote_url:
    raise RuntimeError("Credential leaked into Git remote URL.")

print("Repository                :", REPO)
print("Remote                    :", remote_url)


# ======================================================================================
# 7. PIN CURRENT MAIN
# ======================================================================================

banner("STAGE27 :: PIN CURRENT MAIN")

# Fetch once more with authenticated environment.
run(
    ["git", "fetch", "--prune", "origin", BRANCH],
    cwd=REPO,
    env=git_env,
)

local_head = git("rev-parse", "HEAD")
origin_main = git("rev-parse", f"origin/{BRANCH}")
branch_name = git("branch", "--show-current")
status = git("status", "--porcelain")
commit_subject = git("log", "-1", "--pretty=%s")
commit_time = git("log", "-1", "--format=%cI")

print("branch                    :", branch_name)
print("local HEAD                :", local_head)
print("origin/main               :", origin_main)
print("HEAD subject              :", commit_subject)
print("HEAD committed            :", commit_time)
print("working tree clean        :", status == "")

if branch_name != BRANCH:
    raise RuntimeError(f"Expected branch {BRANCH}, got {branch_name!r}.")

if local_head != origin_main:
    raise RuntimeError(
        "Fresh Stage27 bootstrap is not exactly pinned to origin/main."
    )

if status:
    raise RuntimeError(
        "Repository is unexpectedly dirty immediately after bootstrap:\n"
        + status
    )


# ======================================================================================
# 8. VERIFY REMOTE SHA DIRECTLY
# ======================================================================================

banner("STAGE27 :: REMOTE SHA VERIFICATION")

ls_remote_output = run(
    ["git", "ls-remote", "origin", f"refs/heads/{BRANCH}"],
    cwd=REPO,
    env=git_env,
)

if not ls_remote_output:
    raise RuntimeError("git ls-remote returned no main-branch SHA.")

remote_sha = ls_remote_output.split()[0]

print("local HEAD                :", local_head)
print("origin/main tracking      :", origin_main)
print("remote refs/heads/main    :", remote_sha)

if not (local_head == origin_main == remote_sha):
    raise RuntimeError(
        "Local HEAD, origin/main, and remote main are not identical."
    )

print("[OK] Exact Stage27 execution parent pinned.")


# ======================================================================================
# 9. VERIFY STAGE 26 CLOSURE EVIDENCE
# ======================================================================================

banner("STAGE27 :: VERIFY DURABLE STAGE26 CLOSURE")

stage26_root_candidates = sorted(
    p
    for p in (REPO / "results").glob("stage26*")
    if p.exists()
)

if not stage26_root_candidates:
    raise RuntimeError("No Stage26 result directory exists in the repository.")

print("Stage26 result roots:")
for p in stage26_root_candidates:
    print(" ", p.relative_to(REPO))


stage26_json_files = []

for root in stage26_root_candidates:
    stage26_json_files.extend(root.rglob("*.json"))

stage26_json_files = sorted(set(stage26_json_files))

print()
print("Stage26 JSON files found  :", len(stage26_json_files))


closure_hits = []

for path in stage26_json_files:
    try:
        obj = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        continue

    matches = recursively_find_key(obj, "stage26_status")

    for json_path, value in matches:
        if str(value).strip().upper() == "COMPLETE":
            closure_hits.append(
                {
                    "file": path,
                    "json_path": json_path,
                    "value": value,
                }
            )


# Secondary textual check ONLY as provenance visibility.
# The primary pass condition remains exact stage26_status == COMPLETE.
closure_named_files = [
    p
    for p in stage26_json_files
    if any(
        token in p.name.lower()
        for token in ("closure", "final", "seal", "receipt")
    )
]


print()
print("Exact stage26_status=COMPLETE hits:")

for hit in closure_hits:
    print(
        " ",
        hit["file"].relative_to(REPO),
        "::",
        hit["json_path"],
        "=",
        hit["value"],
    )


if not closure_hits:
    print()
    print("Potential closure/final JSON files:")
    for p in closure_named_files[:50]:
        print(" ", p.relative_to(REPO))

    raise RuntimeError(
        "\nCould not verify an exact durable JSON field:\n"
        '    "stage26_status": "COMPLETE"\n\n'
        "Do not begin Stage27-0A until Stage26 closure provenance is resolved."
    )


print()
print("[OK] Durable Stage26 COMPLETE evidence verified.")


# ======================================================================================
# 10. INVENTORY STAGE22 / STAGE23 / STAGE24 PROVENANCE
# ======================================================================================

banner("STAGE27 :: PRIOR-STAGE PROVENANCE INVENTORY")


def collect_stage_files(stage_number: int):
    stage_token = f"stage{stage_number}"
    files = []

    for base in (REPO / "results", REPO / "docs"):
        if not base.exists():
            continue

        for path in base.rglob("*"):
            if path.is_file() and stage_token in str(path.relative_to(REPO)).lower():
                files.append(path)

    return sorted(set(files))


prior_stage_inventory = {}

for n in (22, 23, 24):
    files = collect_stage_files(n)
    prior_stage_inventory[n] = files

    print()
    print(f"Stage {n}: {len(files)} candidate provenance files")

    interesting = [
        p
        for p in files
        if any(
            word in p.name.lower()
            for word in (
                "taxonomy",
                "family",
                "temporal",
                "support",
                "manifest",
                "receipt",
                "protocol",
                "lock",
                "closure",
                "audit",
                "mapping",
            )
        )
    ]

    for p in interesting[:60]:
        print(" ", p.relative_to(REPO))

    if len(interesting) > 60:
        print(f"  ... {len(interesting) - 60} additional relevant-looking files")


if not prior_stage_inventory[22]:
    raise RuntimeError("No Stage22 provenance candidates found.")

if not prior_stage_inventory[23]:
    raise RuntimeError("No Stage23 provenance candidates found.")

if not prior_stage_inventory[24]:
    raise RuntimeError("No Stage24 provenance candidates found.")


# ======================================================================================
# 11. SEARCH REPOSITORY FOR THE FROZEN SEVEN-FAMILY TAXONOMY
# ======================================================================================

banner("STAGE27 :: TAXONOMY DISCOVERY")

text_suffixes = {
    ".json",
    ".csv",
    ".md",
    ".txt",
    ".yaml",
    ".yml",
    ".py",
}

taxonomy_candidate_hits = []

search_bases = [
    REPO / "results",
    REPO / "docs",
]

for base in search_bases:
    if not base.exists():
        continue

    for path in base.rglob("*"):
        if not path.is_file():
            continue

        if path.suffix.lower() not in text_suffixes:
            continue

        path_l = str(path.relative_to(REPO)).lower()

        # Stage24 is the frozen family-taxonomy authority for Stage27.
        if "stage24" not in path_l:
            continue

        try:
            # Do not read pathological files into memory.
            if path.stat().st_size > 50 * 1024 * 1024:
                continue

            text = path.read_text(
                encoding="utf-8",
                errors="replace",
            )
        except Exception:
            continue

        upper = text.upper()

        found = [
            family
            for family in FROZEN_FAMILIES
            if family in upper
        ]

        if len(found) >= 5:
            taxonomy_candidate_hits.append(
                (path, tuple(found))
            )


print("Frozen Stage27 families:")
for family in FROZEN_FAMILIES:
    print(" ", family)

print()
print("Stage24 taxonomy/support candidate artifacts:")

for path, families in taxonomy_candidate_hits[:50]:
    print(
        " ",
        path.relative_to(REPO),
        f" [{len(families)}/7 families detected]",
    )


if not taxonomy_candidate_hits:
    raise RuntimeError(
        "Could not locate a Stage24 artifact containing the inherited "
        "attack-family taxonomy. Stage27-0A must not invent a replacement."
    )


# ======================================================================================
# 12. INVENTORY KAGGLE INPUT DATA
# ======================================================================================

banner("STAGE27 :: KAGGLE INPUT INVENTORY")

if not INPUT_ROOT.exists():
    raise RuntimeError("/kaggle/input does not exist.")


dataset_dirs = sorted(
    p
    for p in INPUT_ROOT.iterdir()
    if p.is_dir()
)

print("Attached dataset directories:", len(dataset_dirs))

for d in dataset_dirs:
    print(" ", d.name)


all_input_files = sorted(
    p
    for p in INPUT_ROOT.rglob("*")
    if p.is_file()
)

print()
print("Total attached input files  :", len(all_input_files))

suffix_counter = Counter(
    p.suffix.lower() if p.suffix else "<no_extension>"
    for p in all_input_files
)

print()
print("Input extensions:")

for suffix, count in suffix_counter.most_common():
    print(f"  {suffix:15s} {count:6d}")


# Candidate files relevant to CICIDS2017 / Stage27 support recovery.
keywords = (
    "monday",
    "tuesday",
    "wednesday",
    "thursday",
    "friday",
    "cicids",
    "cic-ids",
    "ids2017",
    "workinghours",
    "label",
    "stage22",
    "stage23",
    "stage24",
)

candidate_input_files = [
    p
    for p in all_input_files
    if any(k in p.name.lower() for k in keywords)
]


print()
print("Potential Stage27 support/provenance input files:")

for p in candidate_input_files[:200]:
    try:
        size = human_bytes(p.stat().st_size)
    except Exception:
        size = "?"

    print(
        f"  {str(p.relative_to(INPUT_ROOT)):100s} {size:>12s}"
    )


if len(candidate_input_files) > 200:
    print(
        f"\n  ... {len(candidate_input_files) - 200} additional candidate files"
    )


# ======================================================================================
# 13. CREATE ONLY STAGE27-0A WORK DIRECTORY
# ======================================================================================

banner("STAGE27 :: INITIALIZE ZERO-FIT FEASIBILITY DIRECTORY")

STAGE27_0A.mkdir(parents=True, exist_ok=True)

print("Stage27 root              :", STAGE27_ROOT)
print("Stage27-0A directory      :", STAGE27_0A)

existing_0a_files = sorted(
    p.relative_to(STAGE27_0A)
    for p in STAGE27_0A.rglob("*")
    if p.is_file()
)

if existing_0a_files:
    raise RuntimeError(
        "\nStage27-0A directory already contains files:\n"
        + "\n".join(f"  {p}" for p in existing_0a_files)
        + "\n\nFresh Stage27 bootstrap expected an empty Stage27-0A directory."
    )

print("Stage27-0A currently empty: YES")
print("Stage27-0 created          : NO")
print("Protocol frozen            : NO")


# ======================================================================================
# 14. WRITE LOCAL NON-GIT BOOTSTRAP RECEIPT
# ======================================================================================
#
# This receipt deliberately lives in /kaggle/working OUTSIDE the repository.
# It records the execution parent and discovery state but does not prematurely
# create a Stage27 scientific artifact before the feasibility audit.
# ======================================================================================

banner("STAGE27 :: LOCAL BOOTSTRAP RECEIPT")

bootstrap_receipt = {
    "stage": "Stage27",
    "substage": "fresh_bootstrap",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "repository": {
        "owner": OWNER,
        "name": REPO_NAME,
        "branch": BRANCH,
        "remote_url": REPO_URL,
        "execution_parent_commit": local_head,
        "origin_main_commit": origin_main,
        "remote_main_commit": remote_sha,
        "head_subject": commit_subject,
        "head_commit_time": commit_time,
        "working_tree_clean_before_stage27_0a": True,
    },
    "scientific_policy": {
        "primary_execution_device": PRIMARY_EXECUTION_DEVICE,
        "primary_gpu_budget_hours": PRIMARY_GPU_BUDGET_HOURS,
        "model_fits": 0,
        "model_inference": 0,
        "metrics_computed": 0,
        "thresholds_selected": 0,
        "stage27_0_frozen": False,
        "stage27_0a_required_first": True,
    },
    "frozen_taxonomy_expected": list(FROZEN_FAMILIES),
    "required_stage27_0a_artifacts": list(EXPECTED_STAGE27_0A_FILES),
    "stage26_complete_evidence": [
        {
            "file": str(hit["file"].relative_to(REPO)),
            "json_path": hit["json_path"],
            "value": hit["value"],
            "sha256": sha256_file(hit["file"]),
        }
        for hit in closure_hits
    ],
    "kaggle_input": {
        "dataset_directory_count": len(dataset_dirs),
        "file_count": len(all_input_files),
        "candidate_support_file_count": len(candidate_input_files),
    },
}

LOCAL_BOOTSTRAP_RECEIPT = (
    WORK_ROOT / "stage27_fresh_bootstrap_receipt.json"
)

LOCAL_BOOTSTRAP_RECEIPT.write_text(
    json.dumps(
        bootstrap_receipt,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

print("Local receipt             :", LOCAL_BOOTSTRAP_RECEIPT)
print("Receipt in Git repository : NO")
print("Receipt SHA256            :", sha256_file(LOCAL_BOOTSTRAP_RECEIPT))


# ======================================================================================
# 15. REMOVE TEMPORARY SECRET MATERIAL
# ======================================================================================

banner("STAGE27 :: REMOVE TEMPORARY AUTH MATERIAL")

# Token was never placed in repository or remote URL.
git_env.pop("GITHUB_TOKEN", None)

try:
    ASKPASS.unlink()
    print("Temporary askpass removed : YES")
except FileNotFoundError:
    print("Temporary askpass removed : already absent")

github_token = None

print("Token retained in variable: NO")


# ======================================================================================
# 16. FINAL BOOTSTRAP AUDIT
# ======================================================================================

banner("STAGE27 FRESH BOOTSTRAP COMPLETE")

final_status = git("status", "--porcelain")

print("Repository:")
print(" ", REPO)

print()
print("Execution parent:")
print(" ", local_head)

print()
print("Remote main:")
print(" ", remote_sha)

print()
print("HEAD == origin/main == remote:")
print(" ", local_head == origin_main == remote_sha)

print()
print("Stage26 closure:")
print("  COMPLETE evidence : VERIFIED")
print("  evidence files    :", len(closure_hits))

print()
print("Prior provenance:")
print("  Stage22 candidates:", len(prior_stage_inventory[22]))
print("  Stage23 candidates:", len(prior_stage_inventory[23]))
print("  Stage24 candidates:", len(prior_stage_inventory[24]))
print("  taxonomy candidates:", len(taxonomy_candidate_hits))

print()
print("Kaggle inputs:")
print("  dataset dirs      :", len(dataset_dirs))
print("  total files       :", len(all_input_files))
print("  support candidates:", len(candidate_input_files))

print()
print("STAGE27 SCIENTIFIC STATE")
print("  Stage27-0A started       : DIRECTORY ONLY")
print("  Stage27-0 frozen         : NO")
print("  model fits               : 0")
print("  model inference          : 0")
print("  metrics calculated       : 0")
print("  thresholds selected      : 0")
print("  bootstrap replicates     : 0")
print("  GPU execution            : OFF")
print("  primary GPU budget used  : 0 hours")
print("  Git working tree clean   :", final_status == "")

if final_status:
    raise RuntimeError(
        "\nRepository became dirty during bootstrap:\n"
        + final_status
    )

print()
print("NEXT AUTHORIZED ACTION ONLY:")
print(
    "  Stage27-0A — reconstruct the exact seven-family × weekday support "
    "matrix and benign weekday support from durable Stage24/repository/data "
    "evidence. ZERO model fits. ZERO inference."
)

print("=" * 100)


STAGE27 :: FRESH-RUNTIME PREFLIGHT
timestamp_utc             : 2026-08-20T16:34:58.978472+00:00
hostname                  : c6f469f4a718
python                    : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
python_executable         : /usr/bin/python3
platform                  : Linux-6.12.90+-x86_64-with-glibc2.35
machine                   : x86_64
working_directory         : /kaggle/working
kaggle_working_exists     : True
kaggle_input_exists       : True
logical_cpus              : 4
physical_cores            : 2
ram_total_gib             : 31.348
ram_available_gib         : 30.042

PRIMARY_EXECUTION_DEVICE  : CPU
PRIMARY_GPU_BUDGET_HOURS  : 0
CUDA_VISIBLE_DEVICES      : ''

STAGE27 :: GITHUB SECRET
GitHub secret label       : GITHUB_TOKEN
GitHub secret available   : YES
GitHub token length       : 93
GitHub token printed      : NO

STAGE27 :: CONFIGURE TEMPORARY GIT AUTH
Temporary askpass         : /kaggle/working/.stage27_git_askpass.sh
Interactive prompt        : DISABL

In [3]:
# ======================================================================================
# STAGE 27 — FRESH KAGGLE BOOTSTRAP / PROVENANCE PREFLIGHT
# ======================================================================================
#
# PURPOSE
# -------
# Start Stage 27 from a clean Kaggle runtime and establish provenance BEFORE
# Stage27-0A reads label supports.
#
# THIS CELL:
#   1. Forces primary Stage27 execution policy to CPU / GPU budget = 0
#   2. Prints runtime/system information
#   3. Verifies Kaggle GitHub secret without printing it
#   4. Fresh-clones or safely refreshes:
#          themubasshir/ids2018-validation-safe-ablation
#   5. Pins the exact origin/main SHA used by this notebook
#   6. Verifies clean Git state
#   7. Verifies durable Stage26 COMPLETE evidence exists
#   8. Inventories Stage22 / Stage23 / Stage24 provenance candidates
#   9. Inventories attached Kaggle input files
#  10. Creates ONLY the Stage27-0A working directory
#
# STRICT SCIENTIFIC STATE AFTER THIS CELL:
#   Stage27 model fits       = 0
#   Stage27 model inference  = 0
#   Stage27 metrics          = 0
#   Stage27 thresholds       = 0
#   Stage27 GPU budget used  = 0
#   Stage27-0 frozen         = NO
#
# IMPORTANT:
#   Do NOT manually paste a GitHub token into this notebook.
# ======================================================================================

from __future__ import annotations

import os
import sys
import re
import json
import hashlib
import platform
import socket
import subprocess
import shutil
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict


# ======================================================================================
# 0. STAGE 27 COMPUTE POLICY — FREEZE FOR THIS RUNTIME
# ======================================================================================

STAGE = "Stage27"
PRIMARY_EXECUTION_DEVICE = "CPU"
PRIMARY_GPU_BUDGET_HOURS = 0

# Prevent CUDA-aware libraries imported later from seeing GPUs.
# Stage 27 primary experiment is deliberately CPU-only.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["NVIDIA_VISIBLE_DEVICES"] = ""
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

# Avoid accidental massive thread oversubscription.
LOGICAL_CPUS = os.cpu_count() or 1
CPU_THREADS = max(1, LOGICAL_CPUS)

os.environ.setdefault("OMP_NUM_THREADS", str(CPU_THREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(CPU_THREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(CPU_THREADS))
os.environ.setdefault("NUMEXPR_NUM_THREADS", str(CPU_THREADS))


# ======================================================================================
# 1. CONSTANTS
# ======================================================================================

OWNER = "themubasshir"
REPO_NAME = "ids2018-validation-safe-ablation"
BRANCH = "main"

REPO_URL = f"https://github.com/{OWNER}/{REPO_NAME}.git"

WORK_ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")
REPO = WORK_ROOT / REPO_NAME

STAGE27_ROOT = REPO / "results" / "stage27_loao_unseen_attack"
STAGE27_0A = STAGE27_ROOT / "stage27_0a_family_day_feasibility"

FROZEN_FAMILIES = (
    "BOT",
    "DDOS",
    "DOS",
    "AUTH_BRUTE_FORCE",
    "INFILTRATION",
    "PORT_SCAN",
    "WEB_ATTACK",
)

EXPECTED_STAGE27_0A_FILES = (
    "family_day_support.csv",
    "benign_day_support.csv",
    "taxonomy_receipt.json",
    "source_artifact_receipts.json",
    "temporal_feasibility.json",
    "feasibility_audit.md",
)


# ======================================================================================
# 2. HELPERS
# ======================================================================================

def banner(title: str) -> None:
    print()
    print("=" * 100)
    print(title)
    print("=" * 100)


def run(
    cmd,
    *,
    cwd: Path | None = None,
    env: dict | None = None,
    check: bool = True,
) -> str:
    result = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    if check and result.returncode != 0:
        raise RuntimeError(
            "\nCOMMAND FAILED\n"
            f"Command : {' '.join(str(x) for x in cmd)}\n"
            f"Code    : {result.returncode}\n"
            f"STDOUT:\n{result.stdout}\n"
            f"STDERR:\n{result.stderr}"
        )

    return result.stdout.strip()


def git(*args, env=None) -> str:
    return run(["git", *args], cwd=REPO, env=env)


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def human_bytes(n: int) -> str:
    value = float(n)
    units = ["B", "KiB", "MiB", "GiB", "TiB"]

    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f"{value:,.2f} {unit}"
        value /= 1024

    return f"{n:,} B"


def recursively_find_key(obj, wanted_key: str):
    """
    Return [(path_within_json, value), ...] for exact dictionary-key matches.
    """
    found = []

    def walk(x, path="$"):
        if isinstance(x, dict):
            for key, value in x.items():
                child = f"{path}.{key}"
                if key == wanted_key:
                    found.append((child, value))
                walk(value, child)

        elif isinstance(x, list):
            for i, value in enumerate(x):
                walk(value, f"{path}[{i}]")

    walk(obj)
    return found


# ======================================================================================
# 3. RUNTIME
# ======================================================================================

banner("STAGE27 :: FRESH-RUNTIME PREFLIGHT")

print("timestamp_utc             :", datetime.now(timezone.utc).isoformat())
print("hostname                  :", socket.gethostname())
print("python                    :", sys.version.replace("\n", " "))
print("python_executable         :", sys.executable)
print("platform                  :", platform.platform())
print("machine                   :", platform.machine())
print("working_directory         :", Path.cwd())
print("kaggle_working_exists     :", WORK_ROOT.exists())
print("kaggle_input_exists       :", INPUT_ROOT.exists())
print("logical_cpus              :", LOGICAL_CPUS)

try:
    import psutil

    print("physical_cores            :", psutil.cpu_count(logical=False))
    print("ram_total_gib             :", round(psutil.virtual_memory().total / 2**30, 3))
    print("ram_available_gib         :", round(psutil.virtual_memory().available / 2**30, 3))
except Exception as exc:
    print("psutil_runtime_info       : unavailable:", repr(exc))

print()
print("PRIMARY_EXECUTION_DEVICE  :", PRIMARY_EXECUTION_DEVICE)
print("PRIMARY_GPU_BUDGET_HOURS  :", PRIMARY_GPU_BUDGET_HOURS)
print("CUDA_VISIBLE_DEVICES      :", repr(os.environ["CUDA_VISIBLE_DEVICES"]))

if PRIMARY_EXECUTION_DEVICE != "CPU":
    raise RuntimeError("Stage27 primary execution device must be CPU.")

if PRIMARY_GPU_BUDGET_HOURS != 0:
    raise RuntimeError("Stage27 primary GPU budget must be exactly zero.")


# ======================================================================================
# 4. CHECK GITHUB SECRET
# ======================================================================================

banner("STAGE27 :: GITHUB SECRET")

try:
    from kaggle_secrets import UserSecretsClient
except Exception as exc:
    raise RuntimeError(
        "Kaggle UserSecretsClient is unavailable. "
        "This cell must run inside Kaggle."
    ) from exc


secret_client = UserSecretsClient()

# We know previous notebooks have used these labels.
# Do not display the secret value.
SECRET_LABELS = (
    "GITHUB_TOKEN",
    "github_token",
    "GH_TOKEN",
    "github_pat",
    "GITHUB_PAT",
    "GH_PAT",
)

github_token = None
github_secret_label = None

for label in SECRET_LABELS:
    try:
        candidate = secret_client.get_secret(label)
    except Exception:
        candidate = None

    if candidate and len(candidate.strip()) >= 20:
        github_token = candidate.strip()
        github_secret_label = label
        break


if not github_token:
    raise RuntimeError(
        "\nNo usable GitHub token was accessible from Kaggle Secrets.\n\n"
        "Expected one of these secret labels:\n"
        + "\n".join(f"  - {x}" for x in SECRET_LABELS)
        + "\n\nEnable notebook access to the existing secret and rerun this cell.\n"
        "Do NOT paste the token into notebook source."
    )


print("GitHub secret label       :", github_secret_label)
print("GitHub secret available   : YES")
print("GitHub token length       :", len(github_token))
print("GitHub token printed      : NO")


# ======================================================================================
# 5. SAFE GIT AUTHENTICATION VIA GIT_ASKPASS
# ======================================================================================

banner("STAGE27 :: CONFIGURE TEMPORARY GIT AUTH")

ASKPASS = WORK_ROOT / ".stage27_git_askpass.sh"

ASKPASS.write_text(
    """#!/bin/sh
case "$1" in
    *Username*) echo "x-access-token" ;;
    *Password*) echo "$GITHUB_TOKEN" ;;
    *) echo "" ;;
esac
""",
    encoding="utf-8",
)

ASKPASS.chmod(0o700)

git_env = os.environ.copy()
git_env["GITHUB_TOKEN"] = github_token
git_env["GIT_ASKPASS"] = str(ASKPASS)
git_env["GIT_TERMINAL_PROMPT"] = "0"

print("Temporary askpass         :", ASKPASS)
print("Interactive prompt        : DISABLED")
print("Credential in command URL : NO")


# ======================================================================================
# 6. CLONE / SAFELY REFRESH REPOSITORY
# ======================================================================================

banner("STAGE27 :: REPOSITORY BOOTSTRAP")

if not REPO.exists():
    print("Repository does not exist in this fresh runtime.")
    print("Cloning origin/main...")

    run(
        [
            "git",
            "clone",
            "--branch",
            BRANCH,
            "--single-branch",
            REPO_URL,
            str(REPO),
        ],
        env=git_env,
    )

else:
    print("Repository directory already exists:", REPO)

    if not (REPO / ".git").exists():
        raise RuntimeError(
            f"{REPO} exists but is not a Git repository. "
            "Refusing to delete or overwrite it."
        )

    existing_status = run(
        ["git", "status", "--porcelain"],
        cwd=REPO,
    )

    if existing_status:
        raise RuntimeError(
            "\nExisting repository contains modifications.\n"
            "Refusing to reset or overwrite them.\n\n"
            + existing_status
        )

    print("Existing repository is clean.")
    print("Refreshing origin/main...")

    run(
        ["git", "fetch", "--prune", "origin", BRANCH],
        cwd=REPO,
        env=git_env,
    )

    run(
        ["git", "checkout", BRANCH],
        cwd=REPO,
        env=git_env,
    )

    run(
        ["git", "reset", "--hard", f"origin/{BRANCH}"],
        cwd=REPO,
        env=git_env,
    )


# Ensure remote is canonical and contains no embedded credential.
git("remote", "set-url", "origin", REPO_URL)

remote_url = git("remote", "get-url", "origin")

if github_token in remote_url:
    raise RuntimeError("Credential leaked into Git remote URL.")

print("Repository                :", REPO)
print("Remote                    :", remote_url)


# ======================================================================================
# 7. PIN CURRENT MAIN
# ======================================================================================

banner("STAGE27 :: PIN CURRENT MAIN")

# Fetch once more with authenticated environment.
run(
    ["git", "fetch", "--prune", "origin", BRANCH],
    cwd=REPO,
    env=git_env,
)

local_head = git("rev-parse", "HEAD")
origin_main = git("rev-parse", f"origin/{BRANCH}")
branch_name = git("branch", "--show-current")
status = git("status", "--porcelain")
commit_subject = git("log", "-1", "--pretty=%s")
commit_time = git("log", "-1", "--format=%cI")

print("branch                    :", branch_name)
print("local HEAD                :", local_head)
print("origin/main               :", origin_main)
print("HEAD subject              :", commit_subject)
print("HEAD committed            :", commit_time)
print("working tree clean        :", status == "")

if branch_name != BRANCH:
    raise RuntimeError(f"Expected branch {BRANCH}, got {branch_name!r}.")

if local_head != origin_main:
    raise RuntimeError(
        "Fresh Stage27 bootstrap is not exactly pinned to origin/main."
    )

if status:
    raise RuntimeError(
        "Repository is unexpectedly dirty immediately after bootstrap:\n"
        + status
    )


# ======================================================================================
# 8. VERIFY REMOTE SHA DIRECTLY
# ======================================================================================

banner("STAGE27 :: REMOTE SHA VERIFICATION")

ls_remote_output = run(
    ["git", "ls-remote", "origin", f"refs/heads/{BRANCH}"],
    cwd=REPO,
    env=git_env,
)

if not ls_remote_output:
    raise RuntimeError("git ls-remote returned no main-branch SHA.")

remote_sha = ls_remote_output.split()[0]

print("local HEAD                :", local_head)
print("origin/main tracking      :", origin_main)
print("remote refs/heads/main    :", remote_sha)

if not (local_head == origin_main == remote_sha):
    raise RuntimeError(
        "Local HEAD, origin/main, and remote main are not identical."
    )

print("[OK] Exact Stage27 execution parent pinned.")


# ======================================================================================
# 9. VERIFY STAGE 26 CLOSURE EVIDENCE
# ======================================================================================

banner("STAGE27 :: VERIFY DURABLE STAGE26 CLOSURE")

stage26_root_candidates = sorted(
    p
    for p in (REPO / "results").glob("stage26*")
    if p.exists()
)

if not stage26_root_candidates:
    raise RuntimeError("No Stage26 result directory exists in the repository.")

print("Stage26 result roots:")
for p in stage26_root_candidates:
    print(" ", p.relative_to(REPO))


stage26_json_files = []

for root in stage26_root_candidates:
    stage26_json_files.extend(root.rglob("*.json"))

stage26_json_files = sorted(set(stage26_json_files))

print()
print("Stage26 JSON files found  :", len(stage26_json_files))


closure_hits = []

for path in stage26_json_files:
    try:
        obj = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        continue

    matches = recursively_find_key(obj, "stage26_status")

    for json_path, value in matches:
        if str(value).strip().upper() == "COMPLETE":
            closure_hits.append(
                {
                    "file": path,
                    "json_path": json_path,
                    "value": value,
                }
            )


# Secondary textual check ONLY as provenance visibility.
# The primary pass condition remains exact stage26_status == COMPLETE.
closure_named_files = [
    p
    for p in stage26_json_files
    if any(
        token in p.name.lower()
        for token in ("closure", "final", "seal", "receipt")
    )
]


print()
print("Exact stage26_status=COMPLETE hits:")

for hit in closure_hits:
    print(
        " ",
        hit["file"].relative_to(REPO),
        "::",
        hit["json_path"],
        "=",
        hit["value"],
    )


if not closure_hits:
    print()
    print("Potential closure/final JSON files:")
    for p in closure_named_files[:50]:
        print(" ", p.relative_to(REPO))

    raise RuntimeError(
        "\nCould not verify an exact durable JSON field:\n"
        '    "stage26_status": "COMPLETE"\n\n'
        "Do not begin Stage27-0A until Stage26 closure provenance is resolved."
    )


print()
print("[OK] Durable Stage26 COMPLETE evidence verified.")


# ======================================================================================
# 10. INVENTORY STAGE22 / STAGE23 / STAGE24 PROVENANCE
# ======================================================================================

banner("STAGE27 :: PRIOR-STAGE PROVENANCE INVENTORY")


def collect_stage_files(stage_number: int):
    stage_token = f"stage{stage_number}"
    files = []

    for base in (REPO / "results", REPO / "docs"):
        if not base.exists():
            continue

        for path in base.rglob("*"):
            if path.is_file() and stage_token in str(path.relative_to(REPO)).lower():
                files.append(path)

    return sorted(set(files))


prior_stage_inventory = {}

for n in (22, 23, 24):
    files = collect_stage_files(n)
    prior_stage_inventory[n] = files

    print()
    print(f"Stage {n}: {len(files)} candidate provenance files")

    interesting = [
        p
        for p in files
        if any(
            word in p.name.lower()
            for word in (
                "taxonomy",
                "family",
                "temporal",
                "support",
                "manifest",
                "receipt",
                "protocol",
                "lock",
                "closure",
                "audit",
                "mapping",
            )
        )
    ]

    for p in interesting[:60]:
        print(" ", p.relative_to(REPO))

    if len(interesting) > 60:
        print(f"  ... {len(interesting) - 60} additional relevant-looking files")


if not prior_stage_inventory[22]:
    raise RuntimeError("No Stage22 provenance candidates found.")

if not prior_stage_inventory[23]:
    raise RuntimeError("No Stage23 provenance candidates found.")

if not prior_stage_inventory[24]:
    raise RuntimeError("No Stage24 provenance candidates found.")


# ======================================================================================
# 11. SEARCH REPOSITORY FOR THE FROZEN SEVEN-FAMILY TAXONOMY
# ======================================================================================

banner("STAGE27 :: TAXONOMY DISCOVERY")

text_suffixes = {
    ".json",
    ".csv",
    ".md",
    ".txt",
    ".yaml",
    ".yml",
    ".py",
}

taxonomy_candidate_hits = []

search_bases = [
    REPO / "results",
    REPO / "docs",
]

for base in search_bases:
    if not base.exists():
        continue

    for path in base.rglob("*"):
        if not path.is_file():
            continue

        if path.suffix.lower() not in text_suffixes:
            continue

        path_l = str(path.relative_to(REPO)).lower()

        # Stage24 is the frozen family-taxonomy authority for Stage27.
        if "stage24" not in path_l:
            continue

        try:
            # Do not read pathological files into memory.
            if path.stat().st_size > 50 * 1024 * 1024:
                continue

            text = path.read_text(
                encoding="utf-8",
                errors="replace",
            )
        except Exception:
            continue

        upper = text.upper()

        found = [
            family
            for family in FROZEN_FAMILIES
            if family in upper
        ]

        if len(found) >= 5:
            taxonomy_candidate_hits.append(
                (path, tuple(found))
            )


print("Frozen Stage27 families:")
for family in FROZEN_FAMILIES:
    print(" ", family)

print()
print("Stage24 taxonomy/support candidate artifacts:")

for path, families in taxonomy_candidate_hits[:50]:
    print(
        " ",
        path.relative_to(REPO),
        f" [{len(families)}/7 families detected]",
    )


if not taxonomy_candidate_hits:
    raise RuntimeError(
        "Could not locate a Stage24 artifact containing the inherited "
        "attack-family taxonomy. Stage27-0A must not invent a replacement."
    )


# ======================================================================================
# 12. INVENTORY KAGGLE INPUT DATA
# ======================================================================================

banner("STAGE27 :: KAGGLE INPUT INVENTORY")

if not INPUT_ROOT.exists():
    raise RuntimeError("/kaggle/input does not exist.")


dataset_dirs = sorted(
    p
    for p in INPUT_ROOT.iterdir()
    if p.is_dir()
)

print("Attached dataset directories:", len(dataset_dirs))

for d in dataset_dirs:
    print(" ", d.name)


all_input_files = sorted(
    p
    for p in INPUT_ROOT.rglob("*")
    if p.is_file()
)

print()
print("Total attached input files  :", len(all_input_files))

suffix_counter = Counter(
    p.suffix.lower() if p.suffix else "<no_extension>"
    for p in all_input_files
)

print()
print("Input extensions:")

for suffix, count in suffix_counter.most_common():
    print(f"  {suffix:15s} {count:6d}")


# Candidate files relevant to CICIDS2017 / Stage27 support recovery.
keywords = (
    "monday",
    "tuesday",
    "wednesday",
    "thursday",
    "friday",
    "cicids",
    "cic-ids",
    "ids2017",
    "workinghours",
    "label",
    "stage22",
    "stage23",
    "stage24",
)

candidate_input_files = [
    p
    for p in all_input_files
    if any(k in p.name.lower() for k in keywords)
]


print()
print("Potential Stage27 support/provenance input files:")

for p in candidate_input_files[:200]:
    try:
        size = human_bytes(p.stat().st_size)
    except Exception:
        size = "?"

    print(
        f"  {str(p.relative_to(INPUT_ROOT)):100s} {size:>12s}"
    )


if len(candidate_input_files) > 200:
    print(
        f"\n  ... {len(candidate_input_files) - 200} additional candidate files"
    )


# ======================================================================================
# 13. CREATE ONLY STAGE27-0A WORK DIRECTORY
# ======================================================================================

banner("STAGE27 :: INITIALIZE ZERO-FIT FEASIBILITY DIRECTORY")

STAGE27_0A.mkdir(parents=True, exist_ok=True)

print("Stage27 root              :", STAGE27_ROOT)
print("Stage27-0A directory      :", STAGE27_0A)

existing_0a_files = sorted(
    p.relative_to(STAGE27_0A)
    for p in STAGE27_0A.rglob("*")
    if p.is_file()
)

if existing_0a_files:
    raise RuntimeError(
        "\nStage27-0A directory already contains files:\n"
        + "\n".join(f"  {p}" for p in existing_0a_files)
        + "\n\nFresh Stage27 bootstrap expected an empty Stage27-0A directory."
    )

print("Stage27-0A currently empty: YES")
print("Stage27-0 created          : NO")
print("Protocol frozen            : NO")


# ======================================================================================
# 14. WRITE LOCAL NON-GIT BOOTSTRAP RECEIPT
# ======================================================================================
#
# This receipt deliberately lives in /kaggle/working OUTSIDE the repository.
# It records the execution parent and discovery state but does not prematurely
# create a Stage27 scientific artifact before the feasibility audit.
# ======================================================================================

banner("STAGE27 :: LOCAL BOOTSTRAP RECEIPT")

bootstrap_receipt = {
    "stage": "Stage27",
    "substage": "fresh_bootstrap",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "repository": {
        "owner": OWNER,
        "name": REPO_NAME,
        "branch": BRANCH,
        "remote_url": REPO_URL,
        "execution_parent_commit": local_head,
        "origin_main_commit": origin_main,
        "remote_main_commit": remote_sha,
        "head_subject": commit_subject,
        "head_commit_time": commit_time,
        "working_tree_clean_before_stage27_0a": True,
    },
    "scientific_policy": {
        "primary_execution_device": PRIMARY_EXECUTION_DEVICE,
        "primary_gpu_budget_hours": PRIMARY_GPU_BUDGET_HOURS,
        "model_fits": 0,
        "model_inference": 0,
        "metrics_computed": 0,
        "thresholds_selected": 0,
        "stage27_0_frozen": False,
        "stage27_0a_required_first": True,
    },
    "frozen_taxonomy_expected": list(FROZEN_FAMILIES),
    "required_stage27_0a_artifacts": list(EXPECTED_STAGE27_0A_FILES),
    "stage26_complete_evidence": [
        {
            "file": str(hit["file"].relative_to(REPO)),
            "json_path": hit["json_path"],
            "value": hit["value"],
            "sha256": sha256_file(hit["file"]),
        }
        for hit in closure_hits
    ],
    "kaggle_input": {
        "dataset_directory_count": len(dataset_dirs),
        "file_count": len(all_input_files),
        "candidate_support_file_count": len(candidate_input_files),
    },
}

LOCAL_BOOTSTRAP_RECEIPT = (
    WORK_ROOT / "stage27_fresh_bootstrap_receipt.json"
)

LOCAL_BOOTSTRAP_RECEIPT.write_text(
    json.dumps(
        bootstrap_receipt,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

print("Local receipt             :", LOCAL_BOOTSTRAP_RECEIPT)
print("Receipt in Git repository : NO")
print("Receipt SHA256            :", sha256_file(LOCAL_BOOTSTRAP_RECEIPT))


# ======================================================================================
# 15. REMOVE TEMPORARY SECRET MATERIAL
# ======================================================================================

banner("STAGE27 :: REMOVE TEMPORARY AUTH MATERIAL")

# Token was never placed in repository or remote URL.
git_env.pop("GITHUB_TOKEN", None)

try:
    ASKPASS.unlink()
    print("Temporary askpass removed : YES")
except FileNotFoundError:
    print("Temporary askpass removed : already absent")

github_token = None

print("Token retained in variable: NO")


# ======================================================================================
# 16. FINAL BOOTSTRAP AUDIT
# ======================================================================================

banner("STAGE27 FRESH BOOTSTRAP COMPLETE")

final_status = git("status", "--porcelain")

print("Repository:")
print(" ", REPO)

print()
print("Execution parent:")
print(" ", local_head)

print()
print("Remote main:")
print(" ", remote_sha)

print()
print("HEAD == origin/main == remote:")
print(" ", local_head == origin_main == remote_sha)

print()
print("Stage26 closure:")
print("  COMPLETE evidence : VERIFIED")
print("  evidence files    :", len(closure_hits))

print()
print("Prior provenance:")
print("  Stage22 candidates:", len(prior_stage_inventory[22]))
print("  Stage23 candidates:", len(prior_stage_inventory[23]))
print("  Stage24 candidates:", len(prior_stage_inventory[24]))
print("  taxonomy candidates:", len(taxonomy_candidate_hits))

print()
print("Kaggle inputs:")
print("  dataset dirs      :", len(dataset_dirs))
print("  total files       :", len(all_input_files))
print("  support candidates:", len(candidate_input_files))

print()
print("STAGE27 SCIENTIFIC STATE")
print("  Stage27-0A started       : DIRECTORY ONLY")
print("  Stage27-0 frozen         : NO")
print("  model fits               : 0")
print("  model inference          : 0")
print("  metrics calculated       : 0")
print("  thresholds selected      : 0")
print("  bootstrap replicates     : 0")
print("  GPU execution            : OFF")
print("  primary GPU budget used  : 0 hours")
print("  Git working tree clean   :", final_status == "")

if final_status:
    raise RuntimeError(
        "\nRepository became dirty during bootstrap:\n"
        + final_status
    )

print()
print("NEXT AUTHORIZED ACTION ONLY:")
print(
    "  Stage27-0A — reconstruct the exact seven-family × weekday support "
    "matrix and benign weekday support from durable Stage24/repository/data "
    "evidence. ZERO model fits. ZERO inference."
)

print("=" * 100)


STAGE27 :: FRESH-RUNTIME PREFLIGHT
timestamp_utc             : 2026-08-20T16:35:50.719839+00:00
hostname                  : c6f469f4a718
python                    : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
python_executable         : /usr/bin/python3
platform                  : Linux-6.12.90+-x86_64-with-glibc2.35
machine                   : x86_64
working_directory         : /kaggle/working
kaggle_working_exists     : True
kaggle_input_exists       : True
logical_cpus              : 4
physical_cores            : 2
ram_total_gib             : 31.348
ram_available_gib         : 30.01

PRIMARY_EXECUTION_DEVICE  : CPU
PRIMARY_GPU_BUDGET_HOURS  : 0
CUDA_VISIBLE_DEVICES      : ''

STAGE27 :: GITHUB SECRET
GitHub secret label       : GITHUB_TOKEN
GitHub secret available   : YES
GitHub token length       : 93
GitHub token printed      : NO

STAGE27 :: CONFIGURE TEMPORARY GIT AUTH
Temporary askpass         : /kaggle/working/.stage27_git_askpass.sh
Interactive prompt        : DISABLE

In [4]:
# ======================================================================================
# STAGE27-0A — EVIDENCE INSPECTION
# ======================================================================================
#
# ZERO MODEL FITS
# ZERO MODEL INFERENCE
# ZERO CORPUS DOWNLOADS
# ZERO SCIENTIFIC ARTIFACT WRITES
#
# PURPOSE:
#   Inspect the exact durable evidence that will be used to construct:
#
#       family_day_support.csv
#       benign_day_support.csv
#       taxonomy_receipt.json
#       source_artifact_receipts.json
#       temporal_feasibility.json
#       feasibility_audit.md
#
# We inspect first, write second.
# ======================================================================================

from __future__ import annotations

import json
import hashlib
import subprocess
from pathlib import Path
from collections import Counter
import pandas as pd


# ======================================================================================
# 1. PATHS
# ======================================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

INPUT_ROOT = Path("/kaggle/input")

EXPECTED_HEAD = (
    "e47f44751bc71d219c5d0f3b3fca06d62037fb8b"
)

PROTOCOL = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0c_final_preopening_protocol_lock.json"
)

SOURCE_CONTRACT = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0b2_complete_cicids2017_source_contract.json"
)

SOURCE_RECOVERY = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0b_exact_cicids2017_source_recovery.json"
)

PUBLISHED_RESULT = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_2_primary_target_openings"
    / "stage24_2a_bridge62_published"
    / "stage24_2a_bridge62_published_result.json"
)

STAGE27_0A = (
    REPO
    / "results"
    / "stage27_loao_unseen_attack"
    / "stage27_0a_family_day_feasibility"
)

EXPECTED_FAMILIES = [
    "BOT",
    "DDOS",
    "DOS",
    "AUTH_BRUTE_FORCE",
    "INFILTRATION",
    "PORT_SCAN",
    "WEB_ATTACK",
]


# ======================================================================================
# 2. HELPERS
# ======================================================================================

def banner(title):
    print()
    print("=" * 110)
    print(title)
    print("=" * 110)


def run(cmd):
    r = subprocess.run(
        [str(x) for x in cmd],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    if r.returncode != 0:
        raise RuntimeError(
            f"Command failed:\n{' '.join(cmd)}\n\n"
            f"STDOUT:\n{r.stdout}\n\n"
            f"STDERR:\n{r.stderr}"
        )

    return r.stdout.strip()


def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()

    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def read_json(path):
    if not path.is_file():
        raise FileNotFoundError(path)

    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def walk(obj, path="$"):
    """
    Recursively yield:
        json_path, value
    """
    yield path, obj

    if isinstance(obj, dict):
        for key, value in obj.items():
            yield from walk(
                value,
                f"{path}.{key}",
            )

    elif isinstance(obj, list):
        for i, value in enumerate(obj):
            yield from walk(
                value,
                f"{path}[{i}]",
            )


def dictionaries_containing_labels(obj):
    """
    Find dictionaries that look like canonical label-count dictionaries.
    """
    hits = []

    expected_tokens = {
        "benign",
        "bot",
        "ddos",
        "dos hulk",
        "dos goldeneye",
        "heartbleed",
        "infiltration",
        "portscan",
        "ftp-patator",
        "ssh-patator",
    }

    for path, value in walk(obj):

        if not isinstance(value, dict):
            continue

        keys = {
            str(k).strip().lower()
            for k in value.keys()
        }

        overlap = sorted(
            expected_tokens.intersection(keys)
        )

        if len(overlap) >= 5:

            numeric = True

            for v in value.values():
                if not isinstance(
                    v,
                    (
                        int,
                        float,
                    ),
                ):
                    numeric = False
                    break

            if numeric:
                hits.append(
                    (
                        path,
                        value,
                        overlap,
                    )
                )

    return hits


def find_file_level_lists(obj):
    """
    Locate lists containing dictionaries with day/rows/attack/benign fields.
    """
    hits = []

    for path, value in walk(obj):

        if not isinstance(value, list):
            continue

        if not value:
            continue

        if not all(
            isinstance(x, dict)
            for x in value
        ):
            continue

        keys = set()

        for row in value:
            keys.update(
                str(k)
                for k in row.keys()
            )

        required = {
            "day",
            "rows",
            "attack",
            "benign",
        }

        if required.issubset(keys):
            hits.append(
                (
                    path,
                    value,
                )
            )

    return hits


# ======================================================================================
# 3. GIT GATE
# ======================================================================================

banner("STAGE27-0A :: GIT GATE")

head = run(
    ["git", "rev-parse", "HEAD"]
)

remote = run(
    [
        "git",
        "rev-parse",
        "origin/main",
    ]
)

status = run(
    [
        "git",
        "status",
        "--porcelain",
    ]
)

print("Expected parent :", EXPECTED_HEAD)
print("Local HEAD      :", head)
print("origin/main     :", remote)
print("Repo clean      :", status == "")

if head != EXPECTED_HEAD:
    raise RuntimeError(
        "Unexpected Stage27 execution parent."
    )

if remote != EXPECTED_HEAD:
    raise RuntimeError(
        "origin/main changed since Stage27 bootstrap."
    )

if status:
    raise RuntimeError(
        "Repository is dirty before Stage27-0A inspection."
    )

print("[PASS] Git state unchanged.")


# ======================================================================================
# 4. STAGE27 DIRECTORY GATE
# ======================================================================================

banner("STAGE27-0A :: OUTPUT DIRECTORY GATE")

if not STAGE27_0A.is_dir():
    raise RuntimeError(
        "Expected Stage27-0A directory is missing."
    )

existing = sorted(
    p.relative_to(STAGE27_0A)
    for p in STAGE27_0A.rglob("*")
    if p.is_file()
)

print("Stage27-0A directory:")
print(" ", STAGE27_0A)

print()
print("Existing files:", len(existing))

for p in existing:
    print(" ", p)

if existing:
    raise RuntimeError(
        "Stage27-0A must still be empty before evidence inspection."
    )

print("[PASS] No Stage27-0A artifacts exist yet.")


# ======================================================================================
# 5. DURABLE ARTIFACT INVENTORY
# ======================================================================================

banner("STAGE27-0A :: DURABLE SOURCE ARTIFACTS")

SOURCE_FILES = [
    PROTOCOL,
    SOURCE_CONTRACT,
    SOURCE_RECOVERY,
    PUBLISHED_RESULT,
]

for path in SOURCE_FILES:

    if not path.is_file():
        raise FileNotFoundError(path)

    print()
    print("Artifact:")
    print(" ", path.relative_to(REPO))
    print(" bytes  :", path.stat().st_size)
    print(" sha256 :", sha256_file(path))

print()
print("[PASS] All four Stage24 provenance/result artifacts exist.")


# ======================================================================================
# 6. TAXONOMY
# ======================================================================================

banner("STAGE27-0A :: FROZEN STAGE24 TAXONOMY")

protocol = read_json(PROTOCOL)

taxonomy = protocol.get(
    "attack_family_taxonomy"
)

if not isinstance(
    taxonomy,
    dict,
):
    raise RuntimeError(
        "Stage24 attack_family_taxonomy missing."
    )

for raw_label, family in taxonomy.items():
    print(
        f"{raw_label:35s} -> {family}"
    )


actual_primary_families = sorted(
    {
        str(v)
        for v in taxonomy.values()
        if str(v)
        not in {
            "TARGET_ONLY_UNSEEN",
            "OTHER_ATTACK_UNSEEN_LABEL",
        }
    }
)

print()
print("Primary families recovered:")
for family in actual_primary_families:
    print(" ", family)

if set(actual_primary_families) != set(EXPECTED_FAMILIES):
    raise RuntimeError(
        "\nFrozen family set mismatch.\n"
        f"Expected: {sorted(EXPECTED_FAMILIES)}\n"
        f"Actual  : {actual_primary_families}"
    )

heartbleed_mapping = taxonomy.get(
    "heartbleed"
)

print()
print("Heartbleed mapping:", heartbleed_mapping)

if heartbleed_mapping != "TARGET_ONLY_UNSEEN":
    raise RuntimeError(
        "Heartbleed taxonomy changed."
    )

print("[PASS] Exact seven-family taxonomy recovered.")


# ======================================================================================
# 7. STAGE24 SOURCE PROVENANCE
# ======================================================================================

banner("STAGE27-0A :: EXACT CICIDS2017 SOURCE PROVENANCE")

source_contract = read_json(
    SOURCE_CONTRACT
)

source_recovery = read_json(
    SOURCE_RECOVERY
)

print("Source recovery top-level keys:")
for key in source_recovery.keys():
    print(" ", key)

print()
print("Source contract top-level keys:")
for key in source_contract.keys():
    print(" ", key)


# Print any source-contract/recovery dictionaries containing
# "repo_id", "revision", or exact-verification state.

interesting_source_dicts = []

for json_path, value in walk(
    source_recovery
):

    if not isinstance(
        value,
        dict,
    ):
        continue

    keys = set(
        value.keys()
    )

    if (
        "repo_id" in keys
        or "revision" in keys
        or "all_exact_sources_verified" in keys
    ):
        interesting_source_dicts.append(
            (
                json_path,
                value,
            )
        )


print()
print("Relevant exact-source structures:")

for json_path, value in interesting_source_dicts[:20]:

    print()
    print(json_path)

    for key in [
        "provider",
        "repo_id",
        "revision",
        "all_exact_sources_verified",
        "all_schema_footer_inspections_pass",
    ]:
        if key in value:
            print(
                f"  {key}: {value[key]}"
            )


# ======================================================================================
# 8. STAGE24 PUBLISHED RESULT — FILE/DAY SUPPORT
# ======================================================================================

banner("STAGE27-0A :: STAGE24 FILE/DAY SUPPORT EVIDENCE")

published = read_json(
    PUBLISHED_RESULT
)

file_level_hits = find_file_level_lists(
    published
)

print(
    "File-level descriptive structures found:",
    len(file_level_hits),
)

for json_path, rows in file_level_hits:

    print()
    print("JSON path:", json_path)
    print()

    frame = pd.DataFrame(rows)

    wanted = [
        c
        for c in [
            "file_id",
            "day",
            "remote",
            "rows",
            "benign",
            "attack",
            "start",
            "stop",
            "prevalence",
        ]
        if c in frame.columns
    ]

    print(
        frame[
            wanted
        ].to_string(
            index=False
        )
    )


if not file_level_hits:
    raise RuntimeError(
        "Could not locate committed Stage24 file-level support."
    )


# ======================================================================================
# 9. STAGE24 PUBLISHED RESULT — CANONICAL LABEL COUNTS
# ======================================================================================

banner("STAGE27-0A :: CANONICAL TARGET LABEL COUNTS")

label_dict_hits = dictionaries_containing_labels(
    published
)

print(
    "Candidate canonical label-count dictionaries:",
    len(label_dict_hits),
)

for json_path, counts, overlap in label_dict_hits:

    print()
    print("JSON path:")
    print(" ", json_path)

    print(
        "Recognized labels:",
        len(overlap),
    )

    print()

    for label, count in sorted(
        counts.items(),
        key=lambda x: str(x[0]).lower(),
    ):
        print(
            f"{repr(label):45s} {int(count):12,d}"
        )


# It is possible the canonical counts are represented as a list,
# so also search for attack-label-looking scalar values.

if not label_dict_hits:

    print()
    print(
        "[INFO] No direct canonical-label dictionary detected."
    )
    print(
        "      We will use the attached label summary as the"
        " second durable evidence source."
    )


# ======================================================================================
# 10. ATTACHED KAGGLE LABEL SUMMARY
# ======================================================================================

banner("STAGE27-0A :: ATTACHED LABEL SUMMARY")

summary_candidates = sorted(
    INPUT_ROOT.rglob(
        "all_csv_label_summary.csv"
    )
)

print(
    "all_csv_label_summary.csv candidates:",
    len(summary_candidates),
)

for path in summary_candidates:
    print(" ", path)


if not summary_candidates:

    print()
    print(
        "[INFO] No attached label summary found."
    )

else:

    for path in summary_candidates:

        print()
        print("-" * 110)
        print("FILE:", path)
        print("-" * 110)

        df = pd.read_csv(
            path
        )

        print(
            "shape:",
            df.shape,
        )

        print()
        print(
            "columns:",
            list(df.columns),
        )

        print()
        print(
            df.to_string(
                index=False,
                max_rows=100,
            )
        )


# ======================================================================================
# 11. KNOWN SUPPORT ARITHMETIC — ASSERT ONLY VALUES ALREADY DURABLY RECOVERED
# ======================================================================================

banner("STAGE27-0A :: DURABLE SUPPORT ARITHMETIC")

# These are NOT invented Stage27 values.
#
# They are arithmetic identities over the raw canonical Stage24/Stage20
# label counts and are checked here so any discrepancy becomes fatal
# before Stage27 artifacts are written.

known = {
    "AUTH_BRUTE_FORCE": (
        7_938
        + 5_897
    ),

    "DOS": (
        231_073
        + 10_293
        + 5_796
        + 5_499
    ),

    "WEB_ATTACK": (
        1_507
        + 652
        + 21
    ),

    "BOT":
        1_966,

    "DDOS":
        128_027,

    "INFILTRATION":
        36,

    "PORT_SCAN":
        158_930,

    "HEARTBLEED_TARGET_ONLY_UNSEEN":
        11,
}


for key, value in known.items():
    print(
        f"{key:35s}: {value:,}"
    )


seven_family_total = sum(
    value
    for key, value in known.items()
    if key != "HEARTBLEED_TARGET_ONLY_UNSEEN"
)

all_attack_total = (
    seven_family_total
    + known[
        "HEARTBLEED_TARGET_ONLY_UNSEEN"
    ]
)

print()
print(
    "Seven-family attack total :",
    f"{seven_family_total:,}",
)

print(
    "Heartbleed excluded       :",
    f"{known['HEARTBLEED_TARGET_ONLY_UNSEEN']:,}",
)

print(
    "All non-benign total      :",
    f"{all_attack_total:,}",
)


if known["AUTH_BRUTE_FORCE"] != 13_835:
    raise RuntimeError(
        "AUTH_BRUTE_FORCE arithmetic mismatch."
    )

if known["DOS"] != 252_661:
    raise RuntimeError(
        "DOS arithmetic mismatch."
    )

if known["WEB_ATTACK"] != 2_180:
    raise RuntimeError(
        "WEB_ATTACK arithmetic mismatch."
    )

if seven_family_total != 557_635:
    raise RuntimeError(
        "Seven-family attack total mismatch."
    )

if all_attack_total != 557_646:
    raise RuntimeError(
        "All attack total mismatch."
    )

print()
print("[PASS] Canonical family arithmetic is internally exact.")


# ======================================================================================
# 12. SCIENTIFIC STATE — CONFIRM ZERO-WRITE / ZERO-FIT
# ======================================================================================

banner("STAGE27-0A :: FINAL INSPECTION STATE")

status_after = run(
    [
        "git",
        "status",
        "--porcelain",
    ]
)

stage27_files_after = sorted(
    p.relative_to(
        STAGE27_0A
    )
    for p in STAGE27_0A.rglob("*")
    if p.is_file()
)

print(
    "Stage27 model fits       : 0"
)

print(
    "Stage27 inference        : 0"
)

print(
    "Corpus archives downloaded: 0"
)

print(
    "Release corpus recreated: NO"
)

print(
    "Stage27 artifacts written:",
    len(stage27_files_after),
)

print(
    "Git working tree clean  :",
    status_after == "",
)


if stage27_files_after:
    raise RuntimeError(
        "Evidence inspection unexpectedly wrote Stage27 files."
    )

if status_after:
    raise RuntimeError(
        "Evidence inspection unexpectedly modified Git repository."
    )


print()
print("=" * 110)
print(
    "STAGE27-0A EVIDENCE INSPECTION COMPLETE"
)
print("=" * 110)

print()
print(
    "NEXT:"
)

print(
    "  Construct the exact seven-family × weekday matrix,"
)

print(
    "  benign weekday support, and temporal-feasibility"
)

print(
    "  decision from these durable sources."
)

print()
print(
    "NO MODEL FITTING IS AUTHORIZED YET."
)

print("=" * 110)


STAGE27-0A :: GIT GATE
Expected parent : e47f44751bc71d219c5d0f3b3fca06d62037fb8b
Local HEAD      : e47f44751bc71d219c5d0f3b3fca06d62037fb8b
origin/main     : e47f44751bc71d219c5d0f3b3fca06d62037fb8b
Repo clean      : True
[PASS] Git state unchanged.

STAGE27-0A :: OUTPUT DIRECTORY GATE
Stage27-0A directory:
  /kaggle/working/ids2018-validation-safe-ablation/results/stage27_loao_unseen_attack/stage27_0a_family_day_feasibility

Existing files: 0
[PASS] No Stage27-0A artifacts exist yet.

STAGE27-0A :: DURABLE SOURCE ARTIFACTS

Artifact:
  results/stage24_cross_dataset/stage24_0_protocol_lock/stage24_0c_final_preopening_protocol_lock.json
 bytes  : 20143
 sha256 : 8ef234a9d283f2008f21b9add4361f14328d1f3c1cffa278077f59d9eb9e37c2

Artifact:
  results/stage24_cross_dataset/stage24_0_protocol_lock/stage24_0b2_complete_cicids2017_source_contract.json
 bytes  : 38654
 sha256 : 96f7fc5c0227660fe8ec17f5173630ee2ac71f6be199e81ee37dac0ad25d9779

Artifact:
  results/stage24_cross_dataset/stage24_0

In [5]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Stage27-0A — Family × Day Feasibility Audit
===========================================

Repository:
    themubasshir/ids2018-validation-safe-ablation

Scientific boundary:
    - ZERO model fits
    - ZERO model inference
    - ZERO threshold selection
    - ZERO bootstrap
    - ZERO GPU use
    - writes ONLY the six required Stage27-0A artifacts
    - commits + pushes Stage27-0A to main
    - remotely verifies the resulting commit
    - STOPS before Stage27-0 protocol construction

This script deliberately uses the full Stage24 CICIDS2017 effective target
population for Stage27 family support. The Stage20 compact raw-byte release
corpora are audited as binary-only corpora and therefore are not used as
seven-family support sources.

Expected execution parent:
    e47f44751bc71d219c5d0f3b3fca06d62037fb8b
"""

from __future__ import annotations

import csv
import hashlib
import json
import os
import shutil
import stat
import subprocess
import sys
from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, List, Tuple


# =============================================================================
# 0. FROZEN CONSTANTS
# =============================================================================

REPO_OWNER = "themubasshir"
REPO_NAME = "ids2018-validation-safe-ablation"
REPO_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"

EXPECTED_PARENT = "e47f44751bc71d219c5d0f3b3fca06d62037fb8b"
PRIMARY_EXECUTION_DEVICE = "CPU"
PRIMARY_GPU_BUDGET_HOURS = 0

WEEKDAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]

PRIMARY_FAMILIES = [
    "BOT",
    "DDOS",
    "DOS",
    "AUTH_BRUTE_FORCE",
    "INFILTRATION",
    "PORT_SCAN",
    "WEB_ATTACK",
]

EXPECTED_STAGE24_SOURCE_SHA256 = OrderedDict(
    [
        (
            "results/stage24_cross_dataset/stage24_0_protocol_lock/"
            "stage24_0c_final_preopening_protocol_lock.json",
            "8ef234a9d283f2008f21b9add4361f14328d1f3c1cffa278077f59d9eb9e37c2",
        ),
        (
            "results/stage24_cross_dataset/stage24_0_protocol_lock/"
            "stage24_0b2_complete_cicids2017_source_contract.json",
            "96f7fc5c0227660fe8ec17f5173630ee2ac71f6be199e81ee37dac0ad25d9779",
        ),
        (
            "results/stage24_cross_dataset/stage24_0_protocol_lock/"
            "stage24_0b_exact_cicids2017_source_recovery.json",
            "8a84f1dccc51f224427e00144088058590f48c8b603cfd798f6ba431d659da1c",
        ),
        (
            "results/stage24_cross_dataset/stage24_2_primary_target_openings/"
            "stage24_2a_bridge62_published/stage24_2a_bridge62_published_result.json",
            "1d4b3010fdc8b0f7f62a0fed2c373824ce99cb2f16936c021756749a9ca54749",
        ),
    ]
)

STAGE24_PROTOCOL_LOCK_REL = (
    "results/stage24_cross_dataset/stage24_0_protocol_lock/"
    "stage24_0c_final_preopening_protocol_lock.json"
)
STAGE24_SOURCE_CONTRACT_REL = (
    "results/stage24_cross_dataset/stage24_0_protocol_lock/"
    "stage24_0b2_complete_cicids2017_source_contract.json"
)
STAGE24_SOURCE_RECOVERY_REL = (
    "results/stage24_cross_dataset/stage24_0_protocol_lock/"
    "stage24_0b_exact_cicids2017_source_recovery.json"
)
STAGE24_TARGET_RESULT_REL = (
    "results/stage24_cross_dataset/stage24_2_primary_target_openings/"
    "stage24_2a_bridge62_published/stage24_2a_bridge62_published_result.json"
)

STAGE26_CLOSURE_CANDIDATES = [
    (
        "results/stage26_deployment_profiling/stage26_12_final_synthesis/"
        "stage26_g2_final_stage26_closure_receipt.json"
    ),
    (
        "results/stage26_deployment_profiling/stage26_g2_final_gpu_publication_closure/"
        "stage26_g2_final_stage26_closure_receipt.json"
    ),
]

# Stage20 compact manifests are checked only to prove that the frozen raw-byte
# release corpus contains binary labels, not seven-family labels.
STAGE20_COMPACT_MANIFESTS = OrderedDict(
    [
        (
            "Monday",
            "results/stage20_1e_training/"
            "stage20_1e1m_monday_compact_corpus_manifest.json",
        ),
        (
            "Tuesday",
            "results/stage20_1e_training/"
            "stage20_1e1t_tuesday_compact_corpus_manifest.json",
        ),
        (
            "Wednesday",
            "results/stage20_1e_training/"
            "stage20_1e1w_wednesday_compact_corpus_manifest.json",
        ),
        (
            "Thursday",
            "results/stage20_1e_training/"
            "stage20_1e1v_thursday_validation_compact_corpus_manifest.json",
        ),
        (
            "Friday",
            "results/stage20_1e_training/"
            "stage20_1e4_friday_holdout_compact_corpus_manifest.json",
        ),
    ]
)

OUTPUT_REL = (
    "results/stage27_loao_unseen_attack/"
    "stage27_0a_family_day_feasibility"
)

REQUIRED_OUTPUTS = [
    "family_day_support.csv",
    "benign_day_support.csv",
    "taxonomy_receipt.json",
    "source_artifact_receipts.json",
    "temporal_feasibility.json",
    "feasibility_audit.md",
]

# Frozen Stage24 taxonomy.
RAW_TO_FAMILY = OrderedDict(
    [
        ("bot", "BOT"),
        ("ddos", "DDOS"),
        ("dos goldeneye", "DOS"),
        ("dos hulk", "DOS"),
        ("dos slowhttptest", "DOS"),
        ("dos slowloris", "DOS"),
        ("ftp-patator", "AUTH_BRUTE_FORCE"),
        ("heartbleed", "TARGET_ONLY_UNSEEN"),
        ("infiltration", "INFILTRATION"),
        ("portscan", "PORT_SCAN"),
        ("ssh-patator", "AUTH_BRUTE_FORCE"),
        ("web attack - brute force", "WEB_ATTACK"),
        ("web attack - sql injection", "WEB_ATTACK"),
        ("web attack - xss", "WEB_ATTACK"),
    ]
)

# Explicit counts recovered from the durable Stage24 target population.
EXPECTED_CANONICAL_LABEL_COUNTS = OrderedDict(
    [
        ("benign", 2_273_097),
        ("bot", 1_966),
        ("ddos", 128_027),
        ("dos goldeneye", 10_293),
        ("dos hulk", 231_073),
        ("dos slowhttptest", 5_499),
        ("dos slowloris", 5_796),
        ("ftp-patator", 7_938),
        ("heartbleed", 11),
        ("infiltration", 36),
        ("portscan", 158_930),
        ("ssh-patator", 5_897),
        ("web attack - brute force", 1_507),
        ("web attack - sql injection", 21),
        ("web attack - xss", 652),
    ]
)

MIN_INFERENTIAL_POSITIVE_SUPPORT = 50

COMMIT_MESSAGE = "stage27-0a: freeze family-day feasibility audit"


# =============================================================================
# 1. GENERIC HELPERS
# =============================================================================

def now_utc() -> str:
    return datetime.now(timezone.utc).isoformat()


def banner(title: str, width: int = 110) -> None:
    print()
    print("=" * width)
    print(title)
    print("=" * width)


def run(
    args: List[str],
    *,
    cwd: Path | None = None,
    env: Dict[str, str] | None = None,
    check: bool = True,
    capture: bool = True,
) -> str:
    proc = subprocess.run(
        args,
        cwd=str(cwd) if cwd else None,
        env=env,
        text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.PIPE if capture else None,
    )
    if check and proc.returncode != 0:
        cmd = " ".join(args)
        out = proc.stdout or ""
        err = proc.stderr or ""
        raise RuntimeError(
            f"Command failed ({proc.returncode}): {cmd}\n\nSTDOUT:\n{out}\n\nSTDERR:\n{err}"
        )
    return (proc.stdout or "").strip()


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def write_json(path: Path, obj: Any) -> None:
    payload = json.dumps(
        obj,
        indent=2,
        ensure_ascii=False,
        sort_keys=False,
    ) + "\n"
    path.write_text(payload, encoding="utf-8")


def normalize_label(value: Any) -> str:
    s = str(value).strip().lower()
    # CICIDS2017 Web Attack strings sometimes carry cp1252/en-dash artifacts.
    for ch in ["\u0096", "\u2013", "\u2014", "–", "—"]:
        s = s.replace(ch, "-")
    s = " ".join(s.split())
    return s


def walk_nodes(obj: Any, path: str = "$") -> Iterable[Tuple[str, Any]]:
    yield path, obj
    if isinstance(obj, dict):
        for k, v in obj.items():
            yield from walk_nodes(v, f"{path}.{k}")
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            yield from walk_nodes(v, f"{path}[{i}]")


def contains_pair_anywhere(obj: Any, raw_label: str, family: str) -> bool:
    raw_norm = normalize_label(raw_label)
    fam_norm = normalize_label(family)
    for _, node in walk_nodes(obj):
        if not isinstance(node, dict):
            continue
        for k, v in node.items():
            if normalize_label(k) == raw_norm and normalize_label(v) == fam_norm:
                return True
    return False


def find_repo() -> Path:
    candidates = [
        Path("/kaggle/working") / REPO_NAME,
        Path.cwd() / REPO_NAME,
        Path.cwd(),
    ]
    for p in candidates:
        if (p / ".git").exists() and p.name == REPO_NAME:
            return p.resolve()
    return (Path("/kaggle/working") / REPO_NAME).resolve()


def git_status_clean(repo: Path) -> bool:
    return run(["git", "status", "--porcelain"], cwd=repo) == ""


# =============================================================================
# 2. TEMPORARY GITHUB AUTH
# =============================================================================

def get_kaggle_github_token() -> str:
    try:
        from kaggle_secrets import UserSecretsClient  # type: ignore
    except Exception as exc:
        raise RuntimeError(
            "kaggle_secrets is unavailable. Run this script in Kaggle or provide "
            "authenticated Git credentials another way."
        ) from exc

    client = UserSecretsClient()
    labels = ["GITHUB_TOKEN", "github_token", "GH_TOKEN", "github_pat", "GITHUB_PAT"]

    for label in labels:
        try:
            token = client.get_secret(label)
        except Exception:
            token = None
        if token:
            token = str(token).strip()
            if token:
                print(f"[FOUND] GitHub secret label: {label} (value not printed)")
                return token

    raise RuntimeError(
        "No usable GitHub token found in Kaggle Secrets. Expected one of: "
        + ", ".join(labels)
    )


def make_git_auth_env(token: str) -> Tuple[Dict[str, str], Path]:
    askpass = Path("/kaggle/working/.stage27_git_askpass.sh")
    askpass.write_text(
        "#!/bin/sh\n"
        'case "$1" in\n'
        '  *Username*) printf "%s\\n" "x-access-token" ;;\n'
        '  *Password*) printf "%s\\n" "$STAGE27_GITHUB_TOKEN" ;;\n'
        '  *) printf "%s\\n" "" ;;\n'
        "esac\n",
        encoding="utf-8",
    )
    askpass.chmod(stat.S_IRUSR | stat.S_IWUSR | stat.S_IXUSR)

    env = os.environ.copy()
    env["GIT_ASKPASS"] = str(askpass)
    env["GIT_TERMINAL_PROMPT"] = "0"
    env["STAGE27_GITHUB_TOKEN"] = token
    return env, askpass


# =============================================================================
# 3. REPOSITORY BOOTSTRAP + HARD GATES
# =============================================================================

def ensure_repo(repo: Path, env: Dict[str, str]) -> None:
    banner("STAGE27-0A :: REPOSITORY BOOTSTRAP")

    if not (repo / ".git").exists():
        repo.parent.mkdir(parents=True, exist_ok=True)
        print("Repository not present; cloning origin/main...")
        run(
            ["git", "clone", "--branch", "main", "--single-branch", REPO_URL, str(repo)],
            cwd=repo.parent,
            env=env,
        )

    remote_url = run(["git", "remote", "get-url", "origin"], cwd=repo)
    print("Repository :", repo)
    print("Remote     :", remote_url)

    # Do not permit a dirty starting tree.
    if not git_status_clean(repo):
        raise RuntimeError(
            "Git working tree is not clean. Stage27-0A refuses to overwrite or "
            "co-mingle unrelated work."
        )

    run(["git", "fetch", "origin", "main"], cwd=repo, env=env)

    branch = run(["git", "branch", "--show-current"], cwd=repo)
    head = run(["git", "rev-parse", "HEAD"], cwd=repo)
    origin_main = run(["git", "rev-parse", "origin/main"], cwd=repo)
    remote_main = run(
        ["git", "ls-remote", "origin", "refs/heads/main"],
        cwd=repo,
        env=env,
    ).split()[0]

    banner("STAGE27-0A :: GIT GATE")
    print("Branch          :", branch)
    print("Expected parent :", EXPECTED_PARENT)
    print("Local HEAD      :", head)
    print("origin/main     :", origin_main)
    print("remote main     :", remote_main)
    print("Repo clean      :", git_status_clean(repo))

    if branch != "main":
        raise RuntimeError(f"Expected branch 'main', found {branch!r}")

    if not (head == origin_main == remote_main == EXPECTED_PARENT):
        raise RuntimeError(
            "Pinned execution parent mismatch.\n"
            f"expected    : {EXPECTED_PARENT}\n"
            f"local HEAD  : {head}\n"
            f"origin/main : {origin_main}\n"
            f"remote main : {remote_main}\n\n"
            "Do not rebase or silently continue. Inspect the new remote state first."
        )

    print("[PASS] Exact Stage27 execution parent is pinned and clean.")


# =============================================================================
# 4. PRIOR-STAGE CLOSURE + SOURCE HASH GATES
# =============================================================================

def verify_stage26_closure(repo: Path) -> Dict[str, Any]:
    banner("STAGE27-0A :: VERIFY STAGE26 CLOSURE")

    hits = []
    for rel in STAGE26_CLOSURE_CANDIDATES:
        path = repo / rel
        if not path.exists():
            continue
        data = load_json(path)
        if data.get("stage26_status") == "COMPLETE":
            hits.append(
                {
                    "path": rel,
                    "sha256": sha256_file(path),
                    "bytes": path.stat().st_size,
                    "stage26_status": "COMPLETE",
                }
            )
            print("COMPLETE:", rel)

    if not hits:
        raise RuntimeError("No durable Stage26 stage26_status=COMPLETE receipt found.")

    print("[PASS] Stage26 closure verified.")
    return {"verified": True, "evidence": hits}


def verify_stage24_sources(repo: Path) -> List[Dict[str, Any]]:
    banner("STAGE27-0A :: VERIFY DURABLE STAGE24 SOURCES")

    receipts: List[Dict[str, Any]] = []

    for rel, expected_sha in EXPECTED_STAGE24_SOURCE_SHA256.items():
        path = repo / rel
        if not path.exists():
            raise FileNotFoundError(f"Missing required Stage24 artifact: {rel}")

        actual_sha = sha256_file(path)
        size = path.stat().st_size

        print()
        print("Artifact:", rel)
        print(" bytes   :", size)
        print(" expected:", expected_sha)
        print(" actual  :", actual_sha)

        if actual_sha != expected_sha:
            raise RuntimeError(
                f"SHA256 mismatch for {rel}\n"
                f"expected: {expected_sha}\n"
                f"actual  : {actual_sha}"
            )

        receipts.append(
            {
                "path": rel,
                "bytes": size,
                "sha256": actual_sha,
                "sha256_matches_frozen_expected": True,
            }
        )

    print()
    print("[PASS] All four Stage24 durable artifacts match frozen SHA256 values.")
    return receipts


# =============================================================================
# 5. TAXONOMY + STAGE24 EFFECTIVE POPULATION RECOVERY
# =============================================================================

def verify_taxonomy(repo: Path) -> Dict[str, Any]:
    banner("STAGE27-0A :: VERIFY FROZEN STAGE24 TAXONOMY")

    protocol = load_json(repo / STAGE24_PROTOCOL_LOCK_REL)

    missing_pairs = []
    for raw, fam in RAW_TO_FAMILY.items():
        ok = contains_pair_anywhere(protocol, raw, fam)
        print(f"{raw:<34} -> {fam:<20} {'[OK]' if ok else '[MISSING]'}")
        if not ok:
            missing_pairs.append((raw, fam))

    if missing_pairs:
        raise RuntimeError(
            "Frozen Stage24 taxonomy pairs were not all found in the protocol lock: "
            + repr(missing_pairs)
        )

    recovered_primary = sorted(
        {fam for fam in RAW_TO_FAMILY.values() if fam in PRIMARY_FAMILIES}
    )
    if recovered_primary != sorted(PRIMARY_FAMILIES):
        raise RuntimeError(
            f"Primary taxonomy mismatch: {recovered_primary} vs {sorted(PRIMARY_FAMILIES)}"
        )

    print()
    print("[PASS] Exact seven-family Stage24 taxonomy recovered.")

    return {
        "primary_families": PRIMARY_FAMILIES,
        "raw_label_to_stage27_family": RAW_TO_FAMILY,
        "excluded_from_primary_seven_family_taxonomy": {
            "heartbleed": "TARGET_ONLY_UNSEEN"
        },
    }


def recover_stage24_population(repo: Path) -> Dict[str, Any]:
    banner("STAGE27-0A :: RECOVER STAGE24 EFFECTIVE CICIDS2017 POPULATION")

    result = load_json(repo / STAGE24_TARGET_RESULT_REL)

    try:
        file_level = result["metrics"]["file_level_descriptive"]
        label_counts_raw = result["target_population"]["canonical_label_counts"]
    except Exception as exc:
        raise RuntimeError(
            "Expected Stage24 population structures were not found at "
            "$.metrics.file_level_descriptive and "
            "$.target_population.canonical_label_counts."
        ) from exc

    label_counts = {
        normalize_label(k): int(v)
        for k, v in label_counts_raw.items()
    }

    # Exact canonical label-count gate.
    for label, expected in EXPECTED_CANONICAL_LABEL_COUNTS.items():
        actual = label_counts.get(label)
        if actual != expected:
            raise RuntimeError(
                f"Canonical Stage24 count mismatch for {label!r}: "
                f"expected {expected}, actual {actual}"
            )

    # Day/file support gate.
    expected_files = [
        ("Monday", 529_918, 529_918, 0),
        ("Tuesday", 445_909, 432_074, 13_835),
        ("Wednesday", 692_703, 440_031, 252_672),
        ("Thursday", 288_602, 288_566, 36),
        ("Thursday", 170_366, 168_186, 2_180),
        ("Friday", 225_745, 97_718, 128_027),
        ("Friday", 286_467, 127_537, 158_930),
        ("Friday", 191_033, 189_067, 1_966),
    ]

    if len(file_level) != len(expected_files):
        raise RuntimeError(
            f"Expected {len(expected_files)} Stage24 file-level rows; found {len(file_level)}"
        )

    normalized_rows = []
    for idx, row in enumerate(file_level):
        day = str(row["day"])
        rows = int(row["rows"])
        benign = int(row["benign"])
        attack = int(row["attack"])
        remote = str(row["remote"])

        exp_day, exp_rows, exp_benign, exp_attack = expected_files[idx]
        if (day, rows, benign, attack) != (
            exp_day,
            exp_rows,
            exp_benign,
            exp_attack,
        ):
            raise RuntimeError(
                "Stage24 file-level support mismatch at index "
                f"{idx}: got {(day, rows, benign, attack)}, expected "
                f"{(exp_day, exp_rows, exp_benign, exp_attack)}"
            )

        normalized_rows.append(
            {
                "file_id": int(row.get("file_id", idx)),
                "day": day,
                "remote": remote,
                "rows": rows,
                "benign": benign,
                "attack": attack,
                "start": int(row["start"]),
                "stop": int(row["stop"]),
            }
        )

    benign_by_day = OrderedDict((day, 0) for day in WEEKDAYS)
    rows_by_day = OrderedDict((day, 0) for day in WEEKDAYS)
    all_attack_by_day = OrderedDict((day, 0) for day in WEEKDAYS)

    for row in normalized_rows:
        day = row["day"]
        benign_by_day[day] += row["benign"]
        rows_by_day[day] += row["rows"]
        all_attack_by_day[day] += row["attack"]

    expected_benign = OrderedDict(
        [
            ("Monday", 529_918),
            ("Tuesday", 432_074),
            ("Wednesday", 440_031),
            ("Thursday", 456_752),
            ("Friday", 414_322),
        ]
    )
    if benign_by_day != expected_benign:
        raise RuntimeError(
            f"Benign weekday support mismatch:\nactual={benign_by_day}\n"
            f"expected={expected_benign}"
        )

    if sum(benign_by_day.values()) != 2_273_097:
        raise RuntimeError("Total benign support mismatch.")

    if sum(all_attack_by_day.values()) != 557_646:
        raise RuntimeError("Total non-benign support mismatch.")

    print()
    for row in normalized_rows:
        print(
            f"{row['day']:<10} rows={row['rows']:>7,} "
            f"benign={row['benign']:>7,} attack={row['attack']:>7,}  "
            f"{row['remote']}"
        )

    print()
    print("[PASS] Stage24 effective CICIDS2017 population is exact.")

    return {
        "file_level": normalized_rows,
        "label_counts": label_counts,
        "benign_by_day": benign_by_day,
        "rows_by_day": rows_by_day,
        "all_attack_by_day": all_attack_by_day,
    }


# =============================================================================
# 6. STAGE20 COMPACT CORPUS LIMITATION AUDIT
# =============================================================================

def audit_stage20_compact_binary_only(repo: Path) -> Dict[str, Any]:
    banner("STAGE27-0A :: AUDIT STAGE20 COMPACT RELEASE SEMANTICS")

    manifest_receipts = []
    all_binary_only = True

    for day, rel in STAGE20_COMPACT_MANIFESTS.items():
        path = repo / rel
        if not path.exists():
            raise FileNotFoundError(f"Missing Stage20 compact manifest: {rel}")

        data = load_json(path)
        compact = data.get("compact_corpus", {})
        files = compact.get("files", {})

        keys = sorted(files.keys())
        expected_keys = sorted(
            [
                "encoded_bytes.bin",
                "flow_offsets.npy",
                "labels.npy",
                "packet_lengths.npy",
            ]
        )

        labels_file_present = "labels.npy" in files
        family_label_file_present = any(
            (
                "family" in k.lower()
                or "attack_type" in k.lower()
                or "multiclass" in k.lower()
            )
            for k in files.keys()
        )

        exact_join = data.get("exact_join", {})
        binary_counts = exact_join.get("matched_binary_counts", {})
        binary_key_set = {str(k) for k in binary_counts.keys()}
        binary_only_counts = binary_key_set.issubset({"0", "1"}) and bool(binary_counts)

        schema_ok = (
            keys == expected_keys
            and labels_file_present
            and not family_label_file_present
            and binary_only_counts
        )

        all_binary_only = all_binary_only and schema_ok

        print()
        print(day)
        print("  manifest:", rel)
        print("  files   :", keys)
        print("  matched_binary_counts:", binary_counts)
        print("  family-label array present:", family_label_file_present)
        print("  binary-only schema gate:", schema_ok)

        manifest_receipts.append(
            {
                "day": day,
                "path": rel,
                "bytes": path.stat().st_size,
                "sha256": sha256_file(path),
                "compact_files": keys,
                "flow_count": int(compact.get("flow_count", 0)),
                "matched_binary_counts": {
                    str(k): int(v) for k, v in binary_counts.items()
                },
                "family_label_array_present": family_label_file_present,
                "binary_only_for_stage27_family_identity": schema_ok,
            }
        )

    if not all_binary_only:
        raise RuntimeError(
            "Stage20 compact-corpus binary-only audit did not pass consistently."
        )

    print()
    print(
        "[PASS] All five Stage20 compact manifests persist binary labels only; "
        "they cannot independently certify seven-family LOAO membership."
    )

    return {
        "all_five_compact_manifests_binary_only": True,
        "stage27_family_support_source": "STAGE24_FULL_CICIDS2017_EFFECTIVE_POPULATION",
        "stage20_compact_release_role": (
            "VALID_FOR_STAGE20_BINARY_RAW_BYTE_EXPERIMENTS_BUT_NOT_A_DURABLE_"
            "SEVEN_FAMILY_MEMBERSHIP_SOURCE"
        ),
        "manifests": manifest_receipts,
    }


# =============================================================================
# 7. SUPPORT MATRIX CONSTRUCTION
# =============================================================================

def construct_support_matrix(pop: Dict[str, Any]) -> Tuple[OrderedDict, OrderedDict]:
    banner("STAGE27-0A :: CONSTRUCT SEVEN-FAMILY × WEEKDAY SUPPORT MATRIX")

    c = pop["label_counts"]

    dos_total = (
        c["dos goldeneye"]
        + c["dos hulk"]
        + c["dos slowhttptest"]
        + c["dos slowloris"]
    )
    auth_total = c["ftp-patator"] + c["ssh-patator"]
    web_total = (
        c["web attack - brute force"]
        + c["web attack - sql injection"]
        + c["web attack - xss"]
    )

    family_totals = OrderedDict(
        [
            ("BOT", c["bot"]),
            ("DDOS", c["ddos"]),
            ("DOS", dos_total),
            ("AUTH_BRUTE_FORCE", auth_total),
            ("INFILTRATION", c["infiltration"]),
            ("PORT_SCAN", c["portscan"]),
            ("WEB_ATTACK", web_total),
        ]
    )

    expected_totals = OrderedDict(
        [
            ("BOT", 1_966),
            ("DDOS", 128_027),
            ("DOS", 252_661),
            ("AUTH_BRUTE_FORCE", 13_835),
            ("INFILTRATION", 36),
            ("PORT_SCAN", 158_930),
            ("WEB_ATTACK", 2_180),
        ]
    )
    if family_totals != expected_totals:
        raise RuntimeError(
            f"Family arithmetic mismatch:\nactual={family_totals}\nexpected={expected_totals}"
        )

    matrix = OrderedDict()
    for family in PRIMARY_FAMILIES:
        matrix[family] = OrderedDict((day, 0) for day in WEEKDAYS)

    # CICIDS2017 attack schedule recovered from the exact Stage24 file-level
    # target population:
    matrix["AUTH_BRUTE_FORCE"]["Tuesday"] = auth_total
    matrix["DOS"]["Wednesday"] = dos_total
    matrix["INFILTRATION"]["Thursday"] = c["infiltration"]
    matrix["WEB_ATTACK"]["Thursday"] = web_total
    matrix["DDOS"]["Friday"] = c["ddos"]
    matrix["PORT_SCAN"]["Friday"] = c["portscan"]
    matrix["BOT"]["Friday"] = c["bot"]

    # Cross-check each day against durable Stage24 file-level attack support.
    # Wednesday carries an additional 11 Heartbleed rows excluded from the
    # seven primary families.
    expected_day_attack = OrderedDict(
        [
            ("Monday", 0),
            ("Tuesday", 13_835),
            ("Wednesday", 252_672),
            ("Thursday", 2_216),
            ("Friday", 288_923),
        ]
    )

    primary_day_attack = OrderedDict(
        (
            day,
            sum(matrix[fam][day] for fam in PRIMARY_FAMILIES),
        )
        for day in WEEKDAYS
    )

    heartbleed_by_day = OrderedDict((day, 0) for day in WEEKDAYS)
    heartbleed_by_day["Wednesday"] = c["heartbleed"]

    reconstructed_all_attack = OrderedDict(
        (
            day,
            primary_day_attack[day] + heartbleed_by_day[day],
        )
        for day in WEEKDAYS
    )

    if reconstructed_all_attack != expected_day_attack:
        raise RuntimeError(
            "Reconstructed day attack totals do not match Stage24 file-level evidence.\n"
            f"reconstructed={reconstructed_all_attack}\nexpected={expected_day_attack}"
        )

    if expected_day_attack != pop["all_attack_by_day"]:
        raise RuntimeError(
            "Stage24 file-level attack support differs from expected reconstructed support."
        )

    seven_family_total = sum(family_totals.values())
    all_nonbenign = seven_family_total + c["heartbleed"]

    if seven_family_total != 557_635:
        raise RuntimeError("Seven-family total must equal 557,635.")
    if all_nonbenign != 557_646:
        raise RuntimeError("All non-benign total must equal 557,646.")

    for fam in PRIMARY_FAMILIES:
        vals = matrix[fam]
        print(
            f"{fam:<20} "
            + " ".join(f"{day[:3]}={vals[day]:>7,}" for day in WEEKDAYS)
            + f"  total={sum(vals.values()):>7,}"
        )

    print()
    print("Seven-family attack total :", f"{seven_family_total:,}")
    print("Heartbleed excluded       :", f"{c['heartbleed']:,}")
    print("All non-benign total      :", f"{all_nonbenign:,}")
    print("[PASS] Family/day support arithmetic is exact.")

    summary = OrderedDict(
        [
            ("matrix", matrix),
            ("family_totals", family_totals),
            ("primary_day_attack", primary_day_attack),
            ("heartbleed_by_day", heartbleed_by_day),
            ("all_attack_by_day", reconstructed_all_attack),
            ("seven_family_attack_total", seven_family_total),
            ("heartbleed_target_only_unseen", c["heartbleed"]),
            ("all_nonbenign_total", all_nonbenign),
        ]
    )

    return matrix, summary


# =============================================================================
# 8. TEMPORAL FEASIBILITY
# =============================================================================

def first_positive_day(matrix: OrderedDict, family: str) -> str:
    positive = [day for day in WEEKDAYS if matrix[family][day] > 0]
    if len(positive) != 1:
        raise RuntimeError(
            f"Expected exactly one positive weekday for {family}, found {positive}"
        )
    return positive[0]


def sum_family_support(
    matrix: OrderedDict,
    days: List[str],
    *,
    exclude_family: str | None = None,
) -> OrderedDict:
    out = OrderedDict()
    for fam in PRIMARY_FAMILIES:
        if exclude_family is not None and fam == exclude_family:
            continue
        out[fam] = sum(matrix[fam][d] for d in days)
    return out


def derive_temporal_feasibility(
    matrix: OrderedDict,
    benign_by_day: OrderedDict,
) -> Dict[str, Any]:
    banner("STAGE27-0A :: TEMPORAL FEASIBILITY")

    # Original proposal:
    original_train = ["Monday", "Tuesday", "Wednesday"]
    original_validation = ["Thursday"]
    original_target = ["Friday"]

    family_records = OrderedDict()

    for family in PRIMARY_FAMILIES:
        target_day = first_positive_day(matrix, family)
        target_idx = WEEKDAYS.index(target_day)

        heldout_support = matrix[family][target_day]
        support_status = (
            "INFERENTIAL_ELIGIBLE_SUPPORT"
            if heldout_support >= MIN_INFERENTIAL_POSITIVE_SUPPORT
            else "DESCRIPTIVE_ONLY_SUPPORT_LT_50"
        )

        original_target_support = sum(matrix[family][d] for d in original_target)
        original_train_support = sum(matrix[family][d] for d in original_train)
        original_val_support = sum(matrix[family][d] for d in original_validation)

        original_family_geometry_valid = (
            original_target_support > 0
            and original_train_support == 0
            and original_val_support == 0
        )

        record: Dict[str, Any] = {
            "held_out_family": family,
            "first_and_only_positive_day": target_day,
            "target_positive_support": heldout_support,
            "support_status": support_status,
            "original_mon_wed_train_thu_validation_fri_target": {
                "heldout_train_support": original_train_support,
                "heldout_validation_support": original_val_support,
                "heldout_target_support": original_target_support,
                "family_target_exists_under_original_geometry": original_family_geometry_valid,
            },
        }

        # Day-atomic strict chronology:
        #   TRAIN = all weekdays before VALIDATION
        #   VALIDATION = the weekday immediately preceding TARGET
        #   TARGET = family-positive weekday
        #
        # Additional requirements:
        #   - held-out family absent from TRAIN and VALIDATION
        #   - TRAIN has benign AND at least one known-family positive
        #   - VALIDATION has benign AND at least one known-family positive
        # These conditions are needed for a supervised binary fit and known-family
        # threshold selection without target-family labels.
        if target_idx < 2:
            record.update(
                {
                    "status": "STRUCTURALLY_INELIGIBLE",
                    "reason_code": "INSUFFICIENT_PRETARGET_WEEKDAY_DEPTH",
                    "reason": (
                        f"{family} first appears on {target_day}. Fewer than two "
                        "earlier weekday partitions exist, so separate day-atomic "
                        "TRAIN and VALIDATION partitions cannot both precede TARGET."
                    ),
                    "replacement_geometry": None,
                }
            )
        else:
            validation_day = WEEKDAYS[target_idx - 1]
            train_days = WEEKDAYS[: target_idx - 1]
            validation_days = [validation_day]
            target_days = [target_day]

            train_family = sum_family_support(
                matrix, train_days, exclude_family=family
            )
            val_family = sum_family_support(
                matrix, validation_days, exclude_family=family
            )

            heldout_train = sum(matrix[family][d] for d in train_days)
            heldout_val = sum(matrix[family][d] for d in validation_days)

            train_attack = sum(train_family.values())
            val_attack = sum(val_family.values())

            train_benign = sum(benign_by_day[d] for d in train_days)
            val_benign = sum(benign_by_day[d] for d in validation_days)
            target_benign = benign_by_day[target_day]

            if heldout_train != 0 or heldout_val != 0:
                raise RuntimeError(
                    f"Held-out leakage discovered during feasibility construction for {family}"
                )

            if train_attack <= 0:
                status = "STRUCTURALLY_INELIGIBLE"
                reason_code = "NO_KNOWN_ATTACK_POSITIVES_IN_DAY_ATOMIC_TRAIN"
                reason = (
                    f"The only day-atomic chronology before {target_day} gives "
                    f"TRAIN={train_days}, but TRAIN contains {train_attack} known-family "
                    "attack positives. A supervised attack-vs-benign learner cannot be "
                    "fit honestly under this geometry."
                )
                geometry = None
            elif val_attack <= 0:
                status = "STRUCTURALLY_INELIGIBLE"
                reason_code = "NO_KNOWN_ATTACK_POSITIVES_IN_DAY_ATOMIC_VALIDATION"
                reason = (
                    f"VALIDATION={validation_days} contains no known-family attack "
                    "positives for threshold selection."
                )
                geometry = None
            else:
                status = (
                    "ELIGIBLE_DESCRIPTIVE_ONLY"
                    if heldout_support < MIN_INFERENTIAL_POSITIVE_SUPPORT
                    else "ELIGIBLE"
                )
                reason_code = "DAY_ATOMIC_STRICT_CHRONOLOGY_AVAILABLE"
                reason = (
                    "A strict day-atomic TRAIN < VALIDATION < TARGET geometry exists "
                    "with zero held-out-family support in both TRAIN and VALIDATION."
                )
                geometry = {
                    "train_days": train_days,
                    "validation_days": validation_days,
                    "target_days": target_days,
                    "heldout_family_train_count": heldout_train,
                    "heldout_family_validation_count": heldout_val,
                    "train_benign": train_benign,
                    "train_known_attack": train_attack,
                    "train_known_family_support": train_family,
                    "validation_benign": val_benign,
                    "validation_known_attack": val_attack,
                    "validation_known_family_support": val_family,
                    "target_benign_available_same_day": target_benign,
                    "target_heldout_attack": heldout_support,
                }

            record.update(
                {
                    "status": status,
                    "reason_code": reason_code,
                    "reason": reason,
                    "replacement_geometry": geometry,
                }
            )

        family_records[family] = record

    eligible = [
        fam
        for fam, rec in family_records.items()
        if rec["status"] in {"ELIGIBLE", "ELIGIBLE_DESCRIPTIVE_ONLY"}
    ]
    ineligible = [
        fam
        for fam, rec in family_records.items()
        if rec["status"] == "STRUCTURALLY_INELIGIBLE"
    ]
    descriptive_only = [
        fam
        for fam, rec in family_records.items()
        if rec["status"] == "ELIGIBLE_DESCRIPTIVE_ONLY"
    ]

    original_supported = [
        fam
        for fam, rec in family_records.items()
        if rec["original_mon_wed_train_thu_validation_fri_target"][
            "family_target_exists_under_original_geometry"
        ]
    ]

    # Expected day-atomic outcome from the frozen support schedule.
    expected_eligible = ["BOT", "DDOS", "INFILTRATION", "PORT_SCAN", "WEB_ATTACK"]
    expected_ineligible = ["DOS", "AUTH_BRUTE_FORCE"]
    if sorted(eligible) != sorted(expected_eligible):
        raise RuntimeError(
            f"Unexpected eligible fold set: {eligible}; expected {expected_eligible}"
        )
    if sorted(ineligible) != sorted(expected_ineligible):
        raise RuntimeError(
            f"Unexpected ineligible fold set: {ineligible}; expected {expected_ineligible}"
        )
    if descriptive_only != ["INFILTRATION"]:
        raise RuntimeError(
            f"Expected INFILTRATION to be the only descriptive-only eligible fold; "
            f"got {descriptive_only}"
        )

    print("Original universal Mon-Wed / Thu / Fri geometry: INFEASIBLE FOR 7/7")
    print("Families with Friday targets under original geometry:", original_supported)
    print()
    print("Corrected day-atomic eligible folds :", eligible)
    print("Structurally ineligible folds       :", ineligible)
    print("Descriptive-only eligible folds     :", descriptive_only)
    print()

    for fam, rec in family_records.items():
        print(
            f"{fam:<20} {rec['status']:<28} "
            f"target={rec['first_and_only_positive_day']:<9} "
            f"n={rec['target_positive_support']:,}"
        )

    print()
    print("[PASS] Temporal feasibility derived before any Stage27 model result exists.")

    return {
        "stage": "Stage27-0A",
        "created_at_utc": now_utc(),
        "scientific_boundary": {
            "model_fits": 0,
            "model_inference": 0,
            "threshold_selection": 0,
            "bootstrap_replicates": 0,
            "gpu_execution": False,
            "primary_execution_device": PRIMARY_EXECUTION_DEVICE,
            "primary_gpu_budget_hours": PRIMARY_GPU_BUDGET_HOURS,
        },
        "chronology_semantics": {
            "unit": "WEEKDAY_ATOMIC_PARTITIONS",
            "required_order": "TRAIN < VALIDATION < TARGET",
            "train_requires_benign": True,
            "train_requires_known_attack_positive": True,
            "validation_requires_benign": True,
            "validation_requires_known_attack_positive": True,
            "heldout_family_required_absent_from_train": True,
            "heldout_family_required_absent_from_validation": True,
        },
        "original_proposal": {
            "train_days": original_train,
            "validation_days": original_validation,
            "target_days": original_target,
            "universal_seven_fold_feasible": False,
            "families_with_positive_target_support": original_supported,
            "reason": (
                "Only BOT, DDOS and PORT_SCAN have Friday positives. DOS appears only "
                "Wednesday; AUTH_BRUTE_FORCE only Tuesday; INFILTRATION and WEB_ATTACK "
                "only Thursday. The universal Friday-target geometry therefore cannot "
                "produce all seven held-out-family targets."
            ),
        },
        "replacement_day_atomic_family_specific_geometry": {
            "status": "FEASIBILITY_CORRECTION_PRE_RESULT",
            "eligible_folds": eligible,
            "eligible_fold_count": len(eligible),
            "structurally_ineligible_folds": ineligible,
            "structurally_ineligible_fold_count": len(ineligible),
            "descriptive_only_eligible_folds": descriptive_only,
            "minimum_positive_support_for_inferential_claim": MIN_INFERENTIAL_POSITIVE_SUPPORT,
            "family_records": family_records,
        },
        "important_semantic_limit": (
            "Strict chronology means some non-held-out families occur only on or after "
            "a target day and therefore cannot all be represented in that fold's "
            "training set. Stage27-0 must describe the executable design as a "
            "chronology-first zero-training-exposure family audit and must not imply "
            "that every non-held-out family was necessarily seen in training."
        ),
        "stage27_0_authorization": (
            "AUTHORIZED_ONLY_AFTER_THIS_STAGE27_0A_COMMIT_IS_REMOTELY_VERIFIED"
        ),
    }


# =============================================================================
# 9. WRITE EXACTLY SIX REQUIRED ARTIFACTS
# =============================================================================

def write_family_day_csv(
    outdir: Path,
    matrix: OrderedDict,
) -> None:
    path = outdir / "family_day_support.csv"
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["family", *WEEKDAYS, "Total"])
        for fam in PRIMARY_FAMILIES:
            row = [fam] + [matrix[fam][d] for d in WEEKDAYS]
            row.append(sum(matrix[fam].values()))
            writer.writerow(row)


def write_benign_day_csv(
    outdir: Path,
    benign_by_day: OrderedDict,
) -> None:
    path = outdir / "benign_day_support.csv"
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["day", "benign_support"])
        for day in WEEKDAYS:
            writer.writerow([day, benign_by_day[day]])
        writer.writerow(["TOTAL", sum(benign_by_day.values())])


def markdown_table_family(matrix: OrderedDict) -> str:
    lines = [
        "| Family | Mon | Tue | Wed | Thu | Fri | Total |",
        "|---|---:|---:|---:|---:|---:|---:|",
    ]
    for fam in PRIMARY_FAMILIES:
        vals = [matrix[fam][d] for d in WEEKDAYS]
        lines.append(
            f"| {fam} | "
            + " | ".join(f"{v:,}" for v in vals)
            + f" | {sum(vals):,} |"
        )
    return "\n".join(lines)


def markdown_table_benign(benign_by_day: OrderedDict) -> str:
    lines = [
        "| Day | Benign support |",
        "|---|---:|",
    ]
    for day in WEEKDAYS:
        lines.append(f"| {day} | {benign_by_day[day]:,} |")
    lines.append(f"| **TOTAL** | **{sum(benign_by_day.values()):,}** |")
    return "\n".join(lines)


def markdown_table_feasibility(feas: Dict[str, Any]) -> str:
    records = feas["replacement_day_atomic_family_specific_geometry"]["family_records"]
    lines = [
        "| Family | Target day | Positive support | Stage27-0A status | Corrected geometry |",
        "|---|---|---:|---|---|",
    ]
    for fam in PRIMARY_FAMILIES:
        r = records[fam]
        geom = r["replacement_geometry"]
        if geom is None:
            geom_text = "—"
        else:
            geom_text = (
                f"{','.join(geom['train_days'])} → "
                f"{','.join(geom['validation_days'])} → "
                f"{','.join(geom['target_days'])}"
            )
        lines.append(
            f"| {fam} | {r['first_and_only_positive_day']} | "
            f"{r['target_positive_support']:,} | {r['status']} | {geom_text} |"
        )
    return "\n".join(lines)


def write_artifacts(
    repo: Path,
    stage26_receipt: Dict[str, Any],
    stage24_receipts: List[Dict[str, Any]],
    taxonomy: Dict[str, Any],
    pop: Dict[str, Any],
    compact_audit: Dict[str, Any],
    matrix: OrderedDict,
    support_summary: OrderedDict,
    feasibility: Dict[str, Any],
) -> List[Path]:
    banner("STAGE27-0A :: WRITE SIX REQUIRED ARTIFACTS")

    outdir = repo / OUTPUT_REL

    if outdir.exists():
        existing = [p for p in outdir.iterdir() if p.is_file() or p.is_dir()]
        if existing:
            raise RuntimeError(
                "Stage27-0A output directory is not empty. Refusing to overwrite:\n"
                + "\n".join(str(p) for p in existing)
            )
    outdir.mkdir(parents=True, exist_ok=True)

    write_family_day_csv(outdir, matrix)
    write_benign_day_csv(outdir, pop["benign_by_day"])

    taxonomy_receipt = {
        "stage": "Stage27-0A",
        "type": "TAXONOMY_RECEIPT",
        "created_at_utc": now_utc(),
        "execution_parent": EXPECTED_PARENT,
        "taxonomy_source": STAGE24_PROTOCOL_LOCK_REL,
        "taxonomy_source_sha256": EXPECTED_STAGE24_SOURCE_SHA256[
            STAGE24_PROTOCOL_LOCK_REL
        ],
        "primary_family_count": 7,
        "primary_families": PRIMARY_FAMILIES,
        "raw_label_to_family": RAW_TO_FAMILY,
        "target_only_unseen_not_in_primary_seven_family_taxonomy": {
            "heartbleed": {
                "mapping": "TARGET_ONLY_UNSEEN",
                "support": int(pop["label_counts"]["heartbleed"]),
                "weekday": "Wednesday",
            }
        },
        "taxonomy_mutated": False,
        "family_merge_after_stage24": False,
        "family_split_after_stage24": False,
        "status": "EXACT_STAGE24_TAXONOMY_INHERITED",
    }
    write_json(outdir / "taxonomy_receipt.json", taxonomy_receipt)

    source_receipt = {
        "stage": "Stage27-0A",
        "type": "SOURCE_ARTIFACT_RECEIPTS",
        "created_at_utc": now_utc(),
        "execution_parent": EXPECTED_PARENT,
        "stage26_closure": stage26_receipt,
        "stage24_durable_sources": stage24_receipts,
        "effective_stage27_support_population": {
            "dataset": "CICIDS2017",
            "population_identity": "STAGE24_FULL_EFFECTIVE_TARGET_POPULATION",
            "rows": int(sum(pop["rows_by_day"].values())),
            "benign": int(sum(pop["benign_by_day"].values())),
            "nonbenign_including_heartbleed": int(
                sum(pop["all_attack_by_day"].values())
            ),
            "primary_seven_family_attack": int(
                support_summary["seven_family_attack_total"]
            ),
            "heartbleed_target_only_unseen": int(
                support_summary["heartbleed_target_only_unseen"]
            ),
            "support_source": STAGE24_TARGET_RESULT_REL,
        },
        "stage20_compact_release_population_decision": compact_audit,
        "population_discrepancy_resolution": {
            "discrepancy": (
                "Stage20 compact corpora are much smaller exact-matched raw-byte "
                "corpora, whereas Stage24 exposes the full 2,830,743-row CICIDS2017 "
                "effective target population."
            ),
            "decision": (
                "Use the Stage24 full effective CICIDS2017 population for Stage27 "
                "seven-family feasibility because it durably preserves the canonical "
                "attack-family label counts. Do not substitute Stage20 compact binary "
                "labels for family identities."
            ),
            "reason": (
                "All five committed Stage20 compact manifests persist labels.npy as "
                "binary labels and do not persist a family-label array or durable "
                "seven-family membership mapping."
            ),
            "scientific_adaptation": False,
            "decision_time": "BEFORE_ANY_STAGE27_MODEL_FIT_OR_INFERENCE",
        },
        "model_fits": 0,
        "model_inference": 0,
        "corpus_archives_downloaded_by_stage27_0a": 0,
        "release_corpus_recreated": False,
    }
    write_json(outdir / "source_artifact_receipts.json", source_receipt)

    write_json(outdir / "temporal_feasibility.json", feasibility)

    audit_md = f"""# Stage27-0A — Family × Day Feasibility Audit

**Status:** COMPLETE — ZERO FIT / ZERO INFERENCE FEASIBILITY AUDIT  
**Execution parent:** `{EXPECTED_PARENT}`  
**Primary compute policy:** CPU, GPU budget = 0 hours  
**Created:** {now_utc()}

## Scientific purpose

Stage27-0A determines which chronology-first unseen-family folds are structurally
possible **before** any Stage27 model results exist.

The originally proposed universal geometry was:

```text
Monday–Wednesday -> TRAIN
Thursday         -> VALIDATION
Friday           -> HELD-OUT FAMILY TARGET
```

That geometry is **not feasible for all seven families** because CICIDS2017 attack
families occur on different weekdays.

## Population decision

Stage27-0A uses the **full Stage24 CICIDS2017 effective target population** as the
family-support source.

This is deliberate. The committed Stage20 compact raw-byte release corpora preserve
exact raw-byte flows but persist only:

```text
encoded_bytes.bin
flow_offsets.npy
labels.npy
packet_lengths.npy
```

and the manifest join counts are binary `0/1`. No durable attack-family label array is
present. Therefore the Stage20 compact corpus remains valid for the Stage20 binary
raw-byte experiment, but it is **not** a durable seven-family membership source for
Stage27 LOAO construction.

No release archive was downloaded or recreated during Stage27-0A.

## Frozen seven-family taxonomy

```text
BOT
DDOS
DOS
AUTH_BRUTE_FORCE
INFILTRATION
PORT_SCAN
WEB_ATTACK
```

`heartbleed` remains `TARGET_ONLY_UNSEEN` and is excluded from the seven-family
primary matrix.

## Exact family × weekday support

{markdown_table_family(matrix)}

Seven-family total: **{support_summary['seven_family_attack_total']:,}**

Heartbleed excluded from the primary seven-family matrix: **{support_summary['heartbleed_target_only_unseen']:,}**

All non-benign rows including Heartbleed: **{support_summary['all_nonbenign_total']:,}**

## Benign support by weekday

{markdown_table_benign(pop['benign_by_day'])}

## Original geometry decision

The universal Mon–Wed / Thu / Fri seven-fold geometry is **STRUCTURALLY INFEASIBLE**.

Only the three Friday families — BOT, DDOS and PORT_SCAN — have positive target support
on Friday. DOS appears only Wednesday, AUTH_BRUTE_FORCE only Tuesday, and
INFILTRATION/WEB_ATTACK only Thursday.

The original geometry is therefore rejected **before model fitting**. This is a
feasibility correction, not result-driven adaptation.

## Day-atomic chronology-first feasibility

Stage27-0A applies the following feasibility rule:

```text
TRAIN < VALIDATION < TARGET
```

using weekday-atomic partitions. An executable fold additionally requires:

- zero held-out-family positives in TRAIN;
- zero held-out-family positives in VALIDATION;
- benign and at least one known-family attack positive in TRAIN;
- benign and at least one known-family attack positive in VALIDATION;
- same-target-day benign support for the primary isolation target.

{markdown_table_feasibility(feasibility)}

### Structurally ineligible

- **AUTH_BRUTE_FORCE** — first appears Tuesday. There are not two earlier weekday
  partitions for separate day-atomic TRAIN and VALIDATION, and Monday has no attack
  positives.
- **DOS** — first appears Wednesday. The only day-atomic split is Monday TRAIN /
  Tuesday VALIDATION / Wednesday TARGET, but Monday contains no known attack positives,
  so a supervised binary IDS cannot be trained honestly under that geometry.

These families are retained in the taxonomy and marked
`STRUCTURALLY_INELIGIBLE`; they are not silently dropped.

### Eligible but descriptive-only

- **INFILTRATION** has only **36** positives, below the preregistered recommended
  inferential support threshold of 50. It remains eligible for execution but must be
  marked `DESCRIPTIVE_ONLY`.

## Important semantic limit for Stage27-0

Because chronology is primary, some non-held-out attack families may occur only on or
after a fold's target day and therefore may also be absent from training.

Stage27-0 should therefore describe the executable design precisely as a:

> **chronology-first zero-training-exposure family audit**

and should not imply that every non-held-out family is necessarily represented in
training for every fold.

## Stage27-0A result

```text
Eligible folds:             5
Structurally ineligible:    2
Descriptive-only eligible:  1
Stage27 model fits:         0
Stage27 model inference:    0
GPU execution:              NO
```

Eligible families:

```text
BOT
DDOS
INFILTRATION   [DESCRIPTIVE_ONLY]
PORT_SCAN
WEB_ATTACK
```

Structurally ineligible families:

```text
AUTH_BRUTE_FORCE
DOS
```

## Next gate

Stage27-0 protocol construction is authorized **only after this Stage27-0A artifact
set is committed and the remote `main` SHA is verified**.

This script performs that commit/push/remote verification and then stops. It does not
construct Stage27-0.
"""
    (outdir / "feasibility_audit.md").write_text(audit_md, encoding="utf-8")

    produced = sorted(p.name for p in outdir.iterdir() if p.is_file())
    expected = sorted(REQUIRED_OUTPUTS)

    if produced != expected:
        raise RuntimeError(
            "Stage27-0A produced an unexpected file set.\n"
            f"expected={expected}\nactual={produced}"
        )

    output_paths = [outdir / name for name in REQUIRED_OUTPUTS]

    print("Produced exactly six required files:")
    for path in output_paths:
        print(
            f"  {path.relative_to(repo)}  "
            f"bytes={path.stat().st_size:,}  sha256={sha256_file(path)}"
        )

    print()
    print("[PASS] Exactly six Stage27-0A artifacts written.")
    return output_paths


# =============================================================================
# 10. FINAL LOCAL VALIDATION
# =============================================================================

def validate_outputs(
    repo: Path,
    output_paths: List[Path],
    matrix: OrderedDict,
    pop: Dict[str, Any],
    feasibility: Dict[str, Any],
) -> None:
    banner("STAGE27-0A :: FINAL LOCAL VALIDATION")

    # CSV readback.
    family_csv = repo / OUTPUT_REL / "family_day_support.csv"
    with family_csv.open("r", encoding="utf-8", newline="") as f:
        rows = list(csv.DictReader(f))

    if len(rows) != 7:
        raise RuntimeError(f"family_day_support.csv must contain 7 data rows, got {len(rows)}")

    readback_total = sum(int(r["Total"]) for r in rows)
    if readback_total != 557_635:
        raise RuntimeError(
            f"family_day_support.csv seven-family total mismatch: {readback_total}"
        )

    benign_csv = repo / OUTPUT_REL / "benign_day_support.csv"
    with benign_csv.open("r", encoding="utf-8", newline="") as f:
        benign_rows = list(csv.DictReader(f))

    total_rows = [r for r in benign_rows if r["day"] == "TOTAL"]
    if len(total_rows) != 1 or int(total_rows[0]["benign_support"]) != 2_273_097:
        raise RuntimeError("benign_day_support.csv total mismatch.")

    # JSON readback.
    tf = load_json(repo / OUTPUT_REL / "temporal_feasibility.json")
    eligible_count = tf[
        "replacement_day_atomic_family_specific_geometry"
    ]["eligible_fold_count"]
    ineligible_count = tf[
        "replacement_day_atomic_family_specific_geometry"
    ]["structurally_ineligible_fold_count"]

    if eligible_count != 5 or ineligible_count != 2:
        raise RuntimeError("Temporal-feasibility eligible/ineligible count mismatch.")

    # No Stage27-0 protocol lock may have been created by this script.
    stage27_0 = repo / "results/stage27_loao_unseen_attack/stage27_0_protocol_lock"
    if stage27_0.exists():
        raise RuntimeError(
            "Stage27-0 protocol directory already exists. Stage27-0A must be "
            "remotely verified before Stage27-0 construction."
        )

    # Ensure no unexpected untracked/modified files besides the six intended files.
    porcelain = run(["git", "status", "--porcelain"], cwd=repo).splitlines()
    changed = []
    for line in porcelain:
        if not line.strip():
            continue
        # porcelain path starts after two status chars + space
        changed.append(line[3:] if len(line) >= 4 else line)

    expected_prefix = OUTPUT_REL + "/"
    unexpected = [p for p in changed if not p.startswith(expected_prefix)]
    if unexpected:
        raise RuntimeError(
            "Unexpected working-tree changes exist after artifact generation:\n"
            + "\n".join(unexpected)
        )

    print("Family rows                 : 7")
    print("Seven-family attack total  : 557,635")
    print("Benign total               : 2,273,097")
    print("Eligible folds             : 5")
    print("Structurally ineligible    : 2")
    print("Stage27-0 directory exists : NO")
    print("Model fits                 : 0")
    print("Model inference            : 0")
    print("GPU execution              : OFF")
    print("[PASS] Stage27-0A local validation complete.")


# =============================================================================
# 11. COMMIT + PUSH + REMOTE VERIFY
# =============================================================================

def commit_push_verify(
    repo: Path,
    env: Dict[str, str],
    output_paths: List[Path],
) -> str:
    banner("STAGE27-0A :: GIT COMMIT")

    rel_files = [str(p.relative_to(repo)) for p in output_paths]

    run(["git", "add", "--", *rel_files], cwd=repo)

    staged = run(["git", "diff", "--cached", "--name-only"], cwd=repo).splitlines()

    if sorted(staged) != sorted(rel_files):
        raise RuntimeError(
            "Unexpected staged file set.\n"
            f"expected={sorted(rel_files)}\n"
            f"actual={sorted(staged)}"
        )

    print("Staged exactly:")
    for rel in staged:
        print(" ", rel)

    print()
    print("Staged diff stat:")
    print(run(["git", "diff", "--cached", "--stat"], cwd=repo))

    run(["git", "commit", "-m", COMMIT_MESSAGE], cwd=repo)

    new_head = run(["git", "rev-parse", "HEAD"], cwd=repo)
    parent = run(["git", "rev-parse", "HEAD^"], cwd=repo)

    if parent != EXPECTED_PARENT:
        raise RuntimeError(
            f"New commit parent mismatch: expected {EXPECTED_PARENT}, got {parent}"
        )

    print()
    print("Commit created:", new_head)

    banner("STAGE27-0A :: PUSH MAIN")
    run(["git", "push", "origin", "main"], cwd=repo, env=env)
    print("[OK] Push completed.")

    banner("STAGE27-0A :: REMOTE VERIFICATION")
    remote_head = run(
        ["git", "ls-remote", "origin", "refs/heads/main"],
        cwd=repo,
        env=env,
    ).split()[0]

    run(["git", "fetch", "origin", "main"], cwd=repo, env=env)
    origin_main = run(["git", "rev-parse", "origin/main"], cwd=repo)
    local_head = run(["git", "rev-parse", "HEAD"], cwd=repo)

    print("Local HEAD   :", local_head)
    print("origin/main  :", origin_main)
    print("remote main  :", remote_head)

    if not (local_head == origin_main == remote_head == new_head):
        raise RuntimeError(
            "Remote verification mismatch after Stage27-0A push.\n"
            f"local={local_head}\norigin/main={origin_main}\nremote={remote_head}"
        )

    # Verify the exact six paths exist at remote-tracking main.
    remote_tree = run(
        ["git", "ls-tree", "-r", "--name-only", "origin/main", "--", OUTPUT_REL],
        cwd=repo,
    ).splitlines()

    expected_remote = sorted(
        [f"{OUTPUT_REL}/{name}" for name in REQUIRED_OUTPUTS]
    )
    actual_remote = sorted(remote_tree)

    if actual_remote != expected_remote:
        raise RuntimeError(
            "Remote Stage27-0A tree does not contain exactly the six expected files.\n"
            f"expected={expected_remote}\nactual={actual_remote}"
        )

    if not git_status_clean(repo):
        raise RuntimeError("Working tree is not clean after successful push.")

    print()
    print("[PASS] Stage27-0A commit is remotely verified.")
    print("[PASS] Remote tree contains exactly the six required Stage27-0A artifacts.")
    print("[PASS] Working tree is clean.")

    return new_head


# =============================================================================
# 12. MAIN
# =============================================================================

def main() -> None:
    # Hard-disable CUDA visibility for this CPU-only phase.
    os.environ["CUDA_VISIBLE_DEVICES"] = ""

    banner("STAGE27-0A :: ZERO-FIT FAMILY × DAY FEASIBILITY AUDIT")
    print("timestamp_utc            :", now_utc())
    print("python                   :", sys.version.replace("\n", " "))
    print("working_directory        :", Path.cwd())
    print("PRIMARY_EXECUTION_DEVICE :", PRIMARY_EXECUTION_DEVICE)
    print("PRIMARY_GPU_BUDGET_HOURS :", PRIMARY_GPU_BUDGET_HOURS)
    print("CUDA_VISIBLE_DEVICES     :", repr(os.environ.get("CUDA_VISIBLE_DEVICES")))
    print()
    print("Scientific boundary:")
    print("  model fits              : 0")
    print("  model inference         : 0")
    print("  threshold selection     : 0")
    print("  bootstrap replicates    : 0")
    print("  corpus downloads        : 0")
    print("  release corpus recreate : NO")

    token = get_kaggle_github_token()
    env, askpass = make_git_auth_env(token)

    try:
        repo = find_repo()
        ensure_repo(repo, env)

        outdir = repo / OUTPUT_REL
        banner("STAGE27-0A :: OUTPUT DIRECTORY GATE")
        print("Output directory:", outdir)
        if outdir.exists():
            existing = list(outdir.iterdir())
            print("Existing entries:", len(existing))
            if existing:
                raise RuntimeError(
                    "Stage27-0A output directory is not empty. "
                    "Refusing to overwrite prior artifacts."
                )
        else:
            print("Existing entries: 0 (directory not yet created)")
        print("[PASS] Stage27-0A output is clean.")

        stage26 = verify_stage26_closure(repo)
        stage24_receipts = verify_stage24_sources(repo)
        taxonomy = verify_taxonomy(repo)
        pop = recover_stage24_population(repo)
        compact_audit = audit_stage20_compact_binary_only(repo)
        matrix, support_summary = construct_support_matrix(pop)
        feasibility = derive_temporal_feasibility(
            matrix,
            pop["benign_by_day"],
        )

        output_paths = write_artifacts(
            repo=repo,
            stage26_receipt=stage26,
            stage24_receipts=stage24_receipts,
            taxonomy=taxonomy,
            pop=pop,
            compact_audit=compact_audit,
            matrix=matrix,
            support_summary=support_summary,
            feasibility=feasibility,
        )

        validate_outputs(
            repo=repo,
            output_paths=output_paths,
            matrix=matrix,
            pop=pop,
            feasibility=feasibility,
        )

        commit_sha = commit_push_verify(
            repo=repo,
            env=env,
            output_paths=output_paths,
        )

        banner("STAGE27-0A COMPLETE — REMOTELY VERIFIED")
        print("Execution parent          :", EXPECTED_PARENT)
        print("Stage27-0A commit         :", commit_sha)
        print("Remote main               :", commit_sha)
        print("Artifacts                 : 6/6")
        print("Eligible folds            : 5")
        print("Structurally ineligible   : 2")
        print("Descriptive-only eligible : INFILTRATION (n=36)")
        print("Model fits                : 0")
        print("Model inference           : 0")
        print("GPU execution             : OFF")
        print("Git working tree          : CLEAN")
        print()
        print("NEXT AUTHORIZED STEP:")
        print("  Construct Stage27-0 protocol lock from this remotely verified")
        print("  Stage27-0A feasibility correction.")
        print()
        print("STOPPING HERE. Stage27-0 is NOT created by this script.")
        print("=" * 110)

    finally:
        # Never retain auth material.
        try:
            if askpass.exists():
                askpass.unlink()
        except Exception:
            pass

        env.pop("STAGE27_GITHUB_TOKEN", None)
        os.environ.pop("STAGE27_GITHUB_TOKEN", None)

        # Best-effort overwrite local Python reference.
        token = ""


if __name__ == "__main__":
    main()



STAGE27-0A :: ZERO-FIT FAMILY × DAY FEASIBILITY AUDIT
timestamp_utc            : 2026-08-20T17:00:35.468905+00:00
python                   : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
working_directory        : /kaggle/working
PRIMARY_EXECUTION_DEVICE : CPU
PRIMARY_GPU_BUDGET_HOURS : 0
CUDA_VISIBLE_DEVICES     : ''

Scientific boundary:
  model fits              : 0
  model inference         : 0
  threshold selection     : 0
  bootstrap replicates    : 0
  corpus downloads        : 0
  release corpus recreate : NO
[FOUND] GitHub secret label: GITHUB_TOKEN (value not printed)

STAGE27-0A :: REPOSITORY BOOTSTRAP
Repository : /kaggle/working/ids2018-validation-safe-ablation
Remote     : https://github.com/themubasshir/ids2018-validation-safe-ablation.git

STAGE27-0A :: GIT GATE
Branch          : main
Expected parent : e47f44751bc71d219c5d0f3b3fca06d62037fb8b
Local HEAD      : e47f44751bc71d219c5d0f3b3fca06d62037fb8b
origin/main     : e47f44751bc71d219c5d0f3b3fca06d62037fb8b
remot

RuntimeError: Unexpected working-tree changes exist after artifact generation:
results/stage27_loao_unseen_attack/

In [7]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Stage27-0A — Family × Day Feasibility Audit
===========================================

Repository:
    themubasshir/ids2018-validation-safe-ablation

Scientific boundary:
    - ZERO model fits
    - ZERO model inference
    - ZERO threshold selection
    - ZERO bootstrap
    - ZERO GPU use
    - writes ONLY the six required Stage27-0A artifacts
    - commits + pushes Stage27-0A to main
    - remotely verifies the resulting commit
    - STOPS before Stage27-0 protocol construction

This script deliberately uses the full Stage24 CICIDS2017 effective target
population for Stage27 family support. The Stage20 compact raw-byte release
corpora are audited as binary-only corpora and therefore are not used as
seven-family support sources.

Expected execution parent:
    e47f44751bc71d219c5d0f3b3fca06d62037fb8b
"""

from __future__ import annotations

import csv
import hashlib
import json
import os
import shutil
import stat
import subprocess
import sys
from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, List, Tuple


# =============================================================================
# 0. FROZEN CONSTANTS
# =============================================================================

REPO_OWNER = "themubasshir"
REPO_NAME = "ids2018-validation-safe-ablation"
REPO_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"

EXPECTED_PARENT = "e47f44751bc71d219c5d0f3b3fca06d62037fb8b"
PRIMARY_EXECUTION_DEVICE = "CPU"
PRIMARY_GPU_BUDGET_HOURS = 0

WEEKDAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]

PRIMARY_FAMILIES = [
    "BOT",
    "DDOS",
    "DOS",
    "AUTH_BRUTE_FORCE",
    "INFILTRATION",
    "PORT_SCAN",
    "WEB_ATTACK",
]

EXPECTED_STAGE24_SOURCE_SHA256 = OrderedDict(
    [
        (
            "results/stage24_cross_dataset/stage24_0_protocol_lock/"
            "stage24_0c_final_preopening_protocol_lock.json",
            "8ef234a9d283f2008f21b9add4361f14328d1f3c1cffa278077f59d9eb9e37c2",
        ),
        (
            "results/stage24_cross_dataset/stage24_0_protocol_lock/"
            "stage24_0b2_complete_cicids2017_source_contract.json",
            "96f7fc5c0227660fe8ec17f5173630ee2ac71f6be199e81ee37dac0ad25d9779",
        ),
        (
            "results/stage24_cross_dataset/stage24_0_protocol_lock/"
            "stage24_0b_exact_cicids2017_source_recovery.json",
            "8a84f1dccc51f224427e00144088058590f48c8b603cfd798f6ba431d659da1c",
        ),
        (
            "results/stage24_cross_dataset/stage24_2_primary_target_openings/"
            "stage24_2a_bridge62_published/stage24_2a_bridge62_published_result.json",
            "1d4b3010fdc8b0f7f62a0fed2c373824ce99cb2f16936c021756749a9ca54749",
        ),
    ]
)

STAGE24_PROTOCOL_LOCK_REL = (
    "results/stage24_cross_dataset/stage24_0_protocol_lock/"
    "stage24_0c_final_preopening_protocol_lock.json"
)
STAGE24_SOURCE_CONTRACT_REL = (
    "results/stage24_cross_dataset/stage24_0_protocol_lock/"
    "stage24_0b2_complete_cicids2017_source_contract.json"
)
STAGE24_SOURCE_RECOVERY_REL = (
    "results/stage24_cross_dataset/stage24_0_protocol_lock/"
    "stage24_0b_exact_cicids2017_source_recovery.json"
)
STAGE24_TARGET_RESULT_REL = (
    "results/stage24_cross_dataset/stage24_2_primary_target_openings/"
    "stage24_2a_bridge62_published/stage24_2a_bridge62_published_result.json"
)

STAGE26_CLOSURE_CANDIDATES = [
    (
        "results/stage26_deployment_profiling/stage26_12_final_synthesis/"
        "stage26_g2_final_stage26_closure_receipt.json"
    ),
    (
        "results/stage26_deployment_profiling/stage26_g2_final_gpu_publication_closure/"
        "stage26_g2_final_stage26_closure_receipt.json"
    ),
]

# Stage20 compact manifests are checked only to prove that the frozen raw-byte
# release corpus contains binary labels, not seven-family labels.
STAGE20_COMPACT_MANIFESTS = OrderedDict(
    [
        (
            "Monday",
            "results/stage20_1e_training/"
            "stage20_1e1m_monday_compact_corpus_manifest.json",
        ),
        (
            "Tuesday",
            "results/stage20_1e_training/"
            "stage20_1e1t_tuesday_compact_corpus_manifest.json",
        ),
        (
            "Wednesday",
            "results/stage20_1e_training/"
            "stage20_1e1w_wednesday_compact_corpus_manifest.json",
        ),
        (
            "Thursday",
            "results/stage20_1e_training/"
            "stage20_1e1v_thursday_validation_compact_corpus_manifest.json",
        ),
        (
            "Friday",
            "results/stage20_1e_training/"
            "stage20_1e4_friday_holdout_compact_corpus_manifest.json",
        ),
    ]
)

OUTPUT_REL = (
    "results/stage27_loao_unseen_attack/"
    "stage27_0a_family_day_feasibility"
)

REQUIRED_OUTPUTS = [
    "family_day_support.csv",
    "benign_day_support.csv",
    "taxonomy_receipt.json",
    "source_artifact_receipts.json",
    "temporal_feasibility.json",
    "feasibility_audit.md",
]

# Frozen Stage24 taxonomy.
RAW_TO_FAMILY = OrderedDict(
    [
        ("bot", "BOT"),
        ("ddos", "DDOS"),
        ("dos goldeneye", "DOS"),
        ("dos hulk", "DOS"),
        ("dos slowhttptest", "DOS"),
        ("dos slowloris", "DOS"),
        ("ftp-patator", "AUTH_BRUTE_FORCE"),
        ("heartbleed", "TARGET_ONLY_UNSEEN"),
        ("infiltration", "INFILTRATION"),
        ("portscan", "PORT_SCAN"),
        ("ssh-patator", "AUTH_BRUTE_FORCE"),
        ("web attack - brute force", "WEB_ATTACK"),
        ("web attack - sql injection", "WEB_ATTACK"),
        ("web attack - xss", "WEB_ATTACK"),
    ]
)

# Explicit counts recovered from the durable Stage24 target population.
EXPECTED_CANONICAL_LABEL_COUNTS = OrderedDict(
    [
        ("benign", 2_273_097),
        ("bot", 1_966),
        ("ddos", 128_027),
        ("dos goldeneye", 10_293),
        ("dos hulk", 231_073),
        ("dos slowhttptest", 5_499),
        ("dos slowloris", 5_796),
        ("ftp-patator", 7_938),
        ("heartbleed", 11),
        ("infiltration", 36),
        ("portscan", 158_930),
        ("ssh-patator", 5_897),
        ("web attack - brute force", 1_507),
        ("web attack - sql injection", 21),
        ("web attack - xss", 652),
    ]
)

MIN_INFERENTIAL_POSITIVE_SUPPORT = 50

COMMIT_MESSAGE = "stage27-0a: freeze family-day feasibility audit"


# =============================================================================
# 1. GENERIC HELPERS
# =============================================================================

def now_utc() -> str:
    return datetime.now(timezone.utc).isoformat()


def banner(title: str, width: int = 110) -> None:
    print()
    print("=" * width)
    print(title)
    print("=" * width)


def run(
    args: List[str],
    *,
    cwd: Path | None = None,
    env: Dict[str, str] | None = None,
    check: bool = True,
    capture: bool = True,
) -> str:
    proc = subprocess.run(
        args,
        cwd=str(cwd) if cwd else None,
        env=env,
        text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.PIPE if capture else None,
    )
    if check and proc.returncode != 0:
        cmd = " ".join(args)
        out = proc.stdout or ""
        err = proc.stderr or ""
        raise RuntimeError(
            f"Command failed ({proc.returncode}): {cmd}\n\nSTDOUT:\n{out}\n\nSTDERR:\n{err}"
        )
    return (proc.stdout or "").strip()


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def write_json(path: Path, obj: Any) -> None:
    payload = json.dumps(
        obj,
        indent=2,
        ensure_ascii=False,
        sort_keys=False,
    ) + "\n"
    path.write_text(payload, encoding="utf-8")


def normalize_label(value: Any) -> str:
    s = str(value).strip().lower()
    # CICIDS2017 Web Attack strings sometimes carry cp1252/en-dash artifacts.
    for ch in ["\u0096", "\u2013", "\u2014", "–", "—"]:
        s = s.replace(ch, "-")
    s = " ".join(s.split())
    return s


def walk_nodes(obj: Any, path: str = "$") -> Iterable[Tuple[str, Any]]:
    yield path, obj
    if isinstance(obj, dict):
        for k, v in obj.items():
            yield from walk_nodes(v, f"{path}.{k}")
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            yield from walk_nodes(v, f"{path}[{i}]")


def contains_pair_anywhere(obj: Any, raw_label: str, family: str) -> bool:
    raw_norm = normalize_label(raw_label)
    fam_norm = normalize_label(family)
    for _, node in walk_nodes(obj):
        if not isinstance(node, dict):
            continue
        for k, v in node.items():
            if normalize_label(k) == raw_norm and normalize_label(v) == fam_norm:
                return True
    return False


def find_repo() -> Path:
    candidates = [
        Path("/kaggle/working") / REPO_NAME,
        Path.cwd() / REPO_NAME,
        Path.cwd(),
    ]
    for p in candidates:
        if (p / ".git").exists() and p.name == REPO_NAME:
            return p.resolve()
    return (Path("/kaggle/working") / REPO_NAME).resolve()


def git_status_clean(repo: Path) -> bool:
    return run(["git", "status", "--porcelain"], cwd=repo) == ""


# =============================================================================
# 2. TEMPORARY GITHUB AUTH
# =============================================================================

def get_kaggle_github_token() -> str:
    try:
        from kaggle_secrets import UserSecretsClient  # type: ignore
    except Exception as exc:
        raise RuntimeError(
            "kaggle_secrets is unavailable. Run this script in Kaggle or provide "
            "authenticated Git credentials another way."
        ) from exc

    client = UserSecretsClient()
    labels = ["GITHUB_TOKEN", "github_token", "GH_TOKEN", "github_pat", "GITHUB_PAT"]

    for label in labels:
        try:
            token = client.get_secret(label)
        except Exception:
            token = None
        if token:
            token = str(token).strip()
            if token:
                print(f"[FOUND] GitHub secret label: {label} (value not printed)")
                return token

    raise RuntimeError(
        "No usable GitHub token found in Kaggle Secrets. Expected one of: "
        + ", ".join(labels)
    )


def make_git_auth_env(token: str) -> Tuple[Dict[str, str], Path]:
    askpass = Path("/kaggle/working/.stage27_git_askpass.sh")
    askpass.write_text(
        "#!/bin/sh\n"
        'case "$1" in\n'
        '  *Username*) printf "%s\\n" "x-access-token" ;;\n'
        '  *Password*) printf "%s\\n" "$STAGE27_GITHUB_TOKEN" ;;\n'
        '  *) printf "%s\\n" "" ;;\n'
        "esac\n",
        encoding="utf-8",
    )
    askpass.chmod(stat.S_IRUSR | stat.S_IWUSR | stat.S_IXUSR)

    env = os.environ.copy()
    env["GIT_ASKPASS"] = str(askpass)
    env["GIT_TERMINAL_PROMPT"] = "0"
    env["STAGE27_GITHUB_TOKEN"] = token
    return env, askpass


# =============================================================================
# 3. REPOSITORY BOOTSTRAP + HARD GATES
# =============================================================================

def ensure_repo(repo: Path, env: Dict[str, str]) -> None:
    banner("STAGE27-0A :: REPOSITORY BOOTSTRAP")

    if not (repo / ".git").exists():
        repo.parent.mkdir(parents=True, exist_ok=True)
        print("Repository not present; cloning origin/main...")
        run(
            ["git", "clone", "--branch", "main", "--single-branch", REPO_URL, str(repo)],
            cwd=repo.parent,
            env=env,
        )

    remote_url = run(["git", "remote", "get-url", "origin"], cwd=repo)
    print("Repository :", repo)
    print("Remote     :", remote_url)

    # Do not permit a dirty starting tree.
    if not git_status_clean(repo):
        raise RuntimeError(
            "Git working tree is not clean. Stage27-0A refuses to overwrite or "
            "co-mingle unrelated work."
        )

    run(["git", "fetch", "origin", "main"], cwd=repo, env=env)

    branch = run(["git", "branch", "--show-current"], cwd=repo)
    head = run(["git", "rev-parse", "HEAD"], cwd=repo)
    origin_main = run(["git", "rev-parse", "origin/main"], cwd=repo)
    remote_main = run(
        ["git", "ls-remote", "origin", "refs/heads/main"],
        cwd=repo,
        env=env,
    ).split()[0]

    banner("STAGE27-0A :: GIT GATE")
    print("Branch          :", branch)
    print("Expected parent :", EXPECTED_PARENT)
    print("Local HEAD      :", head)
    print("origin/main     :", origin_main)
    print("remote main     :", remote_main)
    print("Repo clean      :", git_status_clean(repo))

    if branch != "main":
        raise RuntimeError(f"Expected branch 'main', found {branch!r}")

    if not (head == origin_main == remote_main == EXPECTED_PARENT):
        raise RuntimeError(
            "Pinned execution parent mismatch.\n"
            f"expected    : {EXPECTED_PARENT}\n"
            f"local HEAD  : {head}\n"
            f"origin/main : {origin_main}\n"
            f"remote main : {remote_main}\n\n"
            "Do not rebase or silently continue. Inspect the new remote state first."
        )

    print("[PASS] Exact Stage27 execution parent is pinned and clean.")


# =============================================================================
# 4. PRIOR-STAGE CLOSURE + SOURCE HASH GATES
# =============================================================================

def verify_stage26_closure(repo: Path) -> Dict[str, Any]:
    banner("STAGE27-0A :: VERIFY STAGE26 CLOSURE")

    hits = []
    for rel in STAGE26_CLOSURE_CANDIDATES:
        path = repo / rel
        if not path.exists():
            continue
        data = load_json(path)
        if data.get("stage26_status") == "COMPLETE":
            hits.append(
                {
                    "path": rel,
                    "sha256": sha256_file(path),
                    "bytes": path.stat().st_size,
                    "stage26_status": "COMPLETE",
                }
            )
            print("COMPLETE:", rel)

    if not hits:
        raise RuntimeError("No durable Stage26 stage26_status=COMPLETE receipt found.")

    print("[PASS] Stage26 closure verified.")
    return {"verified": True, "evidence": hits}


def verify_stage24_sources(repo: Path) -> List[Dict[str, Any]]:
    banner("STAGE27-0A :: VERIFY DURABLE STAGE24 SOURCES")

    receipts: List[Dict[str, Any]] = []

    for rel, expected_sha in EXPECTED_STAGE24_SOURCE_SHA256.items():
        path = repo / rel
        if not path.exists():
            raise FileNotFoundError(f"Missing required Stage24 artifact: {rel}")

        actual_sha = sha256_file(path)
        size = path.stat().st_size

        print()
        print("Artifact:", rel)
        print(" bytes   :", size)
        print(" expected:", expected_sha)
        print(" actual  :", actual_sha)

        if actual_sha != expected_sha:
            raise RuntimeError(
                f"SHA256 mismatch for {rel}\n"
                f"expected: {expected_sha}\n"
                f"actual  : {actual_sha}"
            )

        receipts.append(
            {
                "path": rel,
                "bytes": size,
                "sha256": actual_sha,
                "sha256_matches_frozen_expected": True,
            }
        )

    print()
    print("[PASS] All four Stage24 durable artifacts match frozen SHA256 values.")
    return receipts


# =============================================================================
# 5. TAXONOMY + STAGE24 EFFECTIVE POPULATION RECOVERY
# =============================================================================

def verify_taxonomy(repo: Path) -> Dict[str, Any]:
    banner("STAGE27-0A :: VERIFY FROZEN STAGE24 TAXONOMY")

    protocol = load_json(repo / STAGE24_PROTOCOL_LOCK_REL)

    missing_pairs = []
    for raw, fam in RAW_TO_FAMILY.items():
        ok = contains_pair_anywhere(protocol, raw, fam)
        print(f"{raw:<34} -> {fam:<20} {'[OK]' if ok else '[MISSING]'}")
        if not ok:
            missing_pairs.append((raw, fam))

    if missing_pairs:
        raise RuntimeError(
            "Frozen Stage24 taxonomy pairs were not all found in the protocol lock: "
            + repr(missing_pairs)
        )

    recovered_primary = sorted(
        {fam for fam in RAW_TO_FAMILY.values() if fam in PRIMARY_FAMILIES}
    )
    if recovered_primary != sorted(PRIMARY_FAMILIES):
        raise RuntimeError(
            f"Primary taxonomy mismatch: {recovered_primary} vs {sorted(PRIMARY_FAMILIES)}"
        )

    print()
    print("[PASS] Exact seven-family Stage24 taxonomy recovered.")

    return {
        "primary_families": PRIMARY_FAMILIES,
        "raw_label_to_stage27_family": RAW_TO_FAMILY,
        "excluded_from_primary_seven_family_taxonomy": {
            "heartbleed": "TARGET_ONLY_UNSEEN"
        },
    }


def recover_stage24_population(repo: Path) -> Dict[str, Any]:
    banner("STAGE27-0A :: RECOVER STAGE24 EFFECTIVE CICIDS2017 POPULATION")

    result = load_json(repo / STAGE24_TARGET_RESULT_REL)

    try:
        file_level = result["metrics"]["file_level_descriptive"]
        label_counts_raw = result["target_population"]["canonical_label_counts"]
    except Exception as exc:
        raise RuntimeError(
            "Expected Stage24 population structures were not found at "
            "$.metrics.file_level_descriptive and "
            "$.target_population.canonical_label_counts."
        ) from exc

    label_counts = {
        normalize_label(k): int(v)
        for k, v in label_counts_raw.items()
    }

    # Exact canonical label-count gate.
    for label, expected in EXPECTED_CANONICAL_LABEL_COUNTS.items():
        actual = label_counts.get(label)
        if actual != expected:
            raise RuntimeError(
                f"Canonical Stage24 count mismatch for {label!r}: "
                f"expected {expected}, actual {actual}"
            )

    # Day/file support gate.
    expected_files = [
        ("Monday", 529_918, 529_918, 0),
        ("Tuesday", 445_909, 432_074, 13_835),
        ("Wednesday", 692_703, 440_031, 252_672),
        ("Thursday", 288_602, 288_566, 36),
        ("Thursday", 170_366, 168_186, 2_180),
        ("Friday", 225_745, 97_718, 128_027),
        ("Friday", 286_467, 127_537, 158_930),
        ("Friday", 191_033, 189_067, 1_966),
    ]

    if len(file_level) != len(expected_files):
        raise RuntimeError(
            f"Expected {len(expected_files)} Stage24 file-level rows; found {len(file_level)}"
        )

    normalized_rows = []
    for idx, row in enumerate(file_level):
        day = str(row["day"])
        rows = int(row["rows"])
        benign = int(row["benign"])
        attack = int(row["attack"])
        remote = str(row["remote"])

        exp_day, exp_rows, exp_benign, exp_attack = expected_files[idx]
        if (day, rows, benign, attack) != (
            exp_day,
            exp_rows,
            exp_benign,
            exp_attack,
        ):
            raise RuntimeError(
                "Stage24 file-level support mismatch at index "
                f"{idx}: got {(day, rows, benign, attack)}, expected "
                f"{(exp_day, exp_rows, exp_benign, exp_attack)}"
            )

        normalized_rows.append(
            {
                "file_id": int(row.get("file_id", idx)),
                "day": day,
                "remote": remote,
                "rows": rows,
                "benign": benign,
                "attack": attack,
                "start": int(row["start"]),
                "stop": int(row["stop"]),
            }
        )

    benign_by_day = OrderedDict((day, 0) for day in WEEKDAYS)
    rows_by_day = OrderedDict((day, 0) for day in WEEKDAYS)
    all_attack_by_day = OrderedDict((day, 0) for day in WEEKDAYS)

    for row in normalized_rows:
        day = row["day"]
        benign_by_day[day] += row["benign"]
        rows_by_day[day] += row["rows"]
        all_attack_by_day[day] += row["attack"]

    expected_benign = OrderedDict(
        [
            ("Monday", 529_918),
            ("Tuesday", 432_074),
            ("Wednesday", 440_031),
            ("Thursday", 456_752),
            ("Friday", 414_322),
        ]
    )
    if benign_by_day != expected_benign:
        raise RuntimeError(
            f"Benign weekday support mismatch:\nactual={benign_by_day}\n"
            f"expected={expected_benign}"
        )

    if sum(benign_by_day.values()) != 2_273_097:
        raise RuntimeError("Total benign support mismatch.")

    if sum(all_attack_by_day.values()) != 557_646:
        raise RuntimeError("Total non-benign support mismatch.")

    print()
    for row in normalized_rows:
        print(
            f"{row['day']:<10} rows={row['rows']:>7,} "
            f"benign={row['benign']:>7,} attack={row['attack']:>7,}  "
            f"{row['remote']}"
        )

    print()
    print("[PASS] Stage24 effective CICIDS2017 population is exact.")

    return {
        "file_level": normalized_rows,
        "label_counts": label_counts,
        "benign_by_day": benign_by_day,
        "rows_by_day": rows_by_day,
        "all_attack_by_day": all_attack_by_day,
    }


# =============================================================================
# 6. STAGE20 COMPACT CORPUS LIMITATION AUDIT
# =============================================================================

def audit_stage20_compact_binary_only(repo: Path) -> Dict[str, Any]:
    banner("STAGE27-0A :: AUDIT STAGE20 COMPACT RELEASE SEMANTICS")

    manifest_receipts = []
    all_binary_only = True

    for day, rel in STAGE20_COMPACT_MANIFESTS.items():
        path = repo / rel
        if not path.exists():
            raise FileNotFoundError(f"Missing Stage20 compact manifest: {rel}")

        data = load_json(path)
        compact = data.get("compact_corpus", {})
        files = compact.get("files", {})

        keys = sorted(files.keys())
        expected_keys = sorted(
            [
                "encoded_bytes.bin",
                "flow_offsets.npy",
                "labels.npy",
                "packet_lengths.npy",
            ]
        )

        labels_file_present = "labels.npy" in files
        family_label_file_present = any(
            (
                "family" in k.lower()
                or "attack_type" in k.lower()
                or "multiclass" in k.lower()
            )
            for k in files.keys()
        )

        exact_join = data.get("exact_join", {})
        binary_counts = exact_join.get("matched_binary_counts", {})
        binary_key_set = {str(k) for k in binary_counts.keys()}
        binary_only_counts = binary_key_set.issubset({"0", "1"}) and bool(binary_counts)

        schema_ok = (
            keys == expected_keys
            and labels_file_present
            and not family_label_file_present
            and binary_only_counts
        )

        all_binary_only = all_binary_only and schema_ok

        print()
        print(day)
        print("  manifest:", rel)
        print("  files   :", keys)
        print("  matched_binary_counts:", binary_counts)
        print("  family-label array present:", family_label_file_present)
        print("  binary-only schema gate:", schema_ok)

        manifest_receipts.append(
            {
                "day": day,
                "path": rel,
                "bytes": path.stat().st_size,
                "sha256": sha256_file(path),
                "compact_files": keys,
                "flow_count": int(compact.get("flow_count", 0)),
                "matched_binary_counts": {
                    str(k): int(v) for k, v in binary_counts.items()
                },
                "family_label_array_present": family_label_file_present,
                "binary_only_for_stage27_family_identity": schema_ok,
            }
        )

    if not all_binary_only:
        raise RuntimeError(
            "Stage20 compact-corpus binary-only audit did not pass consistently."
        )

    print()
    print(
        "[PASS] All five Stage20 compact manifests persist binary labels only; "
        "they cannot independently certify seven-family LOAO membership."
    )

    return {
        "all_five_compact_manifests_binary_only": True,
        "stage27_family_support_source": "STAGE24_FULL_CICIDS2017_EFFECTIVE_POPULATION",
        "stage20_compact_release_role": (
            "VALID_FOR_STAGE20_BINARY_RAW_BYTE_EXPERIMENTS_BUT_NOT_A_DURABLE_"
            "SEVEN_FAMILY_MEMBERSHIP_SOURCE"
        ),
        "manifests": manifest_receipts,
    }


# =============================================================================
# 7. SUPPORT MATRIX CONSTRUCTION
# =============================================================================

def construct_support_matrix(pop: Dict[str, Any]) -> Tuple[OrderedDict, OrderedDict]:
    banner("STAGE27-0A :: CONSTRUCT SEVEN-FAMILY × WEEKDAY SUPPORT MATRIX")

    c = pop["label_counts"]

    dos_total = (
        c["dos goldeneye"]
        + c["dos hulk"]
        + c["dos slowhttptest"]
        + c["dos slowloris"]
    )
    auth_total = c["ftp-patator"] + c["ssh-patator"]
    web_total = (
        c["web attack - brute force"]
        + c["web attack - sql injection"]
        + c["web attack - xss"]
    )

    family_totals = OrderedDict(
        [
            ("BOT", c["bot"]),
            ("DDOS", c["ddos"]),
            ("DOS", dos_total),
            ("AUTH_BRUTE_FORCE", auth_total),
            ("INFILTRATION", c["infiltration"]),
            ("PORT_SCAN", c["portscan"]),
            ("WEB_ATTACK", web_total),
        ]
    )

    expected_totals = OrderedDict(
        [
            ("BOT", 1_966),
            ("DDOS", 128_027),
            ("DOS", 252_661),
            ("AUTH_BRUTE_FORCE", 13_835),
            ("INFILTRATION", 36),
            ("PORT_SCAN", 158_930),
            ("WEB_ATTACK", 2_180),
        ]
    )
    if family_totals != expected_totals:
        raise RuntimeError(
            f"Family arithmetic mismatch:\nactual={family_totals}\nexpected={expected_totals}"
        )

    matrix = OrderedDict()
    for family in PRIMARY_FAMILIES:
        matrix[family] = OrderedDict((day, 0) for day in WEEKDAYS)

    # CICIDS2017 attack schedule recovered from the exact Stage24 file-level
    # target population:
    matrix["AUTH_BRUTE_FORCE"]["Tuesday"] = auth_total
    matrix["DOS"]["Wednesday"] = dos_total
    matrix["INFILTRATION"]["Thursday"] = c["infiltration"]
    matrix["WEB_ATTACK"]["Thursday"] = web_total
    matrix["DDOS"]["Friday"] = c["ddos"]
    matrix["PORT_SCAN"]["Friday"] = c["portscan"]
    matrix["BOT"]["Friday"] = c["bot"]

    # Cross-check each day against durable Stage24 file-level attack support.
    # Wednesday carries an additional 11 Heartbleed rows excluded from the
    # seven primary families.
    expected_day_attack = OrderedDict(
        [
            ("Monday", 0),
            ("Tuesday", 13_835),
            ("Wednesday", 252_672),
            ("Thursday", 2_216),
            ("Friday", 288_923),
        ]
    )

    primary_day_attack = OrderedDict(
        (
            day,
            sum(matrix[fam][day] for fam in PRIMARY_FAMILIES),
        )
        for day in WEEKDAYS
    )

    heartbleed_by_day = OrderedDict((day, 0) for day in WEEKDAYS)
    heartbleed_by_day["Wednesday"] = c["heartbleed"]

    reconstructed_all_attack = OrderedDict(
        (
            day,
            primary_day_attack[day] + heartbleed_by_day[day],
        )
        for day in WEEKDAYS
    )

    if reconstructed_all_attack != expected_day_attack:
        raise RuntimeError(
            "Reconstructed day attack totals do not match Stage24 file-level evidence.\n"
            f"reconstructed={reconstructed_all_attack}\nexpected={expected_day_attack}"
        )

    if expected_day_attack != pop["all_attack_by_day"]:
        raise RuntimeError(
            "Stage24 file-level attack support differs from expected reconstructed support."
        )

    seven_family_total = sum(family_totals.values())
    all_nonbenign = seven_family_total + c["heartbleed"]

    if seven_family_total != 557_635:
        raise RuntimeError("Seven-family total must equal 557,635.")
    if all_nonbenign != 557_646:
        raise RuntimeError("All non-benign total must equal 557,646.")

    for fam in PRIMARY_FAMILIES:
        vals = matrix[fam]
        print(
            f"{fam:<20} "
            + " ".join(f"{day[:3]}={vals[day]:>7,}" for day in WEEKDAYS)
            + f"  total={sum(vals.values()):>7,}"
        )

    print()
    print("Seven-family attack total :", f"{seven_family_total:,}")
    print("Heartbleed excluded       :", f"{c['heartbleed']:,}")
    print("All non-benign total      :", f"{all_nonbenign:,}")
    print("[PASS] Family/day support arithmetic is exact.")

    summary = OrderedDict(
        [
            ("matrix", matrix),
            ("family_totals", family_totals),
            ("primary_day_attack", primary_day_attack),
            ("heartbleed_by_day", heartbleed_by_day),
            ("all_attack_by_day", reconstructed_all_attack),
            ("seven_family_attack_total", seven_family_total),
            ("heartbleed_target_only_unseen", c["heartbleed"]),
            ("all_nonbenign_total", all_nonbenign),
        ]
    )

    return matrix, summary


# =============================================================================
# 8. TEMPORAL FEASIBILITY
# =============================================================================

def first_positive_day(matrix: OrderedDict, family: str) -> str:
    positive = [day for day in WEEKDAYS if matrix[family][day] > 0]
    if len(positive) != 1:
        raise RuntimeError(
            f"Expected exactly one positive weekday for {family}, found {positive}"
        )
    return positive[0]


def sum_family_support(
    matrix: OrderedDict,
    days: List[str],
    *,
    exclude_family: str | None = None,
) -> OrderedDict:
    out = OrderedDict()
    for fam in PRIMARY_FAMILIES:
        if exclude_family is not None and fam == exclude_family:
            continue
        out[fam] = sum(matrix[fam][d] for d in days)
    return out


def derive_temporal_feasibility(
    matrix: OrderedDict,
    benign_by_day: OrderedDict,
) -> Dict[str, Any]:
    banner("STAGE27-0A :: TEMPORAL FEASIBILITY")

    # Original proposal:
    original_train = ["Monday", "Tuesday", "Wednesday"]
    original_validation = ["Thursday"]
    original_target = ["Friday"]

    family_records = OrderedDict()

    for family in PRIMARY_FAMILIES:
        target_day = first_positive_day(matrix, family)
        target_idx = WEEKDAYS.index(target_day)

        heldout_support = matrix[family][target_day]
        support_status = (
            "INFERENTIAL_ELIGIBLE_SUPPORT"
            if heldout_support >= MIN_INFERENTIAL_POSITIVE_SUPPORT
            else "DESCRIPTIVE_ONLY_SUPPORT_LT_50"
        )

        original_target_support = sum(matrix[family][d] for d in original_target)
        original_train_support = sum(matrix[family][d] for d in original_train)
        original_val_support = sum(matrix[family][d] for d in original_validation)

        original_family_geometry_valid = (
            original_target_support > 0
            and original_train_support == 0
            and original_val_support == 0
        )

        record: Dict[str, Any] = {
            "held_out_family": family,
            "first_and_only_positive_day": target_day,
            "target_positive_support": heldout_support,
            "support_status": support_status,
            "original_mon_wed_train_thu_validation_fri_target": {
                "heldout_train_support": original_train_support,
                "heldout_validation_support": original_val_support,
                "heldout_target_support": original_target_support,
                "family_target_exists_under_original_geometry": original_family_geometry_valid,
            },
        }

        # Day-atomic strict chronology:
        #   TRAIN = all weekdays before VALIDATION
        #   VALIDATION = the weekday immediately preceding TARGET
        #   TARGET = family-positive weekday
        #
        # Additional requirements:
        #   - held-out family absent from TRAIN and VALIDATION
        #   - TRAIN has benign AND at least one known-family positive
        #   - VALIDATION has benign AND at least one known-family positive
        # These conditions are needed for a supervised binary fit and known-family
        # threshold selection without target-family labels.
        if target_idx < 2:
            record.update(
                {
                    "status": "STRUCTURALLY_INELIGIBLE",
                    "reason_code": "INSUFFICIENT_PRETARGET_WEEKDAY_DEPTH",
                    "reason": (
                        f"{family} first appears on {target_day}. Fewer than two "
                        "earlier weekday partitions exist, so separate day-atomic "
                        "TRAIN and VALIDATION partitions cannot both precede TARGET."
                    ),
                    "replacement_geometry": None,
                }
            )
        else:
            validation_day = WEEKDAYS[target_idx - 1]
            train_days = WEEKDAYS[: target_idx - 1]
            validation_days = [validation_day]
            target_days = [target_day]

            train_family = sum_family_support(
                matrix, train_days, exclude_family=family
            )
            val_family = sum_family_support(
                matrix, validation_days, exclude_family=family
            )

            heldout_train = sum(matrix[family][d] for d in train_days)
            heldout_val = sum(matrix[family][d] for d in validation_days)

            train_attack = sum(train_family.values())
            val_attack = sum(val_family.values())

            train_benign = sum(benign_by_day[d] for d in train_days)
            val_benign = sum(benign_by_day[d] for d in validation_days)
            target_benign = benign_by_day[target_day]

            if heldout_train != 0 or heldout_val != 0:
                raise RuntimeError(
                    f"Held-out leakage discovered during feasibility construction for {family}"
                )

            if train_attack <= 0:
                status = "STRUCTURALLY_INELIGIBLE"
                reason_code = "NO_KNOWN_ATTACK_POSITIVES_IN_DAY_ATOMIC_TRAIN"
                reason = (
                    f"The only day-atomic chronology before {target_day} gives "
                    f"TRAIN={train_days}, but TRAIN contains {train_attack} known-family "
                    "attack positives. A supervised attack-vs-benign learner cannot be "
                    "fit honestly under this geometry."
                )
                geometry = None
            elif val_attack <= 0:
                status = "STRUCTURALLY_INELIGIBLE"
                reason_code = "NO_KNOWN_ATTACK_POSITIVES_IN_DAY_ATOMIC_VALIDATION"
                reason = (
                    f"VALIDATION={validation_days} contains no known-family attack "
                    "positives for threshold selection."
                )
                geometry = None
            else:
                status = (
                    "ELIGIBLE_DESCRIPTIVE_ONLY"
                    if heldout_support < MIN_INFERENTIAL_POSITIVE_SUPPORT
                    else "ELIGIBLE"
                )
                reason_code = "DAY_ATOMIC_STRICT_CHRONOLOGY_AVAILABLE"
                reason = (
                    "A strict day-atomic TRAIN < VALIDATION < TARGET geometry exists "
                    "with zero held-out-family support in both TRAIN and VALIDATION."
                )
                geometry = {
                    "train_days": train_days,
                    "validation_days": validation_days,
                    "target_days": target_days,
                    "heldout_family_train_count": heldout_train,
                    "heldout_family_validation_count": heldout_val,
                    "train_benign": train_benign,
                    "train_known_attack": train_attack,
                    "train_known_family_support": train_family,
                    "validation_benign": val_benign,
                    "validation_known_attack": val_attack,
                    "validation_known_family_support": val_family,
                    "target_benign_available_same_day": target_benign,
                    "target_heldout_attack": heldout_support,
                }

            record.update(
                {
                    "status": status,
                    "reason_code": reason_code,
                    "reason": reason,
                    "replacement_geometry": geometry,
                }
            )

        family_records[family] = record

    eligible = [
        fam
        for fam, rec in family_records.items()
        if rec["status"] in {"ELIGIBLE", "ELIGIBLE_DESCRIPTIVE_ONLY"}
    ]
    ineligible = [
        fam
        for fam, rec in family_records.items()
        if rec["status"] == "STRUCTURALLY_INELIGIBLE"
    ]
    descriptive_only = [
        fam
        for fam, rec in family_records.items()
        if rec["status"] == "ELIGIBLE_DESCRIPTIVE_ONLY"
    ]

    original_supported = [
        fam
        for fam, rec in family_records.items()
        if rec["original_mon_wed_train_thu_validation_fri_target"][
            "family_target_exists_under_original_geometry"
        ]
    ]

    # Expected day-atomic outcome from the frozen support schedule.
    expected_eligible = ["BOT", "DDOS", "INFILTRATION", "PORT_SCAN", "WEB_ATTACK"]
    expected_ineligible = ["DOS", "AUTH_BRUTE_FORCE"]
    if sorted(eligible) != sorted(expected_eligible):
        raise RuntimeError(
            f"Unexpected eligible fold set: {eligible}; expected {expected_eligible}"
        )
    if sorted(ineligible) != sorted(expected_ineligible):
        raise RuntimeError(
            f"Unexpected ineligible fold set: {ineligible}; expected {expected_ineligible}"
        )
    if descriptive_only != ["INFILTRATION"]:
        raise RuntimeError(
            f"Expected INFILTRATION to be the only descriptive-only eligible fold; "
            f"got {descriptive_only}"
        )

    print("Original universal Mon-Wed / Thu / Fri geometry: INFEASIBLE FOR 7/7")
    print("Families with Friday targets under original geometry:", original_supported)
    print()
    print("Corrected day-atomic eligible folds :", eligible)
    print("Structurally ineligible folds       :", ineligible)
    print("Descriptive-only eligible folds     :", descriptive_only)
    print()

    for fam, rec in family_records.items():
        print(
            f"{fam:<20} {rec['status']:<28} "
            f"target={rec['first_and_only_positive_day']:<9} "
            f"n={rec['target_positive_support']:,}"
        )

    print()
    print("[PASS] Temporal feasibility derived before any Stage27 model result exists.")

    return {
        "stage": "Stage27-0A",
        "created_at_utc": now_utc(),
        "scientific_boundary": {
            "model_fits": 0,
            "model_inference": 0,
            "threshold_selection": 0,
            "bootstrap_replicates": 0,
            "gpu_execution": False,
            "primary_execution_device": PRIMARY_EXECUTION_DEVICE,
            "primary_gpu_budget_hours": PRIMARY_GPU_BUDGET_HOURS,
        },
        "chronology_semantics": {
            "unit": "WEEKDAY_ATOMIC_PARTITIONS",
            "required_order": "TRAIN < VALIDATION < TARGET",
            "train_requires_benign": True,
            "train_requires_known_attack_positive": True,
            "validation_requires_benign": True,
            "validation_requires_known_attack_positive": True,
            "heldout_family_required_absent_from_train": True,
            "heldout_family_required_absent_from_validation": True,
        },
        "original_proposal": {
            "train_days": original_train,
            "validation_days": original_validation,
            "target_days": original_target,
            "universal_seven_fold_feasible": False,
            "families_with_positive_target_support": original_supported,
            "reason": (
                "Only BOT, DDOS and PORT_SCAN have Friday positives. DOS appears only "
                "Wednesday; AUTH_BRUTE_FORCE only Tuesday; INFILTRATION and WEB_ATTACK "
                "only Thursday. The universal Friday-target geometry therefore cannot "
                "produce all seven held-out-family targets."
            ),
        },
        "replacement_day_atomic_family_specific_geometry": {
            "status": "FEASIBILITY_CORRECTION_PRE_RESULT",
            "eligible_folds": eligible,
            "eligible_fold_count": len(eligible),
            "structurally_ineligible_folds": ineligible,
            "structurally_ineligible_fold_count": len(ineligible),
            "descriptive_only_eligible_folds": descriptive_only,
            "minimum_positive_support_for_inferential_claim": MIN_INFERENTIAL_POSITIVE_SUPPORT,
            "family_records": family_records,
        },
        "important_semantic_limit": (
            "Strict chronology means some non-held-out families occur only on or after "
            "a target day and therefore cannot all be represented in that fold's "
            "training set. Stage27-0 must describe the executable design as a "
            "chronology-first zero-training-exposure family audit and must not imply "
            "that every non-held-out family was necessarily seen in training."
        ),
        "stage27_0_authorization": (
            "AUTHORIZED_ONLY_AFTER_THIS_STAGE27_0A_COMMIT_IS_REMOTELY_VERIFIED"
        ),
    }


# =============================================================================
# 9. WRITE EXACTLY SIX REQUIRED ARTIFACTS
# =============================================================================

def write_family_day_csv(
    outdir: Path,
    matrix: OrderedDict,
) -> None:
    path = outdir / "family_day_support.csv"
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["family", *WEEKDAYS, "Total"])
        for fam in PRIMARY_FAMILIES:
            row = [fam] + [matrix[fam][d] for d in WEEKDAYS]
            row.append(sum(matrix[fam].values()))
            writer.writerow(row)


def write_benign_day_csv(
    outdir: Path,
    benign_by_day: OrderedDict,
) -> None:
    path = outdir / "benign_day_support.csv"
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["day", "benign_support"])
        for day in WEEKDAYS:
            writer.writerow([day, benign_by_day[day]])
        writer.writerow(["TOTAL", sum(benign_by_day.values())])


def markdown_table_family(matrix: OrderedDict) -> str:
    lines = [
        "| Family | Mon | Tue | Wed | Thu | Fri | Total |",
        "|---|---:|---:|---:|---:|---:|---:|",
    ]
    for fam in PRIMARY_FAMILIES:
        vals = [matrix[fam][d] for d in WEEKDAYS]
        lines.append(
            f"| {fam} | "
            + " | ".join(f"{v:,}" for v in vals)
            + f" | {sum(vals):,} |"
        )
    return "\n".join(lines)


def markdown_table_benign(benign_by_day: OrderedDict) -> str:
    lines = [
        "| Day | Benign support |",
        "|---|---:|",
    ]
    for day in WEEKDAYS:
        lines.append(f"| {day} | {benign_by_day[day]:,} |")
    lines.append(f"| **TOTAL** | **{sum(benign_by_day.values()):,}** |")
    return "\n".join(lines)


def markdown_table_feasibility(feas: Dict[str, Any]) -> str:
    records = feas["replacement_day_atomic_family_specific_geometry"]["family_records"]
    lines = [
        "| Family | Target day | Positive support | Stage27-0A status | Corrected geometry |",
        "|---|---|---:|---|---|",
    ]
    for fam in PRIMARY_FAMILIES:
        r = records[fam]
        geom = r["replacement_geometry"]
        if geom is None:
            geom_text = "—"
        else:
            geom_text = (
                f"{','.join(geom['train_days'])} → "
                f"{','.join(geom['validation_days'])} → "
                f"{','.join(geom['target_days'])}"
            )
        lines.append(
            f"| {fam} | {r['first_and_only_positive_day']} | "
            f"{r['target_positive_support']:,} | {r['status']} | {geom_text} |"
        )
    return "\n".join(lines)


def write_artifacts(
    repo: Path,
    stage26_receipt: Dict[str, Any],
    stage24_receipts: List[Dict[str, Any]],
    taxonomy: Dict[str, Any],
    pop: Dict[str, Any],
    compact_audit: Dict[str, Any],
    matrix: OrderedDict,
    support_summary: OrderedDict,
    feasibility: Dict[str, Any],
) -> List[Path]:
    banner("STAGE27-0A :: WRITE SIX REQUIRED ARTIFACTS")

    outdir = repo / OUTPUT_REL

    if outdir.exists():
        existing = [p for p in outdir.iterdir() if p.is_file() or p.is_dir()]
        if existing:
            raise RuntimeError(
                "Stage27-0A output directory is not empty. Refusing to overwrite:\n"
                + "\n".join(str(p) for p in existing)
            )
    outdir.mkdir(parents=True, exist_ok=True)

    write_family_day_csv(outdir, matrix)
    write_benign_day_csv(outdir, pop["benign_by_day"])

    taxonomy_receipt = {
        "stage": "Stage27-0A",
        "type": "TAXONOMY_RECEIPT",
        "created_at_utc": now_utc(),
        "execution_parent": EXPECTED_PARENT,
        "taxonomy_source": STAGE24_PROTOCOL_LOCK_REL,
        "taxonomy_source_sha256": EXPECTED_STAGE24_SOURCE_SHA256[
            STAGE24_PROTOCOL_LOCK_REL
        ],
        "primary_family_count": 7,
        "primary_families": PRIMARY_FAMILIES,
        "raw_label_to_family": RAW_TO_FAMILY,
        "target_only_unseen_not_in_primary_seven_family_taxonomy": {
            "heartbleed": {
                "mapping": "TARGET_ONLY_UNSEEN",
                "support": int(pop["label_counts"]["heartbleed"]),
                "weekday": "Wednesday",
            }
        },
        "taxonomy_mutated": False,
        "family_merge_after_stage24": False,
        "family_split_after_stage24": False,
        "status": "EXACT_STAGE24_TAXONOMY_INHERITED",
    }
    write_json(outdir / "taxonomy_receipt.json", taxonomy_receipt)

    source_receipt = {
        "stage": "Stage27-0A",
        "type": "SOURCE_ARTIFACT_RECEIPTS",
        "created_at_utc": now_utc(),
        "execution_parent": EXPECTED_PARENT,
        "stage26_closure": stage26_receipt,
        "stage24_durable_sources": stage24_receipts,
        "effective_stage27_support_population": {
            "dataset": "CICIDS2017",
            "population_identity": "STAGE24_FULL_EFFECTIVE_TARGET_POPULATION",
            "rows": int(sum(pop["rows_by_day"].values())),
            "benign": int(sum(pop["benign_by_day"].values())),
            "nonbenign_including_heartbleed": int(
                sum(pop["all_attack_by_day"].values())
            ),
            "primary_seven_family_attack": int(
                support_summary["seven_family_attack_total"]
            ),
            "heartbleed_target_only_unseen": int(
                support_summary["heartbleed_target_only_unseen"]
            ),
            "support_source": STAGE24_TARGET_RESULT_REL,
        },
        "stage20_compact_release_population_decision": compact_audit,
        "population_discrepancy_resolution": {
            "discrepancy": (
                "Stage20 compact corpora are much smaller exact-matched raw-byte "
                "corpora, whereas Stage24 exposes the full 2,830,743-row CICIDS2017 "
                "effective target population."
            ),
            "decision": (
                "Use the Stage24 full effective CICIDS2017 population for Stage27 "
                "seven-family feasibility because it durably preserves the canonical "
                "attack-family label counts. Do not substitute Stage20 compact binary "
                "labels for family identities."
            ),
            "reason": (
                "All five committed Stage20 compact manifests persist labels.npy as "
                "binary labels and do not persist a family-label array or durable "
                "seven-family membership mapping."
            ),
            "scientific_adaptation": False,
            "decision_time": "BEFORE_ANY_STAGE27_MODEL_FIT_OR_INFERENCE",
        },
        "model_fits": 0,
        "model_inference": 0,
        "corpus_archives_downloaded_by_stage27_0a": 0,
        "release_corpus_recreated": False,
    }
    write_json(outdir / "source_artifact_receipts.json", source_receipt)

    write_json(outdir / "temporal_feasibility.json", feasibility)

    audit_md = f"""# Stage27-0A — Family × Day Feasibility Audit

**Status:** COMPLETE — ZERO FIT / ZERO INFERENCE FEASIBILITY AUDIT  
**Execution parent:** `{EXPECTED_PARENT}`  
**Primary compute policy:** CPU, GPU budget = 0 hours  
**Created:** {now_utc()}

## Scientific purpose

Stage27-0A determines which chronology-first unseen-family folds are structurally
possible **before** any Stage27 model results exist.

The originally proposed universal geometry was:

```text
Monday–Wednesday -> TRAIN
Thursday         -> VALIDATION
Friday           -> HELD-OUT FAMILY TARGET
```

That geometry is **not feasible for all seven families** because CICIDS2017 attack
families occur on different weekdays.

## Population decision

Stage27-0A uses the **full Stage24 CICIDS2017 effective target population** as the
family-support source.

This is deliberate. The committed Stage20 compact raw-byte release corpora preserve
exact raw-byte flows but persist only:

```text
encoded_bytes.bin
flow_offsets.npy
labels.npy
packet_lengths.npy
```

and the manifest join counts are binary `0/1`. No durable attack-family label array is
present. Therefore the Stage20 compact corpus remains valid for the Stage20 binary
raw-byte experiment, but it is **not** a durable seven-family membership source for
Stage27 LOAO construction.

No release archive was downloaded or recreated during Stage27-0A.

## Frozen seven-family taxonomy

```text
BOT
DDOS
DOS
AUTH_BRUTE_FORCE
INFILTRATION
PORT_SCAN
WEB_ATTACK
```

`heartbleed` remains `TARGET_ONLY_UNSEEN` and is excluded from the seven-family
primary matrix.

## Exact family × weekday support

{markdown_table_family(matrix)}

Seven-family total: **{support_summary['seven_family_attack_total']:,}**

Heartbleed excluded from the primary seven-family matrix: **{support_summary['heartbleed_target_only_unseen']:,}**

All non-benign rows including Heartbleed: **{support_summary['all_nonbenign_total']:,}**

## Benign support by weekday

{markdown_table_benign(pop['benign_by_day'])}

## Original geometry decision

The universal Mon–Wed / Thu / Fri seven-fold geometry is **STRUCTURALLY INFEASIBLE**.

Only the three Friday families — BOT, DDOS and PORT_SCAN — have positive target support
on Friday. DOS appears only Wednesday, AUTH_BRUTE_FORCE only Tuesday, and
INFILTRATION/WEB_ATTACK only Thursday.

The original geometry is therefore rejected **before model fitting**. This is a
feasibility correction, not result-driven adaptation.

## Day-atomic chronology-first feasibility

Stage27-0A applies the following feasibility rule:

```text
TRAIN < VALIDATION < TARGET
```

using weekday-atomic partitions. An executable fold additionally requires:

- zero held-out-family positives in TRAIN;
- zero held-out-family positives in VALIDATION;
- benign and at least one known-family attack positive in TRAIN;
- benign and at least one known-family attack positive in VALIDATION;
- same-target-day benign support for the primary isolation target.

{markdown_table_feasibility(feasibility)}

### Structurally ineligible

- **AUTH_BRUTE_FORCE** — first appears Tuesday. There are not two earlier weekday
  partitions for separate day-atomic TRAIN and VALIDATION, and Monday has no attack
  positives.
- **DOS** — first appears Wednesday. The only day-atomic split is Monday TRAIN /
  Tuesday VALIDATION / Wednesday TARGET, but Monday contains no known attack positives,
  so a supervised binary IDS cannot be trained honestly under that geometry.

These families are retained in the taxonomy and marked
`STRUCTURALLY_INELIGIBLE`; they are not silently dropped.

### Eligible but descriptive-only

- **INFILTRATION** has only **36** positives, below the preregistered recommended
  inferential support threshold of 50. It remains eligible for execution but must be
  marked `DESCRIPTIVE_ONLY`.

## Important semantic limit for Stage27-0

Because chronology is primary, some non-held-out attack families may occur only on or
after a fold's target day and therefore may also be absent from training.

Stage27-0 should therefore describe the executable design precisely as a:

> **chronology-first zero-training-exposure family audit**

and should not imply that every non-held-out family is necessarily represented in
training for every fold.

## Stage27-0A result

```text
Eligible folds:             5
Structurally ineligible:    2
Descriptive-only eligible:  1
Stage27 model fits:         0
Stage27 model inference:    0
GPU execution:              NO
```

Eligible families:

```text
BOT
DDOS
INFILTRATION   [DESCRIPTIVE_ONLY]
PORT_SCAN
WEB_ATTACK
```

Structurally ineligible families:

```text
AUTH_BRUTE_FORCE
DOS
```

## Next gate

Stage27-0 protocol construction is authorized **only after this Stage27-0A artifact
set is committed and the remote `main` SHA is verified**.

This script performs that commit/push/remote verification and then stops. It does not
construct Stage27-0.
"""
    (outdir / "feasibility_audit.md").write_text(audit_md, encoding="utf-8")

    produced = sorted(p.name for p in outdir.iterdir() if p.is_file())
    expected = sorted(REQUIRED_OUTPUTS)

    if produced != expected:
        raise RuntimeError(
            "Stage27-0A produced an unexpected file set.\n"
            f"expected={expected}\nactual={produced}"
        )

    output_paths = [outdir / name for name in REQUIRED_OUTPUTS]

    print("Produced exactly six required files:")
    for path in output_paths:
        print(
            f"  {path.relative_to(repo)}  "
            f"bytes={path.stat().st_size:,}  sha256={sha256_file(path)}"
        )

    print()
    print("[PASS] Exactly six Stage27-0A artifacts written.")
    return output_paths


# =============================================================================
# 10. FINAL LOCAL VALIDATION
# =============================================================================

def validate_outputs(
    repo: Path,
    output_paths: List[Path],
    matrix: OrderedDict,
    pop: Dict[str, Any],
    feasibility: Dict[str, Any],
) -> None:
    banner("STAGE27-0A :: FINAL LOCAL VALIDATION")

    # CSV readback.
    family_csv = repo / OUTPUT_REL / "family_day_support.csv"
    with family_csv.open("r", encoding="utf-8", newline="") as f:
        rows = list(csv.DictReader(f))

    if len(rows) != 7:
        raise RuntimeError(f"family_day_support.csv must contain 7 data rows, got {len(rows)}")

    readback_total = sum(int(r["Total"]) for r in rows)
    if readback_total != 557_635:
        raise RuntimeError(
            f"family_day_support.csv seven-family total mismatch: {readback_total}"
        )

    benign_csv = repo / OUTPUT_REL / "benign_day_support.csv"
    with benign_csv.open("r", encoding="utf-8", newline="") as f:
        benign_rows = list(csv.DictReader(f))

    total_rows = [r for r in benign_rows if r["day"] == "TOTAL"]
    if len(total_rows) != 1 or int(total_rows[0]["benign_support"]) != 2_273_097:
        raise RuntimeError("benign_day_support.csv total mismatch.")

    # JSON readback.
    tf = load_json(repo / OUTPUT_REL / "temporal_feasibility.json")
    eligible_count = tf[
        "replacement_day_atomic_family_specific_geometry"
    ]["eligible_fold_count"]
    ineligible_count = tf[
        "replacement_day_atomic_family_specific_geometry"
    ]["structurally_ineligible_fold_count"]

    if eligible_count != 5 or ineligible_count != 2:
        raise RuntimeError("Temporal-feasibility eligible/ineligible count mismatch.")

    # No Stage27-0 protocol lock may have been created by this script.
    stage27_0 = repo / "results/stage27_loao_unseen_attack/stage27_0_protocol_lock"
    if stage27_0.exists():
        raise RuntimeError(
            "Stage27-0 protocol directory already exists. Stage27-0A must be "
            "remotely verified before Stage27-0 construction."
        )

    # Ensure no unexpected untracked/modified files besides the six intended files.
    # IMPORTANT:
    # A fully untracked directory is collapsed by normal `git status --porcelain`
    # to a single line such as:
    #
    #     ?? results/stage27_loao_unseen_attack/
    #
    # even when exactly six files exist beneath it. Force
    # --untracked-files=all so the validator sees the individual artifact paths.
    porcelain = run(
        ["git", "status", "--porcelain=v1", "--untracked-files=all"],
        cwd=repo,
    ).splitlines()

    changed = []
    for line in porcelain:
        if not line.strip():
            continue

        # Porcelain v1 path begins after XY + space.
        path_part = line[3:] if len(line) >= 4 else line

        # Handle rename syntax defensively, although Stage27-0A should create
        # only new files.
        if " -> " in path_part:
            path_part = path_part.split(" -> ", 1)[1]

        changed.append(path_part)

    expected_files = sorted(
        f"{OUTPUT_REL}/{name}"
        for name in REQUIRED_OUTPUTS
    )
    actual_files = sorted(changed)

    if actual_files != expected_files:
        raise RuntimeError(
            "Unexpected working-tree change set after artifact generation.\n"
            f"expected={expected_files}\n"
            f"actual={actual_files}"
        )

    print("Working-tree Stage27-0A files:")
    for rel in actual_files:
        print(" ", rel)

    print("[PASS] Working-tree changes are exactly the six required Stage27-0A files.")

    print("Family rows                 : 7")
    print("Seven-family attack total  : 557,635")
    print("Benign total               : 2,273,097")
    print("Eligible folds             : 5")
    print("Structurally ineligible    : 2")
    print("Stage27-0 directory exists : NO")
    print("Model fits                 : 0")
    print("Model inference            : 0")
    print("GPU execution              : OFF")
    print("[PASS] Stage27-0A local validation complete.")


# =============================================================================
# 11. COMMIT + PUSH + REMOTE VERIFY
# =============================================================================

def commit_push_verify(
    repo: Path,
    env: Dict[str, str],
    output_paths: List[Path],
) -> str:
    banner("STAGE27-0A :: GIT COMMIT")

    rel_files = [str(p.relative_to(repo)) for p in output_paths]

    run(["git", "add", "--", *rel_files], cwd=repo)

    staged = run(["git", "diff", "--cached", "--name-only"], cwd=repo).splitlines()

    if sorted(staged) != sorted(rel_files):
        raise RuntimeError(
            "Unexpected staged file set.\n"
            f"expected={sorted(rel_files)}\n"
            f"actual={sorted(staged)}"
        )

    print("Staged exactly:")
    for rel in staged:
        print(" ", rel)

    print()
    print("Staged diff stat:")
    print(run(["git", "diff", "--cached", "--stat"], cwd=repo))

    run(["git", "commit", "-m", COMMIT_MESSAGE], cwd=repo)

    new_head = run(["git", "rev-parse", "HEAD"], cwd=repo)
    parent = run(["git", "rev-parse", "HEAD^"], cwd=repo)

    if parent != EXPECTED_PARENT:
        raise RuntimeError(
            f"New commit parent mismatch: expected {EXPECTED_PARENT}, got {parent}"
        )

    print()
    print("Commit created:", new_head)

    banner("STAGE27-0A :: PUSH MAIN")
    run(["git", "push", "origin", "main"], cwd=repo, env=env)
    print("[OK] Push completed.")

    banner("STAGE27-0A :: REMOTE VERIFICATION")
    remote_head = run(
        ["git", "ls-remote", "origin", "refs/heads/main"],
        cwd=repo,
        env=env,
    ).split()[0]

    run(["git", "fetch", "origin", "main"], cwd=repo, env=env)
    origin_main = run(["git", "rev-parse", "origin/main"], cwd=repo)
    local_head = run(["git", "rev-parse", "HEAD"], cwd=repo)

    print("Local HEAD   :", local_head)
    print("origin/main  :", origin_main)
    print("remote main  :", remote_head)

    if not (local_head == origin_main == remote_head == new_head):
        raise RuntimeError(
            "Remote verification mismatch after Stage27-0A push.\n"
            f"local={local_head}\norigin/main={origin_main}\nremote={remote_head}"
        )

    # Verify the exact six paths exist at remote-tracking main.
    remote_tree = run(
        ["git", "ls-tree", "-r", "--name-only", "origin/main", "--", OUTPUT_REL],
        cwd=repo,
    ).splitlines()

    expected_remote = sorted(
        [f"{OUTPUT_REL}/{name}" for name in REQUIRED_OUTPUTS]
    )
    actual_remote = sorted(remote_tree)

    if actual_remote != expected_remote:
        raise RuntimeError(
            "Remote Stage27-0A tree does not contain exactly the six expected files.\n"
            f"expected={expected_remote}\nactual={actual_remote}"
        )

    if not git_status_clean(repo):
        raise RuntimeError("Working tree is not clean after successful push.")

    print()
    print("[PASS] Stage27-0A commit is remotely verified.")
    print("[PASS] Remote tree contains exactly the six required Stage27-0A artifacts.")
    print("[PASS] Working tree is clean.")

    return new_head


# =============================================================================
# 12. MAIN
# =============================================================================

def main() -> None:
    # Hard-disable CUDA visibility for this CPU-only phase.
    os.environ["CUDA_VISIBLE_DEVICES"] = ""

    banner("STAGE27-0A :: ZERO-FIT FAMILY × DAY FEASIBILITY AUDIT")
    print("timestamp_utc            :", now_utc())
    print("python                   :", sys.version.replace("\n", " "))
    print("working_directory        :", Path.cwd())
    print("PRIMARY_EXECUTION_DEVICE :", PRIMARY_EXECUTION_DEVICE)
    print("PRIMARY_GPU_BUDGET_HOURS :", PRIMARY_GPU_BUDGET_HOURS)
    print("CUDA_VISIBLE_DEVICES     :", repr(os.environ.get("CUDA_VISIBLE_DEVICES")))
    print()
    print("Scientific boundary:")
    print("  model fits              : 0")
    print("  model inference         : 0")
    print("  threshold selection     : 0")
    print("  bootstrap replicates    : 0")
    print("  corpus downloads        : 0")
    print("  release corpus recreate : NO")

    token = get_kaggle_github_token()
    env, askpass = make_git_auth_env(token)

    try:
        repo = find_repo()
        ensure_repo(repo, env)

        outdir = repo / OUTPUT_REL
        banner("STAGE27-0A :: OUTPUT DIRECTORY GATE")
        print("Output directory:", outdir)

        if outdir.exists():
            existing = sorted(
                p.name
                for p in outdir.iterdir()
                if p.is_file()
            )
            subdirs = sorted(
                p.name
                for p in outdir.iterdir()
                if p.is_dir()
            )

            print("Existing files:", len(existing))
            for name in existing:
                print(" ", name)

            if subdirs:
                raise RuntimeError(
                    "Unexpected subdirectories exist inside Stage27-0A output: "
                    + repr(subdirs)
                )

            if existing:
                # Recovery path for the immediately preceding failed run:
                # exactly the six required artifacts exist and all are still
                # untracked. Nothing was committed or pushed.
                porcelain = run(
                    ["git", "status", "--porcelain=v1", "--untracked-files=all"],
                    cwd=repo,
                ).splitlines()

                actual_untracked = sorted(
                    line[3:]
                    for line in porcelain
                    if line.startswith("?? ")
                    and line[3:].startswith(OUTPUT_REL + "/")
                )

                expected_untracked = sorted(
                    f"{OUTPUT_REL}/{name}"
                    for name in REQUIRED_OUTPUTS
                )

                other_changes = [
                    line
                    for line in porcelain
                    if not (
                        line.startswith("?? ")
                        and line[3:].startswith(OUTPUT_REL + "/")
                    )
                ]

                if (
                    existing == sorted(REQUIRED_OUTPUTS)
                    and actual_untracked == expected_untracked
                    and not other_changes
                ):
                    print()
                    print(
                        "[RECOVERY] Found exactly the six untracked Stage27-0A "
                        "artifacts left by the previous validation-only failure."
                    )
                    print(
                        "[RECOVERY] No commit/push occurred and no unrelated Git "
                        "changes exist."
                    )
                    print(
                        "[RECOVERY] Removing only this uncommitted Stage27-0A "
                        "directory so it can be regenerated cleanly."
                    )
                    shutil.rmtree(outdir)
                else:
                    raise RuntimeError(
                        "Stage27-0A output directory is non-empty and does not "
                        "match the safe recovery state. Refusing to overwrite.\n"
                        f"existing={existing}\n"
                        f"expected={sorted(REQUIRED_OUTPUTS)}\n"
                        f"untracked={actual_untracked}\n"
                        f"other_changes={other_changes}"
                    )
        else:
            print("Existing entries: 0 (directory not yet created)")

        print("[PASS] Stage27-0A output is clean/recovery-safe.")

        stage26 = verify_stage26_closure(repo)
        stage24_receipts = verify_stage24_sources(repo)
        taxonomy = verify_taxonomy(repo)
        pop = recover_stage24_population(repo)
        compact_audit = audit_stage20_compact_binary_only(repo)
        matrix, support_summary = construct_support_matrix(pop)
        feasibility = derive_temporal_feasibility(
            matrix,
            pop["benign_by_day"],
        )

        output_paths = write_artifacts(
            repo=repo,
            stage26_receipt=stage26,
            stage24_receipts=stage24_receipts,
            taxonomy=taxonomy,
            pop=pop,
            compact_audit=compact_audit,
            matrix=matrix,
            support_summary=support_summary,
            feasibility=feasibility,
        )

        validate_outputs(
            repo=repo,
            output_paths=output_paths,
            matrix=matrix,
            pop=pop,
            feasibility=feasibility,
        )

        commit_sha = commit_push_verify(
            repo=repo,
            env=env,
            output_paths=output_paths,
        )

        banner("STAGE27-0A COMPLETE — REMOTELY VERIFIED")
        print("Execution parent          :", EXPECTED_PARENT)
        print("Stage27-0A commit         :", commit_sha)
        print("Remote main               :", commit_sha)
        print("Artifacts                 : 6/6")
        print("Eligible folds            : 5")
        print("Structurally ineligible   : 2")
        print("Descriptive-only eligible : INFILTRATION (n=36)")
        print("Model fits                : 0")
        print("Model inference           : 0")
        print("GPU execution             : OFF")
        print("Git working tree          : CLEAN")
        print()
        print("NEXT AUTHORIZED STEP:")
        print("  Construct Stage27-0 protocol lock from this remotely verified")
        print("  Stage27-0A feasibility correction.")
        print()
        print("STOPPING HERE. Stage27-0 is NOT created by this script.")
        print("=" * 110)

    finally:
        # Never retain auth material.
        try:
            if askpass.exists():
                askpass.unlink()
        except Exception:
            pass

        env.pop("STAGE27_GITHUB_TOKEN", None)
        os.environ.pop("STAGE27_GITHUB_TOKEN", None)

        # Best-effort overwrite local Python reference.
        token = ""


if __name__ == "__main__":
    main()



STAGE27-0A :: ZERO-FIT FAMILY × DAY FEASIBILITY AUDIT
timestamp_utc            : 2026-08-20T17:03:29.676708+00:00
python                   : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
working_directory        : /kaggle/working
PRIMARY_EXECUTION_DEVICE : CPU
PRIMARY_GPU_BUDGET_HOURS : 0
CUDA_VISIBLE_DEVICES     : ''

Scientific boundary:
  model fits              : 0
  model inference         : 0
  threshold selection     : 0
  bootstrap replicates    : 0
  corpus downloads        : 0
  release corpus recreate : NO
[FOUND] GitHub secret label: GITHUB_TOKEN (value not printed)

STAGE27-0A :: REPOSITORY BOOTSTRAP
Repository : /kaggle/working/ids2018-validation-safe-ablation
Remote     : https://github.com/themubasshir/ids2018-validation-safe-ablation.git


RuntimeError: Git working tree is not clean. Stage27-0A refuses to overwrite or co-mingle unrelated work.

In [8]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Stage27-0A — Family × Day Feasibility Audit
===========================================

Repository:
    themubasshir/ids2018-validation-safe-ablation

Scientific boundary:
    - ZERO model fits
    - ZERO model inference
    - ZERO threshold selection
    - ZERO bootstrap
    - ZERO GPU use
    - writes ONLY the six required Stage27-0A artifacts
    - commits + pushes Stage27-0A to main
    - remotely verifies the resulting commit
    - STOPS before Stage27-0 protocol construction

This script deliberately uses the full Stage24 CICIDS2017 effective target
population for Stage27 family support. The Stage20 compact raw-byte release
corpora are audited as binary-only corpora and therefore are not used as
seven-family support sources.

Expected execution parent:
    e47f44751bc71d219c5d0f3b3fca06d62037fb8b
"""

from __future__ import annotations

import csv
import hashlib
import json
import os
import shutil
import stat
import subprocess
import sys
from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, List, Tuple


# =============================================================================
# 0. FROZEN CONSTANTS
# =============================================================================

REPO_OWNER = "themubasshir"
REPO_NAME = "ids2018-validation-safe-ablation"
REPO_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"

EXPECTED_PARENT = "e47f44751bc71d219c5d0f3b3fca06d62037fb8b"
PRIMARY_EXECUTION_DEVICE = "CPU"
PRIMARY_GPU_BUDGET_HOURS = 0

WEEKDAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]

PRIMARY_FAMILIES = [
    "BOT",
    "DDOS",
    "DOS",
    "AUTH_BRUTE_FORCE",
    "INFILTRATION",
    "PORT_SCAN",
    "WEB_ATTACK",
]

EXPECTED_STAGE24_SOURCE_SHA256 = OrderedDict(
    [
        (
            "results/stage24_cross_dataset/stage24_0_protocol_lock/"
            "stage24_0c_final_preopening_protocol_lock.json",
            "8ef234a9d283f2008f21b9add4361f14328d1f3c1cffa278077f59d9eb9e37c2",
        ),
        (
            "results/stage24_cross_dataset/stage24_0_protocol_lock/"
            "stage24_0b2_complete_cicids2017_source_contract.json",
            "96f7fc5c0227660fe8ec17f5173630ee2ac71f6be199e81ee37dac0ad25d9779",
        ),
        (
            "results/stage24_cross_dataset/stage24_0_protocol_lock/"
            "stage24_0b_exact_cicids2017_source_recovery.json",
            "8a84f1dccc51f224427e00144088058590f48c8b603cfd798f6ba431d659da1c",
        ),
        (
            "results/stage24_cross_dataset/stage24_2_primary_target_openings/"
            "stage24_2a_bridge62_published/stage24_2a_bridge62_published_result.json",
            "1d4b3010fdc8b0f7f62a0fed2c373824ce99cb2f16936c021756749a9ca54749",
        ),
    ]
)

STAGE24_PROTOCOL_LOCK_REL = (
    "results/stage24_cross_dataset/stage24_0_protocol_lock/"
    "stage24_0c_final_preopening_protocol_lock.json"
)
STAGE24_SOURCE_CONTRACT_REL = (
    "results/stage24_cross_dataset/stage24_0_protocol_lock/"
    "stage24_0b2_complete_cicids2017_source_contract.json"
)
STAGE24_SOURCE_RECOVERY_REL = (
    "results/stage24_cross_dataset/stage24_0_protocol_lock/"
    "stage24_0b_exact_cicids2017_source_recovery.json"
)
STAGE24_TARGET_RESULT_REL = (
    "results/stage24_cross_dataset/stage24_2_primary_target_openings/"
    "stage24_2a_bridge62_published/stage24_2a_bridge62_published_result.json"
)

STAGE26_CLOSURE_CANDIDATES = [
    (
        "results/stage26_deployment_profiling/stage26_12_final_synthesis/"
        "stage26_g2_final_stage26_closure_receipt.json"
    ),
    (
        "results/stage26_deployment_profiling/stage26_g2_final_gpu_publication_closure/"
        "stage26_g2_final_stage26_closure_receipt.json"
    ),
]

# Stage20 compact manifests are checked only to prove that the frozen raw-byte
# release corpus contains binary labels, not seven-family labels.
STAGE20_COMPACT_MANIFESTS = OrderedDict(
    [
        (
            "Monday",
            "results/stage20_1e_training/"
            "stage20_1e1m_monday_compact_corpus_manifest.json",
        ),
        (
            "Tuesday",
            "results/stage20_1e_training/"
            "stage20_1e1t_tuesday_compact_corpus_manifest.json",
        ),
        (
            "Wednesday",
            "results/stage20_1e_training/"
            "stage20_1e1w_wednesday_compact_corpus_manifest.json",
        ),
        (
            "Thursday",
            "results/stage20_1e_training/"
            "stage20_1e1v_thursday_validation_compact_corpus_manifest.json",
        ),
        (
            "Friday",
            "results/stage20_1e_training/"
            "stage20_1e4_friday_holdout_compact_corpus_manifest.json",
        ),
    ]
)

OUTPUT_REL = (
    "results/stage27_loao_unseen_attack/"
    "stage27_0a_family_day_feasibility"
)

REQUIRED_OUTPUTS = [
    "family_day_support.csv",
    "benign_day_support.csv",
    "taxonomy_receipt.json",
    "source_artifact_receipts.json",
    "temporal_feasibility.json",
    "feasibility_audit.md",
]

# Frozen Stage24 taxonomy.
RAW_TO_FAMILY = OrderedDict(
    [
        ("bot", "BOT"),
        ("ddos", "DDOS"),
        ("dos goldeneye", "DOS"),
        ("dos hulk", "DOS"),
        ("dos slowhttptest", "DOS"),
        ("dos slowloris", "DOS"),
        ("ftp-patator", "AUTH_BRUTE_FORCE"),
        ("heartbleed", "TARGET_ONLY_UNSEEN"),
        ("infiltration", "INFILTRATION"),
        ("portscan", "PORT_SCAN"),
        ("ssh-patator", "AUTH_BRUTE_FORCE"),
        ("web attack - brute force", "WEB_ATTACK"),
        ("web attack - sql injection", "WEB_ATTACK"),
        ("web attack - xss", "WEB_ATTACK"),
    ]
)

# Explicit counts recovered from the durable Stage24 target population.
EXPECTED_CANONICAL_LABEL_COUNTS = OrderedDict(
    [
        ("benign", 2_273_097),
        ("bot", 1_966),
        ("ddos", 128_027),
        ("dos goldeneye", 10_293),
        ("dos hulk", 231_073),
        ("dos slowhttptest", 5_499),
        ("dos slowloris", 5_796),
        ("ftp-patator", 7_938),
        ("heartbleed", 11),
        ("infiltration", 36),
        ("portscan", 158_930),
        ("ssh-patator", 5_897),
        ("web attack - brute force", 1_507),
        ("web attack - sql injection", 21),
        ("web attack - xss", 652),
    ]
)

MIN_INFERENTIAL_POSITIVE_SUPPORT = 50

COMMIT_MESSAGE = "stage27-0a: freeze family-day feasibility audit"


# =============================================================================
# 1. GENERIC HELPERS
# =============================================================================

def now_utc() -> str:
    return datetime.now(timezone.utc).isoformat()


def banner(title: str, width: int = 110) -> None:
    print()
    print("=" * width)
    print(title)
    print("=" * width)


def run(
    args: List[str],
    *,
    cwd: Path | None = None,
    env: Dict[str, str] | None = None,
    check: bool = True,
    capture: bool = True,
) -> str:
    proc = subprocess.run(
        args,
        cwd=str(cwd) if cwd else None,
        env=env,
        text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.PIPE if capture else None,
    )
    if check and proc.returncode != 0:
        cmd = " ".join(args)
        out = proc.stdout or ""
        err = proc.stderr or ""
        raise RuntimeError(
            f"Command failed ({proc.returncode}): {cmd}\n\nSTDOUT:\n{out}\n\nSTDERR:\n{err}"
        )
    return (proc.stdout or "").strip()


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def write_json(path: Path, obj: Any) -> None:
    payload = json.dumps(
        obj,
        indent=2,
        ensure_ascii=False,
        sort_keys=False,
    ) + "\n"
    path.write_text(payload, encoding="utf-8")


def normalize_label(value: Any) -> str:
    s = str(value).strip().lower()
    # CICIDS2017 Web Attack strings sometimes carry cp1252/en-dash artifacts.
    for ch in ["\u0096", "\u2013", "\u2014", "–", "—"]:
        s = s.replace(ch, "-")
    s = " ".join(s.split())
    return s


def walk_nodes(obj: Any, path: str = "$") -> Iterable[Tuple[str, Any]]:
    yield path, obj
    if isinstance(obj, dict):
        for k, v in obj.items():
            yield from walk_nodes(v, f"{path}.{k}")
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            yield from walk_nodes(v, f"{path}[{i}]")


def contains_pair_anywhere(obj: Any, raw_label: str, family: str) -> bool:
    raw_norm = normalize_label(raw_label)
    fam_norm = normalize_label(family)
    for _, node in walk_nodes(obj):
        if not isinstance(node, dict):
            continue
        for k, v in node.items():
            if normalize_label(k) == raw_norm and normalize_label(v) == fam_norm:
                return True
    return False


def find_repo() -> Path:
    candidates = [
        Path("/kaggle/working") / REPO_NAME,
        Path.cwd() / REPO_NAME,
        Path.cwd(),
    ]
    for p in candidates:
        if (p / ".git").exists() and p.name == REPO_NAME:
            return p.resolve()
    return (Path("/kaggle/working") / REPO_NAME).resolve()


def git_status_clean(repo: Path) -> bool:
    return run(["git", "status", "--porcelain"], cwd=repo) == ""


# =============================================================================
# 2. TEMPORARY GITHUB AUTH
# =============================================================================

def get_kaggle_github_token() -> str:
    try:
        from kaggle_secrets import UserSecretsClient  # type: ignore
    except Exception as exc:
        raise RuntimeError(
            "kaggle_secrets is unavailable. Run this script in Kaggle or provide "
            "authenticated Git credentials another way."
        ) from exc

    client = UserSecretsClient()
    labels = ["GITHUB_TOKEN", "github_token", "GH_TOKEN", "github_pat", "GITHUB_PAT"]

    for label in labels:
        try:
            token = client.get_secret(label)
        except Exception:
            token = None
        if token:
            token = str(token).strip()
            if token:
                print(f"[FOUND] GitHub secret label: {label} (value not printed)")
                return token

    raise RuntimeError(
        "No usable GitHub token found in Kaggle Secrets. Expected one of: "
        + ", ".join(labels)
    )


def make_git_auth_env(token: str) -> Tuple[Dict[str, str], Path]:
    askpass = Path("/kaggle/working/.stage27_git_askpass.sh")
    askpass.write_text(
        "#!/bin/sh\n"
        'case "$1" in\n'
        '  *Username*) printf "%s\\n" "x-access-token" ;;\n'
        '  *Password*) printf "%s\\n" "$STAGE27_GITHUB_TOKEN" ;;\n'
        '  *) printf "%s\\n" "" ;;\n'
        "esac\n",
        encoding="utf-8",
    )
    askpass.chmod(stat.S_IRUSR | stat.S_IWUSR | stat.S_IXUSR)

    env = os.environ.copy()
    env["GIT_ASKPASS"] = str(askpass)
    env["GIT_TERMINAL_PROMPT"] = "0"
    env["STAGE27_GITHUB_TOKEN"] = token
    return env, askpass


# =============================================================================
# 3. REPOSITORY BOOTSTRAP + HARD GATES
# =============================================================================

def ensure_repo(repo: Path, env: Dict[str, str]) -> None:
    banner("STAGE27-0A :: REPOSITORY BOOTSTRAP")

    if not (repo / ".git").exists():
        repo.parent.mkdir(parents=True, exist_ok=True)
        print("Repository not present; cloning origin/main...")
        run(
            ["git", "clone", "--branch", "main", "--single-branch", REPO_URL, str(repo)],
            cwd=repo.parent,
            env=env,
        )

    remote_url = run(["git", "remote", "get-url", "origin"], cwd=repo)
    print("Repository :", repo)
    print("Remote     :", remote_url)

    # Normally Stage27-0A requires a clean starting tree.
    #
    # Safe recovery exception:
    # The immediately preceding run may have generated exactly the six required
    # Stage27-0A files and then failed during validation before staging/commit.
    # In that one narrowly-defined case, permit bootstrap to continue so the
    # later OUTPUT DIRECTORY GATE can remove and regenerate only those files.
    porcelain_before = run(
        ["git", "status", "--porcelain=v1", "--untracked-files=all"],
        cwd=repo,
    ).splitlines()

    if porcelain_before:
        expected_safe_untracked = sorted(
            f"?? {OUTPUT_REL}/{name}"
            for name in REQUIRED_OUTPUTS
        )
        actual_status = sorted(
            line for line in porcelain_before if line.strip()
        )

        if actual_status == expected_safe_untracked:
            print(
                "[RECOVERY] Working tree contains exactly the six expected "
                "untracked Stage27-0A artifacts from the prior failed run."
            )
            print(
                "[RECOVERY] No tracked modifications or unrelated untracked "
                "files are present; bootstrap may continue."
            )
        else:
            raise RuntimeError(
                "Git working tree is dirty outside the exact safe Stage27-0A "
                "recovery state. Refusing to overwrite or co-mingle work.\n\n"
                "Observed status:\n"
                + "\n".join(actual_status)
            )

    run(["git", "fetch", "origin", "main"], cwd=repo, env=env)

    branch = run(["git", "branch", "--show-current"], cwd=repo)
    head = run(["git", "rev-parse", "HEAD"], cwd=repo)
    origin_main = run(["git", "rev-parse", "origin/main"], cwd=repo)
    remote_main = run(
        ["git", "ls-remote", "origin", "refs/heads/main"],
        cwd=repo,
        env=env,
    ).split()[0]

    banner("STAGE27-0A :: GIT GATE")
    print("Branch          :", branch)
    print("Expected parent :", EXPECTED_PARENT)
    print("Local HEAD      :", head)
    print("origin/main     :", origin_main)
    print("remote main     :", remote_main)
    clean_now = git_status_clean(repo)
    print("Repo clean      :", clean_now)

    if branch != "main":
        raise RuntimeError(f"Expected branch 'main', found {branch!r}")

    if not (head == origin_main == remote_main == EXPECTED_PARENT):
        raise RuntimeError(
            "Pinned execution parent mismatch.\n"
            f"expected    : {EXPECTED_PARENT}\n"
            f"local HEAD  : {head}\n"
            f"origin/main : {origin_main}\n"
            f"remote main : {remote_main}\n\n"
            "Do not rebase or silently continue. Inspect the new remote state first."
        )

    if clean_now:
        print("[PASS] Exact Stage27 execution parent is pinned and clean.")
    else:
        # Re-validate that the only dirt is the exact six safe untracked files.
        safe_status = sorted(
            line
            for line in run(
                ["git", "status", "--porcelain=v1", "--untracked-files=all"],
                cwd=repo,
            ).splitlines()
            if line.strip()
        )
        expected_safe_status = sorted(
            f"?? {OUTPUT_REL}/{name}"
            for name in REQUIRED_OUTPUTS
        )

        if safe_status != expected_safe_status:
            raise RuntimeError(
                "Repository became dirty outside the exact safe recovery state "
                "during bootstrap.\nObserved:\n"
                + "\n".join(safe_status)
            )

        print(
            "[PASS] Exact Stage27 execution parent is pinned; the only dirty "
            "state is the six safe untracked recovery artifacts."
        )


# =============================================================================
# 4. PRIOR-STAGE CLOSURE + SOURCE HASH GATES
# =============================================================================

def verify_stage26_closure(repo: Path) -> Dict[str, Any]:
    banner("STAGE27-0A :: VERIFY STAGE26 CLOSURE")

    hits = []
    for rel in STAGE26_CLOSURE_CANDIDATES:
        path = repo / rel
        if not path.exists():
            continue
        data = load_json(path)
        if data.get("stage26_status") == "COMPLETE":
            hits.append(
                {
                    "path": rel,
                    "sha256": sha256_file(path),
                    "bytes": path.stat().st_size,
                    "stage26_status": "COMPLETE",
                }
            )
            print("COMPLETE:", rel)

    if not hits:
        raise RuntimeError("No durable Stage26 stage26_status=COMPLETE receipt found.")

    print("[PASS] Stage26 closure verified.")
    return {"verified": True, "evidence": hits}


def verify_stage24_sources(repo: Path) -> List[Dict[str, Any]]:
    banner("STAGE27-0A :: VERIFY DURABLE STAGE24 SOURCES")

    receipts: List[Dict[str, Any]] = []

    for rel, expected_sha in EXPECTED_STAGE24_SOURCE_SHA256.items():
        path = repo / rel
        if not path.exists():
            raise FileNotFoundError(f"Missing required Stage24 artifact: {rel}")

        actual_sha = sha256_file(path)
        size = path.stat().st_size

        print()
        print("Artifact:", rel)
        print(" bytes   :", size)
        print(" expected:", expected_sha)
        print(" actual  :", actual_sha)

        if actual_sha != expected_sha:
            raise RuntimeError(
                f"SHA256 mismatch for {rel}\n"
                f"expected: {expected_sha}\n"
                f"actual  : {actual_sha}"
            )

        receipts.append(
            {
                "path": rel,
                "bytes": size,
                "sha256": actual_sha,
                "sha256_matches_frozen_expected": True,
            }
        )

    print()
    print("[PASS] All four Stage24 durable artifacts match frozen SHA256 values.")
    return receipts


# =============================================================================
# 5. TAXONOMY + STAGE24 EFFECTIVE POPULATION RECOVERY
# =============================================================================

def verify_taxonomy(repo: Path) -> Dict[str, Any]:
    banner("STAGE27-0A :: VERIFY FROZEN STAGE24 TAXONOMY")

    protocol = load_json(repo / STAGE24_PROTOCOL_LOCK_REL)

    missing_pairs = []
    for raw, fam in RAW_TO_FAMILY.items():
        ok = contains_pair_anywhere(protocol, raw, fam)
        print(f"{raw:<34} -> {fam:<20} {'[OK]' if ok else '[MISSING]'}")
        if not ok:
            missing_pairs.append((raw, fam))

    if missing_pairs:
        raise RuntimeError(
            "Frozen Stage24 taxonomy pairs were not all found in the protocol lock: "
            + repr(missing_pairs)
        )

    recovered_primary = sorted(
        {fam for fam in RAW_TO_FAMILY.values() if fam in PRIMARY_FAMILIES}
    )
    if recovered_primary != sorted(PRIMARY_FAMILIES):
        raise RuntimeError(
            f"Primary taxonomy mismatch: {recovered_primary} vs {sorted(PRIMARY_FAMILIES)}"
        )

    print()
    print("[PASS] Exact seven-family Stage24 taxonomy recovered.")

    return {
        "primary_families": PRIMARY_FAMILIES,
        "raw_label_to_stage27_family": RAW_TO_FAMILY,
        "excluded_from_primary_seven_family_taxonomy": {
            "heartbleed": "TARGET_ONLY_UNSEEN"
        },
    }


def recover_stage24_population(repo: Path) -> Dict[str, Any]:
    banner("STAGE27-0A :: RECOVER STAGE24 EFFECTIVE CICIDS2017 POPULATION")

    result = load_json(repo / STAGE24_TARGET_RESULT_REL)

    try:
        file_level = result["metrics"]["file_level_descriptive"]
        label_counts_raw = result["target_population"]["canonical_label_counts"]
    except Exception as exc:
        raise RuntimeError(
            "Expected Stage24 population structures were not found at "
            "$.metrics.file_level_descriptive and "
            "$.target_population.canonical_label_counts."
        ) from exc

    label_counts = {
        normalize_label(k): int(v)
        for k, v in label_counts_raw.items()
    }

    # Exact canonical label-count gate.
    for label, expected in EXPECTED_CANONICAL_LABEL_COUNTS.items():
        actual = label_counts.get(label)
        if actual != expected:
            raise RuntimeError(
                f"Canonical Stage24 count mismatch for {label!r}: "
                f"expected {expected}, actual {actual}"
            )

    # Day/file support gate.
    expected_files = [
        ("Monday", 529_918, 529_918, 0),
        ("Tuesday", 445_909, 432_074, 13_835),
        ("Wednesday", 692_703, 440_031, 252_672),
        ("Thursday", 288_602, 288_566, 36),
        ("Thursday", 170_366, 168_186, 2_180),
        ("Friday", 225_745, 97_718, 128_027),
        ("Friday", 286_467, 127_537, 158_930),
        ("Friday", 191_033, 189_067, 1_966),
    ]

    if len(file_level) != len(expected_files):
        raise RuntimeError(
            f"Expected {len(expected_files)} Stage24 file-level rows; found {len(file_level)}"
        )

    normalized_rows = []
    for idx, row in enumerate(file_level):
        day = str(row["day"])
        rows = int(row["rows"])
        benign = int(row["benign"])
        attack = int(row["attack"])
        remote = str(row["remote"])

        exp_day, exp_rows, exp_benign, exp_attack = expected_files[idx]
        if (day, rows, benign, attack) != (
            exp_day,
            exp_rows,
            exp_benign,
            exp_attack,
        ):
            raise RuntimeError(
                "Stage24 file-level support mismatch at index "
                f"{idx}: got {(day, rows, benign, attack)}, expected "
                f"{(exp_day, exp_rows, exp_benign, exp_attack)}"
            )

        normalized_rows.append(
            {
                "file_id": int(row.get("file_id", idx)),
                "day": day,
                "remote": remote,
                "rows": rows,
                "benign": benign,
                "attack": attack,
                "start": int(row["start"]),
                "stop": int(row["stop"]),
            }
        )

    benign_by_day = OrderedDict((day, 0) for day in WEEKDAYS)
    rows_by_day = OrderedDict((day, 0) for day in WEEKDAYS)
    all_attack_by_day = OrderedDict((day, 0) for day in WEEKDAYS)

    for row in normalized_rows:
        day = row["day"]
        benign_by_day[day] += row["benign"]
        rows_by_day[day] += row["rows"]
        all_attack_by_day[day] += row["attack"]

    expected_benign = OrderedDict(
        [
            ("Monday", 529_918),
            ("Tuesday", 432_074),
            ("Wednesday", 440_031),
            ("Thursday", 456_752),
            ("Friday", 414_322),
        ]
    )
    if benign_by_day != expected_benign:
        raise RuntimeError(
            f"Benign weekday support mismatch:\nactual={benign_by_day}\n"
            f"expected={expected_benign}"
        )

    if sum(benign_by_day.values()) != 2_273_097:
        raise RuntimeError("Total benign support mismatch.")

    if sum(all_attack_by_day.values()) != 557_646:
        raise RuntimeError("Total non-benign support mismatch.")

    print()
    for row in normalized_rows:
        print(
            f"{row['day']:<10} rows={row['rows']:>7,} "
            f"benign={row['benign']:>7,} attack={row['attack']:>7,}  "
            f"{row['remote']}"
        )

    print()
    print("[PASS] Stage24 effective CICIDS2017 population is exact.")

    return {
        "file_level": normalized_rows,
        "label_counts": label_counts,
        "benign_by_day": benign_by_day,
        "rows_by_day": rows_by_day,
        "all_attack_by_day": all_attack_by_day,
    }


# =============================================================================
# 6. STAGE20 COMPACT CORPUS LIMITATION AUDIT
# =============================================================================

def audit_stage20_compact_binary_only(repo: Path) -> Dict[str, Any]:
    banner("STAGE27-0A :: AUDIT STAGE20 COMPACT RELEASE SEMANTICS")

    manifest_receipts = []
    all_binary_only = True

    for day, rel in STAGE20_COMPACT_MANIFESTS.items():
        path = repo / rel
        if not path.exists():
            raise FileNotFoundError(f"Missing Stage20 compact manifest: {rel}")

        data = load_json(path)
        compact = data.get("compact_corpus", {})
        files = compact.get("files", {})

        keys = sorted(files.keys())
        expected_keys = sorted(
            [
                "encoded_bytes.bin",
                "flow_offsets.npy",
                "labels.npy",
                "packet_lengths.npy",
            ]
        )

        labels_file_present = "labels.npy" in files
        family_label_file_present = any(
            (
                "family" in k.lower()
                or "attack_type" in k.lower()
                or "multiclass" in k.lower()
            )
            for k in files.keys()
        )

        exact_join = data.get("exact_join", {})
        binary_counts = exact_join.get("matched_binary_counts", {})
        binary_key_set = {str(k) for k in binary_counts.keys()}
        binary_only_counts = binary_key_set.issubset({"0", "1"}) and bool(binary_counts)

        schema_ok = (
            keys == expected_keys
            and labels_file_present
            and not family_label_file_present
            and binary_only_counts
        )

        all_binary_only = all_binary_only and schema_ok

        print()
        print(day)
        print("  manifest:", rel)
        print("  files   :", keys)
        print("  matched_binary_counts:", binary_counts)
        print("  family-label array present:", family_label_file_present)
        print("  binary-only schema gate:", schema_ok)

        manifest_receipts.append(
            {
                "day": day,
                "path": rel,
                "bytes": path.stat().st_size,
                "sha256": sha256_file(path),
                "compact_files": keys,
                "flow_count": int(compact.get("flow_count", 0)),
                "matched_binary_counts": {
                    str(k): int(v) for k, v in binary_counts.items()
                },
                "family_label_array_present": family_label_file_present,
                "binary_only_for_stage27_family_identity": schema_ok,
            }
        )

    if not all_binary_only:
        raise RuntimeError(
            "Stage20 compact-corpus binary-only audit did not pass consistently."
        )

    print()
    print(
        "[PASS] All five Stage20 compact manifests persist binary labels only; "
        "they cannot independently certify seven-family LOAO membership."
    )

    return {
        "all_five_compact_manifests_binary_only": True,
        "stage27_family_support_source": "STAGE24_FULL_CICIDS2017_EFFECTIVE_POPULATION",
        "stage20_compact_release_role": (
            "VALID_FOR_STAGE20_BINARY_RAW_BYTE_EXPERIMENTS_BUT_NOT_A_DURABLE_"
            "SEVEN_FAMILY_MEMBERSHIP_SOURCE"
        ),
        "manifests": manifest_receipts,
    }


# =============================================================================
# 7. SUPPORT MATRIX CONSTRUCTION
# =============================================================================

def construct_support_matrix(pop: Dict[str, Any]) -> Tuple[OrderedDict, OrderedDict]:
    banner("STAGE27-0A :: CONSTRUCT SEVEN-FAMILY × WEEKDAY SUPPORT MATRIX")

    c = pop["label_counts"]

    dos_total = (
        c["dos goldeneye"]
        + c["dos hulk"]
        + c["dos slowhttptest"]
        + c["dos slowloris"]
    )
    auth_total = c["ftp-patator"] + c["ssh-patator"]
    web_total = (
        c["web attack - brute force"]
        + c["web attack - sql injection"]
        + c["web attack - xss"]
    )

    family_totals = OrderedDict(
        [
            ("BOT", c["bot"]),
            ("DDOS", c["ddos"]),
            ("DOS", dos_total),
            ("AUTH_BRUTE_FORCE", auth_total),
            ("INFILTRATION", c["infiltration"]),
            ("PORT_SCAN", c["portscan"]),
            ("WEB_ATTACK", web_total),
        ]
    )

    expected_totals = OrderedDict(
        [
            ("BOT", 1_966),
            ("DDOS", 128_027),
            ("DOS", 252_661),
            ("AUTH_BRUTE_FORCE", 13_835),
            ("INFILTRATION", 36),
            ("PORT_SCAN", 158_930),
            ("WEB_ATTACK", 2_180),
        ]
    )
    if family_totals != expected_totals:
        raise RuntimeError(
            f"Family arithmetic mismatch:\nactual={family_totals}\nexpected={expected_totals}"
        )

    matrix = OrderedDict()
    for family in PRIMARY_FAMILIES:
        matrix[family] = OrderedDict((day, 0) for day in WEEKDAYS)

    # CICIDS2017 attack schedule recovered from the exact Stage24 file-level
    # target population:
    matrix["AUTH_BRUTE_FORCE"]["Tuesday"] = auth_total
    matrix["DOS"]["Wednesday"] = dos_total
    matrix["INFILTRATION"]["Thursday"] = c["infiltration"]
    matrix["WEB_ATTACK"]["Thursday"] = web_total
    matrix["DDOS"]["Friday"] = c["ddos"]
    matrix["PORT_SCAN"]["Friday"] = c["portscan"]
    matrix["BOT"]["Friday"] = c["bot"]

    # Cross-check each day against durable Stage24 file-level attack support.
    # Wednesday carries an additional 11 Heartbleed rows excluded from the
    # seven primary families.
    expected_day_attack = OrderedDict(
        [
            ("Monday", 0),
            ("Tuesday", 13_835),
            ("Wednesday", 252_672),
            ("Thursday", 2_216),
            ("Friday", 288_923),
        ]
    )

    primary_day_attack = OrderedDict(
        (
            day,
            sum(matrix[fam][day] for fam in PRIMARY_FAMILIES),
        )
        for day in WEEKDAYS
    )

    heartbleed_by_day = OrderedDict((day, 0) for day in WEEKDAYS)
    heartbleed_by_day["Wednesday"] = c["heartbleed"]

    reconstructed_all_attack = OrderedDict(
        (
            day,
            primary_day_attack[day] + heartbleed_by_day[day],
        )
        for day in WEEKDAYS
    )

    if reconstructed_all_attack != expected_day_attack:
        raise RuntimeError(
            "Reconstructed day attack totals do not match Stage24 file-level evidence.\n"
            f"reconstructed={reconstructed_all_attack}\nexpected={expected_day_attack}"
        )

    if expected_day_attack != pop["all_attack_by_day"]:
        raise RuntimeError(
            "Stage24 file-level attack support differs from expected reconstructed support."
        )

    seven_family_total = sum(family_totals.values())
    all_nonbenign = seven_family_total + c["heartbleed"]

    if seven_family_total != 557_635:
        raise RuntimeError("Seven-family total must equal 557,635.")
    if all_nonbenign != 557_646:
        raise RuntimeError("All non-benign total must equal 557,646.")

    for fam in PRIMARY_FAMILIES:
        vals = matrix[fam]
        print(
            f"{fam:<20} "
            + " ".join(f"{day[:3]}={vals[day]:>7,}" for day in WEEKDAYS)
            + f"  total={sum(vals.values()):>7,}"
        )

    print()
    print("Seven-family attack total :", f"{seven_family_total:,}")
    print("Heartbleed excluded       :", f"{c['heartbleed']:,}")
    print("All non-benign total      :", f"{all_nonbenign:,}")
    print("[PASS] Family/day support arithmetic is exact.")

    summary = OrderedDict(
        [
            ("matrix", matrix),
            ("family_totals", family_totals),
            ("primary_day_attack", primary_day_attack),
            ("heartbleed_by_day", heartbleed_by_day),
            ("all_attack_by_day", reconstructed_all_attack),
            ("seven_family_attack_total", seven_family_total),
            ("heartbleed_target_only_unseen", c["heartbleed"]),
            ("all_nonbenign_total", all_nonbenign),
        ]
    )

    return matrix, summary


# =============================================================================
# 8. TEMPORAL FEASIBILITY
# =============================================================================

def first_positive_day(matrix: OrderedDict, family: str) -> str:
    positive = [day for day in WEEKDAYS if matrix[family][day] > 0]
    if len(positive) != 1:
        raise RuntimeError(
            f"Expected exactly one positive weekday for {family}, found {positive}"
        )
    return positive[0]


def sum_family_support(
    matrix: OrderedDict,
    days: List[str],
    *,
    exclude_family: str | None = None,
) -> OrderedDict:
    out = OrderedDict()
    for fam in PRIMARY_FAMILIES:
        if exclude_family is not None and fam == exclude_family:
            continue
        out[fam] = sum(matrix[fam][d] for d in days)
    return out


def derive_temporal_feasibility(
    matrix: OrderedDict,
    benign_by_day: OrderedDict,
) -> Dict[str, Any]:
    banner("STAGE27-0A :: TEMPORAL FEASIBILITY")

    # Original proposal:
    original_train = ["Monday", "Tuesday", "Wednesday"]
    original_validation = ["Thursday"]
    original_target = ["Friday"]

    family_records = OrderedDict()

    for family in PRIMARY_FAMILIES:
        target_day = first_positive_day(matrix, family)
        target_idx = WEEKDAYS.index(target_day)

        heldout_support = matrix[family][target_day]
        support_status = (
            "INFERENTIAL_ELIGIBLE_SUPPORT"
            if heldout_support >= MIN_INFERENTIAL_POSITIVE_SUPPORT
            else "DESCRIPTIVE_ONLY_SUPPORT_LT_50"
        )

        original_target_support = sum(matrix[family][d] for d in original_target)
        original_train_support = sum(matrix[family][d] for d in original_train)
        original_val_support = sum(matrix[family][d] for d in original_validation)

        original_family_geometry_valid = (
            original_target_support > 0
            and original_train_support == 0
            and original_val_support == 0
        )

        record: Dict[str, Any] = {
            "held_out_family": family,
            "first_and_only_positive_day": target_day,
            "target_positive_support": heldout_support,
            "support_status": support_status,
            "original_mon_wed_train_thu_validation_fri_target": {
                "heldout_train_support": original_train_support,
                "heldout_validation_support": original_val_support,
                "heldout_target_support": original_target_support,
                "family_target_exists_under_original_geometry": original_family_geometry_valid,
            },
        }

        # Day-atomic strict chronology:
        #   TRAIN = all weekdays before VALIDATION
        #   VALIDATION = the weekday immediately preceding TARGET
        #   TARGET = family-positive weekday
        #
        # Additional requirements:
        #   - held-out family absent from TRAIN and VALIDATION
        #   - TRAIN has benign AND at least one known-family positive
        #   - VALIDATION has benign AND at least one known-family positive
        # These conditions are needed for a supervised binary fit and known-family
        # threshold selection without target-family labels.
        if target_idx < 2:
            record.update(
                {
                    "status": "STRUCTURALLY_INELIGIBLE",
                    "reason_code": "INSUFFICIENT_PRETARGET_WEEKDAY_DEPTH",
                    "reason": (
                        f"{family} first appears on {target_day}. Fewer than two "
                        "earlier weekday partitions exist, so separate day-atomic "
                        "TRAIN and VALIDATION partitions cannot both precede TARGET."
                    ),
                    "replacement_geometry": None,
                }
            )
        else:
            validation_day = WEEKDAYS[target_idx - 1]
            train_days = WEEKDAYS[: target_idx - 1]
            validation_days = [validation_day]
            target_days = [target_day]

            train_family = sum_family_support(
                matrix, train_days, exclude_family=family
            )
            val_family = sum_family_support(
                matrix, validation_days, exclude_family=family
            )

            heldout_train = sum(matrix[family][d] for d in train_days)
            heldout_val = sum(matrix[family][d] for d in validation_days)

            train_attack = sum(train_family.values())
            val_attack = sum(val_family.values())

            train_benign = sum(benign_by_day[d] for d in train_days)
            val_benign = sum(benign_by_day[d] for d in validation_days)
            target_benign = benign_by_day[target_day]

            if heldout_train != 0 or heldout_val != 0:
                raise RuntimeError(
                    f"Held-out leakage discovered during feasibility construction for {family}"
                )

            if train_attack <= 0:
                status = "STRUCTURALLY_INELIGIBLE"
                reason_code = "NO_KNOWN_ATTACK_POSITIVES_IN_DAY_ATOMIC_TRAIN"
                reason = (
                    f"The only day-atomic chronology before {target_day} gives "
                    f"TRAIN={train_days}, but TRAIN contains {train_attack} known-family "
                    "attack positives. A supervised attack-vs-benign learner cannot be "
                    "fit honestly under this geometry."
                )
                geometry = None
            elif val_attack <= 0:
                status = "STRUCTURALLY_INELIGIBLE"
                reason_code = "NO_KNOWN_ATTACK_POSITIVES_IN_DAY_ATOMIC_VALIDATION"
                reason = (
                    f"VALIDATION={validation_days} contains no known-family attack "
                    "positives for threshold selection."
                )
                geometry = None
            else:
                status = (
                    "ELIGIBLE_DESCRIPTIVE_ONLY"
                    if heldout_support < MIN_INFERENTIAL_POSITIVE_SUPPORT
                    else "ELIGIBLE"
                )
                reason_code = "DAY_ATOMIC_STRICT_CHRONOLOGY_AVAILABLE"
                reason = (
                    "A strict day-atomic TRAIN < VALIDATION < TARGET geometry exists "
                    "with zero held-out-family support in both TRAIN and VALIDATION."
                )
                geometry = {
                    "train_days": train_days,
                    "validation_days": validation_days,
                    "target_days": target_days,
                    "heldout_family_train_count": heldout_train,
                    "heldout_family_validation_count": heldout_val,
                    "train_benign": train_benign,
                    "train_known_attack": train_attack,
                    "train_known_family_support": train_family,
                    "validation_benign": val_benign,
                    "validation_known_attack": val_attack,
                    "validation_known_family_support": val_family,
                    "target_benign_available_same_day": target_benign,
                    "target_heldout_attack": heldout_support,
                }

            record.update(
                {
                    "status": status,
                    "reason_code": reason_code,
                    "reason": reason,
                    "replacement_geometry": geometry,
                }
            )

        family_records[family] = record

    eligible = [
        fam
        for fam, rec in family_records.items()
        if rec["status"] in {"ELIGIBLE", "ELIGIBLE_DESCRIPTIVE_ONLY"}
    ]
    ineligible = [
        fam
        for fam, rec in family_records.items()
        if rec["status"] == "STRUCTURALLY_INELIGIBLE"
    ]
    descriptive_only = [
        fam
        for fam, rec in family_records.items()
        if rec["status"] == "ELIGIBLE_DESCRIPTIVE_ONLY"
    ]

    original_supported = [
        fam
        for fam, rec in family_records.items()
        if rec["original_mon_wed_train_thu_validation_fri_target"][
            "family_target_exists_under_original_geometry"
        ]
    ]

    # Expected day-atomic outcome from the frozen support schedule.
    expected_eligible = ["BOT", "DDOS", "INFILTRATION", "PORT_SCAN", "WEB_ATTACK"]
    expected_ineligible = ["DOS", "AUTH_BRUTE_FORCE"]
    if sorted(eligible) != sorted(expected_eligible):
        raise RuntimeError(
            f"Unexpected eligible fold set: {eligible}; expected {expected_eligible}"
        )
    if sorted(ineligible) != sorted(expected_ineligible):
        raise RuntimeError(
            f"Unexpected ineligible fold set: {ineligible}; expected {expected_ineligible}"
        )
    if descriptive_only != ["INFILTRATION"]:
        raise RuntimeError(
            f"Expected INFILTRATION to be the only descriptive-only eligible fold; "
            f"got {descriptive_only}"
        )

    print("Original universal Mon-Wed / Thu / Fri geometry: INFEASIBLE FOR 7/7")
    print("Families with Friday targets under original geometry:", original_supported)
    print()
    print("Corrected day-atomic eligible folds :", eligible)
    print("Structurally ineligible folds       :", ineligible)
    print("Descriptive-only eligible folds     :", descriptive_only)
    print()

    for fam, rec in family_records.items():
        print(
            f"{fam:<20} {rec['status']:<28} "
            f"target={rec['first_and_only_positive_day']:<9} "
            f"n={rec['target_positive_support']:,}"
        )

    print()
    print("[PASS] Temporal feasibility derived before any Stage27 model result exists.")

    return {
        "stage": "Stage27-0A",
        "created_at_utc": now_utc(),
        "scientific_boundary": {
            "model_fits": 0,
            "model_inference": 0,
            "threshold_selection": 0,
            "bootstrap_replicates": 0,
            "gpu_execution": False,
            "primary_execution_device": PRIMARY_EXECUTION_DEVICE,
            "primary_gpu_budget_hours": PRIMARY_GPU_BUDGET_HOURS,
        },
        "chronology_semantics": {
            "unit": "WEEKDAY_ATOMIC_PARTITIONS",
            "required_order": "TRAIN < VALIDATION < TARGET",
            "train_requires_benign": True,
            "train_requires_known_attack_positive": True,
            "validation_requires_benign": True,
            "validation_requires_known_attack_positive": True,
            "heldout_family_required_absent_from_train": True,
            "heldout_family_required_absent_from_validation": True,
        },
        "original_proposal": {
            "train_days": original_train,
            "validation_days": original_validation,
            "target_days": original_target,
            "universal_seven_fold_feasible": False,
            "families_with_positive_target_support": original_supported,
            "reason": (
                "Only BOT, DDOS and PORT_SCAN have Friday positives. DOS appears only "
                "Wednesday; AUTH_BRUTE_FORCE only Tuesday; INFILTRATION and WEB_ATTACK "
                "only Thursday. The universal Friday-target geometry therefore cannot "
                "produce all seven held-out-family targets."
            ),
        },
        "replacement_day_atomic_family_specific_geometry": {
            "status": "FEASIBILITY_CORRECTION_PRE_RESULT",
            "eligible_folds": eligible,
            "eligible_fold_count": len(eligible),
            "structurally_ineligible_folds": ineligible,
            "structurally_ineligible_fold_count": len(ineligible),
            "descriptive_only_eligible_folds": descriptive_only,
            "minimum_positive_support_for_inferential_claim": MIN_INFERENTIAL_POSITIVE_SUPPORT,
            "family_records": family_records,
        },
        "important_semantic_limit": (
            "Strict chronology means some non-held-out families occur only on or after "
            "a target day and therefore cannot all be represented in that fold's "
            "training set. Stage27-0 must describe the executable design as a "
            "chronology-first zero-training-exposure family audit and must not imply "
            "that every non-held-out family was necessarily seen in training."
        ),
        "stage27_0_authorization": (
            "AUTHORIZED_ONLY_AFTER_THIS_STAGE27_0A_COMMIT_IS_REMOTELY_VERIFIED"
        ),
    }


# =============================================================================
# 9. WRITE EXACTLY SIX REQUIRED ARTIFACTS
# =============================================================================

def write_family_day_csv(
    outdir: Path,
    matrix: OrderedDict,
) -> None:
    path = outdir / "family_day_support.csv"
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["family", *WEEKDAYS, "Total"])
        for fam in PRIMARY_FAMILIES:
            row = [fam] + [matrix[fam][d] for d in WEEKDAYS]
            row.append(sum(matrix[fam].values()))
            writer.writerow(row)


def write_benign_day_csv(
    outdir: Path,
    benign_by_day: OrderedDict,
) -> None:
    path = outdir / "benign_day_support.csv"
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["day", "benign_support"])
        for day in WEEKDAYS:
            writer.writerow([day, benign_by_day[day]])
        writer.writerow(["TOTAL", sum(benign_by_day.values())])


def markdown_table_family(matrix: OrderedDict) -> str:
    lines = [
        "| Family | Mon | Tue | Wed | Thu | Fri | Total |",
        "|---|---:|---:|---:|---:|---:|---:|",
    ]
    for fam in PRIMARY_FAMILIES:
        vals = [matrix[fam][d] for d in WEEKDAYS]
        lines.append(
            f"| {fam} | "
            + " | ".join(f"{v:,}" for v in vals)
            + f" | {sum(vals):,} |"
        )
    return "\n".join(lines)


def markdown_table_benign(benign_by_day: OrderedDict) -> str:
    lines = [
        "| Day | Benign support |",
        "|---|---:|",
    ]
    for day in WEEKDAYS:
        lines.append(f"| {day} | {benign_by_day[day]:,} |")
    lines.append(f"| **TOTAL** | **{sum(benign_by_day.values()):,}** |")
    return "\n".join(lines)


def markdown_table_feasibility(feas: Dict[str, Any]) -> str:
    records = feas["replacement_day_atomic_family_specific_geometry"]["family_records"]
    lines = [
        "| Family | Target day | Positive support | Stage27-0A status | Corrected geometry |",
        "|---|---|---:|---|---|",
    ]
    for fam in PRIMARY_FAMILIES:
        r = records[fam]
        geom = r["replacement_geometry"]
        if geom is None:
            geom_text = "—"
        else:
            geom_text = (
                f"{','.join(geom['train_days'])} → "
                f"{','.join(geom['validation_days'])} → "
                f"{','.join(geom['target_days'])}"
            )
        lines.append(
            f"| {fam} | {r['first_and_only_positive_day']} | "
            f"{r['target_positive_support']:,} | {r['status']} | {geom_text} |"
        )
    return "\n".join(lines)


def write_artifacts(
    repo: Path,
    stage26_receipt: Dict[str, Any],
    stage24_receipts: List[Dict[str, Any]],
    taxonomy: Dict[str, Any],
    pop: Dict[str, Any],
    compact_audit: Dict[str, Any],
    matrix: OrderedDict,
    support_summary: OrderedDict,
    feasibility: Dict[str, Any],
) -> List[Path]:
    banner("STAGE27-0A :: WRITE SIX REQUIRED ARTIFACTS")

    outdir = repo / OUTPUT_REL

    if outdir.exists():
        existing = [p for p in outdir.iterdir() if p.is_file() or p.is_dir()]
        if existing:
            raise RuntimeError(
                "Stage27-0A output directory is not empty. Refusing to overwrite:\n"
                + "\n".join(str(p) for p in existing)
            )
    outdir.mkdir(parents=True, exist_ok=True)

    write_family_day_csv(outdir, matrix)
    write_benign_day_csv(outdir, pop["benign_by_day"])

    taxonomy_receipt = {
        "stage": "Stage27-0A",
        "type": "TAXONOMY_RECEIPT",
        "created_at_utc": now_utc(),
        "execution_parent": EXPECTED_PARENT,
        "taxonomy_source": STAGE24_PROTOCOL_LOCK_REL,
        "taxonomy_source_sha256": EXPECTED_STAGE24_SOURCE_SHA256[
            STAGE24_PROTOCOL_LOCK_REL
        ],
        "primary_family_count": 7,
        "primary_families": PRIMARY_FAMILIES,
        "raw_label_to_family": RAW_TO_FAMILY,
        "target_only_unseen_not_in_primary_seven_family_taxonomy": {
            "heartbleed": {
                "mapping": "TARGET_ONLY_UNSEEN",
                "support": int(pop["label_counts"]["heartbleed"]),
                "weekday": "Wednesday",
            }
        },
        "taxonomy_mutated": False,
        "family_merge_after_stage24": False,
        "family_split_after_stage24": False,
        "status": "EXACT_STAGE24_TAXONOMY_INHERITED",
    }
    write_json(outdir / "taxonomy_receipt.json", taxonomy_receipt)

    source_receipt = {
        "stage": "Stage27-0A",
        "type": "SOURCE_ARTIFACT_RECEIPTS",
        "created_at_utc": now_utc(),
        "execution_parent": EXPECTED_PARENT,
        "stage26_closure": stage26_receipt,
        "stage24_durable_sources": stage24_receipts,
        "effective_stage27_support_population": {
            "dataset": "CICIDS2017",
            "population_identity": "STAGE24_FULL_EFFECTIVE_TARGET_POPULATION",
            "rows": int(sum(pop["rows_by_day"].values())),
            "benign": int(sum(pop["benign_by_day"].values())),
            "nonbenign_including_heartbleed": int(
                sum(pop["all_attack_by_day"].values())
            ),
            "primary_seven_family_attack": int(
                support_summary["seven_family_attack_total"]
            ),
            "heartbleed_target_only_unseen": int(
                support_summary["heartbleed_target_only_unseen"]
            ),
            "support_source": STAGE24_TARGET_RESULT_REL,
        },
        "stage20_compact_release_population_decision": compact_audit,
        "population_discrepancy_resolution": {
            "discrepancy": (
                "Stage20 compact corpora are much smaller exact-matched raw-byte "
                "corpora, whereas Stage24 exposes the full 2,830,743-row CICIDS2017 "
                "effective target population."
            ),
            "decision": (
                "Use the Stage24 full effective CICIDS2017 population for Stage27 "
                "seven-family feasibility because it durably preserves the canonical "
                "attack-family label counts. Do not substitute Stage20 compact binary "
                "labels for family identities."
            ),
            "reason": (
                "All five committed Stage20 compact manifests persist labels.npy as "
                "binary labels and do not persist a family-label array or durable "
                "seven-family membership mapping."
            ),
            "scientific_adaptation": False,
            "decision_time": "BEFORE_ANY_STAGE27_MODEL_FIT_OR_INFERENCE",
        },
        "model_fits": 0,
        "model_inference": 0,
        "corpus_archives_downloaded_by_stage27_0a": 0,
        "release_corpus_recreated": False,
    }
    write_json(outdir / "source_artifact_receipts.json", source_receipt)

    write_json(outdir / "temporal_feasibility.json", feasibility)

    audit_md = f"""# Stage27-0A — Family × Day Feasibility Audit

**Status:** COMPLETE — ZERO FIT / ZERO INFERENCE FEASIBILITY AUDIT  
**Execution parent:** `{EXPECTED_PARENT}`  
**Primary compute policy:** CPU, GPU budget = 0 hours  
**Created:** {now_utc()}

## Scientific purpose

Stage27-0A determines which chronology-first unseen-family folds are structurally
possible **before** any Stage27 model results exist.

The originally proposed universal geometry was:

```text
Monday–Wednesday -> TRAIN
Thursday         -> VALIDATION
Friday           -> HELD-OUT FAMILY TARGET
```

That geometry is **not feasible for all seven families** because CICIDS2017 attack
families occur on different weekdays.

## Population decision

Stage27-0A uses the **full Stage24 CICIDS2017 effective target population** as the
family-support source.

This is deliberate. The committed Stage20 compact raw-byte release corpora preserve
exact raw-byte flows but persist only:

```text
encoded_bytes.bin
flow_offsets.npy
labels.npy
packet_lengths.npy
```

and the manifest join counts are binary `0/1`. No durable attack-family label array is
present. Therefore the Stage20 compact corpus remains valid for the Stage20 binary
raw-byte experiment, but it is **not** a durable seven-family membership source for
Stage27 LOAO construction.

No release archive was downloaded or recreated during Stage27-0A.

## Frozen seven-family taxonomy

```text
BOT
DDOS
DOS
AUTH_BRUTE_FORCE
INFILTRATION
PORT_SCAN
WEB_ATTACK
```

`heartbleed` remains `TARGET_ONLY_UNSEEN` and is excluded from the seven-family
primary matrix.

## Exact family × weekday support

{markdown_table_family(matrix)}

Seven-family total: **{support_summary['seven_family_attack_total']:,}**

Heartbleed excluded from the primary seven-family matrix: **{support_summary['heartbleed_target_only_unseen']:,}**

All non-benign rows including Heartbleed: **{support_summary['all_nonbenign_total']:,}**

## Benign support by weekday

{markdown_table_benign(pop['benign_by_day'])}

## Original geometry decision

The universal Mon–Wed / Thu / Fri seven-fold geometry is **STRUCTURALLY INFEASIBLE**.

Only the three Friday families — BOT, DDOS and PORT_SCAN — have positive target support
on Friday. DOS appears only Wednesday, AUTH_BRUTE_FORCE only Tuesday, and
INFILTRATION/WEB_ATTACK only Thursday.

The original geometry is therefore rejected **before model fitting**. This is a
feasibility correction, not result-driven adaptation.

## Day-atomic chronology-first feasibility

Stage27-0A applies the following feasibility rule:

```text
TRAIN < VALIDATION < TARGET
```

using weekday-atomic partitions. An executable fold additionally requires:

- zero held-out-family positives in TRAIN;
- zero held-out-family positives in VALIDATION;
- benign and at least one known-family attack positive in TRAIN;
- benign and at least one known-family attack positive in VALIDATION;
- same-target-day benign support for the primary isolation target.

{markdown_table_feasibility(feasibility)}

### Structurally ineligible

- **AUTH_BRUTE_FORCE** — first appears Tuesday. There are not two earlier weekday
  partitions for separate day-atomic TRAIN and VALIDATION, and Monday has no attack
  positives.
- **DOS** — first appears Wednesday. The only day-atomic split is Monday TRAIN /
  Tuesday VALIDATION / Wednesday TARGET, but Monday contains no known attack positives,
  so a supervised binary IDS cannot be trained honestly under that geometry.

These families are retained in the taxonomy and marked
`STRUCTURALLY_INELIGIBLE`; they are not silently dropped.

### Eligible but descriptive-only

- **INFILTRATION** has only **36** positives, below the preregistered recommended
  inferential support threshold of 50. It remains eligible for execution but must be
  marked `DESCRIPTIVE_ONLY`.

## Important semantic limit for Stage27-0

Because chronology is primary, some non-held-out attack families may occur only on or
after a fold's target day and therefore may also be absent from training.

Stage27-0 should therefore describe the executable design precisely as a:

> **chronology-first zero-training-exposure family audit**

and should not imply that every non-held-out family is necessarily represented in
training for every fold.

## Stage27-0A result

```text
Eligible folds:             5
Structurally ineligible:    2
Descriptive-only eligible:  1
Stage27 model fits:         0
Stage27 model inference:    0
GPU execution:              NO
```

Eligible families:

```text
BOT
DDOS
INFILTRATION   [DESCRIPTIVE_ONLY]
PORT_SCAN
WEB_ATTACK
```

Structurally ineligible families:

```text
AUTH_BRUTE_FORCE
DOS
```

## Next gate

Stage27-0 protocol construction is authorized **only after this Stage27-0A artifact
set is committed and the remote `main` SHA is verified**.

This script performs that commit/push/remote verification and then stops. It does not
construct Stage27-0.
"""
    (outdir / "feasibility_audit.md").write_text(audit_md, encoding="utf-8")

    produced = sorted(p.name for p in outdir.iterdir() if p.is_file())
    expected = sorted(REQUIRED_OUTPUTS)

    if produced != expected:
        raise RuntimeError(
            "Stage27-0A produced an unexpected file set.\n"
            f"expected={expected}\nactual={produced}"
        )

    output_paths = [outdir / name for name in REQUIRED_OUTPUTS]

    print("Produced exactly six required files:")
    for path in output_paths:
        print(
            f"  {path.relative_to(repo)}  "
            f"bytes={path.stat().st_size:,}  sha256={sha256_file(path)}"
        )

    print()
    print("[PASS] Exactly six Stage27-0A artifacts written.")
    return output_paths


# =============================================================================
# 10. FINAL LOCAL VALIDATION
# =============================================================================

def validate_outputs(
    repo: Path,
    output_paths: List[Path],
    matrix: OrderedDict,
    pop: Dict[str, Any],
    feasibility: Dict[str, Any],
) -> None:
    banner("STAGE27-0A :: FINAL LOCAL VALIDATION")

    # CSV readback.
    family_csv = repo / OUTPUT_REL / "family_day_support.csv"
    with family_csv.open("r", encoding="utf-8", newline="") as f:
        rows = list(csv.DictReader(f))

    if len(rows) != 7:
        raise RuntimeError(f"family_day_support.csv must contain 7 data rows, got {len(rows)}")

    readback_total = sum(int(r["Total"]) for r in rows)
    if readback_total != 557_635:
        raise RuntimeError(
            f"family_day_support.csv seven-family total mismatch: {readback_total}"
        )

    benign_csv = repo / OUTPUT_REL / "benign_day_support.csv"
    with benign_csv.open("r", encoding="utf-8", newline="") as f:
        benign_rows = list(csv.DictReader(f))

    total_rows = [r for r in benign_rows if r["day"] == "TOTAL"]
    if len(total_rows) != 1 or int(total_rows[0]["benign_support"]) != 2_273_097:
        raise RuntimeError("benign_day_support.csv total mismatch.")

    # JSON readback.
    tf = load_json(repo / OUTPUT_REL / "temporal_feasibility.json")
    eligible_count = tf[
        "replacement_day_atomic_family_specific_geometry"
    ]["eligible_fold_count"]
    ineligible_count = tf[
        "replacement_day_atomic_family_specific_geometry"
    ]["structurally_ineligible_fold_count"]

    if eligible_count != 5 or ineligible_count != 2:
        raise RuntimeError("Temporal-feasibility eligible/ineligible count mismatch.")

    # No Stage27-0 protocol lock may have been created by this script.
    stage27_0 = repo / "results/stage27_loao_unseen_attack/stage27_0_protocol_lock"
    if stage27_0.exists():
        raise RuntimeError(
            "Stage27-0 protocol directory already exists. Stage27-0A must be "
            "remotely verified before Stage27-0 construction."
        )

    # Ensure no unexpected untracked/modified files besides the six intended files.
    # IMPORTANT:
    # A fully untracked directory is collapsed by normal `git status --porcelain`
    # to a single line such as:
    #
    #     ?? results/stage27_loao_unseen_attack/
    #
    # even when exactly six files exist beneath it. Force
    # --untracked-files=all so the validator sees the individual artifact paths.
    porcelain = run(
        ["git", "status", "--porcelain=v1", "--untracked-files=all"],
        cwd=repo,
    ).splitlines()

    changed = []
    for line in porcelain:
        if not line.strip():
            continue

        # Porcelain v1 path begins after XY + space.
        path_part = line[3:] if len(line) >= 4 else line

        # Handle rename syntax defensively, although Stage27-0A should create
        # only new files.
        if " -> " in path_part:
            path_part = path_part.split(" -> ", 1)[1]

        changed.append(path_part)

    expected_files = sorted(
        f"{OUTPUT_REL}/{name}"
        for name in REQUIRED_OUTPUTS
    )
    actual_files = sorted(changed)

    if actual_files != expected_files:
        raise RuntimeError(
            "Unexpected working-tree change set after artifact generation.\n"
            f"expected={expected_files}\n"
            f"actual={actual_files}"
        )

    print("Working-tree Stage27-0A files:")
    for rel in actual_files:
        print(" ", rel)

    print("[PASS] Working-tree changes are exactly the six required Stage27-0A files.")

    print("Family rows                 : 7")
    print("Seven-family attack total  : 557,635")
    print("Benign total               : 2,273,097")
    print("Eligible folds             : 5")
    print("Structurally ineligible    : 2")
    print("Stage27-0 directory exists : NO")
    print("Model fits                 : 0")
    print("Model inference            : 0")
    print("GPU execution              : OFF")
    print("[PASS] Stage27-0A local validation complete.")


# =============================================================================
# 11. COMMIT + PUSH + REMOTE VERIFY
# =============================================================================

def commit_push_verify(
    repo: Path,
    env: Dict[str, str],
    output_paths: List[Path],
) -> str:
    banner("STAGE27-0A :: GIT COMMIT")

    rel_files = [str(p.relative_to(repo)) for p in output_paths]

    run(["git", "add", "--", *rel_files], cwd=repo)

    staged = run(["git", "diff", "--cached", "--name-only"], cwd=repo).splitlines()

    if sorted(staged) != sorted(rel_files):
        raise RuntimeError(
            "Unexpected staged file set.\n"
            f"expected={sorted(rel_files)}\n"
            f"actual={sorted(staged)}"
        )

    print("Staged exactly:")
    for rel in staged:
        print(" ", rel)

    print()
    print("Staged diff stat:")
    print(run(["git", "diff", "--cached", "--stat"], cwd=repo))

    run(["git", "commit", "-m", COMMIT_MESSAGE], cwd=repo)

    new_head = run(["git", "rev-parse", "HEAD"], cwd=repo)
    parent = run(["git", "rev-parse", "HEAD^"], cwd=repo)

    if parent != EXPECTED_PARENT:
        raise RuntimeError(
            f"New commit parent mismatch: expected {EXPECTED_PARENT}, got {parent}"
        )

    print()
    print("Commit created:", new_head)

    banner("STAGE27-0A :: PUSH MAIN")
    run(["git", "push", "origin", "main"], cwd=repo, env=env)
    print("[OK] Push completed.")

    banner("STAGE27-0A :: REMOTE VERIFICATION")
    remote_head = run(
        ["git", "ls-remote", "origin", "refs/heads/main"],
        cwd=repo,
        env=env,
    ).split()[0]

    run(["git", "fetch", "origin", "main"], cwd=repo, env=env)
    origin_main = run(["git", "rev-parse", "origin/main"], cwd=repo)
    local_head = run(["git", "rev-parse", "HEAD"], cwd=repo)

    print("Local HEAD   :", local_head)
    print("origin/main  :", origin_main)
    print("remote main  :", remote_head)

    if not (local_head == origin_main == remote_head == new_head):
        raise RuntimeError(
            "Remote verification mismatch after Stage27-0A push.\n"
            f"local={local_head}\norigin/main={origin_main}\nremote={remote_head}"
        )

    # Verify the exact six paths exist at remote-tracking main.
    remote_tree = run(
        ["git", "ls-tree", "-r", "--name-only", "origin/main", "--", OUTPUT_REL],
        cwd=repo,
    ).splitlines()

    expected_remote = sorted(
        [f"{OUTPUT_REL}/{name}" for name in REQUIRED_OUTPUTS]
    )
    actual_remote = sorted(remote_tree)

    if actual_remote != expected_remote:
        raise RuntimeError(
            "Remote Stage27-0A tree does not contain exactly the six expected files.\n"
            f"expected={expected_remote}\nactual={actual_remote}"
        )

    if not git_status_clean(repo):
        raise RuntimeError("Working tree is not clean after successful push.")

    print()
    print("[PASS] Stage27-0A commit is remotely verified.")
    print("[PASS] Remote tree contains exactly the six required Stage27-0A artifacts.")
    print("[PASS] Working tree is clean.")

    return new_head


# =============================================================================
# 12. MAIN
# =============================================================================

def main() -> None:
    # Hard-disable CUDA visibility for this CPU-only phase.
    os.environ["CUDA_VISIBLE_DEVICES"] = ""

    banner("STAGE27-0A :: ZERO-FIT FAMILY × DAY FEASIBILITY AUDIT")
    print("timestamp_utc            :", now_utc())
    print("python                   :", sys.version.replace("\n", " "))
    print("working_directory        :", Path.cwd())
    print("PRIMARY_EXECUTION_DEVICE :", PRIMARY_EXECUTION_DEVICE)
    print("PRIMARY_GPU_BUDGET_HOURS :", PRIMARY_GPU_BUDGET_HOURS)
    print("CUDA_VISIBLE_DEVICES     :", repr(os.environ.get("CUDA_VISIBLE_DEVICES")))
    print()
    print("Scientific boundary:")
    print("  model fits              : 0")
    print("  model inference         : 0")
    print("  threshold selection     : 0")
    print("  bootstrap replicates    : 0")
    print("  corpus downloads        : 0")
    print("  release corpus recreate : NO")

    token = get_kaggle_github_token()
    env, askpass = make_git_auth_env(token)

    try:
        repo = find_repo()
        ensure_repo(repo, env)

        outdir = repo / OUTPUT_REL
        banner("STAGE27-0A :: OUTPUT DIRECTORY GATE")
        print("Output directory:", outdir)

        if outdir.exists():
            existing = sorted(
                p.name
                for p in outdir.iterdir()
                if p.is_file()
            )
            subdirs = sorted(
                p.name
                for p in outdir.iterdir()
                if p.is_dir()
            )

            print("Existing files:", len(existing))
            for name in existing:
                print(" ", name)

            if subdirs:
                raise RuntimeError(
                    "Unexpected subdirectories exist inside Stage27-0A output: "
                    + repr(subdirs)
                )

            if existing:
                # Recovery path for the immediately preceding failed run:
                # exactly the six required artifacts exist and all are still
                # untracked. Nothing was committed or pushed.
                porcelain = run(
                    ["git", "status", "--porcelain=v1", "--untracked-files=all"],
                    cwd=repo,
                ).splitlines()

                actual_untracked = sorted(
                    line[3:]
                    for line in porcelain
                    if line.startswith("?? ")
                    and line[3:].startswith(OUTPUT_REL + "/")
                )

                expected_untracked = sorted(
                    f"{OUTPUT_REL}/{name}"
                    for name in REQUIRED_OUTPUTS
                )

                other_changes = [
                    line
                    for line in porcelain
                    if not (
                        line.startswith("?? ")
                        and line[3:].startswith(OUTPUT_REL + "/")
                    )
                ]

                if (
                    existing == sorted(REQUIRED_OUTPUTS)
                    and actual_untracked == expected_untracked
                    and not other_changes
                ):
                    print()
                    print(
                        "[RECOVERY] Found exactly the six untracked Stage27-0A "
                        "artifacts left by the previous validation-only failure."
                    )
                    print(
                        "[RECOVERY] No commit/push occurred and no unrelated Git "
                        "changes exist."
                    )
                    print(
                        "[RECOVERY] Removing only this uncommitted Stage27-0A "
                        "directory so it can be regenerated cleanly."
                    )
                    shutil.rmtree(outdir)
                else:
                    raise RuntimeError(
                        "Stage27-0A output directory is non-empty and does not "
                        "match the safe recovery state. Refusing to overwrite.\n"
                        f"existing={existing}\n"
                        f"expected={sorted(REQUIRED_OUTPUTS)}\n"
                        f"untracked={actual_untracked}\n"
                        f"other_changes={other_changes}"
                    )
        else:
            print("Existing entries: 0 (directory not yet created)")

        print("[PASS] Stage27-0A output is clean/recovery-safe.")

        stage26 = verify_stage26_closure(repo)
        stage24_receipts = verify_stage24_sources(repo)
        taxonomy = verify_taxonomy(repo)
        pop = recover_stage24_population(repo)
        compact_audit = audit_stage20_compact_binary_only(repo)
        matrix, support_summary = construct_support_matrix(pop)
        feasibility = derive_temporal_feasibility(
            matrix,
            pop["benign_by_day"],
        )

        output_paths = write_artifacts(
            repo=repo,
            stage26_receipt=stage26,
            stage24_receipts=stage24_receipts,
            taxonomy=taxonomy,
            pop=pop,
            compact_audit=compact_audit,
            matrix=matrix,
            support_summary=support_summary,
            feasibility=feasibility,
        )

        validate_outputs(
            repo=repo,
            output_paths=output_paths,
            matrix=matrix,
            pop=pop,
            feasibility=feasibility,
        )

        commit_sha = commit_push_verify(
            repo=repo,
            env=env,
            output_paths=output_paths,
        )

        banner("STAGE27-0A COMPLETE — REMOTELY VERIFIED")
        print("Execution parent          :", EXPECTED_PARENT)
        print("Stage27-0A commit         :", commit_sha)
        print("Remote main               :", commit_sha)
        print("Artifacts                 : 6/6")
        print("Eligible folds            : 5")
        print("Structurally ineligible   : 2")
        print("Descriptive-only eligible : INFILTRATION (n=36)")
        print("Model fits                : 0")
        print("Model inference           : 0")
        print("GPU execution             : OFF")
        print("Git working tree          : CLEAN")
        print()
        print("NEXT AUTHORIZED STEP:")
        print("  Construct Stage27-0 protocol lock from this remotely verified")
        print("  Stage27-0A feasibility correction.")
        print()
        print("STOPPING HERE. Stage27-0 is NOT created by this script.")
        print("=" * 110)

    finally:
        # Never retain auth material.
        try:
            if askpass.exists():
                askpass.unlink()
        except Exception:
            pass

        env.pop("STAGE27_GITHUB_TOKEN", None)
        os.environ.pop("STAGE27_GITHUB_TOKEN", None)

        # Best-effort overwrite local Python reference.
        token = ""


if __name__ == "__main__":
    main()



STAGE27-0A :: ZERO-FIT FAMILY × DAY FEASIBILITY AUDIT
timestamp_utc            : 2026-08-20T17:05:28.392859+00:00
python                   : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
working_directory        : /kaggle/working
PRIMARY_EXECUTION_DEVICE : CPU
PRIMARY_GPU_BUDGET_HOURS : 0
CUDA_VISIBLE_DEVICES     : ''

Scientific boundary:
  model fits              : 0
  model inference         : 0
  threshold selection     : 0
  bootstrap replicates    : 0
  corpus downloads        : 0
  release corpus recreate : NO
[FOUND] GitHub secret label: GITHUB_TOKEN (value not printed)

STAGE27-0A :: REPOSITORY BOOTSTRAP
Repository : /kaggle/working/ids2018-validation-safe-ablation
Remote     : https://github.com/themubasshir/ids2018-validation-safe-ablation.git
[RECOVERY] Working tree contains exactly the six expected untracked Stage27-0A artifacts from the prior failed run.
[RECOVERY] No tracked modifications or unrelated untracked files are present; bootstrap may continue.

STAGE27-0A

RuntimeError: Command failed (128): git commit -m stage27-0a: freeze family-day feasibility audit

STDOUT:


STDERR:
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@c6f469f4a718.(none)')


In [9]:
# ======================================================================================
# STAGE27-0A — COMMIT/PUSH RECOVERY AFTER MISSING GIT AUTHOR IDENTITY
# ======================================================================================
#
# Preconditions:
#   - six Stage27-0A artifacts already generated
#   - six files already staged
#   - prior git commit failed ONLY because user.name/user.email were unset
#
# This cell:
#   1. verifies HEAD is still the Stage27 execution parent
#   2. verifies exactly six intended files are staged
#   3. configures repository-local Git identity
#   4. commits
#   5. pushes main
#   6. verifies local/origin/remote SHA equality
#   7. verifies remote tree contains exactly six Stage27-0A artifacts
#   8. verifies working tree is clean
#
# ZERO model fits / inference / thresholds / bootstrap / GPU
# ======================================================================================

from pathlib import Path
import os
import stat
import subprocess

from kaggle_secrets import UserSecretsClient


REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

EXPECTED_PARENT = (
    "e47f44751bc71d219c5d0f3b3fca06d62037fb8b"
)

OUTPUT_REL = (
    "results/stage27_loao_unseen_attack/"
    "stage27_0a_family_day_feasibility"
)

REQUIRED_OUTPUTS = [
    "family_day_support.csv",
    "benign_day_support.csv",
    "taxonomy_receipt.json",
    "source_artifact_receipts.json",
    "temporal_feasibility.json",
    "feasibility_audit.md",
]

COMMIT_MESSAGE = (
    "stage27-0a: freeze family-day feasibility audit"
)


def banner(title):
    print()
    print("=" * 110)
    print(title)
    print("=" * 110)


def run(args, *, env=None, check=True):
    p = subprocess.run(
        [str(x) for x in args],
        cwd=REPO,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}): {' '.join(map(str, args))}\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return p.stdout.strip()


# ======================================================================================
# 1. CURRENT STATE GATE
# ======================================================================================

banner("STAGE27-0A :: STAGED-STATE RECOVERY GATE")

head = run(
    ["git", "rev-parse", "HEAD"]
)

origin_main = run(
    ["git", "rev-parse", "origin/main"]
)

print("Expected parent :", EXPECTED_PARENT)
print("Local HEAD      :", head)
print("origin/main     :", origin_main)

if head != EXPECTED_PARENT:
    raise RuntimeError(
        "HEAD changed after the failed commit. "
        "Do not continue automatically."
    )

if origin_main != EXPECTED_PARENT:
    raise RuntimeError(
        "origin/main changed after Stage27-0A generation. "
        "Do not continue automatically."
    )


expected_paths = sorted(
    f"{OUTPUT_REL}/{name}"
    for name in REQUIRED_OUTPUTS
)

staged_paths = sorted(
    x
    for x in run(
        [
            "git",
            "diff",
            "--cached",
            "--name-only",
        ]
    ).splitlines()
    if x.strip()
)

print()
print("Expected staged files:")
for path in expected_paths:
    print(" ", path)

print()
print("Actual staged files:")
for path in staged_paths:
    print(" ", path)


if staged_paths != expected_paths:
    raise RuntimeError(
        "\nStaged file universe mismatch.\n"
        f"Expected:\n{expected_paths}\n\n"
        f"Actual:\n{staged_paths}"
    )


# Ensure there are no additional unstaged/untracked changes.
porcelain = run(
    [
        "git",
        "status",
        "--porcelain=v1",
        "--untracked-files=all",
    ]
).splitlines()

unexpected = []

for line in porcelain:

    if not line.strip():
        continue

    path = line[3:]

    if path not in expected_paths:
        unexpected.append(
            line
        )

if unexpected:
    raise RuntimeError(
        "Unexpected repository changes detected:\n"
        + "\n".join(unexpected)
    )


print()
print("[PASS] Exactly six intended Stage27-0A artifacts remain staged.")


# ======================================================================================
# 2. REPOSITORY-LOCAL GIT IDENTITY
# ======================================================================================

banner("STAGE27-0A :: CONFIGURE REPOSITORY-LOCAL GIT IDENTITY")

# Repository-local only. This does NOT alter global Kaggle Git configuration.
run(
    [
        "git",
        "config",
        "--local",
        "user.name",
        "J.M. Mubasshir Rahman",
    ]
)

run(
    [
        "git",
        "config",
        "--local",
        "user.email",
        "themubasshir@users.noreply.github.com",
    ]
)

git_name = run(
    [
        "git",
        "config",
        "--local",
        "user.name",
    ]
)

git_email = run(
    [
        "git",
        "config",
        "--local",
        "user.email",
    ]
)

print("Git author name :", git_name)
print("Git author email:", git_email)

if not git_name or not git_email:
    raise RuntimeError(
        "Repository-local Git identity was not configured."
    )

print("[PASS] Git author identity configured locally.")


# ======================================================================================
# 3. AUTH WITHOUT PRINTING TOKEN
# ======================================================================================

banner("STAGE27-0A :: LOAD GITHUB AUTH")

client = UserSecretsClient()

token = None
token_label = None

for label in [
    "GITHUB_TOKEN",
    "github_token",
    "GH_TOKEN",
    "GITHUB_PAT",
    "github_pat",
]:

    try:
        candidate = client.get_secret(
            label
        )
    except Exception:
        candidate = None

    if candidate:
        token = str(candidate).strip()
        token_label = label
        break


if not token:
    raise RuntimeError(
        "No GitHub token found in Kaggle Secrets."
    )

print(
    f"[FOUND] GitHub secret label: {token_label} "
    "(value not printed)"
)


askpass = Path(
    "/kaggle/working/.stage27_commit_recovery_askpass.sh"
)

askpass.write_text(
    "#!/bin/sh\n"
    'case "$1" in\n'
    '  *Username*) printf "%s\\n" "x-access-token" ;;\n'
    '  *Password*) printf "%s\\n" "$STAGE27_GITHUB_TOKEN" ;;\n'
    '  *) printf "%s\\n" "" ;;\n'
    "esac\n",
    encoding="utf-8",
)

askpass.chmod(
    stat.S_IRUSR
    | stat.S_IWUSR
    | stat.S_IXUSR
)

env = os.environ.copy()

env[
    "GIT_ASKPASS"
] = str(
    askpass
)

env[
    "GIT_TERMINAL_PROMPT"
] = "0"

env[
    "STAGE27_GITHUB_TOKEN"
] = token


# ======================================================================================
# 4. COMMIT
# ======================================================================================

try:

    banner("STAGE27-0A :: COMMIT")

    print(
        run(
            [
                "git",
                "diff",
                "--cached",
                "--stat",
            ]
        )
    )

    commit_output = run(
        [
            "git",
            "commit",
            "-m",
            COMMIT_MESSAGE,
        ]
    )

    print()
    print(commit_output)

    new_head = run(
        [
            "git",
            "rev-parse",
            "HEAD",
        ]
    )

    new_parent = run(
        [
            "git",
            "rev-parse",
            "HEAD^",
        ]
    )

    print()
    print("New commit :", new_head)
    print("Parent     :", new_parent)

    if new_parent != EXPECTED_PARENT:
        raise RuntimeError(
            "Stage27-0A commit parent is not the frozen execution parent."
        )


    # ==================================================================================
    # 5. PUSH
    # ==================================================================================

    banner("STAGE27-0A :: PUSH MAIN")

    push_output = run(
        [
            "git",
            "push",
            "origin",
            "main",
        ],
        env=env,
    )

    print(
        push_output
        if push_output
        else "[OK] Push completed."
    )


    # ==================================================================================
    # 6. REMOTE SHA VERIFICATION
    # ==================================================================================

    banner("STAGE27-0A :: REMOTE SHA VERIFICATION")

    remote_main = run(
        [
            "git",
            "ls-remote",
            "origin",
            "refs/heads/main",
        ],
        env=env,
    ).split()[0]

    run(
        [
            "git",
            "fetch",
            "origin",
            "main",
        ],
        env=env,
    )

    origin_after = run(
        [
            "git",
            "rev-parse",
            "origin/main",
        ]
    )

    local_after = run(
        [
            "git",
            "rev-parse",
            "HEAD",
        ]
    )

    print("Local HEAD  :", local_after)
    print("origin/main :", origin_after)
    print("remote main :", remote_main)

    if not (
        local_after
        == origin_after
        == remote_main
        == new_head
    ):
        raise RuntimeError(
            "Remote SHA verification failed."
        )

    print(
        "[PASS] Local HEAD == origin/main == remote main."
    )


    # ==================================================================================
    # 7. REMOTE TREE VERIFICATION
    # ==================================================================================

    banner("STAGE27-0A :: REMOTE ARTIFACT TREE")

    remote_files = sorted(
        x
        for x in run(
            [
                "git",
                "ls-tree",
                "-r",
                "--name-only",
                "origin/main",
                "--",
                OUTPUT_REL,
            ]
        ).splitlines()
        if x.strip()
    )

    print(
        "Remote Stage27-0A files:"
    )

    for path in remote_files:
        print(" ", path)


    if remote_files != expected_paths:
        raise RuntimeError(
            "\nRemote Stage27-0A file universe mismatch.\n"
            f"Expected:\n{expected_paths}\n\n"
            f"Actual:\n{remote_files}"
        )

    print()
    print(
        "[PASS] Remote contains exactly the six required artifacts."
    )


    # ==================================================================================
    # 8. FINAL CLEANNESS + SCIENTIFIC BOUNDARY
    # ==================================================================================

    banner("STAGE27-0A :: FINAL CLOSURE")

    final_status = run(
        [
            "git",
            "status",
            "--porcelain=v1",
            "--untracked-files=all",
        ]
    )

    if final_status:
        raise RuntimeError(
            "Repository is not clean after Stage27-0A push:\n"
            + final_status
        )


    stage27_0_dir = (
        REPO
        / "results"
        / "stage27_loao_unseen_attack"
        / "stage27_0_protocol_lock"
    )

    if stage27_0_dir.exists():
        raise RuntimeError(
            "Stage27-0 exists unexpectedly."
        )


    print("Stage27-0A commit         :", new_head)
    print("Remote verification       : PASS")
    print("Required artifacts        : 6/6")
    print("Git working tree          : CLEAN")
    print("Stage27-0 created         : NO")
    print()
    print("Eligible folds            : 5")
    print("Structurally ineligible   : 2")
    print("  - DOS")
    print("  - AUTH_BRUTE_FORCE")
    print("Descriptive-only eligible : INFILTRATION (n=36)")
    print()
    print("Model fits                : 0")
    print("Model inference           : 0")
    print("Threshold selection       : 0")
    print("Bootstrap replicates      : 0")
    print("GPU execution             : 0")
    print()
    print(
        "NEXT AUTHORIZED STEP:"
    )
    print(
        "  Construct Stage27-0 protocol lock from this "
        "remotely verified Stage27-0A commit."
    )
    print()
    print(
        "STAGE27-0A COMPLETE."
    )

finally:

    try:
        askpass.unlink(
            missing_ok=True
        )
    except Exception:
        pass

    token = None

    env.pop(
        "STAGE27_GITHUB_TOKEN",
        None,
    )


STAGE27-0A :: STAGED-STATE RECOVERY GATE
Expected parent : e47f44751bc71d219c5d0f3b3fca06d62037fb8b
Local HEAD      : e47f44751bc71d219c5d0f3b3fca06d62037fb8b
origin/main     : e47f44751bc71d219c5d0f3b3fca06d62037fb8b

Expected staged files:
  results/stage27_loao_unseen_attack/stage27_0a_family_day_feasibility/benign_day_support.csv
  results/stage27_loao_unseen_attack/stage27_0a_family_day_feasibility/family_day_support.csv
  results/stage27_loao_unseen_attack/stage27_0a_family_day_feasibility/feasibility_audit.md
  results/stage27_loao_unseen_attack/stage27_0a_family_day_feasibility/source_artifact_receipts.json
  results/stage27_loao_unseen_attack/stage27_0a_family_day_feasibility/taxonomy_receipt.json
  results/stage27_loao_unseen_attack/stage27_0a_family_day_feasibility/temporal_feasibility.json

Actual staged files:
  results/stage27_loao_unseen_attack/stage27_0a_family_day_feasibility/benign_day_support.csv
  results/stage27_loao_unseen_attack/stage27_0a_family_day_feasibility

In [10]:
from pathlib import Path
import re

REPO = Path("/kaggle/working/ids2018-validation-safe-ablation")

patterns = [
    r"scale_pos_weight",
    r"class_weight",
    r"class weighting",
    r"class-weight",
    r"positive weight",
    r"negative.*positive",
    r"benign.*attack.*ratio",
    r"attack.*benign.*ratio",
    r"is_unbalance",
]

rx = re.compile("|".join(patterns), re.I)

roots = [
    REPO / "results/stage22_temporal_session_safe",
    REPO / "results/stage22r_protocol_recovery",
    REPO / "results/stage22r_training",
    REPO / "results/stage16_classical_benchmark_checkpoint",
    REPO / "docs",
    REPO / "scripts",
]

extensions = {
    ".json", ".md", ".py", ".csv", ".yaml", ".yml"
}

hits = []

for root in roots:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        if path.suffix.lower() not in extensions:
            continue

        # Avoid huge generated artifacts.
        if path.stat().st_size > 2_000_000:
            continue

        try:
            text = path.read_text(
                encoding="utf-8",
                errors="replace",
            )
        except Exception:
            continue

        for lineno, line in enumerate(
            text.splitlines(),
            start=1,
        ):
            if rx.search(line):
                hits.append(
                    (
                        str(path.relative_to(REPO)),
                        lineno,
                        line.strip(),
                    )
                )

print("=" * 110)
print("STAGE27-0 :: CLASS-WEIGHT FORMULA PROVENANCE SEARCH")
print("=" * 110)
print("matches:", len(hits))
print()

for path, lineno, line in hits:
    print(f"{path}:{lineno}")
    print(f"  {line}")
    print()

print("=" * 110)
print("ZERO WRITES / ZERO FITS / ZERO INFERENCE")
print("=" * 110)

STAGE27-0 :: CLASS-WEIGHT FORMULA PROVENANCE SEARCH
matches: 155

results/stage22_temporal_session_safe/stage22_2b_devval_input_manifest.json:297
  "positive_class_weight": {

results/stage22_temporal_session_safe/stage22_2a_inherited_model_execution_recipe_lock.json:212
  "development_single_class": "HARD_SCIENTIFIC_INFEASIBILITY_BEFORE_MODEL_FIT; DO_NOT_REASSIGN_DAYS_CHANGE_CLASS_WEIGHTING SUBSTITUTE_MODELS_OR_RELAX_PROTOCOL",

results/stage22_temporal_session_safe/stage22_2a_inherited_model_execution_recipe_lock.json:328
  "class_weight_search": false,

results/stage22_temporal_session_safe/stage22_2a_inherited_model_execution_recipe_lock.json:330
  "positive_class_weight": "DEVELOPMENT_BENIGN_COUNT_DIVIDED_BY_DEVELOPMENT_ATTACK_COUNT",

results/stage22r_training/stage22r_2d_chronological_rebalanced/chronological_rebalanced_xgboost_model.json:1
  {"learner":{"attributes":{"scikit_learn":"{\"_estimator_type\": \"classifier\"}"},"feature_names":[],"feature_types":[],"gradient_booster"

In [11]:
# ======================================================================================
# STAGE27-0
# COMPLETE LOAO UNSEEN-FAMILY PROTOCOL LOCK
#
# ZERO DOWNLOADS
# ZERO MODEL FITS
# ZERO MODEL INFERENCE
# ZERO THRESHOLD SELECTION
# ZERO TARGET OPENINGS
# ZERO BOOTSTRAP EXECUTION
# ZERO GPU
#
# This cell:
#   1. verifies the remotely-anchored Stage27-0A parent,
#   2. recovers all inherited Stage22/24 semantics from durable repository artifacts,
#   3. freezes the complete Stage27 protocol,
#   4. writes exactly the 18 handoff-prescribed JSON artifacts,
#   5. stages exactly those 18 artifacts,
#   6. STOPS before commit/push.
# ======================================================================================

from __future__ import annotations

import csv
import copy
import hashlib
import json
import math
import os
import subprocess
from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Tuple


# ======================================================================================
# 0. CONSTANTS
# ======================================================================================

REPO = Path("/kaggle/working/ids2018-validation-safe-ablation").resolve()

EXPECTED_PARENT = "da1cc26b2759f43e64122bb8ff7a21d112dc420f"

STAGE27_ROOT_REL = Path("results/stage27_loao_unseen_attack")
STAGE27_0A_REL = STAGE27_ROOT_REL / "stage27_0a_family_day_feasibility"
OUT_REL = STAGE27_ROOT_REL / "stage27_0_protocol_lock"
OUT_DIR = REPO / OUT_REL

STAGE22_RECIPE_REL = Path(
    "results/stage22_temporal_session_safe/"
    "stage22_2a_inherited_model_execution_recipe_lock.json"
)

STAGE22_DEVVAL_REL = Path(
    "results/stage22_temporal_session_safe/"
    "stage22_2b_devval_input_manifest.json"
)

STAGE15_FEATURE_REL = Path(
    "results/stage15_transformer_checkpoint/"
    "stage15_1_feature_configuration.json"
)

STAGE24_PROTOCOL_REL = Path(
    "results/stage24_cross_dataset/stage24_0_protocol_lock/"
    "stage24_0c_final_preopening_protocol_lock.json"
)

STAGE24_BRIDGE_REL = Path(
    "results/stage24_cross_dataset/stage24_0_protocol_lock/"
    "stage24_0c_semantic_bridge_spec.json"
)

STAGE24_SOURCE_REL = Path(
    "results/stage24_cross_dataset/stage24_0_protocol_lock/"
    "stage24_0b2_complete_cicids2017_source_contract.json"
)

REQUIRED_STAGE27_0A_FILES = [
    "family_day_support.csv",
    "benign_day_support.csv",
    "taxonomy_receipt.json",
    "source_artifact_receipts.json",
    "temporal_feasibility.json",
    "feasibility_audit.md",
]

REQUIRED_STAGE27_0_FILES = [
    "taxonomy.json",
    "fold_spec.json",
    "temporal_geometry.json",
    "target_population_spec.json",
    "model_inventory.json",
    "fit_budget.json",
    "compute_policy.json",
    "feature_representation.json",
    "class_weight_policy.json",
    "threshold_policy.json",
    "metric_spec.json",
    "support_rules.json",
    "uncertainty_spec.json",
    "similarity_spec.json",
    "target_opening_ledger.json",
    "interpretation_matrix.json",
    "prohibited_claims.json",
    "inherited_receipts.json",
    "freeze_record.json",
]

FAMILY_ORDER = [
    "BOT",
    "DDOS",
    "DOS",
    "AUTH_BRUTE_FORCE",
    "INFILTRATION",
    "PORT_SCAN",
    "WEB_ATTACK",
]

EXPECTED_ELIGIBLE = [
    "BOT",
    "DDOS",
    "INFILTRATION",
    "PORT_SCAN",
    "WEB_ATTACK",
]

EXPECTED_INELIGIBLE = [
    "DOS",
    "AUTH_BRUTE_FORCE",
]

EXPECTED_DESCRIPTIVE_ONLY = [
    "INFILTRATION",
]

PRIMARY_MODEL = "XGBOOST"
REPLICATION_MODEL = "LIGHTGBM"

BOOTSTRAP_REPLICATES = 2000
BOOTSTRAP_SEED = 42

MIN_INFERENTIAL_POSITIVE_SUPPORT = 50

# Explicitly enforce CPU-only Stage27 execution.
os.environ["CUDA_VISIBLE_DEVICES"] = ""


# ======================================================================================
# 1. HELPERS
# ======================================================================================

def banner(title: str) -> None:
    print()
    print("=" * 118)
    print(title)
    print("=" * 118)


def run(
    cmd: List[str],
    cwd: Path = REPO,
    check: bool = True,
) -> str:
    proc = subprocess.run(
        cmd,
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    if check and proc.returncode != 0:
        raise RuntimeError(
            f"Command failed ({proc.returncode}): {' '.join(cmd)}\n\n"
            f"STDOUT:\n{proc.stdout}\n\n"
            f"STDERR:\n{proc.stderr}"
        )

    return proc.stdout.strip()


def now_utc() -> str:
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def canonical_json_bytes(obj: Any) -> bytes:
    return json.dumps(
        obj,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")


def sha256_json(obj: Any) -> str:
    return hashlib.sha256(canonical_json_bytes(obj)).hexdigest()


def load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def write_json(path: Path, obj: Any) -> None:
    path.write_text(
        json.dumps(
            obj,
            ensure_ascii=False,
            indent=2,
            sort_keys=False,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )


def load_csv_rows(path: Path) -> List[Dict[str, str]]:
    with path.open("r", encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))


def file_receipt(path: Path) -> Dict[str, Any]:
    return {
        "path": str(path.relative_to(REPO)),
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }


def recursively_find_key(obj: Any, key: str) -> List[Any]:
    found = []

    if isinstance(obj, dict):
        for k, v in obj.items():
            if k == key:
                found.append(v)
            found.extend(recursively_find_key(v, key))

    elif isinstance(obj, list):
        for item in obj:
            found.extend(recursively_find_key(item, key))

    return found


def ordered_mapping(
    ordered_keys: List[str],
    mapping: Dict[str, str],
) -> OrderedDict:
    return OrderedDict((key, mapping[key]) for key in ordered_keys)


# ======================================================================================
# 2. REPOSITORY / PARENT GATE
# ======================================================================================

banner("STAGE27-0 :: REPOSITORY AND SCIENTIFIC-PARENT GATE")

if not (REPO / ".git").exists():
    raise RuntimeError(f"Repository not found: {REPO}")

head = run(["git", "rev-parse", "HEAD"])
origin_main = run(["git", "rev-parse", "origin/main"])
status_before = run(
    ["git", "status", "--porcelain=v1", "--untracked-files=all"]
)

print("Expected parent :", EXPECTED_PARENT)
print("Local HEAD      :", head)
print("origin/main     :", origin_main)

if head != EXPECTED_PARENT:
    raise RuntimeError(
        f"Unexpected local HEAD.\n"
        f"expected={EXPECTED_PARENT}\n"
        f"actual={head}"
    )

if origin_main != EXPECTED_PARENT:
    raise RuntimeError(
        f"origin/main mismatch.\n"
        f"expected={EXPECTED_PARENT}\n"
        f"actual={origin_main}"
    )

if status_before.strip():
    raise RuntimeError(
        "Git working tree is not clean before Stage27-0 construction:\n\n"
        + status_before
    )

print("[PASS] Stage27-0 begins from the remotely verified Stage27-0A commit.")
print("[PASS] Git working tree is clean.")


# ======================================================================================
# 3. STAGE27-0A COMMITTED ARTIFACT GATE
# ======================================================================================

banner("STAGE27-0 :: VERIFY SEALED STAGE27-0A")

stage27_0a_dir = REPO / STAGE27_0A_REL

if not stage27_0a_dir.is_dir():
    raise RuntimeError("Stage27-0A directory is missing.")

actual_0a = sorted(
    p.name for p in stage27_0a_dir.iterdir()
    if p.is_file()
)

if actual_0a != sorted(REQUIRED_STAGE27_0A_FILES):
    raise RuntimeError(
        "Stage27-0A file set mismatch.\n"
        f"expected={sorted(REQUIRED_STAGE27_0A_FILES)}\n"
        f"actual={actual_0a}"
    )

tracked_0a = sorted(
    line.strip()
    for line in run(
        [
            "git",
            "ls-tree",
            "-r",
            "--name-only",
            "HEAD",
            "--",
            str(STAGE27_0A_REL),
        ]
    ).splitlines()
    if line.strip()
)

expected_tracked_0a = sorted(
    str(STAGE27_0A_REL / name)
    for name in REQUIRED_STAGE27_0A_FILES
)

if tracked_0a != expected_tracked_0a:
    raise RuntimeError(
        "Stage27-0A committed tree mismatch.\n"
        f"expected={expected_tracked_0a}\n"
        f"actual={tracked_0a}"
    )

print("[PASS] Exactly six Stage27-0A artifacts are committed at the parent.")


# ======================================================================================
# 4. OUTPUT DIRECTORY HARD GATE
# ======================================================================================

banner("STAGE27-0 :: OUTPUT DIRECTORY GATE")

if OUT_DIR.exists():
    raise RuntimeError(
        f"Stage27-0 output directory already exists:\n{OUT_DIR}\n\n"
        "Refusing to overwrite an existing protocol lock."
    )

OUT_DIR.mkdir(parents=True, exist_ok=False)

print("Created:", OUT_DIR)


# ======================================================================================
# 5. LOAD DURABLE PROVENANCE
# ======================================================================================

banner("STAGE27-0 :: LOAD DURABLE PROVENANCE")

tf_path = stage27_0a_dir / "temporal_feasibility.json"
tax_receipt_path = stage27_0a_dir / "taxonomy_receipt.json"
source_receipt_path = stage27_0a_dir / "source_artifact_receipts.json"

family_support_path = stage27_0a_dir / "family_day_support.csv"
benign_support_path = stage27_0a_dir / "benign_day_support.csv"

stage22_recipe_path = REPO / STAGE22_RECIPE_REL
stage22_devval_path = REPO / STAGE22_DEVVAL_REL
stage15_feature_path = REPO / STAGE15_FEATURE_REL
stage24_protocol_path = REPO / STAGE24_PROTOCOL_REL
stage24_bridge_path = REPO / STAGE24_BRIDGE_REL
stage24_source_path = REPO / STAGE24_SOURCE_REL

required_sources = [
    tf_path,
    tax_receipt_path,
    source_receipt_path,
    family_support_path,
    benign_support_path,
    stage22_recipe_path,
    stage22_devval_path,
    stage15_feature_path,
    stage24_protocol_path,
    stage24_bridge_path,
    stage24_source_path,
]

for path in required_sources:
    if not path.is_file():
        raise RuntimeError(f"Required source artifact missing: {path}")

temporal = load_json(tf_path)
taxonomy_receipt = load_json(tax_receipt_path)
stage27_0a_sources = load_json(source_receipt_path)

stage22_recipe = load_json(stage22_recipe_path)
stage22_devval = load_json(stage22_devval_path)
stage15_features = load_json(stage15_feature_path)

stage24_protocol = load_json(stage24_protocol_path)
stage24_bridge = load_json(stage24_bridge_path)
stage24_source = load_json(stage24_source_path)

family_support_rows = load_csv_rows(family_support_path)
benign_support_rows = load_csv_rows(benign_support_path)

print("[PASS] Durable Stage27/22/24 provenance loaded.")


# ======================================================================================
# 6. FEASIBILITY INVARIANT VERIFICATION
# ======================================================================================

banner("STAGE27-0 :: VERIFY FROZEN FIVE-FOLD GEOMETRY")

replacement = temporal[
    "replacement_day_atomic_family_specific_geometry"
]

eligible = replacement["eligible_folds"]
ineligible = replacement["structurally_ineligible_folds"]
descriptive_only = replacement["descriptive_only_eligible_folds"]

if eligible != EXPECTED_ELIGIBLE:
    raise RuntimeError(
        f"Eligible-fold identity mismatch.\n"
        f"expected={EXPECTED_ELIGIBLE}\n"
        f"actual={eligible}"
    )

if set(ineligible) != set(EXPECTED_INELIGIBLE):
    raise RuntimeError(
        f"Structurally-ineligible identity mismatch.\n"
        f"expected={EXPECTED_INELIGIBLE}\n"
        f"actual={ineligible}"
    )

if descriptive_only != EXPECTED_DESCRIPTIVE_ONLY:
    raise RuntimeError(
        f"Descriptive-only identity mismatch.\n"
        f"expected={EXPECTED_DESCRIPTIVE_ONLY}\n"
        f"actual={descriptive_only}"
    )

if replacement["eligible_fold_count"] != 5:
    raise RuntimeError("Expected exactly five eligible folds.")

if replacement["structurally_ineligible_fold_count"] != 2:
    raise RuntimeError("Expected exactly two structurally ineligible folds.")

if (
    replacement["minimum_positive_support_for_inferential_claim"]
    != MIN_INFERENTIAL_POSITIVE_SUPPORT
):
    raise RuntimeError("Stage27-0A support threshold mismatch.")

family_records = replacement["family_records"]

for family in EXPECTED_ELIGIBLE:
    rec = family_records[family]
    geom = rec["replacement_geometry"]

    if geom is None:
        raise RuntimeError(f"{family}: eligible fold has no geometry.")

    if int(geom["heldout_family_train_count"]) != 0:
        raise RuntimeError(f"{family}: held-out attack appears in TRAIN.")

    if int(geom["heldout_family_validation_count"]) != 0:
        raise RuntimeError(f"{family}: held-out attack appears in VALIDATION.")

    if int(geom["train_benign"]) <= 0:
        raise RuntimeError(f"{family}: TRAIN has no benign support.")

    if int(geom["train_known_attack"]) <= 0:
        raise RuntimeError(f"{family}: TRAIN has no known attack support.")

    if int(geom["validation_benign"]) <= 0:
        raise RuntimeError(f"{family}: VALIDATION has no benign support.")

    if int(geom["validation_known_attack"]) <= 0:
        raise RuntimeError(f"{family}: VALIDATION has no known attack support.")

    if int(geom["target_benign_available_same_day"]) <= 0:
        raise RuntimeError(f"{family}: target has no same-day benign support.")

    if int(geom["target_heldout_attack"]) <= 0:
        raise RuntimeError(f"{family}: target has no held-out attack support.")

print("Eligible folds:")
for family in EXPECTED_ELIGIBLE:
    g = family_records[family]["replacement_geometry"]
    print(
        f"  {family:18s} "
        f"TRAIN={','.join(g['train_days']):28s} "
        f"VALID={','.join(g['validation_days']):10s} "
        f"TARGET={','.join(g['target_days'])}"
    )

print()
print("Structurally ineligible:")
for family in EXPECTED_INELIGIBLE:
    print(
        f"  {family:18s} "
        f"{family_records[family]['reason_code']}"
    )

print("[PASS] Frozen Stage27-0A feasibility geometry verified.")


# ======================================================================================
# 7. STAGE22 FEATURE REPRESENTATION VERIFICATION
# ======================================================================================

banner("STAGE27-0 :: VERIFY STAGE22 70-FEATURE REPRESENTATION")

feature_policy = stage22_recipe["feature_policy"]

stage22_features = list(feature_policy["order"])
stage15_retained_features = list(stage15_features["retained_features"])

if feature_policy["count"] != 70:
    raise RuntimeError("Stage22 feature count is not 70.")

if len(stage22_features) != 70:
    raise RuntimeError("Stage22 feature-order list is not length 70.")

if stage15_features["retained_feature_count"] != 70:
    raise RuntimeError("Stage15 retained feature count is not 70.")

if stage22_features != stage15_retained_features:
    raise RuntimeError(
        "Stage22 feature order differs from Stage15 frozen retained-feature order."
    )

feature_order_sha256 = sha256_json(stage22_features)

print("Feature count :", len(stage22_features))
print("Feature hash  :", feature_order_sha256)
print("[PASS] Stage22/Stage15 exact 70-feature order agrees.")


# ======================================================================================
# 8. CICIDS2017 70-FEATURE SEMANTIC ADAPTER VERIFICATION
# ======================================================================================

banner("STAGE27-0 :: VERIFY CICIDS2017 70-FEATURE SEMANTIC ADAPTER")

bridge70 = stage24_bridge.get("bridge70")

if not isinstance(bridge70, dict):
    raise RuntimeError("Stage24 semantic bridge lacks bridge70.")

if int(bridge70.get("feature_count", -1)) != 70:
    raise RuntimeError("Stage24 bridge70 feature_count is not 70.")

corrected_mapping = bridge70.get("FLAG_CORRECTED_mapping")

if not isinstance(corrected_mapping, dict):
    raise RuntimeError(
        "Stage24 bridge70 FLAG_CORRECTED mapping is unavailable."
    )

if set(corrected_mapping.keys()) != set(stage22_features):
    missing = sorted(set(stage22_features) - set(corrected_mapping))
    extra = sorted(set(corrected_mapping) - set(stage22_features))
    raise RuntimeError(
        "FLAG_CORRECTED mapping does not cover the exact Stage22 feature set.\n"
        f"missing={missing}\n"
        f"extra={extra}"
    )

cicids2017_adapter = ordered_mapping(
    stage22_features,
    corrected_mapping,
)

adapter_sha256 = sha256_json(cicids2017_adapter)

print("Adapter features :", len(cicids2017_adapter))
print("Adapter hash     :", adapter_sha256)
print(
    "[PASS] Every frozen Stage22 predictor has an exact "
    "FLAG_CORRECTED CICIDS2017 semantic mapping."
)


# ======================================================================================
# 9. STAGE22 CLASS-WEIGHT FORMULA RECOVERY
# ======================================================================================

banner("STAGE27-0 :: RECOVER EXACT STAGE22 CLASS-WEIGHT FORMULA")

positive_weight_objects = recursively_find_key(
    stage22_devval,
    "positive_class_weight",
)

valid_positive_weight_objects = [
    obj
    for obj in positive_weight_objects
    if isinstance(obj, dict)
    and "formula" in obj
]

if not valid_positive_weight_objects:
    raise RuntimeError(
        "Could not recover Stage22 positive_class_weight formula "
        "from stage22_2b_devval_input_manifest.json."
    )

formula_candidates = {
    obj["formula"]
    for obj in valid_positive_weight_objects
}

if formula_candidates != {
    "development_benign / development_attack"
}:
    raise RuntimeError(
        f"Unexpected Stage22 class-weight formulas: {formula_candidates}"
    )

WEIGHT_FORMULA = "train_benign / train_attack"

print("Stage22 formula :", "development_benign / development_attack")
print("Stage27 formula :", WEIGHT_FORMULA)
print("[PASS] No class-weight formula is being invented.")


# ======================================================================================
# 10. RECOVER STAGE22 MODEL CONFIGURATIONS + CPU EXECUTION OVERRIDES
# ======================================================================================

banner("STAGE27-0 :: FREEZE MODEL INVENTORY")

classical = stage22_recipe["classical"]
members = classical["members"]

by_candidate = {
    item["candidate_id"]: item
    for item in members
}

for required_model in [PRIMARY_MODEL, REPLICATION_MODEL]:
    if required_model not in by_candidate:
        raise RuntimeError(
            f"Stage22 inherited recipe lacks {required_model}."
        )

xgb_source = copy.deepcopy(by_candidate["XGBOOST"])
lgbm_source = copy.deepcopy(by_candidate["LIGHTGBM"])

if xgb_source["configuration_id"] != "XGB_11":
    raise RuntimeError("Unexpected XGBoost configuration identity.")

if lgbm_source["configuration_id"] != "LGBM_11":
    raise RuntimeError("Unexpected LightGBM configuration identity.")

xgb_original_params = copy.deepcopy(xgb_source["parameters"])
lgbm_original_params = copy.deepcopy(lgbm_source["parameters"])

xgb_cpu_params = copy.deepcopy(xgb_original_params)
lgbm_cpu_params = copy.deepcopy(lgbm_original_params)

# Stage27 primary compute policy explicitly requires CPU.
xgb_cpu_params["device"] = "cpu"
xgb_cpu_params["tree_method"] = "hist"

lgbm_cpu_params["device_type"] = "cpu"

xgb_changed = {
    key: {
        "inherited": xgb_original_params.get(key),
        "stage27_execution": xgb_cpu_params.get(key),
    }
    for key in sorted(set(xgb_original_params) | set(xgb_cpu_params))
    if xgb_original_params.get(key) != xgb_cpu_params.get(key)
}

lgbm_changed = {
    key: {
        "inherited": lgbm_original_params.get(key),
        "stage27_execution": lgbm_cpu_params.get(key),
    }
    for key in sorted(set(lgbm_original_params) | set(lgbm_cpu_params))
    if lgbm_original_params.get(key) != lgbm_cpu_params.get(key)
}

if set(xgb_changed) != {"device"}:
    raise RuntimeError(
        f"Unexpected XGBoost CPU override set: {xgb_changed}"
    )

if set(lgbm_changed) != {"device_type"}:
    raise RuntimeError(
        f"Unexpected LightGBM CPU override set: {lgbm_changed}"
    )

print("Primary learner     : XGBOOST / XGB_11")
print("Replication learner : LIGHTGBM / LGBM_11")
print("XGBoost override    :", xgb_changed)
print("LightGBM override   :", lgbm_changed)
print("[PASS] Algorithmic hyperparameters unchanged; backend only -> CPU.")


# ======================================================================================
# 11. THRESHOLD PROCEDURE RECOVERY
# ======================================================================================

banner("STAGE27-0 :: FREEZE STAGE22 THRESHOLD PROCEDURE")

stage22_threshold = copy.deepcopy(stage22_recipe["threshold_protocol"])

expected_grid = stage22_threshold["grid"]

if float(expected_grid["start"]) != 0.01:
    raise RuntimeError("Unexpected Stage22 threshold-grid start.")

if float(expected_grid["stop"]) != 0.99:
    raise RuntimeError("Unexpected Stage22 threshold-grid stop.")

if float(expected_grid["step"]) != 0.01:
    raise RuntimeError("Unexpected Stage22 threshold-grid step.")

if float(stage22_threshold["standard"]) != 0.5:
    raise RuntimeError("Unexpected Stage22 standard threshold.")

print("Grid     : 0.01 .. 0.99 inclusive, step 0.01")
print("STANDARD : 0.50")
print("BALANCED : maximize F1 -> min FPR -> higher threshold")
print("SECURITY : FPR <= 0.05 -> maximize F2 -> min FPR -> higher threshold")
print("[PASS] Exact Stage22 threshold semantics recovered.")


# ======================================================================================
# 12. SUPPORT MATRICES
# ======================================================================================

banner("STAGE27-0 :: BUILD SUPPORT LOOKUPS")

family_support = OrderedDict()

for row in family_support_rows:
    family = row.get("Family") or row.get("family")
    if family in FAMILY_ORDER:
        family_support[family] = {
            k: int(v)
            for k, v in row.items()
            if k not in {"Family", "family"} and str(v).strip() != ""
        }

if set(family_support) != set(FAMILY_ORDER):
    raise RuntimeError(
        f"Could not reconstruct all seven family support rows: "
        f"{list(family_support)}"
    )

benign_by_day = {}

for row in benign_support_rows:
    day = row.get("day") or row.get("Day")
    value = row.get("benign_support") or row.get("Benign")
    if day and value not in (None, ""):
        benign_by_day[day] = int(value)

for required_day in [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
]:
    if required_day not in benign_by_day:
        raise RuntimeError(
            f"Missing benign support for {required_day}"
        )

print("Benign support:")
for day in ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]:
    print(f"  {day:10s}: {benign_by_day[day]:,}")


# ======================================================================================
# 13. FOLD-SPEC CONSTRUCTION
# ======================================================================================

banner("STAGE27-0 :: CONSTRUCT SEVEN-FAMILY FOLD SPEC")

fold_records = OrderedDict()

for family in FAMILY_ORDER:
    src = family_records[family]

    fold = OrderedDict()
    fold["held_out_family"] = family
    fold["status"] = src["status"]
    fold["support_status"] = src["support_status"]
    fold["target_positive_support"] = int(src["target_positive_support"])
    fold["reason_code"] = src["reason_code"]
    fold["reason"] = src["reason"]

    if src["replacement_geometry"] is None:
        fold["scientific_model_fit_authorized"] = False
        fold["replacement_geometry"] = None

    else:
        g = src["replacement_geometry"]

        target_day = g["target_days"][0]

        fold["scientific_model_fit_authorized"] = True
        fold["replacement_geometry"] = {
            "train_days": list(g["train_days"]),
            "validation_days": list(g["validation_days"]),
            "target_days": list(g["target_days"]),
            "held_out_family_train_count_expected": int(
                g["heldout_family_train_count"]
            ),
            "held_out_family_validation_count_expected": int(
                g["heldout_family_validation_count"]
            ),
            "train_benign_expected": int(g["train_benign"]),
            "train_known_attack_expected": int(
                g["train_known_attack"]
            ),
            "train_known_family_support": copy.deepcopy(
                g["train_known_family_support"]
            ),
            "validation_benign_expected": int(
                g["validation_benign"]
            ),
            "validation_known_attack_expected": int(
                g["validation_known_attack"]
            ),
            "validation_known_family_support": copy.deepcopy(
                g["validation_known_family_support"]
            ),
            "primary_target_benign_expected": int(
                g["target_benign_available_same_day"]
            ),
            "primary_target_heldout_attack_expected": int(
                g["target_heldout_attack"]
            ),
            "primary_target_rows_expected": (
                int(g["target_benign_available_same_day"])
                + int(g["target_heldout_attack"])
            ),
            "target_day": target_day,
        }

        expected_weight = (
            int(g["train_benign"])
            / int(g["train_known_attack"])
        )

        fold["pre_materialization_expected_class_weight"] = {
            "formula": WEIGHT_FORMULA,
            "value": expected_weight,
            "final_execution_value_must_be_recomputed": True,
            "reason": (
                "The final weight must use the realized frozen training "
                "membership after all preregistered input-validity gates."
            ),
        }

    fold_records[family] = fold

print("Fold statuses:")
for family, fold in fold_records.items():
    print(
        f"  {family:18s} "
        f"{fold['status']:30s} "
        f"positive_support={fold['target_positive_support']:,}"
    )


# ======================================================================================
# 14. SOURCE CONTENT-IDENTITY POLICY
# ======================================================================================

banner("STAGE27-0 :: FREEZE SOURCE CONTENT IDENTITY")

acquisition_records = stage24_source.get("acquisition_records", [])

if len(acquisition_records) < 5:
    raise RuntimeError(
        "Stage24 source contract contains too few acquisition records."
    )

source_assets = []

for rec in acquisition_records:
    if rec.get("status") != "VERIFIED":
        raise RuntimeError(
            "Stage24 source acquisition record is not VERIFIED."
        )

    if rec.get("sha256_match") is not True:
        raise RuntimeError(
            "Stage24 source acquisition SHA verification failed."
        )

    source_assets.append(
        {
            "day": rec["day"],
            "remote": rec["remote"],
            "repo_id": rec["repo_id"],
            "revision": rec["revision"],
            "sha256": rec["sha256"],
            "size_bytes": int(rec["size_bytes"]),
            "physical_rows": int(rec["physical_rows"]),
            "ordered_schema_sha256": rec[
                "ordered_schema_sha256"
            ],
        }
    )

print("Verified Stage24 source assets:", len(source_assets))

for rec in source_assets:
    print(
        f"  {rec['day']:10s} "
        f"{Path(rec['remote']).name:70s} "
        f"{rec['sha256'][:12]}..."
    )


# ======================================================================================
# 15. TAXONOMY.JSON
# ======================================================================================

taxonomy = OrderedDict(
    [
        ("stage", "Stage27-0"),
        (
            "status",
            "FROZEN_BEFORE_STAGE27_MODEL_FIT_INFERENCE_OR_TARGET_OPENING",
        ),
        ("created_at_utc", now_utc()),
        ("parent_commit", EXPECTED_PARENT),
        ("taxonomy_source", str(STAGE24_PROTOCOL_REL)),
        (
            "taxonomy_source_sha256",
            sha256_file(stage24_protocol_path),
        ),
        ("primary_family_order", FAMILY_ORDER),
        (
            "raw_label_to_family",
            copy.deepcopy(stage24_protocol["attack_family_taxonomy"]),
        ),
        (
            "heartbleed_policy",
            stage24_protocol["family_policy"]["heartbleed"],
        ),
        (
            "unknown_nonbenign_family_policy",
            stage24_protocol["family_policy"][
                "unknown_nonbenign_family"
            ],
        ),
        (
            "family_changes_after_stage27_0",
            "FORBIDDEN",
        ),
        (
            "structurally_ineligible_families",
            EXPECTED_INELIGIBLE,
        ),
        (
            "descriptive_only_families",
            EXPECTED_DESCRIPTIVE_ONLY,
        ),
    ]
)

write_json(OUT_DIR / "taxonomy.json", taxonomy)


# ======================================================================================
# 16. FOLD_SPEC.JSON
# ======================================================================================

fold_spec = OrderedDict(
    [
        ("stage", "Stage27-0"),
        (
            "design",
            "CHRONOLOGY_FIRST_ZERO_TRAINING_EXPOSURE_FAMILY_AUDIT",
        ),
        ("parent_commit", EXPECTED_PARENT),
        ("family_count", 7),
        ("eligible_fold_count", 5),
        ("structurally_ineligible_fold_count", 2),
        ("descriptive_only_eligible_fold_count", 1),
        (
            "loao_invariant",
            {
                "held_out_family_train_count_required": 0,
                "held_out_family_validation_count_required": 0,
                "violation": "FOLD_INVALID_HARD_STOP_BEFORE_FIT",
            },
        ),
        (
            "training_population_rule",
            {
                "negative_class": "BENIGN",
                "positive_class": (
                    "ONLY PRIMARY-SEVEN-TAXONOMY ATTACK FAMILIES "
                    "PRESENT IN THE FROZEN TRAIN DAYS EXCEPT THE HELD-OUT FAMILY"
                ),
                "heartbleed": "EXCLUDED_FROM_STAGE27_TRAINING",
                "unknown_nonbenign": "EXCLUDED_FROM_STAGE27_TRAINING",
            },
        ),
        (
            "validation_population_rule",
            {
                "negative_class": "BENIGN",
                "positive_class": (
                    "ONLY PRIMARY-SEVEN-TAXONOMY KNOWN ATTACK "
                    "FAMILIES PRESENT IN THE FROZEN VALIDATION DAY"
                ),
                "held_out_family": "MUST_BE_ABSENT",
                "heartbleed": "EXCLUDED",
                "unknown_nonbenign": "EXCLUDED",
            },
        ),
        ("folds", fold_records),
    ]
)

write_json(OUT_DIR / "fold_spec.json", fold_spec)


# ======================================================================================
# 17. TEMPORAL_GEOMETRY.JSON
# ======================================================================================

temporal_geometry = OrderedDict(
    [
        ("stage", "Stage27-0"),
        ("unit", "WEEKDAY_ATOMIC_PARTITIONS"),
        ("required_order", "TRAIN < VALIDATION < TARGET"),
        (
            "original_universal_geometry",
            copy.deepcopy(temporal["original_proposal"]),
        ),
        (
            "original_geometry_decision",
            "REJECTED_AS_UNIVERSAL_SEVEN_FOLD_DESIGN_BEFORE_MODEL_RESULTS",
        ),
        (
            "replacement_geometry_source",
            str(STAGE27_0A_REL / "temporal_feasibility.json"),
        ),
        (
            "replacement_geometry_source_sha256",
            sha256_file(tf_path),
        ),
        (
            "replacement_geometry_classification",
            "FEASIBILITY_CORRECTION_PRE_RESULT_NOT_PERFORMANCE_ADAPTATION",
        ),
        (
            "eligible_fold_days",
            {
                family: {
                    "train": fold_records[family][
                        "replacement_geometry"
                    ]["train_days"],
                    "validation": fold_records[family][
                        "replacement_geometry"
                    ]["validation_days"],
                    "target": fold_records[family][
                        "replacement_geometry"
                    ]["target_days"],
                }
                for family in EXPECTED_ELIGIBLE
            },
        ),
        (
            "important_semantic_limit",
            temporal["important_semantic_limit"],
        ),
        (
            "cross_fold_role_semantics",
            {
                "calendar_day_may_have_different_roles_in_different_folds": True,
                "example": (
                    "Thursday is TARGET for INFILTRATION/WEB_ATTACK folds "
                    "and VALIDATION for BOT/DDOS/PORT_SCAN folds."
                ),
                "target_opening_ledger_is_fold_specific": True,
                "cross_fold_result_driven_adaptation": "FORBIDDEN",
                "role_assignment_change_after_stage27_0": "FORBIDDEN",
            },
        ),
    ]
)

write_json(
    OUT_DIR / "temporal_geometry.json",
    temporal_geometry,
)


# ======================================================================================
# 18. TARGET_POPULATION_SPEC.JSON
# ======================================================================================

target_population_spec = OrderedDict(
    [
        ("stage", "Stage27-0"),
        (
            "population_identity",
            stage27_0a_sources[
                "effective_stage27_support_population"
            ],
        ),
        (
            "primary_isolation_target",
            {
                "role": "PRIMARY",
                "definition": (
                    "HELD_OUT_FAMILY_ATTACKS_FROM_FROZEN_TARGET_DAY "
                    "PLUS BENIGN_ROWS_FROM_THE_SAME_FROZEN_TARGET_DAY"
                ),
                "positive_class": "HELD_OUT_FAMILY_ONLY",
                "negative_class": "BENIGN_ONLY",
                "known_attacks_in_target_day": "EXCLUDED",
                "other_unknown_attacks": "EXCLUDED",
                "purpose": (
                    "Measure discrimination of a zero-training-exposure "
                    "attack family against temporally matched benign traffic."
                ),
            },
        ),
        (
            "secondary_operational_context_target",
            {
                "role": "SECONDARY_DESCRIPTIVE_ONLY",
                "definition": (
                    "HELD_OUT_FAMILY + KNOWN PRIMARY-TAXONOMY ATTACK "
                    "FAMILIES PRESENT IN TARGET CONTEXT + BENIGN"
                ),
                "binary_attack_label": (
                    "ALL INCLUDED PRIMARY-TAXONOMY ATTACKS -> 1; BENIGN -> 0"
                ),
                "family_identity_prediction_claim": False,
                "threshold_selection_allowed": False,
                "model_selection_allowed": False,
            },
        ),
        (
            "source_identity",
            {
                "dataset": "CICIDS2017",
                "effective_population_rows": 2_830_743,
                "source_contract": str(STAGE24_SOURCE_REL),
                "source_contract_sha256": sha256_file(
                    stage24_source_path
                ),
                "byte_exact_assets": source_assets,
                "materialization_policy": (
                    "Reuse an existing GitHub Release/local cache copy "
                    "only when its bytes match the frozen Stage24 SHA256 "
                    "and size exactly. No raw-data reconstruction or "
                    "source substitution is authorized."
                ),
                "stage20_compact_release_binary_corpus": (
                    "NOT_AUTHORIZED_AS_STAGE27_FAMILY_MEMBERSHIP_SOURCE"
                ),
            },
        ),
        (
            "eligible_primary_target_expected_counts",
            {
                family: {
                    "benign": fold_records[family][
                        "replacement_geometry"
                    ]["primary_target_benign_expected"],
                    "heldout_attack": fold_records[family][
                        "replacement_geometry"
                    ]["primary_target_heldout_attack_expected"],
                    "rows": fold_records[family][
                        "replacement_geometry"
                    ]["primary_target_rows_expected"],
                }
                for family in EXPECTED_ELIGIBLE
            },
        ),
        (
            "target_class_validity_gate",
            {
                "positive_required_gt": 0,
                "negative_required_gt": 0,
                "violation": (
                    "DO_NOT_COMPUTE_ROC_AUC_OR_PR_AUC; HARD_STOP "
                    "AND REPORT STRUCTURAL TARGET INVALIDITY"
                ),
            },
        ),
    ]
)

write_json(
    OUT_DIR / "target_population_spec.json",
    target_population_spec,
)


# ======================================================================================
# 19. MODEL_INVENTORY.JSON
# ======================================================================================

model_inventory = OrderedDict(
    [
        ("stage", "Stage27-0"),
        (
            "model_selection_using_stage27_results",
            "FORBIDDEN",
        ),
        (
            "primary_learner",
            {
                "candidate_id": "XGBOOST",
                "configuration_id": "XGB_11",
                "role": "PRIMARY",
                "inherited_parameters": xgb_original_params,
                "stage27_cpu_parameters": xgb_cpu_params,
                "backend_only_changes": xgb_changed,
                "algorithmic_hyperparameters_changed": False,
                "hyperparameter_search": False,
                "early_stopping": False,
                "validation_rows_in_fit": False,
                "target_rows_in_fit": False,
                "fit_from_scratch_per_eligible_fold": True,
            },
        ),
        (
            "replication_learner",
            {
                "candidate_id": "LIGHTGBM",
                "configuration_id": "LGBM_11",
                "role": "REPLICATION_SENSITIVITY",
                "inherited_parameters": lgbm_original_params,
                "stage27_cpu_parameters": lgbm_cpu_params,
                "backend_only_changes": lgbm_changed,
                "algorithmic_hyperparameters_changed": False,
                "hyperparameter_search": False,
                "early_stopping": False,
                "validation_rows_in_fit": False,
                "target_rows_in_fit": False,
                "fit_from_scratch_per_eligible_fold": True,
            },
        ),
        (
            "ensemble",
            {
                "used_as_primary_stage27_model": False,
                "reason": (
                    "Stage27 handoff specifies XGBoost primary and "
                    "LightGBM replication/sensitivity as separately "
                    "reported learners."
                ),
            },
        ),
        (
            "prohibited",
            [
                "OPTUNA",
                "HYPERPARAMETER_SEARCH",
                "ARCHITECTURE_SEARCH",
                "TARGET_FAMILY_TUNING",
                "NEW_BOOSTING_CONFIGURATION",
                "MODEL_SELECTION_FROM_LOAO_RESULTS",
            ],
        ),
    ]
)

write_json(
    OUT_DIR / "model_inventory.json",
    model_inventory,
)


# ======================================================================================
# 20. FIT_BUDGET.JSON
# ======================================================================================

fit_budget = OrderedDict(
    [
        ("stage", "Stage27-0"),
        ("eligible_folds", EXPECTED_ELIGIBLE),
        ("eligible_fold_count", 5),
        ("models_per_fold", 2),
        ("primary_representation_count", 1),
        ("primary_representation", "FULL_STAGE22_COMPATIBLE_70"),
        (
            "behavior_only_model_sensitivity",
            {
                "included": False,
                "decision_time": "BEFORE_ANY_STAGE27_MODEL_RESULTS",
                "reason": (
                    "Stage27 keeps FULL as the primary experiment and "
                    "does not add the optional Stage23 BEHAVIOR_ONLY "
                    "model branch. Behavioral similarity remains a "
                    "separate preregistered secondary analysis."
                ),
            },
        ),
        ("maximum_authorized_scientific_fits", 10),
        (
            "formula",
            "5 eligible folds * 2 model families * 1 representation = 10",
        ),
        ("fits_completed_at_stage27_0", 0),
        (
            "additional_fit_without_pre_result_amendment",
            "FORBIDDEN",
        ),
    ]
)

write_json(OUT_DIR / "fit_budget.json", fit_budget)


# ======================================================================================
# 21. COMPUTE_POLICY.JSON
# ======================================================================================

compute_policy = OrderedDict(
    [
        ("stage", "Stage27-0"),
        ("primary_execution_device", "CPU"),
        ("primary_gpu_budget_hours", 0),
        ("cuda_visible_devices", ""),
        ("xgboost_backend", "cpu"),
        ("lightgbm_backend", "cpu"),
        (
            "backend_override_semantics",
            (
                "GPU/CUDA -> CPU is an execution-backend change only. "
                "No algorithmic hyperparameter is changed."
            ),
        ),
        (
            "cpu_required_for",
            [
                "XGBoost",
                "LightGBM",
                "preprocessing",
                "fold construction",
                "inference",
                "metrics",
                "bootstrap",
                "figures",
            ],
        ),
        (
            "gpu_use_in_primary_stage27",
            "FORBIDDEN",
        ),
        (
            "future_gpu_extension",
            (
                "Requires a separately frozen pre-result amendment for "
                "a genuinely GPU-dependent extension."
            ),
        ),
    ]
)

write_json(
    OUT_DIR / "compute_policy.json",
    compute_policy,
)


# ======================================================================================
# 22. FEATURE_REPRESENTATION.JSON
# ======================================================================================

feature_representation = OrderedDict(
    [
        ("stage", "Stage27-0"),
        (
            "identity",
            "STAGE22_COMPATIBLE_RAW_FLOAT32_70_ON_CICIDS2017",
        ),
        ("feature_count", 70),
        ("feature_order", stage22_features),
        ("feature_order_sha256", feature_order_sha256),
        (
            "stage22_source",
            {
                "path": str(STAGE22_RECIPE_REL),
                "sha256": sha256_file(stage22_recipe_path),
            },
        ),
        (
            "stage15_feature_identity_source",
            {
                "path": str(STAGE15_FEATURE_REL),
                "sha256": sha256_file(stage15_feature_path),
            },
        ),
        (
            "cicids2017_semantic_adapter",
            {
                "source": str(STAGE24_BRIDGE_REL),
                "source_sha256": sha256_file(stage24_bridge_path),
                "variant": "FLAG_CORRECTED_mapping",
                "mapping": cicids2017_adapter,
                "mapping_sha256": adapter_sha256,
                "semantics": (
                    "The Stage24 mapping is used only as a deterministic "
                    "CICIDS2017 column-name/semantic adapter to materialize "
                    "the exact Stage22 feature order. Stage27 does not "
                    "inherit Stage24 bridge70 as a cross-dataset performance cell."
                ),
            },
        ),
        (
            "numeric_policy",
            {
                "source": "STAGE24_CICIDS2017_FROZEN_NUMERIC_POLICY",
                "parse_dtype": "float64",
                "positive_infinity": "CONVERT_TO_NAN",
                "negative_infinity": "CONVERT_TO_NAN",
                "other_nonnumeric_token": "FAIL_CLOSED",
                "explicit_imputation": "NONE",
                "scaling": "NONE",
                "final_model_matrix_dtype": "float32",
                "target_fitted_scaler": "FORBIDDEN",
                "target_fitted_imputer": "FORBIDDEN",
            },
        ),
        ("feature_search", "FORBIDDEN"),
        ("feature_addition", "FORBIDDEN"),
        ("feature_removal", "FORBIDDEN"),
        ("feature_reordering", "FORBIDDEN"),
        (
            "stage23_behavior_only_model_branch",
            "NOT_INCLUDED_IN_STAGE27_PRIMARY_MODEL_FITS",
        ),
    ]
)

write_json(
    OUT_DIR / "feature_representation.json",
    feature_representation,
)


# ======================================================================================
# 23. CLASS_WEIGHT_POLICY.JSON
# ======================================================================================

expected_weights = {}

for family in EXPECTED_ELIGIBLE:
    g = fold_records[family]["replacement_geometry"]

    train_benign = int(g["train_benign_expected"])
    train_attack = int(g["train_known_attack_expected"])

    if train_attack <= 0:
        raise RuntimeError(
            f"{family}: invalid expected train attack count."
        )

    expected_weights[family] = train_benign / train_attack

class_weight_policy = OrderedDict(
    [
        ("stage", "Stage27-0"),
        (
            "inherited_stage22_formula",
            "development_benign / development_attack",
        ),
        (
            "stage27_fold_formula",
            "train_benign / train_attack",
        ),
        (
            "statistics_source",
            "FINAL_FROZEN_TRAINING_MEMBERSHIP_FOR_THE_CURRENT_FOLD_ONLY",
        ),
        ("negative_class_weight", 1.0),
        (
            "positive_class_weight",
            "train_benign / train_attack",
        ),
        (
            "implementation",
            (
                "Construct fit-time sample_weight with BENIGN=1.0 and "
                "ATTACK=(train_benign/train_attack) for both XGBoost "
                "and LightGBM. This keeps the inherited algorithmic "
                "hyperparameter dictionaries unchanged."
            ),
        ),
        (
            "recompute_independently_per_fold",
            True,
        ),
        (
            "target_rows_in_weight_calculation",
            0,
        ),
        (
            "validation_rows_in_weight_calculation",
            0,
        ),
        (
            "weight_search",
            False,
        ),
        (
            "pre_materialization_expected_values",
            expected_weights,
        ),
        (
            "execution_rule",
            (
                "The actual per-fold receipt must recompute and freeze "
                "the value after final train membership is materialized. "
                "The feasibility-derived values above are audit expectations, "
                "not substitutes for execution-time recomputation."
            ),
        ),
        (
            "single_class_train_rule",
            "HARD_SCIENTIFIC_INFEASIBILITY_BEFORE_MODEL_FIT",
        ),
    ]
)

write_json(
    OUT_DIR / "class_weight_policy.json",
    class_weight_policy,
)


# ======================================================================================
# 24. THRESHOLD_POLICY.JSON
# ======================================================================================

threshold_policy = OrderedDict(
    [
        ("stage", "Stage27-0"),
        (
            "source",
            str(STAGE22_RECIPE_REL),
        ),
        (
            "source_sha256",
            sha256_file(stage22_recipe_path),
        ),
        (
            "selection_population",
            "KNOWN_FAMILY_VALIDATION_ONLY",
        ),
        (
            "held_out_family_in_threshold_selection",
            "FORBIDDEN",
        ),
        (
            "selected_separately_per",
            [
                "HELD_OUT_FAMILY_FOLD",
                "MODEL_FAMILY",
            ],
        ),
        (
            "standard",
            {
                "threshold": 0.5,
                "selected": False,
            },
        ),
        (
            "balanced",
            copy.deepcopy(stage22_threshold["balanced"]),
        ),
        (
            "security",
            copy.deepcopy(stage22_threshold["security"]),
        ),
        (
            "grid",
            {
                "construction": "INTEGER_PERCENT / 100",
                "integer_start": 1,
                "integer_stop_inclusive": 99,
                "integer_step": 1,
                "count": 99,
                "start": 0.01,
                "stop": 0.99,
                "step": 0.01,
            },
        ),
        (
            "target_threshold_search",
            "FORBIDDEN",
        ),
        (
            "post_target_threshold_change",
            "FORBIDDEN",
        ),
    ]
)

write_json(
    OUT_DIR / "threshold_policy.json",
    threshold_policy,
)


# ======================================================================================
# 25. METRIC_SPEC.JSON
# ======================================================================================

metric_spec = OrderedDict(
    [
        ("stage", "Stage27-0"),
        (
            "primary_isolation_target",
            {
                "co_primary_ranking_metrics": [
                    "ROC_AUC",
                    "PR_AUC",
                ],
                "operating_point_metrics": [
                    "RECALL",
                    "FPR",
                    "PRECISION",
                    "F1",
                    "TP",
                    "FP",
                    "TN",
                    "FN",
                ],
                "operating_points": [
                    "STANDARD",
                    "BALANCED",
                    "SECURITY",
                ],
            },
        ),
        (
            "auc_semantics",
            copy.deepcopy(
                stage24_protocol["metric_protocol"][
                    "auc_semantics"
                ]
            ),
        ),
        (
            "pr_auc_chance_anchor",
            {
                "prevalence_symbol": "pi_h",
                "formula": (
                    "target_heldout_attack / "
                    "(target_heldout_attack + target_benign)"
                ),
                "chance_pr_auc": "pi_h",
                "pr_excess": "PR_AUC - pi_h",
                "pr_lift": (
                    "PR_AUC / pi_h when pi_h > 0; "
                    "otherwise undefined"
                ),
            },
        ),
        (
            "known_family_control",
            {
                "mandatory": True,
                "population": (
                    "FROZEN KNOWN-FAMILY VALIDATION ATTACKS + "
                    "VALIDATION BENIGN"
                ),
                "purpose": (
                    "Verify each fold model remains functional on "
                    "attack families represented before its target."
                ),
            },
        ),
        (
            "novelty_generalization_gap",
            {
                "definition": "M_known - M_unseen",
                "primary_compatible_metrics": [
                    "ROC_AUC",
                    "PR_EXCESS",
                    "RECALL_STANDARD",
                    "RECALL_BALANCED",
                    "RECALL_SECURITY",
                ],
                "raw_pr_auc_gap": (
                    "REPORTABLE_DESCRIPTIVELY_BUT_INTERPRET_WITH "
                    "SEPARATE_PREVALENCE_ANCHORS"
                ),
                "causal_interpretation": False,
            },
        ),
        (
            "secondary_operational_context",
            {
                "status": "DESCRIPTIVE_ONLY",
                "ranking_and_threshold_metrics_allowed": True,
                "model_selection_allowed": False,
                "threshold_selection_allowed": False,
                "family_identity_prediction_claim": False,
            },
        ),
        (
            "undefined_metric_policy",
            (
                "Return/report NaN or undefined transparently. "
                "Do not alter denominators or smooth results."
            ),
        ),
        (
            "metric_addition_after_target_results",
            "FORBIDDEN_WITHOUT_EXPLICIT_POST_HOC_LABEL",
        ),
    ]
)

write_json(OUT_DIR / "metric_spec.json", metric_spec)


# ======================================================================================
# 26. SUPPORT_RULES.JSON
# ======================================================================================

support_rules = OrderedDict(
    [
        ("stage", "Stage27-0"),
        (
            "minimum_positive_support_for_inferential_family_claim",
            MIN_INFERENTIAL_POSITIVE_SUPPORT,
        ),
        (
            "below_minimum_policy",
            "DESCRIPTIVE_ONLY_NOT_DROPPED",
        ),
        (
            "eligible_inferential_families",
            [
                family
                for family in EXPECTED_ELIGIBLE
                if family not in EXPECTED_DESCRIPTIVE_ONLY
            ],
        ),
        (
            "eligible_descriptive_only_families",
            EXPECTED_DESCRIPTIVE_ONLY,
        ),
        (
            "structurally_ineligible_families",
            EXPECTED_INELIGIBLE,
        ),
        (
            "ineligible_family_policy",
            "RETAIN_IN_TAXONOMY_AND_REPORT_STRUCTURALLY_INELIGIBLE",
        ),
        (
            "support_threshold_change_after_results",
            "FORBIDDEN",
        ),
    ]
)

write_json(
    OUT_DIR / "support_rules.json",
    support_rules,
)


# ======================================================================================
# 27. UNCERTAINTY_SPEC.JSON
# ======================================================================================

uncertainty_spec = OrderedDict(
    [
        ("stage", "Stage27-0"),
        ("bootstrap_replicates", BOOTSTRAP_REPLICATES),
        ("seed", BOOTSTRAP_SEED),
        (
            "method",
            "STRATIFIED_ROW_BOOTSTRAP",
        ),
        (
            "sampling",
            (
                "WITH_REPLACEMENT_WITHIN BENIGN AND ATTACK STRATA, "
                "PRESERVING ORIGINAL STRATUM SIZE"
            ),
        ),
        (
            "confidence_interval",
            "PERCENTILE_2.5_97.5",
        ),
        (
            "metrics",
            [
                "ROC_AUC",
                "PR_AUC",
                "RECALL",
                "FPR",
                "NOVELTY_GENERALIZATION_GAP_WHERE_MATHEMATICALLY_COMPATIBLE",
            ],
        ),
        (
            "model_retraining_during_bootstrap",
            False,
        ),
        (
            "threshold_reselection_during_bootstrap",
            False,
        ),
        (
            "conditional_interpretation",
            (
                "Intervals quantify target-sampling uncertainty conditional "
                "on the fitted model. They do not include training-seed, "
                "model-selection, or population uncertainty."
            ),
        ),
        (
            "clustered_bootstrap",
            {
                "used": False,
                "reason": (
                    "No previously frozen valid session/time-block grouping "
                    "is being introduced into the Stage27 CICIDS2017 target "
                    "population. Stage27 therefore preregisters the row-stratified "
                    "bootstrap rather than inventing a grouping after results."
                ),
            },
        ),
        (
            "bootstrap_procedure_change_after_results",
            "FORBIDDEN",
        ),
    ]
)

write_json(
    OUT_DIR / "uncertainty_spec.json",
    uncertainty_spec,
)


# ======================================================================================
# 28. SIMILARITY_SPEC.JSON
# ======================================================================================

SIMILARITY_DESCRIPTORS = [
    "Flow Duration",
    "Tot Fwd Pkts",
    "Tot Bwd Pkts",
    "TotLen Fwd Pkts",
    "TotLen Bwd Pkts",
    "Pkt Len Mean",
    "Pkt Len Std",
    "Flow Byts/s",
    "Flow Pkts/s",
    "Down/Up Ratio",
    "Protocol",
]

for feature in SIMILARITY_DESCRIPTORS:
    if feature not in stage22_features:
        raise RuntimeError(
            f"Similarity descriptor not in frozen Stage22 features: {feature}"
        )

similarity_spec = OrderedDict(
    [
        ("stage", "Stage27-0"),
        (
            "status",
            "SECONDARY_PREREGISTERED_DESCRIPTIVE_ANALYSIS",
        ),
        (
            "scientific_question",
            (
                "Is unseen-family transfer descriptively associated with "
                "behavioral similarity between the held-out family and "
                "attack families represented in that fold's training set?"
            ),
        ),
        (
            "descriptor_features",
            SIMILARITY_DESCRIPTORS,
        ),
        (
            "descriptor_feature_sha256",
            sha256_json(SIMILARITY_DESCRIPTORS),
        ),
        (
            "preprocessing",
            {
                "fit_population": "CURRENT_FOLD_TRAINING_ROWS_ONLY",
                "missing_value_fill": "TRAINING_MEDIAN_PER_DESCRIPTOR",
                "centering": "TRAINING_MEAN_AFTER_MEDIAN_FILL",
                "scaling": "TRAINING_STANDARD_DEVIATION",
                "zero_standard_deviation_rule": (
                    "USE_SCALE_1.0_AND_RECORD_ZERO_VARIANCE_DESCRIPTOR"
                ),
                "target_fitted_preprocessing": False,
            },
        ),
        (
            "distance",
            {
                "family_centroid": (
                    "ARITHMETIC_MEAN_OF_STANDARDIZED_DESCRIPTOR_VECTOR "
                    "FOR EACH ATTACK FAMILY"
                ),
                "heldout_centroid": (
                    "ARITHMETIC_MEAN_OF_STANDARDIZED_DESCRIPTOR_VECTOR "
                    "FOR HELD_OUT_TARGET_ATTACK ROWS"
                ),
                "measure": "EUCLIDEAN_DISTANCE",
                "nearest_seen_distance": (
                    "MINIMUM EUCLIDEAN DISTANCE FROM HELDOUT CENTROID "
                    "TO ANY ATTACK-FAMILY CENTROID PRESENT IN TRAINING"
                ),
                "nearest_seen_family": (
                    "LEXICOGRAPHICALLY_SMALLEST_FAMILY_ON_EXACT_DISTANCE_TIE"
                ),
                "similarity_score": "1 / (1 + nearest_seen_distance)",
                "benign_distance": (
                    "EUCLIDEAN_DISTANCE_TO_TRAINING_BENIGN_CENTROID"
                ),
            },
        ),
        (
            "minimum_seen_family_support_for_centroid",
            1,
        ),
        (
            "association_reporting",
            {
                "figure": (
                    "NEAREST_SEEN_SIMILARITY_VS_UNSEEN_RANKING_PERFORMANCE"
                ),
                "statistical_claim": (
                    "EXPLORATORY_DESCRIPTIVE_ASSOCIATION_ONLY"
                ),
                "causal_claim": False,
                "strong_correlation_inference": False,
            },
        ),
        (
            "similarity_definition_change_after_loao_results",
            "FORBIDDEN",
        ),
    ]
)

write_json(
    OUT_DIR / "similarity_spec.json",
    similarity_spec,
)


# ======================================================================================
# 29. TARGET_OPENING_LEDGER.JSON
# ======================================================================================

ledger_families = OrderedDict()

for family in FAMILY_ORDER:
    if family in EXPECTED_INELIGIBLE:
        ledger_families[family] = {
            "status": "STRUCTURALLY_INELIGIBLE",
            "opening_budget": 0,
            "openings_consumed": 0,
            "scientific_target_inference_authorized": False,
        }
    else:
        ledger_families[family] = {
            "status": "SEALED",
            "opening_budget": 1,
            "openings_consumed": 0,
            "scientific_target_inference_authorized": (
                "ONLY_AFTER_FOLD_MEMBERSHIP_MODEL_AND_THRESHOLDS_ARE_FROZEN"
            ),
        }

target_opening_ledger = OrderedDict(
    [
        ("stage", "Stage27-0"),
        (
            "opening_definition",
            (
                "A fold target opening occurs when that fold's frozen "
                "target predictor matrix is first supplied to either "
                "frozen Stage27 learner for scientific target inference."
            ),
        ),
        (
            "same_opening_model_rule",
            (
                "Both already-frozen learners and all already-frozen "
                "operating points for a fold must be evaluated from the "
                "same authorized target opening."
            ),
        ),
        (
            "cross_fold_calendar_day_note",
            (
                "The ledger is fold-specific. A calendar day may serve "
                "as known-family validation in another frozen fold. "
                "Such cross-fold role reuse cannot change any fold's "
                "already-frozen membership, model recipe, weighting rule, "
                "threshold rule, or target definition."
            ),
        ),
        ("families", ledger_families),
        ("eligible_target_opening_budget_total", 5),
        ("openings_consumed_at_stage27_0", 0),
        (
            "reopening_for_adaptation",
            "FORBIDDEN",
        ),
    ]
)

write_json(
    OUT_DIR / "target_opening_ledger.json",
    target_opening_ledger,
)


# ======================================================================================
# 30. INTERPRETATION_MATRIX.JSON
# ======================================================================================

interpretation_matrix = OrderedDict(
    [
        ("stage", "Stage27-0"),
        (
            "success_condition",
            (
                "Stage27 succeeds if the preregistered LOAO audit produces "
                "valid, leakage-controlled, support-aware estimates of "
                "unseen-family discrimination and frozen operating-point "
                "transfer. Scientific success does not require survival "
                "or collapse."
            ),
        ),
        (
            "permitted_outcome_interpretations",
            [
                {
                    "outcome": "ELIGIBLE_FAMILIES_NEAR_CHANCE",
                    "interpretation": (
                        "Little unseen-family ranking signal was detected "
                        "under the tested LOAO protocol."
                    ),
                },
                {
                    "outcome": "SOME_FAMILIES_SURVIVE_WHILE_OTHERS_COLLAPSE",
                    "interpretation": (
                        "Unseen-family transfer is family-dependent and "
                        "appears stronger for some traffic behaviors."
                    ),
                },
                {
                    "outcome": "RANKING_SURVIVES_THRESHOLD_RECALL_COLLAPSES",
                    "interpretation": (
                        "Novel-family ranking information survives, but "
                        "frozen operating points do not translate into "
                        "reliable detection."
                    ),
                },
                {
                    "outcome": "XGB_AND_LGBM_AGREE",
                    "interpretation": (
                        "The family-transfer pattern is less likely to be "
                        "specific to one of the two preregistered learners."
                    ),
                },
                {
                    "outcome": "XGB_AND_LGBM_DISAGREE",
                    "interpretation": (
                        "Novel-family conclusions depend materially on "
                        "learner choice."
                    ),
                },
                {
                    "outcome": "BROAD_TRANSFER_ACROSS_ELIGIBLE_FAMILIES",
                    "interpretation": (
                        "Broad unseen-family discrimination survives across "
                        "the structurally executable benchmark folds."
                    ),
                },
                {
                    "outcome": "SIMILAR_FAMILIES_TRANSFER_BETTER",
                    "interpretation": (
                        "LOAO performance is descriptively associated with "
                        "the preregistered behavioral-similarity measure."
                    ),
                },
                {
                    "outcome": "UNEXPECTED_FAMILY_TRANSFER",
                    "interpretation": (
                        "Retain and investigate without changing the "
                        "preregistered protocol."
                    ),
                },
            ],
        ),
        (
            "valid_high_level_outcomes",
            [
                "UNIVERSAL_COLLAPSE_WITHIN_ELIGIBLE_FOLDS",
                "SELECTIVE_FAMILY_TRANSFER",
                "VOLUMETRIC_TRANSFER",
                "STEALTH_FAMILY_TRANSFER",
                "RANKING_THRESHOLD_DIVERGENCE",
                "LEARNER_DEPENDENCE",
                "BROAD_UNSEEN_FAMILY_TRANSFER",
            ],
        ),
        ("expected_result_requirement", None),
    ]
)

write_json(
    OUT_DIR / "interpretation_matrix.json",
    interpretation_matrix,
)


# ======================================================================================
# 31. PROHIBITED_CLAIMS.JSON
# ======================================================================================

prohibited_claims = OrderedDict(
    [
        ("stage", "Stage27-0"),
        (
            "forbidden_claims",
            [
                "The model detects zero-day attacks.",
                "ROC-AUC approximately 0.5 proves the model is a pure signature matcher.",
                "The model learned zero malicious concepts.",
                "Supervised IDS cannot detect zero-days.",
                "ROC-AUC above 0.8 proves genuine causal behavioral understanding.",
                (
                    "A family failed because it is stealthy, unless the frozen "
                    "behavioral analysis actually supports that characterization."
                ),
            ],
        ),
        (
            "preferred_terminology",
            [
                "unseen attack family",
                "zero-training-exposure family",
                "attack-family novelty",
                "LOAO generalization",
            ],
        ),
        (
            "zero_day_equivalence_claim",
            "FORBIDDEN",
        ),
    ]
)

write_json(
    OUT_DIR / "prohibited_claims.json",
    prohibited_claims,
)


# ======================================================================================
# 32. INHERITED_RECEIPTS.JSON
# ======================================================================================

receipt_paths = [
    stage27_0a_dir / "family_day_support.csv",
    stage27_0a_dir / "benign_day_support.csv",
    stage27_0a_dir / "taxonomy_receipt.json",
    stage27_0a_dir / "source_artifact_receipts.json",
    stage27_0a_dir / "temporal_feasibility.json",
    stage27_0a_dir / "feasibility_audit.md",
    stage22_recipe_path,
    stage22_devval_path,
    stage15_feature_path,
    stage24_protocol_path,
    stage24_bridge_path,
    stage24_source_path,
]

inherited_receipts = OrderedDict(
    [
        ("stage", "Stage27-0"),
        ("scientific_parent_commit", EXPECTED_PARENT),
        (
            "source_artifacts",
            [
                file_receipt(path)
                for path in receipt_paths
            ],
        ),
        (
            "stage27_0a_remote_anchor",
            {
                "commit": EXPECTED_PARENT,
                "required_artifact_count": 6,
                "verified_before_stage27_0": True,
            },
        ),
        (
            "provenance_decisions",
            {
                "family_support_population": (
                    "STAGE24_FULL_EFFECTIVE_CICIDS2017_POPULATION"
                ),
                "stage20_compact_release_family_membership": (
                    "NOT_AUTHORIZED_BINARY_ONLY"
                ),
                "feature_identity": (
                    "EXACT_STAGE22_70_FEATURE_ORDER"
                ),
                "cicids2017_column_semantics": (
                    "STAGE24_FLAG_CORRECTED_70_FEATURE_MAPPING"
                ),
                "class_weight_formula": (
                    "EXACT_STAGE22_DEVELOPMENT_BENIGN_DIVIDED_BY_ATTACK_FORMULA"
                ),
                "threshold_policy": (
                    "EXACT_STAGE22_VALIDATION_ONLY_THRESHOLD_POLICY"
                ),
            },
        ),
    ]
)

write_json(
    OUT_DIR / "inherited_receipts.json",
    inherited_receipts,
)


# ======================================================================================
# 33. ANTI-ADAPTATION CONTENT ADDED TO FREEZE RECORD
# ======================================================================================

ANTI_ADAPTATION = [
    "CHANGE_FAMILY_TAXONOMY",
    "DROP_INCONVENIENT_FAMILY",
    "MOVE_TARGET_ATTACKS_INTO_TRAINING",
    "CHANGE_TARGET_BENIGN_CONTROLS",
    "TARGET_DERIVED_FEATURE_SELECTION",
    "TARGET_DERIVED_SCALING",
    "TARGET_DERIVED_CLASS_WEIGHTING",
    "TARGET_DERIVED_THRESHOLDS",
    "NEW_HYPERPARAMETER_SEARCH",
    "ARCHITECTURE_SEARCH",
    "ADD_OR_REMOVE_REPRESENTATIONS_AFTER_RESULTS",
    "REDEFINE_BEHAVIORAL_SIMILARITY_AFTER_RESULTS",
    "CHANGE_SUPPORT_MINIMUM",
    "CHANGE_BOOTSTRAP_PROCEDURE",
    "REOPEN_SPENT_TARGET",
    "ADD_FAVORABLE_METRICS_AFTER_RESULTS_WITHOUT_POST_HOC_LABEL",
    "CHANGE_FOLD_GEOMETRY_AFTER_RESULTS",
    "CHANGE_MODEL_FAMILY_AFTER_RESULTS",
]


# ======================================================================================
# 34. FREEZE_RECORD.JSON
#
# The hashes below cover the other 17 protocol files.
# freeze_record.json itself is Git-anchored by the commit containing it.
# ======================================================================================

banner("STAGE27-0 :: BUILD FREEZE RECORD")

non_freeze_files = [
    name
    for name in REQUIRED_STAGE27_0_FILES
    if name != "freeze_record.json"
]

artifact_hashes = OrderedDict()

for name in non_freeze_files:
    path = OUT_DIR / name

    if not path.is_file():
        raise RuntimeError(
            f"Protocol artifact missing before freeze record: {name}"
        )

    artifact_hashes[name] = {
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }

freeze_record = OrderedDict(
    [
        ("stage", "Stage27-0"),
        (
            "status",
            "COMPLETE_PROTOCOL_CONTENT_FROZEN_PENDING_GIT_COMMIT_PUSH_REMOTE_VERIFICATION",
        ),
        ("created_at_utc", now_utc()),
        ("scientific_parent_commit", EXPECTED_PARENT),
        (
            "protocol_directory",
            str(OUT_REL),
        ),
        (
            "required_artifact_count",
            len(REQUIRED_STAGE27_0_FILES),
        ),
        (
            "eligible_fold_count",
            5,
        ),
        (
            "structurally_ineligible_fold_count",
            2,
        ),
        (
            "authorized_fit_budget",
            10,
        ),
        ("primary_execution_device", "CPU"),
        ("primary_gpu_budget_hours", 0),
        ("model_fits_completed", 0),
        ("model_inference_completed", 0),
        ("threshold_selections_completed", 0),
        ("target_openings_consumed", 0),
        ("bootstrap_replicates_executed", 0),
        (
            "artifact_hashes_before_freeze_record",
            artifact_hashes,
        ),
        (
            "anti_adaptation_after_remote_freeze",
            ANTI_ADAPTATION,
        ),
        (
            "git_anchor_semantics",
            (
                "The immutable Git commit containing this freeze_record.json "
                "is the Stage27-0 protocol anchor. The commit SHA is not "
                "self-embedded to avoid recursive commit-content dependency."
            ),
        ),
        (
            "next_authorized_step_after_remote_verification",
            (
                "STAGE27-1A SOURCE MATERIALIZATION AND FOLD MEMBERSHIP "
                "CONSTRUCTION ONLY. NO MODEL FIT UNTIL ALL ELIGIBLE FOLD "
                "MEMBERSHIP RECEIPTS PASS HELD-OUT AND VALIDATION EXCLUSION GATES."
            ),
        ),
    ]
)

write_json(
    OUT_DIR / "freeze_record.json",
    freeze_record,
)


# ======================================================================================
# 35. FINAL ARTIFACT-SET VALIDATION
# ======================================================================================

banner("STAGE27-0 :: FINAL LOCAL ARTIFACT VALIDATION")

actual_outputs = sorted(
    p.name
    for p in OUT_DIR.iterdir()
    if p.is_file()
)

expected_outputs = sorted(REQUIRED_STAGE27_0_FILES)

if actual_outputs != expected_outputs:
    raise RuntimeError(
        "Stage27-0 output file set mismatch.\n"
        f"expected={expected_outputs}\n"
        f"actual={actual_outputs}"
    )

print(f"Required artifacts: {len(expected_outputs)}")
print()

for name in REQUIRED_STAGE27_0_FILES:
    path = OUT_DIR / name

    print(
        f"{name:34s} "
        f"bytes={path.stat().st_size:>9,} "
        f"sha256={sha256_file(path)}"
    )

print()
print("[PASS] Exactly 18 prescribed Stage27-0 artifacts exist.")


# ======================================================================================
# 36. SCIENTIFIC CONTENT READBACK
# ======================================================================================

banner("STAGE27-0 :: SCIENTIFIC READBACK")

read_fold_spec = load_json(OUT_DIR / "fold_spec.json")
read_fit_budget = load_json(OUT_DIR / "fit_budget.json")
read_compute = load_json(OUT_DIR / "compute_policy.json")
read_weight = load_json(OUT_DIR / "class_weight_policy.json")
read_uncertainty = load_json(OUT_DIR / "uncertainty_spec.json")
read_ledger = load_json(OUT_DIR / "target_opening_ledger.json")
read_features = load_json(OUT_DIR / "feature_representation.json")

if read_fold_spec["eligible_fold_count"] != 5:
    raise RuntimeError("Readback eligible-fold count mismatch.")

if read_fit_budget["maximum_authorized_scientific_fits"] != 10:
    raise RuntimeError("Readback fit budget mismatch.")

if read_compute["primary_execution_device"] != "CPU":
    raise RuntimeError("Readback compute device mismatch.")

if read_compute["primary_gpu_budget_hours"] != 0:
    raise RuntimeError("Readback GPU budget mismatch.")

if read_weight["stage27_fold_formula"] != WEIGHT_FORMULA:
    raise RuntimeError("Readback class-weight formula mismatch.")

if read_uncertainty["bootstrap_replicates"] != 2000:
    raise RuntimeError("Readback bootstrap replicate mismatch.")

if read_uncertainty["seed"] != 42:
    raise RuntimeError("Readback bootstrap seed mismatch.")

if read_features["feature_count"] != 70:
    raise RuntimeError("Readback feature count mismatch.")

if read_features["feature_order_sha256"] != feature_order_sha256:
    raise RuntimeError("Readback feature hash mismatch.")

for family in EXPECTED_ELIGIBLE:
    ledger = read_ledger["families"][family]

    if ledger["opening_budget"] != 1:
        raise RuntimeError(
            f"{family}: target opening budget != 1"
        )

    if ledger["openings_consumed"] != 0:
        raise RuntimeError(
            f"{family}: target ledger already consumed."
        )

for family in EXPECTED_INELIGIBLE:
    if (
        read_ledger["families"][family]["status"]
        != "STRUCTURALLY_INELIGIBLE"
    ):
        raise RuntimeError(
            f"{family}: ineligible ledger status mismatch."
        )

print("Eligible folds            : 5")
print("Structurally ineligible   : 2")
print("Descriptive-only eligible : 1")
print("Authorized model fits     : 10")
print("Feature count             : 70")
print("Class weight              : train_benign / train_attack")
print("Bootstrap                 : 2000 replicates, seed 42")
print("Target openings consumed  : 0")
print("GPU budget                : 0 hours")

print()
print("[PASS] Stage27-0 scientific readback passed.")


# ======================================================================================
# 37. NON-COMPUTATION ASSERTIONS
# ======================================================================================

banner("STAGE27-0 :: NON-COMPUTATION ASSERTIONS")

scientific_counters = OrderedDict(
    [
        ("corpus_archives_downloaded", 0),
        ("raw_dataset_recreated", 0),
        ("model_fits", 0),
        ("model_inference", 0),
        ("model_probabilities_generated", 0),
        ("threshold_searches", 0),
        ("target_openings", 0),
        ("bootstrap_replicates_executed", 0),
        ("gpu_hours", 0),
    ]
)

for key, value in scientific_counters.items():
    print(f"{key:34s}: {value}")

if any(value != 0 for value in scientific_counters.values()):
    raise RuntimeError(
        "Stage27-0 unexpectedly performed scientific execution."
    )

print()
print("[PASS] Stage27-0 remains a pure preregistration/protocol-lock stage.")


# ======================================================================================
# 38. STAGE EXACTLY THE 18 PROTOCOL ARTIFACTS
# ======================================================================================

banner("STAGE27-0 :: STAGE EXACT PROTOCOL ARTIFACT SET")

for name in REQUIRED_STAGE27_0_FILES:
    run(
        [
            "git",
            "add",
            "--",
            str(OUT_REL / name),
        ]
    )

staged_names = sorted(
    line.strip()
    for line in run(
        [
            "git",
            "diff",
            "--cached",
            "--name-only",
            "--diff-filter=ACMR",
        ]
    ).splitlines()
    if line.strip()
)

expected_staged = sorted(
    str(OUT_REL / name)
    for name in REQUIRED_STAGE27_0_FILES
)

print("Expected staged files:")
for path in expected_staged:
    print(" ", path)

print()
print("Actual staged files:")
for path in staged_names:
    print(" ", path)

if staged_names != expected_staged:
    raise RuntimeError(
        "Unexpected staged file set.\n"
        f"expected={expected_staged}\n"
        f"actual={staged_names}"
    )

unstaged = run(
    [
        "git",
        "diff",
        "--name-only",
    ]
)

if unstaged.strip():
    raise RuntimeError(
        "Unexpected unstaged tracked changes remain:\n"
        + unstaged
    )

status_after = run(
    [
        "git",
        "status",
        "--porcelain=v1",
        "--untracked-files=all",
    ]
)

allowed_status = sorted(
    f"A  {OUT_REL / name}"
    for name in REQUIRED_STAGE27_0_FILES
)

actual_status = sorted(
    line
    for line in status_after.splitlines()
    if line.strip()
)

if actual_status != allowed_status:
    raise RuntimeError(
        "Git status contains something other than the exact "
        "Stage27-0 staged artifact set.\n\n"
        f"Expected:\n" + "\n".join(allowed_status)
        + "\n\nActual:\n" + "\n".join(actual_status)
    )

print()
print("[PASS] Exactly 18 Stage27-0 protocol artifacts are staged.")


# ======================================================================================
# 39. FINAL STOP
# ======================================================================================

banner("STAGE27-0 :: LOCAL PROTOCOL CONSTRUCTION COMPLETE")

print("Scientific parent :", EXPECTED_PARENT)
print("Protocol path     :", OUT_REL)
print("Artifacts         :", "18 / 18")
print()
print("Eligible folds:")
for family in EXPECTED_ELIGIBLE:
    print("  -", family)

print()
print("Structurally ineligible:")
for family in EXPECTED_INELIGIBLE:
    print("  -", family)

print()
print("Descriptive-only:")
for family in EXPECTED_DESCRIPTIVE_ONLY:
    print("  -", family)

print()
print("Primary learner      : XGBoost / XGB_11 / CPU")
print("Replication learner  : LightGBM / LGBM_11 / CPU")
print("Representation       : FULL Stage22-compatible raw float32 70")
print("Fit budget           : 10")
print("Class weighting      : train_benign / train_attack")
print("Threshold source     : exact Stage22 validation-only policy")
print("Bootstrap            : 2000 stratified row replicates / seed 42")
print("Target ledger        : 0 / 5 eligible openings consumed")
print("GPU                   : 0")
print()
print("Git state             : EXACT 18 STAGE27-0 FILES STAGED")
print("Commit performed      : NO")
print("Push performed        : NO")
print()
print("NEXT GATE:")
print(
    "  Inspect this output. If all checks pass, commit/push the exact "
    "Stage27-0 artifact set and remotely verify the new main SHA."
)
print()
print("NO MODEL FIT IS AUTHORIZED YET.")


STAGE27-0 :: REPOSITORY AND SCIENTIFIC-PARENT GATE
Expected parent : da1cc26b2759f43e64122bb8ff7a21d112dc420f
Local HEAD      : da1cc26b2759f43e64122bb8ff7a21d112dc420f
origin/main     : da1cc26b2759f43e64122bb8ff7a21d112dc420f
[PASS] Stage27-0 begins from the remotely verified Stage27-0A commit.
[PASS] Git working tree is clean.

STAGE27-0 :: VERIFY SEALED STAGE27-0A
[PASS] Exactly six Stage27-0A artifacts are committed at the parent.

STAGE27-0 :: OUTPUT DIRECTORY GATE
Created: /kaggle/working/ids2018-validation-safe-ablation/results/stage27_loao_unseen_attack/stage27_0_protocol_lock

STAGE27-0 :: LOAD DURABLE PROVENANCE
[PASS] Durable Stage27/22/24 provenance loaded.

STAGE27-0 :: VERIFY FROZEN FIVE-FOLD GEOMETRY
Eligible folds:
  BOT                TRAIN=Monday,Tuesday,Wednesday     VALID=Thursday   TARGET=Friday
  DDOS               TRAIN=Monday,Tuesday,Wednesday     VALID=Thursday   TARGET=Friday
  INFILTRATION       TRAIN=Monday,Tuesday               VALID=Wednesday  TARGET=Thu

In [12]:
# ======================================================================================
# STAGE27-0 — FINAL ARTIFACT-COUNT CONSISTENCY AUDIT
#
# READ ONLY
# ZERO WRITES
# ZERO FITS
# ZERO INFERENCE
# ZERO TARGET OPENINGS
# ======================================================================================

from pathlib import Path
import json
import subprocess
import re

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

OUT_REL = Path(
    "results/stage27_loao_unseen_attack/"
    "stage27_0_protocol_lock"
)

OUT_DIR = REPO / OUT_REL

EXPECTED_PARENT = (
    "da1cc26b2759f43e64122bb8ff7a21d112dc420f"
)

EXPECTED_FILES = [
    "taxonomy.json",
    "fold_spec.json",
    "temporal_geometry.json",
    "target_population_spec.json",
    "model_inventory.json",
    "fit_budget.json",
    "compute_policy.json",
    "feature_representation.json",
    "class_weight_policy.json",
    "threshold_policy.json",
    "metric_spec.json",
    "support_rules.json",
    "uncertainty_spec.json",
    "similarity_spec.json",
    "target_opening_ledger.json",
    "interpretation_matrix.json",
    "prohibited_claims.json",
    "inherited_receipts.json",
    "freeze_record.json",
]


def run(cmd):
    p = subprocess.run(
        cmd,
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    if p.returncode != 0:
        raise RuntimeError(
            f"Command failed: {' '.join(cmd)}\n"
            f"STDOUT:\n{p.stdout}\n"
            f"STDERR:\n{p.stderr}"
        )

    return p.stdout.strip()


print("=" * 110)
print("STAGE27-0 :: FINAL ARTIFACT-COUNT CONSISTENCY AUDIT")
print("=" * 110)

# ------------------------------------------------------------------
# 1. Parent must still be unchanged
# ------------------------------------------------------------------

head = run(["git", "rev-parse", "HEAD"])
origin = run(["git", "rev-parse", "origin/main"])

print("Expected parent :", EXPECTED_PARENT)
print("Local HEAD      :", head)
print("origin/main     :", origin)

if head != EXPECTED_PARENT:
    raise RuntimeError("Local HEAD changed unexpectedly.")

if origin != EXPECTED_PARENT:
    raise RuntimeError("origin/main changed unexpectedly.")

print("[PASS] Scientific parent unchanged.")


# ------------------------------------------------------------------
# 2. Physical file count
# ------------------------------------------------------------------

actual_files = sorted(
    p.name
    for p in OUT_DIR.iterdir()
    if p.is_file()
)

expected_files = sorted(EXPECTED_FILES)

print()
print("Expected protocol files :", len(expected_files))
print("Actual protocol files   :", len(actual_files))

if actual_files != expected_files:
    raise RuntimeError(
        "Protocol file universe mismatch.\n"
        f"Expected={expected_files}\n"
        f"Actual={actual_files}"
    )

print("[PASS] Exactly 19 protocol files exist.")


# ------------------------------------------------------------------
# 3. Staged file count
# ------------------------------------------------------------------

staged = sorted(
    x
    for x in run(
        [
            "git",
            "diff",
            "--cached",
            "--name-only",
        ]
    ).splitlines()
    if x.strip()
)

expected_staged = sorted(
    str(OUT_REL / name)
    for name in EXPECTED_FILES
)

print()
print("Expected staged files :", len(expected_staged))
print("Actual staged files   :", len(staged))

if staged != expected_staged:
    raise RuntimeError(
        "Staged file universe mismatch.\n"
        f"Expected={expected_staged}\n"
        f"Actual={staged}"
    )

print("[PASS] Exactly 19 protocol files are staged.")


# ------------------------------------------------------------------
# 4. Inspect freeze_record persisted count
# ------------------------------------------------------------------

freeze_path = OUT_DIR / "freeze_record.json"

with freeze_path.open("r", encoding="utf-8") as f:
    freeze = json.load(f)

print()
print(
    "freeze_record.required_artifact_count :",
    freeze.get("required_artifact_count"),
)

artifact_hashes = freeze.get(
    "artifact_hashes_before_freeze_record",
    {}
)

print(
    "freeze_record non-self artifact hashes:",
    len(artifact_hashes),
)

if freeze.get("required_artifact_count") != 19:
    raise RuntimeError(
        "freeze_record.json incorrectly records the "
        "required artifact count."
    )

if len(artifact_hashes) != 18:
    raise RuntimeError(
        "freeze_record.json should hash the other 18 files "
        "before adding itself."
    )

expected_hashed_names = sorted(
    name
    for name in EXPECTED_FILES
    if name != "freeze_record.json"
)

actual_hashed_names = sorted(
    artifact_hashes.keys()
)

if actual_hashed_names != expected_hashed_names:
    raise RuntimeError(
        "freeze_record non-self hash universe mismatch."
    )

print(
    "[PASS] freeze_record correctly records 19 total "
    "artifacts and hashes the other 18."
)


# ------------------------------------------------------------------
# 5. Search persisted artifacts for suspicious artifact-count claims
# ------------------------------------------------------------------

print()
print("Searching persisted protocol files for count wording...")

hits = []

patterns = [
    re.compile(r"\b18\s*/\s*18\b", re.I),
    re.compile(r"exactly\s+18", re.I),
    re.compile(r"required[_ ]artifact[_ ]count[^0-9]{0,20}18", re.I),
    re.compile(r"artifact[_ ]count[^0-9]{0,20}18", re.I),
]

for name in EXPECTED_FILES:
    path = OUT_DIR / name

    text = path.read_text(
        encoding="utf-8",
        errors="replace",
    )

    for lineno, line in enumerate(
        text.splitlines(),
        start=1,
    ):
        if any(rx.search(line) for rx in patterns):
            hits.append(
                (
                    name,
                    lineno,
                    line.strip(),
                )
            )

if hits:
    print()
    print("SUSPICIOUS PERSISTED COUNT REFERENCES:")
    for name, lineno, line in hits:
        print(f"{name}:{lineno}")
        print(" ", line)

    raise RuntimeError(
        "A persisted protocol artifact contains a suspicious "
        "'18 total artifacts' statement. Do not commit."
    )

print("[PASS] No persisted protocol artifact falsely claims 18 total artifacts.")


# ------------------------------------------------------------------
# 6. Ensure no unrelated Git changes
# ------------------------------------------------------------------

status = [
    line
    for line in run(
        [
            "git",
            "status",
            "--porcelain=v1",
            "--untracked-files=all",
        ]
    ).splitlines()
    if line.strip()
]

expected_status = sorted(
    f"A  {OUT_REL / name}"
    for name in EXPECTED_FILES
)

if sorted(status) != expected_status:
    raise RuntimeError(
        "Unexpected Git working-tree state.\n\n"
        + "\n".join(status)
    )

print()
print("[PASS] No unrelated Git changes exist.")


# ------------------------------------------------------------------
# FINAL
# ------------------------------------------------------------------

print()
print("=" * 110)
print("STAGE27-0 ARTIFACT-COUNT AUDIT PASSED")
print("=" * 110)
print("Protocol files             : 19")
print("Non-self hashes in freeze  : 18")
print("Files staged               : 19")
print("Scientific parent          :", EXPECTED_PARENT)
print("Model fits                 : 0")
print("Inference                  : 0")
print("Target openings            : 0")
print("GPU                        : 0")
print()
print("No files were modified.")
print("No commit was performed.")
print("=" * 110)

STAGE27-0 :: FINAL ARTIFACT-COUNT CONSISTENCY AUDIT
Expected parent : da1cc26b2759f43e64122bb8ff7a21d112dc420f
Local HEAD      : da1cc26b2759f43e64122bb8ff7a21d112dc420f
origin/main     : da1cc26b2759f43e64122bb8ff7a21d112dc420f
[PASS] Scientific parent unchanged.

Expected protocol files : 19
Actual protocol files   : 19
[PASS] Exactly 19 protocol files exist.

Expected staged files : 19
Actual staged files   : 19
[PASS] Exactly 19 protocol files are staged.

freeze_record.required_artifact_count : 19
freeze_record non-self artifact hashes: 18
[PASS] freeze_record correctly records 19 total artifacts and hashes the other 18.

Searching persisted protocol files for count wording...
[PASS] No persisted protocol artifact falsely claims 18 total artifacts.

[PASS] No unrelated Git changes exist.

STAGE27-0 ARTIFACT-COUNT AUDIT PASSED
Protocol files             : 19
Non-self hashes in freeze  : 18
Files staged               : 19
Scientific parent          : da1cc26b2759f43e64122bb8ff7a21d1

In [13]:
# ======================================================================================
# STAGE27-0 — COMMIT / PUSH / REMOTE BYTE-VERIFICATION
#
# ZERO MODEL FITS
# ZERO MODEL INFERENCE
# ZERO THRESHOLD SEARCH
# ZERO TARGET OPENINGS
# ZERO BOOTSTRAP
# ZERO GPU
#
# Preconditions:
#   - scientific parent = da1cc26b2759f43e64122bb8ff7a21d112dc420f
#   - exactly 19 Stage27-0 protocol artifacts staged
#   - artifact-count consistency audit passed
#
# This cell:
#   1. re-verifies exact staged universe
#   2. verifies freeze_record self/non-self semantics
#   3. commits Stage27-0
#   4. pushes main
#   5. verifies local == origin/main == remote main
#   6. verifies exactly 19 remote Stage27-0 paths
#   7. verifies remote file bytes SHA256 == local committed file bytes
#   8. verifies clean tree
#   9. verifies Stage27-1 has NOT been created
# ======================================================================================

from pathlib import Path
import hashlib
import json
import os
import stat
import subprocess

from kaggle_secrets import UserSecretsClient


# ======================================================================================
# 0. CONSTANTS
# ======================================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_PARENT = (
    "da1cc26b2759f43e64122bb8ff7a21d112dc420f"
)

OUT_REL = Path(
    "results/stage27_loao_unseen_attack/"
    "stage27_0_protocol_lock"
)

OUT_DIR = REPO / OUT_REL

REQUIRED_FILES = [
    "taxonomy.json",
    "fold_spec.json",
    "temporal_geometry.json",
    "target_population_spec.json",
    "model_inventory.json",
    "fit_budget.json",
    "compute_policy.json",
    "feature_representation.json",
    "class_weight_policy.json",
    "threshold_policy.json",
    "metric_spec.json",
    "support_rules.json",
    "uncertainty_spec.json",
    "similarity_spec.json",
    "target_opening_ledger.json",
    "interpretation_matrix.json",
    "prohibited_claims.json",
    "inherited_receipts.json",
    "freeze_record.json",
]

COMMIT_MESSAGE = (
    "stage27-0: freeze LOAO unseen-family protocol"
)


# ======================================================================================
# 1. HELPERS
# ======================================================================================

def banner(title):
    print()
    print("=" * 116)
    print(title)
    print("=" * 116)


def run(
    cmd,
    *,
    env=None,
    check=True,
    text=True,
):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=REPO,
        env=env,
        text=text,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}): "
            f"{' '.join(map(str, cmd))}\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    if text:
        return p.stdout.strip()

    return p.stdout


def sha256_bytes(data):
    return hashlib.sha256(
        data
    ).hexdigest()


def sha256_file(path):
    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):
            h.update(
                chunk
            )

    return h.hexdigest()


# ======================================================================================
# 2. PRE-COMMIT SCIENTIFIC-PARENT GATE
# ======================================================================================

banner("STAGE27-0 :: PRE-COMMIT PARENT GATE")

head = run(
    [
        "git",
        "rev-parse",
        "HEAD",
    ]
)

origin_before = run(
    [
        "git",
        "rev-parse",
        "origin/main",
    ]
)

print(
    "Expected parent :",
    EXPECTED_PARENT,
)

print(
    "Local HEAD      :",
    head,
)

print(
    "origin/main     :",
    origin_before,
)


if head != EXPECTED_PARENT:
    raise RuntimeError(
        "Local HEAD changed before Stage27-0 commit."
    )

if origin_before != EXPECTED_PARENT:
    raise RuntimeError(
        "origin/main changed before Stage27-0 commit."
    )

print(
    "[PASS] Scientific parent unchanged."
)


# ======================================================================================
# 3. EXACT 19-FILE STAGED-UNIVERSE GATE
# ======================================================================================

banner("STAGE27-0 :: EXACT STAGED UNIVERSE")

expected_paths = sorted(
    str(
        OUT_REL / name
    )
    for name in REQUIRED_FILES
)

staged_paths = sorted(
    x
    for x in run(
        [
            "git",
            "diff",
            "--cached",
            "--name-only",
        ]
    ).splitlines()
    if x.strip()
)

print(
    "Expected staged files:",
    len(expected_paths),
)

print(
    "Actual staged files  :",
    len(staged_paths),
)


if staged_paths != expected_paths:
    raise RuntimeError(
        "Staged file universe changed.\n"
        f"Expected={expected_paths}\n"
        f"Actual={staged_paths}"
    )


status_before = sorted(
    x
    for x in run(
        [
            "git",
            "status",
            "--porcelain=v1",
            "--untracked-files=all",
        ]
    ).splitlines()
    if x.strip()
)

expected_status = sorted(
    f"A  {path}"
    for path in expected_paths
)


if status_before != expected_status:
    raise RuntimeError(
        "Unexpected working-tree state before commit.\n\n"
        + "\n".join(
            status_before
        )
    )


print(
    "[PASS] Exactly 19 Stage27-0 files staged; no unrelated changes."
)


# ======================================================================================
# 4. FREEZE-RECORD CONSISTENCY GATE
# ======================================================================================

banner("STAGE27-0 :: FREEZE RECORD GATE")

freeze_path = (
    OUT_DIR
    / "freeze_record.json"
)

with freeze_path.open(
    "r",
    encoding="utf-8",
) as f:
    freeze = json.load(
        f
    )


required_count = freeze.get(
    "required_artifact_count"
)

hashed = freeze.get(
    "artifact_hashes_before_freeze_record",
    {},
)

print(
    "required_artifact_count :",
    required_count,
)

print(
    "non-self hashes         :",
    len(hashed),
)


if required_count != 19:
    raise RuntimeError(
        "freeze_record required_artifact_count != 19."
    )


expected_nonself = sorted(
    name
    for name in REQUIRED_FILES
    if name != "freeze_record.json"
)


if sorted(
    hashed.keys()
) != expected_nonself:
    raise RuntimeError(
        "freeze_record non-self artifact universe mismatch."
    )


# Recompute all 18 non-self hashes right now.

for name in expected_nonself:

    path = (
        OUT_DIR
        / name
    )

    actual = sha256_file(
        path
    )

    expected = hashed[
        name
    ][
        "sha256"
    ]

    if actual != expected:
        raise RuntimeError(
            f"Freeze-record SHA mismatch for {name}\n"
            f"expected={expected}\n"
            f"actual={actual}"
        )


print(
    "[PASS] freeze_record hashes all other 18 artifacts exactly."
)


# ======================================================================================
# 5. GIT IDENTITY GATE
# ======================================================================================

banner("STAGE27-0 :: GIT AUTHOR IDENTITY")

git_name = run(
    [
        "git",
        "config",
        "--local",
        "user.name",
    ],
    check=False,
)

git_email = run(
    [
        "git",
        "config",
        "--local",
        "user.email",
    ],
    check=False,
)


if not git_name:
    run(
        [
            "git",
            "config",
            "--local",
            "user.name",
            "J.M. Mubasshir Rahman",
        ]
    )

    git_name = run(
        [
            "git",
            "config",
            "--local",
            "user.name",
        ]
    )


if not git_email:
    run(
        [
            "git",
            "config",
            "--local",
            "user.email",
            "themubasshir@users.noreply.github.com",
        ]
    )

    git_email = run(
        [
            "git",
            "config",
            "--local",
            "user.email",
        ]
    )


print(
    "Git author name :",
    git_name,
)

print(
    "Git author email:",
    git_email,
)

print(
    "[PASS] Repository-local Git identity available."
)


# ======================================================================================
# 6. LOAD GITHUB AUTH
# ======================================================================================

banner("STAGE27-0 :: LOAD GITHUB AUTH")

client = UserSecretsClient()

token = None
token_label = None

for label in [
    "GITHUB_TOKEN",
    "github_token",
    "GH_TOKEN",
    "GITHUB_PAT",
    "github_pat",
]:

    try:
        candidate = client.get_secret(
            label
        )

    except Exception:
        candidate = None

    if candidate:

        candidate = str(
            candidate
        ).strip()

        if candidate:
            token = candidate
            token_label = label
            break


if not token:
    raise RuntimeError(
        "No GitHub token found in Kaggle Secrets."
    )


print(
    f"[FOUND] GitHub secret label: "
    f"{token_label} "
    "(value not printed)"
)


askpass = Path(
    "/kaggle/working/"
    ".stage27_0_commit_askpass.sh"
)

askpass.write_text(
    "#!/bin/sh\n"
    'case "$1" in\n'
    '  *Username*) '
    'printf "%s\\n" "x-access-token" ;;\n'
    '  *Password*) '
    'printf "%s\\n" "$STAGE27_GITHUB_TOKEN" ;;\n'
    '  *) printf "%s\\n" "" ;;\n'
    "esac\n",
    encoding="utf-8",
)

askpass.chmod(
    stat.S_IRUSR
    | stat.S_IWUSR
    | stat.S_IXUSR
)

env = os.environ.copy()

env[
    "GIT_ASKPASS"
] = str(
    askpass
)

env[
    "GIT_TERMINAL_PROMPT"
] = "0"

env[
    "STAGE27_GITHUB_TOKEN"
] = token


# ======================================================================================
# 7. COMMIT
# ======================================================================================

try:

    banner("STAGE27-0 :: COMMIT")

    print(
        run(
            [
                "git",
                "diff",
                "--cached",
                "--stat",
            ]
        )
    )

    commit_output = run(
        [
            "git",
            "commit",
            "-m",
            COMMIT_MESSAGE,
        ]
    )

    print()
    print(
        commit_output
    )


    new_head = run(
        [
            "git",
            "rev-parse",
            "HEAD",
        ]
    )

    new_parent = run(
        [
            "git",
            "rev-parse",
            "HEAD^",
        ]
    )


    print()
    print(
        "Stage27-0 commit:",
        new_head,
    )

    print(
        "Commit parent   :",
        new_parent,
    )


    if new_parent != EXPECTED_PARENT:
        raise RuntimeError(
            "Stage27-0 commit parent is not "
            "the remotely verified Stage27-0A commit."
        )


    # ==================================================================================
    # 8. VERIFY COMMITTED LOCAL TREE BEFORE PUSH
    # ==================================================================================

    banner("STAGE27-0 :: LOCAL COMMIT TREE VERIFICATION")

    committed_paths = sorted(
        x
        for x in run(
            [
                "git",
                "ls-tree",
                "-r",
                "--name-only",
                "HEAD",
                "--",
                str(
                    OUT_REL
                ),
            ]
        ).splitlines()
        if x.strip()
    )


    print(
        "Committed protocol files:",
        len(
            committed_paths
        ),
    )


    if committed_paths != expected_paths:
        raise RuntimeError(
            "Committed Stage27-0 tree does not "
            "contain exactly the 19 expected files."
        )


    print(
        "[PASS] Local commit contains exactly 19 Stage27-0 artifacts."
    )


    # ==================================================================================
    # 9. PUSH
    # ==================================================================================

    banner("STAGE27-0 :: PUSH MAIN")

    push_output = run(
        [
            "git",
            "push",
            "origin",
            "main",
        ],
        env=env,
    )

    print(
        push_output
        if push_output
        else "[OK] Push completed."
    )


    # ==================================================================================
    # 10. REMOTE SHA VERIFICATION
    # ==================================================================================

    banner("STAGE27-0 :: REMOTE SHA VERIFICATION")

    remote_main = run(
        [
            "git",
            "ls-remote",
            "origin",
            "refs/heads/main",
        ],
        env=env,
    ).split()[0]


    run(
        [
            "git",
            "fetch",
            "origin",
            "main",
        ],
        env=env,
    )


    origin_after = run(
        [
            "git",
            "rev-parse",
            "origin/main",
        ]
    )

    local_after = run(
        [
            "git",
            "rev-parse",
            "HEAD",
        ]
    )


    print(
        "Local HEAD  :",
        local_after,
    )

    print(
        "origin/main :",
        origin_after,
    )

    print(
        "remote main :",
        remote_main,
    )


    if not (
        local_after
        == origin_after
        == remote_main
        == new_head
    ):
        raise RuntimeError(
            "Stage27-0 remote SHA verification failed."
        )


    print(
        "[PASS] Local HEAD == origin/main == remote main."
    )


    # ==================================================================================
    # 11. EXACT REMOTE TREE VERIFICATION
    # ==================================================================================

    banner("STAGE27-0 :: REMOTE 19-FILE TREE VERIFICATION")

    remote_paths = sorted(
        x
        for x in run(
            [
                "git",
                "ls-tree",
                "-r",
                "--name-only",
                "origin/main",
                "--",
                str(
                    OUT_REL
                ),
            ]
        ).splitlines()
        if x.strip()
    )


    print(
        "Expected remote files:",
        len(
            expected_paths
        ),
    )

    print(
        "Actual remote files  :",
        len(
            remote_paths
        ),
    )


    if remote_paths != expected_paths:
        raise RuntimeError(
            "Remote Stage27-0 artifact universe mismatch.\n"
            f"Expected={expected_paths}\n"
            f"Actual={remote_paths}"
        )


    for path in remote_paths:
        print(
            " ",
            path,
        )


    print()
    print(
        "[PASS] Remote tree contains exactly 19 Stage27-0 artifacts."
    )


    # ==================================================================================
    # 12. BYTE-LEVEL REMOTE VERIFICATION
    # ==================================================================================

    banner("STAGE27-0 :: REMOTE BYTE-LEVEL SHA256 VERIFICATION")

    remote_sha256 = {}

    for rel in expected_paths:

        local_path = (
            REPO
            / rel
        )

        local_sha = sha256_file(
            local_path
        )

        # Read the exact blob from the fetched remote-tracking commit.
        remote_bytes = run(
            [
                "git",
                "show",
                f"origin/main:{rel}",
            ],
            text=False,
        )

        remote_sha = sha256_bytes(
            remote_bytes
        )

        remote_sha256[
            rel
        ] = remote_sha


        if local_sha != remote_sha:
            raise RuntimeError(
                f"Remote byte verification failed for:\n"
                f"{rel}\n"
                f"local ={local_sha}\n"
                f"remote={remote_sha}"
            )


        print(
            f"{local_sha}  {rel}"
        )


    print()
    print(
        "[PASS] All 19 remote files are byte-identical to the committed local files."
    )


    # ==================================================================================
    # 13. FINAL FREEZE-RECORD REMOTE CHECK
    # ==================================================================================

    banner("STAGE27-0 :: REMOTE FREEZE RECORD READBACK")

    remote_freeze_bytes = run(
        [
            "git",
            "show",
            f"origin/main:{OUT_REL / 'freeze_record.json'}",
        ],
        text=False,
    )

    remote_freeze = json.loads(
        remote_freeze_bytes.decode(
            "utf-8"
        )
    )


    if (
        remote_freeze[
            "required_artifact_count"
        ]
        != 19
    ):
        raise RuntimeError(
            "Remote freeze_record required_artifact_count != 19."
        )


    if len(
        remote_freeze[
            "artifact_hashes_before_freeze_record"
        ]
    ) != 18:
        raise RuntimeError(
            "Remote freeze_record should hash 18 non-self artifacts."
        )


    if (
        remote_freeze[
            "scientific_parent_commit"
        ]
        != EXPECTED_PARENT
    ):
        raise RuntimeError(
            "Remote freeze_record scientific parent mismatch."
        )


    if (
        remote_freeze[
            "authorized_fit_budget"
        ]
        != 10
    ):
        raise RuntimeError(
            "Remote freeze_record fit budget mismatch."
        )


    if (
        remote_freeze[
            "target_openings_consumed"
        ]
        != 0
    ):
        raise RuntimeError(
            "Remote freeze_record target openings are not zero."
        )


    print(
        "required_artifact_count :",
        remote_freeze[
            "required_artifact_count"
        ],
    )

    print(
        "non-self artifact hashes:",
        len(
            remote_freeze[
                "artifact_hashes_before_freeze_record"
            ]
        ),
    )

    print(
        "authorized fit budget   :",
        remote_freeze[
            "authorized_fit_budget"
        ],
    )

    print(
        "target openings consumed:",
        remote_freeze[
            "target_openings_consumed"
        ],
    )

    print(
        "[PASS] Remote freeze_record semantics verified."
    )


    # ==================================================================================
    # 14. FINAL CLEANNESS / NEXT-STAGE BOUNDARY
    # ==================================================================================

    banner("STAGE27-0 :: FINAL CLOSURE")

    final_status = run(
        [
            "git",
            "status",
            "--porcelain=v1",
            "--untracked-files=all",
        ]
    )


    if final_status.strip():
        raise RuntimeError(
            "Repository is not clean after Stage27-0 push:\n"
            + final_status
        )


    # Stage27-1 should not exist yet.
    unexpected_next_dirs = []

    for candidate in [
        REPO
        / "results"
        / "stage27_loao_unseen_attack"
        / "stage27_1_source_materialization",

        REPO
        / "results"
        / "stage27_loao_unseen_attack"
        / "stage27_1_fold_membership",

        REPO
        / "results"
        / "stage27_loao_unseen_attack"
        / "stage27_1_training",
    ]:

        if candidate.exists():
            unexpected_next_dirs.append(
                str(
                    candidate
                    .relative_to(
                        REPO
                    )
                )
            )


    if unexpected_next_dirs:
        raise RuntimeError(
            "Unexpected Stage27-1 artifacts/directories already exist:\n"
            + "\n".join(
                unexpected_next_dirs
            )
        )


    print(
        "Stage27-0 scientific parent :",
        EXPECTED_PARENT,
    )

    print(
        "Stage27-0 commit            :",
        new_head,
    )

    print(
        "Remote verification         : PASS",
    )

    print(
        "Protocol artifacts          : 19 / 19",
    )

    print(
        "Freeze non-self hashes      : 18 / 18",
    )

    print(
        "Git working tree            : CLEAN",
    )

    print(
        "Stage27-1 created           : NO",
    )

    print()
    print(
        "Eligible folds              : 5"
    )

    print(
        "Structurally ineligible     : 2"
    )

    print(
        "  - DOS"
    )

    print(
        "  - AUTH_BRUTE_FORCE"
    )

    print(
        "Descriptive-only eligible   : INFILTRATION"
    )

    print()
    print(
        "Authorized future fits      : 10"
    )

    print(
        "Fits completed              : 0"
    )

    print(
        "Inference completed         : 0"
    )

    print(
        "Threshold searches          : 0"
    )

    print(
        "Target openings consumed    : 0"
    )

    print(
        "Bootstrap executed          : 0"
    )

    print(
        "GPU hours                   : 0"
    )

    print()
    print("=" * 116)
    print(
        "STAGE27-0 COMPLETE — PROTOCOL REMOTELY FROZEN"
    )
    print("=" * 116)

    print()
    print(
        "NEXT AUTHORIZED STEP:"
    )

    print(
        "  Stage27-1A — materialize/verify the frozen CICIDS2017 "
        "sources and construct fold membership receipts."
    )

    print()
    print(
        "NO MODEL FIT IS AUTHORIZED UNTIL EVERY ELIGIBLE FOLD "
        "PASSES ITS MEMBERSHIP AND HELD-OUT-EXCLUSION GATES."
    )


finally:

    try:
        askpass.unlink(
            missing_ok=True
        )
    except Exception:
        pass

    token = None

    env.pop(
        "STAGE27_GITHUB_TOKEN",
        None,
    )


STAGE27-0 :: PRE-COMMIT PARENT GATE
Expected parent : da1cc26b2759f43e64122bb8ff7a21d112dc420f
Local HEAD      : da1cc26b2759f43e64122bb8ff7a21d112dc420f
origin/main     : da1cc26b2759f43e64122bb8ff7a21d112dc420f
[PASS] Scientific parent unchanged.

STAGE27-0 :: EXACT STAGED UNIVERSE
Expected staged files: 19
Actual staged files  : 19
[PASS] Exactly 19 Stage27-0 files staged; no unrelated changes.

STAGE27-0 :: FREEZE RECORD GATE
required_artifact_count : 19
non-self hashes         : 18
[PASS] freeze_record hashes all other 18 artifacts exactly.

STAGE27-0 :: GIT AUTHOR IDENTITY
Git author name : J.M. Mubasshir Rahman
Git author email: themubasshir@users.noreply.github.com
[PASS] Repository-local Git identity available.

STAGE27-0 :: LOAD GITHUB AUTH
[FOUND] GitHub secret label: GITHUB_TOKEN (value not printed)

STAGE27-0 :: COMMIT
.../class_weight_policy.json                       |  22 ++
 .../stage27_0_protocol_lock/compute_policy.json    |  21 ++
 .../feature_representation.json  

In [14]:
# ======================================================================================
# STAGE27-1A — EXACT CICIDS2017 SOURCE AVAILABILITY AUDIT
#
# READ ONLY
# ZERO DOWNLOADS
# ZERO COPIES
# ZERO STAGE27 ARTIFACT WRITES
# ZERO LABEL/FAMILY MATERIALIZATION
# ZERO MODEL FITS / INFERENCE / TARGET OPENINGS / GPU
#
# Scientific parent:
#   1ea6bedb141bc3fa6115edbc5296d0f7c6252559
#
# PURPOSE
# -------
# Locate the eight byte-exact CICIDS2017 parquet assets frozen in Stage27-0.
#
# Search priority:
#   1. existing Stage24 working cache
#   2. Kaggle input datasets
#   3. other /kaggle/working caches
#
# Only files whose basename, frozen byte size AND SHA256 all match are accepted.
# ======================================================================================

from __future__ import annotations

import hashlib
import json
import os
import subprocess
from collections import defaultdict
from pathlib import Path


# ======================================================================================
# 0. CONSTANTS
# ======================================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_PARENT = (
    "1ea6bedb141bc3fa6115edbc5296d0f7c6252559"
)

SOURCE_CONTRACT = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0b2_complete_cicids2017_source_contract.json"
)

STAGE27_TARGET_SPEC = (
    REPO
    / "results"
    / "stage27_loao_unseen_attack"
    / "stage27_0_protocol_lock"
    / "target_population_spec.json"
)

STAGE27_FREEZE = (
    REPO
    / "results"
    / "stage27_loao_unseen_attack"
    / "stage27_0_protocol_lock"
    / "freeze_record.json"
)

SEARCH_ROOTS = [
    Path("/kaggle/working/stage24_cicids2017_sources"),
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]


# ======================================================================================
# 1. HELPERS
# ======================================================================================

def banner(title):
    print()
    print("=" * 118)
    print(title)
    print("=" * 118)


def run(cmd):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    if p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}): {' '.join(map(str, cmd))}\n"
            f"STDOUT:\n{p.stdout}\n"
            f"STDERR:\n{p.stderr}"
        )

    return p.stdout.strip()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as f:
        return json.load(f)


def sha256_file(
    path,
    chunk_size=16 * 1024 * 1024,
):
    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                chunk_size
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


# ======================================================================================
# 2. SCIENTIFIC-PARENT / CLEANNESS GATE
# ======================================================================================

banner("STAGE27-1A :: SCIENTIFIC-PARENT GATE")

head = run(
    [
        "git",
        "rev-parse",
        "HEAD",
    ]
)

origin = run(
    [
        "git",
        "rev-parse",
        "origin/main",
    ]
)

status = run(
    [
        "git",
        "status",
        "--porcelain=v1",
        "--untracked-files=all",
    ]
)

print("Expected parent :", EXPECTED_PARENT)
print("Local HEAD      :", head)
print("origin/main     :", origin)
print("Git clean       :", not bool(status.strip()))


if head != EXPECTED_PARENT:
    raise RuntimeError(
        "Local HEAD is not the frozen Stage27-0 commit."
    )


if origin != EXPECTED_PARENT:
    raise RuntimeError(
        "origin/main is not the frozen Stage27-0 commit."
    )


if status.strip():
    raise RuntimeError(
        "Repository is dirty before Stage27-1A:\n"
        + status
    )


print("[PASS] Stage27-0 is the clean execution parent.")


# ======================================================================================
# 3. VERIFY STAGE27-0 FREEZE SEMANTICS
# ======================================================================================

banner("STAGE27-1A :: VERIFY STAGE27-0 FREEZE")

target_spec = load_json(
    STAGE27_TARGET_SPEC
)

freeze = load_json(
    STAGE27_FREEZE
)


if freeze[
    "authorized_fit_budget"
] != 10:
    raise RuntimeError(
        "Unexpected Stage27 fit budget."
    )


if freeze[
    "target_openings_consumed"
] != 0:
    raise RuntimeError(
        "Target-opening ledger is no longer pristine."
    )


if freeze[
    "model_fits_completed"
] != 0:
    raise RuntimeError(
        "A model fit has already occurred unexpectedly."
    )


if freeze[
    "model_inference_completed"
] != 0:
    raise RuntimeError(
        "Model inference has already occurred unexpectedly."
    )


print(
    "Protocol artifacts :",
    freeze[
        "required_artifact_count"
    ],
)

print(
    "Authorized fits    :",
    freeze[
        "authorized_fit_budget"
    ],
)

print(
    "Fits completed     :",
    freeze[
        "model_fits_completed"
    ],
)

print(
    "Target openings    :",
    freeze[
        "target_openings_consumed"
    ],
)

print("[PASS] Stage27 scientific boundary remains sealed.")


# ======================================================================================
# 4. RECOVER EXACT EIGHT-ASSET CONTRACT
# ======================================================================================

banner("STAGE27-1A :: RECOVER FROZEN CICIDS2017 SOURCE CONTRACT")

contract = load_json(
    SOURCE_CONTRACT
)

records = contract.get(
    "acquisition_records",
    []
)


if len(records) != 8:
    raise RuntimeError(
        f"Expected exactly 8 Stage24 source records; "
        f"found {len(records)}."
    )


frozen_assets = []


for rec in records:

    if rec.get(
        "status"
    ) != "VERIFIED":
        raise RuntimeError(
            "Frozen source record is not VERIFIED."
        )


    if rec.get(
        "sha256_match"
    ) is not True:
        raise RuntimeError(
            "Frozen source SHA gate was not verified."
        )


    frozen_assets.append(
        {
            "day": rec["day"],
            "basename": Path(
                rec["remote"]
            ).name,
            "remote": rec[
                "remote"
            ],
            "repo_id": rec[
                "repo_id"
            ],
            "revision": rec[
                "revision"
            ],
            "size_bytes": int(
                rec["size_bytes"]
            ),
            "sha256": rec[
                "sha256"
            ],
            "physical_rows": int(
                rec["physical_rows"]
            ),
            "ordered_schema_sha256": rec[
                "ordered_schema_sha256"
            ],
        }
    )


for i, asset in enumerate(
    frozen_assets,
    start=1,
):

    print()
    print(
        f"[{i}/8] {asset['day']}"
    )

    print(
        "  basename :",
        asset["basename"],
    )

    print(
        "  bytes    :",
        f"{asset['size_bytes']:,}",
    )

    print(
        "  rows     :",
        f"{asset['physical_rows']:,}",
    )

    print(
        "  sha256   :",
        asset["sha256"],
    )


print()
print("[PASS] Exact eight-file source contract recovered.")


# ======================================================================================
# 5. BUILD BASENAME INDEX
# ======================================================================================

banner("STAGE27-1A :: INDEX CURRENT KAGGLE FILESYSTEM")

wanted_names = {
    asset[
        "basename"
    ]
    for asset in frozen_assets
}


candidate_index = defaultdict(
    list
)

visited_files = 0


# We intentionally avoid walking the repository repeatedly.
# /kaggle/working is searched once and known irrelevant subtrees may be skipped.

skip_dirs = {
    ".git",
    "__pycache__",
}


for root in SEARCH_ROOTS:

    if not root.exists():
        print(
            "[MISSING ROOT]",
            root,
        )
        continue


    print(
        "[SEARCH ROOT]",
        root,
    )


    for current_root, dirs, files in os.walk(
        root
    ):

        dirs[:] = [
            d
            for d in dirs
            if d not in skip_dirs
        ]


        for name in files:

            visited_files += 1

            if name not in wanted_names:
                continue


            path = (
                Path(current_root)
                / name
            ).resolve()


            # Avoid duplicate paths because /kaggle/working search also
            # traverses stage24_cicids2017_sources.
            if path not in candidate_index[
                name
            ]:

                candidate_index[
                    name
                ].append(
                    path
                )


print()
print(
    "Filesystem files visited :",
    f"{visited_files:,}",
)

print(
    "Wanted basenames         :",
    len(
        wanted_names
    ),
)

print(
    "Basenames located        :",
    sum(
        bool(
            candidate_index[
                name
            ]
        )
        for name in wanted_names
    ),
)


# ======================================================================================
# 6. SIZE-FIRST / SHA-SECOND EXACT VERIFICATION
# ======================================================================================

banner("STAGE27-1A :: BYTE-EXACT SOURCE VERIFICATION")

verified = []

missing = []

wrong_size = []

wrong_hash = []


for idx, asset in enumerate(
    frozen_assets,
    start=1,
):

    basename = asset[
        "basename"
    ]

    candidates = candidate_index[
        basename
    ]


    print()
    print(
        f"[{idx}/8] {asset['day']} :: {basename}"
    )


    if not candidates:

        print(
            "  status: NOT FOUND"
        )

        missing.append(
            asset
        )

        continue


    print(
        "  candidate paths:",
        len(
            candidates
        ),
    )


    exact_matches = []


    for candidate in candidates:

        size = candidate.stat().st_size


        print()
        print(
            "  candidate:",
            candidate,
        )

        print(
            "    size:",
            f"{size:,}",
        )


        if size != asset[
            "size_bytes"
        ]:

            print(
                "    size gate: FAIL"
            )

            wrong_size.append(
                {
                    "asset": asset,
                    "path": str(
                        candidate
                    ),
                    "actual_size": size,
                }
            )

            continue


        print(
            "    size gate: PASS"
        )

        print(
            "    hashing..."
        )


        digest = sha256_file(
            candidate
        )


        print(
            "    actual SHA256  :",
            digest,
        )

        print(
            "    expected SHA256:",
            asset[
                "sha256"
            ],
        )


        if digest != asset[
            "sha256"
        ]:

            print(
                "    SHA gate: FAIL"
            )

            wrong_hash.append(
                {
                    "asset": asset,
                    "path": str(
                        candidate
                    ),
                    "actual_sha256": digest,
                }
            )

            continue


        print(
            "    SHA gate: PASS"
        )


        exact_matches.append(
            {
                "path": str(
                    candidate
                ),
                "size_bytes": size,
                "sha256": digest,
            }
        )


    if exact_matches:

        verified.append(
            {
                "asset": asset,
                "matches": exact_matches,
            }
        )

        print()
        print(
            "  FINAL STATUS: EXACT SOURCE AVAILABLE"
        )

    else:

        missing.append(
            asset
        )

        print()
        print(
            "  FINAL STATUS: NO BYTE-EXACT SOURCE AVAILABLE"
        )


# ======================================================================================
# 7. SUMMARY
# ======================================================================================

banner("STAGE27-1A :: SOURCE AVAILABILITY SUMMARY")


print(
    "Frozen assets required :",
    len(
        frozen_assets
    ),
)

print(
    "Exact assets available :",
    len(
        verified
    ),
)

print(
    "Exact assets unavailable:",
    len(
        missing
    ),
)

print(
    "Wrong-size candidates  :",
    len(
        wrong_size
    ),
)

print(
    "Wrong-hash candidates  :",
    len(
        wrong_hash
    ),
)


print()
print("Exact matches:")


if not verified:

    print(
        "  NONE"
    )


for item in verified:

    asset = item[
        "asset"
    ]

    print()
    print(
        f"  {asset['day']} :: "
        f"{asset['basename']}"
    )

    for match in item[
        "matches"
    ]:

        print(
            "    ",
            match[
                "path"
            ],
        )


print()
print("Unavailable exact sources:")


if not missing:

    print(
        "  NONE"
    )


for asset in missing:

    print(
        f"  {asset['day']:10s} "
        f"{asset['basename']}"
    )


# ======================================================================================
# 8. KAGGLE INPUT INVENTORY FOR DIAGNOSIS
# ======================================================================================

banner("STAGE27-1A :: KAGGLE INPUT INVENTORY")

input_root = Path(
    "/kaggle/input"
)


if input_root.exists():

    interesting = []

    for path in input_root.rglob(
        "*"
    ):

        if not path.is_file():
            continue

        suffix = path.suffix.lower()

        if suffix in {
            ".csv",
            ".parquet",
            ".pkl",
            ".pickle",
            ".tar",
            ".gz",
            ".zip",
        }:

            interesting.append(
                (
                    str(
                        path
                    ),
                    path.stat().st_size,
                )
            )


    print(
        "Relevant input files:",
        len(
            interesting
        ),
    )


    for path, size in sorted(
        interesting
    ):

        print(
            f"{size:>14,}  {path}"
        )

else:

    print(
        "/kaggle/input does not exist."
    )


# ======================================================================================
# 9. SCIENTIFIC NON-ACTION GATE
# ======================================================================================

banner("STAGE27-1A :: NON-ACTION AUDIT")

repo_status = run(
    [
        "git",
        "status",
        "--porcelain=v1",
        "--untracked-files=all",
    ]
)


if repo_status.strip():

    raise RuntimeError(
        "Repository changed during read-only source audit:\n"
        + repo_status
    )


stage27_1_candidates = [
    REPO
    / "results"
    / "stage27_loao_unseen_attack"
    / "stage27_1_source_materialization",

    REPO
    / "results"
    / "stage27_loao_unseen_attack"
    / "stage27_1_fold_membership",
]


for path in stage27_1_candidates:

    if path.exists():

        raise RuntimeError(
            "Stage27-1 artifact directory appeared "
            "during read-only audit:\n"
            f"{path}"
        )


print(
    "Downloads performed       : 0"
)

print(
    "Files copied              : 0"
)

print(
    "Stage27 artifacts written : 0"
)

print(
    "Rows materialized         : 0"
)

print(
    "Features materialized     : 0"
)

print(
    "Labels processed          : 0"
)

print(
    "Model fits                : 0"
)

print(
    "Model inference           : 0"
)

print(
    "Target openings           : 0"
)

print(
    "GPU hours                 : 0"
)

print(
    "Git working tree          : CLEAN"
)


print()
print("=" * 118)

if len(
    verified
) == 8:

    print(
        "STAGE27-1A SOURCE AUDIT RESULT: "
        "ALL 8 BYTE-EXACT SOURCES ALREADY AVAILABLE"
    )

    print()
    print(
        "NEXT STEP:"
    )

    print(
        "  Materialize deterministic Stage27 source cache "
        "and build five fold-membership receipts."
    )

else:

    print(
        "STAGE27-1A SOURCE AUDIT RESULT: "
        f"{len(verified)}/8 BYTE-EXACT SOURCES AVAILABLE"
    )

    print()
    print(
        "NEXT STEP:"
    )

    print(
        "  Resolve only the missing exact source assets "
        "under the frozen Stage27-0 source contract."
    )

print("=" * 118)


STAGE27-1A :: SCIENTIFIC-PARENT GATE
Expected parent : 1ea6bedb141bc3fa6115edbc5296d0f7c6252559
Local HEAD      : 1ea6bedb141bc3fa6115edbc5296d0f7c6252559
origin/main     : 1ea6bedb141bc3fa6115edbc5296d0f7c6252559
Git clean       : True
[PASS] Stage27-0 is the clean execution parent.

STAGE27-1A :: VERIFY STAGE27-0 FREEZE
Protocol artifacts : 19
Authorized fits    : 10
Fits completed     : 0
Target openings    : 0
[PASS] Stage27 scientific boundary remains sealed.

STAGE27-1A :: RECOVER FROZEN CICIDS2017 SOURCE CONTRACT

[1/8] Monday
  basename : Monday-WorkingHours.pcap_ISCX.csv.parquet
  bytes    : 65,465,382
  rows     : 529,918
  sha256   : dfdcef4b8670e52af54dc4f82174834365a393473e877174cca46d17b12dfd02

[2/8] Tuesday
  basename : Tuesday-WorkingHours.pcap_ISCX.csv.parquet
  bytes    : 52,701,751
  rows     : 445,909
  sha256   : 27e83d518cb093faefd0f883cb4df3ad8b353f150934004f28d0e7962f9f31c4

[3/8] Wednesday
  basename : Wednesday-workingHours.pcap_ISCX.csv.parquet
  bytes    :

In [15]:
# ======================================================================================
# STAGE27-1A — RECOVER 8 BYTE-EXACT CICIDS2017 SOURCES
#
# SOURCE ACQUISITION ONLY
#
# ALLOWED:
#   - download the exact 8 frozen parquet assets
#   - exact pinned Hugging Face revision only
#   - SHA256 / size / parquet-footer verification
#   - materialize a canonical runtime cache OUTSIDE the Git repository
#
# FORBIDDEN:
#   - source substitution
#   - dataset reconstruction
#   - Stage27 repository artifact writes
#   - family/fold membership construction
#   - feature materialization
#   - model fit / inference
#   - threshold search
#   - target opening
#   - GPU
#
# Frozen Stage27-0 parent:
#   1ea6bedb141bc3fa6115edbc5296d0f7c6252559
# ======================================================================================

from __future__ import annotations

import hashlib
import json
import os
import shutil
import subprocess
from pathlib import Path

import pyarrow.parquet as pq

try:
    from huggingface_hub import hf_hub_download
except Exception as exc:
    raise RuntimeError(
        "huggingface_hub is unavailable in this Kaggle runtime."
    ) from exc


# ======================================================================================
# 0. CONSTANTS
# ======================================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_PARENT = (
    "1ea6bedb141bc3fa6115edbc5296d0f7c6252559"
)

SOURCE_CONTRACT = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0b2_complete_cicids2017_source_contract.json"
)

# Canonical Stage27 runtime cache.
# IMPORTANT: deliberately outside Git repository.
DEST_ROOT = Path(
    "/kaggle/working/stage27_cicids2017_sources"
)

# Hugging Face cache can also remain outside Git.
HF_CACHE = Path(
    "/kaggle/working/stage27_hf_cache"
)

EXPECTED_REPO_ID = "bvsam/cic-ids-2017"

EXPECTED_REVISION = (
    "b7e532345512edcd530cb1770dc76636aeb52802"
)

os.environ["CUDA_VISIBLE_DEVICES"] = ""


# ======================================================================================
# 1. HELPERS
# ======================================================================================

def banner(title):
    print()
    print("=" * 118)
    print(title)
    print("=" * 118)


def run(cmd):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    if p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}): "
            f"{' '.join(map(str, cmd))}\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return p.stdout.strip()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as f:
        return json.load(f)


def sha256_file(
    path,
    chunk_size=16 * 1024 * 1024,
):
    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:
            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def human_bytes(n):
    units = [
        "B",
        "KiB",
        "MiB",
        "GiB",
    ]

    value = float(n)

    for unit in units:

        if value < 1024.0 or unit == units[-1]:
            return (
                f"{value:.2f} {unit}"
            )

        value /= 1024.0


# ======================================================================================
# 2. STAGE27-0 PARENT + CLEANNESS GATE
# ======================================================================================

banner("STAGE27-1A SOURCE RECOVERY :: SCIENTIFIC-PARENT GATE")

head = run(
    [
        "git",
        "rev-parse",
        "HEAD",
    ]
)

origin = run(
    [
        "git",
        "rev-parse",
        "origin/main",
    ]
)

status = run(
    [
        "git",
        "status",
        "--porcelain=v1",
        "--untracked-files=all",
    ]
)


print("Expected parent :", EXPECTED_PARENT)
print("Local HEAD      :", head)
print("origin/main     :", origin)
print("Git clean       :", not bool(status.strip()))


if head != EXPECTED_PARENT:
    raise RuntimeError(
        "Local HEAD changed from the frozen Stage27-0 commit."
    )


if origin != EXPECTED_PARENT:
    raise RuntimeError(
        "origin/main changed from the frozen Stage27-0 commit."
    )


if status.strip():
    raise RuntimeError(
        "Repository is dirty before source recovery:\n"
        + status
    )


print("[PASS] Frozen Stage27-0 parent confirmed.")


# ======================================================================================
# 3. LOAD EXACT FROZEN SOURCE CONTRACT
# ======================================================================================

banner("STAGE27-1A SOURCE RECOVERY :: LOAD FROZEN CONTRACT")

contract = load_json(
    SOURCE_CONTRACT
)


hf_source = contract.get(
    "hf_source",
    {}
)


if hf_source.get(
    "repo_id"
) != EXPECTED_REPO_ID:

    raise RuntimeError(
        "Frozen Hugging Face repo_id mismatch.\n"
        f"expected={EXPECTED_REPO_ID}\n"
        f"actual={hf_source.get('repo_id')}"
    )


if hf_source.get(
    "substitution_allowed"
) is not False:

    raise RuntimeError(
        "Frozen contract does not explicitly forbid source substitution."
    )


if hf_source.get(
    "exact_revision_per_artifact"
) is not True:

    raise RuntimeError(
        "Frozen source contract does not require exact revisions."
    )


records = contract.get(
    "acquisition_records",
    []
)


if len(records) != 8:
    raise RuntimeError(
        f"Expected 8 acquisition records; found {len(records)}."
    )


assets = []


for rec in records:

    if rec.get(
        "status"
    ) != "VERIFIED":
        raise RuntimeError(
            "Frozen acquisition record is not VERIFIED."
        )


    if rec.get(
        "sha256_match"
    ) is not True:
        raise RuntimeError(
            "Frozen acquisition record did not pass SHA256."
        )


    repo_id = rec[
        "repo_id"
    ]

    revision = rec[
        "revision"
    ]


    if repo_id != EXPECTED_REPO_ID:
        raise RuntimeError(
            f"Unexpected repo_id in acquisition record: {repo_id}"
        )


    if revision != EXPECTED_REVISION:
        raise RuntimeError(
            f"Unexpected revision in acquisition record:\n"
            f"{revision}"
        )


    assets.append(
        {
            "day": rec["day"],
            "remote": rec["remote"],
            "basename": Path(
                rec["remote"]
            ).name,
            "repo_id": repo_id,
            "revision": revision,
            "size_bytes": int(
                rec["size_bytes"]
            ),
            "sha256": rec[
                "sha256"
            ],
            "physical_rows": int(
                rec["physical_rows"]
            ),
            "ordered_schema_sha256": rec[
                "ordered_schema_sha256"
            ],
        }
    )


print("Hugging Face repo :", EXPECTED_REPO_ID)
print("Pinned revision   :", EXPECTED_REVISION)
print("Required assets   :", len(assets))
print("Substitution      : FORBIDDEN")

print()
print("[PASS] Frozen eight-asset contract recovered.")


# ======================================================================================
# 4. PREPARE RUNTIME CACHE
# ======================================================================================

banner("STAGE27-1A SOURCE RECOVERY :: PREPARE RUNTIME CACHE")

DEST_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

HF_CACHE.mkdir(
    parents=True,
    exist_ok=True,
)


print("Canonical cache :", DEST_ROOT)
print("HF cache        :", HF_CACHE)
print("Inside Git repo :", str(DEST_ROOT).startswith(str(REPO)))

if str(
    DEST_ROOT
).startswith(
    str(REPO)
):
    raise RuntimeError(
        "Canonical source cache must remain outside the Git repository."
    )


# ======================================================================================
# 5. DOWNLOAD / VERIFY EACH EXACT ASSET
# ======================================================================================

banner("STAGE27-1A SOURCE RECOVERY :: EXACT ACQUISITION")

receipts = []

downloaded_count = 0

reused_count = 0

materialization_modes = {}


for index, asset in enumerate(
    assets,
    start=1,
):

    print()
    print("-" * 118)

    print(
        f"[{index}/8] "
        f"{asset['day']} :: "
        f"{asset['basename']}"
    )

    print("-" * 118)


    destination = (
        DEST_ROOT
        / asset[
            "basename"
        ]
    )


    # ------------------------------------------------------------------
    # 5A. Existing canonical copy?
    # ------------------------------------------------------------------

    if destination.exists():

        existing_size = destination.stat().st_size

        print(
            "Existing canonical file:",
            destination,
        )

        print(
            "Existing size          :",
            f"{existing_size:,}",
        )


        if existing_size == asset[
            "size_bytes"
        ]:

            print(
                "Existing size gate    : PASS"
            )

            existing_sha = sha256_file(
                destination
            )

            print(
                "Existing SHA256       :",
                existing_sha,
            )


            if existing_sha == asset[
                "sha256"
            ]:

                print(
                    "Existing SHA gate     : PASS"
                )

                print(
                    "Action                : REUSE EXACT CANONICAL FILE"
                )

                reused_count += 1

                materialization_modes[
                    asset[
                        "basename"
                    ]
                ] = "REUSED_EXISTING_EXACT"

            else:

                raise RuntimeError(
                    "Canonical destination exists with correct size "
                    "but wrong SHA256. Refusing to overwrite silently.\n"
                    f"path={destination}\n"
                    f"expected={asset['sha256']}\n"
                    f"actual={existing_sha}"
                )

        else:

            raise RuntimeError(
                "Canonical destination already exists with wrong byte size. "
                "Refusing to overwrite silently.\n"
                f"path={destination}\n"
                f"expected={asset['size_bytes']}\n"
                f"actual={existing_size}"
            )


    # ------------------------------------------------------------------
    # 5B. Missing -> download exact pinned HF artifact
    # ------------------------------------------------------------------

    else:

        print(
            "Action                : DOWNLOAD EXACT PINNED ASSET"
        )

        print(
            "HF repo               :",
            asset[
                "repo_id"
            ],
        )

        print(
            "HF revision           :",
            asset[
                "revision"
            ],
        )

        print(
            "HF filename           :",
            asset[
                "remote"
            ],
        )

        print(
            "Expected size         :",
            f"{asset['size_bytes']:,}",
            f"({human_bytes(asset['size_bytes'])})",
        )

        print(
            "Expected SHA256       :",
            asset[
                "sha256"
            ],
        )

        print()
        print(
            "Downloading..."
        )


        downloaded_path = Path(
            hf_hub_download(
                repo_id=asset[
                    "repo_id"
                ],
                filename=asset[
                    "remote"
                ],
                repo_type="dataset",
                revision=asset[
                    "revision"
                ],
                cache_dir=str(
                    HF_CACHE
                ),
            )
        ).resolve()


        print(
            "Downloaded cache path :",
            downloaded_path,
        )


        if not downloaded_path.is_file():
            raise RuntimeError(
                "hf_hub_download returned a non-file path."
            )


        downloaded_size = downloaded_path.stat().st_size


        print(
            "Downloaded size       :",
            f"{downloaded_size:,}",
        )


        if downloaded_size != asset[
            "size_bytes"
        ]:

            raise RuntimeError(
                "Downloaded source byte-size mismatch.\n"
                f"asset={asset['remote']}\n"
                f"expected={asset['size_bytes']}\n"
                f"actual={downloaded_size}"
            )


        print(
            "Size gate             : PASS"
        )

        print(
            "Hashing downloaded bytes..."
        )


        downloaded_sha = sha256_file(
            downloaded_path
        )


        print(
            "Downloaded SHA256     :",
            downloaded_sha,
        )

        print(
            "Expected SHA256       :",
            asset[
                "sha256"
            ],
        )


        if downloaded_sha != asset[
            "sha256"
        ]:

            raise RuntimeError(
                "Downloaded source SHA256 mismatch.\n"
                f"asset={asset['remote']}\n"
                f"expected={asset['sha256']}\n"
                f"actual={downloaded_sha}"
            )


        print(
            "SHA gate              : PASS"
        )


        # ------------------------------------------------------------------
        # Materialize canonical runtime path.
        #
        # Prefer a hard link so no duplicate bytes are created.
        # Fall back to copy2 only if filesystem boundaries prevent hardlink.
        # ------------------------------------------------------------------

        try:

            os.link(
                downloaded_path,
                destination,
            )

            mode = "HARDLINK_FROM_HF_CACHE"

        except OSError:

            shutil.copy2(
                downloaded_path,
                destination,
            )

            mode = "COPY_FROM_HF_CACHE"


        materialization_modes[
            asset[
                "basename"
            ]
        ] = mode

        downloaded_count += 1


        print(
            "Canonical materialization:",
            mode,
        )

        print(
            "Canonical path          :",
            destination,
        )


    # ------------------------------------------------------------------
    # 5C. Final canonical byte verification
    # ------------------------------------------------------------------

    final_size = destination.stat().st_size


    if final_size != asset[
        "size_bytes"
    ]:

        raise RuntimeError(
            "Final canonical source size mismatch."
        )


    final_sha = sha256_file(
        destination
    )


    if final_sha != asset[
        "sha256"
    ]:

        raise RuntimeError(
            "Final canonical source SHA mismatch."
        )


    # ------------------------------------------------------------------
    # 5D. Parquet footer verification
    # ------------------------------------------------------------------

    parquet = pq.ParquetFile(
        destination
    )

    actual_rows = parquet.metadata.num_rows

    row_groups = parquet.metadata.num_row_groups

    columns = parquet.schema_arrow.names


    print(
        "Parquet rows            :",
        f"{actual_rows:,}",
    )

    print(
        "Expected rows           :",
        f"{asset['physical_rows']:,}",
    )

    print(
        "Parquet row groups      :",
        row_groups,
    )

    print(
        "Parquet columns         :",
        len(
            columns
        ),
    )


    if actual_rows != asset[
        "physical_rows"
    ]:

        raise RuntimeError(
            "Parquet footer row-count mismatch."
        )


    if len(
        columns
    ) != 85:

        raise RuntimeError(
            f"Expected 85 physical CICIDS2017 columns; "
            f"found {len(columns)}."
        )


    if "Label" not in columns:

        raise RuntimeError(
            "Parquet schema lacks Label column."
        )


    print(
        "Footer/schema gate      : PASS"
    )


    receipts.append(
        {
            "day": asset[
                "day"
            ],
            "remote": asset[
                "remote"
            ],
            "canonical_path": str(
                destination
            ),
            "repo_id": asset[
                "repo_id"
            ],
            "revision": asset[
                "revision"
            ],
            "size_bytes": final_size,
            "sha256": final_sha,
            "physical_rows": actual_rows,
            "parquet_row_groups": row_groups,
            "column_count": len(
                columns
            ),
            "materialization_mode": materialization_modes[
                asset[
                    "basename"
                ]
            ],
            "verified": True,
        }
    )


    print()
    print(
        "FINAL ASSET STATUS      : BYTE-EXACT VERIFIED"
    )


# ======================================================================================
# 6. GLOBAL 8/8 VERIFICATION
# ======================================================================================

banner("STAGE27-1A SOURCE RECOVERY :: GLOBAL 8/8 VERIFICATION")


if len(
    receipts
) != 8:

    raise RuntimeError(
        f"Expected 8 verified source receipts; "
        f"got {len(receipts)}."
    )


for asset, receipt in zip(
    assets,
    receipts,
):

    if receipt[
        "sha256"
    ] != asset[
        "sha256"
    ]:

        raise RuntimeError(
            "Final global SHA verification failed."
        )


    if receipt[
        "size_bytes"
    ] != asset[
        "size_bytes"
    ]:

        raise RuntimeError(
            "Final global size verification failed."
        )


    if receipt[
        "physical_rows"
    ] != asset[
        "physical_rows"
    ]:

        raise RuntimeError(
            "Final global row-count verification failed."
        )


total_bytes = sum(
    receipt[
        "size_bytes"
    ]
    for receipt in receipts
)

total_rows = sum(
    receipt[
        "physical_rows"
    ]
    for receipt in receipts
)


print(
    "Verified assets       :",
    f"{len(receipts)}/8",
)

print(
    "Downloaded this run   :",
    downloaded_count,
)

print(
    "Reused exact files    :",
    reused_count,
)

print(
    "Total verified bytes  :",
    f"{total_bytes:,}",
    f"({human_bytes(total_bytes)})",
)

print(
    "Total physical rows   :",
    f"{total_rows:,}",
)


if total_rows != 2_830_743:

    raise RuntimeError(
        f"Expected total CICIDS2017 population 2,830,743; "
        f"found {total_rows:,}."
    )


print()
print("[PASS] All eight exact Stage27 CICIDS2017 sources are available.")


# ======================================================================================
# 7. PRINT CANONICAL SOURCE MAP
# ======================================================================================

banner("STAGE27-1A SOURCE RECOVERY :: CANONICAL SOURCE MAP")


for receipt in receipts:

    print()
    print(
        receipt[
            "day"
        ],
    )

    print(
        "  path     :",
        receipt[
            "canonical_path"
        ],
    )

    print(
        "  rows     :",
        f"{receipt['physical_rows']:,}",
    )

    print(
        "  bytes    :",
        f"{receipt['size_bytes']:,}",
    )

    print(
        "  sha256   :",
        receipt[
            "sha256"
        ],
    )

    print(
        "  mode     :",
        receipt[
            "materialization_mode"
        ],
    )


# ======================================================================================
# 8. GIT / SCIENTIFIC NON-ACTION AUDIT
# ======================================================================================

banner("STAGE27-1A SOURCE RECOVERY :: NON-ACTION AUDIT")


git_status = run(
    [
        "git",
        "status",
        "--porcelain=v1",
        "--untracked-files=all",
    ]
)


if git_status.strip():

    raise RuntimeError(
        "Repository changed during source recovery:\n"
        + git_status
    )


# No Stage27-1 repository artifact directory should have been created.

for candidate in [

    REPO
    / "results"
    / "stage27_loao_unseen_attack"
    / "stage27_1_source_materialization",

    REPO
    / "results"
    / "stage27_loao_unseen_attack"
    / "stage27_1_fold_membership",

    REPO
    / "results"
    / "stage27_loao_unseen_attack"
    / "stage27_1_training",

]:

    if candidate.exists():

        raise RuntimeError(
            "Unexpected Stage27-1 repository artifact appeared:\n"
            f"{candidate}"
        )


print(
    "Exact source downloads     :",
    downloaded_count,
)

print(
    "Source substitutions       : 0"
)

print(
    "Raw dataset reconstruction : 0"
)

print(
    "Stage27 repo writes        : 0"
)

print(
    "Family labels processed    : 0"
)

print(
    "Fold memberships built     : 0"
)

print(
    "Features materialized      : 0"
)

print(
    "Model fits                 : 0"
)

print(
    "Model inference            : 0"
)

print(
    "Threshold searches         : 0"
)

print(
    "Target openings            : 0"
)

print(
    "Bootstrap replicates       : 0"
)

print(
    "GPU hours                  : 0"
)

print(
    "Git working tree           : CLEAN"
)


# ======================================================================================
# 9. FINAL
# ======================================================================================

banner("STAGE27-1A EXACT SOURCE RECOVERY COMPLETE")

print(
    "Scientific parent :",
    EXPECTED_PARENT,
)

print(
    "HF repository     :",
    EXPECTED_REPO_ID,
)

print(
    "Pinned revision   :",
    EXPECTED_REVISION,
)

print(
    "Verified assets   : 8 / 8"
)

print(
    "Canonical cache   :",
    DEST_ROOT,
)

print(
    "Population rows   : 2,830,743"
)

print(
    "Model fits        : 0"
)

print(
    "Target openings   : 0"
)

print(
    "GPU               : 0"
)

print()
print(
    "NEXT AUTHORIZED STEP:"
)

print(
    "  Stage27-1A membership materialization: "
    "read only the exact eight verified parquet sources, "
    "apply the frozen taxonomy/70-feature adapter, and construct "
    "deterministic row-level membership receipts for all five "
    "eligible folds."
)

print()
print(
    "NO MODEL FIT IS AUTHORIZED YET."
)


STAGE27-1A SOURCE RECOVERY :: SCIENTIFIC-PARENT GATE
Expected parent : 1ea6bedb141bc3fa6115edbc5296d0f7c6252559
Local HEAD      : 1ea6bedb141bc3fa6115edbc5296d0f7c6252559
origin/main     : 1ea6bedb141bc3fa6115edbc5296d0f7c6252559
Git clean       : True
[PASS] Frozen Stage27-0 parent confirmed.

STAGE27-1A SOURCE RECOVERY :: LOAD FROZEN CONTRACT
Hugging Face repo : bvsam/cic-ids-2017
Pinned revision   : b7e532345512edcd530cb1770dc76636aeb52802
Required assets   : 8
Substitution      : FORBIDDEN

[PASS] Frozen eight-asset contract recovered.

STAGE27-1A SOURCE RECOVERY :: PREPARE RUNTIME CACHE
Canonical cache : /kaggle/working/stage27_cicids2017_sources
HF cache        : /kaggle/working/stage27_hf_cache
Inside Git repo : False

STAGE27-1A SOURCE RECOVERY :: EXACT ACQUISITION

----------------------------------------------------------------------------------------------------------------------
[1/8] Monday :: Monday-WorkingHours.pcap_ISCX.csv.parquet
-------------------------------------

traffic_labels/Monday-WorkingHours.pcap_(…):   0%|          | 0.00/65.5M [00:00<?, ?B/s]

Downloaded cache path : /kaggle/working/stage27_hf_cache/datasets--bvsam--cic-ids-2017/blobs/dfdcef4b8670e52af54dc4f82174834365a393473e877174cca46d17b12dfd02
Downloaded size       : 65,465,382
Size gate             : PASS
Hashing downloaded bytes...
Downloaded SHA256     : dfdcef4b8670e52af54dc4f82174834365a393473e877174cca46d17b12dfd02
Expected SHA256       : dfdcef4b8670e52af54dc4f82174834365a393473e877174cca46d17b12dfd02
SHA gate              : PASS
Canonical materialization: HARDLINK_FROM_HF_CACHE
Canonical path          : /kaggle/working/stage27_cicids2017_sources/Monday-WorkingHours.pcap_ISCX.csv.parquet
Parquet rows            : 529,918
Expected rows           : 529,918
Parquet row groups      : 3
Parquet columns         : 85
Footer/schema gate      : PASS

FINAL ASSET STATUS      : BYTE-EXACT VERIFIED

----------------------------------------------------------------------------------------------------------------------
[2/8] Tuesday :: Tuesday-WorkingHours.pcap_ISCX.csv.parquet

traffic_labels/Tuesday-WorkingHours.pcap(…):   0%|          | 0.00/52.7M [00:00<?, ?B/s]

Downloaded cache path : /kaggle/working/stage27_hf_cache/datasets--bvsam--cic-ids-2017/blobs/27e83d518cb093faefd0f883cb4df3ad8b353f150934004f28d0e7962f9f31c4
Downloaded size       : 52,701,751
Size gate             : PASS
Hashing downloaded bytes...
Downloaded SHA256     : 27e83d518cb093faefd0f883cb4df3ad8b353f150934004f28d0e7962f9f31c4
Expected SHA256       : 27e83d518cb093faefd0f883cb4df3ad8b353f150934004f28d0e7962f9f31c4
SHA gate              : PASS
Canonical materialization: HARDLINK_FROM_HF_CACHE
Canonical path          : /kaggle/working/stage27_cicids2017_sources/Tuesday-WorkingHours.pcap_ISCX.csv.parquet
Parquet rows            : 445,909
Expected rows           : 445,909
Parquet row groups      : 2
Parquet columns         : 85
Footer/schema gate      : PASS

FINAL ASSET STATUS      : BYTE-EXACT VERIFIED

----------------------------------------------------------------------------------------------------------------------
[3/8] Wednesday :: Wednesday-workingHours.pcap_ISCX.csv.pa

...

traffic_labels/Friday-WorkingHours-After(…):   0%|          | 0.00/18.6M [00:00<?, ?B/s]

Downloaded cache path : /kaggle/working/stage27_hf_cache/datasets--bvsam--cic-ids-2017/blobs/4d78cee297c27f1a9947b9384793e587a46c7a3ea89db199553dabddc9835d4a
Downloaded size       : 18,632,427
Size gate             : PASS
Hashing downloaded bytes...
Downloaded SHA256     : 4d78cee297c27f1a9947b9384793e587a46c7a3ea89db199553dabddc9835d4a
Expected SHA256       : 4d78cee297c27f1a9947b9384793e587a46c7a3ea89db199553dabddc9835d4a
SHA gate              : PASS
Canonical materialization: HARDLINK_FROM_HF_CACHE
Canonical path          : /kaggle/working/stage27_cicids2017_sources/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv.parquet
Parquet rows            : 286,467
Expected rows           : 286,467
Parquet row groups      : 2
Parquet columns         : 85
Footer/schema gate      : PASS

FINAL ASSET STATUS      : BYTE-EXACT VERIFIED

----------------------------------------------------------------------------------------------------------------------
[8/8] Friday :: Friday-WorkingHours-Morn

traffic_labels/Friday-WorkingHours-Morni(…):   0%|          | 0.00/22.0M [00:00<?, ?B/s]

Downloaded cache path : /kaggle/working/stage27_hf_cache/datasets--bvsam--cic-ids-2017/blobs/2c00236b13a69f4b1c222b8f4a89451dc2148cb04a8cd0c45c2a87af51471774
Downloaded size       : 21,999,571
Size gate             : PASS
Hashing downloaded bytes...
Downloaded SHA256     : 2c00236b13a69f4b1c222b8f4a89451dc2148cb04a8cd0c45c2a87af51471774
Expected SHA256       : 2c00236b13a69f4b1c222b8f4a89451dc2148cb04a8cd0c45c2a87af51471774
SHA gate              : PASS
Canonical materialization: HARDLINK_FROM_HF_CACHE
Canonical path          : /kaggle/working/stage27_cicids2017_sources/Friday-WorkingHours-Morning.pcap_ISCX.csv.parquet
Parquet rows            : 191,033
Expected rows           : 191,033
Parquet row groups      : 1
Parquet columns         : 85
Footer/schema gate      : PASS

FINAL ASSET STATUS      : BYTE-EXACT VERIFIED

STAGE27-1A SOURCE RECOVERY :: GLOBAL 8/8 VERIFICATION
Verified assets       : 8/8
Downloaded this run   : 8
Reused exact files    : 0
Total verified bytes  : 305,935,672 

RuntimeError: Expected total CICIDS2017 population 2,830,743; found 3,119,345.

In [16]:
# ======================================================================================
# STAGE27-1A — SOURCE RECOVERY CLOSURE
#
# RECOVERY FROM:
#   incorrect assertion:
#       sum(PHYSICAL parquet rows) == Stage24 EFFECTIVE population
#
# CORRECT FROZEN SEMANTICS:
#   - 8 physical parquet assets total 3,119,345 rows
#   - Stage24 effective population = 2,830,743 rows
#   - Thursday WebAttacks contains a frozen all-85-column-null EOF suffix:
#         physical rows  = 458,968
#         effective rows = 170,366
#         excluded rows  = 288,602
#
# ZERO DOWNLOADS
# ZERO STAGE27 REPO WRITES
# ZERO FITS / INFERENCE / THRESHOLDS / TARGET OPENINGS / GPU
# ======================================================================================

from __future__ import annotations

import hashlib
import json
import re
import subprocess
from pathlib import Path

import pyarrow.parquet as pq


# ======================================================================================
# 0. CONSTANTS
# ======================================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_PARENT = (
    "1ea6bedb141bc3fa6115edbc5296d0f7c6252559"
)

SOURCE_ROOT = Path(
    "/kaggle/working/stage27_cicids2017_sources"
)

SOURCE_CONTRACT = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0b2_complete_cicids2017_source_contract.json"
)

STAGE24_PROTOCOL = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0c_final_preopening_protocol_lock.json"
)

EXPECTED_PHYSICAL_TOTAL = 3_119_345
EXPECTED_EFFECTIVE_TOTAL = 2_830_743

WEB_REMOTE = (
    "traffic_labels/"
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv.parquet"
)

WEB_EXPECTED_PHYSICAL = 458_968
WEB_EXPECTED_EFFECTIVE = 170_366
WEB_EXPECTED_EXCLUDED = 288_602


# ======================================================================================
# 1. HELPERS
# ======================================================================================

def banner(title):
    print()
    print("=" * 118)
    print(title)
    print("=" * 118)


def run(cmd):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=REPO,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    if p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}): {' '.join(map(str, cmd))}\n"
            f"STDOUT:\n{p.stdout}\n"
            f"STDERR:\n{p.stderr}"
        )

    return p.stdout.strip()


def load_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as f:
        return json.load(f)


def sha256_file(path, chunk_size=16 * 1024 * 1024):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            block = f.read(chunk_size)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


# ======================================================================================
# 2. SCIENTIFIC-PARENT GATE
# ======================================================================================

banner("STAGE27-1A CLOSURE :: SCIENTIFIC-PARENT GATE")

head = run(
    ["git", "rev-parse", "HEAD"]
)

origin = run(
    ["git", "rev-parse", "origin/main"]
)

status = run(
    [
        "git",
        "status",
        "--porcelain=v1",
        "--untracked-files=all",
    ]
)

print("Expected parent :", EXPECTED_PARENT)
print("Local HEAD      :", head)
print("origin/main     :", origin)
print("Git clean       :", not bool(status.strip()))

if head != EXPECTED_PARENT:
    raise RuntimeError(
        "Local HEAD changed from frozen Stage27-0."
    )

if origin != EXPECTED_PARENT:
    raise RuntimeError(
        "origin/main changed from frozen Stage27-0."
    )

if status.strip():
    raise RuntimeError(
        "Repository is dirty:\n" + status
    )

print("[PASS] Stage27-0 remains the clean scientific parent.")


# ======================================================================================
# 3. LOAD FROZEN CONTRACTS
# ======================================================================================

banner("STAGE27-1A CLOSURE :: LOAD FROZEN SOURCE / POPULATION RULES")

contract = load_json(
    SOURCE_CONTRACT
)

protocol = load_json(
    STAGE24_PROTOCOL
)

records = contract[
    "acquisition_records"
]

published_population = protocol[
    "published_population"
]

if len(records) != 8:
    raise RuntimeError(
        f"Expected 8 source records; found {len(records)}."
    )

if len(published_population) != 8:
    raise RuntimeError(
        "Frozen Stage24 published_population must contain 8 file records."
    )

print("Physical source records  :", len(records))
print("Effective-pop records    :", len(published_population))
print("[PASS] Frozen source/population rules loaded.")


# ======================================================================================
# 4. REVERIFY ALL EIGHT CANONICAL FILES
# ======================================================================================

banner("STAGE27-1A CLOSURE :: REVERIFY 8 BYTE-EXACT SOURCES")

physical_by_remote = {}
physical_total = 0

for index, rec in enumerate(records, start=1):

    remote = rec["remote"]

    path = (
        SOURCE_ROOT
        / Path(remote).name
    )

    if not path.is_file():
        raise RuntimeError(
            f"Canonical source is missing:\n{path}"
        )

    expected_size = int(
        rec["size_bytes"]
    )

    expected_sha = rec[
        "sha256"
    ]

    expected_rows = int(
        rec["physical_rows"]
    )

    actual_size = path.stat().st_size

    if actual_size != expected_size:
        raise RuntimeError(
            f"{remote}: byte-size mismatch."
        )

    actual_sha = sha256_file(
        path
    )

    if actual_sha != expected_sha:
        raise RuntimeError(
            f"{remote}: SHA256 mismatch."
        )

    parquet = pq.ParquetFile(
        path
    )

    actual_rows = int(
        parquet.metadata.num_rows
    )

    actual_cols = len(
        parquet.schema_arrow.names
    )

    if actual_rows != expected_rows:
        raise RuntimeError(
            f"{remote}: physical-row mismatch."
        )

    if actual_cols != 85:
        raise RuntimeError(
            f"{remote}: expected 85 columns, got {actual_cols}."
        )

    physical_by_remote[
        remote
    ] = actual_rows

    physical_total += actual_rows

    print(
        f"[{index}/8] "
        f"{rec['day']:10s} "
        f"rows={actual_rows:>9,}  "
        f"sha256={actual_sha[:16]}...  PASS"
    )


print()
print(
    "Physical row total:",
    f"{physical_total:,}",
)

if physical_total != EXPECTED_PHYSICAL_TOTAL:
    raise RuntimeError(
        f"Physical population mismatch: "
        f"expected {EXPECTED_PHYSICAL_TOTAL:,}, "
        f"got {physical_total:,}."
    )

print(
    "[PASS] 8/8 canonical parquet assets remain byte-exact."
)


# ======================================================================================
# 5. VERIFY FROZEN EFFECTIVE-ROW RULES
# ======================================================================================

banner("STAGE27-1A CLOSURE :: VERIFY EFFECTIVE-ROW RULES")

effective_total = 0
excluded_total = 0

effective_records = []

for item in published_population:

    remote = item[
        "remote"
    ]

    physical = int(
        item[
            "physical_rows"
        ]
    )

    effective = int(
        item[
            "effective_rows"
        ]
    )

    rule = item[
        "inclusion_rule"
    ]

    if remote not in physical_by_remote:
        raise RuntimeError(
            f"Effective population references unknown source:\n{remote}"
        )

    if physical_by_remote[
        remote
    ] != physical:
        raise RuntimeError(
            f"{remote}: frozen physical count disagrees "
            "with verified parquet footer."
        )

    if effective < 0 or effective > physical:
        raise RuntimeError(
            f"{remote}: impossible effective-row count."
        )

    excluded = (
        physical
        - effective
    )

    effective_total += effective
    excluded_total += excluded

    effective_records.append(
        {
            "day": item[
                "day"
            ],
            "remote": remote,
            "physical": physical,
            "effective": effective,
            "excluded": excluded,
            "rule": rule,
        }
    )

    print()
    print(
        f"{item['day']} :: "
        f"{Path(remote).name}"
    )

    print(
        "  physical :",
        f"{physical:,}",
    )

    print(
        "  effective:",
        f"{effective:,}",
    )

    print(
        "  excluded :",
        f"{excluded:,}",
    )

    print(
        "  rule     :",
        rule,
    )


print()
print(
    "Physical total :",
    f"{physical_total:,}",
)

print(
    "Effective total:",
    f"{effective_total:,}",
)

print(
    "Excluded total :",
    f"{excluded_total:,}",
)


if effective_total != EXPECTED_EFFECTIVE_TOTAL:
    raise RuntimeError(
        f"Effective population mismatch: "
        f"expected {EXPECTED_EFFECTIVE_TOTAL:,}, "
        f"got {effective_total:,}."
    )


if (
    physical_total
    - effective_total
) != excluded_total:

    raise RuntimeError(
        "Physical/effective exclusion arithmetic failed."
    )


print()
print(
    "[PASS] Effective Stage24 population reconciles exactly."
)


# ======================================================================================
# 6. EXPLICIT THURSDAY WEBATTACKS STRUCTURAL-NULL RULE
# ======================================================================================

banner("STAGE27-1A CLOSURE :: THURSDAY STRUCTURAL-NULL RULE")

web_matches = [
    x
    for x in effective_records
    if x["remote"] == WEB_REMOTE
]

if len(web_matches) != 1:
    raise RuntimeError(
        "Could not identify unique Thursday WebAttacks population rule."
    )

web = web_matches[0]

print(
    "Physical rows :",
    f"{web['physical']:,}",
)

print(
    "Effective rows:",
    f"{web['effective']:,}",
)

print(
    "Excluded rows :",
    f"{web['excluded']:,}",
)

print(
    "Frozen rule   :",
    web["rule"],
)


if web[
    "physical"
] != WEB_EXPECTED_PHYSICAL:
    raise RuntimeError(
        "Thursday WebAttacks physical count mismatch."
    )

if web[
    "effective"
] != WEB_EXPECTED_EFFECTIVE:
    raise RuntimeError(
        "Thursday WebAttacks effective count mismatch."
    )

if web[
    "excluded"
] != WEB_EXPECTED_EXCLUDED:
    raise RuntimeError(
        "Thursday WebAttacks excluded suffix count mismatch."
    )


expected_rule = (
    "INCLUDE_PHYSICAL_ORDINALS_1_THROUGH_170366;"
    "EXCLUDE_FROZEN_ALL_85_COLUMN_NULL_EOF_SUFFIX_170367_THROUGH_458968"
)

if web[
    "rule"
] != expected_rule:
    raise RuntimeError(
        "Unexpected Thursday WebAttacks inclusion rule."
    )


print()
print(
    "[PASS] Frozen Thursday structural-null EOF exclusion "
    "verified exactly."
)


# ======================================================================================
# 7. DAY-LEVEL EFFECTIVE TOTALS
# ======================================================================================

banner("STAGE27-1A CLOSURE :: EFFECTIVE DAY TOTALS")

effective_by_day = {}

for rec in effective_records:

    day = rec[
        "day"
    ]

    effective_by_day[
        day
    ] = (
        effective_by_day.get(
            day,
            0,
        )
        + rec[
            "effective"
        ]
    )


expected_day_totals = {
    "Monday": 529_918,
    "Tuesday": 445_909,
    "Wednesday": 692_703,
    "Thursday": 458_968,
    "Friday": 703_245,
}


for day in [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
]:

    actual = effective_by_day[
        day
    ]

    expected = expected_day_totals[
        day
    ]

    print(
        f"{day:10s}: "
        f"{actual:>9,} "
        f"(expected {expected:,})"
    )

    if actual != expected:
        raise RuntimeError(
            f"{day}: effective day-total mismatch."
        )


if sum(
    effective_by_day.values()
) != EXPECTED_EFFECTIVE_TOTAL:
    raise RuntimeError(
        "Effective day totals do not reconcile globally."
    )


print()
print(
    "[PASS] All five effective weekday populations reconcile."
)


# ======================================================================================
# 8. IMPORTANT PHYSICAL/EFFECTIVE DISTINCTION
# ======================================================================================

banner("STAGE27-1A CLOSURE :: POPULATION SEMANTICS")

print(
    "Physical parquet rows       :",
    f"{physical_total:,}",
)

print(
    "Frozen effective rows       :",
    f"{effective_total:,}",
)

print(
    "Structurally excluded rows  :",
    f"{excluded_total:,}",
)

print()

print(
    "Physical source integrity and effective modeling population "
    "are intentionally different quantities."
)

print(
    "The 8 downloaded parquet files MUST remain byte-identical; "
    "the structural-null suffix is excluded logically during "
    "effective-row materialization and MUST NOT be removed or "
    "rewritten from the source parquet."
)


# ======================================================================================
# 9. FINAL NON-ACTION AUDIT
# ======================================================================================

banner("STAGE27-1A CLOSURE :: NON-ACTION AUDIT")

status_after = run(
    [
        "git",
        "status",
        "--porcelain=v1",
        "--untracked-files=all",
    ]
)

if status_after.strip():
    raise RuntimeError(
        "Repository changed during source closure:\n"
        + status_after
    )


print("Additional downloads      : 0")
print("Source substitutions      : 0")
print("Source parquet rewrites   : 0")
print("Stage27 repo writes       : 0")
print("Family memberships built  : 0")
print("Features materialized     : 0")
print("Model fits                : 0")
print("Model inference           : 0")
print("Threshold searches        : 0")
print("Target openings           : 0")
print("Bootstrap replicates      : 0")
print("GPU hours                 : 0")
print("Git working tree          : CLEAN")


# ======================================================================================
# 10. FINAL
# ======================================================================================

banner("STAGE27-1A EXACT SOURCE RECOVERY — VERIFIED COMPLETE")

print(
    "Scientific parent       :",
    EXPECTED_PARENT,
)

print(
    "Canonical source assets : 8 / 8 BYTE-EXACT"
)

print(
    "Physical parquet rows   :",
    f"{physical_total:,}",
)

print(
    "Effective population    :",
    f"{effective_total:,}",
)

print(
    "Structural-null excluded:",
    f"{excluded_total:,}",
)

print(
    "Canonical source cache  :",
    SOURCE_ROOT,
)

print()
print(
    "SOURCE ACQUISITION STATUS: COMPLETE"
)

print()
print(
    "NEXT AUTHORIZED STEP:"
)

print(
    "  Stage27-1A fold-membership materialization using the "
    "frozen effective-row rules and taxonomy."
)

print()
print(
    "NO MODEL FIT IS AUTHORIZED YET."
)


STAGE27-1A CLOSURE :: SCIENTIFIC-PARENT GATE
Expected parent : 1ea6bedb141bc3fa6115edbc5296d0f7c6252559
Local HEAD      : 1ea6bedb141bc3fa6115edbc5296d0f7c6252559
origin/main     : 1ea6bedb141bc3fa6115edbc5296d0f7c6252559
Git clean       : True
[PASS] Stage27-0 remains the clean scientific parent.

STAGE27-1A CLOSURE :: LOAD FROZEN SOURCE / POPULATION RULES
Physical source records  : 8
Effective-pop records    : 8
[PASS] Frozen source/population rules loaded.

STAGE27-1A CLOSURE :: REVERIFY 8 BYTE-EXACT SOURCES
[1/8] Monday     rows=  529,918  sha256=dfdcef4b8670e52a...  PASS
[2/8] Tuesday    rows=  445,909  sha256=27e83d518cb093fa...  PASS
[3/8] Wednesday  rows=  692,703  sha256=d23a259820b16e1a...  PASS
[4/8] Thursday   rows=  288,602  sha256=5da010354f0fc104...  PASS
[5/8] Thursday   rows=  458,968  sha256=d8110c04a7af9112...  PASS
[6/8] Friday     rows=  225,745  sha256=7c5876d52189fc01...  PASS
[7/8] Friday     rows=  286,467  sha256=4d78cee297c27f1a...  PASS
[8/8] Friday     row

In [17]:
# ======================================================================================
# STAGE27-1A — FIVE-FOLD MEMBERSHIP MATERIALIZATION + EXCLUSION RECEIPTS
#
# Scientific parent:
#   1ea6bedb141bc3fa6115edbc5296d0f7c6252559
#
# PRECONDITION
# ------------
# Eight byte-exact CICIDS2017 parquet sources already exist under:
#   /kaggle/working/stage27_cicids2017_sources
#
# THIS STAGE READS ONLY:
#   - Parquet metadata
#   - Label column for frozen effective rows
#   - committed Stage27-0 / Stage27-0A protocol artifacts
#
# IT DOES NOT READ:
#   - any of the 70 model predictors
#
# IT PERFORMS:
#   - ZERO model fits
#   - ZERO inference
#   - ZERO threshold selection
#   - ZERO target predictor openings
#   - ZERO bootstrap
#   - ZERO GPU
#
# OUTPUT
# ------
# Runtime cache OUTSIDE Git:
#   /kaggle/working/stage27_1a_membership_cache
#
# Small durable receipts INSIDE Git:
#   results/stage27_loao_unseen_attack/stage27_1a_fold_membership/
#
# Exactly nine receipt JSON files are created and staged, but NOT committed/pushed.
# ======================================================================================

from __future__ import annotations

import csv
import hashlib
import json
import os
import re
import shutil
import subprocess
import unicodedata
from collections import Counter, OrderedDict, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Tuple

import numpy as np
import pyarrow.parquet as pq


# ======================================================================================
# 0. CONSTANTS
# ======================================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_PARENT = (
    "1ea6bedb141bc3fa6115edbc5296d0f7c6252559"
)

SOURCE_ROOT = Path(
    "/kaggle/working/stage27_cicids2017_sources"
).resolve()

CACHE_ROOT = Path(
    "/kaggle/working/stage27_1a_membership_cache"
).resolve()

STAGE27_ROOT_REL = Path(
    "results/stage27_loao_unseen_attack"
)

STAGE27_0_REL = (
    STAGE27_ROOT_REL
    / "stage27_0_protocol_lock"
)

STAGE27_0A_REL = (
    STAGE27_ROOT_REL
    / "stage27_0a_family_day_feasibility"
)

OUT_REL = (
    STAGE27_ROOT_REL
    / "stage27_1a_fold_membership"
)

OUT_DIR = (
    REPO
    / OUT_REL
)

FOLD_SPEC_PATH = (
    REPO
    / STAGE27_0_REL
    / "fold_spec.json"
)

TAXONOMY_PATH = (
    REPO
    / STAGE27_0_REL
    / "taxonomy.json"
)

TARGET_SPEC_PATH = (
    REPO
    / STAGE27_0_REL
    / "target_population_spec.json"
)

CLASS_WEIGHT_PATH = (
    REPO
    / STAGE27_0_REL
    / "class_weight_policy.json"
)

FEATURE_SPEC_PATH = (
    REPO
    / STAGE27_0_REL
    / "feature_representation.json"
)

FREEZE_PATH = (
    REPO
    / STAGE27_0_REL
    / "freeze_record.json"
)

STAGE24_SOURCE_CONTRACT = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0b2_complete_cicids2017_source_contract.json"
)

STAGE24_PROTOCOL = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0c_final_preopening_protocol_lock.json"
)

STAGE27_0A_TEMPORAL = (
    REPO
    / STAGE27_0A_REL
    / "temporal_feasibility.json"
)

STAGE27_0A_FAMILY_SUPPORT = (
    REPO
    / STAGE27_0A_REL
    / "family_day_support.csv"
)

STAGE27_0A_BENIGN_SUPPORT = (
    REPO
    / STAGE27_0A_REL
    / "benign_day_support.csv"
)

EXPECTED_EFFECTIVE_ROWS = 2_830_743
EXPECTED_BENIGN = 2_273_097
EXPECTED_PRIMARY_ATTACK = 557_635
EXPECTED_HEARTBLEED = 11
EXPECTED_ALL_NONBENIGN = 557_646
EXPECTED_PHYSICAL_ROWS = 3_119_345
EXPECTED_STRUCTURAL_EXCLUDED = 288_602

ELIGIBLE_FOLDS = [
    "BOT",
    "DDOS",
    "INFILTRATION",
    "PORT_SCAN",
    "WEB_ATTACK",
]

STRUCTURALLY_INELIGIBLE = [
    "DOS",
    "AUTH_BRUTE_FORCE",
]

DESCRIPTIVE_ONLY = [
    "INFILTRATION",
]

DAY_ORDER = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
]

DAY_CODE = {
    day: i + 1
    for i, day in enumerate(DAY_ORDER)
}

# Family-code representation used ONLY for deterministic membership construction.
# The model target remains binary attack-vs-benign.
FAMILY_CODE = OrderedDict(
    [
        ("BENIGN", 0),
        ("BOT", 1),
        ("DDOS", 2),
        ("DOS", 3),
        ("AUTH_BRUTE_FORCE", 4),
        ("INFILTRATION", 5),
        ("PORT_SCAN", 6),
        ("WEB_ATTACK", 7),
        ("TARGET_ONLY_UNSEEN", 8),
        ("OTHER_ATTACK_UNSEEN_LABEL", 9),
    ]
)

CODE_FAMILY = {
    value: key
    for key, value in FAMILY_CODE.items()
}

PRIMARY_FAMILY_CODES = np.array(
    [
        FAMILY_CODE["BOT"],
        FAMILY_CODE["DDOS"],
        FAMILY_CODE["DOS"],
        FAMILY_CODE["AUTH_BRUTE_FORCE"],
        FAMILY_CODE["INFILTRATION"],
        FAMILY_CODE["PORT_SCAN"],
        FAMILY_CODE["WEB_ATTACK"],
    ],
    dtype=np.uint8,
)

REQUIRED_OUTPUT_FILES = [
    "source_effective_population_receipt.json",
    "canonical_label_census.json",
    "family_code_cache_receipt.json",
    "fold_BOT_membership_receipt.json",
    "fold_DDOS_membership_receipt.json",
    "fold_INFILTRATION_membership_receipt.json",
    "fold_PORT_SCAN_membership_receipt.json",
    "fold_WEB_ATTACK_membership_receipt.json",
    "stage27_1a_membership_freeze_record.json",
]

os.environ["CUDA_VISIBLE_DEVICES"] = ""


# ======================================================================================
# 1. HELPERS
# ======================================================================================

def banner(title: str) -> None:
    print()
    print("=" * 118)
    print(title)
    print("=" * 118)


def run(
    cmd: List[str],
    *,
    cwd: Path = REPO,
    check: bool = True,
) -> str:
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}): {' '.join(map(str, cmd))}\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return p.stdout.strip()


def now_utc() -> str:
    return datetime.now(timezone.utc).isoformat()


def load_json(path: Path) -> Any:
    with path.open(
        "r",
        encoding="utf-8",
    ) as f:
        return json.load(f)


def write_json(
    path: Path,
    obj: Any,
) -> None:
    path.write_text(
        json.dumps(
            obj,
            ensure_ascii=False,
            indent=2,
            sort_keys=False,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )


def sha256_file(
    path: Path,
    chunk_size: int = 16 * 1024 * 1024,
) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def content_sha256_array(arr: np.ndarray) -> str:
    contiguous = np.ascontiguousarray(arr)
    return sha256_bytes(
        contiguous.tobytes(order="C")
    )


def file_receipt(path: Path) -> Dict[str, Any]:
    return {
        "path": str(path),
        "bytes": int(path.stat().st_size),
        "sha256": sha256_file(path),
    }


def save_npy_with_receipt(
    path: Path,
    arr: np.ndarray,
) -> Dict[str, Any]:

    np.save(
        path,
        arr,
        allow_pickle=False,
    )

    return {
        "path": str(path),
        "dtype": str(arr.dtype),
        "shape": list(arr.shape),
        "count": int(arr.size),
        "content_sha256": content_sha256_array(arr),
        "npy_bytes": int(path.stat().st_size),
        "npy_sha256": sha256_file(path),
    }


def normalize_unicode_dashes(text: str) -> str:
    dash_chars = (
        "\u2010"  # hyphen
        "\u2011"  # non-breaking hyphen
        "\u2012"  # figure dash
        "\u2013"  # en dash
        "\u2014"  # em dash
        "\u2015"  # horizontal bar
        "\u2212"  # minus sign
        "\ufe58"  # small em dash
        "\ufe63"  # small hyphen-minus
        "\uff0d"  # fullwidth hyphen-minus
    )

    table = str.maketrans(
        {
            ch: "-"
            for ch in dash_chars
        }
    )

    return text.translate(table)


def canonicalize_label(value: Any) -> str:
    if value is None:
        raise RuntimeError(
            "Null Label encountered inside the frozen effective population."
        )

    text = str(value)

    text = normalize_unicode_dashes(text)

    text = text.strip()

    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    if not text:
        raise RuntimeError(
            "Empty Label encountered inside the frozen effective population."
        )

    return text.casefold()


def load_support_csv(
    path: Path,
) -> List[Dict[str, str]]:

    with path.open(
        "r",
        encoding="utf-8",
        newline="",
    ) as f:
        return list(
            csv.DictReader(f)
        )


def parse_family_support(
    rows: List[Dict[str, str]],
) -> Dict[str, Dict[str, int]]:

    out = {}

    for row in rows:

        lower_keys = {
            k.casefold(): k
            for k in row.keys()
        }

        family_key = (
            lower_keys.get("family")
            or lower_keys.get("attack_family")
        )

        if family_key is None:
            raise RuntimeError(
                "Could not find family column in family_day_support.csv"
            )

        family = row[
            family_key
        ].strip()

        per_day = {}

        for day in DAY_ORDER:

            source_key = None

            for k in row.keys():
                if k.casefold() == day.casefold():
                    source_key = k
                    break

            if source_key is None:
                raise RuntimeError(
                    f"Missing {day} column in family_day_support.csv"
                )

            per_day[
                day
            ] = int(
                row[
                    source_key
                ]
            )

        out[
            family
        ] = per_day

    return out


def parse_benign_support(
    rows: List[Dict[str, str]],
) -> Dict[str, int]:

    # Supports either:
    #   day,benign_support
    # or a single row with Monday..Friday columns.
    if not rows:
        raise RuntimeError(
            "benign_day_support.csv is empty."
        )

    result = {}

    keys = list(
        rows[0].keys()
    )

    lower = {
        key.casefold(): key
        for key in keys
    }

    if "day" in lower:

        day_key = lower["day"]

        value_key = (
            lower.get("benign_support")
            or lower.get("benign")
            or lower.get("count")
        )

        if value_key is None:
            raise RuntimeError(
                "Could not identify benign count column."
            )

        for row in rows:
            day = row[day_key].strip()
            result[day] = int(
                row[value_key]
            )

    else:

        row = rows[0]

        for day in DAY_ORDER:

            source_key = None

            for key in keys:
                if key.casefold() == day.casefold():
                    source_key = key
                    break

            if source_key is None:
                raise RuntimeError(
                    f"Could not find {day} in benign support CSV."
                )

            result[day] = int(
                row[source_key]
            )

    return result


def safe_clean_directory(
    path: Path,
) -> None:

    if path.exists():
        # Runtime cache is explicitly non-durable and may be rebuilt,
        # but only when it is exactly the Stage27-1A cache path.
        expected = Path(
            "/kaggle/working/stage27_1a_membership_cache"
        ).resolve()

        if path.resolve() != expected:
            raise RuntimeError(
                f"Refusing to clean unexpected path: {path}"
            )

        shutil.rmtree(path)

    path.mkdir(
        parents=True,
        exist_ok=False,
    )


# ======================================================================================
# 2. SCIENTIFIC PARENT / CLEAN GATE
# ======================================================================================

banner("STAGE27-1A MEMBERSHIP :: SCIENTIFIC-PARENT GATE")

head = run(
    ["git", "rev-parse", "HEAD"]
)

origin = run(
    ["git", "rev-parse", "origin/main"]
)

status = run(
    [
        "git",
        "status",
        "--porcelain=v1",
        "--untracked-files=all",
    ]
)

print("Expected parent :", EXPECTED_PARENT)
print("Local HEAD      :", head)
print("origin/main     :", origin)
print("Git clean       :", not bool(status.strip()))

if head != EXPECTED_PARENT:
    raise RuntimeError(
        "Local HEAD is not the frozen Stage27-0 commit."
    )

if origin != EXPECTED_PARENT:
    raise RuntimeError(
        "origin/main is not the frozen Stage27-0 commit."
    )

if status.strip():
    raise RuntimeError(
        "Repository is dirty before Stage27-1A membership:\n"
        + status
    )

print("[PASS] Frozen Stage27-0 is the clean scientific parent.")


# ======================================================================================
# 3. OUTPUT / CACHE GATES
# ======================================================================================

banner("STAGE27-1A MEMBERSHIP :: OUTPUT GATES")

if OUT_DIR.exists():
    raise RuntimeError(
        f"Stage27-1A durable receipt directory already exists:\n{OUT_DIR}\n"
        "Refusing overwrite."
    )

if not SOURCE_ROOT.is_dir():
    raise RuntimeError(
        f"Canonical source cache is missing:\n{SOURCE_ROOT}"
    )

safe_clean_directory(
    CACHE_ROOT
)

print("Source cache      :", SOURCE_ROOT)
print("Membership cache  :", CACHE_ROOT)
print("Durable output    :", OUT_DIR)
print("Repo writes yet   : 0")


# ======================================================================================
# 4. LOAD FROZEN STAGE27-0 ARTIFACTS
# ======================================================================================

banner("STAGE27-1A MEMBERSHIP :: LOAD FROZEN PROTOCOL")

required_protocol_files = [
    FOLD_SPEC_PATH,
    TAXONOMY_PATH,
    TARGET_SPEC_PATH,
    CLASS_WEIGHT_PATH,
    FEATURE_SPEC_PATH,
    FREEZE_PATH,
    STAGE24_SOURCE_CONTRACT,
    STAGE24_PROTOCOL,
    STAGE27_0A_TEMPORAL,
    STAGE27_0A_FAMILY_SUPPORT,
    STAGE27_0A_BENIGN_SUPPORT,
]

for path in required_protocol_files:
    if not path.is_file():
        raise RuntimeError(
            f"Required frozen artifact missing: {path}"
        )

fold_spec = load_json(
    FOLD_SPEC_PATH
)

taxonomy = load_json(
    TAXONOMY_PATH
)

target_spec = load_json(
    TARGET_SPEC_PATH
)

class_weight_policy = load_json(
    CLASS_WEIGHT_PATH
)

feature_spec = load_json(
    FEATURE_SPEC_PATH
)

freeze = load_json(
    FREEZE_PATH
)

source_contract = load_json(
    STAGE24_SOURCE_CONTRACT
)

stage24_protocol = load_json(
    STAGE24_PROTOCOL
)

temporal_0a = load_json(
    STAGE27_0A_TEMPORAL
)

family_support_expected = parse_family_support(
    load_support_csv(
        STAGE27_0A_FAMILY_SUPPORT
    )
)

benign_support_expected = parse_benign_support(
    load_support_csv(
        STAGE27_0A_BENIGN_SUPPORT
    )
)

if freeze["authorized_fit_budget"] != 10:
    raise RuntimeError(
        "Stage27 fit budget changed."
    )

if freeze["model_fits_completed"] != 0:
    raise RuntimeError(
        "Stage27-0 freeze says fits were already completed."
    )

if freeze["model_inference_completed"] != 0:
    raise RuntimeError(
        "Stage27-0 freeze says inference was already completed."
    )

if freeze["target_openings_consumed"] != 0:
    raise RuntimeError(
        "Stage27 target-opening ledger is no longer pristine."
    )

if feature_spec["feature_count"] != 70:
    raise RuntimeError(
        "Frozen feature count is no longer 70."
    )

print("Eligible folds       :", fold_spec["eligible_fold_count"])
print("Authorized fits      :", freeze["authorized_fit_budget"])
print("Feature count        :", feature_spec["feature_count"])
print("Fits completed       :", freeze["model_fits_completed"])
print("Target openings      :", freeze["target_openings_consumed"])
print("[PASS] Frozen Stage27-0 protocol loaded.")


# ======================================================================================
# 5. VERIFY TAXONOMY + BUILD CANONICAL LOOKUP
# ======================================================================================

banner("STAGE27-1A MEMBERSHIP :: TAXONOMY GATE")

primary_order = taxonomy[
    "primary_family_order"
]

if primary_order != [
    "BOT",
    "DDOS",
    "DOS",
    "AUTH_BRUTE_FORCE",
    "INFILTRATION",
    "PORT_SCAN",
    "WEB_ATTACK",
]:
    raise RuntimeError(
        "Frozen primary family order changed."
    )

raw_mapping = taxonomy[
    "raw_label_to_family"
]

canonical_mapping = {}

for raw_label, family in raw_mapping.items():

    canonical_key = canonicalize_label(
        raw_label
    )

    if (
        canonical_key in canonical_mapping
        and canonical_mapping[canonical_key] != family
    ):
        raise RuntimeError(
            f"Canonical taxonomy collision: {canonical_key}"
        )

    canonical_mapping[
        canonical_key
    ] = family


for family in primary_order:

    if family not in FAMILY_CODE:
        raise RuntimeError(
            f"No family code for {family}"
        )


print("Canonical taxonomy entries :", len(canonical_mapping))
print("Primary families           :", primary_order)
print("Heartbleed policy          :", taxonomy["heartbleed_policy"])
print("Unknown nonbenign policy   :", taxonomy["unknown_nonbenign_family_policy"])
print("[PASS] Frozen taxonomy is executable.")


# ======================================================================================
# 6. RECOVER 8 SOURCE SEGMENTS + EFFECTIVE RULES
# ======================================================================================

banner("STAGE27-1A MEMBERSHIP :: SOURCE SEGMENT CONTRACT")

acquisition_records = source_contract[
    "acquisition_records"
]

published_population = stage24_protocol[
    "published_population"
]

if len(acquisition_records) != 8:
    raise RuntimeError(
        "Expected 8 source acquisition records."
    )

if len(published_population) != 8:
    raise RuntimeError(
        "Expected 8 effective-population records."
    )

acq_by_remote = {
    rec["remote"]: rec
    for rec in acquisition_records
}

segments = []

global_cursor = 0
physical_total = 0
effective_total = 0
excluded_total = 0

for source_index, pop in enumerate(
    published_population
):

    remote = pop["remote"]

    if remote not in acq_by_remote:
        raise RuntimeError(
            f"Population source absent from acquisition contract: {remote}"
        )

    acq = acq_by_remote[
        remote
    ]

    path = (
        SOURCE_ROOT
        / Path(remote).name
    )

    if not path.is_file():
        raise RuntimeError(
            f"Canonical source missing: {path}"
        )

    actual_size = path.stat().st_size

    if actual_size != int(
        acq["size_bytes"]
    ):
        raise RuntimeError(
            f"{remote}: source size mismatch."
        )

    actual_sha = sha256_file(
        path
    )

    if actual_sha != acq[
        "sha256"
    ]:
        raise RuntimeError(
            f"{remote}: source SHA256 mismatch."
        )

    parquet = pq.ParquetFile(
        path
    )

    actual_physical = int(
        parquet.metadata.num_rows
    )

    expected_physical = int(
        pop["physical_rows"]
    )

    if actual_physical != expected_physical:
        raise RuntimeError(
            f"{remote}: physical row mismatch."
        )

    effective_rows = int(
        pop["effective_rows"]
    )

    if not (
        0
        <= effective_rows
        <= actual_physical
    ):
        raise RuntimeError(
            f"{remote}: impossible effective row count."
        )

    global_start = global_cursor

    global_stop = (
        global_start
        + effective_rows
    )

    segment = {
        "source_index": source_index,
        "day": pop["day"],
        "day_code": DAY_CODE[
            pop["day"]
        ],
        "remote": remote,
        "basename": Path(remote).name,
        "path": str(path),
        "sha256": actual_sha,
        "size_bytes": actual_size,
        "physical_rows": actual_physical,
        "effective_rows": effective_rows,
        "excluded_rows": (
            actual_physical
            - effective_rows
        ),
        "inclusion_rule": pop[
            "inclusion_rule"
        ],
        "global_start_zero_based": global_start,
        "global_stop_exclusive": global_stop,
        "physical_ordinal_first_included": (
            1
            if effective_rows > 0
            else None
        ),
        "physical_ordinal_last_included": (
            effective_rows
            if effective_rows > 0
            else None
        ),
    }

    segments.append(
        segment
    )

    global_cursor = global_stop

    physical_total += actual_physical
    effective_total += effective_rows
    excluded_total += (
        actual_physical
        - effective_rows
    )

    print(
        f"[{source_index + 1}/8] "
        f"{pop['day']:10s} "
        f"effective={effective_rows:>9,} "
        f"global=[{global_start:,},{global_stop:,}) "
        f"{Path(remote).name}"
    )


if physical_total != EXPECTED_PHYSICAL_ROWS:
    raise RuntimeError(
        f"Physical total mismatch: {physical_total:,}"
    )

if effective_total != EXPECTED_EFFECTIVE_ROWS:
    raise RuntimeError(
        f"Effective total mismatch: {effective_total:,}"
    )

if excluded_total != EXPECTED_STRUCTURAL_EXCLUDED:
    raise RuntimeError(
        f"Structural exclusion mismatch: {excluded_total:,}"
    )

print()
print("Physical rows :", f"{physical_total:,}")
print("Effective rows:", f"{effective_total:,}")
print("Excluded rows :", f"{excluded_total:,}")
print("[PASS] Eight exact source segments frozen.")


# ======================================================================================
# 7. MATERIALIZE ONLY LABEL-DERIVED CODE ARRAYS
# ======================================================================================

banner("STAGE27-1A MEMBERSHIP :: LABEL-ONLY EFFECTIVE POPULATION MATERIALIZATION")

family_codes = np.empty(
    EXPECTED_EFFECTIVE_ROWS,
    dtype=np.uint8,
)

day_codes = np.empty(
    EXPECTED_EFFECTIVE_ROWS,
    dtype=np.uint8,
)

source_codes = np.empty(
    EXPECTED_EFFECTIVE_ROWS,
    dtype=np.uint8,
)

raw_label_counts = Counter()
family_counts = Counter()
family_day_counts = defaultdict(Counter)
benign_day_counts = Counter()

# Additional audit counters.
null_labels = 0
empty_labels = 0
unknown_nonbenign = 0

for segment in segments:

    path = Path(
        segment["path"]
    )

    start = segment[
        "global_start_zero_based"
    ]

    stop = segment[
        "global_stop_exclusive"
    ]

    n_effective = segment[
        "effective_rows"
    ]

    day = segment[
        "day"
    ]

    local_cursor = 0

    parquet = pq.ParquetFile(
        path
    )

    print()
    print(
        f"Reading Label only: "
        f"{day} :: {segment['basename']}"
    )

    for batch in parquet.iter_batches(
        batch_size=131_072,
        columns=["Label"],
        use_threads=True,
    ):

        if local_cursor >= n_effective:
            break

        values = batch.column(
            0
        ).to_pylist()

        remaining = (
            n_effective
            - local_cursor
        )

        if len(values) > remaining:
            values = values[
                :remaining
            ]

        batch_codes = np.empty(
            len(values),
            dtype=np.uint8,
        )

        for i, value in enumerate(
            values
        ):

            if value is None:
                null_labels += 1
                raise RuntimeError(
                    f"Null effective Label in {segment['basename']} "
                    f"at physical ordinal {local_cursor + i + 1:,}"
                )

            canonical = canonicalize_label(
                value
            )

            if not canonical:
                empty_labels += 1
                raise RuntimeError(
                    "Empty canonical label encountered."
                )

            raw_label_counts[
                canonical
            ] += 1

            if canonical == "benign":

                family = "BENIGN"

            elif canonical in canonical_mapping:

                family = canonical_mapping[
                    canonical
                ]

            else:

                family = taxonomy[
                    "unknown_nonbenign_family_policy"
                ]

                unknown_nonbenign += 1

            if family not in FAMILY_CODE:
                raise RuntimeError(
                    f"Unrecognized frozen family identity: {family}"
                )

            code = FAMILY_CODE[
                family
            ]

            batch_codes[
                i
            ] = code

            family_counts[
                family
            ] += 1

            if family == "BENIGN":

                benign_day_counts[
                    day
                ] += 1

            elif family in primary_order:

                family_day_counts[
                    family
                ][
                    day
                ] += 1

            elif family == "TARGET_ONLY_UNSEEN":

                # Heartbleed remains separate from the primary 7-family matrix.
                pass

            elif family == "OTHER_ATTACK_UNSEEN_LABEL":

                pass

            else:

                raise RuntimeError(
                    f"Unexpected family branch: {family}"
                )

        global_batch_start = (
            start
            + local_cursor
        )

        global_batch_stop = (
            global_batch_start
            + len(batch_codes)
        )

        family_codes[
            global_batch_start:
            global_batch_stop
        ] = batch_codes

        day_codes[
            global_batch_start:
            global_batch_stop
        ] = np.uint8(
            segment[
                "day_code"
            ]
        )

        source_codes[
            global_batch_start:
            global_batch_stop
        ] = np.uint8(
            segment[
                "source_index"
            ]
        )

        local_cursor += len(
            batch_codes
        )

    if local_cursor != n_effective:
        raise RuntimeError(
            f"{segment['basename']}: expected to materialize "
            f"{n_effective:,} effective labels, got {local_cursor:,}."
        )

    print(
        f"  effective labels materialized: {local_cursor:,}"
    )


print()
print("Total label rows:", f"{family_codes.size:,}")

if family_codes.size != EXPECTED_EFFECTIVE_ROWS:
    raise RuntimeError(
        "Global family-code array length mismatch."
    )

print("[PASS] Effective population materialized from Label only.")


# ======================================================================================
# 8. GLOBAL CENSUS VALIDATION
# ======================================================================================

banner("STAGE27-1A MEMBERSHIP :: GLOBAL TAXONOMY CENSUS")

for family in [
    "BENIGN",
    "BOT",
    "DDOS",
    "DOS",
    "AUTH_BRUTE_FORCE",
    "INFILTRATION",
    "PORT_SCAN",
    "WEB_ATTACK",
    "TARGET_ONLY_UNSEEN",
    "OTHER_ATTACK_UNSEEN_LABEL",
]:

    print(
        f"{family:28s}: "
        f"{family_counts.get(family, 0):>10,}"
    )


if family_counts["BENIGN"] != EXPECTED_BENIGN:
    raise RuntimeError(
        "Global benign count mismatch."
    )

primary_total = sum(
    family_counts[
        family
    ]
    for family in primary_order
)

if primary_total != EXPECTED_PRIMARY_ATTACK:
    raise RuntimeError(
        f"Primary seven-family attack total mismatch: {primary_total:,}"
    )

if family_counts[
    "TARGET_ONLY_UNSEEN"
] != EXPECTED_HEARTBLEED:
    raise RuntimeError(
        "TARGET_ONLY_UNSEEN / Heartbleed count mismatch."
    )

if family_counts[
    "OTHER_ATTACK_UNSEEN_LABEL"
] != 0:
    raise RuntimeError(
        "Unexpected non-benign labels outside the frozen taxonomy."
    )

if unknown_nonbenign != 0:
    raise RuntimeError(
        "Unknown non-benign canonical labels were observed."
    )

nonbenign_total = (
    primary_total
    + family_counts[
        "TARGET_ONLY_UNSEEN"
    ]
    + family_counts[
        "OTHER_ATTACK_UNSEEN_LABEL"
    ]
)

if nonbenign_total != EXPECTED_ALL_NONBENIGN:
    raise RuntimeError(
        "Global non-benign count mismatch."
    )

if (
    family_counts["BENIGN"]
    + nonbenign_total
) != EXPECTED_EFFECTIVE_ROWS:
    raise RuntimeError(
        "Global population arithmetic mismatch."
    )

print()
print("Primary seven-family attack:", f"{primary_total:,}")
print("All non-benign             :", f"{nonbenign_total:,}")
print("[PASS] Global taxonomy census exact.")


# ======================================================================================
# 9. FAMILY × DAY + BENIGN × DAY VALIDATION AGAINST STAGE27-0A
# ======================================================================================

banner("STAGE27-1A MEMBERSHIP :: REPRODUCE STAGE27-0A SUPPORT")

for family in primary_order:

    if family not in family_support_expected:
        raise RuntimeError(
            f"Stage27-0A support CSV lacks {family}"
        )

    print()
    print(family)

    for day in DAY_ORDER:

        actual = int(
            family_day_counts[
                family
            ][
                day
            ]
        )

        expected = int(
            family_support_expected[
                family
            ][
                day
            ]
        )

        print(
            f"  {day:10s}: "
            f"{actual:>9,} "
            f"(expected {expected:,})"
        )

        if actual != expected:
            raise RuntimeError(
                f"{family}/{day}: Stage27-0A support mismatch."
            )


print()
print("Benign by day:")

for day in DAY_ORDER:

    actual = int(
        benign_day_counts[
            day
        ]
    )

    expected = int(
        benign_support_expected[
            day
        ]
    )

    print(
        f"  {day:10s}: "
        f"{actual:>9,} "
        f"(expected {expected:,})"
    )

    if actual != expected:
        raise RuntimeError(
            f"{day}: benign support mismatch."
        )


print()
print(
    "[PASS] Row-level Label materialization reproduces the "
    "sealed Stage27-0A family/day and benign/day matrices exactly."
)


# ======================================================================================
# 10. SAVE GLOBAL LABEL-DERIVED CACHE
# ======================================================================================

banner("STAGE27-1A MEMBERSHIP :: SAVE GLOBAL MEMBERSHIP CACHE")

family_codes_path = (
    CACHE_ROOT
    / "global_family_codes.npy"
)

day_codes_path = (
    CACHE_ROOT
    / "global_day_codes.npy"
)

source_codes_path = (
    CACHE_ROOT
    / "global_source_codes.npy"
)

family_codes_receipt = save_npy_with_receipt(
    family_codes_path,
    family_codes,
)

day_codes_receipt = save_npy_with_receipt(
    day_codes_path,
    day_codes,
)

source_codes_receipt = save_npy_with_receipt(
    source_codes_path,
    source_codes,
)

print(
    "family codes:",
    family_codes_receipt[
        "content_sha256"
    ],
)

print(
    "day codes   :",
    day_codes_receipt[
        "content_sha256"
    ],
)

print(
    "source codes:",
    source_codes_receipt[
        "content_sha256"
    ],
)

print(
    "[PASS] Deterministic global code arrays persisted outside Git."
)


# ======================================================================================
# 11. BUILD FIVE ELIGIBLE FOLD MEMBERSHIPS
# ======================================================================================

banner("STAGE27-1A MEMBERSHIP :: BUILD FIVE FROZEN FOLDS")

fold_receipts_runtime = OrderedDict()

primary_mask_global = np.isin(
    family_codes,
    PRIMARY_FAMILY_CODES,
)

binary_included_global = (
    (family_codes == FAMILY_CODE["BENIGN"])
    | primary_mask_global
)

folds = fold_spec[
    "folds"
]

weight_expected_map = class_weight_policy[
    "pre_materialization_expected_values"
]

for family in ELIGIBLE_FOLDS:

    if family not in folds:
        raise RuntimeError(
            f"Frozen fold spec missing eligible family {family}"
        )

    fold = folds[
        family
    ]

    if not fold[
        "scientific_model_fit_authorized"
    ]:
        raise RuntimeError(
            f"{family}: frozen fold unexpectedly not authorized."
        )

    geometry = fold[
        "replacement_geometry"
    ]

    train_days = geometry[
        "train_days"
    ]

    validation_days = geometry[
        "validation_days"
    ]

    target_days = geometry[
        "target_days"
    ]

    if len(
        validation_days
    ) != 1:
        raise RuntimeError(
            f"{family}: expected exactly one validation day."
        )

    if len(
        target_days
    ) != 1:
        raise RuntimeError(
            f"{family}: expected exactly one target day."
        )

    heldout_code = FAMILY_CODE[
        family
    ]

    train_day_codes = np.array(
        [
            DAY_CODE[
                day
            ]
            for day in train_days
        ],
        dtype=np.uint8,
    )

    validation_day_code = np.uint8(
        DAY_CODE[
            validation_days[
                0
            ]
        ]
    )

    target_day_code = np.uint8(
        DAY_CODE[
            target_days[
                0
            ]
        ]
    )

    train_mask = (
        np.isin(
            day_codes,
            train_day_codes,
        )
        & binary_included_global
        & (
            family_codes
            != heldout_code
        )
    )

    validation_mask = (
        (
            day_codes
            == validation_day_code
        )
        & binary_included_global
        & (
            family_codes
            != heldout_code
        )
    )

    primary_target_mask = (
        (
            day_codes
            == target_day_code
        )
        & (
            (
                family_codes
                == FAMILY_CODE["BENIGN"]
            )
            | (
                family_codes
                == heldout_code
            )
        )
    )

    operational_target_mask = (
        (
            day_codes
            == target_day_code
        )
        & binary_included_global
    )

    train_idx = np.flatnonzero(
        train_mask
    ).astype(
        np.int32,
        copy=False,
    )

    validation_idx = np.flatnonzero(
        validation_mask
    ).astype(
        np.int32,
        copy=False,
    )

    primary_target_idx = np.flatnonzero(
        primary_target_mask
    ).astype(
        np.int32,
        copy=False,
    )

    operational_target_idx = np.flatnonzero(
        operational_target_mask
    ).astype(
        np.int32,
        copy=False,
    )

    # ------------------------------------------------------------------
    # Core fold counts
    # ------------------------------------------------------------------

    train_codes = family_codes[
        train_idx
    ]

    validation_codes = family_codes[
        validation_idx
    ]

    primary_target_codes = family_codes[
        primary_target_idx
    ]

    operational_target_codes = family_codes[
        operational_target_idx
    ]

    train_benign = int(
        np.sum(
            train_codes
            == FAMILY_CODE[
                "BENIGN"
            ]
        )
    )

    train_attack = int(
        train_idx.size
        - train_benign
    )

    validation_benign = int(
        np.sum(
            validation_codes
            == FAMILY_CODE[
                "BENIGN"
            ]
        )
    )

    validation_attack = int(
        validation_idx.size
        - validation_benign
    )

    heldout_train_count = int(
        np.sum(
            train_codes
            == heldout_code
        )
    )

    heldout_validation_count = int(
        np.sum(
            validation_codes
            == heldout_code
        )
    )

    target_benign = int(
        np.sum(
            primary_target_codes
            == FAMILY_CODE[
                "BENIGN"
            ]
        )
    )

    target_heldout = int(
        np.sum(
            primary_target_codes
            == heldout_code
        )
    )

    target_other = int(
        primary_target_idx.size
        - target_benign
        - target_heldout
    )

    operational_benign = int(
        np.sum(
            operational_target_codes
            == FAMILY_CODE[
                "BENIGN"
            ]
        )
    )

    operational_attack = int(
        operational_target_idx.size
        - operational_benign
    )

    # ------------------------------------------------------------------
    # Hard frozen expected-count gates
    # ------------------------------------------------------------------

    expected_fields = {
        "train_benign_expected": train_benign,
        "train_known_attack_expected": train_attack,
        "validation_benign_expected": validation_benign,
        "validation_known_attack_expected": validation_attack,
        "primary_target_benign_expected": target_benign,
        "primary_target_heldout_attack_expected": target_heldout,
        "primary_target_rows_expected": int(
            primary_target_idx.size
        ),
    }

    for field, actual in expected_fields.items():

        expected = int(
            geometry[
                field
            ]
        )

        if actual != expected:
            raise RuntimeError(
                f"{family}: {field} mismatch. "
                f"expected={expected:,}, actual={actual:,}"
            )

    if heldout_train_count != 0:
        raise RuntimeError(
            f"{family}: HELD-OUT FAMILY LEAKS INTO TRAIN."
        )

    if heldout_validation_count != 0:
        raise RuntimeError(
            f"{family}: HELD-OUT FAMILY LEAKS INTO VALIDATION."
        )

    if train_benign <= 0 or train_attack <= 0:
        raise RuntimeError(
            f"{family}: training membership is single-class."
        )

    if validation_benign <= 0 or validation_attack <= 0:
        raise RuntimeError(
            f"{family}: validation membership is single-class."
        )

    if target_benign <= 0 or target_heldout <= 0:
        raise RuntimeError(
            f"{family}: primary isolation target is not two-class."
        )

    if target_other != 0:
        raise RuntimeError(
            f"{family}: primary target contains non-benign rows "
            "outside the held-out family."
        )

    # Global row IDs are sorted by source/day chronology.
    if not (
        int(train_idx.max())
        < int(validation_idx.min())
        < int(primary_target_idx.min())
    ):
        raise RuntimeError(
            f"{family}: TRAIN < VALIDATION < TARGET global-order test failed."
        )

    realized_weight = (
        train_benign
        / train_attack
    )

    expected_weight = float(
        weight_expected_map[
            family
        ]
    )

    if not np.isclose(
        realized_weight,
        expected_weight,
        rtol=0.0,
        atol=1e-15,
    ):
        raise RuntimeError(
            f"{family}: realized class weight differs from "
            "Stage27-0 pre-materialization expectation."
        )

    # ------------------------------------------------------------------
    # Save deterministic role membership arrays outside Git.
    # ------------------------------------------------------------------

    fold_cache = (
        CACHE_ROOT
        / family
    )

    fold_cache.mkdir(
        parents=True,
        exist_ok=False,
    )

    role_arrays = OrderedDict(
        [
            ("train_global_idx", train_idx),
            ("validation_global_idx", validation_idx),
            ("primary_target_global_idx", primary_target_idx),
            ("operational_target_global_idx", operational_target_idx),
        ]
    )

    role_receipts = OrderedDict()

    for role_name, arr in role_arrays.items():

        role_path = (
            fold_cache
            / f"{role_name}.npy"
        )

        role_receipts[
            role_name
        ] = save_npy_with_receipt(
            role_path,
            arr,
        )

    # ------------------------------------------------------------------
    # Per-family support within training/validation/operational context.
    # ------------------------------------------------------------------

    train_family_support = OrderedDict()
    validation_family_support = OrderedDict()
    operational_family_support = OrderedDict()

    for candidate in primary_order:

        code = FAMILY_CODE[
            candidate
        ]

        train_family_support[
            candidate
        ] = int(
            np.sum(
                train_codes
                == code
            )
        )

        validation_family_support[
            candidate
        ] = int(
            np.sum(
                validation_codes
                == code
            )
        )

        operational_family_support[
            candidate
        ] = int(
            np.sum(
                operational_target_codes
                == code
            )
        )

    receipt = OrderedDict(
        [
            ("stage", "Stage27-1A"),
            ("type", "FOLD_MEMBERSHIP_RECEIPT"),
            ("created_at_utc", now_utc()),
            ("scientific_parent_commit", EXPECTED_PARENT),
            ("held_out_family", family),
            ("fold_status", fold["status"]),
            ("support_status", fold["support_status"]),
            ("train_days", train_days),
            ("validation_days", validation_days),
            ("target_days", target_days),
            (
                "membership_semantics",
                {
                    "global_row_id": (
                        "ZERO_BASED_INDEX_IN_CONCATENATION_OF_THE_8_FROZEN_"
                        "EFFECTIVE_SOURCE_SEGMENTS_IN_STAGE24_PUBLISHED_POPULATION_ORDER"
                    ),
                    "train": (
                        "BENIGN + PRIMARY-SEVEN KNOWN ATTACK FAMILIES ON FROZEN "
                        "TRAIN DAYS; HELD-OUT FAMILY EXCLUDED; HEARTBLEED/OTHER UNKNOWN EXCLUDED"
                    ),
                    "validation": (
                        "BENIGN + PRIMARY-SEVEN KNOWN ATTACK FAMILIES ON FROZEN "
                        "VALIDATION DAY; HELD-OUT FAMILY EXCLUDED; HEARTBLEED/OTHER UNKNOWN EXCLUDED"
                    ),
                    "primary_target": (
                        "SAME-TARGET-DAY BENIGN + HELD-OUT FAMILY ONLY"
                    ),
                    "operational_target": (
                        "SAME-TARGET-DAY BENIGN + ALL PRIMARY-SEVEN ATTACK FAMILIES"
                    ),
                },
            ),
            (
                "counts",
                {
                    "train_rows": int(train_idx.size),
                    "train_benign": train_benign,
                    "train_attack": train_attack,
                    "validation_rows": int(validation_idx.size),
                    "validation_benign": validation_benign,
                    "validation_attack": validation_attack,
                    "primary_target_rows": int(primary_target_idx.size),
                    "primary_target_benign": target_benign,
                    "primary_target_heldout_attack": target_heldout,
                    "operational_target_rows": int(
                        operational_target_idx.size
                    ),
                    "operational_target_benign": operational_benign,
                    "operational_target_attack": operational_attack,
                },
            ),
            (
                "heldout_exclusion",
                {
                    "train_count": heldout_train_count,
                    "validation_count": heldout_validation_count,
                    "train_required": 0,
                    "validation_required": 0,
                    "status": "PASS",
                },
            ),
            (
                "chronology_and_disjointness",
                {
                    "train_max_global_idx": int(
                        train_idx.max()
                    ),
                    "validation_min_global_idx": int(
                        validation_idx.min()
                    ),
                    "validation_max_global_idx": int(
                        validation_idx.max()
                    ),
                    "target_min_global_idx": int(
                        primary_target_idx.min()
                    ),
                    "strict_global_order": True,
                    "train_validation_disjoint": True,
                    "train_target_disjoint": True,
                    "validation_target_disjoint": True,
                },
            ),
            (
                "class_weight",
                {
                    "formula": "train_benign / train_attack",
                    "train_benign": train_benign,
                    "train_attack": train_attack,
                    "realized_value": realized_weight,
                    "stage27_0_expected_value": expected_weight,
                    "recomputed_from_frozen_train_membership": True,
                    "target_rows_used": 0,
                    "validation_rows_used": 0,
                },
            ),
            (
                "family_support",
                {
                    "train": train_family_support,
                    "validation": validation_family_support,
                    "operational_target": operational_family_support,
                },
            ),
            (
                "runtime_membership_arrays",
                role_receipts,
            ),
            (
                "known_family_control",
                {
                    "membership": "IDENTICAL_TO_FROZEN_VALIDATION_MEMBERSHIP",
                    "threshold_selection_authorized_later": True,
                    "threshold_selection_executed_now": False,
                },
            ),
            (
                "target_opening_semantics",
                {
                    "predictor_columns_read": 0,
                    "label_column_read_for_membership_only": True,
                    "model_inference_on_target": False,
                    "opening_consumed": 0,
                },
            ),
        ]
    )

    fold_receipts_runtime[
        family
    ] = receipt

    print()
    print(
        f"{family}"
    )

    print(
        f"  TRAIN      rows={train_idx.size:,} "
        f"benign={train_benign:,} attack={train_attack:,} "
        f"heldout={heldout_train_count}"
    )

    print(
        f"  VALIDATION rows={validation_idx.size:,} "
        f"benign={validation_benign:,} attack={validation_attack:,} "
        f"heldout={heldout_validation_count}"
    )

    print(
        f"  PRIMARY     rows={primary_target_idx.size:,} "
        f"benign={target_benign:,} heldout={target_heldout:,}"
    )

    print(
        f"  OPERATIONAL rows={operational_target_idx.size:,} "
        f"benign={operational_benign:,} attack={operational_attack:,}"
    )

    print(
        f"  class weight={realized_weight:.15f}"
    )

    print(
        "  exclusion gate: PASS"
    )


print()
print(
    "[PASS] All five eligible folds materialized and "
    "held-out exclusion verified mechanically."
)


# ======================================================================================
# 12. STRUCTURALLY INELIGIBLE FOLD CHECK
# ======================================================================================

banner("STAGE27-1A MEMBERSHIP :: STRUCTURAL INELIGIBILITY PRESERVATION")

for family in STRUCTURALLY_INELIGIBLE:

    fold = folds[
        family
    ]

    print(
        f"{family:18s} "
        f"status={fold['status']} "
        f"reason={fold['reason_code']}"
    )

    if fold[
        "scientific_model_fit_authorized"
    ] is not False:
        raise RuntimeError(
            f"{family}: structurally ineligible fold became fit-authorized."
        )

    if fold[
        "replacement_geometry"
    ] is not None:
        raise RuntimeError(
            f"{family}: structurally ineligible fold unexpectedly has geometry."
        )


print(
    "[PASS] DOS and AUTH_BRUTE_FORCE remain retained but non-executable."
)


# ======================================================================================
# 13. GLOBAL RUNTIME CACHE RE-READ VERIFICATION
# ======================================================================================

banner("STAGE27-1A MEMBERSHIP :: RUNTIME CACHE RE-READ")

family_codes_reload = np.load(
    family_codes_path,
    allow_pickle=False,
    mmap_mode="r",
)

day_codes_reload = np.load(
    day_codes_path,
    allow_pickle=False,
    mmap_mode="r",
)

source_codes_reload = np.load(
    source_codes_path,
    allow_pickle=False,
    mmap_mode="r",
)

if (
    family_codes_reload.shape
    != family_codes.shape
):
    raise RuntimeError(
        "Reloaded family-code shape mismatch."
    )

if (
    content_sha256_array(
        np.asarray(
            family_codes_reload
        )
    )
    != family_codes_receipt[
        "content_sha256"
    ]
):
    raise RuntimeError(
        "Reloaded family-code content hash mismatch."
    )

if (
    content_sha256_array(
        np.asarray(
            day_codes_reload
        )
    )
    != day_codes_receipt[
        "content_sha256"
    ]
):
    raise RuntimeError(
        "Reloaded day-code content hash mismatch."
    )

if (
    content_sha256_array(
        np.asarray(
            source_codes_reload
        )
    )
    != source_codes_receipt[
        "content_sha256"
    ]
):
    raise RuntimeError(
        "Reloaded source-code content hash mismatch."
    )

for family, receipt in fold_receipts_runtime.items():

    for role_name, role_receipt in receipt[
        "runtime_membership_arrays"
    ].items():

        arr = np.load(
            role_receipt[
                "path"
            ],
            allow_pickle=False,
            mmap_mode="r",
        )

        if int(
            arr.size
        ) != int(
            role_receipt[
                "count"
            ]
        ):
            raise RuntimeError(
                f"{family}/{role_name}: reload count mismatch."
            )

        if (
            content_sha256_array(
                np.asarray(
                    arr
                )
            )
            != role_receipt[
                "content_sha256"
            ]
        ):
            raise RuntimeError(
                f"{family}/{role_name}: reload content hash mismatch."
            )

print(
    "[PASS] All global and fold membership arrays re-read byte/content exact."
)


# ======================================================================================
# 14. ONLY NOW CREATE DURABLE REPOSITORY RECEIPTS
# ======================================================================================

banner("STAGE27-1A MEMBERSHIP :: WRITE DURABLE RECEIPTS")

OUT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

source_effective_population_receipt = OrderedDict(
    [
        ("stage", "Stage27-1A"),
        ("type", "SOURCE_EFFECTIVE_POPULATION_RECEIPT"),
        ("created_at_utc", now_utc()),
        ("scientific_parent_commit", EXPECTED_PARENT),
        ("source_root_runtime", str(SOURCE_ROOT)),
        ("source_count", 8),
        ("segments", segments),
        (
            "population",
            {
                "physical_rows": physical_total,
                "effective_rows": effective_total,
                "structural_null_excluded_rows": excluded_total,
                "frozen_effective_population_identity": (
                    "STAGE24_FULL_EFFECTIVE_CICIDS2017_POPULATION"
                ),
            },
        ),
        (
            "predictor_columns_read",
            0,
        ),
        (
            "label_column_read",
            True,
        ),
        (
            "model_fits",
            0,
        ),
        (
            "target_openings",
            0,
        ),
    ]
)

write_json(
    OUT_DIR
    / "source_effective_population_receipt.json",
    source_effective_population_receipt,
)


canonical_label_census = OrderedDict(
    [
        ("stage", "Stage27-1A"),
        ("type", "CANONICAL_LABEL_CENSUS"),
        ("created_at_utc", now_utc()),
        ("scientific_parent_commit", EXPECTED_PARENT),
        (
            "canonicalization",
            [
                "STRIP_OUTER_WHITESPACE",
                "NORMALIZE_UNICODE_DASHES_TO_ASCII_HYPHEN",
                "COLLAPSE_INTERNAL_WHITESPACE",
                "CASEFOLD_FOR_LOOKUP",
            ],
        ),
        (
            "raw_canonical_label_counts",
            OrderedDict(
                sorted(
                    (
                        key,
                        int(value),
                    )
                    for key, value in raw_label_counts.items()
                )
            ),
        ),
        (
            "family_counts",
            OrderedDict(
                (
                    family,
                    int(
                        family_counts.get(
                            family,
                            0,
                        )
                    ),
                )
                for family in FAMILY_CODE.keys()
            ),
        ),
        (
            "family_day_counts",
            OrderedDict(
                (
                    family,
                    OrderedDict(
                        (
                            day,
                            int(
                                family_day_counts[
                                    family
                                ][
                                    day
                                ]
                            ),
                        )
                        for day in DAY_ORDER
                    ),
                )
                for family in primary_order
            ),
        ),
        (
            "benign_day_counts",
            OrderedDict(
                (
                    day,
                    int(
                        benign_day_counts[
                            day
                        ]
                    ),
                )
                for day in DAY_ORDER
            ),
        ),
        (
            "global_arithmetic",
            {
                "benign": int(
                    family_counts[
                        "BENIGN"
                    ]
                ),
                "primary_seven_family_attack": int(
                    primary_total
                ),
                "target_only_unseen": int(
                    family_counts[
                        "TARGET_ONLY_UNSEEN"
                    ]
                ),
                "other_attack_unseen_label": int(
                    family_counts[
                        "OTHER_ATTACK_UNSEEN_LABEL"
                    ]
                ),
                "all_nonbenign": int(
                    nonbenign_total
                ),
                "effective_rows": int(
                    EXPECTED_EFFECTIVE_ROWS
                ),
            },
        ),
        (
            "stage27_0a_support_reproduction",
            "PASS_EXACT",
        ),
    ]
)

write_json(
    OUT_DIR
    / "canonical_label_census.json",
    canonical_label_census,
)


family_code_cache_receipt = OrderedDict(
    [
        ("stage", "Stage27-1A"),
        ("type", "RUNTIME_FAMILY_CODE_CACHE_RECEIPT"),
        ("created_at_utc", now_utc()),
        ("scientific_parent_commit", EXPECTED_PARENT),
        (
            "cache_root",
            str(CACHE_ROOT),
        ),
        (
            "global_row_semantics",
            (
                "ZERO_BASED_INDEX_IN_CONCATENATION_OF_8_FROZEN_EFFECTIVE_"
                "SOURCE_SEGMENTS_IN_STAGE24_PUBLISHED_POPULATION_ORDER"
            ),
        ),
        (
            "family_codebook",
            FAMILY_CODE,
        ),
        (
            "arrays",
            {
                "global_family_codes": family_codes_receipt,
                "global_day_codes": day_codes_receipt,
                "global_source_codes": source_codes_receipt,
            },
        ),
        (
            "rebuild_rule",
            (
                "If the Kaggle runtime cache is lost, rebuild from the same 8 "
                "byte-exact frozen source assets, effective-row rules and taxonomy, "
                "then require these content hashes before any model fit."
            ),
        ),
    ]
)

write_json(
    OUT_DIR
    / "family_code_cache_receipt.json",
    family_code_cache_receipt,
)


for family in ELIGIBLE_FOLDS:

    write_json(
        OUT_DIR
        / f"fold_{family}_membership_receipt.json",
        fold_receipts_runtime[
            family
        ],
    )


# ======================================================================================
# 15. FREEZE RECORD HASHES OTHER EIGHT RECEIPTS
# ======================================================================================

non_freeze_names = [
    name
    for name in REQUIRED_OUTPUT_FILES
    if name
    != "stage27_1a_membership_freeze_record.json"
]

receipt_hashes = OrderedDict()

for name in non_freeze_names:

    path = (
        OUT_DIR
        / name
    )

    if not path.is_file():
        raise RuntimeError(
            f"Durable membership receipt missing: {name}"
        )

    receipt_hashes[
        name
    ] = {
        "bytes": int(
            path.stat().st_size
        ),
        "sha256": sha256_file(
            path
        ),
    }


membership_freeze_record = OrderedDict(
    [
        ("stage", "Stage27-1A"),
        (
            "status",
            "MEMBERSHIP_FROZEN_LOCALLY_PENDING_COMMIT_PUSH_REMOTE_VERIFICATION",
        ),
        ("created_at_utc", now_utc()),
        ("scientific_parent_commit", EXPECTED_PARENT),
        ("required_artifact_count", 9),
        (
            "artifact_hashes_before_freeze_record",
            receipt_hashes,
        ),
        (
            "source_population",
            {
                "physical_rows": EXPECTED_PHYSICAL_ROWS,
                "effective_rows": EXPECTED_EFFECTIVE_ROWS,
                "structural_null_excluded": EXPECTED_STRUCTURAL_EXCLUDED,
                "byte_exact_source_count": 8,
            },
        ),
        (
            "eligible_folds",
            ELIGIBLE_FOLDS,
        ),
        (
            "eligible_fold_count",
            5,
        ),
        (
            "structurally_ineligible",
            STRUCTURALLY_INELIGIBLE,
        ),
        (
            "descriptive_only",
            DESCRIPTIVE_ONLY,
        ),
        (
            "heldout_exclusion_global_status",
            "PASS_ALL_5_FOLDS",
        ),
        (
            "heldout_train_counts",
            {
                family: fold_receipts_runtime[
                    family
                ][
                    "heldout_exclusion"
                ][
                    "train_count"
                ]
                for family in ELIGIBLE_FOLDS
            },
        ),
        (
            "heldout_validation_counts",
            {
                family: fold_receipts_runtime[
                    family
                ][
                    "heldout_exclusion"
                ][
                    "validation_count"
                ]
                for family in ELIGIBLE_FOLDS
            },
        ),
        (
            "scientific_actions_completed",
            {
                "source_downloads_this_stage": 0,
                "label_rows_processed": EXPECTED_EFFECTIVE_ROWS,
                "predictor_columns_read": 0,
                "feature_rows_materialized": 0,
                "model_fits": 0,
                "model_inference": 0,
                "threshold_selections": 0,
                "target_predictor_openings": 0,
                "bootstrap_replicates": 0,
                "gpu_hours": 0,
            },
        ),
        (
            "next_authorized_step_after_remote_verification",
            (
                "MATERIALIZE FROZEN 70-FEATURE TRAIN/VALIDATION MATRICES AND "
                "TRAIN THE TWO PREREGISTERED LEARNERS PER ELIGIBLE FOLD ON CPU, "
                "ONLY AFTER THE MEMBERSHIP RECEIPTS ARE COMMITTED/PUSHED/REMOTE-VERIFIED."
            ),
        ),
        (
            "fit_authorization_now",
            False,
        ),
    ]
)

write_json(
    OUT_DIR
    / "stage27_1a_membership_freeze_record.json",
    membership_freeze_record,
)


# ======================================================================================
# 16. EXACT 9-FILE LOCAL OUTPUT VALIDATION
# ======================================================================================

banner("STAGE27-1A MEMBERSHIP :: FINAL LOCAL RECEIPT VALIDATION")

actual_files = sorted(
    p.name
    for p in OUT_DIR.iterdir()
    if p.is_file()
)

expected_files = sorted(
    REQUIRED_OUTPUT_FILES
)

print("Expected durable receipts:", len(expected_files))
print("Actual durable receipts  :", len(actual_files))

if actual_files != expected_files:
    raise RuntimeError(
        "Stage27-1A durable output universe mismatch.\n"
        f"expected={expected_files}\n"
        f"actual={actual_files}"
    )

for name in REQUIRED_OUTPUT_FILES:

    path = (
        OUT_DIR
        / name
    )

    print(
        f"{name:50s} "
        f"bytes={path.stat().st_size:>10,} "
        f"sha256={sha256_file(path)}"
    )


print()
print(
    "[PASS] Exactly nine Stage27-1A membership receipts exist."
)


# ======================================================================================
# 17. FREEZE RECORD READBACK
# ======================================================================================

banner("STAGE27-1A MEMBERSHIP :: FREEZE READBACK")

read_freeze = load_json(
    OUT_DIR
    / "stage27_1a_membership_freeze_record.json"
)

if read_freeze[
    "required_artifact_count"
] != 9:
    raise RuntimeError(
        "Membership freeze artifact count mismatch."
    )

if len(
    read_freeze[
        "artifact_hashes_before_freeze_record"
    ]
) != 8:
    raise RuntimeError(
        "Membership freeze should hash the other 8 receipts."
    )

for family in ELIGIBLE_FOLDS:

    if read_freeze[
        "heldout_train_counts"
    ][family] != 0:
        raise RuntimeError(
            f"{family}: nonzero heldout train count in freeze record."
        )

    if read_freeze[
        "heldout_validation_counts"
    ][family] != 0:
        raise RuntimeError(
            f"{family}: nonzero heldout validation count in freeze record."
        )

if read_freeze[
    "scientific_actions_completed"
][
    "predictor_columns_read"
] != 0:
    raise RuntimeError(
        "Predictor read counter is nonzero."
    )

if read_freeze[
    "scientific_actions_completed"
][
    "target_predictor_openings"
] != 0:
    raise RuntimeError(
        "Target predictor opening counter is nonzero."
    )

if read_freeze[
    "scientific_actions_completed"
][
    "model_fits"
] != 0:
    raise RuntimeError(
        "Model-fit counter is nonzero."
    )


print("Required receipts          :", read_freeze["required_artifact_count"])
print("Non-self receipt hashes    :", len(read_freeze["artifact_hashes_before_freeze_record"]))
print("Heldout exclusion status   :", read_freeze["heldout_exclusion_global_status"])
print("Predictor columns read     :", read_freeze["scientific_actions_completed"]["predictor_columns_read"])
print("Model fits                 :", read_freeze["scientific_actions_completed"]["model_fits"])
print("Target predictor openings  :", read_freeze["scientific_actions_completed"]["target_predictor_openings"])
print("[PASS] Membership freeze record is internally consistent.")


# ======================================================================================
# 18. STAGE EXACT NINE RECEIPTS — NO COMMIT
# ======================================================================================

banner("STAGE27-1A MEMBERSHIP :: STAGE EXACT RECEIPT SET")

for name in REQUIRED_OUTPUT_FILES:

    run(
        [
            "git",
            "add",
            "--",
            str(
                OUT_REL
                / name
            ),
        ]
    )


staged = sorted(
    line
    for line in run(
        [
            "git",
            "diff",
            "--cached",
            "--name-only",
        ]
    ).splitlines()
    if line.strip()
)

expected_staged = sorted(
    str(
        OUT_REL
        / name
    )
    for name in REQUIRED_OUTPUT_FILES
)

print("Expected staged:", len(expected_staged))
print("Actual staged  :", len(staged))

if staged != expected_staged:
    raise RuntimeError(
        "Unexpected staged membership file universe.\n"
        f"expected={expected_staged}\n"
        f"actual={staged}"
    )


porcelain = sorted(
    line
    for line in run(
        [
            "git",
            "status",
            "--porcelain=v1",
            "--untracked-files=all",
        ]
    ).splitlines()
    if line.strip()
)

expected_porcelain = sorted(
    f"A  {path}"
    for path in expected_staged
)

if porcelain != expected_porcelain:
    raise RuntimeError(
        "Repository contains changes outside the exact "
        "Stage27-1A receipt set.\n\n"
        + "\n".join(
            porcelain
        )
    )


print()
print(
    "[PASS] Exactly nine Stage27-1A membership receipts staged."
)


# ======================================================================================
# 19. FINAL SCIENTIFIC BOUNDARY
# ======================================================================================

banner("STAGE27-1A MEMBERSHIP MATERIALIZATION COMPLETE — LOCAL ONLY")

print("Scientific parent       :", EXPECTED_PARENT)
print("Effective population    :", f"{EXPECTED_EFFECTIVE_ROWS:,}")
print("Primary attack rows     :", f"{primary_total:,}")
print("Benign rows             :", f"{family_counts['BENIGN']:,}")
print("Heartbleed excluded     :", f"{family_counts['TARGET_ONLY_UNSEEN']:,}")
print("Unknown nonbenign       :", f"{family_counts['OTHER_ATTACK_UNSEEN_LABEL']:,}")
print()
print("Eligible fold memberships: 5 / 5")
print("Heldout TRAIN counts     :", {
    family: fold_receipts_runtime[family]["heldout_exclusion"]["train_count"]
    for family in ELIGIBLE_FOLDS
})
print("Heldout VALID counts     :", {
    family: fold_receipts_runtime[family]["heldout_exclusion"]["validation_count"]
    for family in ELIGIBLE_FOLDS
})
print()
print("Predictor columns read   : 0 / 70")
print("Feature matrices built   : 0")
print("Model fits               : 0")
print("Inference                : 0")
print("Threshold selections     : 0")
print("Target predictor openings: 0")
print("Bootstrap replicates     : 0")
print("GPU hours                : 0")
print()
print("Durable receipts         : 9 / 9 STAGED")
print("Commit performed         : NO")
print("Push performed           : NO")
print()
print("NO MODEL FIT IS AUTHORIZED YET.")
print()
print(
    "NEXT GATE: inspect this output, then commit/push/remote-verify "
    "the exact nine Stage27-1A membership receipts."
)



STAGE27-1A MEMBERSHIP :: SCIENTIFIC-PARENT GATE
Expected parent : 1ea6bedb141bc3fa6115edbc5296d0f7c6252559
Local HEAD      : 1ea6bedb141bc3fa6115edbc5296d0f7c6252559
origin/main     : 1ea6bedb141bc3fa6115edbc5296d0f7c6252559
Git clean       : True
[PASS] Frozen Stage27-0 is the clean scientific parent.

STAGE27-1A MEMBERSHIP :: OUTPUT GATES
Source cache      : /kaggle/working/stage27_cicids2017_sources
Membership cache  : /kaggle/working/stage27_1a_membership_cache
Durable output    : /kaggle/working/ids2018-validation-safe-ablation/results/stage27_loao_unseen_attack/stage27_1a_fold_membership
Repo writes yet   : 0

STAGE27-1A MEMBERSHIP :: LOAD FROZEN PROTOCOL
Eligible folds       : 5
Authorized fits      : 10
Feature count        : 70
Fits completed       : 0
Target openings      : 0
[PASS] Frozen Stage27-0 protocol loaded.

STAGE27-1A MEMBERSHIP :: TAXONOMY GATE
Canonical taxonomy entries : 14
Primary families           : ['BOT', 'DDOS', 'DOS', 'AUTH_BRUTE_FORCE', 'INFILTRATION', '

RuntimeError: Primary seven-family attack total mismatch: 555,455

In [18]:
# ======================================================================================
# STAGE27-1A — FIVE-FOLD MEMBERSHIP MATERIALIZATION + EXCLUSION RECEIPTS (FIXED V2)
#
# Scientific parent:
#   1ea6bedb141bc3fa6115edbc5296d0f7c6252559
#
# PRECONDITION
# ------------
# Eight byte-exact CICIDS2017 parquet sources already exist under:
#   /kaggle/working/stage27_cicids2017_sources
#
# THIS STAGE READS ONLY:
#   - Parquet metadata
#   - Label column for frozen effective rows
#   - committed Stage27-0 / Stage27-0A protocol artifacts
#
# IT DOES NOT READ:
#   - any of the 70 model predictors
#
# IT PERFORMS:
#   - ZERO model fits
#   - ZERO inference
#   - ZERO threshold selection
#   - ZERO target predictor openings
#   - ZERO bootstrap
#   - ZERO GPU
#
# OUTPUT
# ------
# Runtime cache OUTSIDE Git:
#   /kaggle/working/stage27_1a_membership_cache
#
# Small durable receipts INSIDE Git:
#   results/stage27_loao_unseen_attack/stage27_1a_fold_membership/
#
# Exactly nine receipt JSON files are created and staged, but NOT committed/pushed.
# ======================================================================================

from __future__ import annotations

import csv
import hashlib
import json
import os
import re
import shutil
import subprocess
import unicodedata
from collections import Counter, OrderedDict, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Tuple

import numpy as np
import pyarrow.parquet as pq


# ======================================================================================
# 0. CONSTANTS
# ======================================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_PARENT = (
    "1ea6bedb141bc3fa6115edbc5296d0f7c6252559"
)

SOURCE_ROOT = Path(
    "/kaggle/working/stage27_cicids2017_sources"
).resolve()

CACHE_ROOT = Path(
    "/kaggle/working/stage27_1a_membership_cache"
).resolve()

STAGE27_ROOT_REL = Path(
    "results/stage27_loao_unseen_attack"
)

STAGE27_0_REL = (
    STAGE27_ROOT_REL
    / "stage27_0_protocol_lock"
)

STAGE27_0A_REL = (
    STAGE27_ROOT_REL
    / "stage27_0a_family_day_feasibility"
)

OUT_REL = (
    STAGE27_ROOT_REL
    / "stage27_1a_fold_membership"
)

OUT_DIR = (
    REPO
    / OUT_REL
)

FOLD_SPEC_PATH = (
    REPO
    / STAGE27_0_REL
    / "fold_spec.json"
)

TAXONOMY_PATH = (
    REPO
    / STAGE27_0_REL
    / "taxonomy.json"
)

TARGET_SPEC_PATH = (
    REPO
    / STAGE27_0_REL
    / "target_population_spec.json"
)

CLASS_WEIGHT_PATH = (
    REPO
    / STAGE27_0_REL
    / "class_weight_policy.json"
)

FEATURE_SPEC_PATH = (
    REPO
    / STAGE27_0_REL
    / "feature_representation.json"
)

FREEZE_PATH = (
    REPO
    / STAGE27_0_REL
    / "freeze_record.json"
)

STAGE24_SOURCE_CONTRACT = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0b2_complete_cicids2017_source_contract.json"
)

STAGE24_PROTOCOL = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0c_final_preopening_protocol_lock.json"
)

STAGE24_PUBLISHED_RESULT = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_2_primary_target_openings"
    / "stage24_2a_bridge62_published"
    / "stage24_2a_bridge62_published_result.json"
)

STAGE27_0A_TEMPORAL = (
    REPO
    / STAGE27_0A_REL
    / "temporal_feasibility.json"
)

STAGE27_0A_FAMILY_SUPPORT = (
    REPO
    / STAGE27_0A_REL
    / "family_day_support.csv"
)

STAGE27_0A_BENIGN_SUPPORT = (
    REPO
    / STAGE27_0A_REL
    / "benign_day_support.csv"
)

EXPECTED_EFFECTIVE_ROWS = 2_830_743
EXPECTED_BENIGN = 2_273_097
EXPECTED_PRIMARY_ATTACK = 557_635
EXPECTED_HEARTBLEED = 11
EXPECTED_ALL_NONBENIGN = 557_646
EXPECTED_PHYSICAL_ROWS = 3_119_345
EXPECTED_STRUCTURAL_EXCLUDED = 288_602

ELIGIBLE_FOLDS = [
    "BOT",
    "DDOS",
    "INFILTRATION",
    "PORT_SCAN",
    "WEB_ATTACK",
]

STRUCTURALLY_INELIGIBLE = [
    "DOS",
    "AUTH_BRUTE_FORCE",
]

DESCRIPTIVE_ONLY = [
    "INFILTRATION",
]

DAY_ORDER = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
]

DAY_CODE = {
    day: i + 1
    for i, day in enumerate(DAY_ORDER)
}

# Family-code representation used ONLY for deterministic membership construction.
# The model target remains binary attack-vs-benign.
FAMILY_CODE = OrderedDict(
    [
        ("BENIGN", 0),
        ("BOT", 1),
        ("DDOS", 2),
        ("DOS", 3),
        ("AUTH_BRUTE_FORCE", 4),
        ("INFILTRATION", 5),
        ("PORT_SCAN", 6),
        ("WEB_ATTACK", 7),
        ("TARGET_ONLY_UNSEEN", 8),
        ("OTHER_ATTACK_UNSEEN_LABEL", 9),
    ]
)

CODE_FAMILY = {
    value: key
    for key, value in FAMILY_CODE.items()
}

PRIMARY_FAMILY_CODES = np.array(
    [
        FAMILY_CODE["BOT"],
        FAMILY_CODE["DDOS"],
        FAMILY_CODE["DOS"],
        FAMILY_CODE["AUTH_BRUTE_FORCE"],
        FAMILY_CODE["INFILTRATION"],
        FAMILY_CODE["PORT_SCAN"],
        FAMILY_CODE["WEB_ATTACK"],
    ],
    dtype=np.uint8,
)

REQUIRED_OUTPUT_FILES = [
    "source_effective_population_receipt.json",
    "canonical_label_census.json",
    "family_code_cache_receipt.json",
    "fold_BOT_membership_receipt.json",
    "fold_DDOS_membership_receipt.json",
    "fold_INFILTRATION_membership_receipt.json",
    "fold_PORT_SCAN_membership_receipt.json",
    "fold_WEB_ATTACK_membership_receipt.json",
    "stage27_1a_membership_freeze_record.json",
]

os.environ["CUDA_VISIBLE_DEVICES"] = ""


# ======================================================================================
# 1. HELPERS
# ======================================================================================

def banner(title: str) -> None:
    print()
    print("=" * 118)
    print(title)
    print("=" * 118)


def run(
    cmd: List[str],
    *,
    cwd: Path = REPO,
    check: bool = True,
) -> str:
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}): {' '.join(map(str, cmd))}\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return p.stdout.strip()


def now_utc() -> str:
    return datetime.now(timezone.utc).isoformat()


def load_json(path: Path) -> Any:
    with path.open(
        "r",
        encoding="utf-8",
    ) as f:
        return json.load(f)


def write_json(
    path: Path,
    obj: Any,
) -> None:
    path.write_text(
        json.dumps(
            obj,
            ensure_ascii=False,
            indent=2,
            sort_keys=False,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )


def sha256_file(
    path: Path,
    chunk_size: int = 16 * 1024 * 1024,
) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def content_sha256_array(arr: np.ndarray) -> str:
    contiguous = np.ascontiguousarray(arr)
    return sha256_bytes(
        contiguous.tobytes(order="C")
    )


def file_receipt(path: Path) -> Dict[str, Any]:
    return {
        "path": str(path),
        "bytes": int(path.stat().st_size),
        "sha256": sha256_file(path),
    }


def save_npy_with_receipt(
    path: Path,
    arr: np.ndarray,
) -> Dict[str, Any]:

    np.save(
        path,
        arr,
        allow_pickle=False,
    )

    return {
        "path": str(path),
        "dtype": str(arr.dtype),
        "shape": list(arr.shape),
        "count": int(arr.size),
        "content_sha256": content_sha256_array(arr),
        "npy_bytes": int(path.stat().st_size),
        "npy_sha256": sha256_file(path),
    }


def normalize_unicode_dashes(text: str) -> str:
    dash_chars = (
        "\u2010"  # hyphen
        "\u2011"  # non-breaking hyphen
        "\u2012"  # figure dash
        "\u2013"  # en dash
        "\u2014"  # em dash
        "\u2015"  # horizontal bar
        "\u2212"  # minus sign
        "\ufe58"  # small em dash
        "\ufe63"  # small hyphen-minus
        "\uff0d"  # fullwidth hyphen-minus
    )

    table = str.maketrans(
        {
            ch: "-"
            for ch in dash_chars
        }
    )

    return text.translate(table)


WEB_ATTACK_CP1252_C1_ALIASES = OrderedDict(
    [
        (
            "web attack \u0096 brute force",
            "web attack - brute force",
        ),
        (
            "web attack \u0096 sql injection",
            "web attack - sql injection",
        ),
        (
            "web attack \u0096 xss",
            "web attack - xss",
        ),
    ]
)


def canonicalize_label_precompat(value: Any) -> str:
    """
    Frozen Stage24 canonicalization before the source-encoding compatibility
    alias is applied.

    CICIDS2017 traffic_labels stores the Web Attack separator as U+0096
    (the C1 control code produced when Windows-1252 byte 0x96, an en dash,
    is preserved under a Latin-1-style decode). Stage24's durable published
    census records those exact canonical strings as:
        web attack \\u0096 brute force
        web attack \\u0096 sql injection
        web attack \\u0096 xss

    We retain that pre-compat form for audit, then map ONLY those three exact
    source strings to the already-frozen ASCII-hyphen taxonomy keys.
    """
    if value is None:
        raise RuntimeError(
            "Null Label encountered inside the frozen effective population."
        )

    text = str(value)

    text = normalize_unicode_dashes(text)

    text = text.strip()

    text = re.sub(
        r"\\s+",
        " ",
        text,
    )

    if not text:
        raise RuntimeError(
            "Empty Label encountered inside the frozen effective population."
        )

    return text.casefold()


def canonicalize_label(value: Any) -> str:
    text = canonicalize_label_precompat(value)

    if "\u0096" in text:
        if text not in WEB_ATTACK_CP1252_C1_ALIASES:
            raise RuntimeError(
                "Unexpected U+0096 source-label variant encountered. "
                "Only the three frozen CICIDS2017 Web Attack aliases are "
                f"authorized; observed={text!r}"
            )

        text = WEB_ATTACK_CP1252_C1_ALIASES[text]

    return text


def load_support_csv(
    path: Path,
) -> List[Dict[str, str]]:

    with path.open(
        "r",
        encoding="utf-8",
        newline="",
    ) as f:
        return list(
            csv.DictReader(f)
        )


def parse_family_support(
    rows: List[Dict[str, str]],
) -> Dict[str, Dict[str, int]]:

    out = {}

    for row in rows:

        lower_keys = {
            k.casefold(): k
            for k in row.keys()
        }

        family_key = (
            lower_keys.get("family")
            or lower_keys.get("attack_family")
        )

        if family_key is None:
            raise RuntimeError(
                "Could not find family column in family_day_support.csv"
            )

        family = row[
            family_key
        ].strip()

        per_day = {}

        for day in DAY_ORDER:

            source_key = None

            for k in row.keys():
                if k.casefold() == day.casefold():
                    source_key = k
                    break

            if source_key is None:
                raise RuntimeError(
                    f"Missing {day} column in family_day_support.csv"
                )

            per_day[
                day
            ] = int(
                row[
                    source_key
                ]
            )

        out[
            family
        ] = per_day

    return out


def parse_benign_support(
    rows: List[Dict[str, str]],
) -> Dict[str, int]:

    # Supports either:
    #   day,benign_support
    # or a single row with Monday..Friday columns.
    if not rows:
        raise RuntimeError(
            "benign_day_support.csv is empty."
        )

    result = {}

    keys = list(
        rows[0].keys()
    )

    lower = {
        key.casefold(): key
        for key in keys
    }

    if "day" in lower:

        day_key = lower["day"]

        value_key = (
            lower.get("benign_support")
            or lower.get("benign")
            or lower.get("count")
        )

        if value_key is None:
            raise RuntimeError(
                "Could not identify benign count column."
            )

        for row in rows:
            day = row[day_key].strip()
            result[day] = int(
                row[value_key]
            )

    else:

        row = rows[0]

        for day in DAY_ORDER:

            source_key = None

            for key in keys:
                if key.casefold() == day.casefold():
                    source_key = key
                    break

            if source_key is None:
                raise RuntimeError(
                    f"Could not find {day} in benign support CSV."
                )

            result[day] = int(
                row[source_key]
            )

    return result


def safe_clean_directory(
    path: Path,
) -> None:

    if path.exists():
        # Runtime cache is explicitly non-durable and may be rebuilt,
        # but only when it is exactly the Stage27-1A cache path.
        expected = Path(
            "/kaggle/working/stage27_1a_membership_cache"
        ).resolve()

        if path.resolve() != expected:
            raise RuntimeError(
                f"Refusing to clean unexpected path: {path}"
            )

        shutil.rmtree(path)

    path.mkdir(
        parents=True,
        exist_ok=False,
    )


# ======================================================================================
# 2. SCIENTIFIC PARENT / CLEAN GATE
# ======================================================================================

banner("STAGE27-1A MEMBERSHIP :: SCIENTIFIC-PARENT GATE")

head = run(
    ["git", "rev-parse", "HEAD"]
)

origin = run(
    ["git", "rev-parse", "origin/main"]
)

status = run(
    [
        "git",
        "status",
        "--porcelain=v1",
        "--untracked-files=all",
    ]
)

print("Expected parent :", EXPECTED_PARENT)
print("Local HEAD      :", head)
print("origin/main     :", origin)
print("Git clean       :", not bool(status.strip()))

if head != EXPECTED_PARENT:
    raise RuntimeError(
        "Local HEAD is not the frozen Stage27-0 commit."
    )

if origin != EXPECTED_PARENT:
    raise RuntimeError(
        "origin/main is not the frozen Stage27-0 commit."
    )

if status.strip():
    raise RuntimeError(
        "Repository is dirty before Stage27-1A membership:\n"
        + status
    )

print("[PASS] Frozen Stage27-0 is the clean scientific parent.")


# ======================================================================================
# 3. OUTPUT / CACHE GATES
# ======================================================================================

banner("STAGE27-1A MEMBERSHIP :: OUTPUT GATES")

if OUT_DIR.exists():
    raise RuntimeError(
        f"Stage27-1A durable receipt directory already exists:\n{OUT_DIR}\n"
        "Refusing overwrite."
    )

if not SOURCE_ROOT.is_dir():
    raise RuntimeError(
        f"Canonical source cache is missing:\n{SOURCE_ROOT}"
    )

safe_clean_directory(
    CACHE_ROOT
)

print("Source cache      :", SOURCE_ROOT)
print("Membership cache  :", CACHE_ROOT)
print("Durable output    :", OUT_DIR)
print("Repo writes yet   : 0")


# ======================================================================================
# 4. LOAD FROZEN STAGE27-0 ARTIFACTS
# ======================================================================================

banner("STAGE27-1A MEMBERSHIP :: LOAD FROZEN PROTOCOL")

required_protocol_files = [
    FOLD_SPEC_PATH,
    TAXONOMY_PATH,
    TARGET_SPEC_PATH,
    CLASS_WEIGHT_PATH,
    FEATURE_SPEC_PATH,
    FREEZE_PATH,
    STAGE24_SOURCE_CONTRACT,
    STAGE24_PROTOCOL,
    STAGE24_PUBLISHED_RESULT,
    STAGE27_0A_TEMPORAL,
    STAGE27_0A_FAMILY_SUPPORT,
    STAGE27_0A_BENIGN_SUPPORT,
]

for path in required_protocol_files:
    if not path.is_file():
        raise RuntimeError(
            f"Required frozen artifact missing: {path}"
        )

fold_spec = load_json(
    FOLD_SPEC_PATH
)

taxonomy = load_json(
    TAXONOMY_PATH
)

target_spec = load_json(
    TARGET_SPEC_PATH
)

class_weight_policy = load_json(
    CLASS_WEIGHT_PATH
)

feature_spec = load_json(
    FEATURE_SPEC_PATH
)

freeze = load_json(
    FREEZE_PATH
)

source_contract = load_json(
    STAGE24_SOURCE_CONTRACT
)

stage24_protocol = load_json(
    STAGE24_PROTOCOL
)

stage24_published = load_json(
    STAGE24_PUBLISHED_RESULT
)

temporal_0a = load_json(
    STAGE27_0A_TEMPORAL
)

family_support_expected = parse_family_support(
    load_support_csv(
        STAGE27_0A_FAMILY_SUPPORT
    )
)

benign_support_expected = parse_benign_support(
    load_support_csv(
        STAGE27_0A_BENIGN_SUPPORT
    )
)

if freeze["authorized_fit_budget"] != 10:
    raise RuntimeError(
        "Stage27 fit budget changed."
    )

if freeze["model_fits_completed"] != 0:
    raise RuntimeError(
        "Stage27-0 freeze says fits were already completed."
    )

if freeze["model_inference_completed"] != 0:
    raise RuntimeError(
        "Stage27-0 freeze says inference was already completed."
    )

if freeze["target_openings_consumed"] != 0:
    raise RuntimeError(
        "Stage27 target-opening ledger is no longer pristine."
    )

if feature_spec["feature_count"] != 70:
    raise RuntimeError(
        "Frozen feature count is no longer 70."
    )

print("Eligible folds       :", fold_spec["eligible_fold_count"])
print("Authorized fits      :", freeze["authorized_fit_budget"])
print("Feature count        :", feature_spec["feature_count"])
print("Fits completed       :", freeze["model_fits_completed"])
print("Target openings      :", freeze["target_openings_consumed"])
print("[PASS] Frozen Stage27-0 protocol loaded.")


# ======================================================================================
# 5. VERIFY TAXONOMY + BUILD CANONICAL LOOKUP
# ======================================================================================

banner("STAGE27-1A MEMBERSHIP :: TAXONOMY GATE")

primary_order = taxonomy[
    "primary_family_order"
]

if primary_order != [
    "BOT",
    "DDOS",
    "DOS",
    "AUTH_BRUTE_FORCE",
    "INFILTRATION",
    "PORT_SCAN",
    "WEB_ATTACK",
]:
    raise RuntimeError(
        "Frozen primary family order changed."
    )

raw_mapping = taxonomy[
    "raw_label_to_family"
]

canonical_mapping = {}

for raw_label, family in raw_mapping.items():

    canonical_key = canonicalize_label(
        raw_label
    )

    if (
        canonical_key in canonical_mapping
        and canonical_mapping[canonical_key] != family
    ):
        raise RuntimeError(
            f"Canonical taxonomy collision: {canonical_key}"
        )

    canonical_mapping[
        canonical_key
    ] = family


for family in primary_order:

    if family not in FAMILY_CODE:
        raise RuntimeError(
            f"No family code for {family}"
        )


print("Canonical taxonomy entries :", len(canonical_mapping))
print("Primary families           :", primary_order)
print("Heartbleed policy          :", taxonomy["heartbleed_policy"])
print("Unknown nonbenign policy   :", taxonomy["unknown_nonbenign_family_policy"])
print("[PASS] Frozen taxonomy is executable.")


# ======================================================================================
# 6. RECOVER 8 SOURCE SEGMENTS + EFFECTIVE RULES
# ======================================================================================

banner("STAGE27-1A MEMBERSHIP :: SOURCE SEGMENT CONTRACT")

acquisition_records = source_contract[
    "acquisition_records"
]

published_population = stage24_protocol[
    "published_population"
]

if len(acquisition_records) != 8:
    raise RuntimeError(
        "Expected 8 source acquisition records."
    )

if len(published_population) != 8:
    raise RuntimeError(
        "Expected 8 effective-population records."
    )

acq_by_remote = {
    rec["remote"]: rec
    for rec in acquisition_records
}

segments = []

global_cursor = 0
physical_total = 0
effective_total = 0
excluded_total = 0

for source_index, pop in enumerate(
    published_population
):

    remote = pop["remote"]

    if remote not in acq_by_remote:
        raise RuntimeError(
            f"Population source absent from acquisition contract: {remote}"
        )

    acq = acq_by_remote[
        remote
    ]

    path = (
        SOURCE_ROOT
        / Path(remote).name
    )

    if not path.is_file():
        raise RuntimeError(
            f"Canonical source missing: {path}"
        )

    actual_size = path.stat().st_size

    if actual_size != int(
        acq["size_bytes"]
    ):
        raise RuntimeError(
            f"{remote}: source size mismatch."
        )

    actual_sha = sha256_file(
        path
    )

    if actual_sha != acq[
        "sha256"
    ]:
        raise RuntimeError(
            f"{remote}: source SHA256 mismatch."
        )

    parquet = pq.ParquetFile(
        path
    )

    actual_physical = int(
        parquet.metadata.num_rows
    )

    expected_physical = int(
        pop["physical_rows"]
    )

    if actual_physical != expected_physical:
        raise RuntimeError(
            f"{remote}: physical row mismatch."
        )

    effective_rows = int(
        pop["effective_rows"]
    )

    if not (
        0
        <= effective_rows
        <= actual_physical
    ):
        raise RuntimeError(
            f"{remote}: impossible effective row count."
        )

    global_start = global_cursor

    global_stop = (
        global_start
        + effective_rows
    )

    segment = {
        "source_index": source_index,
        "day": pop["day"],
        "day_code": DAY_CODE[
            pop["day"]
        ],
        "remote": remote,
        "basename": Path(remote).name,
        "path": str(path),
        "sha256": actual_sha,
        "size_bytes": actual_size,
        "physical_rows": actual_physical,
        "effective_rows": effective_rows,
        "excluded_rows": (
            actual_physical
            - effective_rows
        ),
        "inclusion_rule": pop[
            "inclusion_rule"
        ],
        "global_start_zero_based": global_start,
        "global_stop_exclusive": global_stop,
        "physical_ordinal_first_included": (
            1
            if effective_rows > 0
            else None
        ),
        "physical_ordinal_last_included": (
            effective_rows
            if effective_rows > 0
            else None
        ),
    }

    segments.append(
        segment
    )

    global_cursor = global_stop

    physical_total += actual_physical
    effective_total += effective_rows
    excluded_total += (
        actual_physical
        - effective_rows
    )

    print(
        f"[{source_index + 1}/8] "
        f"{pop['day']:10s} "
        f"effective={effective_rows:>9,} "
        f"global=[{global_start:,},{global_stop:,}) "
        f"{Path(remote).name}"
    )


if physical_total != EXPECTED_PHYSICAL_ROWS:
    raise RuntimeError(
        f"Physical total mismatch: {physical_total:,}"
    )

if effective_total != EXPECTED_EFFECTIVE_ROWS:
    raise RuntimeError(
        f"Effective total mismatch: {effective_total:,}"
    )

if excluded_total != EXPECTED_STRUCTURAL_EXCLUDED:
    raise RuntimeError(
        f"Structural exclusion mismatch: {excluded_total:,}"
    )

print()
print("Physical rows :", f"{physical_total:,}")
print("Effective rows:", f"{effective_total:,}")
print("Excluded rows :", f"{excluded_total:,}")
print("[PASS] Eight exact source segments frozen.")


# ======================================================================================
# 7. MATERIALIZE ONLY LABEL-DERIVED CODE ARRAYS
# ======================================================================================

banner("STAGE27-1A MEMBERSHIP :: LABEL-ONLY EFFECTIVE POPULATION MATERIALIZATION")

family_codes = np.empty(
    EXPECTED_EFFECTIVE_ROWS,
    dtype=np.uint8,
)

day_codes = np.empty(
    EXPECTED_EFFECTIVE_ROWS,
    dtype=np.uint8,
)

source_codes = np.empty(
    EXPECTED_EFFECTIVE_ROWS,
    dtype=np.uint8,
)

precompat_label_counts = Counter()
raw_label_counts = Counter()
cp1252_c1_alias_counts = Counter()
family_counts = Counter()
family_day_counts = defaultdict(Counter)
benign_day_counts = Counter()

# Additional audit counters.
null_labels = 0
empty_labels = 0
unknown_nonbenign = 0

for segment in segments:

    path = Path(
        segment["path"]
    )

    start = segment[
        "global_start_zero_based"
    ]

    stop = segment[
        "global_stop_exclusive"
    ]

    n_effective = segment[
        "effective_rows"
    ]

    day = segment[
        "day"
    ]

    local_cursor = 0

    parquet = pq.ParquetFile(
        path
    )

    print()
    print(
        f"Reading Label only: "
        f"{day} :: {segment['basename']}"
    )

    for batch in parquet.iter_batches(
        batch_size=131_072,
        columns=["Label"],
        use_threads=True,
    ):

        if local_cursor >= n_effective:
            break

        values = batch.column(
            0
        ).to_pylist()

        remaining = (
            n_effective
            - local_cursor
        )

        if len(values) > remaining:
            values = values[
                :remaining
            ]

        batch_codes = np.empty(
            len(values),
            dtype=np.uint8,
        )

        for i, value in enumerate(
            values
        ):

            if value is None:
                null_labels += 1
                raise RuntimeError(
                    f"Null effective Label in {segment['basename']} "
                    f"at physical ordinal {local_cursor + i + 1:,}"
                )

            precompat = canonicalize_label_precompat(
                value
            )

            precompat_label_counts[
                precompat
            ] += 1

            canonical = canonicalize_label(
                value
            )

            if precompat != canonical:
                cp1252_c1_alias_counts[
                    precompat
                ] += 1

            if not canonical:
                empty_labels += 1
                raise RuntimeError(
                    "Empty canonical label encountered."
                )

            raw_label_counts[
                canonical
            ] += 1

            if canonical == "benign":

                family = "BENIGN"

            elif canonical in canonical_mapping:

                family = canonical_mapping[
                    canonical
                ]

            else:

                family = taxonomy[
                    "unknown_nonbenign_family_policy"
                ]

                unknown_nonbenign += 1

            if family not in FAMILY_CODE:
                raise RuntimeError(
                    f"Unrecognized frozen family identity: {family}"
                )

            code = FAMILY_CODE[
                family
            ]

            batch_codes[
                i
            ] = code

            family_counts[
                family
            ] += 1

            if family == "BENIGN":

                benign_day_counts[
                    day
                ] += 1

            elif family in primary_order:

                family_day_counts[
                    family
                ][
                    day
                ] += 1

            elif family == "TARGET_ONLY_UNSEEN":

                # Heartbleed remains separate from the primary 7-family matrix.
                pass

            elif family == "OTHER_ATTACK_UNSEEN_LABEL":

                pass

            else:

                raise RuntimeError(
                    f"Unexpected family branch: {family}"
                )

        global_batch_start = (
            start
            + local_cursor
        )

        global_batch_stop = (
            global_batch_start
            + len(batch_codes)
        )

        family_codes[
            global_batch_start:
            global_batch_stop
        ] = batch_codes

        day_codes[
            global_batch_start:
            global_batch_stop
        ] = np.uint8(
            segment[
                "day_code"
            ]
        )

        source_codes[
            global_batch_start:
            global_batch_stop
        ] = np.uint8(
            segment[
                "source_index"
            ]
        )

        local_cursor += len(
            batch_codes
        )

    if local_cursor != n_effective:
        raise RuntimeError(
            f"{segment['basename']}: expected to materialize "
            f"{n_effective:,} effective labels, got {local_cursor:,}."
        )

    print(
        f"  effective labels materialized: {local_cursor:,}"
    )


print()
print("Total label rows:", f"{family_codes.size:,}")

if family_codes.size != EXPECTED_EFFECTIVE_ROWS:
    raise RuntimeError(
        "Global family-code array length mismatch."
    )

print("[PASS] Effective population materialized from Label only.")


# ======================================================================================
# 8. SOURCE-ENCODING COMPATIBILITY AUDIT
# ======================================================================================

banner("STAGE27-1A MEMBERSHIP :: WEB ATTACK SOURCE-ENCODING AUDIT")

published_label_counts = (
    stage24_published[
        "target_population"
    ][
        "canonical_label_counts"
    ]
)

expected_web_precompat = OrderedDict(
    [
        (
            "web attack \u0096 brute force",
            1507,
        ),
        (
            "web attack \u0096 sql injection",
            21,
        ),
        (
            "web attack \u0096 xss",
            652,
        ),
    ]
)

# The values above are not invented here: the durable Stage24 published
# target census contains these exact three U+0096 canonical labels/counts.
for source_label, expected_count in expected_web_precompat.items():

    published_count = int(
        published_label_counts.get(
            source_label,
            -1,
        )
    )

    observed_count = int(
        precompat_label_counts.get(
            source_label,
            0,
        )
    )

    repaired_count = int(
        cp1252_c1_alias_counts.get(
            source_label,
            0,
        )
    )

    print(
        f"{source_label!r}: "
        f"published={published_count:,} "
        f"observed={observed_count:,} "
        f"mapped={repaired_count:,}"
    )

    if published_count != expected_count:
        raise RuntimeError(
            f"Durable Stage24 published count changed for {source_label!r}: "
            f"expected={expected_count:,}, published={published_count:,}"
        )

    if observed_count != expected_count:
        raise RuntimeError(
            f"Current byte-exact source count mismatch for {source_label!r}: "
            f"expected={expected_count:,}, observed={observed_count:,}"
        )

    if repaired_count != expected_count:
        raise RuntimeError(
            f"Compatibility mapping did not process exactly the frozen rows "
            f"for {source_label!r}: expected={expected_count:,}, "
            f"mapped={repaired_count:,}"
        )

observed_u0096_labels = sorted(
    label
    for label in precompat_label_counts
    if "\u0096" in label
)

expected_u0096_labels = sorted(
    expected_web_precompat.keys()
)

if observed_u0096_labels != expected_u0096_labels:
    raise RuntimeError(
        "Unexpected U+0096 label universe.\n"
        f"expected={expected_u0096_labels!r}\n"
        f"observed={observed_u0096_labels!r}"
    )

if sum(
    cp1252_c1_alias_counts.values()
) != 2180:
    raise RuntimeError(
        "Expected exactly 2,180 Web Attack rows to pass through the "
        "U+0096 -> ASCII-hyphen compatibility mapping."
    )

print()
print(
    "[PASS] Exact Stage24 Web Attack source-encoding aliases reproduced: "
    "1,507 Brute Force + 21 SQL Injection + 652 XSS = 2,180."
)
print(
    "[PASS] This is an implementation compatibility repair only; "
    "the frozen WEB_ATTACK taxonomy is unchanged."
)


# ======================================================================================
# 9. GLOBAL CENSUS VALIDATION
# ======================================================================================

banner("STAGE27-1A MEMBERSHIP :: GLOBAL TAXONOMY CENSUS")

for family in [
    "BENIGN",
    "BOT",
    "DDOS",
    "DOS",
    "AUTH_BRUTE_FORCE",
    "INFILTRATION",
    "PORT_SCAN",
    "WEB_ATTACK",
    "TARGET_ONLY_UNSEEN",
    "OTHER_ATTACK_UNSEEN_LABEL",
]:

    print(
        f"{family:28s}: "
        f"{family_counts.get(family, 0):>10,}"
    )


if family_counts["BENIGN"] != EXPECTED_BENIGN:
    raise RuntimeError(
        "Global benign count mismatch."
    )

primary_total = sum(
    family_counts[
        family
    ]
    for family in primary_order
)

if primary_total != EXPECTED_PRIMARY_ATTACK:
    raise RuntimeError(
        f"Primary seven-family attack total mismatch: {primary_total:,}"
    )

if family_counts[
    "TARGET_ONLY_UNSEEN"
] != EXPECTED_HEARTBLEED:
    raise RuntimeError(
        "TARGET_ONLY_UNSEEN / Heartbleed count mismatch."
    )

if family_counts[
    "OTHER_ATTACK_UNSEEN_LABEL"
] != 0:
    raise RuntimeError(
        "Unexpected non-benign labels outside the frozen taxonomy."
    )

if unknown_nonbenign != 0:
    raise RuntimeError(
        "Unknown non-benign canonical labels were observed."
    )

nonbenign_total = (
    primary_total
    + family_counts[
        "TARGET_ONLY_UNSEEN"
    ]
    + family_counts[
        "OTHER_ATTACK_UNSEEN_LABEL"
    ]
)

if nonbenign_total != EXPECTED_ALL_NONBENIGN:
    raise RuntimeError(
        "Global non-benign count mismatch."
    )

if (
    family_counts["BENIGN"]
    + nonbenign_total
) != EXPECTED_EFFECTIVE_ROWS:
    raise RuntimeError(
        "Global population arithmetic mismatch."
    )

print()
print("Primary seven-family attack:", f"{primary_total:,}")
print("All non-benign             :", f"{nonbenign_total:,}")
print("[PASS] Global taxonomy census exact.")


# ======================================================================================
# 9. FAMILY × DAY + BENIGN × DAY VALIDATION AGAINST STAGE27-0A
# ======================================================================================

banner("STAGE27-1A MEMBERSHIP :: REPRODUCE STAGE27-0A SUPPORT")

for family in primary_order:

    if family not in family_support_expected:
        raise RuntimeError(
            f"Stage27-0A support CSV lacks {family}"
        )

    print()
    print(family)

    for day in DAY_ORDER:

        actual = int(
            family_day_counts[
                family
            ][
                day
            ]
        )

        expected = int(
            family_support_expected[
                family
            ][
                day
            ]
        )

        print(
            f"  {day:10s}: "
            f"{actual:>9,} "
            f"(expected {expected:,})"
        )

        if actual != expected:
            raise RuntimeError(
                f"{family}/{day}: Stage27-0A support mismatch."
            )


print()
print("Benign by day:")

for day in DAY_ORDER:

    actual = int(
        benign_day_counts[
            day
        ]
    )

    expected = int(
        benign_support_expected[
            day
        ]
    )

    print(
        f"  {day:10s}: "
        f"{actual:>9,} "
        f"(expected {expected:,})"
    )

    if actual != expected:
        raise RuntimeError(
            f"{day}: benign support mismatch."
        )


print()
print(
    "[PASS] Row-level Label materialization reproduces the "
    "sealed Stage27-0A family/day and benign/day matrices exactly."
)


# ======================================================================================
# 10. SAVE GLOBAL LABEL-DERIVED CACHE
# ======================================================================================

banner("STAGE27-1A MEMBERSHIP :: SAVE GLOBAL MEMBERSHIP CACHE")

family_codes_path = (
    CACHE_ROOT
    / "global_family_codes.npy"
)

day_codes_path = (
    CACHE_ROOT
    / "global_day_codes.npy"
)

source_codes_path = (
    CACHE_ROOT
    / "global_source_codes.npy"
)

family_codes_receipt = save_npy_with_receipt(
    family_codes_path,
    family_codes,
)

day_codes_receipt = save_npy_with_receipt(
    day_codes_path,
    day_codes,
)

source_codes_receipt = save_npy_with_receipt(
    source_codes_path,
    source_codes,
)

print(
    "family codes:",
    family_codes_receipt[
        "content_sha256"
    ],
)

print(
    "day codes   :",
    day_codes_receipt[
        "content_sha256"
    ],
)

print(
    "source codes:",
    source_codes_receipt[
        "content_sha256"
    ],
)

print(
    "[PASS] Deterministic global code arrays persisted outside Git."
)


# ======================================================================================
# 11. BUILD FIVE ELIGIBLE FOLD MEMBERSHIPS
# ======================================================================================

banner("STAGE27-1A MEMBERSHIP :: BUILD FIVE FROZEN FOLDS")

fold_receipts_runtime = OrderedDict()

primary_mask_global = np.isin(
    family_codes,
    PRIMARY_FAMILY_CODES,
)

binary_included_global = (
    (family_codes == FAMILY_CODE["BENIGN"])
    | primary_mask_global
)

folds = fold_spec[
    "folds"
]

weight_expected_map = class_weight_policy[
    "pre_materialization_expected_values"
]

for family in ELIGIBLE_FOLDS:

    if family not in folds:
        raise RuntimeError(
            f"Frozen fold spec missing eligible family {family}"
        )

    fold = folds[
        family
    ]

    if not fold[
        "scientific_model_fit_authorized"
    ]:
        raise RuntimeError(
            f"{family}: frozen fold unexpectedly not authorized."
        )

    geometry = fold[
        "replacement_geometry"
    ]

    train_days = geometry[
        "train_days"
    ]

    validation_days = geometry[
        "validation_days"
    ]

    target_days = geometry[
        "target_days"
    ]

    if len(
        validation_days
    ) != 1:
        raise RuntimeError(
            f"{family}: expected exactly one validation day."
        )

    if len(
        target_days
    ) != 1:
        raise RuntimeError(
            f"{family}: expected exactly one target day."
        )

    heldout_code = FAMILY_CODE[
        family
    ]

    train_day_codes = np.array(
        [
            DAY_CODE[
                day
            ]
            for day in train_days
        ],
        dtype=np.uint8,
    )

    validation_day_code = np.uint8(
        DAY_CODE[
            validation_days[
                0
            ]
        ]
    )

    target_day_code = np.uint8(
        DAY_CODE[
            target_days[
                0
            ]
        ]
    )

    train_mask = (
        np.isin(
            day_codes,
            train_day_codes,
        )
        & binary_included_global
        & (
            family_codes
            != heldout_code
        )
    )

    validation_mask = (
        (
            day_codes
            == validation_day_code
        )
        & binary_included_global
        & (
            family_codes
            != heldout_code
        )
    )

    primary_target_mask = (
        (
            day_codes
            == target_day_code
        )
        & (
            (
                family_codes
                == FAMILY_CODE["BENIGN"]
            )
            | (
                family_codes
                == heldout_code
            )
        )
    )

    operational_target_mask = (
        (
            day_codes
            == target_day_code
        )
        & binary_included_global
    )

    train_idx = np.flatnonzero(
        train_mask
    ).astype(
        np.int32,
        copy=False,
    )

    validation_idx = np.flatnonzero(
        validation_mask
    ).astype(
        np.int32,
        copy=False,
    )

    primary_target_idx = np.flatnonzero(
        primary_target_mask
    ).astype(
        np.int32,
        copy=False,
    )

    operational_target_idx = np.flatnonzero(
        operational_target_mask
    ).astype(
        np.int32,
        copy=False,
    )

    # ------------------------------------------------------------------
    # Core fold counts
    # ------------------------------------------------------------------

    train_codes = family_codes[
        train_idx
    ]

    validation_codes = family_codes[
        validation_idx
    ]

    primary_target_codes = family_codes[
        primary_target_idx
    ]

    operational_target_codes = family_codes[
        operational_target_idx
    ]

    train_benign = int(
        np.sum(
            train_codes
            == FAMILY_CODE[
                "BENIGN"
            ]
        )
    )

    train_attack = int(
        train_idx.size
        - train_benign
    )

    validation_benign = int(
        np.sum(
            validation_codes
            == FAMILY_CODE[
                "BENIGN"
            ]
        )
    )

    validation_attack = int(
        validation_idx.size
        - validation_benign
    )

    heldout_train_count = int(
        np.sum(
            train_codes
            == heldout_code
        )
    )

    heldout_validation_count = int(
        np.sum(
            validation_codes
            == heldout_code
        )
    )

    target_benign = int(
        np.sum(
            primary_target_codes
            == FAMILY_CODE[
                "BENIGN"
            ]
        )
    )

    target_heldout = int(
        np.sum(
            primary_target_codes
            == heldout_code
        )
    )

    target_other = int(
        primary_target_idx.size
        - target_benign
        - target_heldout
    )

    operational_benign = int(
        np.sum(
            operational_target_codes
            == FAMILY_CODE[
                "BENIGN"
            ]
        )
    )

    operational_attack = int(
        operational_target_idx.size
        - operational_benign
    )

    # ------------------------------------------------------------------
    # Hard frozen expected-count gates
    # ------------------------------------------------------------------

    expected_fields = {
        "train_benign_expected": train_benign,
        "train_known_attack_expected": train_attack,
        "validation_benign_expected": validation_benign,
        "validation_known_attack_expected": validation_attack,
        "primary_target_benign_expected": target_benign,
        "primary_target_heldout_attack_expected": target_heldout,
        "primary_target_rows_expected": int(
            primary_target_idx.size
        ),
    }

    for field, actual in expected_fields.items():

        expected = int(
            geometry[
                field
            ]
        )

        if actual != expected:
            raise RuntimeError(
                f"{family}: {field} mismatch. "
                f"expected={expected:,}, actual={actual:,}"
            )

    if heldout_train_count != 0:
        raise RuntimeError(
            f"{family}: HELD-OUT FAMILY LEAKS INTO TRAIN."
        )

    if heldout_validation_count != 0:
        raise RuntimeError(
            f"{family}: HELD-OUT FAMILY LEAKS INTO VALIDATION."
        )

    if train_benign <= 0 or train_attack <= 0:
        raise RuntimeError(
            f"{family}: training membership is single-class."
        )

    if validation_benign <= 0 or validation_attack <= 0:
        raise RuntimeError(
            f"{family}: validation membership is single-class."
        )

    if target_benign <= 0 or target_heldout <= 0:
        raise RuntimeError(
            f"{family}: primary isolation target is not two-class."
        )

    if target_other != 0:
        raise RuntimeError(
            f"{family}: primary target contains non-benign rows "
            "outside the held-out family."
        )

    # Global row IDs are sorted by source/day chronology.
    if not (
        int(train_idx.max())
        < int(validation_idx.min())
        < int(primary_target_idx.min())
    ):
        raise RuntimeError(
            f"{family}: TRAIN < VALIDATION < TARGET global-order test failed."
        )

    realized_weight = (
        train_benign
        / train_attack
    )

    expected_weight = float(
        weight_expected_map[
            family
        ]
    )

    if not np.isclose(
        realized_weight,
        expected_weight,
        rtol=0.0,
        atol=1e-15,
    ):
        raise RuntimeError(
            f"{family}: realized class weight differs from "
            "Stage27-0 pre-materialization expectation."
        )

    # ------------------------------------------------------------------
    # Save deterministic role membership arrays outside Git.
    # ------------------------------------------------------------------

    fold_cache = (
        CACHE_ROOT
        / family
    )

    fold_cache.mkdir(
        parents=True,
        exist_ok=False,
    )

    role_arrays = OrderedDict(
        [
            ("train_global_idx", train_idx),
            ("validation_global_idx", validation_idx),
            ("primary_target_global_idx", primary_target_idx),
            ("operational_target_global_idx", operational_target_idx),
        ]
    )

    role_receipts = OrderedDict()

    for role_name, arr in role_arrays.items():

        role_path = (
            fold_cache
            / f"{role_name}.npy"
        )

        role_receipts[
            role_name
        ] = save_npy_with_receipt(
            role_path,
            arr,
        )

    # ------------------------------------------------------------------
    # Per-family support within training/validation/operational context.
    # ------------------------------------------------------------------

    train_family_support = OrderedDict()
    validation_family_support = OrderedDict()
    operational_family_support = OrderedDict()

    for candidate in primary_order:

        code = FAMILY_CODE[
            candidate
        ]

        train_family_support[
            candidate
        ] = int(
            np.sum(
                train_codes
                == code
            )
        )

        validation_family_support[
            candidate
        ] = int(
            np.sum(
                validation_codes
                == code
            )
        )

        operational_family_support[
            candidate
        ] = int(
            np.sum(
                operational_target_codes
                == code
            )
        )

    receipt = OrderedDict(
        [
            ("stage", "Stage27-1A"),
            ("type", "FOLD_MEMBERSHIP_RECEIPT"),
            ("created_at_utc", now_utc()),
            ("scientific_parent_commit", EXPECTED_PARENT),
            ("held_out_family", family),
            ("fold_status", fold["status"]),
            ("support_status", fold["support_status"]),
            ("train_days", train_days),
            ("validation_days", validation_days),
            ("target_days", target_days),
            (
                "membership_semantics",
                {
                    "global_row_id": (
                        "ZERO_BASED_INDEX_IN_CONCATENATION_OF_THE_8_FROZEN_"
                        "EFFECTIVE_SOURCE_SEGMENTS_IN_STAGE24_PUBLISHED_POPULATION_ORDER"
                    ),
                    "train": (
                        "BENIGN + PRIMARY-SEVEN KNOWN ATTACK FAMILIES ON FROZEN "
                        "TRAIN DAYS; HELD-OUT FAMILY EXCLUDED; HEARTBLEED/OTHER UNKNOWN EXCLUDED"
                    ),
                    "validation": (
                        "BENIGN + PRIMARY-SEVEN KNOWN ATTACK FAMILIES ON FROZEN "
                        "VALIDATION DAY; HELD-OUT FAMILY EXCLUDED; HEARTBLEED/OTHER UNKNOWN EXCLUDED"
                    ),
                    "primary_target": (
                        "SAME-TARGET-DAY BENIGN + HELD-OUT FAMILY ONLY"
                    ),
                    "operational_target": (
                        "SAME-TARGET-DAY BENIGN + ALL PRIMARY-SEVEN ATTACK FAMILIES"
                    ),
                },
            ),
            (
                "counts",
                {
                    "train_rows": int(train_idx.size),
                    "train_benign": train_benign,
                    "train_attack": train_attack,
                    "validation_rows": int(validation_idx.size),
                    "validation_benign": validation_benign,
                    "validation_attack": validation_attack,
                    "primary_target_rows": int(primary_target_idx.size),
                    "primary_target_benign": target_benign,
                    "primary_target_heldout_attack": target_heldout,
                    "operational_target_rows": int(
                        operational_target_idx.size
                    ),
                    "operational_target_benign": operational_benign,
                    "operational_target_attack": operational_attack,
                },
            ),
            (
                "heldout_exclusion",
                {
                    "train_count": heldout_train_count,
                    "validation_count": heldout_validation_count,
                    "train_required": 0,
                    "validation_required": 0,
                    "status": "PASS",
                },
            ),
            (
                "chronology_and_disjointness",
                {
                    "train_max_global_idx": int(
                        train_idx.max()
                    ),
                    "validation_min_global_idx": int(
                        validation_idx.min()
                    ),
                    "validation_max_global_idx": int(
                        validation_idx.max()
                    ),
                    "target_min_global_idx": int(
                        primary_target_idx.min()
                    ),
                    "strict_global_order": True,
                    "train_validation_disjoint": True,
                    "train_target_disjoint": True,
                    "validation_target_disjoint": True,
                },
            ),
            (
                "class_weight",
                {
                    "formula": "train_benign / train_attack",
                    "train_benign": train_benign,
                    "train_attack": train_attack,
                    "realized_value": realized_weight,
                    "stage27_0_expected_value": expected_weight,
                    "recomputed_from_frozen_train_membership": True,
                    "target_rows_used": 0,
                    "validation_rows_used": 0,
                },
            ),
            (
                "family_support",
                {
                    "train": train_family_support,
                    "validation": validation_family_support,
                    "operational_target": operational_family_support,
                },
            ),
            (
                "runtime_membership_arrays",
                role_receipts,
            ),
            (
                "known_family_control",
                {
                    "membership": "IDENTICAL_TO_FROZEN_VALIDATION_MEMBERSHIP",
                    "threshold_selection_authorized_later": True,
                    "threshold_selection_executed_now": False,
                },
            ),
            (
                "target_opening_semantics",
                {
                    "predictor_columns_read": 0,
                    "label_column_read_for_membership_only": True,
                    "model_inference_on_target": False,
                    "opening_consumed": 0,
                },
            ),
        ]
    )

    fold_receipts_runtime[
        family
    ] = receipt

    print()
    print(
        f"{family}"
    )

    print(
        f"  TRAIN      rows={train_idx.size:,} "
        f"benign={train_benign:,} attack={train_attack:,} "
        f"heldout={heldout_train_count}"
    )

    print(
        f"  VALIDATION rows={validation_idx.size:,} "
        f"benign={validation_benign:,} attack={validation_attack:,} "
        f"heldout={heldout_validation_count}"
    )

    print(
        f"  PRIMARY     rows={primary_target_idx.size:,} "
        f"benign={target_benign:,} heldout={target_heldout:,}"
    )

    print(
        f"  OPERATIONAL rows={operational_target_idx.size:,} "
        f"benign={operational_benign:,} attack={operational_attack:,}"
    )

    print(
        f"  class weight={realized_weight:.15f}"
    )

    print(
        "  exclusion gate: PASS"
    )


print()
print(
    "[PASS] All five eligible folds materialized and "
    "held-out exclusion verified mechanically."
)


# ======================================================================================
# 12. STRUCTURALLY INELIGIBLE FOLD CHECK
# ======================================================================================

banner("STAGE27-1A MEMBERSHIP :: STRUCTURAL INELIGIBILITY PRESERVATION")

for family in STRUCTURALLY_INELIGIBLE:

    fold = folds[
        family
    ]

    print(
        f"{family:18s} "
        f"status={fold['status']} "
        f"reason={fold['reason_code']}"
    )

    if fold[
        "scientific_model_fit_authorized"
    ] is not False:
        raise RuntimeError(
            f"{family}: structurally ineligible fold became fit-authorized."
        )

    if fold[
        "replacement_geometry"
    ] is not None:
        raise RuntimeError(
            f"{family}: structurally ineligible fold unexpectedly has geometry."
        )


print(
    "[PASS] DOS and AUTH_BRUTE_FORCE remain retained but non-executable."
)


# ======================================================================================
# 13. GLOBAL RUNTIME CACHE RE-READ VERIFICATION
# ======================================================================================

banner("STAGE27-1A MEMBERSHIP :: RUNTIME CACHE RE-READ")

family_codes_reload = np.load(
    family_codes_path,
    allow_pickle=False,
    mmap_mode="r",
)

day_codes_reload = np.load(
    day_codes_path,
    allow_pickle=False,
    mmap_mode="r",
)

source_codes_reload = np.load(
    source_codes_path,
    allow_pickle=False,
    mmap_mode="r",
)

if (
    family_codes_reload.shape
    != family_codes.shape
):
    raise RuntimeError(
        "Reloaded family-code shape mismatch."
    )

if (
    content_sha256_array(
        np.asarray(
            family_codes_reload
        )
    )
    != family_codes_receipt[
        "content_sha256"
    ]
):
    raise RuntimeError(
        "Reloaded family-code content hash mismatch."
    )

if (
    content_sha256_array(
        np.asarray(
            day_codes_reload
        )
    )
    != day_codes_receipt[
        "content_sha256"
    ]
):
    raise RuntimeError(
        "Reloaded day-code content hash mismatch."
    )

if (
    content_sha256_array(
        np.asarray(
            source_codes_reload
        )
    )
    != source_codes_receipt[
        "content_sha256"
    ]
):
    raise RuntimeError(
        "Reloaded source-code content hash mismatch."
    )

for family, receipt in fold_receipts_runtime.items():

    for role_name, role_receipt in receipt[
        "runtime_membership_arrays"
    ].items():

        arr = np.load(
            role_receipt[
                "path"
            ],
            allow_pickle=False,
            mmap_mode="r",
        )

        if int(
            arr.size
        ) != int(
            role_receipt[
                "count"
            ]
        ):
            raise RuntimeError(
                f"{family}/{role_name}: reload count mismatch."
            )

        if (
            content_sha256_array(
                np.asarray(
                    arr
                )
            )
            != role_receipt[
                "content_sha256"
            ]
        ):
            raise RuntimeError(
                f"{family}/{role_name}: reload content hash mismatch."
            )

print(
    "[PASS] All global and fold membership arrays re-read byte/content exact."
)


# ======================================================================================
# 14. ONLY NOW CREATE DURABLE REPOSITORY RECEIPTS
# ======================================================================================

banner("STAGE27-1A MEMBERSHIP :: WRITE DURABLE RECEIPTS")

OUT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

source_effective_population_receipt = OrderedDict(
    [
        ("stage", "Stage27-1A"),
        ("type", "SOURCE_EFFECTIVE_POPULATION_RECEIPT"),
        ("created_at_utc", now_utc()),
        ("scientific_parent_commit", EXPECTED_PARENT),
        ("source_root_runtime", str(SOURCE_ROOT)),
        ("source_count", 8),
        ("segments", segments),
        (
            "population",
            {
                "physical_rows": physical_total,
                "effective_rows": effective_total,
                "structural_null_excluded_rows": excluded_total,
                "frozen_effective_population_identity": (
                    "STAGE24_FULL_EFFECTIVE_CICIDS2017_POPULATION"
                ),
            },
        ),
        (
            "predictor_columns_read",
            0,
        ),
        (
            "label_column_read",
            True,
        ),
        (
            "model_fits",
            0,
        ),
        (
            "target_openings",
            0,
        ),
    ]
)

write_json(
    OUT_DIR
    / "source_effective_population_receipt.json",
    source_effective_population_receipt,
)


canonical_label_census = OrderedDict(
    [
        ("stage", "Stage27-1A"),
        ("type", "CANONICAL_LABEL_CENSUS"),
        ("created_at_utc", now_utc()),
        ("scientific_parent_commit", EXPECTED_PARENT),
        (
            "canonicalization",
            [
                "STRIP_OUTER_WHITESPACE",
                "NORMALIZE_UNICODE_DASHES_TO_ASCII_HYPHEN",
                "COLLAPSE_INTERNAL_WHITESPACE",
                "CASEFOLD_FOR_LOOKUP",
                (
                    "MAP_ONLY_THE_THREE_FROZEN_CICIDS2017_WEB_ATTACK_"
                    "U+0096_CP1252_EN_DASH_COMPATIBILITY_ALIASES_TO_"
                    "THE_ALREADY_FROZEN_ASCII_HYPHEN_TAXONOMY_KEYS"
                ),
            ],
        ),
        (
            "source_encoding_compatibility",
            {
                "issue": (
                    "The byte-exact CICIDS2017 traffic_labels Web Attack "
                    "strings preserve Windows-1252 byte 0x96 as Unicode "
                    "U+0096. The durable Stage24 published census records "
                    "these exact strings."
                ),
                "taxonomy_change": False,
                "authorized_aliases": WEB_ATTACK_CP1252_C1_ALIASES,
                "expected_and_observed_alias_counts": OrderedDict(
                    (
                        label,
                        int(
                            cp1252_c1_alias_counts[
                                label
                            ]
                        ),
                    )
                    for label in expected_web_precompat
                ),
                "total_rows_mapped": int(
                    sum(
                        cp1252_c1_alias_counts.values()
                    )
                ),
                "durable_evidence_path": str(
                    STAGE24_PUBLISHED_RESULT.relative_to(
                        REPO
                    )
                ),
                "durable_evidence_sha256": sha256_file(
                    STAGE24_PUBLISHED_RESULT
                ),
                "status": "PASS_EXACT",
            },
        ),
        (
            "precompat_canonical_label_counts",
            OrderedDict(
                sorted(
                    (
                        key,
                        int(value),
                    )
                    for key, value in precompat_label_counts.items()
                )
            ),
        ),
        (
            "raw_canonical_label_counts",
            OrderedDict(
                sorted(
                    (
                        key,
                        int(value),
                    )
                    for key, value in raw_label_counts.items()
                )
            ),
        ),
        (
            "family_counts",
            OrderedDict(
                (
                    family,
                    int(
                        family_counts.get(
                            family,
                            0,
                        )
                    ),
                )
                for family in FAMILY_CODE.keys()
            ),
        ),
        (
            "family_day_counts",
            OrderedDict(
                (
                    family,
                    OrderedDict(
                        (
                            day,
                            int(
                                family_day_counts[
                                    family
                                ][
                                    day
                                ]
                            ),
                        )
                        for day in DAY_ORDER
                    ),
                )
                for family in primary_order
            ),
        ),
        (
            "benign_day_counts",
            OrderedDict(
                (
                    day,
                    int(
                        benign_day_counts[
                            day
                        ]
                    ),
                )
                for day in DAY_ORDER
            ),
        ),
        (
            "global_arithmetic",
            {
                "benign": int(
                    family_counts[
                        "BENIGN"
                    ]
                ),
                "primary_seven_family_attack": int(
                    primary_total
                ),
                "target_only_unseen": int(
                    family_counts[
                        "TARGET_ONLY_UNSEEN"
                    ]
                ),
                "other_attack_unseen_label": int(
                    family_counts[
                        "OTHER_ATTACK_UNSEEN_LABEL"
                    ]
                ),
                "all_nonbenign": int(
                    nonbenign_total
                ),
                "effective_rows": int(
                    EXPECTED_EFFECTIVE_ROWS
                ),
            },
        ),
        (
            "stage27_0a_support_reproduction",
            "PASS_EXACT",
        ),
    ]
)

write_json(
    OUT_DIR
    / "canonical_label_census.json",
    canonical_label_census,
)


family_code_cache_receipt = OrderedDict(
    [
        ("stage", "Stage27-1A"),
        ("type", "RUNTIME_FAMILY_CODE_CACHE_RECEIPT"),
        ("created_at_utc", now_utc()),
        ("scientific_parent_commit", EXPECTED_PARENT),
        (
            "cache_root",
            str(CACHE_ROOT),
        ),
        (
            "global_row_semantics",
            (
                "ZERO_BASED_INDEX_IN_CONCATENATION_OF_8_FROZEN_EFFECTIVE_"
                "SOURCE_SEGMENTS_IN_STAGE24_PUBLISHED_POPULATION_ORDER"
            ),
        ),
        (
            "family_codebook",
            FAMILY_CODE,
        ),
        (
            "arrays",
            {
                "global_family_codes": family_codes_receipt,
                "global_day_codes": day_codes_receipt,
                "global_source_codes": source_codes_receipt,
            },
        ),
        (
            "rebuild_rule",
            (
                "If the Kaggle runtime cache is lost, rebuild from the same 8 "
                "byte-exact frozen source assets, effective-row rules and taxonomy, "
                "then require these content hashes before any model fit."
            ),
        ),
    ]
)

write_json(
    OUT_DIR
    / "family_code_cache_receipt.json",
    family_code_cache_receipt,
)


for family in ELIGIBLE_FOLDS:

    write_json(
        OUT_DIR
        / f"fold_{family}_membership_receipt.json",
        fold_receipts_runtime[
            family
        ],
    )


# ======================================================================================
# 15. FREEZE RECORD HASHES OTHER EIGHT RECEIPTS
# ======================================================================================

non_freeze_names = [
    name
    for name in REQUIRED_OUTPUT_FILES
    if name
    != "stage27_1a_membership_freeze_record.json"
]

receipt_hashes = OrderedDict()

for name in non_freeze_names:

    path = (
        OUT_DIR
        / name
    )

    if not path.is_file():
        raise RuntimeError(
            f"Durable membership receipt missing: {name}"
        )

    receipt_hashes[
        name
    ] = {
        "bytes": int(
            path.stat().st_size
        ),
        "sha256": sha256_file(
            path
        ),
    }


membership_freeze_record = OrderedDict(
    [
        ("stage", "Stage27-1A"),
        (
            "status",
            "MEMBERSHIP_FROZEN_LOCALLY_PENDING_COMMIT_PUSH_REMOTE_VERIFICATION",
        ),
        ("created_at_utc", now_utc()),
        ("scientific_parent_commit", EXPECTED_PARENT),
        ("required_artifact_count", 9),
        (
            "artifact_hashes_before_freeze_record",
            receipt_hashes,
        ),
        (
            "source_population",
            {
                "physical_rows": EXPECTED_PHYSICAL_ROWS,
                "effective_rows": EXPECTED_EFFECTIVE_ROWS,
                "structural_null_excluded": EXPECTED_STRUCTURAL_EXCLUDED,
                "byte_exact_source_count": 8,
            },
        ),
        (
            "eligible_folds",
            ELIGIBLE_FOLDS,
        ),
        (
            "eligible_fold_count",
            5,
        ),
        (
            "structurally_ineligible",
            STRUCTURALLY_INELIGIBLE,
        ),
        (
            "descriptive_only",
            DESCRIPTIVE_ONLY,
        ),
        (
            "source_label_compatibility_status",
            "PASS_EXACT_STAGE24_U0096_WEB_ATTACK_ALIAS_REPRODUCTION",
        ),
        (
            "heldout_exclusion_global_status",
            "PASS_ALL_5_FOLDS",
        ),
        (
            "heldout_train_counts",
            {
                family: fold_receipts_runtime[
                    family
                ][
                    "heldout_exclusion"
                ][
                    "train_count"
                ]
                for family in ELIGIBLE_FOLDS
            },
        ),
        (
            "heldout_validation_counts",
            {
                family: fold_receipts_runtime[
                    family
                ][
                    "heldout_exclusion"
                ][
                    "validation_count"
                ]
                for family in ELIGIBLE_FOLDS
            },
        ),
        (
            "scientific_actions_completed",
            {
                "source_downloads_this_stage": 0,
                "label_rows_processed": EXPECTED_EFFECTIVE_ROWS,
                "predictor_columns_read": 0,
                "feature_rows_materialized": 0,
                "model_fits": 0,
                "model_inference": 0,
                "threshold_selections": 0,
                "target_predictor_openings": 0,
                "bootstrap_replicates": 0,
                "gpu_hours": 0,
            },
        ),
        (
            "next_authorized_step_after_remote_verification",
            (
                "MATERIALIZE FROZEN 70-FEATURE TRAIN/VALIDATION MATRICES AND "
                "TRAIN THE TWO PREREGISTERED LEARNERS PER ELIGIBLE FOLD ON CPU, "
                "ONLY AFTER THE MEMBERSHIP RECEIPTS ARE COMMITTED/PUSHED/REMOTE-VERIFIED."
            ),
        ),
        (
            "fit_authorization_now",
            False,
        ),
    ]
)

write_json(
    OUT_DIR
    / "stage27_1a_membership_freeze_record.json",
    membership_freeze_record,
)


# ======================================================================================
# 16. EXACT 9-FILE LOCAL OUTPUT VALIDATION
# ======================================================================================

banner("STAGE27-1A MEMBERSHIP :: FINAL LOCAL RECEIPT VALIDATION")

actual_files = sorted(
    p.name
    for p in OUT_DIR.iterdir()
    if p.is_file()
)

expected_files = sorted(
    REQUIRED_OUTPUT_FILES
)

print("Expected durable receipts:", len(expected_files))
print("Actual durable receipts  :", len(actual_files))

if actual_files != expected_files:
    raise RuntimeError(
        "Stage27-1A durable output universe mismatch.\n"
        f"expected={expected_files}\n"
        f"actual={actual_files}"
    )

for name in REQUIRED_OUTPUT_FILES:

    path = (
        OUT_DIR
        / name
    )

    print(
        f"{name:50s} "
        f"bytes={path.stat().st_size:>10,} "
        f"sha256={sha256_file(path)}"
    )


print()
print(
    "[PASS] Exactly nine Stage27-1A membership receipts exist."
)


# ======================================================================================
# 17. FREEZE RECORD READBACK
# ======================================================================================

banner("STAGE27-1A MEMBERSHIP :: FREEZE READBACK")

read_freeze = load_json(
    OUT_DIR
    / "stage27_1a_membership_freeze_record.json"
)

if read_freeze[
    "required_artifact_count"
] != 9:
    raise RuntimeError(
        "Membership freeze artifact count mismatch."
    )

if len(
    read_freeze[
        "artifact_hashes_before_freeze_record"
    ]
) != 8:
    raise RuntimeError(
        "Membership freeze should hash the other 8 receipts."
    )

for family in ELIGIBLE_FOLDS:

    if read_freeze[
        "heldout_train_counts"
    ][family] != 0:
        raise RuntimeError(
            f"{family}: nonzero heldout train count in freeze record."
        )

    if read_freeze[
        "heldout_validation_counts"
    ][family] != 0:
        raise RuntimeError(
            f"{family}: nonzero heldout validation count in freeze record."
        )

if read_freeze[
    "scientific_actions_completed"
][
    "predictor_columns_read"
] != 0:
    raise RuntimeError(
        "Predictor read counter is nonzero."
    )

if read_freeze[
    "scientific_actions_completed"
][
    "target_predictor_openings"
] != 0:
    raise RuntimeError(
        "Target predictor opening counter is nonzero."
    )

if read_freeze[
    "scientific_actions_completed"
][
    "model_fits"
] != 0:
    raise RuntimeError(
        "Model-fit counter is nonzero."
    )


print("Required receipts          :", read_freeze["required_artifact_count"])
print("Non-self receipt hashes    :", len(read_freeze["artifact_hashes_before_freeze_record"]))
print("Heldout exclusion status   :", read_freeze["heldout_exclusion_global_status"])
print("Predictor columns read     :", read_freeze["scientific_actions_completed"]["predictor_columns_read"])
print("Model fits                 :", read_freeze["scientific_actions_completed"]["model_fits"])
print("Target predictor openings  :", read_freeze["scientific_actions_completed"]["target_predictor_openings"])
print("[PASS] Membership freeze record is internally consistent.")


# ======================================================================================
# 18. STAGE EXACT NINE RECEIPTS — NO COMMIT
# ======================================================================================

banner("STAGE27-1A MEMBERSHIP :: STAGE EXACT RECEIPT SET")

for name in REQUIRED_OUTPUT_FILES:

    run(
        [
            "git",
            "add",
            "--",
            str(
                OUT_REL
                / name
            ),
        ]
    )


staged = sorted(
    line
    for line in run(
        [
            "git",
            "diff",
            "--cached",
            "--name-only",
        ]
    ).splitlines()
    if line.strip()
)

expected_staged = sorted(
    str(
        OUT_REL
        / name
    )
    for name in REQUIRED_OUTPUT_FILES
)

print("Expected staged:", len(expected_staged))
print("Actual staged  :", len(staged))

if staged != expected_staged:
    raise RuntimeError(
        "Unexpected staged membership file universe.\n"
        f"expected={expected_staged}\n"
        f"actual={staged}"
    )


porcelain = sorted(
    line
    for line in run(
        [
            "git",
            "status",
            "--porcelain=v1",
            "--untracked-files=all",
        ]
    ).splitlines()
    if line.strip()
)

expected_porcelain = sorted(
    f"A  {path}"
    for path in expected_staged
)

if porcelain != expected_porcelain:
    raise RuntimeError(
        "Repository contains changes outside the exact "
        "Stage27-1A receipt set.\n\n"
        + "\n".join(
            porcelain
        )
    )


print()
print(
    "[PASS] Exactly nine Stage27-1A membership receipts staged."
)


# ======================================================================================
# 19. FINAL SCIENTIFIC BOUNDARY
# ======================================================================================

banner("STAGE27-1A MEMBERSHIP MATERIALIZATION COMPLETE — LOCAL ONLY")

print("Scientific parent       :", EXPECTED_PARENT)
print("Effective population    :", f"{EXPECTED_EFFECTIVE_ROWS:,}")
print("Primary attack rows     :", f"{primary_total:,}")
print("Benign rows             :", f"{family_counts['BENIGN']:,}")
print("Heartbleed excluded     :", f"{family_counts['TARGET_ONLY_UNSEEN']:,}")
print("Unknown nonbenign       :", f"{family_counts['OTHER_ATTACK_UNSEEN_LABEL']:,}")
print("Web Attack U+0096 alias :", f"{sum(cp1252_c1_alias_counts.values()):,} rows / PASS")
print()
print("Eligible fold memberships: 5 / 5")
print("Heldout TRAIN counts     :", {
    family: fold_receipts_runtime[family]["heldout_exclusion"]["train_count"]
    for family in ELIGIBLE_FOLDS
})
print("Heldout VALID counts     :", {
    family: fold_receipts_runtime[family]["heldout_exclusion"]["validation_count"]
    for family in ELIGIBLE_FOLDS
})
print()
print("Predictor columns read   : 0 / 70")
print("Feature matrices built   : 0")
print("Model fits               : 0")
print("Inference                : 0")
print("Threshold selections     : 0")
print("Target predictor openings: 0")
print("Bootstrap replicates     : 0")
print("GPU hours                : 0")
print()
print("Durable receipts         : 9 / 9 STAGED")
print("Commit performed         : NO")
print("Push performed           : NO")
print()
print("NO MODEL FIT IS AUTHORIZED YET.")
print()
print(
    "NEXT GATE: inspect this output, then commit/push/remote-verify "
    "the exact nine Stage27-1A membership receipts."
)



STAGE27-1A MEMBERSHIP :: SCIENTIFIC-PARENT GATE
Expected parent : 1ea6bedb141bc3fa6115edbc5296d0f7c6252559
Local HEAD      : 1ea6bedb141bc3fa6115edbc5296d0f7c6252559
origin/main     : 1ea6bedb141bc3fa6115edbc5296d0f7c6252559
Git clean       : True
[PASS] Frozen Stage27-0 is the clean scientific parent.

STAGE27-1A MEMBERSHIP :: OUTPUT GATES
Source cache      : /kaggle/working/stage27_cicids2017_sources
Membership cache  : /kaggle/working/stage27_1a_membership_cache
Durable output    : /kaggle/working/ids2018-validation-safe-ablation/results/stage27_loao_unseen_attack/stage27_1a_fold_membership
Repo writes yet   : 0

STAGE27-1A MEMBERSHIP :: LOAD FROZEN PROTOCOL
Eligible folds       : 5
Authorized fits      : 10
Feature count        : 70
Fits completed       : 0
Target openings      : 0
[PASS] Frozen Stage27-0 protocol loaded.

STAGE27-1A MEMBERSHIP :: TAXONOMY GATE
Canonical taxonomy entries : 14
Primary families           : ['BOT', 'DDOS', 'DOS', 'AUTH_BRUTE_FORCE', 'INFILTRATION', '

In [19]:
# ======================================================================================
# STAGE27-1A — COMMIT / PUSH / REMOTE VERIFICATION
#
# Preconditions:
#   scientific parent:
#     1ea6bedb141bc3fa6115edbc5296d0f7c6252559
#
#   exactly 9 Stage27-1A membership receipts staged
#
# This script performs ONLY:
#   - staged-state verification
#   - freeze-record verification
#   - git commit
#   - push
#   - remote SHA verification
#   - exact 9-file remote-tree verification
#   - byte-level SHA256 verification
#   - clean-tree closure
#
# ZERO model fits
# ZERO inference
# ZERO threshold selection
# ZERO target predictor openings
# ZERO bootstrap
# ZERO GPU
# ======================================================================================

from __future__ import annotations

import hashlib
import json
import os
import stat
import subprocess
from pathlib import Path

from kaggle_secrets import UserSecretsClient


# ======================================================================================
# 0. CONSTANTS
# ======================================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_PARENT = (
    "1ea6bedb141bc3fa6115edbc5296d0f7c6252559"
)

OUT_REL = Path(
    "results/stage27_loao_unseen_attack/"
    "stage27_1a_fold_membership"
)

OUT_DIR = REPO / OUT_REL

REQUIRED_FILES = [
    "source_effective_population_receipt.json",
    "canonical_label_census.json",
    "family_code_cache_receipt.json",
    "fold_BOT_membership_receipt.json",
    "fold_DDOS_membership_receipt.json",
    "fold_INFILTRATION_membership_receipt.json",
    "fold_PORT_SCAN_membership_receipt.json",
    "fold_WEB_ATTACK_membership_receipt.json",
    "stage27_1a_membership_freeze_record.json",
]

FREEZE_FILE = (
    "stage27_1a_membership_freeze_record.json"
)

COMMIT_MESSAGE = (
    "stage27-1a: freeze LOAO fold memberships"
)


# ======================================================================================
# 1. HELPERS
# ======================================================================================

def banner(title: str) -> None:
    print()
    print("=" * 118)
    print(title)
    print("=" * 118)


def run(
    cmd,
    *,
    env=None,
    text=True,
    check=True,
):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=REPO,
        env=env,
        text=text,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}): "
            f"{' '.join(map(str, cmd))}\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    if text:
        return p.stdout.strip()

    return p.stdout


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        for block in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(
        data
    ).hexdigest()


def load_json(path: Path):
    with path.open(
        "r",
        encoding="utf-8",
    ) as f:
        return json.load(f)


# ======================================================================================
# 2. PRE-COMMIT SCIENTIFIC-PARENT GATE
# ======================================================================================

banner("STAGE27-1A :: PRE-COMMIT SCIENTIFIC-PARENT GATE")

head = run(
    [
        "git",
        "rev-parse",
        "HEAD",
    ]
)

origin_before = run(
    [
        "git",
        "rev-parse",
        "origin/main",
    ]
)

print("Expected parent :", EXPECTED_PARENT)
print("Local HEAD      :", head)
print("origin/main     :", origin_before)

if head != EXPECTED_PARENT:
    raise RuntimeError(
        "Local HEAD changed before Stage27-1A commit."
    )

if origin_before != EXPECTED_PARENT:
    raise RuntimeError(
        "origin/main changed before Stage27-1A commit."
    )

print("[PASS] Scientific parent unchanged.")


# ======================================================================================
# 3. EXACT NINE-FILE STAGED UNIVERSE
# ======================================================================================

banner("STAGE27-1A :: EXACT STAGED UNIVERSE")

expected_paths = sorted(
    str(
        OUT_REL / name
    )
    for name in REQUIRED_FILES
)

staged_paths = sorted(
    line
    for line in run(
        [
            "git",
            "diff",
            "--cached",
            "--name-only",
        ]
    ).splitlines()
    if line.strip()
)

print("Expected staged files:", len(expected_paths))
print("Actual staged files  :", len(staged_paths))

if staged_paths != expected_paths:
    raise RuntimeError(
        "Staged file universe changed.\n"
        f"expected={expected_paths}\n"
        f"actual={staged_paths}"
    )

status_before = sorted(
    line
    for line in run(
        [
            "git",
            "status",
            "--porcelain=v1",
            "--untracked-files=all",
        ]
    ).splitlines()
    if line.strip()
)

expected_status = sorted(
    f"A  {path}"
    for path in expected_paths
)

if status_before != expected_status:
    raise RuntimeError(
        "Unexpected repository state before commit.\n\n"
        + "\n".join(status_before)
    )

print(
    "[PASS] Exactly 9 Stage27-1A membership receipts staged; "
    "no unrelated changes."
)


# ======================================================================================
# 4. FREEZE RECORD / RECEIPT HASH GATE
# ======================================================================================

banner("STAGE27-1A :: MEMBERSHIP FREEZE RECORD GATE")

freeze_path = (
    OUT_DIR
    / FREEZE_FILE
)

freeze = load_json(
    freeze_path
)

print(
    "required_artifact_count   :",
    freeze.get(
        "required_artifact_count"
    ),
)

print(
    "non-self artifact hashes  :",
    len(
        freeze.get(
            "artifact_hashes_before_freeze_record",
            {},
        )
    ),
)

print(
    "heldout exclusion status  :",
    freeze.get(
        "heldout_exclusion_global_status"
    ),
)

if freeze.get(
    "scientific_parent_commit"
) != EXPECTED_PARENT:
    raise RuntimeError(
        "Membership freeze scientific parent mismatch."
    )

if freeze.get(
    "required_artifact_count"
) != 9:
    raise RuntimeError(
        "Membership freeze required_artifact_count != 9."
    )

hashed = freeze[
    "artifact_hashes_before_freeze_record"
]

expected_nonself = sorted(
    name
    for name in REQUIRED_FILES
    if name != FREEZE_FILE
)

if sorted(
    hashed.keys()
) != expected_nonself:
    raise RuntimeError(
        "Membership freeze non-self receipt universe mismatch."
    )

for name in expected_nonself:

    path = (
        OUT_DIR
        / name
    )

    actual_sha = sha256_file(
        path
    )

    expected_sha = hashed[
        name
    ][
        "sha256"
    ]

    if actual_sha != expected_sha:
        raise RuntimeError(
            f"Receipt SHA mismatch for {name}\n"
            f"expected={expected_sha}\n"
            f"actual={actual_sha}"
        )

    actual_bytes = int(
        path.stat().st_size
    )

    expected_bytes = int(
        hashed[
            name
        ][
            "bytes"
        ]
    )

    if actual_bytes != expected_bytes:
        raise RuntimeError(
            f"Receipt byte-size mismatch for {name}."
        )

if freeze.get(
    "heldout_exclusion_global_status"
) != "PASS_ALL_5_FOLDS":
    raise RuntimeError(
        "Held-out exclusion status is not PASS_ALL_5_FOLDS."
    )

for family in [
    "BOT",
    "DDOS",
    "INFILTRATION",
    "PORT_SCAN",
    "WEB_ATTACK",
]:

    if freeze[
        "heldout_train_counts"
    ][
        family
    ] != 0:
        raise RuntimeError(
            f"{family}: held-out TRAIN count is nonzero."
        )

    if freeze[
        "heldout_validation_counts"
    ][
        family
    ] != 0:
        raise RuntimeError(
            f"{family}: held-out VALIDATION count is nonzero."
        )

actions = freeze[
    "scientific_actions_completed"
]

required_zero_actions = [
    "predictor_columns_read",
    "feature_rows_materialized",
    "model_fits",
    "model_inference",
    "threshold_selections",
    "target_predictor_openings",
    "bootstrap_replicates",
    "gpu_hours",
]

for key in required_zero_actions:

    if int(
        actions[
            key
        ]
    ) != 0:
        raise RuntimeError(
            f"Scientific boundary violation: {key} != 0"
        )

print(
    "[PASS] Membership freeze hashes other 8 receipts exactly."
)

print(
    "[PASS] Held-out TRAIN/VALIDATION counts are zero for all 5 folds."
)

print(
    "[PASS] Predictor/model/target-opening counters remain zero."
)


# ======================================================================================
# 5. READBACK FIVE FOLD RECEIPTS
# ======================================================================================

banner("STAGE27-1A :: FOLD RECEIPT READBACK")

fold_names = [
    "BOT",
    "DDOS",
    "INFILTRATION",
    "PORT_SCAN",
    "WEB_ATTACK",
]

for family in fold_names:

    path = (
        OUT_DIR
        / f"fold_{family}_membership_receipt.json"
    )

    receipt = load_json(
        path
    )

    exclusion = receipt[
        "heldout_exclusion"
    ]

    counts = receipt[
        "counts"
    ]

    weight = receipt[
        "class_weight"
    ]

    if exclusion[
        "status"
    ] != "PASS":
        raise RuntimeError(
            f"{family}: exclusion receipt not PASS."
        )

    if exclusion[
        "train_count"
    ] != 0:
        raise RuntimeError(
            f"{family}: nonzero TRAIN held-out count."
        )

    if exclusion[
        "validation_count"
    ] != 0:
        raise RuntimeError(
            f"{family}: nonzero VALID held-out count."
        )

    if receipt[
        "target_opening_semantics"
    ][
        "opening_consumed"
    ] != 0:
        raise RuntimeError(
            f"{family}: target opening already consumed."
        )

    if receipt[
        "target_opening_semantics"
    ][
        "predictor_columns_read"
    ] != 0:
        raise RuntimeError(
            f"{family}: target predictor columns were read."
        )

    print()
    print(family)

    print(
        "  TRAIN      :",
        f"{counts['train_rows']:,}",
        "rows |",
        f"{counts['train_benign']:,}",
        "benign |",
        f"{counts['train_attack']:,}",
        "attack",
    )

    print(
        "  VALIDATION :",
        f"{counts['validation_rows']:,}",
        "rows |",
        f"{counts['validation_benign']:,}",
        "benign |",
        f"{counts['validation_attack']:,}",
        "attack",
    )

    print(
        "  PRIMARY    :",
        f"{counts['primary_target_rows']:,}",
        "rows |",
        f"{counts['primary_target_benign']:,}",
        "benign |",
        f"{counts['primary_target_heldout_attack']:,}",
        "held-out attack",
    )

    print(
        "  class wt   :",
        f"{weight['realized_value']:.15f}",
    )

    print(
        "  exclusion  : PASS"
    )

print()
print("[PASS] Five fold receipts read back successfully.")


# ======================================================================================
# 6. GIT IDENTITY
# ======================================================================================

banner("STAGE27-1A :: GIT AUTHOR IDENTITY")

git_name = run(
    [
        "git",
        "config",
        "--local",
        "user.name",
    ],
    check=False,
)

git_email = run(
    [
        "git",
        "config",
        "--local",
        "user.email",
    ],
    check=False,
)

if not git_name:
    run(
        [
            "git",
            "config",
            "--local",
            "user.name",
            "J.M. Mubasshir Rahman",
        ]
    )

    git_name = run(
        [
            "git",
            "config",
            "--local",
            "user.name",
        ]
    )

if not git_email:
    run(
        [
            "git",
            "config",
            "--local",
            "user.email",
            "themubasshir@users.noreply.github.com",
        ]
    )

    git_email = run(
        [
            "git",
            "config",
            "--local",
            "user.email",
        ]
    )

print("Git author name :", git_name)
print("Git author email:", git_email)
print("[PASS] Repository-local Git identity available.")


# ======================================================================================
# 7. GITHUB AUTH
# ======================================================================================

banner("STAGE27-1A :: LOAD GITHUB AUTH")

client = UserSecretsClient()

token = None
token_label = None

for label in [
    "GITHUB_TOKEN",
    "github_token",
    "GH_TOKEN",
    "GITHUB_PAT",
    "github_pat",
]:

    try:
        candidate = client.get_secret(
            label
        )
    except Exception:
        candidate = None

    if candidate:

        candidate = str(
            candidate
        ).strip()

        if candidate:
            token = candidate
            token_label = label
            break

if not token:
    raise RuntimeError(
        "No GitHub token found in Kaggle Secrets."
    )

print(
    f"[FOUND] GitHub secret label: "
    f"{token_label} "
    "(value not printed)"
)

askpass = Path(
    "/kaggle/working/"
    ".stage27_1a_commit_askpass.sh"
)

askpass.write_text(
    "#!/bin/sh\n"
    'case "$1" in\n'
    '  *Username*) printf "%s\\n" "x-access-token" ;;\n'
    '  *Password*) printf "%s\\n" "$STAGE27_GITHUB_TOKEN" ;;\n'
    '  *) printf "%s\\n" "" ;;\n'
    "esac\n",
    encoding="utf-8",
)

askpass.chmod(
    stat.S_IRUSR
    | stat.S_IWUSR
    | stat.S_IXUSR
)

env = os.environ.copy()

env[
    "GIT_ASKPASS"
] = str(
    askpass
)

env[
    "GIT_TERMINAL_PROMPT"
] = "0"

env[
    "STAGE27_GITHUB_TOKEN"
] = token


# ======================================================================================
# 8. COMMIT / PUSH / VERIFY
# ======================================================================================

try:

    banner("STAGE27-1A :: COMMIT")

    print(
        run(
            [
                "git",
                "diff",
                "--cached",
                "--stat",
            ]
        )
    )

    commit_output = run(
        [
            "git",
            "commit",
            "-m",
            COMMIT_MESSAGE,
        ]
    )

    print()
    print(commit_output)

    new_head = run(
        [
            "git",
            "rev-parse",
            "HEAD",
        ]
    )

    new_parent = run(
        [
            "git",
            "rev-parse",
            "HEAD^",
        ]
    )

    subject = run(
        [
            "git",
            "show",
            "-s",
            "--format=%s",
            "HEAD",
        ]
    )

    print()
    print("Stage27-1A commit:", new_head)
    print("Commit parent    :", new_parent)
    print("Commit subject   :", subject)

    if new_parent != EXPECTED_PARENT:
        raise RuntimeError(
            "Stage27-1A commit parent mismatch."
        )

    if subject != COMMIT_MESSAGE:
        raise RuntimeError(
            "Stage27-1A commit subject mismatch."
        )


    # ------------------------------------------------------------------
    # Local committed tree
    # ------------------------------------------------------------------

    banner("STAGE27-1A :: LOCAL COMMIT TREE")

    committed_paths = sorted(
        line
        for line in run(
            [
                "git",
                "ls-tree",
                "-r",
                "--name-only",
                "HEAD",
                "--",
                str(
                    OUT_REL
                ),
            ]
        ).splitlines()
        if line.strip()
    )

    print(
        "Committed Stage27-1A files:",
        len(
            committed_paths
        ),
    )

    if committed_paths != expected_paths:
        raise RuntimeError(
            "Committed Stage27-1A tree mismatch."
        )

    print(
        "[PASS] Local commit contains exactly 9 membership receipts."
    )


    # ------------------------------------------------------------------
    # Push
    # ------------------------------------------------------------------

    banner("STAGE27-1A :: PUSH MAIN")

    push_output = run(
        [
            "git",
            "push",
            "origin",
            "main",
        ],
        env=env,
    )

    print(
        push_output
        if push_output
        else "[OK] Push completed."
    )


    # ------------------------------------------------------------------
    # Remote SHA
    # ------------------------------------------------------------------

    banner("STAGE27-1A :: REMOTE SHA VERIFICATION")

    remote_main = run(
        [
            "git",
            "ls-remote",
            "origin",
            "refs/heads/main",
        ],
        env=env,
    ).split()[0]

    run(
        [
            "git",
            "fetch",
            "origin",
            "main",
        ],
        env=env,
    )

    local_after = run(
        [
            "git",
            "rev-parse",
            "HEAD",
        ]
    )

    origin_after = run(
        [
            "git",
            "rev-parse",
            "origin/main",
        ]
    )

    print("Local HEAD  :", local_after)
    print("origin/main :", origin_after)
    print("remote main :", remote_main)

    if not (
        local_after
        == origin_after
        == remote_main
        == new_head
    ):
        raise RuntimeError(
            "Remote SHA verification failed."
        )

    print(
        "[PASS] Local HEAD == origin/main == remote main."
    )


    # ------------------------------------------------------------------
    # Exact remote tree
    # ------------------------------------------------------------------

    banner("STAGE27-1A :: REMOTE 9-FILE TREE VERIFICATION")

    remote_paths = sorted(
        line
        for line in run(
            [
                "git",
                "ls-tree",
                "-r",
                "--name-only",
                "origin/main",
                "--",
                str(
                    OUT_REL
                ),
            ]
        ).splitlines()
        if line.strip()
    )

    print("Expected remote files:", len(expected_paths))
    print("Actual remote files  :", len(remote_paths))

    if remote_paths != expected_paths:
        raise RuntimeError(
            "Remote Stage27-1A file universe mismatch.\n"
            f"expected={expected_paths}\n"
            f"actual={remote_paths}"
        )

    for path in remote_paths:
        print(" ", path)

    print()
    print(
        "[PASS] Remote tree contains exactly 9 Stage27-1A receipts."
    )


    # ------------------------------------------------------------------
    # Byte-exact remote verification
    # ------------------------------------------------------------------

    banner("STAGE27-1A :: REMOTE BYTE-LEVEL SHA256 VERIFICATION")

    for rel in expected_paths:

        local_path = (
            REPO
            / rel
        )

        local_sha = sha256_file(
            local_path
        )

        remote_bytes = run(
            [
                "git",
                "show",
                f"origin/main:{rel}",
            ],
            text=False,
        )

        remote_sha = sha256_bytes(
            remote_bytes
        )

        if local_sha != remote_sha:
            raise RuntimeError(
                f"Remote byte mismatch:\n{rel}\n"
                f"local ={local_sha}\n"
                f"remote={remote_sha}"
            )

        print(
            f"{local_sha}  {rel}"
        )

    print()
    print(
        "[PASS] All 9 remote receipts are byte-identical."
    )


    # ------------------------------------------------------------------
    # Remote freeze readback
    # ------------------------------------------------------------------

    banner("STAGE27-1A :: REMOTE MEMBERSHIP FREEZE READBACK")

    remote_freeze_bytes = run(
        [
            "git",
            "show",
            f"origin/main:{OUT_REL / FREEZE_FILE}",
        ],
        text=False,
    )

    remote_freeze = json.loads(
        remote_freeze_bytes.decode(
            "utf-8"
        )
    )

    if remote_freeze[
        "required_artifact_count"
    ] != 9:
        raise RuntimeError(
            "Remote membership artifact count != 9."
        )

    if len(
        remote_freeze[
            "artifact_hashes_before_freeze_record"
        ]
    ) != 8:
        raise RuntimeError(
            "Remote membership freeze should hash 8 non-self receipts."
        )

    if remote_freeze[
        "scientific_parent_commit"
    ] != EXPECTED_PARENT:
        raise RuntimeError(
            "Remote membership freeze scientific parent mismatch."
        )

    if remote_freeze[
        "heldout_exclusion_global_status"
    ] != "PASS_ALL_5_FOLDS":
        raise RuntimeError(
            "Remote held-out exclusion status mismatch."
        )

    for family in fold_names:

        if remote_freeze[
            "heldout_train_counts"
        ][family] != 0:
            raise RuntimeError(
                f"Remote {family} TRAIN exclusion failed."
            )

        if remote_freeze[
            "heldout_validation_counts"
        ][family] != 0:
            raise RuntimeError(
                f"Remote {family} VALIDATION exclusion failed."
            )

    remote_actions = remote_freeze[
        "scientific_actions_completed"
    ]

    for key in required_zero_actions:

        if int(
            remote_actions[
                key
            ]
        ) != 0:
            raise RuntimeError(
                f"Remote scientific counter {key} != 0"
            )

    print("Required artifacts        :", remote_freeze["required_artifact_count"])
    print("Non-self hashes           :", len(remote_freeze["artifact_hashes_before_freeze_record"]))
    print("Held-out exclusion        :", remote_freeze["heldout_exclusion_global_status"])
    print("Predictor columns read    :", remote_actions["predictor_columns_read"])
    print("Model fits                :", remote_actions["model_fits"])
    print("Target predictor openings :", remote_actions["target_predictor_openings"])

    print(
        "[PASS] Remote membership freeze semantics verified."
    )


    # ------------------------------------------------------------------
    # Final clean state
    # ------------------------------------------------------------------

    banner("STAGE27-1A :: FINAL CLOSURE")

    final_status = run(
        [
            "git",
            "status",
            "--porcelain=v1",
            "--untracked-files=all",
        ]
    )

    if final_status.strip():
        raise RuntimeError(
            "Repository is not clean after Stage27-1A push:\n"
            + final_status
        )

    print("Stage27-1A scientific parent :", EXPECTED_PARENT)
    print("Stage27-1A commit            :", new_head)
    print("Remote verification          : PASS")
    print("Membership receipts          : 9 / 9")
    print("Freeze non-self hashes       : 8 / 8")
    print("Git working tree             : CLEAN")

    print()
    print("Eligible fold memberships    : 5 / 5")
    print("Heldout TRAIN counts         : all zero")
    print("Heldout VALIDATION counts    : all zero")
    print("Web Attack U+0096 mapping    : PASS_EXACT")

    print()
    print("Predictor columns read       : 0 / 70")
    print("Feature matrices built       : 0")
    print("Model fits                   : 0")
    print("Inference                    : 0")
    print("Threshold selections         : 0")
    print("Target predictor openings    : 0")
    print("Bootstrap replicates         : 0")
    print("GPU hours                    : 0")

    print()
    print("=" * 118)
    print(
        "STAGE27-1A COMPLETE — MEMBERSHIP REMOTELY FROZEN"
    )
    print("=" * 118)

    print()
    print(
        "NEXT AUTHORIZED PHASE:"
    )

    print(
        "  Materialize the frozen Stage22-compatible 70-feature "
        "TRAIN and VALIDATION matrices from the exact source rows, "
        "verify hashes/counts/nonfinite policy, and only then begin "
        "the preregistered CPU model-fit sequence."
    )

    print()
    print(
        "TARGET PREDICTOR MATRICES MUST REMAIN SEALED UNTIL "
        "THE CORRESPONDING FOLD MODEL AND THRESHOLDS ARE FROZEN."
    )


finally:

    try:
        askpass.unlink(
            missing_ok=True
        )
    except Exception:
        pass

    token = None

    env.pop(
        "STAGE27_GITHUB_TOKEN",
        None,
    )



STAGE27-1A :: PRE-COMMIT SCIENTIFIC-PARENT GATE
Expected parent : 1ea6bedb141bc3fa6115edbc5296d0f7c6252559
Local HEAD      : 1ea6bedb141bc3fa6115edbc5296d0f7c6252559
origin/main     : 1ea6bedb141bc3fa6115edbc5296d0f7c6252559
[PASS] Scientific parent unchanged.

STAGE27-1A :: EXACT STAGED UNIVERSE
Expected staged files: 9
Actual staged files  : 9
[PASS] Exactly 9 Stage27-1A membership receipts staged; no unrelated changes.

STAGE27-1A :: MEMBERSHIP FREEZE RECORD GATE
required_artifact_count   : 9
non-self artifact hashes  : 8
heldout exclusion status  : PASS_ALL_5_FOLDS
[PASS] Membership freeze hashes other 8 receipts exactly.
[PASS] Held-out TRAIN/VALIDATION counts are zero for all 5 folds.
[PASS] Predictor/model/target-opening counters remain zero.

STAGE27-1A :: FOLD RECEIPT READBACK

BOT
  TRAIN      : 1,668,519 rows | 1,402,023 benign | 266,496 attack
  VALIDATION : 458,968 rows | 456,752 benign | 2,216 attack
  PRIMARY    : 416,288 rows | 414,322 benign | 1,966 held-out attack
  

In [20]:
# ======================================================================================
# STAGE27-1B — PREFIT 70-FEATURE MATERIALIZATION (MONDAY–WEDNESDAY ONLY)
#
# Scientific parent:
#   c5ad77b346f5d98e41ca5eccd4336087f8c1786e
#
# PURPOSE
# -------
# Materialize the exact frozen Stage22-compatible 70-feature population needed
# for ALL five TRAIN sets plus the two Thursday-target VALIDATION sets, while
# keeping Thursday and Friday predictor rows unmaterialized.
#
# This stage reads predictor columns ONLY from:
#   Monday
#   Tuesday
#   Wednesday
#
# It persists only the known binary population:
#   BENIGN + seven-primary-family attacks
#
# It EXCLUDES:
#   Heartbleed / TARGET_ONLY_UNSEEN (11 Wednesday rows)
#   OTHER_ATTACK_UNSEEN_LABEL
#
# One shared chronological cache is built:
#
#   [ Monday + Tuesday ] [ Wednesday excluding Heartbleed ]
#      975,827 rows          692,692 rows
#
# Total:
#   1,668,519 rows
#
# Therefore:
#   INFILTRATION / WEB_ATTACK:
#       TRAIN      = prefix [0 : 975,827)
#       VALIDATION = suffix [975,827 : 1,668,519)
#
#   BOT / DDOS / PORT_SCAN:
#       TRAIN      = full [0 : 1,668,519)
#
# No Thursday predictors are read in this stage.
# No Friday predictors are read in this stage.
#
# ZERO MODEL FITS
# ZERO INFERENCE
# ZERO THRESHOLD SELECTION
# ZERO TARGET INFERENCE OPENINGS
# ZERO BOOTSTRAP
# ZERO GPU
#
# Runtime cache OUTSIDE Git:
#   /kaggle/working/stage27_1b_prefit_feature_cache
#
# Durable receipts INSIDE Git:
#   results/stage27_loao_unseen_attack/stage27_1b_prefit_feature_materialization/
#
# Exactly 4 JSON receipts are staged, but NOT committed/pushed.
# ======================================================================================

from __future__ import annotations

import hashlib
import json
import os
import shutil
import subprocess
from collections import OrderedDict, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List

import numpy as np
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq


# ======================================================================================
# 0. CONSTANTS
# ======================================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_PARENT = (
    "c5ad77b346f5d98e41ca5eccd4336087f8c1786e"
)

SOURCE_ROOT = Path(
    "/kaggle/working/stage27_cicids2017_sources"
).resolve()

MEMBERSHIP_CACHE_ROOT = Path(
    "/kaggle/working/stage27_1a_membership_cache"
).resolve()

CACHE_ROOT = Path(
    "/kaggle/working/stage27_1b_prefit_feature_cache"
).resolve()

STAGE27_ROOT_REL = Path(
    "results/stage27_loao_unseen_attack"
)

STAGE27_0_REL = (
    STAGE27_ROOT_REL
    / "stage27_0_protocol_lock"
)

STAGE27_1A_REL = (
    STAGE27_ROOT_REL
    / "stage27_1a_fold_membership"
)

OUT_REL = (
    STAGE27_ROOT_REL
    / "stage27_1b_prefit_feature_materialization"
)

OUT_DIR = (
    REPO
    / OUT_REL
)

FEATURE_SPEC_PATH = (
    REPO
    / STAGE27_0_REL
    / "feature_representation.json"
)

MODEL_INVENTORY_PATH = (
    REPO
    / STAGE27_0_REL
    / "model_inventory.json"
)

COMPUTE_POLICY_PATH = (
    REPO
    / STAGE27_0_REL
    / "compute_policy.json"
)

STAGE27_0_FREEZE_PATH = (
    REPO
    / STAGE27_0_REL
    / "freeze_record.json"
)

MEMBERSHIP_FREEZE_PATH = (
    REPO
    / STAGE27_1A_REL
    / "stage27_1a_membership_freeze_record.json"
)

SOURCE_RECEIPT_PATH = (
    REPO
    / STAGE27_1A_REL
    / "source_effective_population_receipt.json"
)

FAMILY_CACHE_RECEIPT_PATH = (
    REPO
    / STAGE27_1A_REL
    / "family_code_cache_receipt.json"
)

FOLD_RECEIPT_PATHS = {
    family: (
        REPO
        / STAGE27_1A_REL
        / f"fold_{family}_membership_receipt.json"
    )
    for family in [
        "BOT",
        "DDOS",
        "INFILTRATION",
        "PORT_SCAN",
        "WEB_ATTACK",
    ]
}

THURSDAY_TARGET_FOLDS = [
    "INFILTRATION",
    "WEB_ATTACK",
]

FRIDAY_TARGET_FOLDS = [
    "BOT",
    "DDOS",
    "PORT_SCAN",
]

EXPECTED_MON_TUE_ROWS = 975_827
EXPECTED_WED_KNOWN_ROWS = 692_692
EXPECTED_MON_WED_KNOWN_ROWS = 1_668_519
EXPECTED_HEARTBLEED_EXCLUDED = 11

EXPECTED_FEATURE_COUNT = 70
EXPECTED_MODEL_DTYPE = np.dtype("float32")
EXPECTED_PARSE_DTYPE = np.dtype("float64")

REQUIRED_OUTPUT_FILES = [
    "prefit_source_feature_receipt.json",
    "monwed_known_feature_cache_receipt.json",
    "fold_feature_binding_receipt.json",
    "stage27_1b_prefit_feature_freeze_record.json",
]

os.environ["CUDA_VISIBLE_DEVICES"] = ""


# ======================================================================================
# 1. HELPERS
# ======================================================================================

def banner(title: str) -> None:
    print()
    print("=" * 118)
    print(title)
    print("=" * 118)


def run(
    cmd: List[str],
    *,
    cwd: Path = REPO,
    check: bool = True,
) -> str:

    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}): {' '.join(map(str, cmd))}\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return p.stdout.strip()


def now_utc() -> str:
    return datetime.now(timezone.utc).isoformat()


def load_json(path: Path) -> Any:
    with path.open(
        "r",
        encoding="utf-8",
    ) as f:
        return json.load(f)


def write_json(
    path: Path,
    obj: Any,
) -> None:

    path.write_text(
        json.dumps(
            obj,
            ensure_ascii=False,
            indent=2,
            sort_keys=False,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )


def sha256_file(
    path: Path,
    chunk_size: int = 16 * 1024 * 1024,
) -> str:

    h = hashlib.sha256()

    with path.open("rb") as f:
        while True:
            block = f.read(
                chunk_size
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def sha256_array_content(
    arr: np.ndarray,
    block_rows: int = 131_072,
) -> str:

    h = hashlib.sha256()

    if arr.ndim == 1:
        for start in range(
            0,
            arr.shape[0],
            block_rows,
        ):
            stop = min(
                start + block_rows,
                arr.shape[0],
            )

            block = np.ascontiguousarray(
                arr[start:stop]
            )

            h.update(
                block.tobytes(
                    order="C"
                )
            )

    elif arr.ndim == 2:
        for start in range(
            0,
            arr.shape[0],
            block_rows,
        ):
            stop = min(
                start + block_rows,
                arr.shape[0],
            )

            block = np.ascontiguousarray(
                arr[start:stop, :]
            )

            h.update(
                block.tobytes(
                    order="C"
                )
            )

    else:
        raise RuntimeError(
            f"Unsupported ndim for content hashing: {arr.ndim}"
        )

    return h.hexdigest()


def npy_receipt(
    path: Path,
    *,
    content_arr: np.ndarray | None = None,
) -> Dict[str, Any]:

    arr = np.load(
        path,
        allow_pickle=False,
        mmap_mode="r",
    )

    receipt = {
        "path": str(path),
        "shape": list(
            arr.shape
        ),
        "dtype": str(
            arr.dtype
        ),
        "npy_bytes": int(
            path.stat().st_size
        ),
        "npy_sha256": sha256_file(
            path
        ),
    }

    if content_arr is None:
        content_arr = arr

    receipt[
        "content_sha256"
    ] = sha256_array_content(
        content_arr
    )

    return receipt


def clean_exact_cache(
    path: Path,
) -> None:

    expected = Path(
        "/kaggle/working/stage27_1b_prefit_feature_cache"
    ).resolve()

    if path.resolve() != expected:
        raise RuntimeError(
            f"Refusing to clean unexpected path: {path}"
        )

    if path.exists():
        shutil.rmtree(
            path
        )

    path.mkdir(
        parents=True,
        exist_ok=False,
    )


def cast_numeric_float64(
    arr: pa.Array,
    *,
    source_column: str,
    source_file: str,
) -> np.ndarray:

    try:
        casted = pc.cast(
            arr,
            pa.float64(),
            safe=False,
        )

    except Exception as exc:
        raise RuntimeError(
            f"Numeric parse failed for column {source_column!r} "
            f"in source {source_file!r}. "
            "Frozen policy requires hard failure for nonnumeric tokens."
        ) from exc

    try:
        values = casted.to_numpy(
            zero_copy_only=False
        )

    except Exception as exc:
        raise RuntimeError(
            f"Could not convert Arrow float64 column {source_column!r} "
            f"to NumPy."
        ) from exc

    values = np.asarray(
        values,
        dtype=np.float64,
    )

    return values


# ======================================================================================
# 2. SCIENTIFIC PARENT / CLEAN GATE
# ======================================================================================

banner("STAGE27-1B :: SCIENTIFIC-PARENT GATE")

head = run(
    [
        "git",
        "rev-parse",
        "HEAD",
    ]
)

origin = run(
    [
        "git",
        "rev-parse",
        "origin/main",
    ]
)

status = run(
    [
        "git",
        "status",
        "--porcelain=v1",
        "--untracked-files=all",
    ]
)

print("Expected parent :", EXPECTED_PARENT)
print("Local HEAD      :", head)
print("origin/main     :", origin)
print("Git clean       :", not bool(status.strip()))

if head != EXPECTED_PARENT:
    raise RuntimeError(
        "Local HEAD is not the frozen Stage27-1A commit."
    )

if origin != EXPECTED_PARENT:
    raise RuntimeError(
        "origin/main is not the frozen Stage27-1A commit."
    )

if status.strip():
    raise RuntimeError(
        "Repository is dirty before Stage27-1B:\n"
        + status
    )

print("[PASS] Stage27-1A is the clean scientific parent.")


# ======================================================================================
# 3. OUTPUT / RUNTIME GATES
# ======================================================================================

banner("STAGE27-1B :: OUTPUT / RUNTIME GATES")

if OUT_DIR.exists():
    raise RuntimeError(
        f"Stage27-1B durable output already exists:\n{OUT_DIR}\n"
        "Refusing overwrite."
    )

if not SOURCE_ROOT.is_dir():
    raise RuntimeError(
        f"Exact CICIDS2017 source cache missing:\n{SOURCE_ROOT}"
    )

if not MEMBERSHIP_CACHE_ROOT.is_dir():
    raise RuntimeError(
        f"Stage27-1A membership cache missing:\n{MEMBERSHIP_CACHE_ROOT}\n"
        "Rebuild Stage27-1A runtime membership cache before continuing."
    )

clean_exact_cache(
    CACHE_ROOT
)

print("Exact source cache    :", SOURCE_ROOT)
print("Membership cache      :", MEMBERSHIP_CACHE_ROOT)
print("New feature cache     :", CACHE_ROOT)
print("Durable output        :", OUT_DIR)
print("Thursday predictors   : SEALED / NOT READ")
print("Friday predictors     : SEALED / NOT READ")


# ======================================================================================
# 4. LOAD FROZEN RECEIPTS / PROTOCOL
# ======================================================================================

banner("STAGE27-1B :: LOAD FROZEN PROTOCOL + MEMBERSHIP RECEIPTS")

required_paths = [
    FEATURE_SPEC_PATH,
    MODEL_INVENTORY_PATH,
    COMPUTE_POLICY_PATH,
    STAGE27_0_FREEZE_PATH,
    MEMBERSHIP_FREEZE_PATH,
    SOURCE_RECEIPT_PATH,
    FAMILY_CACHE_RECEIPT_PATH,
    *FOLD_RECEIPT_PATHS.values(),
]

for path in required_paths:
    if not path.is_file():
        raise RuntimeError(
            f"Required frozen artifact missing: {path}"
        )

feature_spec = load_json(
    FEATURE_SPEC_PATH
)

model_inventory = load_json(
    MODEL_INVENTORY_PATH
)

compute_policy = load_json(
    COMPUTE_POLICY_PATH
)

stage27_0_freeze = load_json(
    STAGE27_0_FREEZE_PATH
)

membership_freeze = load_json(
    MEMBERSHIP_FREEZE_PATH
)

source_receipt = load_json(
    SOURCE_RECEIPT_PATH
)

family_cache_receipt = load_json(
    FAMILY_CACHE_RECEIPT_PATH
)

fold_receipts = {
    family: load_json(
        path
    )
    for family, path in FOLD_RECEIPT_PATHS.items()
}

if stage27_0_freeze[
    "authorized_fit_budget"
] != 10:
    raise RuntimeError(
        "Frozen fit budget changed."
    )

if membership_freeze[
    "heldout_exclusion_global_status"
] != "PASS_ALL_5_FOLDS":
    raise RuntimeError(
        "Stage27-1A held-out exclusion is not frozen PASS."
    )

actions = membership_freeze[
    "scientific_actions_completed"
]

for key in [
    "predictor_columns_read",
    "feature_rows_materialized",
    "model_fits",
    "model_inference",
    "threshold_selections",
    "target_predictor_openings",
    "bootstrap_replicates",
    "gpu_hours",
]:
    if int(
        actions[
            key
        ]
    ) != 0:
        raise RuntimeError(
            f"Stage27-1A boundary counter {key} is nonzero."
        )

if compute_policy[
    "primary_execution_device"
] != "CPU":
    raise RuntimeError(
        "Frozen Stage27 compute device is not CPU."
    )

if int(
    compute_policy[
        "primary_gpu_budget_hours"
    ]
) != 0:
    raise RuntimeError(
        "Frozen GPU budget is nonzero."
    )

if feature_spec[
    "feature_count"
] != EXPECTED_FEATURE_COUNT:
    raise RuntimeError(
        "Frozen feature count changed."
    )

print("Feature count          :", feature_spec["feature_count"])
print("Authorized future fits :", stage27_0_freeze["authorized_fit_budget"])
print("Membership exclusion   :", membership_freeze["heldout_exclusion_global_status"])
print("Primary device         :", compute_policy["primary_execution_device"])
print("[PASS] Frozen protocol + membership receipts loaded.")


# ======================================================================================
# 5. VERIFY STAGE27-1A RUNTIME CODE ARRAYS
# ======================================================================================

banner("STAGE27-1B :: VERIFY MEMBERSHIP RUNTIME CACHE")

global_arrays = family_cache_receipt[
    "arrays"
]

family_codes_path = Path(
    global_arrays[
        "global_family_codes"
    ][
        "path"
    ]
)

day_codes_path = Path(
    global_arrays[
        "global_day_codes"
    ][
        "path"
    ]
)

source_codes_path = Path(
    global_arrays[
        "global_source_codes"
    ][
        "path"
    ]
)

for name, path, receipt_key in [
    (
        "global_family_codes",
        family_codes_path,
        "global_family_codes",
    ),
    (
        "global_day_codes",
        day_codes_path,
        "global_day_codes",
    ),
    (
        "global_source_codes",
        source_codes_path,
        "global_source_codes",
    ),
]:

    if not path.is_file():
        raise RuntimeError(
            f"Membership runtime array missing: {path}"
        )

    actual_npy_sha = sha256_file(
        path
    )

    expected_npy_sha = global_arrays[
        receipt_key
    ][
        "npy_sha256"
    ]

    if actual_npy_sha != expected_npy_sha:
        raise RuntimeError(
            f"{name}: runtime NPY SHA mismatch."
        )

family_codes = np.load(
    family_codes_path,
    allow_pickle=False,
    mmap_mode="r",
)

day_codes = np.load(
    day_codes_path,
    allow_pickle=False,
    mmap_mode="r",
)

source_codes = np.load(
    source_codes_path,
    allow_pickle=False,
    mmap_mode="r",
)

if family_codes.shape != (
    2_830_743,
):
    raise RuntimeError(
        "Unexpected family-code population shape."
    )

if day_codes.shape != family_codes.shape:
    raise RuntimeError(
        "day-code population shape mismatch."
    )

if source_codes.shape != family_codes.shape:
    raise RuntimeError(
        "source-code population shape mismatch."
    )

print("Family code rows :", f"{family_codes.shape[0]:,}")
print("Day code rows    :", f"{day_codes.shape[0]:,}")
print("Source code rows :", f"{source_codes.shape[0]:,}")
print("[PASS] Stage27-1A global membership cache verified.")


# ======================================================================================
# 6. VERIFY FOLD MEMBERSHIP ARRAY IDENTITIES
# ======================================================================================

banner("STAGE27-1B :: VERIFY SHARED FOLD ROLE IDENTITIES")

def load_role_array(
    family: str,
    role_name: str,
) -> np.ndarray:

    role = fold_receipts[
        family
    ][
        "runtime_membership_arrays"
    ][
        role_name
    ]

    path = Path(
        role[
            "path"
        ]
    )

    if not path.is_file():
        raise RuntimeError(
            f"{family}/{role_name}: runtime membership file missing."
        )

    actual_npy_sha = sha256_file(
        path
    )

    if actual_npy_sha != role[
        "npy_sha256"
    ]:
        raise RuntimeError(
            f"{family}/{role_name}: NPY SHA mismatch."
        )

    arr = np.load(
        path,
        allow_pickle=False,
        mmap_mode="r",
    )

    return arr


# Thursday-target folds must share TRAIN and VALIDATION membership.
inf_train = load_role_array(
    "INFILTRATION",
    "train_global_idx",
)

web_train = load_role_array(
    "WEB_ATTACK",
    "train_global_idx",
)

inf_valid = load_role_array(
    "INFILTRATION",
    "validation_global_idx",
)

web_valid = load_role_array(
    "WEB_ATTACK",
    "validation_global_idx",
)

if not np.array_equal(
    inf_train,
    web_train,
):
    raise RuntimeError(
        "INFILTRATION and WEB_ATTACK TRAIN memberships differ."
    )

if not np.array_equal(
    inf_valid,
    web_valid,
):
    raise RuntimeError(
        "INFILTRATION and WEB_ATTACK VALIDATION memberships differ."
    )


# Friday-target folds must share TRAIN membership.
bot_train = load_role_array(
    "BOT",
    "train_global_idx",
)

ddos_train = load_role_array(
    "DDOS",
    "train_global_idx",
)

port_train = load_role_array(
    "PORT_SCAN",
    "train_global_idx",
)

if not np.array_equal(
    bot_train,
    ddos_train,
):
    raise RuntimeError(
        "BOT and DDOS TRAIN memberships differ."
    )

if not np.array_equal(
    bot_train,
    port_train,
):
    raise RuntimeError(
        "BOT and PORT_SCAN TRAIN memberships differ."
    )


if int(
    inf_train.size
) != EXPECTED_MON_TUE_ROWS:
    raise RuntimeError(
        "Thursday-target TRAIN row count mismatch."
    )

if int(
    inf_valid.size
) != EXPECTED_WED_KNOWN_ROWS:
    raise RuntimeError(
        "Thursday-target VALIDATION row count mismatch."
    )

if int(
    bot_train.size
) != EXPECTED_MON_WED_KNOWN_ROWS:
    raise RuntimeError(
        "Friday-target TRAIN row count mismatch."
    )


# Strong structural relationship:
# Friday-target TRAIN must be exact concatenation of:
# Thursday-target TRAIN + Thursday-target VALIDATION.
concat_expected = np.concatenate(
    [
        np.asarray(
            inf_train,
            dtype=np.int32,
        ),
        np.asarray(
            inf_valid,
            dtype=np.int32,
        ),
    ]
)

if not np.array_equal(
    concat_expected,
    bot_train,
):
    raise RuntimeError(
        "Friday-target TRAIN is not exactly "
        "MonTue TRAIN + Wednesday VALIDATION."
    )

if int(
    inf_train.max()
) >= int(
    inf_valid.min()
):
    raise RuntimeError(
        "MonTue/Wed chronology boundary failed."
    )

if int(
    bot_train.max()
) >= 1_668_530:
    raise RuntimeError(
        "Friday-target TRAIN unexpectedly extends beyond Wednesday."
    )

print("Mon-Tue known rows      :", f"{inf_train.size:,}")
print("Wednesday known rows    :", f"{inf_valid.size:,}")
print("Mon-Wed known rows      :", f"{bot_train.size:,}")
print("Excluded Wed Heartbleed :", f"{1_668_530 - bot_train.size:,}")

if (
    1_668_530
    - bot_train.size
) != EXPECTED_HEARTBLEED_EXCLUDED:
    raise RuntimeError(
        "Expected exactly 11 Wednesday rows excluded from known population."
    )

print("[PASS] Shared fold role identities and chronology verified.")


# ======================================================================================
# 7. FROZEN FEATURE ORDER + SOURCE ADAPTER
# ======================================================================================

banner("STAGE27-1B :: FEATURE ADAPTER GATE")

feature_order = list(
    feature_spec[
        "feature_order"
    ]
)

if len(
    feature_order
) != EXPECTED_FEATURE_COUNT:
    raise RuntimeError(
        "Feature-order length mismatch."
    )

adapter = feature_spec[
    "cicids2017_semantic_adapter"
]

if adapter[
    "variant"
] != "FLAG_CORRECTED_mapping":
    raise RuntimeError(
        "Unexpected CICIDS2017 semantic adapter variant."
    )

mapping = adapter[
    "mapping"
]

if list(
    mapping.keys()
) != feature_order:
    raise RuntimeError(
        "Adapter key order does not exactly match frozen feature order."
    )

unique_source_columns = []

source_to_positions = defaultdict(
    list
)

for model_position, model_feature in enumerate(
    feature_order
):

    source_column = mapping[
        model_feature
    ]

    source_to_positions[
        source_column
    ].append(
        model_position
    )

    if source_column not in unique_source_columns:
        unique_source_columns.append(
            source_column
        )

print("Frozen model features :", len(feature_order))
print("Unique source columns :", len(unique_source_columns))
print("Adapter SHA256        :", adapter["mapping_sha256"])
print("[PASS] Exact 70-feature Stage27 adapter recovered.")


# ======================================================================================
# 8. SOURCE SEGMENTS — AUTHORIZE MONDAY/TUESDAY/WEDNESDAY ONLY
# ======================================================================================

banner("STAGE27-1B :: AUTHORIZED SOURCE SEGMENTS")

segments = source_receipt[
    "segments"
]

authorized_segments = [
    segment
    for segment in segments
    if segment[
        "day"
    ] in {
        "Monday",
        "Tuesday",
        "Wednesday",
    }
]

forbidden_segments = [
    segment
    for segment in segments
    if segment[
        "day"
    ] in {
        "Thursday",
        "Friday",
    }
]

if len(
    authorized_segments
) != 3:
    raise RuntimeError(
        "Expected exactly 3 authorized Mon-Wed source segments."
    )

if len(
    forbidden_segments
) != 5:
    raise RuntimeError(
        "Expected exactly 5 sealed Thu/Fri source segments."
    )

for segment in authorized_segments:

    path = Path(
        segment[
            "path"
        ]
    )

    if not path.is_file():
        raise RuntimeError(
            f"Authorized source file missing: {path}"
        )

    if sha256_file(
        path
    ) != segment[
        "sha256"
    ]:
        raise RuntimeError(
            f"Source SHA mismatch: {path.name}"
        )

    parquet = pq.ParquetFile(
        path
    )

    schema_names = set(
        parquet.schema_arrow.names
    )

    missing_columns = [
        source_column
        for source_column in unique_source_columns
        if source_column not in schema_names
    ]

    if missing_columns:
        raise RuntimeError(
            f"{path.name}: missing frozen feature columns:\n"
            + "\n".join(
                missing_columns
            )
        )

    print(
        f"AUTHORIZED  {segment['day']:10s} "
        f"{path.name}"
    )

for segment in forbidden_segments:

    print(
        f"SEALED      {segment['day']:10s} "
        f"{segment['basename']}"
    )

print()
print(
    "[PASS] Predictor materialization is hard-scoped to Monday–Wednesday only."
)


# ======================================================================================
# 9. BUILD THE MONDAY–WEDNESDAY KNOWN-POPULATION GLOBAL ROW LIST
# ======================================================================================

banner("STAGE27-1B :: BUILD PREFIT GLOBAL ROW LIST")

selected_global_idx = np.asarray(
    bot_train,
    dtype=np.int32,
)

if selected_global_idx.size != EXPECTED_MON_WED_KNOWN_ROWS:
    raise RuntimeError(
        "Selected Mon-Wed row count mismatch."
    )

if not np.all(
    selected_global_idx[
        1:
    ] > selected_global_idx[
        :-1
    ]
):
    raise RuntimeError(
        "Selected global row IDs are not strictly increasing."
    )

if int(
    selected_global_idx[
        0
    ]
) != 0:
    raise RuntimeError(
        "Mon-Wed known population should begin at global row 0."
    )

if int(
    selected_global_idx[
        -1
    ]
) >= 1_668_530:
    raise RuntimeError(
        "Selected known population extends beyond Wednesday."
    )

selected_family_codes = np.asarray(
    family_codes[
        selected_global_idx
    ],
    dtype=np.uint8,
)

if np.any(
    selected_family_codes == 8
):
    raise RuntimeError(
        "TARGET_ONLY_UNSEEN / Heartbleed leaked into prefit known population."
    )

if np.any(
    selected_family_codes == 9
):
    raise RuntimeError(
        "OTHER_ATTACK_UNSEEN_LABEL leaked into prefit known population."
    )

y_binary = (
    selected_family_codes
    != 0
).astype(
    np.uint8,
    copy=False,
)

expected_benign = int(
    np.sum(
        selected_family_codes == 0
    )
)

expected_attack = int(
    np.sum(
        selected_family_codes != 0
    )
)

if expected_benign != 1_402_023:
    raise RuntimeError(
        f"Mon-Wed benign mismatch: {expected_benign:,}"
    )

if expected_attack != 266_496:
    raise RuntimeError(
        f"Mon-Wed attack mismatch: {expected_attack:,}"
    )

print("Selected known rows :", f"{selected_global_idx.size:,}")
print("Benign              :", f"{expected_benign:,}")
print("Attack              :", f"{expected_attack:,}")
print("Heartbleed           : 0")
print("Unknown nonbenign    : 0")
print("[PASS] Prefit known population identity frozen.")


# ======================================================================================
# 10. ALLOCATE PREFIT RUNTIME ARTIFACTS
# ======================================================================================

banner("STAGE27-1B :: ALLOCATE FLOAT32 PREFIT CACHE")

feature_path = (
    CACHE_ROOT
    / "monwed_known_features_float32.npy"
)

label_path = (
    CACHE_ROOT
    / "monwed_known_binary_labels_uint8.npy"
)

global_idx_path = (
    CACHE_ROOT
    / "monwed_known_global_idx_int32.npy"
)

X = np.lib.format.open_memmap(
    feature_path,
    mode="w+",
    dtype=np.float32,
    shape=(
        EXPECTED_MON_WED_KNOWN_ROWS,
        EXPECTED_FEATURE_COUNT,
    ),
)

np.save(
    label_path,
    y_binary,
    allow_pickle=False,
)

np.save(
    global_idx_path,
    selected_global_idx,
    allow_pickle=False,
)

print("Feature matrix :", feature_path)
print("Shape          :", X.shape)
print("Dtype          :", X.dtype)
print("Approx payload :", f"{X.nbytes / (1024**3):.3f} GiB")
print("[PASS] Runtime artifacts allocated outside Git.")


# ======================================================================================
# 11. MATERIALIZE FEATURES FROM MONDAY/TUESDAY/WEDNESDAY ONLY
# ======================================================================================

banner("STAGE27-1B :: MATERIALIZE 70 FEATURES")

nonfinite_global = {
    "positive_inf_to_nan": 0,
    "negative_inf_to_nan": 0,
    "nan_cells_before_inf_conversion": 0,
    "nan_cells_after_inf_conversion_float64": 0,
    "float32_inf_after_conversion": 0,
    "float32_nan_cells": 0,
}

per_source_numeric = []

out_cursor = 0
source_access_log = []

for segment in authorized_segments:

    path = Path(
        segment[
            "path"
        ]
    )

    seg_start = int(
        segment[
            "global_start_zero_based"
        ]
    )

    seg_stop = int(
        segment[
            "global_stop_exclusive"
        ]
    )

    left = int(
        np.searchsorted(
            selected_global_idx,
            seg_start,
            side="left",
        )
    )

    right = int(
        np.searchsorted(
            selected_global_idx,
            seg_stop,
            side="left",
        )
    )

    seg_selected_global = selected_global_idx[
        left:right
    ]

    local_selected = (
        seg_selected_global.astype(
            np.int64
        )
        - seg_start
    )

    expected_selected_rows = int(
        local_selected.size
    )

    print()
    print(
        f"{segment['day']} :: {path.name}"
    )

    print(
        "  effective source rows :",
        f"{segment['effective_rows']:,}",
    )

    print(
        "  selected model rows   :",
        f"{expected_selected_rows:,}",
    )

    parquet = pq.ParquetFile(
        path
    )

    local_row_cursor = 0
    selected_cursor = 0
    source_written = 0

    source_stats = {
        "day": segment[
            "day"
        ],
        "basename": path.name,
        "sha256": segment[
            "sha256"
        ],
        "effective_rows": int(
            segment[
                "effective_rows"
            ]
        ),
        "selected_rows": expected_selected_rows,
        "positive_inf_to_nan": 0,
        "negative_inf_to_nan": 0,
        "nan_cells_before_inf_conversion": 0,
        "nan_cells_after_inf_conversion_float64": 0,
        "float32_inf_after_conversion": 0,
        "float32_nan_cells": 0,
        "feature_columns_requested": len(
            unique_source_columns
        ),
    }

    for batch in parquet.iter_batches(
        batch_size=65_536,
        columns=unique_source_columns,
        use_threads=True,
    ):

        batch_len = len(
            batch
        )

        batch_start = local_row_cursor
        batch_stop = (
            batch_start
            + batch_len
        )

        sel_left = int(
            np.searchsorted(
                local_selected,
                batch_start,
                side="left",
            )
        )

        sel_right = int(
            np.searchsorted(
                local_selected,
                batch_stop,
                side="left",
            )
        )

        if sel_right > sel_left:

            selected_local_rows = local_selected[
                sel_left:sel_right
            ]

            relative_rows = (
                selected_local_rows
                - batch_start
            ).astype(
                np.int64,
                copy=False,
            )

            relative_arrow = pa.array(
                relative_rows,
                type=pa.int64(),
            )

            n_selected_batch = int(
                relative_rows.size
            )

            batch_matrix64 = np.empty(
                (
                    n_selected_batch,
                    EXPECTED_FEATURE_COUNT,
                ),
                dtype=np.float64,
            )

            name_to_array = {
                name: batch.column(
                    batch.schema.get_field_index(
                        name
                    )
                )
                for name in unique_source_columns
            }

            for source_column in unique_source_columns:

                selected_arrow = pc.take(
                    name_to_array[
                        source_column
                    ],
                    relative_arrow,
                )

                values64 = cast_numeric_float64(
                    selected_arrow,
                    source_column=source_column,
                    source_file=path.name,
                )

                pos_inf = int(
                    np.isposinf(
                        values64
                    ).sum()
                )

                neg_inf = int(
                    np.isneginf(
                        values64
                    ).sum()
                )

                nan_before = int(
                    np.isnan(
                        values64
                    ).sum()
                )

                if pos_inf or neg_inf:

                    values64 = values64.copy()

                    values64[
                        np.isinf(
                            values64
                        )
                    ] = np.nan

                nan_after = int(
                    np.isnan(
                        values64
                    ).sum()
                )

                source_stats[
                    "positive_inf_to_nan"
                ] += pos_inf

                source_stats[
                    "negative_inf_to_nan"
                ] += neg_inf

                source_stats[
                    "nan_cells_before_inf_conversion"
                ] += nan_before

                source_stats[
                    "nan_cells_after_inf_conversion_float64"
                ] += nan_after

                for model_position in source_to_positions[
                    source_column
                ]:
                    batch_matrix64[
                        :,
                        model_position
                    ] = values64

            batch_matrix32 = batch_matrix64.astype(
                np.float32,
                copy=False,
            )

            float32_inf = int(
                np.isinf(
                    batch_matrix32
                ).sum()
            )

            float32_nan = int(
                np.isnan(
                    batch_matrix32
                ).sum()
            )

            source_stats[
                "float32_inf_after_conversion"
            ] += float32_inf

            source_stats[
                "float32_nan_cells"
            ] += float32_nan

            if float32_inf != 0:
                raise RuntimeError(
                    f"{path.name}: float64->float32 conversion "
                    f"created {float32_inf:,} infinite cells."
                )

            out_start = out_cursor
            out_stop = (
                out_start
                + n_selected_batch
            )

            X[
                out_start:out_stop,
                :
            ] = batch_matrix32

            expected_global_batch = selected_global_idx[
                out_start:out_stop
            ]

            actual_global_batch = (
                seg_start
                + selected_local_rows
            ).astype(
                np.int32,
                copy=False,
            )

            if not np.array_equal(
                expected_global_batch,
                actual_global_batch,
            ):
                raise RuntimeError(
                    f"{path.name}: global row alignment failure."
                )

            out_cursor = out_stop
            source_written += n_selected_batch

        local_row_cursor = batch_stop

    if local_row_cursor < int(
        segment[
            "effective_rows"
        ]
    ):
        raise RuntimeError(
            f"{path.name}: failed to scan full authorized effective segment."
        )

    if source_written != expected_selected_rows:
        raise RuntimeError(
            f"{path.name}: selected-write count mismatch. "
            f"expected={expected_selected_rows:,}, actual={source_written:,}"
        )

    for key in nonfinite_global:
        nonfinite_global[
            key
        ] += int(
            source_stats[
                key
            ]
        )

    source_access_log.append(
        {
            "day": segment[
                "day"
            ],
            "basename": path.name,
            "predictor_columns_read": len(
                unique_source_columns
            ),
            "selected_rows_materialized": source_written,
        }
    )

    per_source_numeric.append(
        source_stats
    )

    print(
        "  written rows            :",
        f"{source_written:,}",
    )

    print(
        "  +inf -> NaN             :",
        f"{source_stats['positive_inf_to_nan']:,}",
    )

    print(
        "  -inf -> NaN             :",
        f"{source_stats['negative_inf_to_nan']:,}",
    )

    print(
        "  final float32 NaN cells :",
        f"{source_stats['float32_nan_cells']:,}",
    )

    print(
        "  final float32 inf cells :",
        source_stats[
            "float32_inf_after_conversion"
        ],
    )


if out_cursor != EXPECTED_MON_WED_KNOWN_ROWS:
    raise RuntimeError(
        f"Global feature-write count mismatch. "
        f"expected={EXPECTED_MON_WED_KNOWN_ROWS:,}, actual={out_cursor:,}"
    )

X.flush()

del X

print()
print(
    "[PASS] Exact 70-feature Mon-Wed known population materialized."
)


# ======================================================================================
# 12. SEALED-DAY ACCESS AUDIT
# ======================================================================================

banner("STAGE27-1B :: SEALED-DAY ACCESS AUDIT")

accessed_days = [
    item[
        "day"
    ]
    for item in source_access_log
]

if accessed_days != [
    "Monday",
    "Tuesday",
    "Wednesday",
]:
    raise RuntimeError(
        f"Unexpected predictor source access days: {accessed_days}"
    )

print(
    "Predictor-source days read :",
    accessed_days,
)

print(
    "Thursday predictor sources : 0 files read"
)

print(
    "Friday predictor sources   : 0 files read"
)

print(
    "Target inference openings  : 0"
)

print(
    "[PASS] Thursday and Friday predictors remain sealed."
)


# ======================================================================================
# 13. RE-READ FEATURE CACHE + NUMERIC / LABEL / ROW-IDENTITY GATES
# ======================================================================================

banner("STAGE27-1B :: PREFIT CACHE RE-READ")

X_read = np.load(
    feature_path,
    allow_pickle=False,
    mmap_mode="r",
)

y_read = np.load(
    label_path,
    allow_pickle=False,
    mmap_mode="r",
)

idx_read = np.load(
    global_idx_path,
    allow_pickle=False,
    mmap_mode="r",
)

if X_read.shape != (
    EXPECTED_MON_WED_KNOWN_ROWS,
    EXPECTED_FEATURE_COUNT,
):
    raise RuntimeError(
        "Feature matrix shape mismatch after reload."
    )

if X_read.dtype != EXPECTED_MODEL_DTYPE:
    raise RuntimeError(
        f"Feature matrix dtype mismatch: {X_read.dtype}"
    )

if y_read.shape != (
    EXPECTED_MON_WED_KNOWN_ROWS,
):
    raise RuntimeError(
        "Binary-label shape mismatch."
    )

if y_read.dtype != np.uint8:
    raise RuntimeError(
        "Binary-label dtype mismatch."
    )

if idx_read.shape != (
    EXPECTED_MON_WED_KNOWN_ROWS,
):
    raise RuntimeError(
        "Global-index shape mismatch."
    )

if idx_read.dtype != np.int32:
    raise RuntimeError(
        "Global-index dtype mismatch."
    )

if not np.array_equal(
    idx_read,
    selected_global_idx,
):
    raise RuntimeError(
        "Reloaded global row IDs differ from frozen Friday-train membership."
    )

if not np.array_equal(
    y_read,
    y_binary,
):
    raise RuntimeError(
        "Reloaded binary labels differ from frozen family-code derivation."
    )

final_inf_cells = 0
final_nan_cells = 0

for start in range(
    0,
    X_read.shape[0],
    131_072,
):
    stop = min(
        start + 131_072,
        X_read.shape[0],
    )

    block = np.asarray(
        X_read[
            start:stop,
            :
        ]
    )

    final_inf_cells += int(
        np.isinf(
            block
        ).sum()
    )

    final_nan_cells += int(
        np.isnan(
            block
        ).sum()
    )

if final_inf_cells != 0:
    raise RuntimeError(
        f"Reloaded feature matrix contains {final_inf_cells:,} inf cells."
    )

if final_nan_cells != nonfinite_global[
    "float32_nan_cells"
]:
    raise RuntimeError(
        "Reloaded NaN count differs from materialization audit."
    )

print("Shape               :", X_read.shape)
print("Dtype               :", X_read.dtype)
print("Benign labels       :", f"{int((y_read == 0).sum()):,}")
print("Attack labels       :", f"{int((y_read == 1).sum()):,}")
print("Final inf cells     :", f"{final_inf_cells:,}")
print("Final NaN cells     :", f"{final_nan_cells:,}")
print("[PASS] Prefit cache re-read exactly.")


# ======================================================================================
# 14. VERIFY FOLD SLICE BINDINGS
# ======================================================================================

banner("STAGE27-1B :: FOLD FEATURE BINDINGS")

# The shared matrix ordering is exactly BOT/DDOS/PORT_SCAN train_global_idx.
# Therefore the Thursday-target train/valid roles must be contiguous slices.

if not np.array_equal(
    idx_read[
        :EXPECTED_MON_TUE_ROWS
    ],
    inf_train,
):
    raise RuntimeError(
        "Thursday-target TRAIN is not the expected prefix of feature cache."
    )

if not np.array_equal(
    idx_read[
        EXPECTED_MON_TUE_ROWS:
    ],
    inf_valid,
):
    raise RuntimeError(
        "Thursday-target VALIDATION is not the expected suffix of feature cache."
    )

bindings = OrderedDict(
    [
        (
            "INFILTRATION",
            {
                "train_slice": [
                    0,
                    EXPECTED_MON_TUE_ROWS,
                ],
                "validation_slice": [
                    EXPECTED_MON_TUE_ROWS,
                    EXPECTED_MON_WED_KNOWN_ROWS,
                ],
                "target_predictors_materialized": False,
            },
        ),
        (
            "WEB_ATTACK",
            {
                "train_slice": [
                    0,
                    EXPECTED_MON_TUE_ROWS,
                ],
                "validation_slice": [
                    EXPECTED_MON_TUE_ROWS,
                    EXPECTED_MON_WED_KNOWN_ROWS,
                ],
                "target_predictors_materialized": False,
            },
        ),
        (
            "BOT",
            {
                "train_slice": [
                    0,
                    EXPECTED_MON_WED_KNOWN_ROWS,
                ],
                "validation_slice": None,
                "validation_status": (
                    "SEALED_THURSDAY_UNTIL_THURSDAY_TARGET_FOLD_MODELS_"
                    "AND_THRESHOLDS_ARE_FROZEN"
                ),
                "target_predictors_materialized": False,
            },
        ),
        (
            "DDOS",
            {
                "train_slice": [
                    0,
                    EXPECTED_MON_WED_KNOWN_ROWS,
                ],
                "validation_slice": None,
                "validation_status": (
                    "SEALED_THURSDAY_UNTIL_THURSDAY_TARGET_FOLD_MODELS_"
                    "AND_THRESHOLDS_ARE_FROZEN"
                ),
                "target_predictors_materialized": False,
            },
        ),
        (
            "PORT_SCAN",
            {
                "train_slice": [
                    0,
                    EXPECTED_MON_WED_KNOWN_ROWS,
                ],
                "validation_slice": None,
                "validation_status": (
                    "SEALED_THURSDAY_UNTIL_THURSDAY_TARGET_FOLD_MODELS_"
                    "AND_THRESHOLDS_ARE_FROZEN"
                ),
                "target_predictors_materialized": False,
            },
        ),
    ]
)

for family, binding in bindings.items():
    print()
    print(family)
    print("  train slice      :", binding["train_slice"])
    print("  validation slice :", binding.get("validation_slice"))
    print("  target features  :", binding["target_predictors_materialized"])

print()
print("[PASS] Prefit matrix bindings are deterministic and fold-safe.")


# ======================================================================================
# 15. HASH RUNTIME PREFIT ARTIFACTS
# ======================================================================================

banner("STAGE27-1B :: HASH PREFIT RUNTIME ARTIFACTS")

feature_receipt = npy_receipt(
    feature_path
)

label_receipt = npy_receipt(
    label_path
)

global_idx_receipt = npy_receipt(
    global_idx_path
)

print("Feature NPY SHA256 :", feature_receipt["npy_sha256"])
print("Feature content SHA:", feature_receipt["content_sha256"])
print("Label NPY SHA256   :", label_receipt["npy_sha256"])
print("Index NPY SHA256   :", global_idx_receipt["npy_sha256"])
print("[PASS] Runtime prefit artifacts cryptographically frozen.")


# ======================================================================================
# 16. ONLY NOW CREATE DURABLE RECEIPTS
# ======================================================================================

banner("STAGE27-1B :: WRITE DURABLE RECEIPTS")

OUT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

prefit_source_feature_receipt = OrderedDict(
    [
        ("stage", "Stage27-1B"),
        ("type", "PREFIT_SOURCE_FEATURE_RECEIPT"),
        ("created_at_utc", now_utc()),
        ("scientific_parent_commit", EXPECTED_PARENT),
        (
            "feature_representation",
            {
                "identity": feature_spec[
                    "identity"
                ],
                "feature_count": EXPECTED_FEATURE_COUNT,
                "feature_order": feature_order,
                "feature_order_sha256": feature_spec[
                    "feature_order_sha256"
                ],
                "adapter_variant": adapter[
                    "variant"
                ],
                "adapter_mapping_sha256": adapter[
                    "mapping_sha256"
                ],
                "parse_dtype": "float64",
                "final_model_matrix_dtype": "float32",
                "positive_infinity": "CONVERT_TO_NAN",
                "negative_infinity": "CONVERT_TO_NAN",
                "explicit_imputation": "NONE",
                "scaling": "NONE",
            },
        ),
        (
            "authorized_predictor_source_days",
            [
                "Monday",
                "Tuesday",
                "Wednesday",
            ],
        ),
        (
            "sealed_predictor_source_days",
            [
                "Thursday",
                "Friday",
            ],
        ),
        (
            "source_access_log",
            source_access_log,
        ),
        (
            "numeric_audit_by_source",
            per_source_numeric,
        ),
        (
            "numeric_audit_global",
            nonfinite_global,
        ),
        (
            "target_predictor_openings_consumed",
            0,
        ),
    ]
)

write_json(
    OUT_DIR
    / "prefit_source_feature_receipt.json",
    prefit_source_feature_receipt,
)


monwed_cache_receipt = OrderedDict(
    [
        ("stage", "Stage27-1B"),
        ("type", "MONWED_KNOWN_FEATURE_CACHE_RECEIPT"),
        ("created_at_utc", now_utc()),
        ("scientific_parent_commit", EXPECTED_PARENT),
        (
            "population",
            {
                "global_row_id_source": (
                    "FROZEN_STAGE27_1A_GLOBAL_ROW_ID"
                ),
                "rows": EXPECTED_MON_WED_KNOWN_ROWS,
                "benign": expected_benign,
                "attack": expected_attack,
                "heartbleed_target_only_unseen": 0,
                "other_attack_unseen_label": 0,
                "days": [
                    "Monday",
                    "Tuesday",
                    "Wednesday",
                ],
                "ordering": "STRICT_GLOBAL_ROW_ID_ASCENDING",
            },
        ),
        (
            "runtime_cache_root",
            str(CACHE_ROOT),
        ),
        (
            "artifacts",
            {
                "features_float32": feature_receipt,
                "binary_labels_uint8": label_receipt,
                "global_idx_int32": global_idx_receipt,
            },
        ),
        (
            "role_boundaries",
            {
                "mon_tue_end_exclusive": EXPECTED_MON_TUE_ROWS,
                "wednesday_known_start": EXPECTED_MON_TUE_ROWS,
                "wednesday_known_end_exclusive": EXPECTED_MON_WED_KNOWN_ROWS,
            },
        ),
        (
            "rebuild_policy",
            (
                "Rebuild only from the same byte-exact CICIDS2017 sources, "
                "Stage27-1A membership hashes, frozen 70-feature order and "
                "FLAG_CORRECTED adapter; require identical content hashes "
                "before any scientific fit."
            ),
        ),
    ]
)

write_json(
    OUT_DIR
    / "monwed_known_feature_cache_receipt.json",
    monwed_cache_receipt,
)


fold_feature_binding_receipt = OrderedDict(
    [
        ("stage", "Stage27-1B"),
        ("type", "FOLD_FEATURE_BINDING_RECEIPT"),
        ("created_at_utc", now_utc()),
        ("scientific_parent_commit", EXPECTED_PARENT),
        (
            "shared_feature_cache",
            str(
                feature_path
            ),
        ),
        (
            "shared_label_cache",
            str(
                label_path
            ),
        ),
        (
            "shared_global_idx_cache",
            str(
                global_idx_path
            ),
        ),
        (
            "bindings",
            bindings,
        ),
        (
            "fit_readiness",
            {
                "INFILTRATION": {
                    "train_features_ready": True,
                    "validation_features_ready": True,
                    "fit_authorized_after_remote_freeze": True,
                    "threshold_selection_authorized_after_remote_freeze": True,
                },
                "WEB_ATTACK": {
                    "train_features_ready": True,
                    "validation_features_ready": True,
                    "fit_authorized_after_remote_freeze": True,
                    "threshold_selection_authorized_after_remote_freeze": True,
                },
                "BOT": {
                    "train_features_ready": True,
                    "validation_features_ready": False,
                    "fit_authorized_after_remote_freeze": True,
                    "threshold_selection_authorized": False,
                },
                "DDOS": {
                    "train_features_ready": True,
                    "validation_features_ready": False,
                    "fit_authorized_after_remote_freeze": True,
                    "threshold_selection_authorized": False,
                },
                "PORT_SCAN": {
                    "train_features_ready": True,
                    "validation_features_ready": False,
                    "fit_authorized_after_remote_freeze": True,
                    "threshold_selection_authorized": False,
                },
            },
        ),
        (
            "target_predictor_matrix_count",
            0,
        ),
    ]
)

write_json(
    OUT_DIR
    / "fold_feature_binding_receipt.json",
    fold_feature_binding_receipt,
)


# ======================================================================================
# 17. STAGE27-1B FREEZE RECORD
# ======================================================================================

non_freeze_names = [
    name
    for name in REQUIRED_OUTPUT_FILES
    if name
    != "stage27_1b_prefit_feature_freeze_record.json"
]

artifact_hashes = OrderedDict()

for name in non_freeze_names:

    path = (
        OUT_DIR
        / name
    )

    artifact_hashes[
        name
    ] = {
        "bytes": int(
            path.stat().st_size
        ),
        "sha256": sha256_file(
            path
        ),
    }


prefit_freeze_record = OrderedDict(
    [
        ("stage", "Stage27-1B"),
        (
            "status",
            "PREFIT_FEATURE_CACHE_FROZEN_LOCALLY_PENDING_COMMIT_PUSH_REMOTE_VERIFICATION",
        ),
        ("created_at_utc", now_utc()),
        ("scientific_parent_commit", EXPECTED_PARENT),
        ("required_artifact_count", 4),
        (
            "artifact_hashes_before_freeze_record",
            artifact_hashes,
        ),
        (
            "prefit_population",
            {
                "rows": EXPECTED_MON_WED_KNOWN_ROWS,
                "features": EXPECTED_FEATURE_COUNT,
                "matrix_dtype": "float32",
                "days_read": [
                    "Monday",
                    "Tuesday",
                    "Wednesday",
                ],
                "thursday_predictors_read": False,
                "friday_predictors_read": False,
                "heartbleed_predictors_persisted": 0,
            },
        ),
        (
            "fold_readiness",
            {
                "INFILTRATION": "TRAIN_AND_VALIDATION_READY",
                "WEB_ATTACK": "TRAIN_AND_VALIDATION_READY",
                "BOT": "TRAIN_READY_VALIDATION_SEALED",
                "DDOS": "TRAIN_READY_VALIDATION_SEALED",
                "PORT_SCAN": "TRAIN_READY_VALIDATION_SEALED",
            },
        ),
        (
            "scientific_actions_completed",
            {
                "predictor_source_days_read": 3,
                "predictor_source_days_sealed": 2,
                "feature_rows_materialized": EXPECTED_MON_WED_KNOWN_ROWS,
                "model_fits": 0,
                "model_inference": 0,
                "threshold_selections": 0,
                "target_predictor_inference_openings": 0,
                "bootstrap_replicates": 0,
                "gpu_hours": 0,
            },
        ),
        (
            "next_authorized_step_after_remote_verification",
            (
                "FIT THE TWO FROZEN CPU LEARNERS FOR THE FIVE ELIGIBLE FOLDS "
                "USING ONLY THEIR FROZEN TRAIN SLICES. FOR INFILTRATION AND "
                "WEB_ATTACK, SELECT/FREEZE VALIDATION-ONLY THRESHOLDS FROM "
                "THE FROZEN WEDNESDAY VALIDATION SLICE. DO NOT READ THURSDAY "
                "PREDICTORS UNTIL THOSE THURSDAY-TARGET MODEL/THRESHOLD "
                "ARTIFACTS ARE FROZEN."
            ),
        ),
        (
            "target_predictor_opening_authorized_now",
            False,
        ),
    ]
)

write_json(
    OUT_DIR
    / "stage27_1b_prefit_feature_freeze_record.json",
    prefit_freeze_record,
)


# ======================================================================================
# 18. EXACT FOUR-FILE LOCAL VALIDATION
# ======================================================================================

banner("STAGE27-1B :: FINAL LOCAL RECEIPT VALIDATION")

actual_outputs = sorted(
    path.name
    for path in OUT_DIR.iterdir()
    if path.is_file()
)

expected_outputs = sorted(
    REQUIRED_OUTPUT_FILES
)

print("Expected durable receipts:", len(expected_outputs))
print("Actual durable receipts  :", len(actual_outputs))

if actual_outputs != expected_outputs:
    raise RuntimeError(
        "Stage27-1B receipt universe mismatch.\n"
        f"expected={expected_outputs}\n"
        f"actual={actual_outputs}"
    )

for name in REQUIRED_OUTPUT_FILES:

    path = (
        OUT_DIR
        / name
    )

    print(
        f"{name:52s} "
        f"bytes={path.stat().st_size:>10,} "
        f"sha256={sha256_file(path)}"
    )

print()
print("[PASS] Exactly four Stage27-1B durable receipts exist.")


# ======================================================================================
# 19. FREEZE READBACK
# ======================================================================================

banner("STAGE27-1B :: FREEZE READBACK")

read_freeze = load_json(
    OUT_DIR
    / "stage27_1b_prefit_feature_freeze_record.json"
)

if read_freeze[
    "required_artifact_count"
] != 4:
    raise RuntimeError(
        "Stage27-1B freeze artifact count mismatch."
    )

if len(
    read_freeze[
        "artifact_hashes_before_freeze_record"
    ]
) != 3:
    raise RuntimeError(
        "Stage27-1B freeze should hash the other 3 receipts."
    )

if read_freeze[
    "prefit_population"
][
    "thursday_predictors_read"
] is not False:
    raise RuntimeError(
        "Freeze record claims Thursday predictors were read."
    )

if read_freeze[
    "prefit_population"
][
    "friday_predictors_read"
] is not False:
    raise RuntimeError(
        "Freeze record claims Friday predictors were read."
    )

for key in [
    "model_fits",
    "model_inference",
    "threshold_selections",
    "target_predictor_inference_openings",
    "bootstrap_replicates",
    "gpu_hours",
]:

    if int(
        read_freeze[
            "scientific_actions_completed"
        ][
            key
        ]
    ) != 0:
        raise RuntimeError(
            f"Stage27-1B scientific counter {key} != 0"
        )

print("Required receipts         :", read_freeze["required_artifact_count"])
print("Non-self receipt hashes   :", len(read_freeze["artifact_hashes_before_freeze_record"]))
print("Prefit feature rows       :", f"{read_freeze['prefit_population']['rows']:,}")
print("Thursday predictors read  :", read_freeze["prefit_population"]["thursday_predictors_read"])
print("Friday predictors read    :", read_freeze["prefit_population"]["friday_predictors_read"])
print("Model fits                :", read_freeze["scientific_actions_completed"]["model_fits"])
print("Target inference openings :", read_freeze["scientific_actions_completed"]["target_predictor_inference_openings"])
print("[PASS] Stage27-1B freeze record is internally consistent.")


# ======================================================================================
# 20. STAGE EXACT FOUR RECEIPTS — NO COMMIT
# ======================================================================================

banner("STAGE27-1B :: STAGE EXACT RECEIPT SET")

for name in REQUIRED_OUTPUT_FILES:

    run(
        [
            "git",
            "add",
            "--",
            str(
                OUT_REL
                / name
            ),
        ]
    )

staged = sorted(
    line
    for line in run(
        [
            "git",
            "diff",
            "--cached",
            "--name-only",
        ]
    ).splitlines()
    if line.strip()
)

expected_staged = sorted(
    str(
        OUT_REL
        / name
    )
    for name in REQUIRED_OUTPUT_FILES
)

if staged != expected_staged:
    raise RuntimeError(
        "Unexpected staged Stage27-1B file universe.\n"
        f"expected={expected_staged}\n"
        f"actual={staged}"
    )

porcelain = sorted(
    line
    for line in run(
        [
            "git",
            "status",
            "--porcelain=v1",
            "--untracked-files=all",
        ]
    ).splitlines()
    if line.strip()
)

expected_porcelain = sorted(
    f"A  {path}"
    for path in expected_staged
)

if porcelain != expected_porcelain:
    raise RuntimeError(
        "Repository contains changes outside the exact Stage27-1B receipt set.\n\n"
        + "\n".join(
            porcelain
        )
    )

print("Expected staged:", len(expected_staged))
print("Actual staged  :", len(staged))
print("[PASS] Exactly four Stage27-1B receipts staged.")


# ======================================================================================
# 21. FINAL
# ======================================================================================

banner("STAGE27-1B PREFIT FEATURE MATERIALIZATION COMPLETE — LOCAL ONLY")

print("Scientific parent         :", EXPECTED_PARENT)
print("Feature representation    : Stage22-compatible frozen 70")
print("Feature cache rows        :", f"{EXPECTED_MON_WED_KNOWN_ROWS:,}")
print("Feature cache shape       :", f"({EXPECTED_MON_WED_KNOWN_ROWS:,}, 70)")
print("Feature cache dtype       : float32")
print("Benign                    :", f"{expected_benign:,}")
print("Attack                    :", f"{expected_attack:,}")
print("Heartbleed persisted      : 0")
print()
print("INFILTRATION train/valid  : READY / READY")
print("WEB_ATTACK train/valid    : READY / READY")
print("BOT train/valid           : READY / SEALED")
print("DDOS train/valid          : READY / SEALED")
print("PORT_SCAN train/valid     : READY / SEALED")
print()
print("Predictor days read       : Monday, Tuesday, Wednesday")
print("Thursday predictors       : SEALED")
print("Friday predictors         : SEALED")
print("Target predictor matrices : 0")
print()
print("Model fits                : 0")
print("Inference                 : 0")
print("Threshold selections      : 0")
print("Target inference openings : 0")
print("Bootstrap replicates      : 0")
print("GPU hours                 : 0")
print()
print("Durable receipts          : 4 / 4 STAGED")
print("Commit performed          : NO")
print("Push performed            : NO")
print()
print("NO TARGET INFERENCE IS AUTHORIZED.")
print()
print(
    "NEXT GATE: inspect this output, then commit/push/remote-verify "
    "the exact four Stage27-1B prefit feature receipts."
)



STAGE27-1B :: SCIENTIFIC-PARENT GATE
Expected parent : c5ad77b346f5d98e41ca5eccd4336087f8c1786e
Local HEAD      : c5ad77b346f5d98e41ca5eccd4336087f8c1786e
origin/main     : c5ad77b346f5d98e41ca5eccd4336087f8c1786e
Git clean       : True
[PASS] Stage27-1A is the clean scientific parent.

STAGE27-1B :: OUTPUT / RUNTIME GATES
Exact source cache    : /kaggle/working/stage27_cicids2017_sources
Membership cache      : /kaggle/working/stage27_1a_membership_cache
New feature cache     : /kaggle/working/stage27_1b_prefit_feature_cache
Durable output        : /kaggle/working/ids2018-validation-safe-ablation/results/stage27_loao_unseen_attack/stage27_1b_prefit_feature_materialization
Thursday predictors   : SEALED / NOT READ
Friday predictors     : SEALED / NOT READ

STAGE27-1B :: LOAD FROZEN PROTOCOL + MEMBERSHIP RECEIPTS
Feature count          : 70
Authorized future fits : 10
Membership exclusion   : PASS_ALL_5_FOLDS
Primary device         : CPU
[PASS] Frozen protocol + membership receipts loa

In [21]:
# ======================================================================================
# STAGE27-1B — COMMIT / PUSH / REMOTE VERIFICATION
#
# Scientific parent:
#   c5ad77b346f5d98e41ca5eccd4336087f8c1786e
#
# Preconditions:
#   exactly 4 Stage27-1B prefit feature receipts are staged.
#
# This script performs ONLY:
#   - staged-state verification
#   - receipt/freeze-record verification
#   - runtime cache hash readback
#   - git commit
#   - push
#   - remote SHA verification
#   - exact 4-file remote-tree verification
#   - byte-level SHA256 verification
#   - clean-tree closure
#
# ZERO model fits
# ZERO inference
# ZERO threshold selection
# ZERO target predictor inference openings
# ZERO bootstrap
# ZERO GPU
# ======================================================================================

from __future__ import annotations

import hashlib
import json
import os
import stat
import subprocess
from pathlib import Path

import numpy as np
from kaggle_secrets import UserSecretsClient


# ======================================================================================
# 0. CONSTANTS
# ======================================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_PARENT = (
    "c5ad77b346f5d98e41ca5eccd4336087f8c1786e"
)

OUT_REL = Path(
    "results/stage27_loao_unseen_attack/"
    "stage27_1b_prefit_feature_materialization"
)

OUT_DIR = REPO / OUT_REL

REQUIRED_FILES = [
    "prefit_source_feature_receipt.json",
    "monwed_known_feature_cache_receipt.json",
    "fold_feature_binding_receipt.json",
    "stage27_1b_prefit_feature_freeze_record.json",
]

FREEZE_FILE = (
    "stage27_1b_prefit_feature_freeze_record.json"
)

COMMIT_MESSAGE = (
    "stage27-1b: freeze prefit feature materialization"
)


# ======================================================================================
# 1. HELPERS
# ======================================================================================

def banner(title: str) -> None:
    print()
    print("=" * 118)
    print(title)
    print("=" * 118)


def run(
    cmd,
    *,
    env=None,
    text=True,
    check=True,
):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=REPO,
        env=env,
        text=text,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}): "
            f"{' '.join(map(str, cmd))}\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return p.stdout.strip() if text else p.stdout


def sha256_file(
    path: Path,
    chunk_size: int = 16 * 1024 * 1024,
) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        while True:
            block = f.read(chunk_size)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(
        data
    ).hexdigest()


def sha256_array_content(
    arr: np.ndarray,
    block_rows: int = 131_072,
) -> str:

    h = hashlib.sha256()

    if arr.ndim == 1:

        for start in range(
            0,
            arr.shape[0],
            block_rows,
        ):
            stop = min(
                start + block_rows,
                arr.shape[0],
            )

            block = np.ascontiguousarray(
                arr[start:stop]
            )

            h.update(
                block.tobytes(
                    order="C"
                )
            )

    elif arr.ndim == 2:

        for start in range(
            0,
            arr.shape[0],
            block_rows,
        ):
            stop = min(
                start + block_rows,
                arr.shape[0],
            )

            block = np.ascontiguousarray(
                arr[start:stop, :]
            )

            h.update(
                block.tobytes(
                    order="C"
                )
            )

    else:
        raise RuntimeError(
            f"Unsupported ndim={arr.ndim} for content hashing."
        )

    return h.hexdigest()


def load_json(path: Path):
    with path.open(
        "r",
        encoding="utf-8",
    ) as f:
        return json.load(f)


# ======================================================================================
# 2. PRE-COMMIT SCIENTIFIC-PARENT GATE
# ======================================================================================

banner("STAGE27-1B :: PRE-COMMIT SCIENTIFIC-PARENT GATE")

head = run(
    [
        "git",
        "rev-parse",
        "HEAD",
    ]
)

origin_before = run(
    [
        "git",
        "rev-parse",
        "origin/main",
    ]
)

print("Expected parent :", EXPECTED_PARENT)
print("Local HEAD      :", head)
print("origin/main     :", origin_before)

if head != EXPECTED_PARENT:
    raise RuntimeError(
        "Local HEAD changed before Stage27-1B commit."
    )

if origin_before != EXPECTED_PARENT:
    raise RuntimeError(
        "origin/main changed before Stage27-1B commit."
    )

print("[PASS] Scientific parent unchanged.")


# ======================================================================================
# 3. EXACT FOUR-FILE STAGED UNIVERSE
# ======================================================================================

banner("STAGE27-1B :: EXACT STAGED UNIVERSE")

expected_paths = sorted(
    str(
        OUT_REL / name
    )
    for name in REQUIRED_FILES
)

staged_paths = sorted(
    line
    for line in run(
        [
            "git",
            "diff",
            "--cached",
            "--name-only",
        ]
    ).splitlines()
    if line.strip()
)

print("Expected staged files:", len(expected_paths))
print("Actual staged files  :", len(staged_paths))

if staged_paths != expected_paths:
    raise RuntimeError(
        "Staged file universe changed.\n"
        f"expected={expected_paths}\n"
        f"actual={staged_paths}"
    )

status_before = sorted(
    line
    for line in run(
        [
            "git",
            "status",
            "--porcelain=v1",
            "--untracked-files=all",
        ]
    ).splitlines()
    if line.strip()
)

expected_status = sorted(
    f"A  {path}"
    for path in expected_paths
)

if status_before != expected_status:
    raise RuntimeError(
        "Unexpected repository state before commit.\n\n"
        + "\n".join(
            status_before
        )
    )

print(
    "[PASS] Exactly 4 Stage27-1B receipts staged; no unrelated changes."
)


# ======================================================================================
# 4. FREEZE RECORD GATE
# ======================================================================================

banner("STAGE27-1B :: PREFIT FREEZE RECORD GATE")

freeze = load_json(
    OUT_DIR / FREEZE_FILE
)

print(
    "required_artifact_count  :",
    freeze.get(
        "required_artifact_count"
    ),
)

print(
    "non-self artifact hashes :",
    len(
        freeze.get(
            "artifact_hashes_before_freeze_record",
            {},
        )
    ),
)

print(
    "prefit rows              :",
    freeze[
        "prefit_population"
    ][
        "rows"
    ],
)

print(
    "Thursday predictors read :",
    freeze[
        "prefit_population"
    ][
        "thursday_predictors_read"
    ],
)

print(
    "Friday predictors read   :",
    freeze[
        "prefit_population"
    ][
        "friday_predictors_read"
    ],
)

if freeze.get(
    "scientific_parent_commit"
) != EXPECTED_PARENT:
    raise RuntimeError(
        "Stage27-1B freeze scientific parent mismatch."
    )

if freeze.get(
    "required_artifact_count"
) != 4:
    raise RuntimeError(
        "Stage27-1B required_artifact_count != 4."
    )

hashed = freeze[
    "artifact_hashes_before_freeze_record"
]

expected_nonself = sorted(
    name
    for name in REQUIRED_FILES
    if name != FREEZE_FILE
)

if sorted(
    hashed.keys()
) != expected_nonself:
    raise RuntimeError(
        "Stage27-1B non-self receipt universe mismatch."
    )

for name in expected_nonself:

    path = OUT_DIR / name

    actual_sha = sha256_file(
        path
    )

    expected_sha = hashed[
        name
    ][
        "sha256"
    ]

    if actual_sha != expected_sha:
        raise RuntimeError(
            f"{name}: receipt SHA256 mismatch."
        )

    if int(
        path.stat().st_size
    ) != int(
        hashed[
            name
        ][
            "bytes"
        ]
    ):
        raise RuntimeError(
            f"{name}: receipt byte-size mismatch."
        )

if freeze[
    "prefit_population"
][
    "rows"
] != 1_668_519:
    raise RuntimeError(
        "Unexpected prefit population row count."
    )

if freeze[
    "prefit_population"
][
    "features"
] != 70:
    raise RuntimeError(
        "Unexpected feature count."
    )

if freeze[
    "prefit_population"
][
    "matrix_dtype"
] != "float32":
    raise RuntimeError(
        "Unexpected prefit matrix dtype."
    )

if freeze[
    "prefit_population"
][
    "thursday_predictors_read"
] is not False:
    raise RuntimeError(
        "Thursday predictors are no longer sealed."
    )

if freeze[
    "prefit_population"
][
    "friday_predictors_read"
] is not False:
    raise RuntimeError(
        "Friday predictors are no longer sealed."
    )

actions = freeze[
    "scientific_actions_completed"
]

for key in [
    "model_fits",
    "model_inference",
    "threshold_selections",
    "target_predictor_inference_openings",
    "bootstrap_replicates",
    "gpu_hours",
]:

    if int(
        actions[
            key
        ]
    ) != 0:
        raise RuntimeError(
            f"Scientific boundary violation: {key} != 0."
        )

print(
    "[PASS] Stage27-1B freeze record and all three non-self receipts verified."
)


# ======================================================================================
# 5. FEATURE / CACHE RECEIPT READBACK
# ======================================================================================

banner("STAGE27-1B :: RUNTIME PREFIT CACHE HASH READBACK")

cache_receipt = load_json(
    OUT_DIR
    / "monwed_known_feature_cache_receipt.json"
)

feature_info = cache_receipt[
    "artifacts"
][
    "features_float32"
]

label_info = cache_receipt[
    "artifacts"
][
    "binary_labels_uint8"
]

index_info = cache_receipt[
    "artifacts"
][
    "global_idx_int32"
]

runtime_artifacts = [
    (
        "features_float32",
        feature_info,
        (1_668_519, 70),
        np.dtype("float32"),
    ),
    (
        "binary_labels_uint8",
        label_info,
        (1_668_519,),
        np.dtype("uint8"),
    ),
    (
        "global_idx_int32",
        index_info,
        (1_668_519,),
        np.dtype("int32"),
    ),
]

for name, info, expected_shape, expected_dtype in runtime_artifacts:

    path = Path(
        info[
            "path"
        ]
    )

    if not path.is_file():
        raise RuntimeError(
            f"Runtime cache artifact missing: {path}"
        )

    actual_file_sha = sha256_file(
        path
    )

    if actual_file_sha != info[
        "npy_sha256"
    ]:
        raise RuntimeError(
            f"{name}: runtime NPY SHA mismatch."
        )

    arr = np.load(
        path,
        allow_pickle=False,
        mmap_mode="r",
    )

    if tuple(
        arr.shape
    ) != expected_shape:
        raise RuntimeError(
            f"{name}: shape mismatch "
            f"{arr.shape} != {expected_shape}."
        )

    if arr.dtype != expected_dtype:
        raise RuntimeError(
            f"{name}: dtype mismatch "
            f"{arr.dtype} != {expected_dtype}."
        )

    actual_content_sha = sha256_array_content(
        arr
    )

    if actual_content_sha != info[
        "content_sha256"
    ]:
        raise RuntimeError(
            f"{name}: runtime content SHA mismatch."
        )

    print()
    print(name)
    print("  path        :", path)
    print("  shape       :", arr.shape)
    print("  dtype       :", arr.dtype)
    print("  npy sha256  :", actual_file_sha)
    print("  content sha :", actual_content_sha)

print()
print(
    "[PASS] Runtime prefit feature/label/index cache still matches frozen receipts."
)


# ======================================================================================
# 6. SOURCE/SEALED-DAY RECEIPT READBACK
# ======================================================================================

banner("STAGE27-1B :: SEALED-DAY RECEIPT READBACK")

source_receipt = load_json(
    OUT_DIR
    / "prefit_source_feature_receipt.json"
)

authorized_days = source_receipt[
    "authorized_predictor_source_days"
]

sealed_days = source_receipt[
    "sealed_predictor_source_days"
]

access_days = [
    entry[
        "day"
    ]
    for entry in source_receipt[
        "source_access_log"
    ]
]

print("Authorized predictor days:", authorized_days)
print("Observed access days      :", access_days)
print("Sealed predictor days    :", sealed_days)

if authorized_days != [
    "Monday",
    "Tuesday",
    "Wednesday",
]:
    raise RuntimeError(
        "Authorized predictor-day set changed."
    )

if access_days != authorized_days:
    raise RuntimeError(
        "Observed predictor access differs from authorized Mon-Wed set."
    )

if sealed_days != [
    "Thursday",
    "Friday",
]:
    raise RuntimeError(
        "Sealed predictor-day set changed."
    )

if source_receipt[
    "target_predictor_openings_consumed"
] != 0:
    raise RuntimeError(
        "Target predictor opening counter is nonzero."
    )

print(
    "[PASS] Thursday and Friday predictors remain sealed."
)


# ======================================================================================
# 7. FOLD BINDING RECEIPT READBACK
# ======================================================================================

banner("STAGE27-1B :: FOLD FEATURE BINDING READBACK")

binding_receipt = load_json(
    OUT_DIR
    / "fold_feature_binding_receipt.json"
)

bindings = binding_receipt[
    "bindings"
]

readiness = binding_receipt[
    "fit_readiness"
]

expected_readiness = {
    "INFILTRATION": (
        True,
        True,
    ),
    "WEB_ATTACK": (
        True,
        True,
    ),
    "BOT": (
        True,
        False,
    ),
    "DDOS": (
        True,
        False,
    ),
    "PORT_SCAN": (
        True,
        False,
    ),
}

for family, (
    train_ready,
    validation_ready,
) in expected_readiness.items():

    actual = readiness[
        family
    ]

    if actual[
        "train_features_ready"
    ] is not train_ready:
        raise RuntimeError(
            f"{family}: train readiness mismatch."
        )

    if actual[
        "validation_features_ready"
    ] is not validation_ready:
        raise RuntimeError(
            f"{family}: validation readiness mismatch."
        )

    if bindings[
        family
    ][
        "target_predictors_materialized"
    ] is not False:
        raise RuntimeError(
            f"{family}: target predictors unexpectedly materialized."
        )

    print(
        f"{family:14s} "
        f"TRAIN={'READY' if train_ready else 'SEALED':6s} "
        f"VALID={'READY' if validation_ready else 'SEALED':6s} "
        f"TARGET=SEALED"
    )

if binding_receipt[
    "target_predictor_matrix_count"
] != 0:
    raise RuntimeError(
        "Target predictor matrix count is nonzero."
    )

print(
    "[PASS] Fold feature bindings/readiness remain frozen."
)


# ======================================================================================
# 8. GIT IDENTITY
# ======================================================================================

banner("STAGE27-1B :: GIT AUTHOR IDENTITY")

git_name = run(
    [
        "git",
        "config",
        "--local",
        "user.name",
    ],
    check=False,
)

git_email = run(
    [
        "git",
        "config",
        "--local",
        "user.email",
    ],
    check=False,
)

if not git_name:

    run(
        [
            "git",
            "config",
            "--local",
            "user.name",
            "J.M. Mubasshir Rahman",
        ]
    )

    git_name = run(
        [
            "git",
            "config",
            "--local",
            "user.name",
        ]
    )

if not git_email:

    run(
        [
            "git",
            "config",
            "--local",
            "user.email",
            "themubasshir@users.noreply.github.com",
        ]
    )

    git_email = run(
        [
            "git",
            "config",
            "--local",
            "user.email",
        ]
    )

print("Git author name :", git_name)
print("Git author email:", git_email)
print("[PASS] Repository-local Git identity available.")


# ======================================================================================
# 9. GITHUB AUTH
# ======================================================================================

banner("STAGE27-1B :: LOAD GITHUB AUTH")

client = UserSecretsClient()

token = None
token_label = None

for label in [
    "GITHUB_TOKEN",
    "github_token",
    "GH_TOKEN",
    "GITHUB_PAT",
    "github_pat",
]:

    try:
        candidate = client.get_secret(
            label
        )
    except Exception:
        candidate = None

    if candidate:

        candidate = str(
            candidate
        ).strip()

        if candidate:
            token = candidate
            token_label = label
            break

if not token:
    raise RuntimeError(
        "No GitHub token found in Kaggle Secrets."
    )

print(
    f"[FOUND] GitHub secret label: {token_label} "
    "(value not printed)"
)

askpass = Path(
    "/kaggle/working/"
    ".stage27_1b_commit_askpass.sh"
)

askpass.write_text(
    "#!/bin/sh\n"
    'case "$1" in\n'
    '  *Username*) printf "%s\\n" "x-access-token" ;;\n'
    '  *Password*) printf "%s\\n" "$STAGE27_GITHUB_TOKEN" ;;\n'
    '  *) printf "%s\\n" "" ;;\n'
    "esac\n",
    encoding="utf-8",
)

askpass.chmod(
    stat.S_IRUSR
    | stat.S_IWUSR
    | stat.S_IXUSR
)

env = os.environ.copy()

env[
    "GIT_ASKPASS"
] = str(
    askpass
)

env[
    "GIT_TERMINAL_PROMPT"
] = "0"

env[
    "STAGE27_GITHUB_TOKEN"
] = token


# ======================================================================================
# 10. COMMIT / PUSH / REMOTE VERIFY
# ======================================================================================

try:

    banner("STAGE27-1B :: COMMIT")

    print(
        run(
            [
                "git",
                "diff",
                "--cached",
                "--stat",
            ]
        )
    )

    commit_output = run(
        [
            "git",
            "commit",
            "-m",
            COMMIT_MESSAGE,
        ]
    )

    print()
    print(
        commit_output
    )

    new_head = run(
        [
            "git",
            "rev-parse",
            "HEAD",
        ]
    )

    new_parent = run(
        [
            "git",
            "rev-parse",
            "HEAD^",
        ]
    )

    subject = run(
        [
            "git",
            "show",
            "-s",
            "--format=%s",
            "HEAD",
        ]
    )

    print()
    print("Stage27-1B commit:", new_head)
    print("Commit parent    :", new_parent)
    print("Commit subject   :", subject)

    if new_parent != EXPECTED_PARENT:
        raise RuntimeError(
            "Stage27-1B commit parent mismatch."
        )

    if subject != COMMIT_MESSAGE:
        raise RuntimeError(
            "Stage27-1B commit subject mismatch."
        )


    # ------------------------------------------------------------------
    # Local committed tree
    # ------------------------------------------------------------------

    banner("STAGE27-1B :: LOCAL COMMIT TREE")

    committed_paths = sorted(
        line
        for line in run(
            [
                "git",
                "ls-tree",
                "-r",
                "--name-only",
                "HEAD",
                "--",
                str(
                    OUT_REL
                ),
            ]
        ).splitlines()
        if line.strip()
    )

    print(
        "Committed Stage27-1B files:",
        len(
            committed_paths
        ),
    )

    if committed_paths != expected_paths:
        raise RuntimeError(
            "Committed Stage27-1B tree mismatch."
        )

    print(
        "[PASS] Local commit contains exactly 4 Stage27-1B receipts."
    )


    # ------------------------------------------------------------------
    # Push
    # ------------------------------------------------------------------

    banner("STAGE27-1B :: PUSH MAIN")

    push_output = run(
        [
            "git",
            "push",
            "origin",
            "main",
        ],
        env=env,
    )

    print(
        push_output
        if push_output
        else "[OK] Push completed."
    )


    # ------------------------------------------------------------------
    # Remote SHA verification
    # ------------------------------------------------------------------

    banner("STAGE27-1B :: REMOTE SHA VERIFICATION")

    remote_main = run(
        [
            "git",
            "ls-remote",
            "origin",
            "refs/heads/main",
        ],
        env=env,
    ).split()[0]

    run(
        [
            "git",
            "fetch",
            "origin",
            "main",
        ],
        env=env,
    )

    local_after = run(
        [
            "git",
            "rev-parse",
            "HEAD",
        ]
    )

    origin_after = run(
        [
            "git",
            "rev-parse",
            "origin/main",
        ]
    )

    print("Local HEAD  :", local_after)
    print("origin/main :", origin_after)
    print("remote main :", remote_main)

    if not (
        local_after
        == origin_after
        == remote_main
        == new_head
    ):
        raise RuntimeError(
            "Stage27-1B remote SHA verification failed."
        )

    print(
        "[PASS] Local HEAD == origin/main == remote main."
    )


    # ------------------------------------------------------------------
    # Exact remote tree
    # ------------------------------------------------------------------

    banner("STAGE27-1B :: REMOTE 4-FILE TREE VERIFICATION")

    remote_paths = sorted(
        line
        for line in run(
            [
                "git",
                "ls-tree",
                "-r",
                "--name-only",
                "origin/main",
                "--",
                str(
                    OUT_REL
                ),
            ]
        ).splitlines()
        if line.strip()
    )

    print("Expected remote files:", len(expected_paths))
    print("Actual remote files  :", len(remote_paths))

    if remote_paths != expected_paths:
        raise RuntimeError(
            "Remote Stage27-1B file universe mismatch.\n"
            f"expected={expected_paths}\n"
            f"actual={remote_paths}"
        )

    for path in remote_paths:
        print(" ", path)

    print()
    print(
        "[PASS] Remote tree contains exactly 4 Stage27-1B receipts."
    )


    # ------------------------------------------------------------------
    # Byte-level remote verification
    # ------------------------------------------------------------------

    banner("STAGE27-1B :: REMOTE BYTE-LEVEL SHA256 VERIFICATION")

    for rel in expected_paths:

        local_path = (
            REPO
            / rel
        )

        local_sha = sha256_file(
            local_path
        )

        remote_bytes = run(
            [
                "git",
                "show",
                f"origin/main:{rel}",
            ],
            text=False,
        )

        remote_sha = sha256_bytes(
            remote_bytes
        )

        if local_sha != remote_sha:
            raise RuntimeError(
                f"Remote byte mismatch:\n{rel}\n"
                f"local ={local_sha}\n"
                f"remote={remote_sha}"
            )

        print(
            f"{local_sha}  {rel}"
        )

    print()
    print(
        "[PASS] All 4 remote Stage27-1B receipts are byte-identical."
    )


    # ------------------------------------------------------------------
    # Remote freeze readback
    # ------------------------------------------------------------------

    banner("STAGE27-1B :: REMOTE PREFIT FREEZE READBACK")

    remote_freeze_bytes = run(
        [
            "git",
            "show",
            f"origin/main:{OUT_REL / FREEZE_FILE}",
        ],
        text=False,
    )

    remote_freeze = json.loads(
        remote_freeze_bytes.decode(
            "utf-8"
        )
    )

    if remote_freeze[
        "required_artifact_count"
    ] != 4:
        raise RuntimeError(
            "Remote Stage27-1B artifact count != 4."
        )

    if len(
        remote_freeze[
            "artifact_hashes_before_freeze_record"
        ]
    ) != 3:
        raise RuntimeError(
            "Remote Stage27-1B freeze should hash 3 non-self receipts."
        )

    if remote_freeze[
        "scientific_parent_commit"
    ] != EXPECTED_PARENT:
        raise RuntimeError(
            "Remote Stage27-1B scientific parent mismatch."
        )

    if remote_freeze[
        "prefit_population"
    ][
        "thursday_predictors_read"
    ] is not False:
        raise RuntimeError(
            "Remote freeze says Thursday predictors were read."
        )

    if remote_freeze[
        "prefit_population"
    ][
        "friday_predictors_read"
    ] is not False:
        raise RuntimeError(
            "Remote freeze says Friday predictors were read."
        )

    remote_actions = remote_freeze[
        "scientific_actions_completed"
    ]

    for key in [
        "model_fits",
        "model_inference",
        "threshold_selections",
        "target_predictor_inference_openings",
        "bootstrap_replicates",
        "gpu_hours",
    ]:

        if int(
            remote_actions[
                key
            ]
        ) != 0:
            raise RuntimeError(
                f"Remote scientific counter {key} != 0."
            )

    print("Required artifacts        :", remote_freeze["required_artifact_count"])
    print("Non-self hashes           :", len(remote_freeze["artifact_hashes_before_freeze_record"]))
    print("Prefit rows               :", f"{remote_freeze['prefit_population']['rows']:,}")
    print("Thursday predictors read  :", remote_freeze["prefit_population"]["thursday_predictors_read"])
    print("Friday predictors read    :", remote_freeze["prefit_population"]["friday_predictors_read"])
    print("Model fits                :", remote_actions["model_fits"])
    print("Target inference openings :", remote_actions["target_predictor_inference_openings"])

    print(
        "[PASS] Remote Stage27-1B freeze semantics verified."
    )


    # ------------------------------------------------------------------
    # Final clean state
    # ------------------------------------------------------------------

    banner("STAGE27-1B :: FINAL CLOSURE")

    final_status = run(
        [
            "git",
            "status",
            "--porcelain=v1",
            "--untracked-files=all",
        ]
    )

    if final_status.strip():
        raise RuntimeError(
            "Repository is not clean after Stage27-1B push:\n"
            + final_status
        )

    print("Stage27-1B scientific parent :", EXPECTED_PARENT)
    print("Stage27-1B commit            :", new_head)
    print("Remote verification          : PASS")
    print("Durable receipts             : 4 / 4")
    print("Freeze non-self hashes       : 3 / 3")
    print("Git working tree             : CLEAN")

    print()
    print("Feature cache                : 1,668,519 x 70 float32")
    print("INFILTRATION train/valid     : READY / READY")
    print("WEB_ATTACK train/valid       : READY / READY")
    print("BOT train/valid              : READY / SEALED")
    print("DDOS train/valid             : READY / SEALED")
    print("PORT_SCAN train/valid        : READY / SEALED")

    print()
    print("Thursday predictors          : SEALED")
    print("Friday predictors            : SEALED")
    print("Model fits                   : 0")
    print("Inference                    : 0")
    print("Threshold selections         : 0")
    print("Target inference openings    : 0")
    print("Bootstrap replicates         : 0")
    print("GPU hours                    : 0")

    print()
    print("=" * 118)
    print(
        "STAGE27-1B COMPLETE — PREFIT FEATURE CACHE REMOTELY FROZEN"
    )
    print("=" * 118)

    print()
    print(
        "NEXT AUTHORIZED PHASE:"
    )

    print(
        "  Execute the preregistered CPU model-fit sequence using only "
        "the frozen TRAIN slices. INFILTRATION and WEB_ATTACK may also "
        "use their frozen Wednesday validation slice for validation-only "
        "threshold selection. Thursday and Friday predictor sources remain sealed."
    )


finally:

    try:
        askpass.unlink(
            missing_ok=True
        )
    except Exception:
        pass

    token = None

    env.pop(
        "STAGE27_GITHUB_TOKEN",
        None,
    )



STAGE27-1B :: PRE-COMMIT SCIENTIFIC-PARENT GATE
Expected parent : c5ad77b346f5d98e41ca5eccd4336087f8c1786e
Local HEAD      : c5ad77b346f5d98e41ca5eccd4336087f8c1786e
origin/main     : c5ad77b346f5d98e41ca5eccd4336087f8c1786e
[PASS] Scientific parent unchanged.

STAGE27-1B :: EXACT STAGED UNIVERSE
Expected staged files: 4
Actual staged files  : 4
[PASS] Exactly 4 Stage27-1B receipts staged; no unrelated changes.

STAGE27-1B :: PREFIT FREEZE RECORD GATE
required_artifact_count  : 4
non-self artifact hashes : 3
prefit rows              : 1668519
Thursday predictors read : False
Friday predictors read   : False
[PASS] Stage27-1B freeze record and all three non-self receipts verified.

STAGE27-1B :: RUNTIME PREFIT CACHE HASH READBACK

features_float32
  path        : /kaggle/working/stage27_1b_prefit_feature_cache/monwed_known_features_float32.npy
  shape       : (1668519, 70)
  dtype       : float32
  npy sha256  : d86e0c0272a33000ce7ed8e0c5a25005db8847939f793b0957d97ce0917a1ea6
  content

In [22]:

======================================================================================================================
STAGE27-1B :: PRE-COMMIT SCIENTIFIC-PARENT GATE
======================================================================================================================
Expected parent : c5ad77b346f5d98e41ca5eccd4336087f8c1786e
Local HEAD      : c5ad77b346f5d98e41ca5eccd4336087f8c1786e
origin/main     : c5ad77b346f5d98e41ca5eccd4336087f8c1786e
[PASS] Scientific parent unchanged.

======================================================================================================================
STAGE27-1B :: EXACT STAGED UNIVERSE
======================================================================================================================
Expected staged files: 4
Actual staged files  : 4
[PASS] Exactly 4 Stage27-1B receipts staged; no unrelated changes.

======================================================================================================================
STAGE27-1B :: PREFIT FREEZE RECORD GATE
======================================================================================================================
required_artifact_count  : 4
non-self artifact hashes : 3
prefit rows              : 1668519
Thursday predictors read : False
Friday predictors read   : False
[PASS] Stage27-1B freeze record and all three non-self receipts verified.

======================================================================================================================
STAGE27-1B :: RUNTIME PREFIT CACHE HASH READBACK
======================================================================================================================

features_float32
  path        : /kaggle/working/stage27_1b_prefit_feature_cache/monwed_known_features_float32.npy
  shape       : (1668519, 70)
  dtype       : float32
  npy sha256  : d86e0c0272a33000ce7ed8e0c5a25005db8847939f793b0957d97ce0917a1ea6
  content sha : fc3137b10bbb2542240df3d380286b88f6ea282aba200fd893e6445391855ccf

binary_labels_uint8
  path        : /kaggle/working/stage27_1b_prefit_feature_cache/monwed_known_binary_labels_uint8.npy
  shape       : (1668519,)
  dtype       : uint8
  npy sha256  : c3c8dc29b60a579a04f329c850dc7318044f582ebf3581e309dec6eee53ed529
  content sha : cc0d2a0d3b8d57461a0374e808dbbf942832a81fc9fcb5541eb8769cb207235b

global_idx_int32
  path        : /kaggle/working/stage27_1b_prefit_feature_cache/monwed_known_global_idx_int32.npy
  shape       : (1668519,)
  dtype       : int32
  npy sha256  : 7f87ae6455bce853677236670250bfc0608b99ba3a285c901cd0b6bb6ee65be5
  content sha : b001047298984aa5db11b80ebbdb12fe48a2277ee7ed62ad1a27817ef67adc7e

[PASS] Runtime prefit feature/label/index cache still matches frozen receipts.

======================================================================================================================
STAGE27-1B :: SEALED-DAY RECEIPT READBACK
======================================================================================================================
Authorized predictor days: ['Monday', 'Tuesday', 'Wednesday']
Observed access days      : ['Monday', 'Tuesday', 'Wednesday']
Sealed predictor days    : ['Thursday', 'Friday']
[PASS] Thursday and Friday predictors remain sealed.

======================================================================================================================
STAGE27-1B :: FOLD FEATURE BINDING READBACK
======================================================================================================================
INFILTRATION   TRAIN=READY  VALID=READY  TARGET=SEALED
WEB_ATTACK     TRAIN=READY  VALID=READY  TARGET=SEALED
BOT            TRAIN=READY  VALID=SEALED TARGET=SEALED
DDOS           TRAIN=READY  VALID=SEALED TARGET=SEALED
PORT_SCAN      TRAIN=READY  VALID=SEALED TARGET=SEALED
[PASS] Fold feature bindings/readiness remain frozen.

======================================================================================================================
STAGE27-1B :: GIT AUTHOR IDENTITY
======================================================================================================================
Git author name : J.M. Mubasshir Rahman
Git author email: themubasshir@users.noreply.github.com
[PASS] Repository-local Git identity available.

======================================================================================================================
STAGE27-1B :: LOAD GITHUB AUTH
======================================================================================================================
[FOUND] GitHub secret label: GITHUB_TOKEN (value not printed)

======================================================================================================================
STAGE27-1B :: COMMIT
======================================================================================================================
.../fold_feature_binding_receipt.json              |  93 +++++++++++
 .../monwed_known_feature_cache_receipt.json        |  60 +++++++
 .../prefit_source_feature_receipt.json             | 173 +++++++++++++++++++++
 .../stage27_1b_prefit_feature_freeze_record.json   |  54 +++++++
 4 files changed, 380 insertions(+)

[main 9bdf2c1] stage27-1b: freeze prefit feature materialization
 4 files changed, 380 insertions(+)
 create mode 100644 results/stage27_loao_unseen_attack/stage27_1b_prefit_feature_materialization/fold_feature_binding_receipt.json
 create mode 100644 results/stage27_loao_unseen_attack/stage27_1b_prefit_feature_materialization/monwed_known_feature_cache_receipt.json
 create mode 100644 results/stage27_loao_unseen_attack/stage27_1b_prefit_feature_materialization/prefit_source_feature_receipt.json
 create mode 100644 results/stage27_loao_unseen_attack/stage27_1b_prefit_feature_materialization/stage27_1b_prefit_feature_freeze_record.json

Stage27-1B commit: 9bdf2c1fca362312855b8612c579da685a390ffe
Commit parent    : c5ad77b346f5d98e41ca5eccd4336087f8c1786e
Commit subject   : stage27-1b: freeze prefit feature materialization

======================================================================================================================
STAGE27-1B :: LOCAL COMMIT TREE
======================================================================================================================
Committed Stage27-1B files: 4
[PASS] Local commit contains exactly 4 Stage27-1B receipts.

======================================================================================================================
STAGE27-1B :: PUSH MAIN
======================================================================================================================
[OK] Push completed.

======================================================================================================================
STAGE27-1B :: REMOTE SHA VERIFICATION
======================================================================================================================
Local HEAD  : 9bdf2c1fca362312855b8612c579da685a390ffe
origin/main : 9bdf2c1fca362312855b8612c579da685a390ffe
remote main : 9bdf2c1fca362312855b8612c579da685a390ffe
[PASS] Local HEAD == origin/main == remote main.

======================================================================================================================
STAGE27-1B :: REMOTE 4-FILE TREE VERIFICATION
======================================================================================================================
Expected remote files: 4
Actual remote files  : 4
  results/stage27_loao_unseen_attack/stage27_1b_prefit_feature_materialization/fold_feature_binding_receipt.json
  results/stage27_loao_unseen_attack/stage27_1b_prefit_feature_materialization/monwed_known_feature_cache_receipt.json
  results/stage27_loao_unseen_attack/stage27_1b_prefit_feature_materialization/prefit_source_feature_receipt.json
  results/stage27_loao_unseen_attack/stage27_1b_prefit_feature_materialization/stage27_1b_prefit_feature_freeze_record.json

[PASS] Remote tree contains exactly 4 Stage27-1B receipts.

======================================================================================================================
STAGE27-1B :: REMOTE BYTE-LEVEL SHA256 VERIFICATION
======================================================================================================================
c2bcac1ef7617a8b14d5d3a68e55b553736ee19236c713d2c0cf50da317f0b3c  results/stage27_loao_unseen_attack/stage27_1b_prefit_feature_materialization/fold_feature_binding_receipt.json
0ed8418a2e96ed09470fd53b6db4729a7c8b0381c07b538955a6f6d74f62b109  results/stage27_loao_unseen_attack/stage27_1b_prefit_feature_materialization/monwed_known_feature_cache_receipt.json
ccf8732fcaf8cf3ca83b17410262fbccf16911e1651169e9c50d3cfc177e9c9c  results/stage27_loao_unseen_attack/stage27_1b_prefit_feature_materialization/prefit_source_feature_receipt.json
c0d71088da234067b7dbf4d1f312670fd88fff17a75af3ba5455360fee22e6ea  results/stage27_loao_unseen_attack/stage27_1b_prefit_feature_materialization/stage27_1b_prefit_feature_freeze_record.json

[PASS] All 4 remote Stage27-1B receipts are byte-identical.

======================================================================================================================
STAGE27-1B :: REMOTE PREFIT FREEZE READBACK
======================================================================================================================
Required artifacts        : 4
Non-self hashes           : 3
Prefit rows               : 1,668,519
Thursday predictors read  : False
Friday predictors read    : False
Model fits                : 0
Target inference openings : 0
[PASS] Remote Stage27-1B freeze semantics verified.

======================================================================================================================
STAGE27-1B :: FINAL CLOSURE
======================================================================================================================
Stage27-1B scientific parent : c5ad77b346f5d98e41ca5eccd4336087f8c1786e
Stage27-1B commit            : 9bdf2c1fca362312855b8612c579da685a390ffe
Remote verification          : PASS
Durable receipts             : 4 / 4
Freeze non-self hashes       : 3 / 3
Git working tree             : CLEAN

Feature cache                : 1,668,519 x 70 float32
INFILTRATION train/valid     : READY / READY
WEB_ATTACK train/valid       : READY / READY
BOT train/valid              : READY / SEALED
DDOS train/valid             : READY / SEALED
PORT_SCAN train/valid        : READY / SEALED

Thursday predictors          : SEALED
Friday predictors            : SEALED
Model fits                   : 0
Inference                    : 0
Threshold selections         : 0
Target inference openings    : 0
Bootstrap replicates         : 0
GPU hours                    : 0

======================================================================================================================
STAGE27-1B COMPLETE — PREFIT FEATURE CACHE REMOTELY FROZEN
======================================================================================================================

NEXT AUTHORIZED PHASE:
  Execute the preregistered CPU model-fit sequence using only the frozen TRAIN slices. INFILTRATION and WEB_ATTACK may also use their frozen Wednesday validation slice for validation-only threshold selection. Thursday and Friday predictor sources remain sealed.

SyntaxError: invalid decimal literal (3063825707.py, line 2)

In [23]:
from pathlib import Path

SCRIPT = Path(
    "/kaggle/working/stage27_2a_all10_fits_wednesday_threshold_freeze.py"
)

if not SCRIPT.is_file():
    raise FileNotFoundError(
        f"""
Stage27-2A script was not found:

    {SCRIPT}

Upload the actual .py file to the Kaggle notebook first.
Do NOT paste the previous console output into a code cell.
"""
    )

text = SCRIPT.read_text(
    encoding="utf-8"
)

print("Script path :", SCRIPT)
print("Script bytes:", SCRIPT.stat().st_size)

# Strong identity checks.
required_markers = [
    "STAGE27-2A",
    "ALL 10 PREFROZEN MODEL FITS",
    "EXPECTED_PARENT =",
    "9bdf2c1fca362312855b8612c579da685a390ffe",
]

for marker in required_markers:
    if marker not in text:
        raise RuntimeError(
            f"Wrong/incomplete script uploaded. Missing marker: {marker!r}"
        )

# Catch accidental pasted console logs before executing anything.
bad_prefixes = [
    "STAGE27-1B ::",
    "STAGE27-1A ::",
    "Expected parent :",
    "Local HEAD      :",
]

first_nonempty = next(
    (
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ),
    "",
)

if any(
    first_nonempty.startswith(prefix)
    for prefix in bad_prefixes
):
    raise RuntimeError(
        "The uploaded file appears to contain console output, not Python source."
    )

# Parse the ENTIRE file before any execution.
compile(
    text,
    str(SCRIPT),
    "exec",
)

print("[PASS] Stage27-2A script syntax and identity verified.")
print()
print("Launching Stage27-2A...")
print("=" * 100)

exec(
    compile(
        text,
        str(SCRIPT),
        "exec",
    ),
    {
        "__name__": "__main__",
        "__file__": str(SCRIPT),
    },
)

FileNotFoundError: 
Stage27-2A script was not found:

    /kaggle/working/stage27_2a_all10_fits_wednesday_threshold_freeze.py

Upload the actual .py file to the Kaggle notebook first.
Do NOT paste the previous console output into a code cell.


In [24]:
# STAGE27-2A NO-UPLOAD BOOTSTRAP
# Paste this ENTIRE file into ONE Kaggle code cell and run it.
# It reconstructs the full Stage27-2A script locally, verifies SHA256, then executes it.

from pathlib import Path
import base64
import bz2
import hashlib

OUT = Path("/kaggle/working/stage27_2a_all10_fits_wednesday_threshold_freeze.py")
EXPECTED_SHA256 = "e134201f1d8770351a6b606910ab2db4e0584097fd18e3efacb0fef19127cf34"

PAYLOAD = r"""
QlpoOTFBWSZTWbXP6YoALpR/5X//ZAB5////P////7////9AAASACAAQAGBLHveqkE9bQDz7PVHubq2d5nzxe96dzVIoBUUhCUCbNL7Blg0D3r7wGAdb4il7
N7aSd3XrVt27Lx9vvs2QPqj61tWld9d3Roe8wO+zo+vpIufd7fe76u775t9t2e+s7tQ862Uylvu6d9d951Pp2GPer74PXN52uNzfV7S72k7esvPJVl7dzXc7
m2Nd33Lxvve596PDpAnxtrazZ6NG5oSrskxOA33neNfX3eB3Y7jexryx28pdN9e++fen1hKaQgARo0CaZAJkyqfiYaRptIFNNHqaNDIyHqPUeoDQGEpoIITQ
IIyaJtEaU0bQxTNIPUNGhkANAAAAAEgiiRTyJomR6j1PTRDTTQaDIAAAAGgBkbUAABJpJCCCT0TCmyYk01M1B7SmxQeoADQGjQAAGgABEkQg1MTTKTwIRT9P
U0ZGm1GqfpNMk2p5T0TRp6TMp6nimg9TyhoDQRJCBABAQGhVH6ZSPxKPU9TamIeoAAAAAAMgN4giodkDpB+wgB/XD7I/TE/5iBP8+uypuiw0kFKlYCyEqBUk
/lYVh7/z7JW7W2WAwoRVx/AcMaK6OxxZfYzEc0IjlxMzC4DEXBsTMouKw6yhGwK0HNhbCiSGk2Y1rRfZiOFlNJ/ghjiFdFqtSmRUMMsU/xTEKfEipRij/S6V
yA2h8SVADZOrJixQ8lhBIqSMvlsOyz1Gia59mw8pi+/YBsoq4EGVLQCq43dJWFeM1gunNrpmEFARnuZDHGTxdutLpd7+VhQFFFRUUUIIkigKCkYIisTTQQTz
fP807Un8ggYIQa2AjCCNSAkDGYqjiB/uhWGKhplX6PHznwMj3Rs4fs+N107OwnboR+H2KdMS8DSF0aDHy/Nco14Zy94PmUPuVc9nt7dUbYcSIOt/2fGKen1v
wV/5n4INcbQOob3d74bVdl8Riug8Q4qFj8j5e3eWr7bg6zD3nchh+c9BPxQbknr3ulDBp14I9ZwqwVBVg+ooHqcSRYKwwsDHkpOfV5mPBP3KJKKJdhCZCTDk
v3IcQHkE2TFVXpkejKrGrbaHxIZ6OM8zpgMib2zpge0TNUasXU9FnpM69t0G6cCdiYkDQ4kl5UmthJUhSliKUSNRaVsiCIbMk2QYnstNgZJDahYeDvl3QJZs
WxgBQSyb4SU3pUIiReaEKmUOtqJGmUxQZW6amJQUW0sQ1TDKMjaFwtSpXKXeDJAsGAOJsAejV4qxv9zIHkM0jzIaKS5BRWP4z6uCEoTmmUttVP2LV/s/QVD2
i4pJiFFhtQ5TCDm6ZgZtCzqhDoRRiVp4ZFXLy8ZGKxKC019Hc6lUTlfjhH1i5ThvsekLfK1ebVXI9ybpmrJRPeDxl5Mic9lHofDaZSr3uZGU6OGAHcZKGCw5
EK13DDMqWiyOVAwIZCPt7T+uk3XWcKe6ZoxLdQmqdRo5SVhNbhZNagajMFmU9YkUiKqbVifF3VkGiPJRMeEyqDAutzQikmp6CwKoCdWVu5mX15DEystDVxwC
qjErJWUuhmZC2NasEUYVilRpSoG+eLotkDZDapxDIhJ73YhIyLtUHgOcqTvPFYQ8+OJRiWYL30J66eJl6YhY9Tk4SaEaakrXn4yYBwMIw4ncgPPf529CAZHW
Jp4NFEfKhqAYIyCncOFChSm0LidzNCGLEGQ0CJRNCiUTKQCjKmNjTq7C2qloNDEiQ6oHtZkEeW5h50MNZEyLdF0MKOMzCEqcsQyDkeOtjKDIS2yWGWWqhLXL
EffO9vLty6785Mu83h7yngQ8ZkkTIeK3oyTjO9n25qePFsERE8aVERGO4hWm+Zcso2Yyyc2oxmd3ftr1GxgwkIVTDFhJgSqqk+HdHBO/77w3jy91pI/o6uE/
D8Ozc3wQH2ImQJGx3Y4YTv+1/2LlrHmSF+DzyVK3szHGHlcRQWoc0djm5m5RDFJzEDjGyAkRBGAsFCOe43ZmyNFs2WDjJRiqowUWAiwYZiZhhzwk3ZpQZm9P
RpTAgbIAF7MDKqirIrZVDrz++vW7awBHumD5Nn4+r4rF8QEOUjIBIMggTft3+iAIKIGEUR8SaQUFqKKBIAHTlfY/Gj/PnTg88rZVWiqC4qkVRxRpex92FQQr
FCIhVTLFXEFcsUohlADQTAvh9VXfF2zkVvlDK2X19JYgEitubmSTYSoxdEKFy5PAktYf77WqdrR/jezLKYc8sfR/xz+JlzMKJ3yVMuv1/3TW/12j/sNQmrGS
FXwcgMmdN/UL1iZTUQCoLOJ+efg5bSQ5T5adUnNnCSquFHhS6M7zribf1/LyMwn+kX+vm9/u4n4/I1nobXq6+diKFi3WwFPnL/tx+JFv2/HR2aMTsgQivh8U
atWn2+zabGqJvANvWPrMIhIEIL1Ydh9t3AnmoR8hkfdhch7PM+xv1Wk8LF2533NlX84bSOJ0JfX9l4M+HhjfRXlXSEwhHTF0QOSmcPBER31mW5HX0u75Uc1a
vu1JTuRTHjtKNtDCQzWU+Pt1tjqPJJmNocKTOeRSeXR6O0v8WO9sZrnPFWLnk+UGa4JIb3447NzWrcNcsVkLFaeX52u9Nimwrk5kmlomROXETskQnSnkklS8
3TpvJnWOMotENlGInprqRy3bVbmShy5Bk1MhB+z/vWD8Vtd9cmr7wT9YTUpOXzj5ljnXXBjvXazdurHKNs5KEpEpHs+w7QBrSajpfzf+d8ygvoO92jz88Pal
2Rr0lxpFYa/Tc13hJut819auu1zuHoqPMTkVR8uMCrGNBrz4bvwgGkqhrcgtM+RvD1K2Mzr0vDDEcooA2sTTyoLx2o2o0VdY37I3Paq/UweixuocTPot9iWb
sGs0ftZajGIrWrqd/S7/BWo5ts/PVdCftIZivZW5dx48ZVlhNVD6NU+/aGzlw4qrhz3308KKDHhmhCjS9FuHR9MbrrzCwzwpfPVS1cD5dKOYe+6vMeKjbjh0
fVf2RvwYCyyrVxutxu0aezxV+PHkzz5xkxqLb0UIx5Izj5Nlyrkx0LZDPx6mGZi6MczU0gtePQuk2w9teRvsstsZjuvvkr+JPmUxiy68qYu3tEaKItPptZ4/
wShhs3WcmWlPwz5bVOqoa8QVPbP4KnbT4Z+zQynObXl0ZfDLTpxTX8Mi2MWwIdEHoIQgTbu9XlHSZs5RR3wOoTEEyQWSKYbfHpr1MjXFdHjuzZuaqEJ1asKX
jzEtJleEkDcDOJhZ+wPSPoUe5THznXOxrp7hHui1Sm/zn13xMY27bqcVEnuIfRTuaW4Tlr10tJfm8ZHyTYmExhmvBRo3ux1njn7Neu2OttMdehcu7OhJmE4c
ds+BY8dcGt/ix4f2j0t35C7vnyRx19Gat31PT0Q+Mas+ixpGRComp5Lxt3ayjVSXg51Ps9N+Hqzbl1HbJyRXnXLJzT7IJ32RXLTeXe666ii4iGhdK15jGhoz
enGNefB95YQlLhFjV+s716N0l1GXN8u70abjGtmdqmTVtVRtutsu1PjBjCtkud9zeE4vDTWvoZzkJEpaIMSvlu6PJTX031EOHXTQ1CW9tiyZqWufTQ/NRK52
lzzSCiRfOlnI6GmpIyJp+ajfzRuMPum8SuyVV/uj+aFGtis8yBGLQDCCfypbYE00D7X76iWp5ghmbXrztk2uAwOzdOPcoKv0CsxcuWmLPAUhsYy9uHxn9HB5
+R3uWYoUavRfMyxA7Sg5uaNpcskFrdNZu+pA9ztwSSAWQ9eIIcRUm+FUMVq3pPRf2tQ8u5wqAR8zTwJ4lJ26PV9TZMJJWeikvqjKXHxIE/ZtnpBkgwbDB2bV
q8uo8yCBMwjLpyVuS+MxJ1EMEDJosx5h2aHGsfKfZURAzePnuU5TMTNXfTfJX1UtsmxV9ku96najRcwprJm2m72t4RjbNNHAu+0qaM1/KTt4YP8/UwgIyYDu
0Y7e8xt5UZDLfcSd7IpM78OwRcbMTx5yeyjkh7K3rMn228wDsKUZNN8ppQ5NGrlzxn9Rrup1LqoraUR79C8+uN90rIhnzWZPbMarp9kzmJFFJRUinmneXBb1
DffRSmQSElZGe+BDVW/EwqqqtcwMYg6NOdq66Wlc8MpZ69lJNLN6ttWgKR0zdTFF1Tx44+vGrNwMbbLKdOfzYmBZpq1zx3ym5Ylvdn04bKZ5+suHnejp7Fn+
ei+nDfXmtVx2BHlv24pomdWb4LWc77tk9xnvrJ2zzEM1aMcIgpxJxJuJxU4tVtPGfhwE+Fxa93Y8tpOsxfg4tOl5JP7Ay6VkNqhYcQ3/iOgtnO6GPt33y2X9
vX+ST4ZqX9kQPpij/eRf6f00+x8W38+7t7rSDIjIiSEB1OEo/KiYj2fHhlS7c3g0XLeH8hO0XjKfujJNIeuad89sBJITJAklrUU06iLvTD593Lj3UluFbi46
FjjVj6bHSqwZ0wVBzBCZjiDcxfYOj75VZ0lzsINzlamBytvJDIp+R20ZZvfjcWt7nfAt1xyy06CrVVLYYLFgNIRTCQWGGaBHK364QjXxjVH7Y7UxbbSNFWYj
u0zo/f0jC3wPiT2p0VNnPv34nsRRc9TpcWt23vjp5WYYYeiH4gZ6STabTnwjw2j/9HTo1uznj4csV0ReDL1rSX3wKVU6Itt8wxoH9I3N1zXRYdJAyTMrOmzl
ttHsPuVxM3qYr9N3XdvkMX/JPt5fOo4+OcDMdalYRvNWaRNMICr0EH6VmF4GZkFHgEG99zkWrQInE/r3HJlRAJIdE8FOW2a7SYujsulmems5KDu3vo5puB2B
zpKXsbJlydm1d23st6kiVtxDfe0eFrEWsxufmX7tnDyPWOXcPt2UQS0uJGuus7Tc8749fsx7K+K1vF7l7/H78NOdeznrpOuyZvopvm7c7WxS+eqgqhZtok0x
mz9RUothc5RmrePGg/oO70dBLogmEPBlV73FPimmb7G+uzo9aH0T4azAttVw5GVYNLUthGniZsZWcwxtvN4xY+GR6tZdxZkMTzDgZ1Ah8D5RpwcoMRpUFbcg
YNNUjKsJoNrk2toDgw8UOOTiHEoyEhnF2ZuFz993ozKScX4RysCvPrjtbK93RI8u47vliJpGYiBUPZCHMCtarTbjdzi3gmqvedzSKc1muqJyFrsCtkwHaPnd
o9NzL2nSPneeZjnKc17dOwks4bNJOWxjnOZsiySFcG29DJGypzoqRtu4M2zOXKRp188cauFo2jLR0plzIdWkoxonKDW1DMIDeXLjsnBK8eCxZdZjWxkw52W7
eMCdDjqvGnFo5Fucql5yED8EhMEAhwIpRtLbRZpKkqjBoPq5/KXC3rfr6i/QDUVAlHS+owoI9UWWQ+xSgw3hWjmBvKqz5R9vL1/HJeZh+NfPcjxzGoDIqdmY
Jp9cHExJAxGiW4QzoRq7oLiLBA24Fx6SaamG/AUDcLE2ed/jMIqx+b8Huzz0Ff8NhzdXKQshyC1QWsBPsqODpN4sMzXwSXLFjlI5DPy1LEmtMDsmGOZrytih
2dx7YfUh1eYjQc4kETMFmNUT26UFvHCCEzBqcSm0hhrFm6wdhiop/piNi+SPWojrFsJzgXn2ZosDHY1QcpdvgjaNKiNimZrWB2+anaSnYqbsT1maGHayahux
OPPnrfuPHs5cHS9c4Ow58HPmxR6UGyoNaJaVGYLBFEfUaZNJDKKTBfu2maMRRNkhUikUiOI2yGWioMlxpDSGRiyVJCtZRkcpUWDlJRFbSqGRPJZdWyRJ7dw0
kK1kKorulEwKxVpRxWSYhFBGTEhUJjJgg4y7QDEEw+/13DQp3vTjmGIUvIpwkyfYOob72+Y7N51SVTllTzWdGQ2dHprpmft8mLqFQzXSECr0A6BhLyZ4qVNM
yYJQzBkLFVVYFUMKiM+j9s+I224z1ZroOexc/TKIp0SmSYdIUO4fBr3A/TF97uKBsbLV9M2AvoN9C+NDE85Q9DB11cJyqYmKx7YdTb66J/rn+/70UggEbCPZ
1+svYmCidvP9rxOXRgzQAmDAcYgyJlEEx9jOPqtsxZRZtjtUsw51vCOvuj6lOxaMu1UQTIn9+0EbctZuIBm313t6ulxWc0Fi4r/LetWk8sF1HD9Hjh4iHdS6
gtKmk9Q8KyZ3nyyVaw9mxWHjC4Enht+LodNTJk7qJ2PRyc3+Y9ZgUkiTVirP3QvgMVqi3KMc7O+ACspsqbjA6Fn2PTfTQD0pJmlWB+cd8TF529shTMGMCRHQ
Trpw9NidOGQ7l5b5jjtlikzM4iuEkIkdu5BXZiuImi7FppTwljdyEhy3GA4m0CaQhihJK9c1totozWmsB35ZFieeeL1w1zu+smx5d6Dqe/eWQ6lOxhXd4Mrm
uxEqlzd+a42vpi0YEmIgOiIOjzmaeaeE0U9Q95vGNiHDrzzFBu3Tjoxse+c1t+j4dlbSxwiuRpPrmDUs0Va7qF1+bWzHsbt70k3f3qmS9R0Rs0Pr5P9nIFkR
+FxQPzn4e15fkk6MDe/JPjPq+Pfp4HpFB8dJn1ZUeCL/hJs38JJ9XiaYsPeCxjCwXRJ12+ennqgUCTiJvuUJnb9Ppx8u/v7PCWfo/L3yJtMeTOmm+MZqdY2r
Tbu+/9PQ4eXoRFl6939AjNxd5eLcL4x1xYwsJAkOhy5Mi1v97stj6J1tv42PPbMovdcB0mRKbqmwuZK7TCDMvvcHh3hqUdsnPfSZrbo6U3M7skJ/UKdDNqII
4e4REM/6lqcpaikF8/hfSiodhrl1El6BxLVM7NHuL/P+xz7EvU+nKPY8t91yYBIWzceK3PwTEc0j2LF7MNo3o7jsIUGsWsx9Nu7JsUbvCraa+j59deVunXML
Xuag+nLQi72uHUlhe5PEi4uHQ43v79vSC+b0lzsgJdbxpahC8khTgOep4CQxw97cqsOv2QsPNbSsm26cc8JzTSTzNUA8db49cp4NNws3od/vU3I2+Wy2hifl
t1l7wNMNYGCnn1MMDSLFK+77zqGcUs6PQ+xEpNEPaMbOVB5HkvPbyDCKXozTRWIEnmcXXAubBvqJzz1pz3jzqdwzCzNMPmX6JPYpuWcIEyZv0wDTA2RS5uH9
tqC5T7xxOUsXneaqxE7oJUsEeLIop2lNYmZB/PHt0LJpNtyC8nsTFF9ED92uFjlB9xowMTziTonw8dNnXvpMM+y/ERfrBi945wv8KifsgBgB8UQyBIgSAQCQ
YpEiam2c4wPRFOPujYiHRi+aFHI7onMfZuaz6Z8iDIIxiqIiT3Hpzr4BlpCosN4oDV7ngSRG5yvR3wsnnbDs5bH/voaZJaWo9SAr8zr8blvQ9PHZsok20Oeh
4UeinANS7qgZeHHdCQFd1eVd+vhjUmaIddrIevKEGg2vnLjYU2z0SbG4IiEVPnOiHJsz5RZIJ2W64o9fs9fS/1JoJ+8e36I+SsWvN4jX3ni1R6VKzLfTN/nV
209jnNdc6na9sTGuqn1rpv1TdMM9eF5jQrygae05fV7qKRIhCrQWMnrvs94Gk845obs42g4xB6a3hNRuIPp4dff7+31fB5KA5a++2/D6fpkP2/d/ff2+PoCK
zx+1E7F8fufgKLbbb5igRBfjod/meTNZ27oujPa8Oxh5hZCEqDXTL2enAoAiRIfXT4NMpvGM47tllvE30UI76D2B+qaDZFBYp6qWQ24ODNIdrYIaPPMfWTpo
zfl+W+rcL5sHNe54/k5LTPBlv5B4d2h5E+6Ewkvcj4I/hFXSWt+vt417oboKMbun6jyJyz1tvZcMGE748nNn7/q6ztG/m26t0fwjD8oEGoQFKD74zJFWWf3j
A0YjVRf1kJkCCyCwGMigshpIVkhEQFWCxGIrIhYg1GEU6jecTe+M2buIR+Ggr+fPWdh++g3kXwVUgVFB3gA9Xg2hPr2zwuIAywmygoyMhbk381v3ai7v+iit
Pip6vnd0F79b4fQdU7gAjD4oJRP3HiH6QaPXYQT0N/xqFPIUNB6ftIXiefYaAFxT19cMaNYhT/wi/Y6Bw5QrRyDlgB9ohga0f6j47Wge/++HGKZ2FwF1ITrm
usNDAZbRAj2P9wu8l1U5PB3wM5Z47bWYMCC8gn4wz2hRt3TVkCc5BsXCCSy/f0MBgwxM0p5tlrpwP1GTB8BDNUMVywAkbBsEzCk+jH7jQQohyFtLpQsHUBfq
IQU1g0ublk0hi4NKwIX+b9MrTHKMJa8obBMYmsg5KWRYukiXGygQDYdnaPLmIcLivbmgS4djiaSRyZJlyalURtCoJKujpRdhxiVFNw4BY0r9VcVHwNhPXoPB
2BhwUsC0ByzAy5AFh5NJi3YBTCoIQCCE0HaIYhsQ/h1amXT+vo/bGzWHmL2sempTlFscXfGWrDOs2dpM3VmJUPiVsCNjNyTUMxPkn/cS6uAmqLZmY+IGdsaZ
FrF3H75OHCopNq+LyO2DYDVrCWGA4AglInqsMLcHSwJyYkZ9ep2QMMDTBMUIhto2N4SBcg2TcBnCjKSKcn6OxroYAdTaXIurV+yUdtUy8c5iBwwhc7vcOs93
C27agxNgMcADhA1HSLmnSRwsjg7B3AdIE75wpJ3J23Lt599VVVW3sU5ch5ARDy8CSd3shH37vzRUgUPqh9DsGd0xhr2JNiYQ3Cd472gpU0B4eDtPVh1+tNXJ
zyvZiKxLNbvtZA4SaXR1bEZ1wwNg7TtDR9UFPRI+gTCMSbwUiCqIwIzclK/yPvJwHnGDZB1gL1p2SEJAgkeCjz2rY3ci4NWdgH3kFgFPztJJDKGZjjqCQYAw
S85IGIndnYqgwCfMOb9B5FA5MRZ9TLI2dUOEck4nN1BgQDGMx1KKE9hoDqlm7zNYvRZw1NONoFEHh9TKDo4yYgEYKR+f0Oj0B2RJB7R5jcWzNBcfU1HHIOEy
i1E3GFJayBxIthW0bFmYOxE9/JXRsdTBceYJ28adQiUSUQorztUEMB2xkiHLyQx7b+Mf5bs8XgPkOquTywc3gDoi3T9GEy5AbGSwi2OxHGLwwJ6wqk7WCWEh
BCa5yRjDepiIqTqxODPt5rUCBu4OIXZttrhgSSkc9hB2nEDbtgJw3O/aUFY9UCiKdlzDIlTBlVKIIWqjPxDVDzWbQzaAFMYNHmPvA3YBC+HIDcGJIPkPOrmD
OKSMzp7IWpuBeyFigtcN4XLKm5rbSRG75I/CHgNRrxDQIF68VHngD1+AHcZAJGDAsDZHogesZADcQOB2ZNxkJdHIgPiAXUy6euhtruMoVe0hEsu9wByDiBXR
E3rJy5m3HCZ2Nwz1X19we68n3S1z2xmutAmKhtOO+xn60U6P8UClUxPCi9LYDFjc5Ju4yG5ACziph2ccYW+1mlAcRr0ISoMiX2PnBPb5EhIhyyM/A4sG/jgD
1PWl3pyH7APBAooBlBJo0vyLDR7voO9HpdpDgyKRIMGKaToHm8RWKA8dDoT0xKLkW+FAtKTUW/dTn92fjAuVWPoAUHWfdqLs9HhSafZ6qnVzQuam3Q6KX2QN
kNRIDg0E1AzA6cpuuG2juLyQIKpr2+pr7qro3ISPvGK4aNdr4KWHW8aai8LVztc0Z4ZYTKOrW3ICNrDEI8z2unw/gaeaaAJWhcU4CfOQCIbkMt4tZmOGG2k/
2CGwMmumhom0HRzfPwlDwBLgXkjLhKCwSJ76o2aM00DHYO6B279SztrIrIjGQICZlgyHHUZazr6yItFsoZNRZHnV7OO5UOrvzWGtTPQl3ukzpGRqEQ9ok6kt
DQ9a7sNQoHAOy9u0vvhudCbzahIb5u4AZkBtEOF+HD0M9TSbn5mvQG4odtm8cmDsSHPpPtbRDGIFg59kIHEgGSazpJypLRBgEQxW3R2/VLAOBTHcMYaLduiT
udHhIh5cw5rCaHiwJuJA2A2QWKUBJqSxmh3tq0ziO3UGD2gdsPADtiKrB9kD2i2IEDWAbBzH2Af4DfSmOoHqHp0Km4Tl4w1dPtRzzKeLQlhGwwcwckPFTyo9
jtYzzU9BoOwVILnOnnypNDwc8TKtFFWK81Gmuvx1VQ1htgPS4OhDHPNL3oqmYqsdFw77zRliWwOGjQ0TfGMJRdJONJ3EIQtaCJRygqhtompFlUXtPBBRDywC
zuD5fq/R/QQZfUrNxX549+fnAaoyI2UbCcIy+M4pfubFo+tFXumsWkTF2PNvFtPgtgsgP8CMkM+H6Z8nl+RO5BnvEy/a65uEeEhvculkPodhWu0hxOGokbSl
yP42fhAioYmKDPmPVBxa0EdSBj669GAdLAQzzBA/PP9TnvIER4EW19wIGKBxvnSUAw8A+5BEpJommFGFbLT7x00aZBh7kCGBxXSA6A+FBWpglNFH7BNQOf81
GRquSwRWIWU0AZPzuvrDk5Sk+7ZyUh2+Hu9EjC6HywD7msn7NTnq8+oT3D3fd84FhcRk+v5ryCksIgoxGQoJNgKwwMFIwdf+cj+0xo2MPU8Do7X8N1g0Iuoh
g4bZE4Z3KTITZj3NHuV5m0g+iCRL5Jp050cHb1L0xGamprrPEjRSn00Lcw+js5xB/N5k80sXYTIDchuebSHJcaFrZ1xbQklRL30byVUrYNYcHW8xidKeN8kA
8ntC0+ZPL3TG7+g8k+yGoG5N9bxBIVJ05AoqlyfeG7AfdlhPXhz2CfhKdZ+OWgaqbRVSxA5E64uxMf4hGeHz1pBQUSd/wOJf26kkqGrrpqa5nYzjICy4hYdi
1FRZ3OoNcBWAwCJbjZEHfwH6s9SGMPPXa2QK1CBAkNBqUmtO2rNNOBkTQ4ga8Ntc9n8KVra1R7MpOgcibCjBVihgcS9JeVW+CdxveELuKjhx4cHZM0ZlXH0m
s0SHOGo3umsMxeAZFLO8G84X0g4gcZZRloHS06zVOkS10yZJKoilEWRKLEIUnaHHHfG0ba0tOyglO2sYahgYBg5DYLFmLsOku3B2AbN0LTdPMzEpOm5kYQ5m
lUWnhxhNoCdJ2/XCzTFSGjwq3hhyjJ3h0nPlaNKNiNG73MW52maXNK5MnIKHRBSyG5oiDsHTntC2WB0DjCYDtaKjHJuYc6bZNoqiKtwm/hpDAaOZMyrWMHQP
3tQkGEUs7/3iBoxzDQ7LpoEhCLtmU4I6nmL3ou60bSRkkbNqKgWSV6ji/angMsCt5xTBaii8D3SZGYfDxQt+GgBqgft7InYubAnkeue9YRVsOCg+mONPZVZR
W1mJkJfCGoYQPycxIen018HyV4OmQWBA54jSJrp04QuWJE6i6YE3G2KoRIL6oGB27TiYkjwrKzRDFVD4kyMgTBQdMJIdTynx686hWRiU0c6FwuOP4FzJpUP1
QRf1gFhy5SE+cryMtOEen4+tltu2snUcVArkXE5i1Sd6G9SeKitkCyOn22oyPDU1TxnGF3V1DANPiiY2C9Pb0O2ffvWdiNbAEmfBD6fMR6FON43VNvtwkCBI
RE0GSCt8DwfGcxtovxB3o8y2huACls+2uvzbhhhGoFJl5HXtUZDCcxMiHl07Kx7q8Yn2yTMNIQ8lDbgJQgXQsocl2O9hrCFznLutOYQ+aDIQGIxIIKMRkkRh
ugb3fkJwVQd5iP9NHUYwj2RQnFMV3ETRoAzVx9KDxPcX0km1TjzIEIvjE5qm5A0seUU8cmjIiVzhKc6+MzzP5YctoCRUhAgBwK6/Unk1afSzdLQI0gfp+4oQ
uDE/cKChTU4SPnZDvb4gkOJ/wK7MEgz8UCqIsPKWG0cCzUAutTe4cg2Cd4wWIjErc+TIk4iu5jdAcZQ2u1kzw4qECzSFtcbPEepAkUjBJwOjC5x6u3z8aO/L
TqAm5GZLJcthcJiXv+fzBpyLjpTQhFbEPGdGfIycY0h9rBcMgU4jodIKZIRWvuj3IQkhC5oCSoSHnUpCwRs/kbtAHCGDUNPAuRAyDVQh9eJ629KYUloSvDrd
TrFbwshjRt15cz6kWENvFAE70FiUpKqFRLK2mkgnMLCLU0HXbhDvVzhDYOva3GUnAPcZ4B1AoeCQ64sEp17L0cbjZdXDuhvN9TaCVhkGCTIIj2mrNCok6GXe
E0URq4YOQzeroaN22Uj4RwMOVVeTyRUj4pD32BjklVIV77JYwjDAigQrIUSwFsKJEQRCkD4BCc+hnVx34BgYpAN/passwdyr2I02JlySBJEJBjNeMJHaqaSs
RJCNBIQbBazBkdaC0W4ZjSYzLBNqSGtEM2oe2uChYpdCDCReJhrozBoIa4qmi8ZMbw871s5zgY0Th1YJirbSqMy+630M1y5XF5mUANIBoQFHeEbQg22wEBkY
JEFgycgsECloCFZB/FPsnk35gH2p5JzAA8yqqiyCijEVME7xDquQK6UDGwHE40jGVIQAToG5bc+CGkqCnuMBkekhRCsadDsB0UIC5ZQOtyDUCayg0kHuAxTp
TSAVphCcnekSQ2i4hoSM2nXofA6lFHw3R39iNWMC0DsEWiJp2ClivoZY2hbt7oHK7SRuUUu7qmnjohcTI7ndi/kUmUTmHHIxMCoZmYfViFt36aFYursaKZVS
vhH2qdvKjOjzWjPtqU0WWbbnK9XqOHHJvbU1oe4ZYqFi7ybD2zoUU6NIaapWVEcshtO0DzeJWw5swsy8feiGkfCaN4eDZprih01MslSb30LCVxXunzYlrign
CYLcEmZQiAqHFqgoozkci91fth38uu12ce+8iNdW6bYaQpt2Bg/KgpjdDiMqIdIjMRHSg0sWejYLCzZFSri1GOgWDiW4hCnJnQJWQSgvIROS50ZEMGY0YTim
aXdQm8wNuDmF0YgIqG5yPxmQqQmdspIUaKOkOkLoIe5AE4OQTzkkEM9wBcDxxBM0ANaULcTQEkVgxIkQEwz7jSMe8Bu/qUS/lz2XYqXfY1e04t270GBNO4YH
R411yaHnlh+8hui7EvTJHCweOMUMkGZdKpTRCSEBY0tHvEcAy6HXo2xUm1JYQiEd1BwIpcfvp4BkUmCxhLJUDfKkJCTcBkC2OIGg89xg5GxSicRCKPxUPU6r
ubXNkU0QcyykJCQbxYFgyVCVFklZYDCkSCxYiNERsvy9vX3eJMnmQmxRxMTv6jXFkJqv2nmt+3yaXBnK5gZ9LzpYvrI8gdEIjW6hMXpIj6FkzNFvPv4Hcfg6
mInuc/BQsqcWafhmcjnOGQo04rYBRKI8IkeIXaBoHdDug++RvddHubWA+MkhE2sHxMo28bdpgrCVYgyuN0UvbIGTdFRovBARJ4Y3UkcROFENAwmVxp4gjo5n
ArFsDBoeYGBsFjgB+ZA0T9ey4fRNN1psFPCGh5MEcjbhYyZmD0HM4pZ9su+70PJ0grS32WLFtvUY+W8VvUs2iLHAW1Flw1LghZaSwWtYMDNDAwWMwIOHO7FP
PrlmIeWpnoQAJAXslkA3KQ2MpTboUdZNVaxbfFVkO6HTytUEAh1MJZ7J60K5V+GROpB3h7bxafay45xy24PeUJ9F9QE93YnadLfT3o+OZSSBInwi0RAkFEZI
KHpQCqiJD9pJUU7Nd/LvQGIqB4MKp8nyMsViKsYkMBNBYlltCQmKTq3t4lNFFNcz8f1XPgcRQr36bCbg34UxgdwWMgMZNuH6iiEyDHqesFH8zp86VtMBc2Xl
wsWJqpqDtAyHIZv0BMwEA3Sc4QRIie8MDD4zz3W2FLKIWlZbdDxFRWMp2+Qlh26bu52wVsNJ5VomkwzM4k4iDrfKzkMDkILwhug8bBQxHECCBQWhzQGPx6xo
qwRGkjkziDR1DM6FaIgZBDpOhM4TRqzB4d6D1O06Wa+ZJk1hdTRNZ1Nih2NgHVnDJxDWpbKKo7xTCBwXdyG108p7p/r6DExCYVRBAqqqCBHRbu9VkkhCEYyQ
knBIOe/ZkhpYqJGJc4fMLhA2cWgdze4aIwWD5BEqqKBLt0VygQ7jmfe5eF5w4WtVtFDfh2OAU2mPO3yOouWHNHWTUpyqOBEe8wnkJTRAUiBHIghoZeLFQJcK
QLjmMDtfBIBIkALctLIAZyF3iOTX571wvBtkWcX4ZCBUjaKgUrMswsmBBQOZUciyFkydqc+kOZ+zqAeQG51ka+V4kiKIICeIL00VnzDFijERj66Fm79wN8P0
yelAGQm4ExiLcvjVGPLYChABIpuPbCGu6TuKRMjdCq0liQ9e3nPQPMsfLWFCYwo6Nnonq8YtejkIDJA4ZcpRUGlSUSX1OCYSsbKHQkDiDq4JYUzd6R2Dd8CI
elIuxJAwHSofpdTQeqV7Q/O6jaIH47ZoF/yuIrwdw6gAHJ+cQ6C6H2c6X5RJAwg1BKiREY+P1dRwNmMdkIelgBwQO5Hs5Ta+49dMl8Q7mVBqc8h1AkSRfF8c
Zx59sXJ/P+NrcuC0j8+aUYMztpMzhRHUsOpbowM0kajEzc94HQBLY8SiC6RBKiJBCAQjGBzNHZYQ9smDp9gOoCwrxdAUDurEVDrgKsFYkYhBZIIjGQSAopGI
kgRkidvHZJ4jK97IEiB2Aym9euiltyGkV6dKS0khIQ5gUGCg6oqQ5nRWOxToQLTxQefv9hIBwew5ONwDefAPbaLJ+ozGVJ1U5ESUnV9IzDahbZBLQXoGwByf
vXQOCY4g+Zcp0TiwhEd+1RwMYECOTaOUUzLlhJEwUhAIEYPzDl8JwJBH3qNH1hQTx84AfGAMAg5kTtdwFJj4dPf1NGjaw8boDlQNN5t0OFF4z2R6zXnZsVCp
ggXh7RsRJIXpCNMcestiigwZaoqEj0aVB4BEaMgFjIIwYCEQyhRgFXC5IgaTUlPTd1GBGMIyjb0lxUzeRYOwfbEZkxJjXcjh9Tn0C8CQQuxVIMFyyTrAIz5w
LZN0T3YxbEs3+QWwJhXpwMP5SdKR8TCuMQTDmgBEW7O6GoZcgDGOgTAyWGGpItooKZypGyEkLzFAuCGVk0/252OmPs5yEa0B2CHQoZoA55mJzPzWdpcOcx4a
6CER0qP80kR1sE8p4g0gkk4nfRHsE5gswDAfAPQczV6EIJwiFKBAK+KFohFyOASzNzb2mVjuiPkFZmKPcEVcohABDwoA0CSE6G+Yczu7LKk0rKXABRNUmetu
Q4bDOOjhOHJo6i5Ibq1Az+2DXjYVmpgJzPKLbjsbMfNmDjEzlrVa1SKC8d2JCRNjbwXZLjuNqhspkZwPbLa+4yBVyM2APFHIC+WmhcrTveYbUBiZTzst7L2I
4NFJsgbJCYtZMoruLEkKZhgkFdAlxI0bgwG7DggyCR3VCohgjDFFQ7yxcM8HMOS+C0ZnEzo8dNpqeLHhS+550zEg1vhQIJiYzwyZ6W4VvI1I+MjYXAhIPqxY
1YtZvkokUNDJqO1gpXBaaMTfA/5UUFEkMYGkMSKc9rszplJOGdgWqglC1YxQ5UaWnNu/LitzQKnRhWHMEBpWbInBTOkSTaMzsT3k6pz1c91ZZ1iJnKmJwDId
LULlkUMojQuYR0RIbKqiqRQYwBJuYUx1ZJS048vHydPA3hwzmB2qXMUqTCIxWlnKEy37KqxrKswdyEIxgQJui2NR7bKPqiQRuwBMz2Fhsf33aDCtLobKwEjB
EOA0EpXWPBlwuDcEsQkN1JQhANAu6/2M1bIl5sOb193ILmA8ihTEX1gpwh1j7g+9aAf3AhkXJYLDnH6wAuyKfUrAw0PA4JAhJOZ0e8OsUsSxFO5owRkYmmtr
GgtAo0gQqSEUgDFFYREkGAkSTQwsKQMld9xS5cUWePC43LqUBLCUBGBybc9WC4KRMdIeBaDsfAHT8Xr85dgdGHv0HfIYNME1njBbQBYyAARDf1cx/o4aCA9x
76UGitw/A/ARGPFPPp+P9Defkivl8gOFbTeBTyPIkWQFiCsIxAEYKCJxIGOoMlcbfJH1HuKUJIRAOvKIptlMkIgqxgsjIwUOiWALAVn1PwIGTcwqoA6kQSwQ
E0TA9MBKWwNjangFg1+WSSGpdIBjlEU45B5w/USSEQkQk6Cn8c2D6Q8qGFB9aZRsZAeKNJR5RgCMnrS7WApKMr4oSQwOQIYcAcBOA+E5BwfcLZPnofNBODnA
UERQb7ejketH3XJHYAbFTrVhFJSEDTBm2atPUMK9A+QlQA8lB/ZsFehNswyeO3RAujxAsn4J76+Eh9QNnzgVrXDbhJFOg0sIs7AzV8O1+brmMhTaCZKKiZYc
cjvl/HkotaUxa0uEosW6aomsQJEXh6VKlKmehNqpkzHM01aA2rt09I9HnZLEgPvhwOaWlgOi+tvAq5IP3MTBEMDo0yA/jBLEYSEhfExunG+NijGgu433vvm5
jDlKoiBJGqqC1QToI/uihge58OvBso9Sg2hrsW4hkVoBoZ9NaNu4i4Dg/pFHEhHcg6SWDLtV/lPAxrETeVcTJ0Tr2t3ya69QOZ6HpByg+OWM6vqJ9mdFE/9n
ZIkVYTDghkCZuicthgOERKTCBDeEOchyCmQZksLujg6PaIIjRzMKF4NgzRhBFA4aa+oSnz7vZChSTrAse9P5gQ8Ef19zCWnrGygD2+vzgYgsiz0MRJVGKsGS
IiAyCkioIojGwYaYxrOsmpCHJDNa1o0WUkiRhEFpCKosmmiViyiCDvGIWGi4RYmXywwFGBiwIwnBEwjQxUDsqnYAO6Phn1Ae4cTQPDZcYrcMLW4oufvTIdVh
uZhWKqyB9JAZIe8w20yhedoUwxqJ3tUSMA6QNx2PIDvY84PgnhhPYuTt18vHB8HoDWjblu5zlERAuXBFUe6QasCHNZm+SiKFJQL1nUdFHwQYRFvlB/QVv2uL
sXHdvphB5ZKV2JaOgeWNAL+Z7oibl1L5aNG1PjbInn6T+MqBqa8oZQ8YbElqdeCRZExL7dZkTl3AbUQyhQBwB4o8R4pYC47x0aUvvaQQKT1Gu1DJZgJkAbgJ
5wGrDCKQhFcNFIqUnpmCB3e8fDv6SsBx5q8FPRFTmGA7I7tXWQ+JXttnofP3ghacJDt8ygxQRVVJ4eOAsknk+mEkMPKYhQStKLSsEpI6EAEDJFBEqQqKIAxY
AoMUNDiWt27n6FISQkQgb3gHKHr8N3S/DOkix4dcqrW1/Pz05c5CFShg0paAtvlOv6OmAr3FHk5SCq+TzjQVvPPvvjWz+vDxs+sOa1Uxyk+SPOKXDUeF5und
LiaawcMHI7SdMt0ix5PfnA7vBbze/ItU2d1TmrU9Kkz9pN25Kje2afosLdGiZmSQkjEwfkl0sUQGUOiF3NqWKx0xFKplxcCal4h4fepo2s7lh95rSwkWvUZC
YVtnsTSZMi1YriJKjCDkk6THPGcWrLkt7iHOnBip3uxKnKcgWga3mx3444bft1IXBOhYtiGZJS16iWInfUX3jzGtQ1Z3BjtCh5ANCCUghGRuKTC5mzMHJpqJ
m3KneOj3j1kDaMaTiGeIYJmQCsSnWFF844Bu3JFgQiQNwdGelFSzhaFwsi1BcUyXdJVFGu3VomacEhkx2klrDOcj2cTRiaHIgsgIDlq7RyL3d8mBU0J0jCsx
flLBCSEiaOa82B1VAwdzF3xN6Wm8kg5Fk0G9JStN7Fl6d1ccU3wOcqGq8Zz610ta1sBIQJRA0ZoFp4Dedyqw6DSatIGhYEiRhwicZdgPTk1i9Ovgw6tNMQqV
E4gV8RlznDEFVhsZdlMJAGBnUI7DvJefE3Oth78aJGFb5cxVMneuXLTF3mRplop/IMNO5TCU57yxE6adhe08Ttu9xJh3jK1MJ99h4xxgWmogQzI9mbkFjbf6
LJH3KSux6uVOG0HnYvkRzGC3mCYa8Jhs0fj/z/Ws4rHtPgbeZBEN7vk+O+LfJm3TyTQWNKKSxqZcnWVEWKbDfcze1wtsXNBhauOUWiJ09k2mtk6iIUg7Bd9b
vLWqLvOhi0We8FtqvbFKMVSkvOR0UqoloHlXoIYdsuVOzJmt9cgYItTREkLFZkVphaXIi0EEOrD7BtYyPjKLIei9iiRpBMKBOVL2os4suIohokQMijRsM0pS
pILinIckQSYwaVTaXiWBkFkMIbUJJnBCY4ZH2Jos5jheHaCRQi2WlPpLZUK1DlPLqyaklOpeTGOnlsxhMDIEgNDYZ3OUGgjKatZJTGm3AskwbiECGNG3m04d
s3oQuBi6bJkBlSSKXHF1Nqc7hiBYbrWg6JYMJh0gSF8C5CZ5scZlBuLJ3FMgP3RgEQSABodEoOgHsfSivYiNxUyTfu0ugcheChApK4R4kYQPwysBZhOVnnog
HpoQmiTlLy0OGhTyMlhDjQwHQJ17MQYDwKhtHNBcBsR0XFzhk64DWCwfyYSqqURa2GUaFzvMJbgVCotQKdQ+u5sndIq7z76GiISCAdh7QTrag1CttUb5k+0T
RTNtYPSX8b1e7aJmjhYse+biEJKqGsoNQphBy0DuGYHAd720snX6VRbfYAWqwPjtXlDKwGsjEhIaGSpRAEYsOTMcSgwiNiFgkokDrBKtYbfGa0sUEQNE7+7s
NiBnyPY8A6BkG/iK07khBZCQNN7E239/hAdCdzjMkCO2h+BwYA0YawxcQyLfK4aY0hKl8dazT2HY2UXpCHpEJrSTEmrdH7GDmjcChkt2B25c2Qn8VUeXYrGK
cp3Ht6oaJxGmSvEpOchrkuFNwCA7kspqPMPNXX2ClwzF4WHKIG9EiBxMhz2qQA+9OUnBlJtSCGvXkp0C/RiGas17PKob7byKsD/yX2cq3X1fZdiPE0uYheCE
MwKEzGb2iwYB69IniWo4myBYf7SKkUPmfVLr0YPn2GP5nbJIkhImQ4gci4kQLuwSkSXTYLZKgkhHZ1G0T0c3g4nPGaO95+nK501SnjuRoLJzTU+Do/0Yh1cK
FZhwnMPsR3AepAdentXm5Cb3TEy4MUm4j+tyvfa82ZTM3tVWtL3sXtcyMBVHd1+wnPuU0wcGRDgPHA874eWZQ7XF4b+WDTZWTcpnEtuIXqkHq30HFyiQzaQK
gMxIa91G466yxX123h3BWRpnnBmicqwfA63Zoe1CTCEAEgy+zt9n3XzDaWosYJqICRgSCPiInUr2cKbnSWlcTX1NsePFemEsj8trUNSeA8/dEZ3I2WCGQNZr
0sUmuDOBJdEN2YCROz00DhWG2wLiTG+JIbnrln0KMZuWQOjNkS2ATGBaAsNYyZLs4mucZ/OgigNbjNBoGmAoLF64mZ5lF/PAWwF6PCxT5F9A2d9FKh7Qr4nV
7gGO0UfCpchCRYERYAHWi8TXC7SPAJGRnBnr7OVmi0IBRBE9otP6w+lSFFCl3GE/Wf+T01MauQSSmkqqp3R8zyeCBFzfVk3/PzBxH1JJ9OIjm/V9MaZg4FL8
6aHe4dVgSSEJv69wen7Xm9BHjtRCESQhE0iUwtQ91YeLmxMQkDqOf+K//L/GwH/4u5IpwoSFrn9MUA==
"""

source_bytes = bz2.decompress(base64.b64decode(PAYLOAD))
actual_sha256 = hashlib.sha256(source_bytes).hexdigest()

if actual_sha256 != EXPECTED_SHA256:
    raise RuntimeError(
        f"Embedded Stage27-2A payload SHA mismatch: {actual_sha256} != {EXPECTED_SHA256}"
    )

OUT.write_bytes(source_bytes)

print("Reconstructed :", OUT)
print("Bytes         :", OUT.stat().st_size)
print("SHA256        :", actual_sha256)

source_text = source_bytes.decode("utf-8")
compiled = compile(source_text, str(OUT), "exec")

print("[PASS] Full Stage27-2A script reconstructed and syntax-verified.")
print("Launching Stage27-2A...")
print("=" * 100)

exec(
    compiled,
    {
        "__name__": "__main__",
        "__file__": str(OUT),
    },
)


Reconstructed : /kaggle/working/stage27_2a_all10_fits_wednesday_threshold_freeze.py
Bytes         : 82742
SHA256        : e134201f1d8770351a6b606910ab2db4e0584097fd18e3efacb0fef19127cf34
[PASS] Full Stage27-2A script reconstructed and syntax-verified.
Launching Stage27-2A...

STAGE27-2A :: SCIENTIFIC-PARENT GATE
Expected parent : 9bdf2c1fca362312855b8612c579da685a390ffe
Local HEAD      : 9bdf2c1fca362312855b8612c579da685a390ffe
origin/main     : 9bdf2c1fca362312855b8612c579da685a390ffe
Git clean       : True
[PASS] Stage27-1B is the clean scientific parent.

STAGE27-2A :: LOAD FROZEN EXECUTION CONTRACT
Eligible folds        : 5
Learners/fold         : 2
Authorized total fits : 10
Primary device        : CPU
Target openings       : 0 / 5
[PASS] Frozen Stage27 execution contract loaded.

STAGE27-2A :: ENVIRONMENT GATE
python        : 3.12.13
numpy         : 2.0.2
scikit_learn  : 1.6.1
xgboost       : 3.2.0
lightgbm      : 4.6.0
joblib        : 1.5.3
CUDA_VISIBLE_DEVICES: ''
[PASS] Inheri

RuntimeError: Command failed (1): git add -- results/stage27_loao_unseen_attack/stage27_2a_preopening_models/validation/INFILTRATION_XGBOOST_wednesday_known_validation_prob_float32.npy

STDOUT:


STDERR:
The following paths are ignored by one of your .gitignore files:
results/stage27_loao_unseen_attack/stage27_2a_preopening_models/validation/INFILTRATION_XGBOOST_wednesday_known_validation_prob_float32.npy
hint: Use -f if you really want to add them.
hint: Turn this message off by running
hint: "git config advice.addIgnoredFile false"


In [25]:
# STAGE27-2A GITIGNORE RECOVERY — NO-UPLOAD BOOTSTRAP
# Paste this ENTIRE file into ONE new Kaggle code cell and run it.
# ZERO model fits. ZERO inference. ZERO target openings.

from pathlib import Path
import base64
import bz2
import hashlib

OUT = Path("/kaggle/working/stage27_2a_gitignore_recovery_commit_push_verify.py")
EXPECTED_SHA256 = "214a57af8e3f97e7d0bb8e36e2107f6e20fd44a82173f8c9b026a47838b3e4b6"

PAYLOAD = r"""
QlpoOTFBWSZTWYOp9UAACdp/4X/0wAB+///7f////7////5AAAQAEABgHT7wNPncejWC8wYe5d3KFidNOvbdPdvVuZBo00E7Ntru3dji4EvMyo6p7tVLa1ZT
urhwyAairW7y71e3e7pXs1p65glNEIIATaCNJtVPwTE00CniTepmqep6TGU9J6jymntUNANNGg00CEyIBJijyTEyMQZHqAAAAAA0AAAcDQNANA0NABoZDTQA
00A0AAZDEABoJNJImhAqfkCmTR4kaR6TaJtTJk9qmRp6QAAAyaAGjCKIkyGg1NU/Jqnmk9QnqNDIA09QNHoRoNDRoGmho0AAiSQBAJoBoRMEFPSaPIeqMQNB
6hpoAAA9QeoZ0QB2K+Dxaoo6FgPqhDukCExPimSGqF93LHU/GA5Bgf+8csm399mIKH+STokDGbev4T+d9tqh/4kNPcyG0vLjRXArcRHZPlO+Bfy9/Bvtvo1x
qmuMxURRplMgxU3KVysVgVUBYsUBRZUqoi6wCZgNaU1mZLmzFE6nM7cCuiFN+jKTU6HBzY7ch/0cECOqXqcBmm7nUvMzTuJhYqSYmpTt98sbB7fLL9X7A6ox
sGDnsTnyy4ccqkllmrMGSJqlSTz2xUQREYCJ/h8Gvg0bLEEFFkYqqA+V5/TkgQgWCt8RRNcTKfmgXIkIgJDIuv/wkvAvxpQwLU2rdK6OhYy0tu/c9XEdkB4a
wlFErOt9OhYxFJsvir4NjTnyu7PmjwsbFXtEJENTmlA9qACQgHndltv9npoGixG/ma7fOs/Tw2kVE+j7D9vHt5Z8vBu1Fkx9m6mGTM3E5vZvwnBLe9JIECic
4bHJSqTcVMvsrtReLSuocic2kQxryEx9+MsOmwST3AGxuxs01b8Y23HcjThxOlYmaV8VpNHh3a/7uNrrYylA4/aSb0f7+giaqet35nzT/wWRVenwX3WoGNlx
umm3ZV9tI09KfvcHN3+TFfttVndrLAoZyZUqnfHRY+GRXWLPyUCyldujuWeTVuhaV6dd6swYPjLj3uWdYV5wyXe570rqsW/bhM3/icVXMmbhtIOQA32B1bQz
OzGcaI357tN/k7smnP0Ium+/hPI5XwSxy4+DDglkwZeAkE9yV24+SxzHRCP84tjNHI6y063Py9rY2/avQ9stxKtQsbqTHGsxhe6FuHWlZgCBbmh+Ok0fN9YD
bM2souUEmg8LxFS2mFDnmI4aCO27EbpWCZ1JxjTYjvEpfMLdV2HQGRS0FCSQoMij4mmUdrC0lkyJ29l50peeKAYAu6jOtdS6Hhu2DWTnG4Vs3DTG3MmJyYkX
Zoz7wnrePbc9vMx+m3fwA3bf6Z+usCmAMOBtWXO2dtPDiKkmcvWtGvmzwreGxs8PNa2MCQKUKGGpdPQ8TfOaPGx78ear5X9lvfejnf6+ednad2fe4M3J3sEZ
ERMaPp7u3hsOpt05jq9/MVnLXRt8FrfSG1CYHIN+T6kRY5kIxCXJNgGlM+rEIFk0LQnDqa9otZQVXyQ1+GciTFy6ZgwCQCSCiCF3jINbwFkuHlnBAZQApzXS
YNv8HJalsAinN3RaTJkT06XFKLQcWxtDxnnEwOz3q530Iuu+w24pKqrsUIrthdcCjedKKTTit2yOe2zTAVE6unQ4JcHzl7q8VGAa/hx7V4DAJUBBDO7uyEqc
nrg9CyB0b0Yf3W+pMYBmkejDALy2cCpMnPoiEJS609914Hhip6N6JA4tdvH8fn8Gk8fI5UJFjc6YK5TRFXMsnd3ntypzkSSSSRMP3+ODwkiyKdP4bkoOLvjd
B3oZcTKzmb8b6TlyvV1O8vbWgPFOXQxi7dShT3Q64QlQk5+iiJ3nedtoaDxYfktFnxqrAKL+pNGQ/m5wq40TdNwZWbJiTMu2qQKUpJLoDKKb4XWtZGGScTw5
nP9i6tuHdwuli3BYQpCDpdXpcsYyay3YRa8t/ELgusEMZVUqnghhakzn6G++u9PNcGhak0CBAIfSEQQISJCA1DCZENtiZiiQTMn0ZtTBwdBUNsJSMKMbFBQr
Km+CnybO0NhTVd5PrZgOpqJr4Ub7KqzAZicGfTXMI2sQMQOOiE/FtKBiAJEWCggqgMQD3UogKiLIDsX0po8xQPRTJo9exmrGcs2HXKe7YnXCA3dLmzz4yZNo
FYO2S9/LJ992yb7zWN1ywW5bejvwvjJ6GT7F9Py6nmE63not+CceJ91J3hDYt66Aq8z7Cp2LyZLswVQXs9VdVdAlFMZrEIWstTBGsXi95FqJQEAMDDKYUcNa
u8sgZY29h3gFVBluAcScpZelqpCSjLgEDUHXWzugE3ChQ04YOQUObrU6zuS1k1dymQ2+pfoTPE0zrHEqKFWuvQL123NWVNjEk2Wb2TcQuVYM28NUqrHa3Q+7
Q1iEgZNc4O72fWee15DPIxMKeyKUHRoVkRbEQHdFONjTl4X/IeEpAL/vhSp8QE2tuN5TL9S2o37dt18v7240U4aQNRNjm0qGkabUbMUbTFo0NOTJsdsW9L9v
yu7a9/wc/zdfm139ujgjOueV24lnIxx34zLWhyFFoRAcnoVah4O7r5iF/GBIBost3VXmTWsa+7X1Jp8x7tuKwNbDvTAYGfu1dwLxw8z4vE2OxsxIDso1A3SS
MVUZGHosLYUMkDHLuyO4mzJfocjs+eJpfK8uhqTGPefCHcdrF3sKPpBJVbQoiDefnO3W0pjCAPYMIO8dB6YZg5EHUfrHoFYYXFp3PcB1HXQIUIhIYtMooZg5
SjfqOiHMLkFIfIWE8D8mH3DhJvh4nBqTacAQ3uCIAQxfZNJEGQ5CCVYtXhLuPp6ejyyzo26erw92XV0xpTkypAqRUiWfLF400hbFbZuWZQVIxmovz2lEkpPh
hXta10Ojf73vPh8Pu7eXeZu2j09PcIO/ZwN2I+53D+KxCYos4+cRo8TlNu7hM67czmbtKoyK5tqcNWi9H03q1mpPb7qAOf5PZ7zsuvf7WJ8R/PBaisEIJIMY
VRUbZIHrDc8cITYCHwmH3sCEEZ2kFxikYG0kgRh9XtI8jV4vIwH0IATJklufbA/s/m9NTs1VIIPi7Xy+Ixh461u9sLguKnAH6qZTZGMs9VAD+rE/6cVqGozW
MXAl2FUR7VTctYYn3SCUzLelpie/ws68hAWlQ+MDIP1wlVCiMin1joGHC69RmYVQpR5w6ae6dk8MPV5QOBc21zz1B2G7DjuKonP7S6zplaWTsJfgYinPtQym
fT7VuRNFwM8/GOq4K0Q4xJ0pauoDF34KxIbMliEthKNrba1ogVcbeZcxgnFA69S6rw5URsDWOltgYWlsPehxwthHVCDFJEJAxSk5bEi/Ij3iOQGNGBXjIUUX
MCiQw4Mi9lkuGzcHECJLfMWLuoNnsyPAMHeR7XAC+xiu+duxv9A3GJpFN52HgB5pzrrJszIVtqy5nAqzMS244h6JzlBx3AciEgyQLnnfx9mqYaUueGfas5zH
EwhLafns9ejv4yd8m04JTgJz0kcNdLey2Yyo4XAOhvpGxdfdxcQtcDIYVhCTC+Qe+HFN+1o7tzzTx9bzAx3DpYoOQHDtbja5bXu41F32M94xzt/W7fKqP7JK
LQ2oHAL+7wg0/JzRtzPJeOHQ49i+bUuuLNxUs2h91FuTjBLk3sOG5c6dAId+MOcQMUwPBbcQCBvmaHlQd/eSqh5vF2m6gx1Cs0udtWQMvRkWQIiN2708+ozj
xeWTr0L3YlJ2TDIljCJwbm8A3/qpmF+u/I95yXvm1Ho0ZP0GNISMYSQvxx516y3pVS++9tN84X34FXaywcetb1W9Msk+nRbufcBm73YnRNrRsK1EhkncuJre
uKZoa00ujVtF4iZQh5M4m3cmp9qZZk2mLs+fnhkwA27D1TnG5AuH1Jxouva5g0Jd57JO0gB2by1T32MXHYGrgJ64fa+2Za39X44/Z+6UEwufOcArGR9yfNrb
lbKIcpjP5NGjSLWn9esMGdTD971STtc3Si87erm5j6Xf1/7bDe9LUKUGqJ9ftc8nmc4FuouQeFMw/QP4I/GBZ8ImnQpE7SO7xoG+D9ZNlGvy+5Pin6A+WyxC
ne3JavqOMlnJcedg5y4olELQ18rLxwcLkPq+3xPqHm+mq5KeWer2kYHBr+wpEht2qB1Fg4ztRig1D4BaQDQaPeFbfyD8Rw3GQbgp8NG8/eYGDVJ8rOG7diSf
3ySTI1H9E7Cc18SbY7w24M4LS2/4lA4ObNPKhqurUn2Xs6xwEub9DC8q2zTYZDeGi4Hq7pOD3bKBwPsMhSvyJpXQB28B0zkLap4eJYJGQOpuNACJsxIAjmzu
hU8n1FwNpQIJ85AQjHfCYjlvRT5PFo/X/yA4n5IVjfTRFJJCMGg5pCdh8J6IL5qWRFGVKCFn4P4xBQUB/o7jt29m3XgGEGO+wGqGY02xAZ03hR8qBN6BfXh1
FQ+OZf7wfc9sZAiBcthuXo+bClD55bfvF7e2Y/XWa8T9rbaLyDl8CAEG3PaGOIEYEIKVeMiHnBTCtlCXbuDwvrDFBk22Go0Yng7pAQIDjzJMkJhFsj+ZfaBN
0M1qA9VGHZuzVVkJsavjhneXeTgJhCqHB1syiQdIPggcW0rLAaEMXizAfNMaLKcIxlGQsavhAPvBf4wLdAJEM8o8o6Ht3W+kPrI6bZMapobfte/Zw9TdnTJO
vpv0uVrWYaCpGUGHn34kMJoiFA8ydiwWKAMkgikglIiRFkpA+eMDyzg2sNJ2IUZPJD2YZmki+wcHEuSmK5bNnYTZ2uGSJhTBAwTbxyZD613uTcZMEn1xgaZN
gYbCDBAPYeJ1n1WSB6RFSCIoci+F2Fp1diUi8LuHeXfA6I70b07joYHf3w1MUbOjzZvqSEYbaDXPA1MMgv2QfSeDds6m3buPuPCxvreEJ31Su+W2xdzji8/i
l2RLLgbcGxVFJWoYYejAKDUSMoENlVw2FyEgYhRQMQa17RElBOFxm9WSRLOFjEQkoqYFJAqOhECKwxztlhjkYIjcpaWJsGkLMbyBclUGt5ucEIECRvNCLEoL
JtpGz637kw6ASIwSQScwm/q8O7tKO+FxWdu9tud17DVzVm2Xn3HGRdK1WQ3F+5PVmdbYKMDpDqVQpQh9j6wL7QuS+AByToBW8N1WhCTQKypRlYKUjDQD82wb
wJsAe+GOAeR6MLrLh9aUwIQIqbHPj1EYsmsaOw5K10jkpr7sN0F67wkZGQ0pC/A9hGQPk+oLre8LnTNA4kDsCVE5D6R2ykCHMXQtRtZiXBvtTQNWPAJFYU0p
CKcTbIa5MqBThPCVTAg+25TeI8R3hmQSw0BIDEYEwxFQxJgKBnBri03oAvBc4ktT+VRBQWMib16I+QAdlnd1EbqUoiNRFkZGQD5+Jw+k2kgfTTQedSWreWuX
VtsNhRA1VZHLn6uqQ+k0Hn6lMYuuDnIBoOXGBlYGfJ9HqsQ1nM0RTGIJkNp5hMxS2Q0J2AZUWvoaYRhFxbNkL9gFX6LaBqMZLABYjAaeIdkUgiJGE9rsGiBD
QF2a6FVRjcV4aRzzfsg4F4HYmnR8lsEq0qQZOlge9lHQQQ0HhIz49TYDNd6VP1MVvu1axQ4HtotJDljSDZcTMs2UC4/AlDyPIPBC1BRXYjaJEQuH5su/hlgG
iHM83iwZGIPvCNlfOliMfCgpobStCkxLYq7BdBKGhyjQkQpizW73JXUB08iFEQ5AU/pIaIjg5pd0iSBYGSRZTCcpEwQhbI1o3BqAKBm4RwMsbLpOUpINHPI1
MEDqToh08iQgxGIHMgAgbw4Ixz6MwPUtUK8gY0R1q2nXJbzohPkR3jmwwxgj5BI76uikmgUMPDrPZNTkPU8TA5mZqEdEuG/mFYAa1A7k2HKA9oUcj0MM6Hwl
HqMLNylPy5/KBfk36BtHxdk3Sp7iNAqqyDCoZKYpMSrYUYxYkSzGlwwJ65XVeBxn3jghRFjEO27WVCkOdiWK3mnDlAzRyLltFzshoJeD0Cgu3owe21LiOItA
xBpcsviBDWHjQ35lqoCz5w3OIfN0bbtMzOw2urNwpjYVldAxg7Na695PFZJWYZWiUmawrTVhlcDYZLh4QYRRJyQ2vf5JtQVyaIxTv3jWENbnB0IjrWULEC3K
oaQTNWd4dpRpUg9sCGxDoBQxzFKqJaFZQQoznmTFlq3tEUcLlGgmnIfwSFsUDMugtLBodgTYyWAVYGnLQZZ7UlSCQXKWG0VmYMZDQXNFwZaHmzTqazTGDpe0
aloFVSVc5hCBCBC/sQINg7lOmxzLlMRgjmEQgsM2BFLgqNw3CF4aPX1Fl8hN9KO74JYeo5au0IDtgu4ojaFBCDSIlRGBok0CGmeEKIjWGUawrQbDi5yO3yGs
aDk5AaA3/ANADCqS8e586YgggUlrxPMeG1TzQb93pTl74kkgEgNuxAjIhGBA6vAKcLihThehgNegS5XFbkYP5j2GvQyy8QgcAz95x432tELQoINopwRGwlPz
TUDFD2wsb+yqhIwDP2eLUJB9tG1Mwx2IagdglOBY8dQXYgnqDLcATMdQ8eZE6jZGFYl7onUA7XgeZt7wK0PeQlFBfB2bR+HtG17LyPvaSDKXJ930KBsNsF6j
rE6BSRDPymnTZXByAxTjiN4wjv4xM5JIiKqsRRESLFiqiowRRiJJKoqXBcMY+aYvUEPgeeOO3yXuHlDhmph0pYeehYGJJWKBo6UAMAGTXKBSRxRKUlH1as0Q
5E2Dr1Iw8A+zYFQSDBIpQdB05ciJDxGEbZ0Q6DgR2HEee0PPqKub6ISkkzOGVJB7UdyeaetYsSB4czmHnslUX7DT6U8Mo5NwYvd6evPtheFVKZWV11danYDw
C41OMTPO6zDvm180HRmq1N44ysWNRmwL0WuY1OplziGgAHmB67CeAXxzWhwENMnt7kRsHUEhcbGKqnLn88IS9cEFFrQKHUmEBkjdQWYkY0w6zwONcgK8qscQ
mGfEg4NWB6cM0DNAzc7OHUcpb43msrk65F32cQyuXI6iKMSFLk1pyFCQilAQRkq3hJzi3f2FMc90/utRfNbAOoq4JTQeviZy8smVxyqEZZVhZa7ToOp5r71S
bCjETx2ZzCgxYZNxCEbb3NpsbA5hYdpOPWG9vSqwwGzQXIj3rApfESMrdv6UkdOaGIN9+y9wMGiX7Ug2xHRnttW09K+/HN1JcYLQYIcJDgg4nZVMW0ztyXDq
Vx9ZQtEsGplTfdGKMMg0KdlKrA1OwHt5cHsvrTEJ1UU7cW+zcTJrG13PBk2bL/fD8ZH2WrKHMMgKoKJqhGMbQzbmy70dm2mPS+C3igSvAgi2c6ib7OYqnTpC
hQxRrMGh7Gyg9uQwIk0Iaa0mUyALnkb6NtQ2ptMHe60OjHN5c21xDrYSaOIVtdQKGZBLN1DfFyxrG+8tiYVWLa667QkywDXK8TThyFXjrzmaHQZotm+xDjUs
Ms3BMCgiYim0eDze6ENSCknEPGHvkmIboJiuUQoYWSRD4RKaCUKa3mmyIpULaw8CHXWc9gA3nxE3BOc6REDs5OHd1qzc8DcuQhiFCxDCJqspQ5zIkxQ3wOC4
t7TkWOSrSFREeEuEODAhB7SIbI4EuF8yrDZTTZPLbvt6r6uu9eKDkTVdyzunFBnQoGTn1KQ5cMUT0Wi+9yya0XSFjTvDfgl4t163CepLxu5HgL+yeb9X0bhP
Z8EcMOIPGvStRoK7dcLecHHUA50R4mtM/MQyO116gO+4NQLCd212noP3fu94duOlkhISODY/HA2XAdc7U8hnuYBt3CR7JqEhBkJ7OfpOlBMmRrTSC8WJrwSw
FQXgg4dpprYUjvG9Jktwy1H6yAQgyMJGAvgkHkvXyqdtrW6UKD46koryN4ZHlNa1N1FIat22IZaG029YA4hsDDzrEBBikBh3QnnI+JvZpGxsQwk2YE8ZaiQU
TSGTcd/3F2HnfPG2HlnqjJwvMKjrnuW/bQLQTTjRZQvfkfkZD/4u5IpwoSEHU+qA
"""

source = bz2.decompress(base64.b64decode(PAYLOAD))
actual = hashlib.sha256(source).hexdigest()

if actual != EXPECTED_SHA256:
    raise RuntimeError(
        f"Recovery payload SHA mismatch: {actual} != {EXPECTED_SHA256}"
    )

OUT.write_bytes(source)

print("Reconstructed :", OUT)
print("Bytes         :", OUT.stat().st_size)
print("SHA256        :", actual)

text = source.decode("utf-8")
compiled = compile(text, str(OUT), "exec")

print("[PASS] Stage27-2A recovery script reconstructed and syntax-verified.")
print("Scientific actions: ZERO fits / ZERO inference / ZERO target openings.")
print("Launching recovery...")
print("=" * 100)

exec(
    compiled,
    {
        "__name__": "__main__",
        "__file__": str(OUT),
    },
)


Reconstructed : /kaggle/working/stage27_2a_gitignore_recovery_commit_push_verify.py
Bytes         : 23644
SHA256        : 214a57af8e3f97e7d0bb8e36e2107f6e20fd44a82173f8c9b026a47838b3e4b6
[PASS] Stage27-2A recovery script reconstructed and syntax-verified.
Scientific actions: ZERO fits / ZERO inference / ZERO target openings.
Launching recovery...

STAGE27-2A RECOVERY :: SCIENTIFIC-PARENT GATE
Expected parent : 9bdf2c1fca362312855b8612c579da685a390ffe
Local HEAD      : 9bdf2c1fca362312855b8612c579da685a390ffe
origin/main     : 9bdf2c1fca362312855b8612c579da685a390ffe
[PASS] No Stage27-2A commit occurred before the staging failure.

STAGE27-2A RECOVERY :: EXISTING 18-ARTIFACT GATE
Expected artifacts: 18
Actual artifacts  : 18
[PASS] Exactly 18 durable Stage27-2A artifacts already exist.

STAGE27-2A RECOVERY :: FREEZE-RECORD HASH GATE
Fits completed           : 10 / 10
Fits remaining           : 0
Validation vectors       : 4
Thursday predictor reads : 0
Friday predictor reads   : 0
Targe

In [26]:
# STAGE27-2B THURSDAY OPENINGS — NO-UPLOAD BOOTSTRAP
# Paste this ENTIRE file into ONE new Kaggle code cell and run it.

from pathlib import Path
import base64
import bz2
import hashlib

OUT = Path("/kaggle/working/stage27_2b_thursday_openings.py")
EXPECTED_SHA256 = "8c41d8b37486230e4306323a3fa1a32b5c72e85112791b39f396b8df8dd0d253"

PAYLOAD = r"""
...
C40s
"""

source = bz2.decompress(base64.b64decode(PAYLOAD))
actual = hashlib.sha256(source).hexdigest()

if actual != EXPECTED_SHA256:
    raise RuntimeError(
        f"Stage27-2B payload SHA mismatch: {actual} != {EXPECTED_SHA256}"
    )

OUT.write_bytes(source)

print("Reconstructed :", OUT)
print("Bytes         :", OUT.stat().st_size)
print("SHA256        :", actual)

text = source.decode("utf-8")
compiled = compile(text, str(OUT), "exec")

print("[PASS] Stage27-2B script reconstructed and syntax-verified.")
print("Launching Stage27-2B...")
print("=" * 100)

exec(
    compiled,
    {
        "__name__": "__main__",
        "__file__": str(OUT),
    },
)


Reconstructed : /kaggle/working/stage27_2b_thursday_openings.py
Bytes         : 84519
SHA256        : 8c41d8b37486230e4306323a3fa1a32b5c72e85112791b39f396b8df8dd0d253
[PASS] Stage27-2B script reconstructed and syntax-verified.
Launching Stage27-2B...

STAGE27-2B :: SCIENTIFIC-PARENT GATE
Expected parent : 8994c42e1fa31501784a7653d10b34c30a07c26c
Local HEAD      : 8994c42e1fa31501784a7653d10b34c30a07c26c
origin/main     : 8994c42e1fa31501784a7653d10b34c30a07c26c
Git clean       : True
[PASS] Stage27-2A is the clean scientific parent.

STAGE27-2B :: LOAD FROZEN CONTRACT
Frozen models               : 10 / 10
Additional model fits        : FORBIDDEN
INFILTRATION thresholds      : FROZEN
WEB_ATTACK thresholds        : FROZEN
BOT/DDOS/PORT_SCAN thresholds: Thursday validation pending
Target openings before stage : 0 / 5
[PASS] Stage27-2B transition is authorized.

STAGE27-2B :: FROZEN MODEL HASH GATE
BOT            XGBOOST   3d7b8747f911ae406894dbb07ec6870ceda91653e09792702b85e5777c5d43c9
BO

KeyError: 'numeric_audit'

In [27]:
# STAGE27-2B FIXED V2 — NO-UPLOAD BOOTSTRAP
# Paste this ENTIRE file into ONE new Kaggle code cell.
#
# V2 fixes ONLY:
#   1) Stage24 numeric audit path: root["numeric_audit"]
#   2) surgical cleanup of the unsealed partial Thursday feature file created
#      by the V1 KeyError, but ONLY before any validation/target inference/opening.
#
# ZERO model fits are added. Existing Stage27-2A models remain frozen.

from pathlib import Path
import base64
import bz2
import hashlib

OUT = Path("/kaggle/working/stage27_2b_thursday_openings_FIXED_V2.py")
EXPECTED_SHA256 = "39bc2258808062ac8a75249eed588921ff4c457e7e669f93bc16384553ee74ea"

PAYLOAD = r"""
...
/CRdgA==
"""

source = bz2.decompress(base64.b64decode(PAYLOAD))
actual = hashlib.sha256(source).hexdigest()

if actual != EXPECTED_SHA256:
    raise RuntimeError(
        f"Stage27-2B V2 payload SHA mismatch: {actual} != {EXPECTED_SHA256}"
    )

OUT.write_bytes(source)

print("Reconstructed :", OUT)
print("Bytes         :", OUT.stat().st_size)
print("SHA256        :", actual)

source_text = source.decode("utf-8")
compiled = compile(source_text, str(OUT), "exec")

print("[PASS] Stage27-2B FIXED V2 reconstructed and syntax-verified.")
print("V1 scientific state: no Thursday inference / no target opening consumed.")
print("Launching V2...")
print("=" * 100)

exec(
    compiled,
    {
        "__name__": "__main__",
        "__file__": str(OUT),
    },
)


Reconstructed : /kaggle/working/stage27_2b_thursday_openings_FIXED_V2.py
Bytes         : 85789
SHA256        : 39bc2258808062ac8a75249eed588921ff4c457e7e669f93bc16384553ee74ea
[PASS] Stage27-2B FIXED V2 reconstructed and syntax-verified.
V1 scientific state: no Thursday inference / no target opening consumed.
Launching V2...

STAGE27-2B :: SCIENTIFIC-PARENT GATE
Expected parent : 8994c42e1fa31501784a7653d10b34c30a07c26c
Local HEAD      : 8994c42e1fa31501784a7653d10b34c30a07c26c
origin/main     : 8994c42e1fa31501784a7653d10b34c30a07c26c
Git clean       : True
[PASS] Stage27-2A is the clean scientific parent.

STAGE27-2B :: LOAD FROZEN CONTRACT
Frozen models               : 10 / 10
Additional model fits        : FORBIDDEN
INFILTRATION thresholds      : FROZEN
WEB_ATTACK thresholds        : FROZEN
BOT/DDOS/PORT_SCAN thresholds: Thursday validation pending
Target openings before stage : 0 / 5
[PASS] Stage27-2B transition is authorized.

STAGE27-2B :: FROZEN MODEL HASH GATE
BOT            X

In [28]:
# STAGE27-2B COMMIT/PUSH/REMOTE VERIFY — NO-UPLOAD BOOTSTRAP
# Paste this ENTIRE file into ONE new Kaggle code cell.
# ZERO fits / ZERO inference / ZERO target reopening.

from pathlib import Path
import base64
import bz2
import hashlib

OUT = Path("/kaggle/working/stage27_2b_commit_push_verify.py")
EXPECTED_SHA256 = "15b2670a6514616eb231b5bb771b506b15f496fc780bb83a5c52398842a7b0ac"

PAYLOAD = r"""
QlpoOTFBWSZTWRkKU9wAEzz/4X/3QAB+///7f////7////5AAAQAEABgJD7wvMz7uvPerfV7aBIdbe3H00M+6Nr7Z2vc8zxz7ePH0++p7roB333A0ej77763
2e9989zuujvt7ZZ7Z76OfW9upWtLMprS9biB82q7u0DZ3DW+zdwd0PuanoKUeV6EppCAExDQQJ6qe2iekZJtKeTU/JT1Bo9GUMjR6jQaDTQAJTQjQkKYnqnp
pT2mmk9Sfqn6oeU9Rso9QNAB6IaAAGgAASCUUwqYJoBoAGnpPSANAaaAaBoAxqAAACTURFNMp4qn7Up+EyTU09T01DMp6jINGQGgxDQGmgAANBEkRNDImmTS
p7VB56qD09SaNGg2ieoM1A09T1A0AHqeo0AAiSECEYjQjE0nqNTTUzTTU2iaNDJoDQA0GQAA0N6oqvavqgfwgFQNy/j9j2VufDG8KheDZJIL/O6LURaZSC/t
dSS6Psoe4y0KkFwFTuTCstD/gpz0afFUNGRQNiBSCk21RCZ8fOM+A/2fNJUFSMQuihnqJV1SwYKKKKpThlF5soFVi3WKqSQswYTMznHT3epeK5uYG7LYb7Vy
sYXuMqhJRA2hRkYT76UyMZr2Ze1FzBGtP5IAxkI44Q7ndydWBqEhN8mGRl0e8mmutZTFQFoSqah1MK1Ki0VQYRVBRhFBEikQtlJRIF1JSjUeZLJsLl2IyMI5
RR5ESxSlVLWsRKlFAHOM9W7nhgBaGWm7HLFtju5/EbNASsJGykhFIkXkcd+VEx2SoGRxTcTkcyGcTS+Zo4YQgOqoFbWSkiqLq4hYwbaRoMSiKhBjEEVYgsGL
EKOYh79orG1pTbkTQxrBkA1NhbRrVAzQRBIECuIxaUk4ziP4/n89WBxaXvVbZDN9HtPh74QGX0x0DNl56EfX/blgsL0+pmNSohTFax0QXKlFxRJp49AasgkS
BSA6oCnp7Lc7oZSMJIyLIskiSBIQDfNA1BUEGlBXwajVYEV89jXPRHXC8wEQJplZRE535fr+Hbe8yFFqt9hAN1sWFKIripKOiGdsQzj0tmGaJmfOuHjyYC5D
VaRDlLuMJAuDyh0fXO9memvD2tjsv3QHYlwlfE3T8LWHYTW+zN+LgX5HKYw9/uoDS3SgNYzZN8ofKN76afcPz1c7+bnOsRGGrhnBjk4jA4e+MRbq549hTOOC
r1eTNwmAxPJwuFzcM21KFLCp3WY0YULFcyGY5J3T5U8N9CKOMd75QaQvGzByNvu1XpjHrtqszvG3VRqjRCejzUvz30YW1vNm8Q9iy/uuM/1xiVMPtE+J0qD1
fLZyuD5Rpq7aYReL9j8YzNtNhPn+k2zc2z/vleG6uPUHklbEeBMFuBL0b57vLbGlSJlQxmqNB4dmkjeFKPEnTFV5qp2UfezwCajfTZh8CDVGGbMXeKylb9La
F3vfMRzPcaOd6klFlqnZy/ASz0rSJ8Nc4VHx1FxmI77S1I2gMF9Sn5BrpX6rIbb7Jqa7i0lRSdbibS7fOjBMQKLjRjYsLpo7r4mbdZgvFdshYPqvr9KrYNIS
TVZzwsEel00qiyaXJB5k97uxHLjxPyki63hre68W9RlqKktot4vN8qBWOq/TnwxtikGDaFUne1xzXeO/FRSBY9Xo24npzT7apF3+YvPepawlKBDNfuWMrrOD
hCdpmrSSZMkOol8cSti5h5h+uGKYTToLK9j1YGnFNqg9pWklfPUSmLleKOM7pYUU3ojOq5VSos7yei/vu+y1sc5EtHRef6mipvM9U9YUx9ON0r856/VR4MLO
Aypq1VJ7IG+gdR3uV5SHpM9PEkcZBYNBnnitxvaHQtp4Ro6T9f5ijoiCW+jLMZri1ft8WFhNNWNt2QgMmk4Pn0XXbV29eTbune8G3BMONOy6N1zv0qjkpXnU
8tpk4mY6zpfe1W27Fx6OOvF/hw0HePVo5JjaqYap5mmjUv+Uzzz+nbb15baONNW90xu3vDcbbNgvHfBIygz7sltl614lncSUpnQ1cLb9s1t/wnjVo1C6PWYu
uVWR46sG68zYvKC5V7LZ1GmEMGVVOv4rWasjNpnDNFZTyXhom0rFSTj0rhcGVt77ZPNak7aazGd975RoBZwR0FvYkmw7e2LDAZLcYehmoQMFPs5Npra99AoZ
0T2NSOhEUQZ8Ju8vNr9SqfOm4W6264flZKNs2qi85YDgm0OTTOP1OHC565PSQ5I4RkblEi83LB8Hnd5oyQePyZUWI3wF7rG8AeaUO7L2phZMGo25vBT5lcrn
vJupMV1YXHUaRKRl3UfThuPX14mq1WvNc3JeC5o05qIO7uKLxOZuNOCYsvrjycn8iHgI1nMOiYsVBOvtmj6JPa7ajq6RvLjmDW5oL/XcRRACs3kCEEwXMZnu
oMJrcxbUX0slR90yjuvWmfHQwzfqP0pcYmHEJMhYb+iTjrw1NXx80HdyGjMyRU/AeCUEfMGuwoKTcDLLo72+jIXTEEEVXA3C4VicGm5S4JdFjmqRi6CUZDBc
cgKY7GaCzMSDNHcMg6FdsGxthWoNoZEjIizxLOIQgim5llMqMZyHvwOaHYx00TJm2hrcpaFEGoYsdYtZgy6SGjjrtpW4txSaarR2NvHo5blzNYSkCKMka1oW
kE3JMZGVlcpjJRGNVYoeqXA5tQD3SSyRCQYoIT1plm8Jb4K02/754/JFWbm7DDp0NoY609WTZQP8lXYEiIrBR9qgKSMHVZAaMmt2Y6Srwi6H37ZBeRMESGPC
Z4wphFIoQVqB57eWNihwEOSnh07DkRljG0rFZITSgxiuTs4nWh05ngaYeK+zJg0Wors20Fp9FqAqK5dW+ZrEWqoR6LvPbbFVE4XuA1YsepZIaxFo7UTSIjY1
WE2uJs0bykDuOOqd1Lyk9SIVLnRcJEtItDmVTcu+KtF8G1cwe2rYw2Z1GnuhmzCjGxi0Zw2VY025rVpjP+G2XoaqcKnnCo7/uEPmpsaYrTuEPblAiI8zy7Sc
73unWza9mk1QWLH1fY8VVgb7SpcPaaQwkfp6+7juLb3zFcKJgx0X4PFizKQphsYS2OtDbFzW4g4I2ghkb/GrDicDL6Cn5Piwp4fa+LbJwIdvb0+jwA+2ZFC0
C0gfmEX4wsLFp9S9T3nYXdFByHFg0ESBONcmgEZcQ0SRtXBmnmkqCaSImeUPk2Lkfi1r1Y0/v7ptW2y1LyX9+Uu7fPB4wglDuDvrncS51k3Vv1UZurwOsp1h
8e29Kcw493jLxG2z9Z7S/jcu1aePThzsZTrtpokv2/i8ltk106dHbTbNPEY0eW77vSfho5CRt97hxJRvqFZdwzUk+YYooLFPM+bs9jkHq7ihPi26bTfnQmUL
ao8JNpPLNaJ4/J6dwpnojo4PBZ644BZrVGqB5PO/RMG4v77h4SZmGAnQ8dD5iqHWQTdB732alHEsHNVgeDZNIx7VRIZPZxEgPumx0gxBAkSMzD7iCwmofcNm
5JmaHkp9YunWcaCTIi+w5H0mzeRJvh4n0Go9RBgXyJ3wqJN8jW1FrS1rFoaGwWxcwugK3e7dxMiQGkxLkIplSmsH5JSa6Nxba15rOzxLa/YdacS6+J9GvBwi
IT9u/XtmMlfPXRlIzO9LwrjLN2xvvwvUHhTaq6wsL5reZ5VPuMfBXCEsZle8OW6qSS4emyzsqsGPq+rs6/X87j0t+zsbQOzpzzN3SNd7I9j9t6s4GPj58a93
jla3bHDQJj1I4/PCSZIaXBHDw8VVd6SMXfM98JstUqpYzDhx6p4Vo3BwMCDMGHr6H6m6ZN8GXrZDCAZJoRJEIgLIe+iKtVABYHpB7Z8j08MkFcQVs955bCtW
nYXDb1YGh23Zw+9/i3S3wW8sA5/c5JP1HP1gUk/vP5epPxoHvmQIp7z+fibDQH4tD+mOLhA68jWEsX/Qf1QuPGxidbj5tSh6leRsmUWoOAuIQ8QYO9qe8iek
gIdhhf8/oe04C3HX+4jvF1NyYb5liF8Q8ELHdceu4VszUBoR3fE5iJi0nnCkCzeSYiEdgIewdhpJwDq1JfMduJuwey+rwE7UL33bZK8JIbjfrJf+K76VwK1G
Obb0YHXq5vXECZ7IGWgnNPNP+BawOq1poeWjIQgeUGjrN66gGBipy867OvKyG4/OSMOoCwnQNSkOpuctzrohni5tbROp0Cg1HrDo+2Dy7V3cDSSZ/GAhm9w8
tbO1WE8qFvuLflsmUToZ+1PsJdNgkIQjGSMdile5fjXyRe55jsukzCtxJCoajCpDXxCeNL7fPCZDLjkvgsQl/mLXv7R8OrQ3BnvCBdwBl8hBbHKvLhtAeo2d
iPQpA4nsHd2ieodu85keopdeRpVsM3lszHVzR3JxlHDej1wSMYXHs5GN0MWPF7sLbDNNZJMNme+WGa7kd4rmFkykWNHww2TRoZ25O+BywNdqUDai7kO73o0Q
cGObrRlg2SiWKdm5lcBuFhuYMZNLcAxkS0mHCfP7a5hvOLFrjSQnV1pc7U8ve+EOimnJXfrdbHavE7jI85manBH2r2ryo2XjyWg43+6Pw38Id34z85fonYc+
wg/DU71bdx6r0E6Ze46Pmmh4BwMoHxHoFWHll9fqHaerk+n5qA1ZIcA+JXMfjJStUc7td/Ljc94vxtm+y5hh962OYjFzQx1Qcf0OO/YxDRuKGzhtx7lu9IxI
c3gemJLiLtpBONBluNUIQhCQIAhxz5ad07GJHsQzDLINrDqMhvLSy1hIWY0xWGhgExP65TO+GDCQLDc1L7K7z2tjAfORTVkgqLBEVvidx39t+G+WOjvXLqrd
OEdKqtmLMWOWZRoTQysHPiLkI5R6hhoth0oCH5N635+C7BqHhwWjh1QN6hDr1TnzFzcOQhol5sJv2s2DjiiVV0L4umoGoSRhHxaLGuZqz7QnGKGmHDp0Doxo
KWt68qrATAXWHMtUeDcXoVdsc0oC2/5OznuLm+i9iX9eZOW5NEKDXoHkkkOJrpEy+seazK3oMg2PIMgxWSHPQCL3Ln1Ikhx2r1+stct7owU/8Jw83sPkcyQM
6IdjUVa27Frfk5yAnzdpF1OT1fK+KlrP4XK2eZxxxNT9zfz/NuH2kIr2JcLWiFBTQVGEFkQZ60PMh9dJcyPCBU0HqYc2MZgzH1QoUKBTbwtcKPZg+wHD9X1q
dAS8/o7DBHFzTamq/SO3u0a7g7LfFtNyWvrl4Ma0eSt1OEK6WwYH7nmPjQ9lhZ1dnfyiA6DdvSSj5UuxwSE30b5R80fzpJBeTCYxugTrqHfJbWCUWVJixuSM
VqQ3lqC5kWTUkNnQlkuOwmE1LqUdNb6Q/2Y5Y0v8Sh3vh2NiGh1bQya+ngN28mNJAlyTN4b9DLIJGEAkDJbBmNE1kk2pqa3fJKtBo4EtTkERhg1wCcgUjNkj
c/AczNqzltLDoSTuO1TcwzU5LEl62Pt+FJJ3dKTEvVfFwbQ1jY2qYG5WI7Bxj1H7h5pm3MB+/wbTfbxOhkZrkbP+UIRft+KmiH5JRhcK29qfUJU1StXMTnGL
BhA37Axi9RppABkE+Lxfs+oH4ORo2UIQcyaAIb31Og+MMgsMIlGxqQfWKYGKDZAzcLrDBGqOX1TKn6hMa3F6lXccJm1E3p5K2R6vEEx3WbwzRunyA8g9YkIM
iyRircGwy6d4eTSunrEKPgdXUD5+41+9W9eR+PrLQOanDQ6hwBMSLxQdqOAZyAmSEAHFoGwUYIVIM5GbLSLA/6VUsNC+W4msCYAnRgxO2i18tFFlnyPYbcqs
fWP1OEwJimUganf3SMgRI6YG8KSD+1eqZwA+/ilJacPL6NZkTUHw917WICCTKUno6NyhXoPOGN+Sh853ySoJu64bv2MKaPwX2JsgQ+gNvcxsdJJEkEmwOcfB
1Lr+Lvv9X24R4Zm6qGpKaO42Dxg6O7QZtuGnXRXLW++04FycySyXIsTXEApbCQhQ5nGgzZGQkWCjGRFoaCJAkQpA/CQTbCxwCJRpTIPSUSoRo4JLJGMQ3CDh
DGtLCuWOwBtKWw4sLg+TLRkQhZ/AoPnJMoBpIgOIMhGMhl2GxPHFHcsWMZIESIxIJqM3gdiUg4PacbZqeJGA2zTsO436jnLUUGboQLeD8QHbdDFsLazYy2HO
3Um93aXEP6J+eR06Crr11k0SDpPIBgiYIizuNIeFk0lLsT4K2I3WZwpbwV5FezqEcN8xfSYiLkbt8yyWVsbkF8CTKXsb9cBGkXWth8mThwTs8efFIhsvGyzk
NDnccBhidzbtk5nGNXAYI0lWLI3S5vZklwyBiMRmiKFQ0gl3KdvELIxIsMBshEJdnePOjGAd2kLoHPIORYQu+r/45QgbRZBjAWDuQHdOAzjsEKNgx9fNgiI3
RTxngN504wtmYowVDiVkGk0olkgJNA7ptCAlJEgyAZalMQbMQrClA3pFPpKzkgd6pYiO/JKLahd3YRhkT2dIawJ29up47+ttqcA9x6sMrgfVIBGMYEkUcJv6
BzRYJ1copLhfUzUwzf5fZ4bCEnZoCwVQ6KWGXWaQ+H1BcuFns0A5yR8AooOw9Ttp8QsR8YWsCQ9o0QsWTHJfessmamVHIqwcxMaquyqehldfkCrxRSmgJW6i
uoyVKh5b9kQ4r1hqRCytJICEigQrJfd13QLGa9qHMzK+QRvHR51jKhIT1ig8hapejIvzA80PcoHX2Xeo7QhilaYhURCQiJJXQbfEHoVWFPhEtFXelEScdTnC
YhCfhNlGxy50bJqUnI7CKfNVkgZQzlmE7V1RxEShuP2iJWOMyzrmFZWGzCqkVqKGLIagwzA0DcMKdDBrJNOrBjSlaJGRchKswMCDjtBrH557+amBADQYq0KM
IQGJzUqxTnICyJCDEgB6xKmAuia8BHh1WKqjS55BpmWOOW5Q6eUJ507rNiyVRKGgcUPS1EbySKEVuPWx+hLuBe2+mvq55DjewU21BBj1HsUeSYQ7MqBCyzQ2
KCwK3MgKhEX4Q7E8g6HoBewUt+8Qso0ieO85ZBgNywOh2Qisi9RVUUNEo6aLSWpSQZQXRUtDSNb1HvR4FaweCm4bm4soZHYh65DRF6B3iFUo2MUNYAmWjomC
0rJILE5s+QhQm1Jew9/z701AMPTHi8UfZEk6mfCPZKKKBq9AvAdo7ckCERUhGcXURSwHQkVnKp7I9+oG3hNYCHaAcOo8ln2GnLVGEXBKAJDMYxgRHdARAdr+
O5rWNxM58TaHN2ppBcxvhJAC1HMLGMjo0Q7k3nCA8lp5nkJrE8JR6GMXMKdmZg/n1+U3u4yJvJ2k3PAafINQUiS0KpqCClpdLQK20FMBkSMEYMh8vdqI55HG
azwhyZ0X4yrSBbCDBJ2Z5kSdNWVRW3ZNxRzA1U0LD3E9ZrO8g8sl619TUX0CwYUfx9UmYZGnyGxQHIPDZMxDZPqY9J46LDw8FO/niD8Orrv4WqeB3PaHB7TV
jQnWPKIV7LL29E988btWu2Id+LK6mkNE503IB08TnVFodRC16br7fTyw5EX7GuIB6egBkHR63B0oShUKAL2ADUA9mouUHlnAThkvEGHWCOJcAHIdFmR1ob5z
IUkkiUaDfarXbEpDUjpK3XKKQ98Y0DBNQgrXo1K8eNbCibYasgKDDRLSipNaosWMhSlUUkw0FoxspUDMZUCJr1HozdbpL7thzIbtSjcFsFhgOcEQRByHYEIB
YNvdGT+bQuuYxRdFYoRHUSogqICIAYCktMEqENKQ693Wc9GODAdO9bCctuxM2ANRB0KCNoUQkIBSECjEONspa0URABDJCCiaS5r5fim3ocvcqdqjP3AkIAUE
fHzZ8xEhFaege2bXntU6eL5xFIR9eC0kW2gbO+MkgEIpy6JiAEGDBkWdlmSnKXYCboB455gQa+IJ8+EXUDCh8v3SHL3Hjqm/1oN8QsBaNISE9IwkKaDNduh2
dqsjQQewBCkKIYML8IJdXjsm0WesN83Jo5mwuwh0EpyLPkG5B+1hhEryBHLh6pv3dOIhyOpMB5nbB7UmWWA4LSjyff0hw9gFcvxxj8QpeQxsG3yRdDxhU4Xv
zUvPgUmXEI2Gz3zl0FTer5yCDl4MDuXxBSRu9OmltRVcTnrYwkYmQ7hMFxSdp5eXW7cSZnFAwYlIqMUViIosVFRFiKRYSCRSTQMCQIB5h7N5ECE+ydi5bQ21
96eQmU1R1liqCljEYQCMk07DKwoRisihC6hJIQgwqjjlRpsmzk3IkSE5BoWMhQwg+1DdhM2QKzTM8kYIUQkSipISoMeVAUxCcE55j8t4noYuEB/FQUkA5DsD
1IeqfXkgSCmDIYx4JybGehPBOvjRequgkpOuunq58lF3VukhO7+XEy/Qx0AyIbR4EOm0HR2AwcNeig8czNCUew7yBxNJqZDjh1Crek7+9TvEspyUM/ZAFcgN
58uBOwM4bgcxNEgBqHt8IED0joJxOUOOosFhunuAQHfN4goMaoUIG4bCEgkUe9Ni8pJIpy7KeaDhXOZdJJ2ZM3LsN2dXCPjd2C8LNvDl708xK55Ig9t3UXIL
ZabLO20Vh5RKVAok6Pc2wxbYs5nbgTNw1pqVmC5MhFtXN7akvTdLw0kxCtt5bH+osXHLrfJMCWiGO7cw5yBrk5UZCyMpClC4ExDOc27FirkI8W+y2LIVTlCE
WSJFLFXKL3oHrcOGxuA5nEulimuXToy3WKHTnsAgWFCRIZPCioJ1o+nVm8m/O3JppKJg3jBzH6WYjvPGj1jom0uZDQZL3keIIXyC+c8HPfDQeSBr9pU2EwmW
DcmZnIkkSxKdoDfPMd1o77j5/Jytvz9LNtUo1phAK007NGTfjOayT75+D69tYeaagUZxyJxV70C66IFmkMv6nyd24Ks4bNkH3uGZ3rQQ4Wc1JKfmUxhi6SFf
JCM4LF7WKLW9FiyYtMUOmV5RDMRJalKB9EKnMoqhOwUsXHKwVaRIh2oWAd0CVtAG8BQyIpfFZwyyL3MNWi9MJaoNsMxRh8hdQbBTYy7pONkLFMgNdgyLBTAz
QhHYjlDTWlud63X8IIQZENjgp2C+cA7mq6JYiwIBkBsZaWAcM88YrcYGIkjIPfEqEoKEdW8OoxnZbVU0E4VrdSiEiYkkiBmh8qZRdEo2ogwicN/s9D5/KtHt
DUDcRXU5nMthbg3Po3tQJGS3nEyRNGsOWSNAQwavaoxUSlTlAhCEA7yIlMhrET0MGMYHC3z269/XbvxV7+uivgQXZyczreSZI7yJ3YucYeC4GBpuKDaYZtEN
ZC5e7aNyJezTeFXY2KO1OK4HAtC74pkMAbgnzF+Z7RfwnufvfZ5IfSj7eMD9A7Ly9+aAYBe+5B2ng8dquIe8IhiPcWMchPgORdLHESEFKDcKFAO53PkP6Pu8
ZEx2+za6QDMLLSc55J4jPkIBi2qdk0HgMgT1y8/ZDLJgQY6G6lLhtZLrc923r7zJe7L5z23Bt16bJ9UGDOKVGIqQhxFUhfcdlXQ0R7WZzlZeZG6zWLEK6oGY
jouZo5HEpRGTHXZdS8HMgucCzCwGXo/fiNXmanc0yApFFCCThJHgG/3h8AMJYciVd9HWUWymg5BhZG8LbdXGvnfqg26T4ZQJTTUZ/s9hTDioZqhMxHaZ5uz4
PjPjOR/8XckU4UJAZClPcA==
"""

source = bz2.decompress(
    base64.b64decode(
        PAYLOAD
    )
)

actual = hashlib.sha256(
    source
).hexdigest()

if actual != EXPECTED_SHA256:
    raise RuntimeError(
        f"Stage27-2B closure payload SHA mismatch: {actual} != {EXPECTED_SHA256}"
    )

OUT.write_bytes(
    source
)

print("Reconstructed :", OUT)
print("Bytes         :", OUT.stat().st_size)
print("SHA256        :", actual)

source_text = source.decode(
    "utf-8"
)

compiled = compile(
    source_text,
    str(OUT),
    "exec",
)

print("[PASS] Stage27-2B closure script reconstructed and syntax-verified.")
print("Scientific actions: ZERO fits / ZERO inference / ZERO target reopening.")
print("Launching closure...")
print("=" * 100)

exec(
    compiled,
    {
        "__name__": "__main__",
        "__file__": str(OUT),
    },
)


Reconstructed : /kaggle/working/stage27_2b_commit_push_verify.py
Bytes         : 36644
SHA256        : 15b2670a6514616eb231b5bb771b506b15f496fc780bb83a5c52398842a7b0ac
[PASS] Stage27-2B closure script reconstructed and syntax-verified.
Scientific actions: ZERO fits / ZERO inference / ZERO target reopening.
Launching closure...

STAGE27-2B :: PRE-COMMIT SCIENTIFIC-PARENT GATE
Expected parent : 8994c42e1fa31501784a7653d10b34c30a07c26c
Local HEAD      : 8994c42e1fa31501784a7653d10b34c30a07c26c
origin/main     : 8994c42e1fa31501784a7653d10b34c30a07c26c
[PASS] Scientific parent unchanged.

STAGE27-2B :: EXACT STAGED UNIVERSE
Expected staged artifacts: 16
Actual staged artifacts  : 16
[PASS] Exactly 16 Stage27-2B artifacts staged; no unrelated changes.

STAGE27-2B :: LOCAL FREEZE/HASH GATE
Required artifacts       : 16
Non-self artifact hashes : 15
[PASS] Freeze hashes exactly cover the other 15 artifacts.

STAGE27-2B :: SCIENTIFIC BOUNDARY GATE
Fits in Stage27-2B       : 0
Model refits     

In [29]:
# STAGE27-2C FINAL FRIDAY OPENINGS — NO-UPLOAD BOOTSTRAP
# Paste this ENTIRE file into ONE new Kaggle code cell and run it.

from pathlib import Path
import base64
import bz2
import hashlib

OUT = Path("/kaggle/working/stage27_2c_friday_final_openings.py")
EXPECTED_SHA256 = "ab93a38c1729d987f0950adea4616db72525ddf33dc8d959ac4771c94ee0454c"

PAYLOAD = r"""
...
0fcs
"""

source = bz2.decompress(
    base64.b64decode(
        PAYLOAD
    )
)

actual = hashlib.sha256(
    source
).hexdigest()

if actual != EXPECTED_SHA256:
    raise RuntimeError(
        f"Stage27-2C payload SHA mismatch: {actual} != {EXPECTED_SHA256}"
    )

OUT.write_bytes(
    source
)

print("Reconstructed :", OUT)
print("Bytes         :", OUT.stat().st_size)
print("SHA256        :", actual)

source_text = source.decode(
    "utf-8"
)

compiled = compile(
    source_text,
    str(OUT),
    "exec",
)

print("[PASS] Stage27-2C script reconstructed and syntax-verified.")
print("Scientific rule: ZERO fits / ZERO threshold search; exactly 3 remaining openings.")
print("Launching Stage27-2C...")
print("=" * 100)

exec(
    compiled,
    {
        "__name__": "__main__",
        "__file__": str(OUT),
    },
)


Reconstructed : /kaggle/working/stage27_2c_friday_final_openings.py
Bytes         : 100849
SHA256        : ab93a38c1729d987f0950adea4616db72525ddf33dc8d959ac4771c94ee0454c
[PASS] Stage27-2C script reconstructed and syntax-verified.
Scientific rule: ZERO fits / ZERO threshold search; exactly 3 remaining openings.
Launching Stage27-2C...

STAGE27-2C :: SCIENTIFIC-PARENT GATE
Expected parent : bffa77c536cad211dea9ef46b102e5482e01e68e
Local HEAD      : bffa77c536cad211dea9ef46b102e5482e01e68e
origin/main     : bffa77c536cad211dea9ef46b102e5482e01e68e
Git clean       : True
[PASS] Stage27-2B is the clean scientific parent.

STAGE27-2C :: LOAD FROZEN CONTRACT
Frozen models               : 10 / 10
All five threshold sets     : FROZEN
Inherited target openings   : 2 / 5
Remaining authorized openings: 3
Model fits authorized       : 0
[PASS] Final Friday opening transition is authorized.

STAGE27-2C :: FROZEN MODEL HASH GATE
BOT        XGBOOST   3d7b8747f911ae406894dbb07ec6870ceda91653e09792702

In [30]:
# STAGE27-2C COMMIT/PUSH/REMOTE VERIFY — NO-UPLOAD BOOTSTRAP
# Paste this ENTIRE file into ONE new Kaggle code cell.
# ZERO fits / ZERO inference / ZERO target reopening.

from pathlib import Path
import base64
import bz2
import hashlib

OUT = Path("/kaggle/working/stage27_2c_commit_push_verify.py")
EXPECTED_SHA256 = "68256e54c8f07382fb8a0517053479d6f8b85a4fdcc5fc83034739eb9a882a74"

PAYLOAD = r"""
QlpoOTFBWSZTWV2B5RoAEjJ/4X//QAB+///7f////7////5AAAQAEABgIV7vC9vjs+7TenrQIChFBvOZ7l2dnrzM9vb7zwb6dqoe9jTJ19sd7rrzuc2Ogd7D
T3o7pJKKoAH3Z00PF13b0DQ9GjXhDfcNyhR17twSmkQARkDSYmk8Q0KeQp4nqmnlAHqeUxBtCNmqAADQSmghAhNCaGiantGiR6T0yTT0h6mQAPUNAAGnqHqG
QBIKKaFPVPU8oD9U8oAeoMnpAMQAGg0ADQAAABJqIhNBJT9U9NG0p5I0yemoeo8o0eo9Q0DRoeUAAAAABEpCmmkY2pPUeoaGj1BkBoaNMmgAA0BoaaA0DQAR
JEAgBNEyGkyT1E8k8mSamjGmowh6nqAyDQNPUHqY1GvCgK7tdmPvgFIGQurys9azWiVV0KoNIjYq0NpJEYEJPypCuMjD7HohWHqMPtJaFEFwFTvswqW0f7m9
zT9dEyyCgcGQpDigTPs7ifiP9eKFRgpdFUyvCluKwqkuKLBUabZRebKIxEF5HJ35efqk65OU3A6dRyGqKLptFClUUVR1pkLKhDaRGo1PvxjdCG767xI4UGxU
fyoGdk7c+m6yAsDuPq7ySQZkYs8jF0OKVTRPUQNRWWNMJMJFGLGFpQylQtS1ApmcbTUvdgpJImcAkVesSWJaA2tKKIkLAHWDP18r3XlDPfwzcGsOHl/7NDeC
VhZSQixIc+W2NGHsiYnINjmdB6tNA1rBLSp0KkK2sllRK4hVoak1qRIDEQVWMBUQo2D03ZG2ZuBYLLEomdCqTMZkhagh0MY0w0UIp+fzPjy8jlqpbMw36/z/
sX1/NGMSSHulysNn88qP64n2XH8q8RPGZ20vPIqFvfUvRV8Z8zepVhAjQqZMJD9/cv1cAbrBFVUWAsgrJzdZwAFVSkEXHzysAK/6b6T/VDSSGCAJw787KCF6
tr5aYYYggyrbXFHTPDK4rWUkoyAe6HPRE6GxjEiWJ8qOPXoGDHSPnScdkhCE2QnZoQ+r1HXXyVCi7WVVjwctcsnUlvyXrbp+Mn7TyZnZrlf8v6JQzt0IDSM2
dvYMexfKUcTfQu7SdhveGYyLj5OQGuzsA4x7RhQg8ZfLLLL7BBNIG6XHZPMC1hrFRlni2DsBwNAcoKcm3NzkcCEOvk3q5UO8MScqF2DBsbpx4Beu2qirK8a9
E2iM8Jp/JQ/XbMwuivHlaIeqX+VlPsxjnUw9RPcdK4eRkW2fCOajyvPGPctQM3/vpNJPqOTbzHCfUvG8m6ePEYurmiTd04WExJVXNBWd1MlCkm3UDU3IwSFo
6sbdS4IKrjCuejvyQCWei/TSX/WWNWXYshTXxXDOHyX/Fde2MxzXw13kuzLAtFKy5KGcjlM3ZlLhShQXUGIrtJNPCwLG1sMBirU9cbcLIcmeqWe0rjNOja3O
7OjEmgk/78tddo9N8bTl0V5h88vnVAFzRXHeBDpdNPIKnrXxLjFG+Hin1vpIikeGrzPFecyLsX1ott4r9Hy7t3CjB+bR/Dt2ZtSN6wRvvOZzXxfickXi6BY5
fNwabZKUa7TRyP7N12izt39/h5DmvbduR2IdGC5yu7pCZJO8nCeOrbmZ7CfXIvzm3N9HHMmExZUjhjsvu0s2eju2ewkhLhJu04RlUpZ33IiyyrKr+ktfd6df
Zrw5CdR+X5rRbqjW+oZE+jktZHPyjxj6DM7xu5rKMQd+h1KRbt9KRg6EiwOFrsP2qfBoxkoDRCnDI9PWOdXKwk2xtrmv39W/ItacDM7gh3eKv8f314+u8151
QKitDY0SkUGjQ9sxe6mMb10T1zxqZOcyjunNrni2cLYx32TrwbXZuGyzTOhxCvdKyQm5BuJjDOtgtXu7a8c83bldn2k+AvtFEVuUBdNyIYTJwfawHhqmRZMD
2MLdfbPfe//CeMvnhd/6ZtldcVZXvRp3t1IjS1Q4mzkVXwz4S0VM1eRzJVbC6GVEEDs9i4Uu0/fXORg+58UWVrI35etJN4/VjXkkAYOhcpj9DNRJ5OjfAgkJ
dInlYYLdm3WmNeGeLryrCe8KR0AkwhJCSb7C+qak0UEO8ly71gcEyKpx9Hbx7Pku5YyJvBTu0QgG6EUdWbLO0ntoblDulhhd60wrwNpotn7VE9KbvQfh/by7
aX37IuvsRoWHZ67jPdIIHaDucrFO6iO3b/GG8jT4h0SnPX3TxpHX+J4lsjZTspGPH0YgzubzH6rCKIAVHEgQgmC1i57Zy6WzEWUmOhkp/OZ4x3mRgb6T6Euw
TDiEmQrePVI4/DDBrkTbXfXsxduaTVMUeINOOJ8KhOHFLxQmU1GxgYZERSosYzC7Ju8F1Y5ShBSIOJdSOJFClwiQHDgcGKhTKQtpFFwN6VjDCgS/BydHz3LU
pQiUV16Gmuz+pG5OlKIHCFMQUyJ2cYtTEjBYJaVJaqETLu1J3iB3ZwShcBSZ1LLZW8PLj1K5xjOZKQIoy1aqEGgw1huQdirhLJINhzE16COsJaCTQSEiRT7H
GPQLv6stTr+U6Y3Ov4CZJUNEPYeGUragsmFWLIxQVgpER87UFEWMSOVkBwaaXnPcKsAwebTQMIyZgQw9mWBgn14qBQhFbRfLbP43sZlBCryFDoHhRfWVF7Pi
UyqIa734rXh2Exz0dpOmPnR62TBo0VyqVvyqFosbQ1YCiy9qpIiuuKDDhMVVCPXd57tiJXB4yasWnQuqg1IPNSjMsUbGKsJ5bzXilJCEbah4StIhWXYRRJZU
uTV+AtkXTaYPhTCyfvQDghJEqva045b1fWOecviz87Z2M5jkD6vfPw66tRdO4Q/LnhG9zuV+oa9W554Fa0ymxDeZHm8+uSLRRH4QcspZqqPRuPUuJw70ZSVy
ND2diSeaO5DkpgdMGUEtI50Jb4XNLiDgRtBDE/VKDc3pDx9/wx/A4AH39Na/J93mB+UyKFoFpA8hR+kLCxaXKuh9Pbi6ZzUKDXONYmg2ltI0CJmwfqZppKKJ
pERMsnXzU4YeHNHOfJ82qv0cK7caV2r0YRbnd5Ulqj6IZpHB6qk3aoXG2yhZVTw1uhRUp+bwXJKyyiFul+pvXRTTm8e9dzrZT7Xju0eOr/H1VpTskP7dm0au
cbM8Nn17B+O5wME56aBfwQ6a25fESajPGIooLJ43xvf9zcO1HwpkpMw0EwrTYbAkqHTdHicEkjmUR9JOQOr1iGsNAjCQg2l50ghsQxfb3RqxhNkoh3hD2Uk4
iB7x5Cg6bFTYos06mF5LI5WEqrCc24W3Xv6ciFbfy1CKsgxl3bwhoCw2A8xco32hNq3dZtse4UWa5iGcPGUTIcqEID1G4uDU0lIHTWSPfTIc2a45I1jlMMDN
FmvHA1l4PNVLORa+QrNEt16Eisw7ezqPLkj3bJHEYSPbpaAXufHz5Ix0wmeDcFHCqiLWKQeGOzFYsTy0qgahKajlePNRB089l23FKoZkcfXLi7Z+36vd7/C1
m8ISe/7nvhwKfD4S/mqJkNBd4p0pXPDpYKtDhkExh8Z5ZBwk50dW6zkKnx4o98YLw1aY58UpAreah22adEKkbg45IDBuNTNpx3NsfYzUIBFGQVgsEgEgmrCE
kJVKsi/API58rySAuIC2fxH8FgWlUvlYba9YY5uq2W9jaq3s0kC7rZUH4F3AUh93Z4j5oB4CxUkPFi08NpKTeSpuRxEsofVCi3vWWGS9awwfkGMaiRIZgjUI
y4kfzOBP4D+60AuN/Ew7BmaJg/yqGhJAAsVV3sBY7ZBYNk0A5UeXzOw2qtYLgZZ9Rdigpv1IdAyY1LvQ8TO55ZD2Q6gW3664o6m/Ft+yu0K79y3JjjaVdDdG
FUYkBq7LBSx2w/qEhkzTYxg6WITClMbGq0LuCInP9lZdOGV0Nv+iVvAiHgGp2shxTTnlvZtsBpa5jmcXMKDReVTg+c+qBw7LwMiF/kiBHuHSnTOwnuoLuf+6
kxgY+h9aWHRjJE0opkIyQdAoTuHxD2gvkm9nENnUK74qUnCCUpw7wLJ2eDEMkc5VbQCZl1C0jWWTHKE1YIItyMTcBBAudtdFoUIYaLyPQNN6H0h19+45kYun
IlWZuMTTfzT2J2nPiDziEN8qSEv37Gd8k6+W29wzNcnNL9cAj0tyy0M5C5u/VbnDcbGwNN+oHIDHRaL6OpM44jHRTe6SyhqwveY0wSUGJTDBnTSFI3AMW7cP
X0roHHdCLXWkhOHE5p7vHshj0Noksj1CvCB7plJ568BPMeIdwlGo4HJmcJo+djnqrRm/IfAjkwZpc+RDPzrQM0NBuZswEhmXKZtjGBYNuNo8yTF1n1WsU7ml
bb+5wKBkWB4hc0+RKRtLWLdCBXjz5fBNKbUkwC76rzAcFz+SHvsdv6qKqvlG03GWS6U7j6eNiQCwdPGrSUhGMOBwcLZmQCBAkJAmGA4Y5OxkX8ba37McaewG
iWN2Aa7y0wWrSzGmCwvuSDL+hw3wyZTCgsF2iXGoVcw0GibbBCQIQkLkvRjcdznqO7F+Gz2d4thoQ9kspijjlWZlA5dgwQcCcFhk2HKl/Tst+vsDU4a0Tett
4ajpTmlczdzDRxrIDRNUN1NBwwolVuAvhd1XSPtaObxXY/YGQBm4lH4epUPOhrO3PRUCALajmyu1xLIDxZ87OBCyysyzqJbEq/FtNufICwZ7k+SfPG3xi3Fa
FHyVKWYvz+6HpIIe58GGxaE4o84wP8he1STEk+4F7ZOOyqYuoIvo7oBe3p43Pdx73x/fsu0/xPf3+xNNNDWe7w8H/ZwPzIQXsFws7okKGFUQaKKDGqfbA8Sf
RQ0xNjSyFBQ/XYcMZDNPmBmZw8X5jnlPlXXDr5XzHQ54dPmqDRdmo+nqKtDnJXo4rXjP0IWqMslJagydJk2bPE5vP5Bi5Tod9ytt9HU3hgkLqnyw9O/kSSCx
BdMVpAnX7B3yrPBKuyEu7XpkC9JiSGbiS4H9Q4A4Gb/Hzkk+Gr/WHA/uIm2lDyfbubkOJzy1DNt+TomQ4BnEhMFLyna+Bua7qCxZFDYsiahg58VoLGGQORf7
ZcKMDtI1YptVLfpOkxuUlOd1hx3di7lqMM16GWj2a3riwb2HQx75geNGbRX/No2nG3IwMXA0f6yEIv2d0PQD9MowAwVt+7PtJUylZck6R3lDhrczuU8bhgRe
RweEUGQT7vF/c+AH8HQzsoQgmRmiBuPeefMcKISDadGQdE1JWxqwa+HGBiwoNkdW4fzFuR9ymFsoLvUNTdrWdoFkr0Rghv81DDws3ie4zVwfUHmnxYSAQhIK
F0sMuniHsaV0Eh9HDgD7fcZ/fWy8z9HAtA6LvzOZQMS5uledwumeYEZCAB1smBMaehWdiftKxG1auKjYQ5dC6daS2DhoKFn0PI15VY/CWGxFcTw7SBIESbYG
8KCLP+MnSmtxJPRpCgtXp9D0bBsGofy7MHFEBgjSIdFIdh9FcBwXWZDIM0EMcasSKvTKDU7xtKD5H1ht7o2O6SATUTKPsdC6n6PK/I+4jia76Kkpo76vpFzm
7EjtZMfKisd1u0tLkmBnBsFhYZXVpSwwhQ9I7MgSIQRKoVpSIH+UgnhLSMIpuJ61TiVQ2AlCTrVWSdphoKPxcXVhCyyp7TMMgWMFMli0czRPlAe0UbvPZ/xH
Q8Esi4viVsnwRngdE1PMxPjfEcbZ85JJ2E4u7Hxm65pq13MQbO56a7hHvF91HMeJvJ2tCpIdJCphgJOLKpEOZrsRXvzsNuXnSwPDhXuDGmITR2o3uXmbsQh5
zE0wF4Mjfo7GuYkyp7EhvzN4t6LrQ2HzMzfvTs8eTF1DJ9FjAI6zZmYEdJitA0EZa5ZaYYYCTRJJAkAodK0VQ2YFxiM0RQomkEu5Q8+h1hhiKGZqrEtY4n44
wihdsg3DEoAxfk/3J2EhvIc+HKk6cBCoW2zsbROpqpfM9S8pmQ8hsa3Sw4M6YrAsNTcKoYxUlqiwYBWcyTC6xCsqQTin6ykfsVknZBjRCE44AUWlDVUrEuQ4
uQvtPb9NbtMig957ETOJDyrIxixEFYSZDft9adxqKCddsFCy7DuVNTLOfh83VsQgetoCwVZ26WHvy5Pznz+8Lm4LPhqD2kj5BRQeBt7e9gtZ85ayhDzGoUmH
Qay9R+6waIYmOXUlwtoaGirwrSyu2yuw8PApbw5ZSzYEo50V3TNPu2AOacgMyCWQpZBWRUKyX1ugVqvkp4mpTaGr3pkPrledB+8WqSTEbcgPevqgnDyu8DqE
MKQKYJUFCQIQCuZ2ugHlZT3xLRV6NDDtam4Odyih4G8hLHLnWwa08vog/VVlgYwylghOo6CGEPoIgWTJPIaQOt2eqZbqyXGVTFVqCqcLDccHeVQSYGKvjnYM
saAKGRc37aQoC7LjlvRw+s/HvBuRQ4DFApRYQgkTxSrFO+SQFhAIkQPpg1MAuqaGuwju42KqjS57Q0zLGyj4/RCe+nWzYsNVKGl1ofG1EbySKERuPRj+7LuA
XtwoqfftmNisXYvmSEQjxPtpwh0wpWwM1NCgsqFz8tHznQPeHYPet7DQ38QLFIh6bjpiGAbLHudokJFPaSpRRKGjxoqDIELRSgkY57lHwEN5S7xkdhsahZQw
OqHyxGiHxfVAgBbFDRFDV3JcaQhFYnRnzIUJnSXsPin2GyyJwAMDwmHpyR9kF/BBhOpBatSIYm4TTdRbmiDCJaAsAzEG+Z452ILmGKiBgBt0aY+TkVGmc/Ym
eE1AJSCQzF5gkd0wSA5R07NsHfbF7nY2De7IOw2LyQWeAUYBuCpInsTY3QTmNPQ946RPSUephYupR/tM/xGzmbx9B4vRp9oaIpBCIJGSkulqRRtoKgkQiEGE
SIfHbiQmHU45PtekedqpWnvfKMHtViqKydtubHIiYw8ggcLnyX6NwveCBqPi1Su1j666zQ0A4B0PoxhgNA9Cbt0Mjx8V8OmEH8G/hfvap3O72TeZvUpTyQnl
0TwH1CGNjvcvTbv50ZwcioZntj1s4QDv7zrVWjxIWvTcT2+uvzxXIi+X49soBl8Acg7vJwIJFEO4hewJqD56AnTGAnDBDLmLXIBcpbQIiORDfJaMrjkRGjnO
DjSt4COyUoNp3qpDQNQiLcwCeBdKGUbkaDSBsGLRlcIlqko0RIkIiBMqha1NNFJCKRYF9Nj1Ztbu292g7YlGZaAbhuCgiDWTskBmA37j4Gn8HAmScCJADRGI
sENWomsAqCRJIBCIJQOiUUDswQrwg7W3aZEcEMxn8+BufTINBgiMQtxEDVZBgxRJJEaBhSVtkKRQjCInV935/GtvXn6iHdEn+MJCAFAjvuAzkMGEQKOHSMGM
ph4SsxguZ0ZJAIRTh4BhFiQIpOlmSsZdg6QDHFYPn9D9mADoBgCfP98id9fcQPPZK+G6xCDeDcqJeFgZA97UYEIOQ69PxKSIdEQOwYGD7YDgB5ReeqbRZ8ob
5uE0cjZDYQ7iU4+yemAQNwp2x+2QiFuIC5cfoA4b+YhQbx6pLdS4a0AcX17Q19gFO/5kJTW5jYNveLwvBFYzYZJywMQxcIRrOX0zvRCN6p+mSRgLJII5HfxX
1H3JH4fDfllVdDwNGMSOEzcI8EKf3utc3LUmx0BDEgUkIrIBIjBJDEMBjB9Q8diAPiUfUdRy2htr6p6DjM0dYWCEBhAhJM+pjZUGIApImCAqiJEqu9pRtuG5
pdlmB6BmYmKoYKP1u2CGQ1icFYAZkJBoqpUAo7UBTBId9j4nQ9TtdGKfvxKE5m+BtwQ+IH3yRgRUxMiMCHLOjrLkfJZxrfxQq1oeoaIzfO27fblQtljbHLbp
iybJ4gyjTSinvCo6hz8fkrwx9JjkSjxHicZkYDrv4Creg8PAA8ESyHJQyUFwAt85cWTmFQ0TFfJJIpmHl6Igcw4HQw4SQROPlhAHnzGKLKpDmRsgUkGQnhTV
ekkkU6dafCBwy7GhUlLF5bH0elcJ5ORg6xN0dFknE0Dud4u0EovllF1CISsES3jw3RJfUiwWCUKGYlx7DqnamxnJJ23dWHqHQtbdNHzUWK3q8kwk+YmPF4Qc
4FeDijDbpdizuRCcTvTOacZLUbkdO1LQpF5MkhM4TDxBsRhDi03bGynQ5FF+fu7nd5d+uqIFgqDFMHGEGBDrudBg7x/IzvHY86PujoPC5sNgxXxI8lb4hc48
Hhm5jyR0+wE1E2TAvIyRMCUVAM4JfByU2tHfdfT6vptvy9bBfAnAZL229faLz63TZ3V8x5/bvdO+A7gJqtKK5g0jBixKLJFeQNHp9fMFGRsmEH4OzWpJ1Tyu
E+3r3cysRiwhnSu75DoixMyVYp8WPLeLvQ7te0ZZVYek5GV1OEzWwWIL0ye6Z7WFAFpq6Ja5N6kSIdoAbRAr1OaAYdMwNcQxVnDGb3tctYuz2ztmyWWBrsz2
vGQSoLXZEWbGxMGjA24BqWFRDMBEOAzV2qgLHduL+dAIsiGhuF6Ke2AdmHZaIhCQi4ocDGhC+7bdFLjAwQkZFifDSmKNSpCdOrqM3riaszZsQ6VnMlCK4Qkk
BM8gPmOUXLQNSESMHhu8vefj9K1fANANzFdA68C2ANwL7DdExYZHRRi1EUIJyi9SJhFsFQQ9xgXuNxtDLTf7eFu+FXv8MjsRDVxcg4hyTFHcRDB2jDxsDBy5
lBucYidgmtymWyVElpVxllHUDk4DgLSu6A4jBDdgofhMOp7Bf5D3P5vq6Ifaj584H7R1HpiC3BeNoJqeX02q4h+hUDEfYY5ofzHUo6KQipmG1CHoP7fy8ZGM
JnBJBIEAZ2OC1sahlvEBJDLdujo2SaEII5a0hcMXS44Lh7/j8Djj18MvrPZgNuWWyffAINcEgxtpCOJI5DcZoQZAHxsxcRilRHGMXMVfqwuJoVYqiyUevehM
aFSGiFmw44eqBRQ+BQkgEYqQDgJA4NntFpBM0GvHhC7QOQZSwcgyzHBsdmrQulzbFPjyxvU0/9vUTS6ahqEqSSfR8/2h9oXh/8XckU4UJBdgeUaA
"""

source = bz2.decompress(
    base64.b64decode(
        PAYLOAD
    )
)

actual = hashlib.sha256(
    source
).hexdigest()

if actual != EXPECTED_SHA256:
    raise RuntimeError(
        f"Stage27-2C closure payload SHA mismatch: {actual} != {EXPECTED_SHA256}"
    )

OUT.write_bytes(
    source
)

print("Reconstructed :", OUT)
print("Bytes         :", OUT.stat().st_size)
print("SHA256        :", actual)

source_text = source.decode(
    "utf-8"
)

compiled = compile(
    source_text,
    str(OUT),
    "exec",
)

print("[PASS] Stage27-2C closure script reconstructed and syntax-verified.")
print("Scientific actions: ZERO fits / ZERO inference / ZERO target reopening.")
print("Launching closure...")
print("=" * 100)

exec(
    compiled,
    {
        "__name__": "__main__",
        "__file__": str(OUT),
    },
)


Reconstructed : /kaggle/working/stage27_2c_commit_push_verify.py
Bytes         : 33653
SHA256        : 68256e54c8f07382fb8a0517053479d6f8b85a4fdcc5fc83034739eb9a882a74
[PASS] Stage27-2C closure script reconstructed and syntax-verified.
Scientific actions: ZERO fits / ZERO inference / ZERO target reopening.
Launching closure...

STAGE27-2C :: PRE-COMMIT SCIENTIFIC-PARENT GATE
Expected parent : bffa77c536cad211dea9ef46b102e5482e01e68e
Local HEAD      : bffa77c536cad211dea9ef46b102e5482e01e68e
origin/main     : bffa77c536cad211dea9ef46b102e5482e01e68e
[PASS] Scientific parent unchanged.

STAGE27-2C :: EXACT STAGED UNIVERSE
Expected staged artifacts: 11
Actual staged artifacts  : 11
[PASS] Exactly 11 Stage27-2C artifacts staged; no unrelated changes.

STAGE27-2C :: LOCAL FREEZE/HASH GATE
Required artifacts       : 11
Non-self artifact hashes : 10
[PASS] Freeze hashes exactly cover the other 10 artifacts.

STAGE27-2C :: SCIENTIFIC BOUNDARY / CLOSED LEDGER GATE
Friday feature rows       : 70

In [31]:
# STAGE27-3A BOOTSTRAP UNCERTAINTY — NO-UPLOAD BOOTSTRAP
# Paste this ENTIRE file into ONE new Kaggle code cell and run it.
# Post-opening only: ZERO model inference / ZERO target reopening.

from pathlib import Path
import base64
import bz2
import hashlib

OUT = Path("/kaggle/working/stage27_3a_bootstrap_uncertainty.py")
EXPECTED_SHA256 = "6d23c28597b4e9b4d8680f4e6da2e29a518f2b82d991922664fc7f86333926a8"

PAYLOAD = r"""
QlpoOTFBWSZTWTAMNTQAMmD/5X//bsh6////P////r////5AAASACAAQAGBL2cF7d3vee+3yb5d1Ud73Hl1p647e+k3tH0UAFKJ7NdZUB6Ggab4oB733tDTA
XNgfbucGgGn3fO217ub2envvdu6D2mdPu0DmtoT677wDXaMbK997XCX157d7Ofb4t1r2b13b7u7GXbOtFsbvbn0+kPph3wbvsrmDCs2evHt7r2dx9j3u73e5
3Wk+c1sGls3yB7lt2MOs1Na3Z7Z3HN42PV7NL2nBX33ffFMtfD3teet3ZVu5zWxp9U53tt2d7fCU0hAAEBNDQEZGmE01GiZpT9JM0amINMmI8IamnqDQASmg
QhJkCDQI2kampkNNo1D9JMmjQZAAaAAANAAkEiiTTQyRPaapp7VPJDyT1GhoaaDQPUAANAANBoAGQSaSQgQk2hkmUaNJgIZGmmg0DQNDQPUAaDQAAAIkiaCa
E0TRT2Rqp+j1NU/VPJMh6nqfqm0yT1HlMTQyAek0ZBo0DQAESQgQAQI0NFMVP9U2in6GlHlD1AAAAAGg0aAAHeQJIBHs9XWg/37UFGex6UUG3RmM/mwoxJsx
/Ie8gLHjboWLJGz2vl7KCjI20osBYjP50zagisWGkYK5lWLFlawnCaFkIv37TLWaZqhSadDF2tkuqGKMNrQ6JOboTQudsDcdRalE+/2zSTUICMiNGqcDRP7E
pmjMBrZMSUTLYL8MgfWQc1n2LkxKh3p+7ftVARtuoUa0k2CgK9qFyCRGjo5bbz9mI6wFPxwQ1IJiJIH5fhH+qAe2JGMGCMIgpKUIW2QZVSVBQqG1B8LNOzbU
IEyEf6/bqpnxqN2q6AUK5RV6vmeA17LH93RTsbc3xcM/xCstKpUa70FyzVS46fd1mM04UmLgfSB2Bqy5CW81P4iCeV0rr1VNca4+/wrHqzNauVlpYMu9/p8B
lSlog/Sj5cEVYEVVVVVZFU9iQ+P1L6STs9zwoCsmu89OYrA+5NgqDMVgI+6dH5TtFAQ2njP5KH+8dUYlSYnoGcdxH8xAPAvDm/lxZmAMc+tW06UuxC0QbkRm
lKFhbkpd0AzBtTZ3dQSIIaSTEhpIYyE1Qk8+FsAWrBGZcwUdYQkgNQYnlQurnwFQYwwcQMOMhQkjB05cWBiKEpLGKaZUUjIiCCMVFy4OCDiNlYxgrDGqC2xi
UcTgP+nrhRLpBDD/y0HfFCXoqD8428IVwpx0iY+KoULHN0ptE++PCPD2Vg94vLnVIUm/ldifXz569XSdIW5dqevpiyW+ZFmndcAAWhQ4kLHB6F99spX2dbz5
n0w2a8z0cigTWa+3eTk2kMEZDJGMVip0ww62KdHJzQt6u/rFycAZ4ObAfXcZcuQymKuc/9O74fLg34ZI+jJgYCdQKq6CotcoUQYuZoiGlGJyjKQwSqcQsqZh
QcpEZbKQXsmtXUaNiggbZbKeNpUeSPENMGS1WJlqTctKb5O2BSDJsoxhRGFrycEYIiCpnmZjgk0woJhjmrqYIjKWiyFZVYaEDMZBYCKkUHc18Npx6HBWJuXR
dtemjGFNmJ9ajv8kWe9CSPYQEqZioRAYbVyuz77K1BE7Lh6fTmRd1vurY1iE2t0Z/EOVS7giBBhIc+lMOflSWb6EPja5xinBtrGcogsffKVanZsyyKkt68uD
W20N0jE4FiQGhYY8ttZLK7s3Qo77bKIZTAwlZQSsEO2aE0RQNTMYpUFbo4mk6nSksVHYQtllboB6HA9ApoWmhshhFCkaaGMpQxOZlgNGIqpNZWKadZF0liAt
ZpkkUw4IucfXtH1NG+DsyZgeC2W37Gc3LJ6L4FwWMEwiHrKnle46Awh63PYcURjH5V5+wiMGOvaoDkTF7/RdZeYhiGY5DhAh56gYMHcNZMKIVwaiYw5LhkWF
KFlYMxyg4WoxRhaHkMKy23x59+NP1idQjv7XN9twcEhMhnVthBpSUqGzJQuUSlrEQtBEssME8/HCa0sJAkyGzHMD8vK3Lc/3PP9m6XX9emudsQc/Ycn7QtIR
jEf6bn1Uv1GxTUy+v/L7PAu9fcNSmGwgpqITxHO1+xExrz6WYNX9WdzH5pOhfC7mCKUygvZ2cGqzj/Tb6Pz19XPhXZLyHiGkNNvvNRgE+De2uotb8u5mQK2b
vJmCogoLBVWRjEQ2A0JlDNXBTELvK1tQO8B4xKgsyuExU3WALQDAh9mIJp+tOuyL8Xv+a1+ZQ6RBDjE3iMgHKi4qKHYgkiCgChdQAfTM+2yIJyiKqW+7w4WJ
D5aB0t+pxWigAH3wUUfJ/ZdrbTJEBL1LWRRSq1K8bXAEG3slr3VRK89I+yCm0E0UM+HPji/TAvWkOr/RtnwaOEPFWRBEFHf2HpiCh7BJysdl4/fGRMy1O847
eRyoO3kJ+P4eqXby+ifXVlOb5rOOrLon/I9dI19J8QFojaL/RSDt8lcozFJeNoFvpzgxoTOhI/5UpRgkSdyLMp3My6dY/t93Ka/i9v86jk6G84MOzMaPArh5
Bxe9FfH6erm7L+Bg2sM2H2aY5jCZm8RS/E858kx6em7HlJ2q5CqmCUAf0wiS0z13qaKmSTLjBi695ypsZ/82lvo+rFyyY0J2+WXzw5b26Tecgw5FsSEUodJD
cneDQ3eLt33e1husXCN8Fm5vXuarstebdlc3hTSQO9e31OWJ/gsVl8I740TtjClNTRdNCBFCi/o00Z6c72J+ZHkph7BbDVMrr4J6JT6UiKFrhJIqPl1RIllW
a7eojbjT0sPEQeWosmvvky2rs8zLy0YWl0ktZXaU20zz1nyCakuLsBRVgsJqMiaNGdM8UnUZ8IW0ZBom++h7ri1NCnvOE5wGIVdPv9/VHfv8fd9/Rz7PRfml
e2nvvYb1ezf23dV83Z2RjYTiKPVheYH1GPdh8FvcQshdu3fKptEw/Ddrq1zRTRiGwB3KJ5VKd61NqaETMWqYmu34cE00wo7L/j+3fNOrsP/KpQL4QbgYg2KX
H2/b+X+G3osdPseD8emOXeK+PDiD/iyz0jtq+Nyr4sOviVcrLeDOUSWRZHPk+S9qu2oz+PjxKW1zhOHznSQGiKB26IaMVn89np3cei3BmCNmFnbN1dWri55+
nuotCsNheaCj1fw6otVs6Ide023bc4nq09M7YZ5HXs1Ki6Zi6GNceKo9TAwZHFNNpfg0VUc/ydz1PdXbpGb0wKssIFM5ZsY82qo/ApPzShP6MdPFWkXa9FHD
o0+mJUJluun57eAm1bdlRQ8ZOR4masYyBohZ4rq0724YQkmXl9ueplfBOgj7Ky2T0+pyJ+Yikzysx9o5en3OFnDTujYaa4mh4NdBl+iZXRmjzq1HX7YT27YO
3g0+MJWttX8aWkoW/LCiErC9Jq3hWfLNQvwU8tMqd/zHZnvJ68XcU29uc+jhM/DkCd36naU3W3CGwifQTK+8PfS+WcVE05V3YL+vPCXLYs31WmiFUrst84Yc
E6moRQsm7YV00yVEKYaKdmGRG+hCV6oLY89ZOFeJAMRb6KpyTXpsKSurrvKQgymTsnZ5p0YVTdFnGy5qtdCKs4PPw8sSmi24efB6VqmOzp0IZNMnitbYsSSE
qrVcshjA1t3YwRJWz/6Dt5Wu8bejs+9+Pqznp3brYHaGLGXq80Iq+XSbNsjHXBQ5VlzkOo3jJQEiwkiOeOfblGBpJWorjK+zZCaC8GJ1COPbZ7OfOfKWeZz8
lUr5BOXlDcKhr1TqbRs3tUX1C0UExE+nqntGaKHTiV9Ni2o3A3Rj3The0fydPi10OWxbuEl6bls4/DwnHH1VvikhZ21TXXnV1u7xywkxIxjHTM12UIkll8GO
GTg26by97WYjDHbLy97UTQEgdKb7B7GyuM/To6nwvjCyckNMxGJQTcUqKN0NuufRWefblcrjdCtzXi18hzNtzuK2u625HZz7OeU/uIci+Df1kv07cc7dTRjW
2pnG/gY8VHTcxpBnz9Y2azUxPN40Oz+voxfeLWGFfbGWaStHlTxsaK2LykyqZp0alGtZvqhAI5eOOnOb78MIel19W+m3xg7PN1TtnwWm6UOj0V3cXYJzHFFW
vj1aMaDyarKOLTaLy7GWFIiSTsjpqVlC0SDJGd0BqgRpfTG/ZmyLuaevBhjlY92201WaaFo0cF0yKIi4Ybxb9DT6uLKxBVmhah2xjBUQSwq3KuD8UYNC5KPB
l3KOmmReebOqUzXDtMWRo4oZcVFkqfdjNBnrGzt6dCU4r7rct+qnBcN8+VV89laexTIqmnB6EHl4OU92mjhweytIC9kxoHcc+jfOFEWHXLNkHLr9vyU/rX6U
37vqdmaKZhvzrAPx1v8vmQwOm+ZzT6/j+jjjofyBI85zw8ne1eZcPY2LjddMWZ0T51yHFfCDF9tArFBBNtc/LCUaM/6OIkeak9PoFaUrHAmLPh06CDk4WG5C
G+VBRkNLsf8O3i9WqjddUK43XwgkZBqJSDBbnyEkJY2IpKsb8NEpt2v0Su1up0VwgQIZwe1prgvECFbTNOJZlEzTmioNuAxY0EbHPmNUSCBL402GOnjFTsRs
/zoD0ot3nzvNTttz5P5onp6rikkugmTzB1Dt8zdTIbwNe/h0Jw4uzQRtphrmXGiaCjTJpBEhobq7d69zpA5DuKOee+0EDyZJmSGSDRh/TrpDHdjjEExthFO1
Vs57/BiPytwZdNBlpnAzxzz+HQZshMZJVTWWwjrJQUyiYOw7FfGgvUau0H6wqcg45KqFMdRSg6RHV9M3Ltroyv3cha8n08ClJ4KEHgoRwoXTDJQd2usLDRb2
u2vYX8ln44ebpZCSSj5Nt2dBSbjaQJz0h66nlfXlfYVk8sKGvpSzjGgzc+qX2ozSIDxK6Fu0LcQWwkTshDPc4qLYM7ube5cvEU5TcpZFcbkKGKYIQ7q2my3R
w8J0eXGQCIofadkWkSkRDfOL+JZQXZgMxVJf3yXq6aVLu7u9siLmhaetZC222W2W4TMhkK43OAHDDEaI7KzaYGbNF2ZmQw3DJlLkwxRYcDRTRw22ZuOGrbXb
C4BrRTRtbKaMpGIViHYsn+NhoYgousmLGLYmM4jE2mBZIh0zVBaOB1RcM4xM4Rkre25jZYpxHLmGnrebrY8m7umEHXWSNJtQce5mUMlx73re8W8YMY6btLpw
dmV5Jj5d7M4buOTLvCmszAtPy6ISGNOQ1mN6wIdgyuyGQjG9NZuFtl081xVVrU1G+Tc1mo2SHIctqze7LBqQ4jprVHSnDLZGtqJXWBjXF4zSJsIxawpjxk3m
aNZrVx5S3MNu3TZKM0zG7LMex12RYGTXIxMTQmUmM9ER/nVXMWAriVAqioCCrFK1LbIo2ytZKxfiP5OPA9x7/eZBdfQy7ggVK8U2hSP0qmAQYe+JhITr21JB
Bg9j8i29XpY22P0lix62nLsH0oJSZhYTY5ePy+WfF4VDhl4FOQIjmFmtGo+EVgTualzDm9MSwvuimm3hnL5TW1bqmjHPT4pbGVxoL734vjDZ+/nYwTFQnb09
R501y7QwMzpm3N6ZJu26sdSjKHtlgzmtsloZIGF3NtRujKoTck6foPuJpuDUjrix7UVxbo4VbmHzqnQ9RbQUz72MjRORHF9Djwo3rfbUq16kMuyW9FQ1FO7e
lAgY2nRNWDvzTUUfCiWCYc2MMN3R2Jm7+hSi5yQwbSYdnjBNKCvFhoOJO2xKvYrx4VM3+Dbby+b8MlQE/Qyoo+WhPXlFYixAdewtwUxlwrFYNtFREFFFUgpU
WVFK1kNJmWRNWxFhjjjIuJQxii6ZVQUFVYlEBNpshu13QDGW0FzGYYGVKrIpEUy2G7jJiAVKmssiwwQUjksI2hqaYMRwiKbwIdM4gtntOpc8Tuz+Gm0mUCDm
4zEcfJFHhVbcfi76ER5CbBBKH5ZSAYGMEf0SMR0XGvjyutmF0pgFQYkhpUCQ0Phrw6XXrlRvZVX14c5+iWbmJvuapbhvAn1E9b8B1s56fbX2L9ZfnscmfjUr
3qJ20ze3LkT9OKaX1JM2+bnZ/tn4cCBqc6k4Jg+PyeWMsJiTcZwJUCDnL3sOGQ+ODXnkmaYECZdQnGJGb89735VhJBXfEBC0ag4npP2IEU/1ncGZpx37838n
vWJnDvB8Ovuu6vTm30cNEEbP174vHNzFbbm0qdj2vgNJO6zjY2eLatdkBnkvDYnMg5yYfxh0vO1+gppVEVC9520yrIKJUfzg4G1hs5fJJtJR6n4JtSOPbQe3
4rnT0JweEA576GaNYKeBhlciJEYBYr6nb7ctcGI2gyBKrv0mRpi2nmNF7oP2jwaU0tzSbkl2+noWusxgoz6F5mz4Xv234Wx4eicTjnygLvt+vMISKQHXpzUT
DA5UDkPiBqtye53QLgcgmnizBqH2K9UGFVFHJLCimyedb1ftbalU+ESRrw14O3fS4cUnzQBcyYwcrgOxzKSCqUgtvcvQcwnFQgoR8+qk4+1FgTjacqdEIDRU
VPNvFkEbR2qeBDh3TSifTUFa5T4OPYU/npYbGshP83JLH6D3TG+aUaumwcsa3reL5YdNUmzj0Ehb/7UzfhbzJu/y7/WQceArQtPhsbmFkMoxQgLQ6PF/Ainz
OQuFmOXd1pCS95xb4Hd46SMra9wegIcCyh6CQRabRrlGh/1Ex7A4KpnKadMipFB5eETuG31/N0xlJOdSyRPLomplhGjX49r918uaHL7Cqm6aiaHn8suHK98f
GbRCEy0QJS2V0aJOLCVMUXKmsdpKHjH2WS2YWQ7EWTd/sb110yJks1fi+G3PQJvUzbYCBJQEDkj143W28aGZeAhylEDYzhQjX5C15lNvp7rylmDqTkzszOAm
SZ+fyhQ3p/Xt3aQwUFoQTid4Eos6d6GwTX2rtU8eQ1OwWNs5vj0NKLI8cY2kSzFxtrVw2kM03BGOeTzkwhEiFl0HFjEyPwNyDhoxYoA7MEeckDDX5vPqaG6f
AGbyb4G7hh00GHQDjn0qPrccdweIurzwLbeO63PlLzUyIP2zGefLNp1FE85izDdXnzzSPyCZa2d/EHkbQ01PJKzgYk7R0GMp0rGd/IcYYMDDGQhWgXND6MjG
Kr52FBIr97CfOFNLCfrLhi0zPwr9RBwXhSOzWQfjqlp+c2P2Oj2htn+07cwyxkn2wfwj4QTZidx7UDeJJ+JOE7DMcoMm1LQcOLW6aQ7bs4kDO+oIiZoIRTAu
CWqtNLFefiG11a8KD2mobljvRHieCHxowAkWJIyAhEAiwvZ1FO4wXleg7EwNTUAlKT8XtPTzpMHBeO65qii1hgsAqIbJ50ZcaS4Mi3o4WOZXWSHxht8M8/j6
5EJvpwGhB7IKEZd1twWMmGGidkN8pjbrAB/LinsUa1LmSJiwqYy4gczejJVCBEMZtUxQUItKMykxkQp/RRpKR2pFxcUCHpcKCvy8F4iOTc5CPq7SUVGh+/8D
lFk3N1PwzFnRXxoN9NF93eYXkoy4mfR7IglPaN/XX14s73e03y6cx29+GmW37YiXY+jYLbmWKNxdDVYzTCDnLPjed3efueuyWdXYnqHOM2pMxxfuRyfb+/6o
fZ9f7/4VAEpnab21FaiMp3gjD8mtQCm7Ew0OUGHmcob833XaJbV/HVQEPOGHwxCLbY3fdjrm+6gPlqhR5xaIBKo/k142Uf6VYLL5x9r5fl2/h0M1MCAKGagJ
0KJEC6rsarRHlxuFyJza5q1LbnNI38z4IoXAJ4nuDxD0BqRVws41/dwP4brWG87cLFf4e7j936/oXqTkdm6/5BJFSRWQE+epKr/GwQYWiFFytH+rKYpEJFiy
GkKDBKsDGICjPnNAQaQOeYorqCjTaTICKYdAhAyFwmg1ZsdBk9yWtoAmxwP2V0tDzygQ8ACHHKH1+gs+TT9U3AhlQOtaMhGxy517+pzPuYDQ+a9fQL1RU7q6
+v8Hl+ZG0Q8RgRJfcP6HhRF6nsL0dMbA0H6w/qNyQcCOPxETIEC/4aU+nX++inKf28ET844OmQqTLMuAfe5DAxvE7NOpE/hrkqc1N65iGPxUwBiMWbH33tgJ
r/cLoJpgrvyDBwaujmjvY7TcObB4aFhYQDMQeImu0q9qQsZGxtALx4DakAohx22x8z54QgcOhsKcyxspcYFz8SWn7/7eRuMl8hewZ6wYupUdaPL1rrfrYbMz
mOGirctRgs2jR0iOdClGx99DrxmAtCCYL0o3Cjg0UTMkUT3DYR9eskRLodWwWUvU+QDYv06FNuPCOzphHZu790KiDCQhAtFoawNokxhG4PKWsppYB5CPNaQi
5BcOZckmKSpvhZ7UWulERUEUS0KCMVVUS6Dg1yKM022TzF+/TBiHKwuyWqyJMxxjfPSybWELoQT1glwDbEDT06hUJpyOngfjsXoPX8SrZZwayISupSITN7Df
OVswAV1auHTqQ/4CWFG7xrcDiyJ72yG527Y6lqxUtgmA44oXMDOaBrWtJcHRdbmguS+6aoUq8rUS/ijwudDNrqbyLqcbzz370ga4ZilN5Ec+LuDrxxppvDmy
6A42pO1jv2nEPorOpYrUbHO3E6FHNS+r053Imlqq0Q4wmSRGwDv4EQshnmGjyl7LOg2GjV5j4hsA0T19k+X7jAdtAf4PJeIdZyGuL2fJPIK9GBkkcwZv/BBC
PZJ4pO06VOgvlhR8FsJuFTd/Ah6fg2MDpFikZQSfGzJj8+B5gk0ebRhqRBR6gb5hFBQUikROo8D3AQO4NxOoEzM+MqauzZ2T0RR7pSAMN/RkpttKzNhGBaE9
5lyHn2HAiTGID4Yhc80Gre+1c4FHxN+N+am9yK1TiJtTtGQDJW6ccLcKOBYSgwXV49+R1C79iJxEJutJCb5XbYMVailPonAwqc958jtzSAbwT2SbjpmTSG3a
anzCDzIIW2eCbr9nI3p8NndJEO7GlLvJU650BTIkhRTY6N0S/n2xAfEdbbta3OoaLzDctBqjQjcsY8uiOhgMjCcICmZyI6bmuRwDQSDsmqPQ8rlMFPOyaXsp
BcK0KVBNqxF0sXsohYMZ2FIhEhyYrUfbJpzzcmhUsxaC2HBweGTAeG1lwEu3wwOQN0ywmlrSw91WiYqRFdLTBhDlIUsCXTok0rO0cKAoxFEYBHsRxNtYDnol
pyojHBNWjNQuRA0gXRlrqVe90bRhuNbFtzgPQ0JdqBk7IezATzQPA5eNUyELeIWAgQR8g6XH1Q9G6PuiOToFKdVvEatu7QiXLo4K9xmkBWIHwz6aXsbZx1kG
bt4LXqPI7FyPCRJzigc4Iltd68zXhxT0Ht9z78SfwS172NXgRE8bInvfAnugnnwkU9D99ShB6ZnzBpDQ8nHiGibvOH+g9X3LEHdQUIaHyeu4U2TUx0eAp6hc
Pnsa5mmOzYH4LyB21BVBRRVwD1jOnU9RJ82d3PXTy6UlHhZ1DZiKZp6QeCQnTaIfYCcQYCbW7LstYlakS6dnBOQmIAvhRaaBWkxhBU7Hn3U7qKoy2sRa0VRG
MJGUQZvunYtxAvNtRYLbtl5RVRHvp9FHgjRa346bktyo3PdQrQY2hLu3ASHWsLnuJ4knY4ghjPO5pQ6FLr1hJIkhIQh620OuAeN3zOtfdM/gtbTRdRd48YHG
gaUpKANMBkQezwN8l8005AJrTSO5xFkYgh4/H4ZEa+Zxq5Xrre7LVtHO1X89fK1cI7aPHWTmww63qikRm3NOBGMHXv55rsW7lrid9di2GNJ57WQ6DB4cZSmj
m756ETmXQtxv/MZHRHH53I48ELoGRFim9Q5YwjE9aPsWlqi2WqL9FqLktWRvOO1AgwOq1jkbHaoFwuYqiY4YEMlDbbadqmJNvVghAr+O646F2CYgkJIEyZJA
IvaqYfoOHdqJkqZ+Sd+1J2mFkHjPNeZ6AgsBQZVnqYduSmYIGikAeHDvOvaWL4xXr8rGWdfSFGFtqrLggXjtITCEousgbCmJQIBseIcANYLJg6WoqG8bt8xZ
1SmwkTcRHyCzvOJFKknLyJbBtw6sDdDTZFNhJW0SXbTabFc1AzHCnGRMMSQWLZLiOqvG5SabN0htbamRqqaKca56DGaBd0oc24lzknFYyPiLQbPp/3OaqbbW
QoPWHMe8DoDwdz2PXy8pY6wNyNQI6ikDkkOKJ26O8heXaRKSB2OHhnCjlg32umJ2vcrJrNCEKLWZS1jEFCOVrCPR5TWVVO0kkYNirAnM1Mi1qleCiLxNLkEu
IoED9Mn5T459yEbhUaqN0+WJ7WBgxgx2hGv7h8F/dSccecT43DQYisJ3TLLH2yrlFDSO9PE7mGFua38TIY/FnlLnFFhrsf7hjFucLJGRwlEpEfVROE8W85Sk
7pY3BB3HjX3+/EVYQynsIEbR5pDeBNy202Djt9G2gSpr85KnKuZjb6tw2+3DuBV2eZeBI1POgBT7PH6MPTKQuGUq6kZwWBIPifU+olhby6F0I8g/yjDAuUjA
iTFENoOBvRVVIRPfUyS+x5efaaKZCeigKHiAwIbhOCGeZ+3ZzsPCd8uhFv3HxMkNMBEYIs1dU9Xovw8umEOv39MOvFiP2jHWHc4MHMCYAsk6SEdjH10RcJgo
7f6BjwsQkJVYpFixVUe9QUVZ+bJ17p3ywJ9roWHhdYRhGEcwaxRY0AcViBOlfj4BedNr9BA2my1CYr7XVp3d3ebo4brJhhdGAT31LlDyUKhQ6VPBJLpUlVQU
KhQ7sb8GInTM9uwG1Nlq44wdsSEIgkUkDuRM2k30mqOeWCEsuSdq6rolpCKbnKMkIlCd5zURc0goqzWbBMsO6cd3VV49Jt28OCiqnb2eIHd9ALqb6O7wYSSG
TUEQfwXqpSQTtH2Se/db0OkxFVedHjj4uEyUT6HbfOetMux17ZaS92GskZEL4utuC6FBc4ZzUpDM9s9vA/JN5uM46jEURX5VsgsXrC7cy6W4jIBJJJJeDoTU
NHe9FdAJCl4N+evO24rqIkYOHIbOhwH35neHGJZZiN6SrGnWR2NrrFg/9OM+MqmiVHmIRbDHKYLaFKsgooObTrV3zMzgjOA+2lnTtfg8PjLV1w3X8MxrWw5N
UK+MMiwdJckDWIObD/VQx9Q447zW5oDScTuN6TsJGIR/vNvUrFmljRFolENHU5STMNhwwLkOxddEclvAhQH5tITrekDXnxTlpro8U5SQkkZepGhIxo05BeIm
5088mw69DVLStpVGvtIwmJtHkgjujEXNSU5c7JSLV1CBdNTRtJISZ4ElxOuUjGDgDQj37aErW2aYrnHv34JsQTs6ySSWwuhQbSZ3jCBvpnc0ioi8ccOp8nZB
vPlyBQWRRd4Zo2ZbIcoRkwcXM2IEJmTamx2LnMMJkZ9uhNDUtvpxRyNjR0TS45CTwPOjUOCajlsHkfXyiSMnfQhPiwDzLIjmTS/9t/b+g8I2SnaHaAH2/MJa
KzV97zoY4BgPQQ7qPG1VUoklLFPdAstIfseT4vkbW+x+/Xx0erpJEgBFGXzbFkPkG/HDkIp+Q/h5B4H3RmI05URs/dFARAg5SHpd46/Jc75owyeMjWlJzFLo
aTaGYMowDEMlmSYRdeEcJzgSH2yfuEfT9zHGxA1P64rBd/PAPjEA7gjlAQLF1zIdJJMmxyiciudP2rZRpHqLjnL8DoeZwHrNmyfVZYkixEUcrQlUUikjCAlN
FAEmwLEo8gOR7RNyUochPbJFIxUDV7YuFEMebwJT4U+ovC75KeoHxDvf5Fq5HFAobPtu3Zt8IcUA7lOyFwSFgeWyiw2LiZFseIAaZhQDoHgXRPRI+gBJCzzZ
TZQnjsFtSAXiJIjh6K/FekSmLCQgwglRCiInXmcB/28dMtyLzQq4NsNRJibENe00w7+Jg3oByO8s6K0B7eM0+V1PBPOcPv2lUvYZwNXjZR5qZZFk8jum/25V
ZLYo+h/PKLsA+8h+Yn5gIAHCL1+SPTx8+WA5P6Bq54K2XjEHYIBTGSRXt2YZ0Q28AovshQt4hENnYX6WSlpOHm5o6mMg8EtMNgynjDSmGEIQlK5ChRHq8rl+
vpEHM9UCC1TCHMMqaSOCNouZRiy0pIGAM0naBmwoaCLRcoTcbh/PmXDCuKBCRRP3kHaD+bCAWXw30uFAykdMyPEMwgpqEyIALSgLEGDkZo1MYzzCWYgoPnEI
TfFfIrvhzHnyCuRUISVEoslgDXi5M57FHCY2nQKo9JjyoW7wDM+pOnX453k30RopYO60N+lmSP096CHoS0HQghkUX0HB0NsMIyE5EhSYs73ZNQC6p2gLhLGQ
56cMGm1ZDIqZ35zGAKosSxvDY6inZDSHpZDJDBAqwZIkKEh5qxGSqSZYijCgpPtKhOU6db329k1hhmvvekPDXQhx1s6fPpEfYxVMcQ9uYOQ9DIbfMswYbFpR
jGicMqpDIIoGk43sNY5QMYsYUGAcF4oQbfh1aceMrQYK/yKbs5fapCbCaYHChMBCChSFkYMIqEUsv2Tt0A+lPCYEPNRZx5IdppikIEImZb4qne5592kqDCU1
RBiJtoTz88ZFrbZdxH5movmIbGQhbse6Ae/Hs5B0mZ+I4LJ6nUrVC4UbzzSWaNSQ4uULWMqHlE69exuTbmGAwdigKfiU65mY5j3THhptcoYYta9uf7nwNaQk
seH1R7YwjreExKDa+J0iCJ2Kd6Ih6JYkaR6f6yiZamymb16MUU7ajMeG2WBgSzCzA8WTcsRLyPrDlNoaIT0nyEj7JnCwIIN9tRkYczsE5an+pJtMkZT/mcO5
L71Oc1nkDrc4a2FCd+L1Dpt32MTA7KuMxSiSmEQcD2V6rJN02kD5aXcciRAaVNoRJtSXsudy5ZknENrF46wwSYhvH7H8Oe0ddcMVq9rz2CnOb7HhCAfcNTiT
sodrixhI1prxtqUck4FksagOEEKVpC2hfJ0zjwbKcCyuBMygiVKCJ0StlLF4RNytsi5jmpNPg6lS1AOIsNXecUsohvkvEOSpQo6iGKVLYYQiyMIAwXcht9BJ
0I8iVw7yFFZ42ywQdied86twJoWO2FpcacY3h7Q6IHu5Vu20RuS3J2kh8QXEENIN3mtBuuOTlrh0m5AxocdkCI2LbFXW+TQHCxqbuaZL6zPaxMxffI+SIfRo
oHIBgfrsI/D5UF+aDfhqgHbYXQGDENCDIBKaELr7n1/WV440HkoeJC/p4cJp64+vBZtl4Kc3ddvqAe2J0yO5cJzDmXHvR6T2x3ivfq+kglrBBkRjAPdscAhV
FEZCFEo5lrWr69jfy+IZvk70XYDPw3PaTObJY90zi2A8cJYL+whwXLPw8VfOigNB1LB0sIZd9LI74p8IAEIiEJnrC58iXwxPUxqHh5+6weVvI73CQXiwUrOU
sLt4mLh0CwMi+rUwZgkKN/EQwuo/xkSsHoGAaSZVg4C5D+wIdTzUCZBDJ5CBq3J8OHhORNzRDqT0YbEbZvn+Sz927Sb8njdMr2CGbUONxLOJIQcchPtbrL2I
guHa8WTBaWU1CwaQyzMwLsHJcQuoaF1QNISQJ0HdiJ6H6oeRCGAA9NsxOsBZEEPNDC8FDhb5kAzb4bKhAU1MhCKbpueTVBzb3aDnKmdI5AXb26YNyNS9DudN
LvF4qEzMjEdj95A4QT7FA+XZXrA+32Pip5GZTCSKWhChb70JiioAooKAwYQR9Xlrx5eKevelYFv2bgaWKVrfVtimrrN3WLENCNojKhkHLKlZp8ZvPbOsCXyR
2DIa4pmoND2w1tssPLoPnTFsHl80QeGxLl8bEVDp+gPcLUDtzUe8RLEDUB6HM44ueDv18rk3lNEKRkQvGoXjWiU+gZE6JLq3nM+DDlxrjbqCJphsyImWlGl3
kxNgOXkC+5rIbVWtw0CK6x2i2zohV2SKA4wfok7CUpn0xcSaYdxKWb3yC5ZDHgN35bBGDAQgxiKCAsWCMQQZBRRBBnLc9RzGgFQE2kevduBxEihEiZ5vqX3P
YALTgRR2YKEXQwpjhyGvcM7GzhxZ7U3ch2CMhIcoVIxeXbcuNgsd6ed5KG53iHj3UOxGwQIQtKhIAECRkWGxEDVl4N1pBYsBcDlEwyjMYfCaJIlBZ1cakGZ3
Hnu7h5R6amkTyb2w03FwUSqjFIKMVmpubAFySKJkniwoLRccLhScSkD2TAej7Jus7T0k+SkWLBKh3yFQvYK+aUYgh7wYKWJPudgIsVHWIpL+eOZzBnsgw1V9
IZkDtPQu1lctD5XP3C+7wTkmy7FhRFDZJLUWUpZKwsIKMhYxOsgmQdo0ZoPdR7lCGqSSLTHSLdubF/b6/dDBTyKgpG4Sjz/VQD00Aru43EcxReh8UYegD28c
HvdrTuwCUakqoR6e7iy77rSZVaNYwZMxbWEGZNt1Ri09y1FF8PO57uHTumeSea3PXWZzaZxzkzblnxMFGCVHEAOo+HmDhO3To5NiR8XdEYCBUVkYJAghI+BB
KWFn3Fe5KRSzvNHWFQjfVQNg3HuVgnIskqEEZKoKRAk8wO0GFT1jaMgDdAnQeIHWeW3hyJFNYB4QCgh+kOynNBCB8h6KFdvytn0P5ekg+CPv79V1Pn31MDEO
8zE80IhyH0j9yJyDjxWA9xYo1FpF87dtEdjDsi3DCZgtlfENw+EEe6jXlK8fj201MrfDj87FQLYCj30F7JdaZAsIFkopD7DxH7sHE4vAibiI8NoItiA5MkCP
jYyYJb3U0QToeiiucjjuL6mZnAsW2Zm7fQzcxdbg/iMQu9hdrkO7PAmtjbEuUyGWVQltFBzBLkolFGIKmSuWNLUbcZlgksrLGBYwCLEs2ooLFBIh6dzR7fk1
zA0Imuh5MucjjCctqQqbJCUZYMk18N0cIkWepBiGBngqvcxi7AEa3Ow7CA7AhhnjtO4t3A8FA6nUQk3B4nFLGbmO7cW8FIzvrITQE0PyJSqHbJArLBu3Gocz
uVmQzCiIugn7IlVRvqgqJA3JSn3HwTRE5sTgfFBfT6ESwegSAeieIG5O4EFYvFQkBIpQvEXH9ONHRDsnzDWrfLVpcF705CB8TpCbsITpxORMEOqiMARYhRRS
hf5YIQ9vnOaD9fXFvwmSO4YkAIQM/Lk/CAFXUgWmQwYyMiEOGovZ45yCkQUGTXg0mKCwICwDg+HqbKQRpnYQhaGJYhgg2Z28II9v1oP3fHuufF+/MmpsmIJR
ORzQOiRBXLD0yKFO6nUc8nogUpBcnxM1k3WIrPLuhmkOCUCcBsJzKJkXRE7sg7c+yiX6D380wpObp5uD4qTW4igYNG0NJUtrE64SyVG5BspHhHNa1mPUcrpb
DZaGGeCkNzAMu1KMu+guJ/0pFgkhshWQUm4cFA2ZDrBKCWWsU3bLKbFLytutJrIWXVkwyWLopYc3lEm4rOA4heY0BNIkUQoQ6B4oeM9AfnZpzgKGeNKPqeJu
HGbOcjlGBzW0YgosWwpZaoIp3sk6vwcvGHYYHTo0T6/dnkMgo8qN1XqUCSbyQ49gF8AXq6hY2l+SfgzKHIiH9iikhFYbL2i8IpUHbpaUc3FQl6hIS+MJnqcQ
fYWIvAIIG5+Z/e6Lrx7zlmIbsQ1gpIMIA7iRA4N7KNBiiosWEM5kkyETRBGZGSaA1DPP1rEQLQU4WpN3GB2tzD8yH7oyALYU7x5jz7s7vpmhYNI0MgasCDjD
wsUau4ZM+M3mGoVH0w7d8e/b5CmhBiL64fM4K2VWMnlAysZPb5hhBXUjQEsFoQCpAFUCKEEW0IUVJBkSSrGESgLhBqgZZiPrdPGCEMFLhVSgomCxSpZRotAo
YjCDSoahuuENrlpmfnEPm/V+P8yj88h+DmfxFVPoJIKkXy9PmaGMEhE/EgUjYP0qfeh6MSH6yBGMIiUgc3tD3frbJ8n5jrQA9B3jDKTaMhRXEbNY04oPhCJA
YRXBfChg+rpGRO8aasPxiyQUk924IsJ6EhUWMkUBIKCoSIe3ehkYHCEwSS0RMDkj1RJBdrGHQ6Zq3FbRU+EXb7Y0QGQ+8xYrGQA+CePuseRSZ9IfeFLj+oDw
9h7vODpIzEIQoiUeMkPVvRLaYOxKOogHcc+cNbC0VQNEQ5wcIIWohM1zU0Mz7yw4JIEIMYAn6r/hx4Nvgr+kPPqPEeIvoIsBopQzgvNIvCClQfEEn56B85z4
4dQRAFAGIkOZ9tRhLYNI+i1gwopzq/YoNIrZPgO6vJO+Hf73oqo0UndJwUp3bgt0WJIKBQKE4nj0r6hbOFtzoMy6GrZ29peJqjIMOhpjXQxK0GITTIeN7zgr
wOHCyDnETEUM/VQypQppqmSRi4dr2XgMBC4DudW2hokkECEkXRVZlmDlch8S9jNljt7xQhbJISDAYaB+8YNoPV2ojExjZqmznqM0ip4B6Rzx0eX2GMIMhA66
pGDAGIYAfsghkHpYTCrubgUDR0tRKaR0sl9c9zqUUg0Kht2sby7w3J+uSESRnT7BKPx7NdKXrvJS7qlj5Hx2Y2D0x1X+dA/Cf3LwLYbzgQxSYtDHNA3NASc4
qDk1uhS6EhTFglTHWR5g8gD3n1vqdM/YOU0k27EstLn1s0wXfAXGhjudjTlK6WmjgCAqQaWpUPEIyTbaXOwiGa3rAPEDDS3OZdph1M6WqwO0kFvBLAXAain3
N0z9vaQNCrlS0YxYIVknpDqcoBYB6fZSpeG4ZDWWRYgyCgCxEDxnc5d7DaDtAnRy0BEJGMiIYAU7rCWRkQEQCziNgVEtiBq7QUZjUgxaQLElGHMR7Ah1XxDA
fE4vYScjUVTCBypvdPo8gAr4PIDRdEpA/YCRahJLSgediNKW3nnuTonSk5H5H06LyJ+QMAZS5376l3aKSOLUFETtGs7I0vn83ZCIkYQwneoGDIw0lGmu692D
D161UsbzfazqL2POKUEGwXeNNLsnq3KLr4QHLgwYMC9rkJkQhISZNPUpu4stkHKvJ9nPXhpjjzAA7aYRs/aKPY9kwct+cgW70XORRvSExRNlhWbkRJEhIQY6
mMiIdl7CA381DxIA9EuvVAwjvqXXUR7z2UIDqncfpCUi5AmsCQzFahsohy57zYBZzvU8i6GwmL8hOeZzAyPcghwHV6TLwRWHpE0DklQ4fdixIxkhIxiLqB1K
yCB0+7AgYUfELGtC6GQiGSMaMLIwI0oFI0g15fA69Q/KBIBN4evEfdXl0ui3yyXPo8b76Dz75OgeCHoMMhgoIIjAQhQNJCeXz9zx7tpgoo6lKk8T8KnCnyG1
/Eqv2Y9PsLf39HEP0cXFwWaW1OD6JHh9nq3Nz35MyJnogO2Tp2XfUtvBmF17cuM7HX0ufmqDZI32Oyp5fc1HVpFvAMkJGdXNqEUfUOUENL0dJhJueeWNRzUV
0bGp3hkuhBsP1lyVX4HJpzZZxw8JG2B02rqToyjhPFmOb1i7yKygo7C2Nh2TqwfApadxevOrtoumzCdnYsUePVVNNZIkx3xlSJDzc2afl1u3RZTdXDjvzw8r
kWzTkSUlsQYk8ibCEMVhTiTmNQkTIVuYQxiA4abZvgGmNUgSdBJch0mjB6WuaOmAYbkRhqHdIwLgIjUZ0ApuZNGQoI0PMv49EnlQueGdmKRjEJCEmXgrY2Yf
C+kZBxZa95Ql3nQFsOPWHd2J7tyCHD9vS4YaaQagPJN/ZzISOyby68eNuyGMXvVuLUCoHMqs7vkaXNvy59Htc0Ed0Ti1k+qCeQK9mMIZOKVsaL2CwNxRxXmP
Ha/HqeJG1gtDUHn4q8OESSMc6qEISczgouuqJxPxVbkVcdtV+KdTaXvZU4by/jNynU7E92Iwvf4c80DjSbJKkSJzDr2A+dbaoq20LLV+nj92hTu8DY3vw4V9
tvj5lnnZsVY78ArKa77BGE3XmCTrzRy7VGdFImfH+LWih3Lt4/J8tKxH4YUk9KilHkhxIR5JyDgvoJjYhyVZ9ucJJIhYzdBe3LPtaMV4I/B/lr9T3F+7xIrx
imdI+U2Uem3gRNy5OWky9O8NNYwXmoYymi0pi1VTAsV3zhGKuMO9NKx3lUTcCmHbx3g1LXh1BLNOzkjVTEJ3gJmk2lX3Uzs1kjlKGfCrfeipq6BQaUuNvBEW
ymDBAYJnZNxJUJAow9WxsTZsbXomorERUYqxYorEGCAkREIdoQIZEAnjetzk1Xdw96vYxdouoc1cRFXNTjM4425hpEYCqkRERYmkNKqbrXDZyc5NdpG0MWMm
VKKANFb22NNMBdi4bHCBmMC45UNAQi5DtRCWC0aDA0NMRxDMLlmwOszdFzAyCzdY8ApCGO4Ml+6AEUHMM1A0UN/L2InuZAvsaxZ8z2/EJqfN2LjrD3M4BFJK
K9sLxthrFzYCiAQRlbaRk+jVMY+Ph/HWvBrM3pN8xni9qsYIcB8ex5SmQkldrq0J5SEsH4Rfw7DXFCaUYIxXW+GaEqxkYA5u+42EXNr56vvTZ91KyatpWTYG
FJkEAZmw0HcmcQkfxoM7CVAw7pFigCKCRVWE7/HvmSTmkPrIUOtACxBHmHcgyAXhhhfepmrcL+PpW4TNoTnd3kQSNLkEIh+XhdNQpXDGMCdm6BA6Ubh5EGkW
ymRnLJzba73EUtbGvbVSpffDcN/YaT006Pdqh3MqR2sLEbaxWtBYFgzTASPDQ0Q5dfDRsKKqyRMJlCbQnH7DuG90eIOgVBJqZmkOm709bL3O8h1d4A81ex9Z
ZUTfACUocQgJZKRcgaDiUg8zmy/f2kyuScMsYDKsWpLFwuZXmAtRfAF7slMghkdlFNQ0eQBZroAFkmvHgqvNoKCMKFKJKGgKUI/APRgbiQB5bcXzB/2TKilI
YTcHcPrbhxcFwCpJSJSHx5HnS6nwbp2aB6h3EN8LQvEiBOIeRQlkiRPeinIXmXLYTYHqPiSSJ399e61x9Qt77XnUi5HbAz9IXuYD6MCXPYVUCGNBHc+SEQQi
B7CyD7xEODF9p1gUimOv29sgSSK5bEOOAEZmLwgJMNqIF+Ri/xcXFDgn4XBySFGQWD16Icdn0q0hhxDaeL8BjyFh96MOXr2u27iuN7xwiiomjOpt75CphGaN
Y1uY7MQYrZYMxzhi9470mDLE42Z5bOkjJFiRPJ1Gm/OVUywVyIuLotxiCWOht09QX02vm6RHicjuEtBvJwxiGMschsfU0Y6sVCy1SslD8pgiEKGwWTBCLEZC
LJ2BTuHnRUI9Apo8yP22TJOeunrppLUFmGVFYfjyPIJDTTwZFe2qJsQmHJp9UCTryPpasprg0PRDoaGY6ws0mA7CMkHYGgYSOHnZsXcViCXzM0uDhGsLYkhI
Gw+A3YRlIYgLZ2fX4jk/5FBgxgHBEEKwQyx21n8GYXy0c2MAGJyNxeNQUOxEoIQIJFGKeAK+I0FngprF7geoe7v8CpIGCmEQD4RLW/EkfgI4LQhGD7+DfCcA
/YpeMzU1R9SHb5bFxPJDE7vk9XR94jvT7g5f1TQPOcUIQSPMn4Bp0kHPSkziFibgOTg44M6EZRoDaBGARPv344b/ny2NTVr3F7a9zpD/4u5IpwoSBgGGpoA=
"""

source = bz2.decompress(
    base64.b64decode(
        PAYLOAD
    )
)

actual = hashlib.sha256(
    source
).hexdigest()

if actual != EXPECTED_SHA256:
    raise RuntimeError(
        f"Stage27-3A payload SHA mismatch: {actual} != {EXPECTED_SHA256}"
    )

OUT.write_bytes(
    source
)

print("Reconstructed :", OUT)
print("Bytes         :", OUT.stat().st_size)
print("SHA256        :", actual)

source_text = source.decode(
    "utf-8"
)

compiled = compile(
    source_text,
    str(OUT),
    "exec",
)

print("[PASS] Stage27-3A bootstrap script reconstructed and syntax-verified.")
print("Scientific rule: persisted probabilities only; 5/5 target ledger remains closed.")
print("Launching Stage27-3A...")
print("=" * 100)

exec(
    compiled,
    {
        "__name__": "__main__",
        "__file__": str(OUT),
    },
)


Reconstructed : /kaggle/working/stage27_3a_bootstrap_uncertainty.py
Bytes         : 91145
SHA256        : 6d23c28597b4e9b4d8680f4e6da2e29a518f2b82d991922664fc7f86333926a8
[PASS] Stage27-3A bootstrap script reconstructed and syntax-verified.
Scientific rule: persisted probabilities only; 5/5 target ledger remains closed.
Launching Stage27-3A...

STAGE27-3A :: SCIENTIFIC-PARENT GATE
Expected parent : 9771631b18009883da32680b9218d5e40a036942
Local HEAD      : 9771631b18009883da32680b9218d5e40a036942
origin/main     : 9771631b18009883da32680b9218d5e40a036942
Git clean       : True
[PASS] Stage27-2C is the clean scientific parent.

STAGE27-3A :: LOAD FROZEN UNCERTAINTY CONTRACT
Bootstrap replicates : 2000
Root seed            : 42
Method               : STRATIFIED_ROW_BOOTSTRAP
Clustered bootstrap  : False
Target openings      : 5 / 5 CLOSED
Model retraining     : FORBIDDEN
Threshold reselection: FORBIDDEN
[PASS] Frozen post-opening uncertainty contract loaded.

STAGE27-3A :: FROZEN LABEL/M

In [32]:
# STAGE27-3A COMMIT/PUSH/REMOTE VERIFY — NO-UPLOAD BOOTSTRAP
# Paste this ENTIRE file into ONE new Kaggle code cell.
# ZERO bootstrap / ZERO inference / ZERO target reopening.

from pathlib import Path
import base64
import bz2
import hashlib

OUT = Path("/kaggle/working/stage27_3a_commit_push_verify.py")
EXPECTED_SHA256 = "d226ba9c5e7008e95f66ec911ecdcbef08287fc0c0997827af5dc118c7e80904"

PAYLOAD = r"""
QlpoOTFBWSZTWRxMMYcAC+v/4X1/4AB+///7f////7////5AAAQAEABgG1z6KaYzdohUpabuOLIHYccsunL7wAddAFuW3dAB2MXsNVRVFVQU5MqUaNVSROYO
3cUKOy4SmkECZGg01NDU9U/QCaU/SntU8k9T1P1Gpp5Ro2oHpPBQaeoaAAaamTQTIQmVNtUeo21NTepP1T0jTQyBoNABoBoAAD1DQ40NDQ0A0BiBoDIAANNA
A0AyAAABhJpJEEAk8JU/QwqbeqgPSaeU/UgeSA0PU9RoAPUA9QAASSmk9EQyAAAGgAAaGgA0DQAAAAAESRAmkwmjQEyDUQ9BT0nkjTINMjaQaGhpoeoZD1A0
O0AkA9JJ/b9tUPzeqSn87PxWgUYLCCkX7dqLNgNX9ZM9VD8MiZBmzG3/vlZt9lgGMH6ohjrSBj7tkf+/sP5o2opt8NIFzCgs0zhh0WlGjYuSChjURMKp9m5N
Gu7tAsR/wvE3KdxnDa6ThQKMWehhwENS6yIMEIsixSC41WtlJJlhVGmNE3FhXLQK3NFgd8aj0bY11f7GGYhE0zo60vgho5HWde2+4cZr7dXKQLRt0F3bAxEh
12ICQ5RwGs1kmQxvhvdh2ITSBBCBOCR+z73q8VBetDZvs1Aer2/T9Vn1wyS1Aae0rR6e+He6+LfTSvdlM2/HnKooOL8eBDfVknBCB/d450aOHCixFkVYCqw8
Hkc4EIC0gjllQKpWc/lgWUAtXn6quipV7Gl1UprLG6AWL2llGPrhDjWNAex1LmtIfk9OAbZJwJsohSkhNa3W92BJq5Xt9WGl/M68l9jamD80nBA8u52iMkwy
QT0QKc3p9Ew00o47L2+hvRT6Z7jdvH+vlOcRCCk+AzaQ3O3Hh6QhL0CBBf+damYHr119W8N+bKCylo7TM5JZ4ntdF8lRcHciJtTdpAvjWdcodm2qnEwmsp79
0aHnm8sIR2D+ZxnOcFEjGG/qdShPoITWRUo0T+1649UuAYOnioOPwL/vrPDi3NzH7fEnSD9TdhFjbgHAtqD0GMnOLgSHY3tzFbYc0u1UbQKAbPC3UFaC/2LU
5gfMaNrpq46LEHimkBjfLirECYWSWG6Y6YQBLDXotbopLpHJ3GKTys8gOdObyVSx41fIotI3/IatN/Bte7chss1/fj3DEeZ1Lqh1d+/KdA+WqpzUREsYRRGT
yh90pKwmokozfcpfb1QK9teMCNliuceCnmj16MpGp6trhHCrr12FtsqEJEpGuZrTV6ZQXeqrW+/IoZqLXKJyneJCBIkShV4sKdAcE82tBZNG0IEELY5xyJnW
WJDJSKv6It27fvXTZOLrMrde75gdmYaL78+fpCqqNdHFiOo8KCutCFEEEJA8VmqOIBakLtMy0dasgI7e1So5Ym4vcWN+/2q1EpTqBAoQSxJSmGarIFoYPk3U
ZlUVE14xqsIdDJ4qZSxQpMs9bXcB8k4s4shTLmvztTY6d3tG4MXRqVVNyCULRoiOQSgIKV136dBYsIuSRCCAQrg2DP/omudBPk80Tc6C+aSimWeI2DXmMRTQ
AcyK4vGUL6cpnV5WOeac4hvlC5b5EdxaXtNOnz5TB03Ukm66s0QBsy4b96SH3H39fw64bFEWKqG+ko1DQmuw/B7fdIKNDtnDoth7Mn40wrRtB3eOnzqIeNTG
EtD+75/l68/YvlQkNZi+Ao1dH0X4VPQoCg/CMcToORR79nhnBaeQJcARyDq9dbKAZwAolYVlg/kdYa6jOiEz742UbVXQBXYGA4nwFiy8A5C7oiflfT+ZgUSi
owmlnzF9f0eru9t8/hDtZ+q+wPH6y9NlFYwxL2a680qk2mp+6ZDfu4AbZo4PBkq6poNCWqlQ0yROGjfWw7BbRJHfbbNGkornjy9Hxj/vdr28GwjwkA7a9KCY
RxxxiFsp1q5STRdyKqTPbQQkLrFqNerXjjHW6TpO45FoBFQIzaBICQSYaRNPOUiZEJzd3cd3JRMSU4FAoxoohQ486aIMjuShuzc4bGzhrZjlmOZYLDNEhl8h
Wb9KKfK3juDR/JlXLxLVVYDxvDlAo9iyEhH7lLQTAQuY3tyrBSwRa52se+6UDFIgkIyRIsUkweSeN9edByKAlNJcZcc+/xDp5F3TM2eXHMHTvgZTfWtmcXNd
r1Xy7YeGd6D8ueOxsbtKpy53y0UdMLHZsS22d+Zx6+HhtrpOLPVbjxy0Gi7Mxcp0ZMRyW/O0xj9OZuauLeGrT4OWuN4GxveOdrxdK9/bwgVPbriZUvecsmqz
lys7uHS7cqTirCWgKqNVNAeds8Kjlv2EvH3TeLKmr0O6O7bQkRPIEkPxrDpo+acvAeU37KdYd+C/PGJFI/czPpkzqVhtN1LeOMS4Y0FvC+VxHAhaCGJ+xRpF
uxR6IDv9mXh2YNu/3WCQS8GS+zyn7JdQ/Qqnqebw6pPp2evQ6SWg7I083mNi5v9Er0XLDBh9GnK6FmjP5fp+Pm2W5V1XOLvGSDwE9+yOaMfFMja81O5QGPG0
skF8ZGRo0vDwzfcyjhyUNHkl6uF6FYh332FjVXjFhtm9vhOygzERvPOYayKMtEFJczXoQgY369chj3rNz4Axh4dOY09cm3ML8VFt53AxF9xmQ/bn1aio/Iak
ux8xhRWLwUX9LYPhOB9nxG5IedAx0lXEVHKVs4TYNCD95cFJ+XMCQ1R2AHNFkSM1Mg0nvMjQsfGqnWHUb7rlVGcLCbDcXbkT2wzyBo+Ew7htR7VMDiYDSqS1
FZAC5qZItLiZA1ESXDecZr38651gptQjLB/Ly7Zrr2WC1nUjAA7L87cMs8ysLZnLDAzx5ILTaoVdKm3jlz6bfm6asGPHj7ff5vv/Chmn7JQeXtq7pFMOs9yG
mi4TGk1Hd3pHmRvpwM7ZM3w1P8JUe/4fup15KJZHbisx4LJZMtNFkjoUk7vo874HybT+mP0MWIApAZIwGRSfUoqNshFJ7Q+k7+3SqQNwgYeZ+phApDORCzsi
WnBFl3luXkajL2AHrTDDpxMySOz+JrXqeEILgUIIZig2NtLyCB7pCGLZHIgp9wkBLNXAX9Q5Zm0lVDB8jkQ2G4MxLBhh+Is7n6kq6uSf0vS09P47JExA5Q+T
4qcMHVD2l9Uu3JGJNCi3QanTbeBxbZZZg4muH1k6YcrV/V3U54SjKPZnMLWRK7UITP63FfAxM3HNcw4xB+uPSF6A3foSkcU5HKyHEN2wJpoBDK5416nIMzpO
x7yD1rpqGAe0QcHsDh28QpHxpw2/lpOiGPvCg0AgHJO9NUHFMIGBXZIQqJDHuSQaGw8796RJ9koKDmBiTBL8AicGTkhgHROzVPUevUx0Q8DZ4AewOny2nEjE
zJTOBbA13cXcnKUXPUAcAgdd+Pf7Q1TEM3FMm+3Do2J0Gcl/8a6o+HSGxvQHUHEDNY33psQ5aDY6705LbcOIasiFM1NJEQjdLu4h8fjXINQJ3s6U9XUhhNq8
YcBMfDMTv2PBEOT0M0WHCcfIF2r2Oe54IzGqtBPNsQz8ulmIaD2LgeB096W617/WeK34Guevm4Nvz0BqHqQ9ZEJkBp6kNTdm4Bc8F6AS69aGoU9vbRVaF08o
d0AtnpZLjNKDWh2dQSDIpBBt08MZhHRxTPgYvA2SYDKYYMUhmAX/NRRCcUq5kl3jDYljVTm0XfiIJmySMkYwh7eBkcrFjgVVpJZ3zTliPHg6CmPAvYzzG9J+
rVb8Nh6Job9yXDbE2CQ3TNOhfc5mqZOhlhWwDDZZtnCRs2vdOCPa7Nr0J9YYhfJhSGDp7/LG+8oXXM8eqg3N/IrHAOSRbbfHsOos18DkJtTYA4LpQm6AF4A7
Wc8vYLwKfWkmOdClIENn7/v1pOIUpmGJp2Nrc2H69tT6vTCWfX0QKfJnYFBw5zHuvOVpJ73kSqH1U9qOPCNMgTDJgavBhRMkF+LOYnMnJC0WIiKAI5UCkyby
kDnCaTVhwAeYanZMfnO7A8+iTlx1nv297mpjrV0jWyhjz7Uz11lfN0+THvJxnNm1JKIczroH5RoDH1VnrEOdCRh00dd4DiN4yeDhEYK1LWJd4faiB0xEt8tH
YdNZZBk8cAsfESTV5fnz0mjUZGQZDiHCfr+pVVeidhwhxhuTv7GCZDmGtmZX4LVxvKMOBVZEchN7scBwNDdXM/y6Xt3ehYToJb6U1X+LoeuuZjc8J9sZUWl9
/iG4JGQhCBglwC3vnvJj/HzTlGJhQ1N2wMobQU9h4P3f0gbiGEAORVMXiIOp7TkkvIFLUhGg86LECmw/d/CXv98C9YxXVzNnDlCo6J3i2B8wC9de8EqKljzU
9xxJEiREobDzDVoAyTykitGu2A5iHf96afPV01XefgsWgcFNvQRWNtqFM7Yq5ZDGMIAFYSXmFUTFDWwO4oIHHuI4cH/o7C+6rH+B6FjpPZJJCJNbE2kYHwL+
C6UH7fbinc5CRCUYh3fpsFzCNTcGkD7BwUqKZmOw/0wUTL1gebpOkIMSCviTbACwaSA1CrO62PSdogquVCdqcT0ocXnsznkO/vo9PDJbcHhsTAwONsnDaFIY
RpOznRUFiyRkgMtlISkSB+ESZDvtOq1EJXqD2Niyg0e+MNRCQZEwJGeV5SJR/mUfLG8HBICZhE+U0Ok6H0jltOPSlCl+wrYponUdxc9L4B1mQat4ml9DHBPK
HhIWmnfzhfq2n6yjDAQbnXFyJXa2Fc+DbhonydAayDp4YnSZceTyyUG+F7QoJhGa0BCl+EBMpgcHLCy1uUVV4RVDMMhmfbUD6i2gZabMtLiDZA0liRCmBeBZ
KoNeIYRgyFyNB0UOj7X/xOIgZAdycitTjoSo3KdInYbSrm83hVFlPmM6gbM6j00N5E1CBUkJFkXBLDSQdQIiHbBJ4wHxiVEcBK2S1bIMME0RPZ3ZZYHuPJhh
YMp8YRhCJAHJ5cHrnUkgnYYMC2d9nADn5uwE79AoKp6KBtt7kPZ8wXt1Bd5aAHGQgEIIyJ1hRQdJzWJ1l8gBPADA9gHxkM0HoLkOcNTyWteWFdntKKLzCwGM
moUFrbbA9pdoPZvEMU13JmRbpSMgDErMPo9TcA1XmPYdBzt4kwm5vUjO8oPnLVIYBbi1A8QPUiHQdt3geIm1ALEhUCMSF9B2/LD5RCvrExScJMTRm+EJoi0G
/4eAFwemAlDyH4zLc9+u6YNVacaQ1sidpaC7bwOF32slBYRYsqQhfYt9ANgwQsKEJFInSlWOYIevBLCHQg7NupVUY3O5M6dEOIZd9RiTwlIUhpQ9dqIF5BCC
3Xexs3zHpS/d3JtuB1H5lMVO3bAwBIPA4yzPmWAuBDsDvAqkEo4hadSl+e7EbBkBRx7WMFUiM8ijZT0UrBxWSxZ2dMJOQO0oDoE0GxQhxA9Vxohh4uAlCDnO
UNzyQXCLINUQ9o1GB50NZOQdwN4Fw5huHn507ZRwKG8Uzc4Z7kFJJCGnqs3TZ7YukRS4Hk8j7Q41sH1Ggg9avtm6H5SJJ1IMtCxOq4FHRm+YXeo4GgcXRTSQ
ZfqTYmAQDmmpvgHWlHAxPEgZ1SQO2Utj1GNzBSj+h07ja6KdwazjKntJQWlCpCiwsykZhbWhWRAYxGDB03E90q7w6K8KA530lUPKiWriZAYGg+ZJGQo+ELi+
IUgm7+DOT2GhoBtDtrNPmNvDUOybCz7GYv5Rue7LS+DvsHwbTpKWwMoGZlrwYx2hPEMMVJUyEqWNDgYcZe7jXDwjETbNvLIJi1pp1gptE8vJXHUjnJOt3o5C
D1hfEulK/BsROOKkRN+Dw10E3oJiGoFJKJCBRWiwROiy4kb1lctdYGjP3JBBlLLrUxMtChQ6AhsgxiGkxKpZEQglU0FVC7YKtGNBhs0PNmluTbwNANSblOIY
xGHPGcwUEQR8gBDn8bF/FmFXAxGCOYsQgOaUxQwSo3C6BgnZt7GFt8ANtCcdNE2EEagOpQRtCxRVJQCFYRDJcZM6EDCBNIwKW4xXi27v3Zs8dPAE/CkQOoI9
od3gfYKYkBp8j5jnqp05geeoZ+uMkiEIBheASRZEnHgyVjcoDDAd0oe3ycFXMC6BXylQb/nGSSYEA2GbiE2zbkZy0EREzoFkz84Ej1MToEGnUwftQwQ457fN
qEgz2zUDNy0AOYmriWDmaq+TI/I4g28UEuomuzrOCBZPF6klPJeh4+YJs7wK4+hCUqVshGK+kn5P3SZyHYTzKDMuQhxPP6T5yAdRJMTEOsPJBPBI+e86XAJC
QjdyjqBTm8gSHsOyE9CFVBFUVRFVVRVQkkkYyBIRHAO0MzrYfE+/2YmmdeSeQne8oTm5ILAYvoOWAEySJuBVESJvxsaaJ2WtZybPFzUwUNoh8RSllEy1DNHw
Jmx54PU8iqPvRN48x1H1h9+BAgDz4nCdd7HBD7yd88uymHAQXyvh6+exnjTMtR1w1qvkB5SFnUMQSck5hhjBIVq+KaI2CmpHh6wN18JPaV1hQbzdMTEdoIV1
puE6BDRBGwHwRLA94YQwDcHbu1fAQe4wI5p/CIM0SDuHUApIJSL8AbHgQBYdvdSbdq8KKCligikOr3erp89+rx35ZwfN3H3KXBnNeVOU3x5cS812DNzXg4dJ
dIZASUwaMWA6RrTB1cT2uOh3cPLGkyEaUpBpRgpkXeAHV8dIJmc0ExDexCoFNYMh01swuREINt9CamMM7uLCkDe4N2xyKTj1HgHR18chBsFQYBh5Gygp2bAI
bBg3wH5mddsbandR6R2O6XMhoMQwj1EcwMMm71vRm5IvBXP5ADQTVMLwjJEwKqqXOLfYSG3JPs+npboy87O4NCNZ5+eTJfL6D9qzb7NZQ8FE35gWMY5Eu9oW
QDIKpEIOItvNXEfFFWLcmMMYOCXHCqpRCUmkpZt4cCAcyCxQiEhHnZFDpoQLGQiUaMg6hSb6yOhvHfgfw7kOhBYTNBU5oBDkMDC9OyGOtZZYFYZOWOuWwmmK
l8MbZuEsYuZCJRoJlYLBHOguQgmrjoc/3EWwyAaYwzTJXpXpE6SMGmmJVDTDJcnCUDNkkpEz2IViiRGWkspZJWTp6ZnVwQhqJtqwmEkkQMsn0TGEDMNCAwYr
t29nj8d6yeTkGyEYgYcZW8MBLFwuiYMMTgiwDjBB4ziYaRka6irWhnawlRQruJYbJw29/fwt5YVe/nkObgcRNqcEd6kUS4dB1BkBho6OosOXOIIehpPuWoXM
uTDtUTPBbibcBuDknwF+T1CfiPJ+j7XFDu4a0/rHjrip5wcFHmd7MWIbT3Ve6B84g9xhmB+M6qOS5JAdPAf1/PwHGJC72id6WT0nWngMuQQF+/g25RhMKVjV
NUzO1m+e4Whd812QRbe5ZPmYIxQRWEDsh8Qw6/LNIGlw8CFjq6UVRQLNGsIUzXvCUnFgMQIhnZkGQs/PgDwZO5dqHIKUHIMtBzGa3p1bssXhl0tE96d/Sk/j
UwIaWvveAeAaR/4u5IpwoSA4mGMO
"""

source = bz2.decompress(
    base64.b64decode(
        PAYLOAD
    )
)

actual = hashlib.sha256(
    source
).hexdigest()

if actual != EXPECTED_SHA256:
    raise RuntimeError(
        f"Stage27-3A closure payload SHA mismatch: {actual} != {EXPECTED_SHA256}"
    )

OUT.write_bytes(
    source
)

print("Reconstructed :", OUT)
print("Bytes         :", OUT.stat().st_size)
print("SHA256        :", actual)

source_text = source.decode(
    "utf-8"
)

compiled = compile(
    source_text,
    str(OUT),
    "exec",
)

print("[PASS] Stage27-3A closure script reconstructed and syntax-verified.")
print("Scientific actions: ZERO bootstrap / ZERO inference / ZERO target reopening.")
print("Launching Stage27-3A closure...")
print("=" * 100)

exec(
    compiled,
    {
        "__name__": "__main__",
        "__file__": str(OUT),
    },
)


Reconstructed : /kaggle/working/stage27_3a_commit_push_verify.py
Bytes         : 24344
SHA256        : d226ba9c5e7008e95f66ec911ecdcbef08287fc0c0997827af5dc118c7e80904
[PASS] Stage27-3A closure script reconstructed and syntax-verified.
Scientific actions: ZERO bootstrap / ZERO inference / ZERO target reopening.
Launching Stage27-3A closure...

STAGE27-3A :: PRE-COMMIT SCIENTIFIC-PARENT GATE
Expected parent : 9771631b18009883da32680b9218d5e40a036942
Local HEAD      : 9771631b18009883da32680b9218d5e40a036942
origin/main     : 9771631b18009883da32680b9218d5e40a036942
[PASS] Scientific parent unchanged.

STAGE27-3A :: EXACT 14-FILE STAGED UNIVERSE
Expected staged artifacts: 14
Actual staged artifacts  : 14
[PASS] Exactly 14 Stage27-3A artifacts staged; no unrelated changes.

STAGE27-3A :: LOCAL FREEZE / HASH GATE
Required artifacts       : 14
Non-self artifact hashes : 13
[PASS] All Stage27-3A frozen artifact hashes verified.

STAGE27-3A :: BOOTSTRAP SCIENTIFIC BOUNDARY GATE
Bootstrap task

In [2]:
# ======================================================================================
# STAGE27 — FRESH KAGGLE BOOTSTRAP AFTER STAGE27-3A
#
# Remote scientific parent expected:
#   17b734f778c4881d200c90967a19baf367347116
#
# Purpose:
#   - clone/fetch clean repository
#   - hard-pin to remotely frozen Stage27-3A commit
#   - verify exact Stage27-3A artifact universe + hashes
#   - verify closed target-opening ledger
#   - audit fresh Kaggle environment / attached inputs
#
# ZERO scientific computation.
# ======================================================================================

from pathlib import Path
import hashlib
import json
import os
import platform
import subprocess
import sys

EXPECTED_HEAD = "17b734f778c4881d200c90967a19baf367347116"

REPO_URL = "https://github.com/themubasshir/ids2018-validation-safe-ablation.git"
REPO = Path("/kaggle/working/ids2018-validation-safe-ablation")

STAGE3A_REL = Path(
    "results/stage27_loao_unseen_attack/"
    "stage27_3a_bootstrap_uncertainty"
)

FREEZE_NAME = "stage27_3a_uncertainty_freeze_record.json"


def banner(title):
    print()
    print("=" * 120)
    print(title)
    print("=" * 120)


def run(cmd, cwd=None):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    if p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}): {' '.join(map(str, cmd))}\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return p.stdout.strip()


def sha256_file(path, chunk_size=16 * 1024 * 1024):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            h.update(block)

    return h.hexdigest()


# ======================================================================================
# 1. FRESH RUNTIME
# ======================================================================================

banner("FRESH KAGGLE RUNTIME")

print("Python executable :", sys.executable)
print("Python version    :", sys.version.split()[0])
print("Platform          :", platform.platform())
print("Working directory :", Path.cwd())

os.environ["CUDA_VISIBLE_DEVICES"] = ""

print("CUDA_VISIBLE_DEVICES:", repr(os.environ["CUDA_VISIBLE_DEVICES"]))


# ======================================================================================
# 2. CLONE / RESET TO REMOTE MAIN
# ======================================================================================

banner("REPOSITORY RECOVERY")

if not REPO.exists():
    print("Repository not present — cloning fresh.")
    run([
        "git",
        "clone",
        REPO_URL,
        str(REPO),
    ])

else:
    print("Repository already exists — refreshing from origin.")

    if not (REPO / ".git").is_dir():
        raise RuntimeError(
            f"{REPO} exists but is not a Git repository."
        )

    run([
        "git",
        "fetch",
        "--prune",
        "origin",
    ], cwd=REPO)

    run([
        "git",
        "reset",
        "--hard",
        "origin/main",
    ], cwd=REPO)

    run([
        "git",
        "clean",
        "-fd",
    ], cwd=REPO)


# ======================================================================================
# 3. HEAD GATE
# ======================================================================================

banner("REMOTE SCIENTIFIC-PARENT GATE")

run([
    "git",
    "fetch",
    "origin",
    "main",
], cwd=REPO)

head = run([
    "git",
    "rev-parse",
    "HEAD",
], cwd=REPO)

origin = run([
    "git",
    "rev-parse",
    "origin/main",
], cwd=REPO)

subject = run([
    "git",
    "show",
    "-s",
    "--format=%s",
    "HEAD",
], cwd=REPO)

parent = run([
    "git",
    "rev-parse",
    "HEAD^",
], cwd=REPO)

status = run([
    "git",
    "status",
    "--porcelain=v1",
    "--untracked-files=all",
], cwd=REPO)

print("Expected HEAD :", EXPECTED_HEAD)
print("Local HEAD    :", head)
print("origin/main   :", origin)
print("Parent        :", parent)
print("Subject       :", subject)
print("Git clean     :", not bool(status.strip()))

if head != EXPECTED_HEAD:
    raise RuntimeError(
        f"Unexpected local HEAD:\n{head}\nexpected:\n{EXPECTED_HEAD}"
    )

if origin != EXPECTED_HEAD:
    raise RuntimeError(
        f"Unexpected origin/main:\n{origin}\nexpected:\n{EXPECTED_HEAD}"
    )

if subject != "stage27-3a: freeze bootstrap uncertainty":
    raise RuntimeError(
        f"Unexpected Stage27-3A commit subject: {subject}"
    )

if status.strip():
    raise RuntimeError(
        "Repository is not clean after fresh bootstrap:\n" + status
    )

print("[PASS] Exact remotely frozen Stage27-3A commit recovered.")


# ======================================================================================
# 4. STAGE27-3A FREEZE
# ======================================================================================

banner("STAGE27-3A ARTIFACT / HASH GATE")

stage3a = REPO / STAGE3A_REL
freeze_path = stage3a / FREEZE_NAME

if not freeze_path.is_file():
    raise RuntimeError(
        f"Missing Stage27-3A freeze record:\n{freeze_path}"
    )

freeze = json.loads(
    freeze_path.read_text(encoding="utf-8")
)

if freeze["scientific_parent_commit"] != (
    "9771631b18009883da32680b9218d5e40a036942"
):
    raise RuntimeError(
        "Stage27-3A freeze scientific parent mismatch."
    )

if freeze["required_artifact_count"] != 14:
    raise RuntimeError(
        "Stage27-3A required artifact count is not 14."
    )

frozen_hashes = freeze[
    "artifact_hashes_before_freeze_record"
]

if len(frozen_hashes) != 13:
    raise RuntimeError(
        "Stage27-3A freeze must contain exactly 13 non-self hashes."
    )

expected_files = sorted(
    list(frozen_hashes.keys())
    + [FREEZE_NAME]
)

actual_files = sorted(
    str(path.relative_to(stage3a))
    for path in stage3a.rglob("*")
    if path.is_file()
)

print("Expected artifacts :", len(expected_files))
print("Actual artifacts   :", len(actual_files))

if actual_files != expected_files:
    raise RuntimeError(
        "Stage27-3A artifact universe mismatch.\n"
        f"EXPECTED:\n{expected_files}\n\n"
        f"ACTUAL:\n{actual_files}"
    )

for rel, receipt in sorted(frozen_hashes.items()):
    path = stage3a / rel

    actual_sha = sha256_file(path)
    actual_bytes = path.stat().st_size

    if actual_sha != receipt["sha256"]:
        raise RuntimeError(
            f"SHA256 mismatch:\n{rel}\n"
            f"expected={receipt['sha256']}\n"
            f"actual={actual_sha}"
        )

    if actual_bytes != receipt["bytes"]:
        raise RuntimeError(
            f"Byte-size mismatch:\n{rel}\n"
            f"expected={receipt['bytes']}\n"
            f"actual={actual_bytes}"
        )

print("[PASS] 14/14 Stage27-3A artifacts recovered.")
print("[PASS] 13/13 non-self artifact hashes exact.")


# ======================================================================================
# 5. SCIENTIFIC-BOUNDARY READBACK
# ======================================================================================

banner("STAGE27 SCIENTIFIC-BOUNDARY READBACK")

contract = freeze["bootstrap_contract"]
actions = freeze["scientific_actions_completed"]
ledger = freeze["target_opening_status"]

checks = {
    "bootstrap_tasks": 10,
    "bootstrap_replicates_per_task": 2000,
    "total_task_replicates": 20000,
    "raw_target_predictor_reads": 0,
    "model_fits": 0,
    "model_refits": 0,
    "model_inference": 0,
    "target_reopenings": 0,
    "threshold_reselections": 0,
    "gpu_hours": 0,
}

for key, expected in checks.items():
    if actions[key] != expected:
        raise RuntimeError(
            f"Frozen counter mismatch for {key}: "
            f"{actions[key]} != {expected}"
        )

if ledger["total_consumed"] != 5:
    raise RuntimeError("Target-opening ledger is not 5/5.")

if ledger["remaining"] != 0:
    raise RuntimeError("Target-opening ledger is not closed.")

if ledger["target_reopening_authorized"] is not False:
    raise RuntimeError(
        "Freeze unexpectedly authorizes target reopening."
    )

print("Bootstrap tasks       :", actions["bootstrap_tasks"], "/ 10")
print("Replicates/task       :", actions["bootstrap_replicates_per_task"])
print("Total task-replicates :", actions["total_task_replicates"])
print("Root seed             :", contract["root_seed"])
print()
print("Target openings       :", ledger["total_consumed"], "/ 5 CLOSED")
print("Target reopenings     :", actions["target_reopenings"])
print("Model fits            :", actions["model_fits"])
print("Model refits          :", actions["model_refits"])
print("Model inference       :", actions["model_inference"])
print("Threshold reselection :", actions["threshold_reselections"])
print("GPU hours             :", actions["gpu_hours"])

print()
print("[PASS] Stage27 scientific boundary recovered exactly.")


# ======================================================================================
# 6. GIT IDENTITY
# ======================================================================================

banner("GIT IDENTITY")

run([
    "git",
    "config",
    "--local",
    "user.name",
    "J.M. Mubasshir Rahman",
], cwd=REPO)

run([
    "git",
    "config",
    "--local",
    "user.email",
    "themubasshir@users.noreply.github.com",
], cwd=REPO)

print(
    "user.name :",
    run(["git", "config", "--local", "user.name"], cwd=REPO),
)

print(
    "user.email:",
    run(["git", "config", "--local", "user.email"], cwd=REPO),
)


# ======================================================================================
# 7. KAGGLE INPUT AUDIT
# ======================================================================================

banner("KAGGLE INPUT AUDIT")

input_root = Path("/kaggle/input")

if not input_root.exists():
    print("[WARN] /kaggle/input does not exist.")
else:
    input_dirs = sorted(
        path
        for path in input_root.iterdir()
        if path.is_dir()
    )

    print("Attached input datasets:", len(input_dirs))

    for path in input_dirs:
        file_count = sum(
            1
            for p in path.rglob("*")
            if p.is_file()
        )

        print(
            f"  {path.name:60s} files={file_count}"
        )


# ======================================================================================
# 8. FINAL
# ======================================================================================

banner("FRESH STAGE27 BOOTSTRAP COMPLETE")

print("Repository :", REPO)
print("HEAD       :", head)
print("Git clean  : True")

print()
print("Stage27-3A bootstrap uncertainty : REMOTELY FROZEN")
print("Target-opening ledger            : 5 / 5 PERMANENTLY CLOSED")
print("Raw target model inference       : FORBIDDEN")
print("Additional model fitting         : FORBIDDEN")
print("Threshold reselection            : FORBIDDEN")

print()
print("NEXT AUTHORIZED STAGE:")
print("  Stage27-3B — preregistered secondary behavioral similarity analysis.")


FRESH KAGGLE RUNTIME
Python executable : /usr/bin/python3
Python version    : 3.12.13
Platform          : Linux-6.12.90+-x86_64-with-glibc2.35
Working directory : /kaggle/working
CUDA_VISIBLE_DEVICES: ''

REPOSITORY RECOVERY
Repository already exists — refreshing from origin.

REMOTE SCIENTIFIC-PARENT GATE
Expected HEAD : 17b734f778c4881d200c90967a19baf367347116
Local HEAD    : 17b734f778c4881d200c90967a19baf367347116
origin/main   : 17b734f778c4881d200c90967a19baf367347116
Parent        : 9771631b18009883da32680b9218d5e40a036942
Subject       : stage27-3a: freeze bootstrap uncertainty
Git clean     : True
[PASS] Exact remotely frozen Stage27-3A commit recovered.

STAGE27-3A ARTIFACT / HASH GATE
Expected artifacts : 14
Actual artifacts   : 14
[PASS] 14/14 Stage27-3A artifacts recovered.
[PASS] 13/13 non-self artifact hashes exact.

STAGE27 SCIENTIFIC-BOUNDARY READBACK
Bootstrap tasks       : 10 / 10
Replicates/task       : 2000
Total task-replicates : 20000
Root seed             : 42


In [3]:
# STAGE27-3B0 SIMILARITY IMPLEMENTATION LOCK — NO-UPLOAD BOOTSTRAP
# Paste this ENTIRE file into ONE fresh Kaggle code cell.
# ZERO predictor-value reads. ZERO similarity computation.

from pathlib import Path
import base64
import bz2
import hashlib

OUT = Path("/kaggle/working/stage27_3b0_similarity_implementation_lock.py")
EXPECTED_SHA256 = "cc00f33a494f5ad1b0459af6b4924131f12380aaa15f7873e7ff9e0f8f0c3adf"

PAYLOAD = r"""
QlpoOTFBWSZTWXOR/poAB5Nf4X9Ufv///j/v//+////+QAAEABAAYBn+8Gn1tPVRDOSKZdHceuRq89NtHTSls1Xs1e9689U7aAWptRegV13c4N07jmHdtgdO
Tr3se1Nloe1nPDEhDSZDTTIJqNo9RhTRhNCNkCaaZGRkPKGgxBoJIgBASZKfpqp+U8intSep5TygDTQ9T1ND1AAAaANAcaGTTTJoAGCANBk0AAyaAAAGTIBo
JNREjQmJMSn6k8ap+knqeobUPUHpA0yA0ZPUDQ9QAACJRNAqe0p5BMKfqZING1HqA0BoAeoGgA0DQACREIAQaFME0aVP00NU8TUyaZGag9TQ9QaaaD1GgAZq
IJvJp+dE7Ake/BeqHpy++FNiSLLQNH/b736qmfkqB/PSlIezycT4ngi91UMEUf9FS5yrhrcW1fx1tdWavPYbsav1XCUyhKpSmQUlVTS0hKGkeFuzXHHQrcZA
4pUQm5FhBQioMIsU06CjfjMstDb7EoMHDdWKbZKLxLtu96MVhSFbaLEw0a3sM0ISMJkRMmQLl6OTY/31VByxdfqv5dRYV30yGNkElChcqxAEuilYaurJPk4Y
8WDRGIIxFWaV8Bqad0AIAYJCBt+DWrhCGRiSLxiR+xgBLNP8GB5rE6sDh4/Z5rPlG2W0Vx/v+/fq/wnWSfhPEQUoGSkOz1GLZ9lqMAoggEkmKEtj4X68FnTM
LbyCscdoybCmHR4tLkvYRIlwAXE/Kcog1puxsrWnjJ7HP8COC6+enotvL6Mz6T05zY/QA08+UVUatlF+n5ScLP+LoDfE/DWz4yNJrY2S8Tc0kFP2h7LbdNy3
tWnq9ssISBhgFg9X/axrvTESS39bfDsCXhvQzzxqpGobMfShPh3QRvNeTK8Eu8EWn3QlGvSifrhOvJykXjK99QYn4dnsPlpi0Hv7l0MbQ9L9ERIMgeMG66bd
lWTBhjJSGej7szb2ldTo408mYg1PJgh9TXw/2cfW+NsvPnntPV9IUzUb8rts/814BTVa54TlhxOE16rHpxcUYOy2SvNlbwWkWVve3ahiB0LKuqZ9YQk71Glp
5tIaNk9uTaWhrrX+5e+7Xboo7Rci3ljlbQms3NGAlHpbglSUyBUaFcNAJMBEdHGMApjH/Dl3NMuXKwpk0cE1KUovLSPXyvThocwwuloeQjOaSD677JBdkUa7
ngnGL/gbo7yXPBg/WgFysb2FjhCN3E022/5URfAehIm09TvHqs53gypQw+FHlvY+5d7axCKvqIhCGjvOXSOqGSIZ8BHlhHB+vTbPpdTfhGCAmhQ8XpeHZru1
43bOAbWRWQv6HxLgbw3A4ucgRj+D474Qhf1fIDaBu/+Ugt305GsGLzlaT9LJJXSYCSBnIZWAoSc7qNOhFI6PeRfhJzmH9La0LpmHV4eEx0O/G7zUO3fVdRy6
q9cobu0T42kifTnwlFhojU0aNMF3UaZgqVZUmcYuFOBzEIhOtCkgzXHo1GPw7eqfbwvj/dz306dh3R3ORnddSkfGs7KLVI27oVXJD479uyceM0mGHqcNtSzJ
nFtrgu4OKxZgWNYxAKLlN1eLSzCyK4pLnys0V25DGml1o6VZd1isQoLwua2XkbaoxebmEKKgUXUsqxSmoliUgP61Mzf+gQfuKzWwXr+fCO0dRxysmQUiRnHz
MgWkUQPU0qIxUmXDMZKh2Vc7PDmzHeXV25cbdZpn75eLqqqpvSSrS73aUOyYyUJIMwYb1MBSHwU0om0Po4IRnSxDOhxsK+/ngneaLPh9bDyBzPqZke6tDmtQ
kXKjBS2tG18Xh6TYK7ZnvEHMu1Ni0nJK66Or1deu06uGL3jOWq1y9Pg3Pb6K55vNFKYt7Y+5WjUvXtKHuI0gc8o5sTp9efPt8kjSOhvnxPhb/Lc3a1SfSeXl
h1bKD0IfFsOtDwp0n3UtMY3FlPX5O00TGnWU/MaK43aWfDReKc2XMiUUDQmEc4xjLZRVrZSCUjWTGIYWtfb3XkFZxHR63lU0xplb40mEhHJhd09PLAoVq20J
EsOv6Q4E9QBHIaRj0zb8wvlRQeKyGLsqu4XvCPRtjIjCosyc2bwsWKcFxOGWUL10xGy9yzXmU7qp71VrrN5uMVDuBxHpjADK9CiUuUKGDaie2EQB4Nf4yOyj
GFYzI8CN2IUO/3CoJ8tA5jwHTY7GPeo5Sxd983F8ZjsCLdKmdfbnD+0J6NYmMZBqGMee8PvoICRAtFJEiyMQEg9nMPMYPcGMA5zQEpJp70fYegTwHnZgIPg7
0YHmmh5C1gaBD7ZuPEYRUIAAzwCGArMJhIMAsMBw8yObtpk+5272sUcz4GM6mU7Plc2edhXOMeTopAJ6M663AEeZdKHXcxKlQwJxAqVKABD8FPGpuqvrcF12
uWlpaGIUaWj38sI+uY9Zj3SBAIiRgnmyUog753XXpLgvD1656/59KQSGSBDwObyvExd4qdfIkPySHkt5/cHBJwDvhnXvQG+dEG32YAskwphRf1+A/b7m5gFh
yoZGJCkr4Wef7j9GwEywuMwUF361YWFrAewzI/hdSzca+dqhhMGONVyUdHSB5fHGtlCeyLoYGQZNyerTVEctutDFDL+oNQMGaGuNEZU/tazcq8ReqJMKr0K9
5QmpLqdaKpRN1rqc+6MWZZtdNWzi1a1sE50CjulBmirHE62P8j2XEhE5j4JoSjzTKXUrmUeRxVsnDReLOpiyxXUtKoRZ1UJoiAh5lfC/NFSi54/e2b4c2TLu
2zgDJHaNjExhxkMxeSS4ZjnDaBSc8pLXQjTegi5AgareJ3cS6ldxWmnbkdRjp30FOm6WHhyyG9jl16LTGqIsuc3OPgCNQ15vih8f7v6e6yuu3avUfdNGPBwy
sB5d3zVQwdR806pJchOmI72xlsrsIlKhQ6AlYOAlW4cRUjfEImBQGd41jQdzLZmWmO6YaA6ZW132XmJdpcA6pagt1xK3lawsW8rxXdhrNBw4bnVGaRii557j
u1tA6I6BAKKJ5F5Qv7LjzNUZM6CGcAZNu5vfZRsC76UGU7MqIoz8+va4X5LO5jfe/zY5l92nZSSc42o7okrZhsjjRLStUoVnKGutQVRb9A4hcVREJ7tREZDU
q5w3GXUyHJ1uIjEm9YFSaaVEibCdEmUaUSmhWdWoKyrFFFHMRIySEEzFOSPDYFjwODS4ok6nOxRHjKO8GK3RPYJnkjwUhSWFUWsoOsWDrkKBPgjrOEEZ12Rs
cDmzfxhDvItVBnoRFUTiuQMbsgDpBISchLjzHc940G0GTO4AyjJA+WpQKNjRAbwsdLmJevvupS/9dEdnxPMsryTWIk9XYCgZsOCNTicBYk+CUEo0cUAx4KqW
3BlYJlb4i6iK8Es+8Q3SSE9JkE1SmjorBEEpUwZb/iFxnBpTeMmGQ29hbAi3npzLtuU2YKYUxFrmfelsTmaSqsSH7qHFuJ4s67Sj6zkTdMxpLZ5q+g2mjbMw
uFFyP0yY0VWzGZsfTbkXW+UdAYO6Y60x5zU9JrKYqiPINfDXh778a8YwYYYY5bdgaIS1YgZMpo5cuUzWuWaWta1qXv24NLElCDIMulM0GxpZFkQUWQ4pMGrG
RK5pTihZJVdbmZQOqaoh1MsrWlXTijl6xPTPu8Uah78raaVMJiEEd/WtN7bdOdZ6OhZLc8NMNquJN9ANbsC1kFk0EWduLaNeSghy+DJmH7WOEmq5ujKkyDE6
y6qsOSL7vovU+p8R6FKrME6YTKzB75AyDMGXI5nMmXtKZebMDCeKPTt+R85+6YeSZPZMw/Bmc/BQHnnuvu2H1OVB7TJ0qiorBfGiRAIAzx4OP2EGL7mZULUb
mkDcA2hXR4VqQvd+Rmv8UIXF1yPZUlqTNYuzelYeDV2T+hSxKJlBU9Nt/VaRXq+inWNWYZqlMVk+1GzS4UoQjAmUkHFqloohGV1Z4opnJjhZYX4DKMcQoiDe
sys5teqIdMqZ9TLaZa76WM2hExpkTjEppkfftLn628V5k/t/f4Q8zOq/OftfY/qstR/kK/gvzRWrCpeFJMClrnBiEIiJKkT5GRYRiOEnvge5e5i/wUn8iPlI
S32K6Ei807mBMM14j3fNtfK5W1ZxPwuXtEsVqq0a7BOXQY/S79GlRN5FXoHVvenHU4BwxDO6dY4Cl5vadR5EhFgzL5YfsHxutfZsqwieG0cLPJGBrZFjOA7k
jXIgyjlttxvmcFHB9b8THNpjcRhBQGftxoHkNH2784x7+JmWX3n6huCpMnd1SW6Tr9cyNnY0qZsMkgPk8FlPBHkIaa2FIwRkPgP1vOR1m1EcF+aT+Jcvx14/
iSLgJRsBqvumNAWecPAgJR4dEqVXU4Q0w9qKeZ+JpGSMvgmxoOySVBULd/SDWdtiQ/H9jr+SMUpwDXLSr4MOSi5cSEMVLGpKvvQ0MQpquC3/8lu5y6DBwJkL
5YMtMkgqhGBQ5s5dKoyJy735FuknuF7zK0v/I7I/RXiM2fzpEryH3ff4GSLG5eXGFQ8f1xkSdkblDmwN/I9ZiSD0GY4MmyMdZ/SKzKrgdowBIKqqPRU5GU9A
ZOHkGL0kz0G49ekJhfM9HwslE1ZWYTu6GU7hU6+jmuzD5s6Ka7ZKxGpTKZx3w4FmM1ixJiSHEo6FaqmDICMagyuqwssR8Fb7RPK0nlQwCWdmQwl6UkJ8JcCl
kn1H1zVhQLosljCCDjqBcNyS+BjdNsT305Eo8kqE5gXz94krjUczJL1Sk0/TFQw1DR2aMkXnYsvqMMCrTE7jpmuDbG7DE1ztGua6MO9+y5DEAR53jydwf53N
yBblOtp2pSSdic/OdipAMjE4Bd3HdneDo9UAuwXyHtpIjGs0psRU4+pk8zMEpzhgtG1QRiSd8Rao4As4pGRKhVUKJZKKSb4WVBhapOwIkCBI2M9eiWK2Nah1
ylxB0ed8KWAMVcu/jL2B6ZEmfGIiBdydtaa0lqWluMjEL6ooXQqFyCEjFL1dx3yvoVbAbuJ6tVcwQocIZKlHCq760FK1zEflCiFscg4psa6WJnmgajAuQeXp
82gsQX0eVybTY2c2RwRi0QBzRFD0+sMt1oUXp4XJdGl1GoyCKSNvVx6knsP1Ns9nrQb8ibdjEnroae15OCBwQN6KYUh4M13HZze9BCWG0yAmO0omJCgEIJht
zb1wtwYjNouDRHWnY89ii6uwdoD7vfM0NQH5nskOZxzO7DBYKMSCzv6zy15VXvK4O2V84/kRgh7gUg6cv1Hay3aVtlugricQYik2EfFOmGtVWISKJpzTbyRF
UU+zqvMnbgM8GHDnAK22rfo1iBFghKitIleENhotIhlYLmTaEPGGtyWVqF9k6XxL3NQCpxqdTeFIRxyZrjN45oV6eBtB0NjyPcv1OE0ZUNRoZJJPyE7b+8sD
PQO8LdoBwYk0HZ0LhUDvCEQRXhWgSg5dEz1x1csYD6UguRiMDtrI0lAtdcvNJKPRoPBg5AaMECt8WDYC0Oq2cNhCMiC7xIERCaNyEvkkVJV2YPxN+isUQY92
kHu4RQcKlDoq1KOrIaRyNUGiMuiFNtxio4xSQkTk4b0OaKmh27U2l8HnyzBGiIfG5p6SyFoORXsZubsgOpykTZD+UmTWeRZo8DgBYipZaWLJcnKus2QDjE56
7xzCZx42LQ7RbeLmwNMes4GoeRzYusMaFViiMpuwK3bR+bXcnCGZ7cknT4yxGbwcdjWqIA3KEO4spyWF55CJKKF4gzINcPtbea5fNdm1cYsWQHqqdynu+Wks
Tfu6qde2OXY6SkdixAG4ED8zffBsxaMAsHwlGJ5IzHo7l4h5Qtt+Fa0O5FOCy6kzDIRWuY5MR3QNpoAoBQwhs9EuZRBUDtoURhAe7CCigOgpAcZF0UO9qyll
0VzqjCVZajdjZ8YyJLpri4B0yZ6LsDIsyG2GmCVJKKMIiUhUlUVMsyKzGJjZagiWsZVB3makMaWFGvYdOhg1YIKApuNk2GJjCTDDIGg8S3Pc39r9TrzMcBAy
IjnA2ahoQjGI2xWhDbRUqAxaoxBaQbuV8CvNJl7MZKR3SAUkQghulAXJDBUMMThvuR0D4eDc58BNwvX6PcKtk3BSsFw27odESidrDFPwnv9SwBdb/Z72WCQM
MA6P4UbEdkd1mJgMExDKxDYZtIKHZfW07HoU+KLLAPnRp7NQzBrvoFvm3Oaq4GMaNQQyEIdsmFy8mY1IHf8PYoYvkg7IvVuQ1es8kjqJohXnlIXAqPxQGete
HeGeZmgNTxbWrpmUGatGAg9nHnm0E4/MMfWVxoXMOz5HwPBEd3vFCrfAWBjmsGdxGPH2rrPfAN0K67A6IXAmF0bspyN0yC4uRRrHkGZqYK1tkvIG22A0MB84
Vg9YVuzhdMoISR8DkGeHrvKYF5DGDabBtLjUogkQMwE1Lte2jEIwZDniQ4zlEWIg+fJo5DoNkAuByBlFUVIs49AddNnhyPIqKmXDqjRfZ7BGxcc1oeCmi6b+
Otetzgq9kHQSgwuhoG9p3SJJ81p3XorduLNFiUZWAI8SuaKcCGKp4qXqthdEwhDBME3R1g6AimqRU6jV2C+0ERiTkiW4BMghBH1JiKO6TORYMSHZzWJZXkso
msCu7eq4IJ0mIZ4bko0ZmgPXhFurvrAUYwh79F8vd3Xia7GC5qOONNqDvdC+hMHlAUwj3M4W4yYvjLdM8JXJlE7n5ZWZhbFbVzZVMeHBWiWM25cI0yUBSvKh
v5i6eBZJEXAjpxQ0hlbnvrAYNUAh4ErAj6Frn4s+q6GoqODWBc14M3TQ7DPk2dKWyTNIs4Q4FdQkpFxbooDT9KEeN0ZMVLIZJjUL1YYFYutI1BlRheZFmXnt
2gp5tsJpomOVfCCkSKs4mIEWcaMPCaZpTqYPA9nxYR9jHeGEFy5OzKo1zoZHB+X9e87Iyx5odE0GUPAj3Si9GiRaM/jM3uii6aXOlVWtzqQYNILWgzphVtCU
y4JWLxiZZtbdJdbndiVjRu9DLtGrM6DuF2amAwzLcLcB1FH2gjCGdhwuAczqgVJuiKQkjupVWMxX5IuEy9sPMcA1smAzUKFKoyeLUHSMssHit2DGdRhcXWrr
l3w7dVuqKzEivJRRFKalBEjtqMy5IgDkmCL++iOd5gNKDCE3k+RWBBUZcTEpkIjJaY92/CfKxFKeu/MbG0JjFiuWuyRxRIoDEzhBRDOWephlRwWl+l0xEknA
OJcxaNCok0Izw5sV8UJF7bhsOX4zsmsN0DTYdM0WL/jmFUvPOIakwNJAjoceBzbUc8spH3AR0Uci8kBxIMiQfMgtSInGdW8bC1IGpM6mjgnLf48304knomBx
9vRaaM18ksaxiGmNRSiYDT2+d9Dn0n5T2kbFki4vojhvC+rUbGMDVDhpI92vr5wHhEetNMU211Co18vOxymhC0WN1EQRh3bhO+B2lxggMOqTmWc5J5n2nxue
vBc9dZsMxyTV4dzeyNZrgPdY2LQSPdlj8x8xiH/i7kinChIOcj/TQA==
"""

source = bz2.decompress(base64.b64decode(PAYLOAD))
actual = hashlib.sha256(source).hexdigest()

if actual != EXPECTED_SHA256:
    raise RuntimeError(
        f"Stage27-3B0 payload SHA mismatch: {actual} != {EXPECTED_SHA256}"
    )

OUT.write_bytes(source)

print("Reconstructed :", OUT)
print("Bytes         :", OUT.stat().st_size)
print("SHA256        :", actual)

source_text = source.decode("utf-8")
compiled = compile(source_text, str(OUT), "exec")

print("[PASS] Stage27-3B0 lock script reconstructed and syntax-verified.")
print("Scientific actions: ZERO descriptor values / ZERO inference / ZERO reopening.")
print("Launching Stage27-3B0...")
print("=" * 100)

exec(
    compiled,
    {
        "__name__": "__main__",
        "__file__": str(OUT),
    },
)


Reconstructed : /kaggle/working/stage27_3b0_similarity_implementation_lock.py
Bytes         : 17867
SHA256        : cc00f33a494f5ad1b0459af6b4924131f12380aaa15f7873e7ff9e0f8f0c3adf
[PASS] Stage27-3B0 lock script reconstructed and syntax-verified.
Scientific actions: ZERO descriptor values / ZERO inference / ZERO reopening.
Launching Stage27-3B0...

STAGE27-3B0 :: SCIENTIFIC-PARENT GATE
Expected parent : 17b734f778c4881d200c90967a19baf367347116
Local HEAD      : 17b734f778c4881d200c90967a19baf367347116
origin/main     : 17b734f778c4881d200c90967a19baf367347116
Git clean       : True
[PASS] Exact Stage27-3A parent recovered.

STAGE27-3B0 :: LOAD FROZEN SIMILARITY CONTRACT
Descriptors: 11
  Flow Duration          <- Flow Duration
  Tot Fwd Pkts           <- Total Fwd Packets
  Tot Bwd Pkts           <- Total Backward Packets
  TotLen Fwd Pkts        <- Total Length of Fwd Packets
  TotLen Bwd Pkts        <- Total Length of Bwd Packets
  Pkt Len Mean           <- Packet Length Mean
  Pkt L

In [1]:
# ======================================================================================
# STAGE27 — FRESH KAGGLE RECOVERY AFTER STAGE27-3B0
#
# Expected scientific parent:
#   283690948d44345123d838fd47a8764441f491c2
#
# ZERO scientific computation.
# ======================================================================================

from pathlib import Path
import hashlib
import json
import os
import platform
import subprocess
import sys

EXPECTED_HEAD = "283690948d44345123d838fd47a8764441f491c2"
EXPECTED_PARENT = "17b734f778c4881d200c90967a19baf367347116"

REPO_URL = "https://github.com/themubasshir/ids2018-validation-safe-ablation.git"
REPO = Path("/kaggle/working/ids2018-validation-safe-ablation")

LOCK_REL = Path(
    "results/stage27_loao_unseen_attack/"
    "stage27_3b_similarity/"
    "stage27_3b0_implementation_lock"
)

LOCK_NAME = "stage27_3b0_similarity_implementation_lock.json"
FREEZE_NAME = "stage27_3b0_implementation_freeze_record.json"

EXPECTED_LOCK_SHA = (
    "0bcfd61b9e4f397f1f2c8bc60f50059ee8d320cb75481ab11c7dd49ad68e8376"
)

EXPECTED_FREEZE_SHA = (
    "dbe0bb7eb40d37ae66e11298a5ec48fc2e84878a240fb7418a3324e3918e445f"
)


def banner(title):
    print()
    print("=" * 120)
    print(title)
    print("=" * 120)


def run(cmd, cwd=None):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    if p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}): {' '.join(map(str, cmd))}\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return p.stdout.strip()


def sha256_file(path, chunk_size=16 * 1024 * 1024):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            block = f.read(chunk_size)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


# ======================================================================================
# 1. FRESH RUNTIME
# ======================================================================================

banner("FRESH KAGGLE RUNTIME")

os.environ["CUDA_VISIBLE_DEVICES"] = ""

print("Python executable :", sys.executable)
print("Python version    :", sys.version.split()[0])
print("Platform          :", platform.platform())
print("Working directory :", Path.cwd())
print("CUDA_VISIBLE_DEVICES:", repr(os.environ["CUDA_VISIBLE_DEVICES"]))


# ======================================================================================
# 2. RECOVER REPOSITORY
# ======================================================================================

banner("REPOSITORY RECOVERY")

if not REPO.exists():

    print("Repository absent — cloning fresh.")

    run([
        "git",
        "clone",
        REPO_URL,
        str(REPO),
    ])

else:

    if not (REPO / ".git").is_dir():
        raise RuntimeError(
            f"{REPO} exists but is not a Git repository."
        )

    print("Repository exists — resetting to origin/main.")

    run([
        "git",
        "fetch",
        "--prune",
        "origin",
    ], cwd=REPO)

    run([
        "git",
        "reset",
        "--hard",
        "origin/main",
    ], cwd=REPO)

    run([
        "git",
        "clean",
        "-fd",
    ], cwd=REPO)


# ======================================================================================
# 3. SCIENTIFIC-PARENT GATE
# ======================================================================================

banner("STAGE27-3B0 SCIENTIFIC-PARENT GATE")

run([
    "git",
    "fetch",
    "origin",
    "main",
], cwd=REPO)

head = run([
    "git",
    "rev-parse",
    "HEAD",
], cwd=REPO)

origin = run([
    "git",
    "rev-parse",
    "origin/main",
], cwd=REPO)

parent = run([
    "git",
    "rev-parse",
    "HEAD^",
], cwd=REPO)

subject = run([
    "git",
    "show",
    "-s",
    "--format=%s",
    "HEAD",
], cwd=REPO)

status = run([
    "git",
    "status",
    "--porcelain=v1",
    "--untracked-files=all",
], cwd=REPO)

print("Expected HEAD :", EXPECTED_HEAD)
print("Local HEAD    :", head)
print("origin/main   :", origin)
print("Parent        :", parent)
print("Subject       :", subject)
print("Git clean     :", not bool(status.strip()))

if head != EXPECTED_HEAD:
    raise RuntimeError(
        f"Local HEAD mismatch:\n{head}\nexpected:\n{EXPECTED_HEAD}"
    )

if origin != EXPECTED_HEAD:
    raise RuntimeError(
        f"origin/main mismatch:\n{origin}\nexpected:\n{EXPECTED_HEAD}"
    )

if parent != EXPECTED_PARENT:
    raise RuntimeError(
        f"Stage27-3B0 parent mismatch:\n{parent}"
    )

if subject != "stage27-3b0: freeze similarity implementation":
    raise RuntimeError(
        f"Unexpected commit subject:\n{subject}"
    )

if status.strip():
    raise RuntimeError(
        "Repository is not clean:\n" + status
    )

print("[PASS] Exact Stage27-3B0 scientific parent recovered.")


# ======================================================================================
# 4. STAGE27-3B0 BYTE GATE
# ======================================================================================

banner("STAGE27-3B0 IMPLEMENTATION-LOCK BYTE GATE")

lock_dir = REPO / LOCK_REL
lock_path = lock_dir / LOCK_NAME
freeze_path = lock_dir / FREEZE_NAME

if not lock_path.is_file():
    raise RuntimeError(
        f"Missing implementation lock:\n{lock_path}"
    )

if not freeze_path.is_file():
    raise RuntimeError(
        f"Missing freeze record:\n{freeze_path}"
    )

lock_sha = sha256_file(lock_path)
freeze_sha = sha256_file(freeze_path)

print("Implementation lock SHA :", lock_sha)
print("Expected                :", EXPECTED_LOCK_SHA)
print()
print("Freeze record SHA       :", freeze_sha)
print("Expected                :", EXPECTED_FREEZE_SHA)

if lock_sha != EXPECTED_LOCK_SHA:
    raise RuntimeError(
        "Stage27-3B0 implementation-lock SHA mismatch."
    )

if freeze_sha != EXPECTED_FREEZE_SHA:
    raise RuntimeError(
        "Stage27-3B0 freeze-record SHA mismatch."
    )

actual_files = sorted(
    str(p.relative_to(lock_dir))
    for p in lock_dir.rglob("*")
    if p.is_file()
)

expected_files = sorted([
    LOCK_NAME,
    FREEZE_NAME,
])

if actual_files != expected_files:
    raise RuntimeError(
        f"Unexpected Stage27-3B0 artifact universe:\n{actual_files}"
    )

print("[PASS] Exact 2/2 Stage27-3B0 artifacts recovered.")


# ======================================================================================
# 5. IMPLEMENTATION / SCIENTIFIC BOUNDARY READBACK
# ======================================================================================

banner("STAGE27-3B0 SCIENTIFIC BOUNDARY")

lock = json.loads(
    lock_path.read_text(encoding="utf-8")
)

freeze = json.loads(
    freeze_path.read_text(encoding="utf-8")
)

if lock["status"] != (
    "FROZEN_BEFORE_ANY_STAGE27_3B_DESCRIPTOR_VALUE_ACCESS"
):
    raise RuntimeError(
        "Unexpected Stage27-3B0 lock status."
    )

if lock["numeric_implementation"]["standard_deviation_ddof"] != 0:
    raise RuntimeError(
        "Frozen similarity ddof is not 0."
    )

actions = freeze["scientific_actions_completed"]
ledger = freeze["target_opening_ledger"]

expected_zero = [
    "source_predictor_rows_read",
    "target_descriptor_rows_read",
    "similarity_values_computed",
    "model_fits",
    "model_inference",
    "target_reopenings",
    "threshold_reselection",
    "bootstrap_replicates",
    "gpu_hours",
]

for key in expected_zero:
    if actions[key] != 0:
        raise RuntimeError(
            f"Unexpected Stage27-3B0 action: {key}={actions[key]}"
        )

if ledger["consumed"] != 5:
    raise RuntimeError("Target ledger is not 5/5.")

if ledger["remaining"] != 0:
    raise RuntimeError("Target ledger has remaining openings.")

if ledger["reopening_authorized"] is not False:
    raise RuntimeError(
        "Target reopening unexpectedly authorized."
    )

print("Descriptor values read :", actions["target_descriptor_rows_read"])
print("Similarity values      :", actions["similarity_values_computed"])
print("Model inference        :", actions["model_inference"])
print("Target reopenings      :", actions["target_reopenings"])
print("Target ledger          :", ledger["consumed"], "/ 5 CLOSED")
print("Similarity ddof        :", lock["numeric_implementation"]["standard_deviation_ddof"])

print()
print("[PASS] Stage27-3B0 scientific boundary recovered exactly.")


# ======================================================================================
# 6. GIT IDENTITY
# ======================================================================================

banner("GIT IDENTITY")

run([
    "git",
    "config",
    "--local",
    "user.name",
    "J.M. Mubasshir Rahman",
], cwd=REPO)

run([
    "git",
    "config",
    "--local",
    "user.email",
    "themubasshir@users.noreply.github.com",
], cwd=REPO)

print(
    "user.name :",
    run(
        ["git", "config", "--local", "user.name"],
        cwd=REPO,
    ),
)

print(
    "user.email:",
    run(
        ["git", "config", "--local", "user.email"],
        cwd=REPO,
    ),
)


# ======================================================================================
# 7. ATTACHED DATASET AUDIT
# ======================================================================================

banner("KAGGLE INPUT AUDIT")

input_root = Path("/kaggle/input")

if not input_root.exists():

    raise RuntimeError(
        "/kaggle/input is missing."
    )

datasets = sorted(
    p
    for p in input_root.iterdir()
    if p.is_dir()
)

print("Attached input datasets:", len(datasets))

for dataset in datasets:

    files = [
        p
        for p in dataset.rglob("*")
        if p.is_file()
    ]

    parquet_files = [
        p
        for p in files
        if p.suffix.lower() == ".parquet"
    ]

    print(
        f"  {dataset.name:60s} "
        f"files={len(files):3d} "
        f"parquet={len(parquet_files):3d}"
    )


# ======================================================================================
# 8. FINAL
# ======================================================================================

banner("FRESH STAGE27-3B0 RECOVERY COMPLETE")

final_status = run([
    "git",
    "status",
    "--porcelain=v1",
    "--untracked-files=all",
], cwd=REPO)

if final_status.strip():
    raise RuntimeError(
        "Git tree became dirty during recovery:\n"
        + final_status
    )

print("Repository                :", REPO)
print("HEAD                      :", head)
print("Git clean                 : True")
print()
print("Stage27-3A uncertainty    : REMOTELY FROZEN")
print("Stage27-3B0 impl lock     : REMOTELY FROZEN")
print("Target-opening ledger     : 5 / 5 PERMANENTLY CLOSED")
print("Descriptor similarity run : NOT YET EXECUTED")
print()
print("NEXT AUTHORIZED STAGE:")
print("  Stage27-3B1 behavioral-similarity execution.")


FRESH KAGGLE RUNTIME
Python executable : /usr/bin/python3
Python version    : 3.12.13
Platform          : Linux-6.12.90+-x86_64-with-glibc2.35
Working directory : /kaggle/working
CUDA_VISIBLE_DEVICES: ''

REPOSITORY RECOVERY
Repository absent — cloning fresh.

STAGE27-3B0 SCIENTIFIC-PARENT GATE
Expected HEAD : 283690948d44345123d838fd47a8764441f491c2
Local HEAD    : 283690948d44345123d838fd47a8764441f491c2
origin/main   : 283690948d44345123d838fd47a8764441f491c2
Parent        : 17b734f778c4881d200c90967a19baf367347116
Subject       : stage27-3b0: freeze similarity implementation
Git clean     : True
[PASS] Exact Stage27-3B0 scientific parent recovered.

STAGE27-3B0 IMPLEMENTATION-LOCK BYTE GATE
Implementation lock SHA : 0bcfd61b9e4f397f1f2c8bc60f50059ee8d320cb75481ab11c7dd49ad68e8376
Expected                : 0bcfd61b9e4f397f1f2c8bc60f50059ee8d320cb75481ab11c7dd49ad68e8376

Freeze record SHA       : dbe0bb7eb40d37ae66e11298a5ec48fc2e84878a240fb7418a3324e3918e445f
Expected             

In [2]:
from pathlib import Path
import os
import zipfile
import tarfile

ROOT = Path("/kaggle/input")

print("=" * 120)
print("KAGGLE INPUT DEEP INVENTORY")
print("=" * 120)

files = sorted(
    p for p in ROOT.rglob("*")
    if p.is_file()
)

print("Total files:", len(files))
print()

for i, p in enumerate(files, 1):
    rel = p.relative_to(ROOT)
    size = p.stat().st_size

    print(
        f"[{i:02d}] {rel}"
        f"\n     bytes={size:,}"
        f"\n     suffix={p.suffix!r}"
    )

    # Read only a tiny header for file-type identification.
    try:
        with p.open("rb") as f:
            magic = f.read(16)

        print("     magic =", magic.hex(" "))

    except Exception as e:
        print("     magic read failed:", repr(e))

    # List ZIP members without extracting.
    try:
        if zipfile.is_zipfile(p):
            with zipfile.ZipFile(p, "r") as z:
                names = z.namelist()

            print(f"     ZIP members={len(names)}")

            for name in names[:30]:
                print("       ", name)

            if len(names) > 30:
                print("       ...")

    except Exception as e:
        print("     ZIP inspection failed:", repr(e))

    # List TAR members without extracting.
    try:
        if tarfile.is_tarfile(p):
            with tarfile.open(p, "r:*") as t:
                names = t.getnames()

            print(f"     TAR members={len(names)}")

            for name in names[:30]:
                print("       ", name)

            if len(names) > 30:
                print("       ...")

    except Exception:
        pass

    print()

print("=" * 120)
print("SEARCH FOR CICIDS2017 BASENAME FRAGMENTS")
print("=" * 120)

needles = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "WorkingHours",
    "ISCX",
    "cicids",
]

for p in files:
    name = str(p.relative_to(ROOT))

    if any(
        needle.casefold() in name.casefold()
        for needle in needles
    ):
        print(name)

KAGGLE INPUT DEEP INVENTORY
Total files: 29

[01] datasets/jmmubasshirrahman/ai-ids-research-kaggle/data/processed/merged_balanced_ids2018_safe.csv
     bytes=114,122,375
     suffix='.csv'
     magic = 44 73 74 20 50 6f 72 74 2c 50 72 6f 74 6f 63 6f

[02] datasets/jmmubasshirrahman/ai-ids-research-kaggle/models/final_xgboost_ids_model.pkl
     bytes=773,138
     suffix='.pkl'
     magic = 80 04 95 ce 03 00 00 00 00 00 00 8c 0f 78 67 62

[03] datasets/jmmubasshirrahman/ai-ids-research-kaggle/results/additional_baseline_model_results.csv
     bytes=530
     suffix='.csv'
     magic = 4d 6f 64 65 6c 2c 41 63 63 75 72 61 63 79 2c 50

[04] datasets/jmmubasshirrahman/ai-ids-research-kaggle/results/all_csv_label_summary.csv
     bytes=1,169
     suffix='.csv'
     magic = 66 69 6c 65 2c 42 65 6e 69 67 6e 2c 42 6f 74 2c

[05] datasets/jmmubasshirrahman/ai-ids-research-kaggle/results/cnn_baseline_result.csv
     bytes=243
     suffix='.csv'
     magic = 4d 6f 64 65 6c 2c 41 63 63 75 72 61 63 7

In [3]:
# STAGE27 CICIDS2017 SOURCE RECOVERY — NO-UPLOAD BOOTSTRAP
# Paste this ENTIRE file into ONE Kaggle code cell and run it.
# Data recovery only; ZERO descriptor/similarity/model computation.

from pathlib import Path
import base64
import bz2
import hashlib

OUT = Path("/kaggle/working/stage27_cicids2017_source_recovery.py")
EXPECTED_SHA256 = "3c496bb2c31b3f819a37c389e10ddc848777625df61263a6490540f25935dfe9"

PAYLOAD = r"""
QlpoOTFBWSZTWa/aklgABxZ/4X70QAB4///7P+/ffr////pAAAQAEABgEP3VKL5cXg7uPS955Sjt9Dew+n0a32C+972+2j1WqN9g7pYF77zvNSA1wlCJo0VP
MSn+oZEaT0xRmqfoyo9JoAP1QNM0nlNAAGDRMgE00TTKMKNGTQGgyAAAAAAAAJEEhTVPMqbU9PKmMp6nhTTT1GQAAaeoNAAAAAk1EiGink0KaMjaQBoAA0AA
xGmgNADQDaUoTymNIYQNNAAAA0AAGgABoAAkSCATQBNBoTEMo9SNqeoaaeoNPUZNGgaaDQBuCEkOUJwe2naqi/nKIWqRKoWBT3k2CYvoxB7gqDxSs/78vFIp
hTX5uWksNKmHlccgeDuUzbMv2z5NK4d7GTjkVjJvEL1Z1UPmMzcQwED1E+kQqEwYxmWR+EXbuA81thwajYrMCZ6FokzUumkJSrMJqdsMyYBk5JKLgI6DNf20
ZjlRsqnWJQtF3cwxjVQBTUQkQ5TomSCIWeyp0xgyoimlbG2iGK2UZ3rfgvXao16fd5t/Yy4viE7fcm517ycNmpPGe+c34cpzo6fRejSYEVViKsdoFAhDJknR
svvuAmsYKIiwKmfkhACBYEDrYkJV2ny6UjXrgASsGkqYIMmIvs34azMDBfZjASAwyLy2/kToOO2Z3H51tX4Rr2yPreP5PdmX5Hd1lvoQ9m/5AlKTYU/0cNdS
q4Mx6IxFHMdZxxvcb5+rXmtpz5Z0iuDex+lYGXakp4ZMHCJd3s/XmGEKJDvI/1Fk3iUp5yXpPsfh/9m9wsqVX3bacq4iaMuX3emcuLr1wnpvmPRNp9EZ8LZk
RiuypE6nwezy2GBEQfrWUsbZT70SXNSaM9lGppvBhZE8OeE+4eri/zU4tuBkdpFYpjjOksK7LmzZ3sB06Y/d0pMMZliDokxsp8kGfe49/J6rXlJAzKbmwGRA
Ks1b4zkuOwJsji0YTFpNWk8EAqNMZ3yjs9H6GL5EP2YfhTd4c7zIzDs2FYVheQwOIJsZBAeMFVQxw2JP0cDCZpXVEClwyOA15qOvwjdm7I3BjlJ+izZrO7mG
KqTmD2CLHmxGjSQR7WITMX0UX3eg2bgnQbfCKmferYdazXEMbn2VlbCNuoLljDW5lpC4bZkSAyGOBAeJ8vj8h8IxsbM+/ZojxRRB8b8nU2ZQ2ijQXaAqVWdB
MmOv7T6xF89xSkklwXbiu2CgVmw5yD0Jv69joxp2tqlZhwhwnWHb8JEdMDCWrluXFKJogboi5goThuocU5KIcT4+iZ2+Xl75O/OOTRavh64njqFrIXBb17Ii
7TdIiGMpq0/jCh55rp60KalYpNunrTkgHHmwUTj2DFZEuTBVuKrEGYTV1xTTvo6ZNKEXE6FQhiYaaR3Iz2wYshq2DIJeB5RMlkqTkxUmmvwmyHuO69tJkRDB
VFhwrNwMOpvdZgQyHUj2UsS2qNShggZgzMzCI0POMLIKBXXfiLBc+nUtfWcQkeU0L7Zhfay0NTRHCKFPRduCQ4913lpjPbnNZfEipISoMk21ZNQF1puc5vy+
SPjVlbosJw/m85pcbI+1nLltHBskqJ48pvMfzcZYTxLV/iZj8B0M6uOE3C3j1X97dffyzUURWxld8B8sxssG5/hoPefNDWt54PJ59fJHNohWFJpVg7Cc1SCe
rIqnVOfmdfcs+DXno+TmL7T9j2EzI5gx7v5ChDSaFQNbHrH3b11bY9BdOcVDoReoSKpUdBqHcsCs7UPDjS15Ofd2CHjRLkAsqz0M4rL7zcpWiZkeE8BeBYaV
BuPUVqYcK5zvSNFRWYdKxJodKiYNsB9Zc/hMgMUmCoLVfMmEkPONBgMFY4ylxm7Nq45fG+Cs5k6BSdVyRsiq+/BVaq9nbfq6X+rJl292gO7eKMfRGzg1fXvj
nsc5187GPEd+x8gojBydbNGCsIvIOqLsH3cez1Nmkn4qIxA7tdpES4IkRkix9Uy3zQFASsYk0Y6DKCCyevfDrIy+OyCq8zp/FraXUt4VT2E3Eq4KIODybtbK
lpMWf2JWZSS976rXB3r8aOBqSGiaJo9y94SC3UgqA0MhpEs2fT6oi+d6LUrRRrm94Y/bCCRNahxsSaRegmiaLSWsiBg0i2bJi2wB9lMJivCWZwe3hBrYQ+1J
HZfJbNag1IasdwyaGU7GqBIOCWQfwcDGYoc4bkSV04Ym5bGNH78cDcNU06JAEFYMQhNJoXS9/uUfC0QErWFqHEDVbEsLQGFgWSHPLWFcA229VwciaakNSHKB
RNTSgj1NVW8TFkEFgXKmRMB1DVMi4DNIz7bEFptztu5fn3qWwDSb0ftMg3IKAzUiwXO0bxKenadKRzVEMsOi1Ezo3RUL0EiWKbUEpvFpOm5DbSRO1ELxuLF1
hp7qlIrMLFPHHEWLILpMYxuzjzkENuIsaiSjhjhcls2X65lA+q5EIN+KoTJ5TMwoLO9j3CfWakiroM3jApMSxQdX08sE5OJ5ty9uQ7RO5XuRkNyqpNYFE7bj
s1tbzOV6qPKSmTJhgHhyLrUHFavfKxt00xJlg+Y+cPBF/Z9vpS6QjoGMfqEes8+HHmJGsfnFD8J1nsKg9a+9L5U/oCQPfVTA9696IGF/6slgWX5EFVfFW29A
rYhz7cEzMqGcFzYUoY7WuRuWaje36Vb61HviNbSLRntuxu029JrQU+cSZ7j6xsG8moXYpdv341MwzYiQKoKiqGauBUgGhMoOed1lp1UCyGtAUlKpLZ9ECBBC
ZN8ArqYTJclokOKAhRqhYkqrsSnTcU9BX8HsRn6WLZtCaMDEEURaSOtj6og+tvlIVQ8u8L9KWUoS8ZgxDQVN2Y4cV/4pknkMFTNc50jWLI95z8e+gT3C2L7m
5aLH4JFA+oMXy4zyeSPsvvJw4bQ3HrvG7P4nNOtuaxHfnnER24eerHgbyTQNjAY2XaAgtRv5qtKqgyYLdo4qKprF1J4RtsosUExTia7clQRTaMagQjCaxrZA
oCIQb2O+GfLkZiIzpNDNzD/KKgclkElq+aldTJxNHrHUbeUyvQWjhGjVJiQ6zQap+SGUJ00gxMWHIUyObJldmJmZWesbl3F3KTAYKFGRmxQbQ733QQ7II1M4
nbAyagKRB1HIgTnEuXNR0LdBJh4i6Rrms9WKUrR2HnAwMPVJlI0MmI5BtUQ1kYG+QA34SBxojosgaWIACzeBQM7JJEBRjVhXZW3yss+UbaJgLjPDRbhOe6jh
0HHYZWVOx0ZKyjIm7VoYS202mMG/5VXcay4M2a1y43JI4mSvQS0ATMiKa2pmIiGIsLqC3Ek1zmSkYshoW40A+fMe4UDuiSOchL6GXXuhfrgMOFjvhM6jolRh
N7B3+eCMyWGxIYXBqOfYPIaioxQQhlDwzlqLEaQ5MO19nc/g3eMjvq+mFmepXSudlkEmSZBvsHs67J00u+sHXSRSCJ0gNoqsQRETbXD4R6VwQR5HpzMCe5B6
/aSNuUQxwmMgNLk5EQg0Ms9EIlANRuNrFKDuJFpWLgSNyGlJU1hjyFAJ4wjcyQ+pJEKahSCRLQZxEoXTZsszqjwMekMeaRhHBqff1GRfYS5lPmFrRE2jfItl
B0asQwY9MlIYxqIIUtBlyYoNvIOnSIkmqUSRq+Jl95mMhmJCLuYmBA34SUHsjGGLBV2PnqYmgct/wBLZcixBXGxeXHBYJIYjBUAdc0EiiogiDDLBNIdCZ1GN
NlYUpKcAq2UkXgazbzgsBSFczxF/VYyIPE0Y7TathAzhwU0lyaFAM8bcGdajdd2he2PrYDGoOQbySCgd9geAzQrpxJBym4Qp0GHI+B1jjNjEbZEgiMCcMvvX
NpbmutZV8NuyYjS1CtTU9pi5jhpxHfGLOhy6LzaYbQ2GWeqRZRDSZpskkuBww4paFnnp8OCwEGaSHZijXIAoKQcRs8hTkgtDYL4LFkSQOgbIIEyionVuLdWW
7K8EkQyS5MAt2sRBKFEjXXQidSYxgakmIlZQZj0TVZo3AWfKbC0kZJiMm84TXKLSbA1oN50pZlkYltxIESozgJiSF1MWXCy3gHWOrEhadAL3aUQhFNvOHhB6
j/mjeOfb1Bgl9E1rJsExjTGwPnRvgU0HcWLYuneQqI1+pK5tw1YIVFE+x/AdSLaoWguVfRNGvkk2P4qqXEBuzRcB4sDisUhr2jogw9G9HXf5DdCFQZ4p63Jj
cHrlI7/zGfmArt50QRZx3IoC7VoHKBoWuarqXQgMwNXlBJNpjT60YWhuM1hioRlIpJtJjFhkMkPFsllVELl4uTYE3SGWfDnkliKqibV51YRWOIbxSnMDtv0P
FCCAOcHcC59MVUbtQKCKMMjPYjB0YtGkCoC1Di7EXiCEg5h4QQtGRhHQlvbnuEKqA9TLPQ08CBNQnKS8k6QJ1I2ji0zWmb/TqJK5sc0rjvCcvLnhTRBE1vtL
TeVRkK5pRZZUzszqhujWOmlmqYM4yrO+2JRBCuRY9rPZmRQ+EgBacRrLAzQLBCO9I6w2Yh4TRUyn3WxQgMksWjFTSVNLgWIMtLQOpKwVhaoWpHiaS0zCkJjO
AMgs9gaVMJXg/RU9Ep6qjdyvR3nvra6huih6YtHv9fubLNy4Y6Kro2GIO1GHGRkSYBk3EQWiZguC3dK6KadpIbSOpiM2jcEkSFGc5UtjznWtlbpWFAYp8l8I
EkFqvCFpXF6+OaKBeWq2QKYDRAoH2tumSNA0g0pIw9JCtSvLkjo6MMKosYguRNhDtKDSRiA5razFgucmXEyHaRdYUWnV68BWJMJTNyV7EdoyB1OvQTQs6IkC
ZtxGwx114IWjySGjG0DL7T5iqCNasLLEG0qdmJ+NJHgS4IDR1jR8zSyaNuBU30XLI5UsXccdF+8KpcdcBaC+5oQZlvXu6rRT9IPsv9JUIpBZCbzXp4SczjCj
Q01i4by7idBJC1AgyQMxF0N7lwkIgbIfq00KKj9C/xhbv0uM7MzxMSl841dVOZ5FYr65ez2We0XckU4UJCv2pJYA
"""

source = bz2.decompress(
    base64.b64decode(
        PAYLOAD
    )
)

actual = hashlib.sha256(
    source
).hexdigest()

if actual != EXPECTED_SHA256:
    raise RuntimeError(
        f"Stage27 source-recovery payload SHA mismatch: {actual} != {EXPECTED_SHA256}"
    )

OUT.write_bytes(
    source
)

print("Reconstructed :", OUT)
print("Bytes         :", OUT.stat().st_size)
print("SHA256        :", actual)

source_text = source.decode(
    "utf-8"
)

compiled = compile(
    source_text,
    str(OUT),
    "exec",
)

print("[PASS] Source-recovery script reconstructed and syntax-verified.")
print("Scientific actions: ZERO descriptor values / ZERO similarity / ZERO inference.")
print("Launching exact CICIDS2017 source recovery...")
print("=" * 100)

exec(
    compiled,
    {
        "__name__": "__main__",
        "__file__": str(OUT),
    },
)


Reconstructed : /kaggle/working/stage27_cicids2017_source_recovery.py
Bytes         : 13634
SHA256        : 3c496bb2c31b3f819a37c389e10ddc848777625df61263a6490540f25935dfe9
[PASS] Source-recovery script reconstructed and syntax-verified.
Scientific actions: ZERO descriptor values / ZERO similarity / ZERO inference.
Launching exact CICIDS2017 source recovery...

STAGE27 SOURCE RECOVERY :: SCIENTIFIC-PARENT GATE
Expected HEAD : 283690948d44345123d838fd47a8764441f491c2
Local HEAD    : 283690948d44345123d838fd47a8764441f491c2
origin/main   : 283690948d44345123d838fd47a8764441f491c2
Git clean     : True
[PASS] Exact Stage27-3B0 parent recovered.

STAGE27 SOURCE RECOVERY :: FROZEN SOURCE CONTRACT
Dataset  : CICIDS2017
HF repo  : bvsam/cic-ids-2017
Revision : b7e532345512edcd530cb1770dc76636aeb52802
Assets   : 8
[PASS] Stage27-0 and Stage27-3B0 source contracts agree exactly.

STAGE27 SOURCE RECOVERY :: HUGGING FACE CLIENT
huggingface_hub: 1.11.0

STAGE27 SOURCE RECOVERY :: BYTE-EXACT MATERIA

traffic_labels/Monday-WorkingHours.pcap_(…):   0%|          | 0.00/65.5M [00:00<?, ?B/s]

  downloaded path: /kaggle/working/huggingface_stage27_cache/datasets--bvsam--cic-ids-2017/blobs/dfdcef4b8670e52af54dc4f82174834365a393473e877174cca46d17b12dfd02
  materialization: HARDLINK_FROM_VERIFIED_HF_CACHE
  final path     : /kaggle/working/stage27_cicids2017_sources/Monday-WorkingHours.pcap_ISCX.csv.parquet
  status         : PASS_EXACT

[2/8] Tuesday-WorkingHours.pcap_ISCX.csv.parquet
  expected bytes : 52,701,751
  expected SHA   : 27e83d518cb093faefd0f883cb4df3ad8b353f150934004f28d0e7962f9f31c4


traffic_labels/Tuesday-WorkingHours.pcap(…):   0%|          | 0.00/52.7M [00:00<?, ?B/s]

  downloaded path: /kaggle/working/huggingface_stage27_cache/datasets--bvsam--cic-ids-2017/blobs/27e83d518cb093faefd0f883cb4df3ad8b353f150934004f28d0e7962f9f31c4
  materialization: HARDLINK_FROM_VERIFIED_HF_CACHE
  final path     : /kaggle/working/stage27_cicids2017_sources/Tuesday-WorkingHours.pcap_ISCX.csv.parquet
  status         : PASS_EXACT

[3/8] Wednesday-workingHours.pcap_ISCX.csv.parquet
  expected bytes : 76,512,727
  expected SHA   : d23a259820b16e1ad54f9f3b58d5727c5032d383015f90bc7c07cebbdf8a7140


...

  downloaded path: /kaggle/working/huggingface_stage27_cache/datasets--bvsam--cic-ids-2017/blobs/7c5876d52189fc01af54bad6cf23afe9f7fbc0e3ca6c3595920754f0c3ba8f66
  materialization: HARDLINK_FROM_VERIFIED_HF_CACHE
  final path     : /kaggle/working/stage27_cicids2017_sources/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv.parquet
  status         : PASS_EXACT

[7/8] Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv.parquet
  expected bytes : 18,632,427
  expected SHA   : 4d78cee297c27f1a9947b9384793e587a46c7a3ea89db199553dabddc9835d4a


traffic_labels/Friday-WorkingHours-After(…):   0%|          | 0.00/18.6M [00:00<?, ?B/s]

  downloaded path: /kaggle/working/huggingface_stage27_cache/datasets--bvsam--cic-ids-2017/blobs/4d78cee297c27f1a9947b9384793e587a46c7a3ea89db199553dabddc9835d4a
  materialization: HARDLINK_FROM_VERIFIED_HF_CACHE
  final path     : /kaggle/working/stage27_cicids2017_sources/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv.parquet
  status         : PASS_EXACT

[8/8] Friday-WorkingHours-Morning.pcap_ISCX.csv.parquet
  expected bytes : 21,999,571
  expected SHA   : 2c00236b13a69f4b1c222b8f4a89451dc2148cb04a8cd0c45c2a87af51471774


traffic_labels/Friday-WorkingHours-Morni(…):   0%|          | 0.00/22.0M [00:00<?, ?B/s]

  downloaded path: /kaggle/working/huggingface_stage27_cache/datasets--bvsam--cic-ids-2017/blobs/2c00236b13a69f4b1c222b8f4a89451dc2148cb04a8cd0c45c2a87af51471774
  materialization: HARDLINK_FROM_VERIFIED_HF_CACHE
  final path     : /kaggle/working/stage27_cicids2017_sources/Friday-WorkingHours-Morning.pcap_ISCX.csv.parquet
  status         : PASS_EXACT

STAGE27 SOURCE RECOVERY :: FINAL SOURCE ROOT GATE
Monday-WorkingHours.pcap_ISCX.csv.parquet                                      bytes=65,465,382 SHA=dfdcef4b8670e52af54dc4f82174834365a393473e877174cca46d17b12dfd02
Tuesday-WorkingHours.pcap_ISCX.csv.parquet                                     bytes=52,701,751 SHA=27e83d518cb093faefd0f883cb4df3ad8b353f150934004f28d0e7962f9f31c4
Wednesday-workingHours.pcap_ISCX.csv.parquet                                   bytes=76,512,727 SHA=d23a259820b16e1ad54f9f3b58d5727c5032d383015f90bc7c07cebbdf8a7140
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv.parquet            bytes=27,901,448 SHA

In [4]:
# STAGE27-3B1 BEHAVIORAL SIMILARITY V2 — RECOVERED-SOURCE NO-UPLOAD BOOTSTRAP
# Paste this ENTIRE file into ONE Kaggle code cell and run it.
#
# Runtime-only patch:
#   source root = /kaggle/working/stage27_cicids2017_sources
#
# Scientific methodology unchanged.
# ZERO learner inference / ZERO target reopening.

from pathlib import Path
import base64
import bz2
import hashlib

OUT = Path("/kaggle/working/stage27_3b1_behavioral_similarity_RECOVERED_SOURCE_V2.py")
EXPECTED_SHA256 = "1e3c548d4feef38dc7fe7045cd73f9627428ed349af0bfe36ac375ad225558ca"

PAYLOAD = r"""
QlpoOTFBWSZTWTKVXecAK7X/5X/9xkB5////P////r////9AAASACAAQAGBHHvB6ESgees6DmwO21xd7p9W9PoSAII6NSKooB9nra2a84aawB5ipezTT0G+z
rPdwds+be9dvnce9Wcno9Dz7VkdYvj1YDr559Xvr7vHtbrYvZ97K72d7d7UuxvjedfHN07dq2e49fXgu+6qNd673vrjvvt55ZbGPLR1qRamfV2uzfa+eUH08
q6XfPeZWL32c1vFquxqHuD1vPF9V917b4SXeAPXt3vO+u7vuUq+++vd8hKaQQAJoExGgjQA00aE00CJj1DUaNMRp6jyjymj1ANBKaBCCEE0TJiYimnkaIxAG
Q0ADRpoAADQABIKTRNFU/EyeqT9TNKeUD0gA2oZDTQ0NDQGjQAyAA0ASaSIhBDUyaaFNN6RqH6kBoZHlNANADQ0BpoMg0AARJETRiCak3pqNNVP2lNhUfqnk
elPaptTag2p6QHqbU9TQNDQNAPUAESRAIAmQJqZMQp5TGpPSbap6j01A/VABoD9UAHqAAZOaiAj3xQVCT+7p3Ifjin2/s1+eFr0pRB/B9XCadOOJVtJfxrU0
DDQky0qtTo/4GFXTbSGP9Scujf+VmCxxskfu6673N4/3shMYiMnKG06aaBjxdYfctXw1Uwoogz9bCzlgdt0/Aljp7hDp4UWH3dl4O+0A7a6GGJqc5RXCpRqA
ooW76RFxBzMCWcKUq9G64MDG35EgjB+c9O5aaYZllH7XTeQmh0SI5IoFzKBjMd5SLpgVhpJOnb0Zt/N4Oc+dOj1VkkPFk46NhqICk+96PR9Wf7n1rPuBlqZB
AWIOFKStUEUWJMsIlUEa2VJW0xWle44em14ej3OXLE0OgbJAzKq5L386YZjuve1WlRqwbX93NuNybyepvVTWEZohEacijg8XW68fa7sm2WTUnLz5j2/JofnN
DjVm/vaO4+mijA71pYf0ENfCdWz4BFVVZBZCtQFVVhOvj19L7702c/Ls4beNQF9u9Oq57GYK8/mZvgmsMndyT0aIwjBk8UKjuG9GGMEUYpXUMqD5rOOrU9P3
8h3TV7nxyzr3FBtIfSTQgPxWb2CUqpWInghmPgXDLfgymDAYIVUkOYySYO7YQWdEzdumyVEZiVUKsPSMwIrE1bGWqpWAVpQY9K0RqrIhaSOVZWAsCp5zoGYE
DcGEv4ns8YShvcuyaHy6mYjsLvhvxF+iPZviCnbEx7ayY2Nt/he6mXWp9mK8OegQaE0Ro92JoaN6n40R2kpJ5abANHw6rWKCM76HsHg+KZOYcOPPOUiFO1tt
wnsQ46GYSmf8jR4e7r6L1Y1rrDiE0OsrInbotRfZBtqrGZM7ZLEE7vjXsNKtlGHgMwUwqK0k6WzFhwEpQbG2q03AsMIsnaZh2PynvILI2NsOpNFaXE33eXqV
gNUW9jJD43GG2u44FzGKQEykqyYifhGjWk0W4GJZcKlGUSjVWDmQPj6n0be443AeClQElFi29MlguNELYVVLATI4GWxAURAaUXMKCIYFSvh7POezRZ6L2+N9
ONNbllK9aacIuFVU+Uor/cJUWCCd5Z0Z4Ndk47bmtsxKh8jO+tCfZPOQzSkh1LxoyjqRazEFFDJkXjzHC9N8GnVYjwvPgx4OhQkTGk0TptTTVqbKWCb4w2mp
MFRCMFBX4TgJuu82XvDwZuX6OFDc4iTaFbpnkc7HaWGrmMxRHpdQdMDWWBbRZkyFmGUZS3RZo1KxCqiVtEKlWB7wMPfZnF7rg5cjZiTm1tiiMLCpYYltCwXE
ttGKzLZbYipCi+WzOe3cZj6D3/kjAwhmteps36O3PIsArkPKKpNx4LjhG0Sjq66evZHa0o95DimUN7YMiMddj08VpnJoyrpLa5cuZco2iOF5x0Zhq2pfBs4R
h6cdCWU8Dug3vZ57Ada7u6jOiIO3bkSaID38nLOKI2gVo2K0r4uZW0pCQF1ARJSAdhMmEkw2HT9n53vvfvI61ZF6YWYJofkX1pBL/aBY3m/F4QseF8dCv5u9
x0R8TMEiUJQTuP4HPEiWp9zMHCGMNQL6HiiAvmRdF4/M5+WDtTt72/pme0MGAkZAfGeWOLPpfLrts36uPvGcE9OrMUaBtjDJRqrhUY2ljWADLNFic/SOQfia
H8X8dCmLlJJl4biwl4RhylRMVDpFCJhIZjjJemAMRTMdXd/Lvxzqp6/bTjaqWrQdiZrkI5xCSRhJAU6U8Q5M5Txiih+CICkgKKgBsARD1QUNP9d9kBF8L/B4
fltcv/DMVBBu1SAC4+V2wiL8YAgOlW9txEStNLAIFiAqmaNbMrKGo7oBhB5EJAkYMGB+XgZxngbWKkZyZy+/n6TN6xmXwOtuKZitzKDA9pzQWrk/D+FueqBW
kzL+0XpkwZJUlPt5IHV+brTdpRMYZxftggiSEx/z+6iIEUTLuT+H9HH05nXtb6DdwQ29qQ6huVvuQzc9J9X9x+jP1pfnx+who+UWOjLc24ZmzdmIQ8UYRivx
2QJcxR8stPzqhNSZM6lnjMe3ZnbqYpRcZgiH4yONGMpwhh4YRJ/RTfUrJUUKTnRZBNluYaHj62dc9mHvGMWbOBjx/QHdZ5pdpgdRf8aiehovjXztZ4EHCmRs
RDCt68NUCaXTdq5atbY9CaoLBGd1pB8W2ECtJlB4LTPEWWZmo1YDzhVdGhUWeMegpLxM012RPcp3qEnxvnjcith47qBzfuhoQctla501zwWelr5ld4sdN4N3
5eaSdNxWdu94PuGecE7qPqdxAentgaBNbc/Qe8BjYU128m/eS+VcMiu87zafsidF9J3w7/XMr8xoabHm75X78F+X78yNItF1/yxep/u2lLRq9CKvbTsQTkLB
UminS2EGCidK3xhCiWOZJmKZVJbPl2VaNs//Nnhlbht5qtkpLxu+kpospl01ckNlb5ylt1l0CzXUipjqUm7xrWKt7I2HoWrBgK6PXos7XO3napsk5ozw04Ew
thg2DZ2Grfp0W91L+XfZ02GjczMwnpol1aKsldjOqY8J52WTKExg5zr4U83ilLxS9GdSyu8HCFXNoJ5Y26obdMHyralbaBYUHPwKcscDf0wz3bsbenymJoEx
pTJJCSM2ds/g7bIUyQuGuO1dXX6I6vkojAjw4wO5N6m6rc+pn8jmjvTn2tnZVeI2uclHHkxoMq9OFRhidun2c/Sfd9W6wkp68mn53fEHBr0TUrGFXWPsvqw8
6Q+dfazvigfvfCQ4/s6bSYgfoj3Ry65cXHqo0c5LpbbOcezHM30WmffwNe3Wd/JyXPGDeuz77ebeWMeFzyqrnb3TpFY9fbnxvpOnOArXl4+XdsFcsQmjrIuf
NdIjchCXT3p6bGr8Tx3ywz9qgsrVDeHlfmRRYpFz2VHIPDl0QkajJ2vQy9z42vFq4omcfD+jpaRLKfS5x5JwZGY4nuuum8MtsZDkXmp9sMpN8ZbdVO5qbaU1
vRn9X/FDSYOe0tHgLNPcn+b4TpTUOztKKOYf6zyjo/HBzNueFa1TcrTlzzyy9fK9zzfj5eEOHisgkgCcPug6Ejgn+WjRejQXws9TnDrw1xp05fPWBGWNMTeX
rZr1zooP/bgruYkSHE7DPqpPGbCIVpNk+uuu92saksfwPdt16B7yX00tZYJGnS45TGg1ZE6TMJfK2NfNf0No82W8zoc0RnZ5en7vr52XTlnZNUQRsnUmNCSG
D8iqZvcI7oPoq2WVtEhvy9OrRPzz81lZVnohfulNIs9scMeJyaPnPfxnG5i6/IV9DFuObbXZJ0XHq+Tlvk0Fm86Em+rt3GJ9TcT1bWmvaX6N1Jkmmv3G5CmM
yquvFrAoU603k4ys5McDglCuWBHGI15I5/j0UcfkNlFR33vaxy1Z3uXkObp7qOW0awgjW2q3QXkr7oFIowvlM2qreVzK6mKUIq0XEKNVcHeJvaLW2F4dG62O
W9GjR3+e77kKUB8o3Taqe890FFFB11192q9lQFKS6Sdm3RdA67HgvcnuWet4zvZid5DkfBclUCFBNu3QtePdyTer6cInO7eDLHXO1QjE4Dx57ddjdkgBIHEg
SRAh5MJEUaZeKgLrebm7uWw/agBCA/WJhrNPvMvTU57mSASSR87joSo8vtyMkLWiSMtFTVCSZJIXvc+mGop0Iq5V7bTS8PAoIU3dM9foypJQIywhMGzceXvj
6a4xlvrPuuOlDe9yqhRvm+qtpCZkmPem+i/Xd4jl+j3Q3rRdfTee/CMlmOqFPi8FwSWLNsTExt4vkf2WRVjt+lcMffsfo8HvhJAhNffRHdG6xYu9+b8vn7nI
7HkFCd33/wBqRfZ0N4Nd9D8bYJDwgX1PAzeYnhQ0fkHI3lO5+sXEjDyafEM9XY3GxV/m5H9qmhNSh/Xv39Oqrv7MDKCD1NupJ0u10lVkssPNIDlTVfHz/X06
dOHt0kExhqWojjTCCYecFD2xI0aIQNr5wOdjFrIRt5iJFKWe23o2zxKSsya6Sd5XXcy7dHav5W9OGXsDrQklL2n29D7aJZ7Ltsoap6hF/XGrdKcHi8XgAm5z
g8Yvj8GLrriwcISu010Xizm02vcNadWZGhMohOoSaUFianKT0PxxXqoyL7H5+ugcb3cnCUbY8/oO70ecPwopBX0ufSrp8KEXbx6naqdLNS5qut5NOkrHjWcN
NNpwUpZZww4dJW5NDk25Nwf3kysS8TboV3ehmkfF2mDDiKRNbSa4lJmqNJZyd/CqzNMlo0yS1sTi7+92687tLvnIqI1AcC5emWXo0uJoVRRW5JjTNGnB2XOd
BtuzQGiwM5yRQqdQiMTBczOVRClBA0VjBcDj3YXnu1errNpaCdLmt4bVy6qOY2mDha3FYVzJadZc2N31OeuPTnmaxzkqSrE2jDHLiWQKbO8JhrXFJwGW4jIa
pzYa3nWOrveXEUWzXEXOJre3hK7R6Qgf41f34KDhGRALj+WXTofXCdS3fvMPP4/kJl50PqbeiqVDTFgTGnjDGEEhFgS3nYLYtH1uJPzcZhXAsjvzBi2n8l7S
qT2dzg33USNKGCzJtTtyGUvL4WZW5nQss/K0czilvjDLQhkoy/wq9SfqQPtkghjM6ijMXJ1RxbgddrY3k21iseyfTTJhhlffi9t42sXHLSEFbSbb2Xw5IiAz
A2JTzJfNo7tB0Sb2XDls4YdNupvzQZDYfGfjFFXb+j2DR1rkby0z8F3iaja5fZlOE46KVJsS5uh2Yx7Abw33hCthcpziIWm2YteEd8LSThbtWJhgjmOXXCF2
5TG8qGkZHxumHfl5gHMQ5Yjqfeo6lYxdL0t5IA+l8QiqmMS099mMWI4kYgXLKJFmJWAW2K5SjMSsbahWyT06oKRiqkNXEntZVh+RxycJKwqBltGKVYuIXFVM
hy5lUFjC2gsNILAS3EKCsFHSGJgVIpMDUVk9zYqLDZKLKHJKMoKWHMwESdCrl5ocCdEC4oJzSBjXD15nie/6+NyLywAO4qqqtK/NkO/VBsId4DOJjynOUO0S
hgZVi4LESYBQQS8/z9On3zzKT6iBfm+FzCebUMCdiahCLIUG5IsxtOXV67BuCC/BxvP3Tg3wQNq7nts18T5ryC7USofvfae54Du7ljls3cij5Fplubfv39v4
PKJC0w7OuGFXNx8el6/wPOtXu0nBuOJGwhCNk6i1WMp0TTMRYpOonVO7pWeKu3xc1iSsWxEQSpgnSdwBrYy05GmNvnK+9sqptKYE7b6sJx1weBrYfsjbW1Oo
xShFDoTni5c09OQeS8V3ZcymV7LuMbIXOiaCyDUn1zcbyx0U6zrn5yLzpOCy0ym7OzrIi7MF6/XEoRehaaZ2Idr9cjOoTbfyBqq/SgD51un79RPq1B0nwPRL
GK19PxUxj4tM6gNlxsswwRR3OXm1evsmPFlW/AxKcZow2zTOHmqg44okbw5YxMsU5gg1Xxty1m7RVVlTnVpQ/bgURFMa7JWU2UMxWKitYzF6b8U1przzfEtW
ESBTjM6bTJK1OsHLXOuomME84k9vnmPbrGX/AcQY+Pa6R7bz61Vd/8zzbutWVJBxQiYaebRxhvA+RjyN5G87nmd3h/VY02ZJmgM431swxrYgHpQzdn5v1CcP
AI+kh29f5eTkw3dB0Sa6aPIeJeNnVvrg0F9shgwtaPXB3eb6j6Nc7qSTHoTdzmrT2eP1it4d7vi7PPTuhPwU36KrKKrbZt8PRjnHthbLTCWh6lJvYkjl6aGl
vrgJerkeEa+RN5QZ/ZxrP7ym9vY8UcWY/hDxti/PFO/EzUzv87O+IHa9ndia4p9dBQNpPuXJvfcgxPN+/khFJaXOWO+Wl4NCSSE6UbWyvIYaachtYgmw3u1m
zzSCeQ309kfla4sLt2d2VLYjtY0sWgSKxIEFbUTxHd3gNINREQq9Pu4UMFd1HCEyRr29raUAy2szOCZOmHc7judjveTaVF5oVRxSiW69mUXDXUOVBzlZ+pLA
d8lQZeRxaPvEgdl5vJzN8sCrf4lEbaQXFqj9PkU450PWx/pgYPMbysS8Xg+7a+d/HJt95s2+La7/oMO/NHAmyKT71JMZxEJJU6XYCut21moG9IXxSapOJF6e
JM64X8jZdjRzZoaz2ew2kz0fjuUIIhoXCBBAz+XlPKdTSmdiHc8v5PEZBJBOfzkB7iwSQahUZABMCCrdAHxbWWA3Bakka1MSr8LeNEiglynf2BonmAjFXQ4B
FdT5pvoN0QJ6qkva81NYHkMTk0d3ecS+T+7+ajTnSaa08DeT64efJFw4mTOg6IHTBz8yMSvOA2DY44xtuObg3GyS3IolV9BwmXy11fM68dtoCGxpsVcDyTmH
TpewUftoI235otNhyl9FeEDUWBhdRQbaDnCLUaw7fQvd3+MIMvSvX8sZQmnXeLfSaY+pRKfAQr4So5OqFt99SbC2KH6PG/woHk8UUKo1bYaZveh/UJ77q4M8
XaNZAzukc03g7zvehEERXZ69B89H56+tf1/ql8kX425A0UVJmH+33bPvy+37fw/CsZjBVISPChrHdiS+s2+LaXPA+rnge2lmH6Q151lw+7n9m06uxKQ74HxQ
TrAo3W+erhLSU3I2gfjjvrr98zMai5v11IOn1/fnndyxUkcCN69Jr7z7/wf74Pzb64bqZ7fwehV107jevoPJE8GuByIbYybUCdjXAhG5hsp6D0OWfH0edrPT
Lf/HluZu+v9fnSE4zAkC2fbUPu6MRcpU01bGSkSQljDSYfYGackmmomUCjBRRQxiOYUiWIIMhURoilEFIwPpOfbe/JltfeX7oxbm414p+O+TRUXAYxAGNhzV
O9li8JMBhQNym6ps59rsM2j4u/KgeiimzmyGv7R7Pw/d2/VbwQB8ZzOv5PhZI/yhm4fi9SlvvJ/owqUEfkLgpEsx9SPx+8Sk/f/E2+KPlj7NmBtmGsbBsYXI
lw2D8v+YOA9eYOLmOeTpIwidDQyjHAlIEAl/ypGgQ+3Q120OqHYUbtOlAb2iiFuDOCNg4JewvetjmTKfM+K6oz2XjIhE4sdfshSlgdvfXVld7mtddkRE60t/
wFM5n8jYsD1ytC/bGW56+RkhC4GOdiZv0exbEgV5yDBg3bdpgxIap0xDtDgXqAF/hpmmBhHiBIEKIVQhrRJBA1RoTRi0lQ5FnLOl6Fw0HG57R4w5TOJhDAwW
+U2ITatLpstieXO2DTa9qBw3rQZaHAfs3SalqgMXYCAhyy61BJhrvCn83MQ5u2pWSesQxndSb75vwHbraoTaJuJvOmYpPHm3c+l5LTXBk0Em6B61bawMTqIF
dr6H7V1E4DBNBMME08G64JjsNKpLcaHUS11i/R1P0+t/yIkTqJQL0sHU0muHV6oUWenGetBrVozDpsJKTKLNEO7vb3NTQnW6BzvaQkQx5tyGkxzPqaejcOVs
duy6+fHSmlqT4pEMIYCmxLWi5YOoTHcQegkM/Ex4hKHJlR4908THHp+48l1Hpx6m/xS7Z6eIxaTFtkZGGWSP3qpzHgHdZQdInyOfS89E69E0DPJLgQMaz29I
0YOstE+GXVQ4x8PnC84BrRIZC5+gooJuYnO/vPsBTXeO3BIJnr87bNr7PrASHehKWxmlqwjQfDBokD2gFZbMK7Pvntc55cfXR+FFjEwjYShMCuQmnwBiQRAJ
tMmCdwNjt5F1idUFwhqyelIa0rXFTDbLoboKdU8HueJzECAaaoGEMYeKcDqZ3LUDbVzgng47uQdhOSkWuAJyyoCiQhISxgeQOKPXEgeL3yL+CGSa12yzNtTI
TCwmjt7RNDI2zA96GBmc7kQwreWY1d3JHPQkIE1MuHMyQz2ODa3AvdDpniDmj4mzjoPt5yltbr7Lf36GaG2mAZDgSMbWTOV5KkpwWUoiW5amjLVRw0mUbZkv
qs0NoUc3LFXASSGJijmJA+aF7pZIYRBVgkGZZYGB9qzMLOu7K9iUugZAPrQtQprqvoj5nuJlFtwD0CHmUOKW9V6QGn/g694WLpxg5hY7ESwG/yk7mSR8AupW
Cmgp35BIBu3nGBumw5MeRwHTSgzxAyFPehWR23yJHgbuxxS6G8Q2R5m2R7E+igPn7215PfLW0TxQ3CnRHFQ4dnscL0BE+5HCoSwIYHkJzW3TA7e77TuJmegt
PmUWsY8SEEkzIWnyckIeF5VjhGBoBmOgj/Ez2YCTp0MUNCTT7QSAwE7G+0perZ611dkcAOFIUFggRCGkz3obCnSBmvnW1JgC3hWwHO/MTurn/Zmut9PRohvw
dTUY4sMkO8QLo4kb/i4mJOTv8NxoDikFoDTbM6CmJstgNA4fJL4QkvAwItnSwhIY7kgA3mpvmXt07u/y+ceEUgrUNBSJ8c/CCVla0EIhaGomMlklJtHcNu25
IwEtlVQuCSOEgvhvhOA7uxb1RRs1VvVovdH0enyXG1OGGHmk+zRqTjwBRYoiQnm0HHYkvE6SaxWq7oS4J62Qx4QGwxRXAzUgXchnAzKEkgxRixILrkPP7PVn
imRLdd3yOLvXhhpKlYm0zvTkoQxKDLcpkCnhA3hv4ZITlynMyJr9k5WDiJxyzmoaJi5pj42QDfZxrlLuekzTbfxXQOWGvA1JKtSFoJXMCybjelhtgUCao4jo
ZIWOPzAu0G+tG81oDVTbMUp0smUXAy1QNYmFjY3m6hzx3LCXwHIy6JsMSe9PogUH7T5FFYFBvOQ7Gwa/+uo9CAHRDrqSEOYhnc9myPxyd+3k0GDDNOCYN1S6
R4aBp42cQwavexBL5JhhiNI4obQW/TpKxPz5B5d6PjiW+3A3q54VvCI0c+8L8GdQ8A5DysjXb8LXBltJvTg9U8UWAtDuhXvzWz5gWiri9J/Z9r1fRfq4pfPW
cFwp6nuZy9CJ0SKKa6E552CLoeLJybvvH3SpSZsDJzW0uWnY9Ig+/DTxMppYqfoh4DPGwqdhcfchs2e+ATQKDOuJfeZYBoWKE8HKor98AgU/Sfdbm2Wg5lij
h/sW8HK7r1R4fJnLfeeoOqDZHA0pyI8GhkohVkOqGQlQnc09TDJqmpDZQS9nwPiOXkIQuUWaDuT5yA5HWIQfv+8DCJcQwwRqAJuVSH6VP8QNhOAE7aOHhu+e
ATEUKJV7DYv0Rny/kuYdY0JkPdQ07ndF0x7MKdkJl5H7l3MMEwTBAxNQ1E1DINI/we/7f9aseU+z7D4/sc+PhqEketmrPevVLf8OX4MS9fcyTsOUMhr7udhg
puN+bYDTcAsYJZqY/LpJKLr8xQ2cJCCwIP3O0GbasPgY/ZwlY89sViXHmbKLe08VVXMM6z2JbTTjC4hheA5B1mIeKeqOnfxHk5O4KB4SoqJ7zsngcBTwl4RE
elvFnE4mICbCMxmPnoyMyjFMN1egQvV2e0PQzdvqF5PrEcxh14n5PRRVV3VYLE+fW6PzHBo+uM3ljVNo3LK5HHKX46GiRnDP5iGPYZtGOQXMje9oxkrTC0q9
kvRVbCmsdmZDvcvmZf5x35KRstCZxRDJyFo2mVkDe7lxJomWiJhZaGgRQUQXU4VzeuMy5gYFFVsmUNrl5FZKvSZPP7DV5LmYtoG6WgQkhYjkLZ+tdQmphhN4
cngZ9XZTLrfY05XhVNUbG8qBpcPt24oa5EX/jc0gZ/WazU/o4g0s9Hiq+YwUDv7NTyO3QNAqRTd7lR8Cjsy2F9NHFWymXQ2FQC/FJgWMiNEqimQpiSxkG/db
+K5kGncNGKYwI6B4HYeJjmbiN44xIX5DY26fu4zNtmO51g5wyNVuUuFdDv7cTTQbPUBxrVTcXLF4y2dLWEzKVqR2ISSEubarujlgoRdDUzS+FBEOwQldYo54
g7pthjm3Ohvp1wMKNf3EkR805atmpm56kr/UQOKB3vI55DQcsd68FoLEeEZK5XbSBguHSZJIQh4do7zVI7WOxXqGePDy3NBunccTHaVXIyW5eBk4LkG+X51+
FXlzu/h4riMI2Ps7YFE7D4iECAH1cnW1V6LPK3VRRhDZ9CB60tsRisie0ECyfpzr6H7Nv7LVaVsPnFIjiVTF93obAbwEPt+CfHhedEO8ehe1NmCtEZIEAZZp
VK2SytNuziYRL+HzclklVTU0gXchTDSNcjhHu0CDCiKlUcRqEBezkT7F905+rrMB66SWyCcsAV5iCXYKhWnOQkkJJIUFVRLQqpaUrFhTzqAnfWJAqCGpZCy7
kqNA2bcQ7xg7OKvzfTty3ZyEsdpobQ2TMLL8fkDkNK6KPA/USEGERHmBzFQuDgvyHd0/DGfht7HzKb1N5ch539EsTZ4rglz5btczBegJEPu7dhNCep6QC4mS
obx3aGfZ20HV89Yia8QoAM05m8MCBAYBIHVtMeiYG7C9UnATphSNMjFCoiMRjoM+7CAyxj1jsIOst8JvorhQRTV4F1vX52BbVC6nQeAdF6/G/fx4dFTQU0Vx
jCIfD34pe6TYr4eXunIVx3bz4x3p2ATFSQYQYoHWiiRkImfZr0+Usn3Kfmj7vz4Khw6i8zl4n6Oz2lz1EMkleqx4zaTvqzBXYMMjzq1hWSg6GtjbvuC+JGFt
Lq1mDWsQvbUJi/Kl7EyDB37+xiQTnNpsh6rn8/Dq7/UYM1xJhAhoRUBuw+rtqk1IWjIDlQNBM1+C16MtD8IT4TbSIx+QENLv3Eci4OFu7v8vZD1gQJAiIIgR
GzkpDJICwjBIwGD5bIBknok73qJJ3MtcLcgipDj5Xoco9SSmmb7kVa6gaQkVK2pZScdLHBZiY8T3uOngd59g7ChPOmbJndWNtVYtQLCJkabAJj4vBzhur8K1
5nHEvvXM+LjlKeIxLOmK7M/lkdgIi7kEshgwSYxYSRMCTMn10hj28tvessWhaeoHPoTO0jg6GQjDujlEWvKzRKa+Phlxd2WI99THG+qmaPiZlw2ImsVsF43s
yFSoDGDUFwMUl0iC99eEfkfXpA9aLInS5CCjGKDApgxIQirIlBEWAWGL80V3Kl0N/C+NNWVbJ8NsyOkwy/AkVRRSiTskuUfcUKjzTUYe56M069mjDlrqTLUi
KxKEmn5v3KPNrEZMDIgQ2UNTMvh8D6x85JjjZvu3eXPrZQwmMApMChCigrQyC4Q3sRl2SJGLEhhKRn28VTt7/3tHxYhwwt4NvcZE5kehuw+UWkmeD29qsy3v
H1iWZvX8jb7iSkuvW0zsPqoXIBPf4aIGOBzgbiQie3HQW/YGPItgY0M3PbfeTiQiOzQTMwKkUvC6932ZWNuyG9XPkvhASzv1LDsu5V6W3J2vvc8UhyornKIO
kWoSFVwhc4CHh1E4HqxxLru5mCJM52nGi4S0kHS5QWjgvR4h9fC1xoNhdQzFQXt4cTFCRh2cR4OZTc1iooi5CEUR3v4atow1qJ1al6rOob2XtDsS4n2aDBqd
NgAbbKd8/YU1oN7dkL7SuDA2adUPl6ikxK3NquH0bj8OfX2OObh9cow/37krpXTPTmXyZzt0rbChYORGdTxPGGBpF2rJXgwx5ul066PbkdaJLYuplpLdp2Rj
YliypWncIN96CmYojI7ZTk0iFISa6zlaXV+XipxWdi8TXKklMcDDNDbqX4iHN9dUuXNLeiHIKicwHOeijkcCKpgOZL1nWOrNW6mJBHgkKLG+ytOo0YDiGJ7K
3NTakciBgpnYQ6y+TfTUwzDG4eGxkkiE/ZSmLum+Ccg/bBS4OCc1TmB+NhArV0sCayS5lEJIkEIuSL9KHU7byYeHIwlyqDSieFipDdCu6gqG7dc+YvbjMW7P
JMQBuRjSbNlxDs6CWYERho71VJYPqsyMCghnr1J0CHA4GrlU8e4FcBHgIeM8I+BUrvPMWCw+lrlk6fYestF8R6SzgIwvy1OIgXvFhGHFoIlbqJCgLEkSkl1s
p0HoMRNouRinx5n0m7QI5rQHc/GAHJQ1uGJ8VIpcOpBnkJCQYxe+yQiNkvELxYkQOkF5JhJDMqEgGe88tQ9OiIUoX55G/ixoN9lChslDuiMv1QL8rd+yo+iZ
ho4hy4lkA0ihtEPeRV0aoNTAx2psWh3b4OMTQkm6FGRtD4B9CGuSzxDUl93FIyZKGOr8AcvwguRogZq28xC5p4TPE7rAhu/vHK+At+XcgHZJhKvipLw4cm3f
srrFvO0VamkL2XamvlNs7xtjZjcljy7iUspOB/EvbdErG6cWHtCRKcMs9BFi1jBmsZ2m8nOdDDZrocU/X2KooCyQO2RR45BhFyEN9FAEdxo0A/RkYWQabmJC
RCJFlDSI06X2d5owcG4hjEwPPI8r4XxZm38KGbtR+x9sOYg4EngaWR5ou0ycCl+KhY/eDOsao+qY5yry+ROylT7AN0E+Qj4v09Ke5Dp4Py+qUYb6VFSKKsPB
CVD6FoKRQFFUFktpG8+j255a7JtgMrKwvTJkYIsVwQsQYiDq9opvzweQ+p9U+/d2mwHVQsHaPSVDNDEKgFtUse8+MD/MO7LzrrcAoMhwqbAuzjbDvEy4tfAF
PYWaq4r46U5hNpNwCw2BbDR5flHidpai+MtKZjzovMGc80WfCdKEDFy/U0vNnJrN0ZJMgavKOxBKd/Xr30O1ztk3MZhZz2pm4qWonQ+DwuJnHRcXAQim8l86
579MNi9ax1vUZugYmDzDWTmGqOqY0RcprArp1aLEcUxrkw24dJz1Cbs+SE95mbjGB0TqyY8XSRkhIWCCc5IS+tjXty7gZ+D+tCdAA+qIVAkDvKKdpmR+posK
PNiBHWs0jFVgQLWz8FwNrzBt0KVIxAP1to4A7Q6DxHE1Q27cTx72vyLTAmISjA0MarUR2WBaePnhoJ6w9Mw9oHuERE91lHsmJiMIVRSEFzpQpImkvLRhSr6a
GkwOCQMJrewH4SLipt5TGybByjfiZ1ysWOy9k2LuHSmSOljsa911CrlSMIkUz0ovAyY9kos2C5yKcSvA7HkgQCP6DT1A2PcmMSAwCLr48C5iQSGjmVlafoWi
AkH0UkBtBvFhAs0hYfvzLfDnkBofyUvTz7fBqFj73UI4IB8ULs7uGsm0ZGrLa7UMjgNEGrLKq6nfDTDQ0YmJbOCQdY0bwHwOovSQ3pi+4x/GcTJM0QzM8IJo
e/8wqHx3+kil4t0YjUlCiENYORuBNoo0H7YkigHdO/SqqFB0+NrJeQXl7bMhcB7ESoCSLIIx04Rh51MbwiAaHgTLY8bCFw90c4WDPQoXMixzh03yGD7yeTim
Ww/MtF1d5Ud13MK6lo6hYyIBEgRbwogkOPFQOyBp7WKwnE7qpUFkAhIEikhGA8yDzO4o9hDBX2Y28ifUdiP5gLHebheB3S0XjJFkVgQhESQBhFDwO8D5TgNK
ehRpBYvR8gapll0L5m0gvBK0LuqqXHDtFpeUo/ViL7K/L9cI6ChPewp/u48Pqq8xGJQEgXGHUXssZw+g4KWCE+4Mej2eRRqBigXgqz64F4IGxhgSBjgAQcrA
VSFOq/D4L8e9PSRh8gcS3WkqKnVQr0SvudMU4gVFR8qBUBIiDH2ygYCBkBkp9OHwOIhdyeHCnMmmoUdgZ4BAOXZCt54IcSb2wSMKeoMyIOFBGZYYRFMZiKVJ
RYmIzLKJZZQsAwGQKMjJaqijPxNDb2tU24T9P24ZEfz/c9gNlyuKhZsH1+soIQ41Xs6XSyEEIAwNPooy0j9PTVJCUZphpTU2IwhQJyy7BH8l4jHhxyyYbZih
qQu+7ioUJZPukhvNLpmdSM5zwsV01fzlHC54tSSZTOAog2uphqDIyMg66x8UIY4WFgHpUkI5rwgynPJXJ8oo+UH2QFuPecCjp0c5YKJKgSQh33B4PIDzeKcu
yY2oIKzDtYRDCC3QTAULfgyCy2RTxwjGHst657NAGd28gfIkBrSREhDXKWUiwgbfOBIDVr0QoLUv2EA6fImQD9k/oeC6gk8on0wZMoH2QJ6Upn0MIDHGG2di
xn31UOPwNgdYSKEkiEAGBi5el/am84YYcDwFBzOGMZ8dTdEYwhoQiuyIqvWMx2ZgQbkTeuWWv26bczkZjBUIoCZOcFxINQbg1M5XW7kPYdPID3NywdRmDLY2
DAdGYLZwYIyRIoRRIZocu9ybpbW9eC2RsdNZiUstJeYzy5Sy70XU5htB55TU5G+TZNsVCKQ6JeiU9LZhZrPlwNG+mG0ON0wyZO68YHJnCyTGCk4RjDdomtZt
0hjMemfdW0IECxEgI0aK0RkxK0sAzKUYc0FCeeF3hYZHEcpzs1BMMrly6PN4eO6mk3VsEUFCi8c8UphTU2V7Fksyw7YrmwzvA78IbZtEpsRYxUQhU4KTSGmQ
vDa2rgDuBciJqBhLSYNEBHr3Id3FJRoPkrXMYROjLKglQIZJcXRCAJQG3sInTqZ3dpabsodVZIqIowZBUSBgyPmDy0s0BqxRoTiQ4hxpIGpqmoxRI9A6Pg9+
dbsE8bUlTK15Z47cAPRWBkc3P/LyNMAQ+MkRYQhF5EVYwRNRItdzqiFpEYMjEkYgIsOZYAm4IsDcLIEA4Gvp2w+D3MT3efey6TwiWIlGNeSfpQKQ2gH1CFZN
WmFHP+zBZcehYDQ68AetodZv5OtoKzqHKRsdIgZCLsEGCoXfMCKuMlBAowZJCoQihIUZaygDVZArIYMIMC5COEDMAE0QLS0ElVW0oEIJibikYKxKhRnsQye6
8fPB501DVeO4TBBuhtfj6eaAhIUHHCRQ7mFZEX1iC9vZ2GOA+7FGgA8KQb/F61H5PISQiUilvhekHybdoblR+x5AHV1B7wSEAy2PBAjEzQvXx3QsqGKzAAfY
fvFJNw38TBgHntYKRUYCRQjIHkU0rgrg9EhoZJNIS6h9n9Dm1hsTD24PG78qbVzVeEHJ9zX07G1Q6vpew11kp1SQldWd3w11TVPgE4kQvS6LH79UoSXhJRn0
ethgr8aQokRObQ4DykkElwIQAqKG7xS8XNTz+arjFJIm326cTvU+QeZl0jQxA0E4cSTlVPFEIRYBCijfc34PwYNuOqXiVBkSms0sfSHroeXPU0dZqHC3rxQT
4MhEA5B+uKoeB+95ew/q+ap6dTzHdRfQDgj2dtAFFKlCJygD6reLfPO62FNIgcXGmUhBPmjcQGARjWJBDJ2kGwdKcPevSNudyJki38GVgpaQXvnDFDP9px+I
4jexhsycLBfBs0dOB0ACHUT0ovN7wihti/Lt5TT7nDY3vAz8nQ7tDhE6PJ9ApROww0fk+2/hPQdGjEkkhBkZoeNW6TqcAxBICZTrrnw5uJxOqnNBxb7lphMz
hWpVqsWZGPnPiR2hKmKb04eJbKOXGxczLTBDuzbHrmpwi0caKhMtMMTDFVMEG7KSMHwDQFx6HuS0FIQAClPd5U8vV75IYJUunB2+Tq9c3R70Dwh0BBigKDYl
kIIqQiIxAUiwiwTyZfJnBlAnglzDOrJLDVgwDWEKjzESyiWlguimS0sLYDRcSFULQwgQE6FF1Q+eCPiEdADuHfN5Bf+IOGYahIvpjpyGohramabbbw4ABNN4
qEWDFNxnX1qFhKvpQWkbMKMIDa9UkSGYdbvnDa58j4B6p8ku4TFinEqpIRYsBjoHSBCKEOg+oRvjq7B/BBBnTtAI9nmGO7iZWC8nkOb0+4N5laLGNVwRTZtb
f1/Ta8mKP4RCQKMynug7r4lzp3TcZ7bsz4HBRtrfoFFrUGUqJVbdMEJA9NjLIxegBqCBgUUBBPViw3fccgur5UyFRxHqgvtQ/cgbhULjigmyB3ELg2AOA6AB
iqPny6+HWutjkAemaeLi+UAdT2631e3Gyet2VYJgcK+U3vKxCbaKGBxgz5qLQIwk5PWMgBp4kSRT3wE6CfWql7JLdsKqli2sQiDSp0LN2EEvBT/aKWLFAUoQ
v2qetXR188Hz8sdu6tyWg2mgKfr7X8fTKj6b47hpjQ38qlMYUo8H1XGlR87+6Hbe43lj5o0iZda9Kj8+eE07p82zmygRAmabUcyxtB9Kbzk4eo619jadt931
KOnfqXZY44zts9QZ2bKZaUi4glMsbRWKD9LmbRsJvn3B+LNModSIeG3im7xyIohONcSuVtco2wJHS5NJ9llFSYjYyYIIMW7w8beBJH40qMJ5SxlTbPnwnpdd
fNl7szTCipK1nhhp82z4VCq3blO3XzQUmSFddiXBEs7ZITad43UvViled8Py7YMqPQ2GsFUyUgXqaItCYtGUxwmFlowJ4WgUcGqu3oseA6P2bcE7pInp6HtJ
9vtLjVQuDg4I7O5FeZAieDZVJUhuvRytdVjsLG03a0zkU+mcPZc33wpcMPZgXWXONtgueCZLI4vsdjstgEnFtVZ5AUZdcH1w3x8Td2bORudIsIYtbA0daMDC
o1knVQgaJ2PE3OJsaH+i1LC0KiMooaopia9wOkjd3owwKCHlEhbmORWgzCRnoGdzIkxXU0A+EOmp4KhgY7AmsNV/AEdyN4kRMhi/ZiBEudmEJWrKbhpbcdpQ
x3FZMG112NA0LPjtbGe26+zlqBUT3e3Y8Jp4ce9RkPOmleCMEEEIIjKgHp1rD1xlm0hYkMKqEWzTSOQTVA1IPkThZ4SqonZMkJ5mE5zdlaFk6haIxEeWhpEF
idFIEZemwvY7qNvs1J0bvPRUyLJkyHOfWxeyQj770M8oa6DenipnoauJjoTtmKDdA5PGa73w7pqKpjZLs90n7H7X8Jf4plqHu+fmcp8UODib1+FfUmlTqffM
rC8vrsi/k19FZMMJD+c00kb6aLiG1vU5lnat3HnNlXLvgxgQ8cPhKat20ELJFFppNCLixWYiAo+myWWVtL3FmZJzELNUxT4cLRCYzZaOaeXlqQ7MzGTEN45b
wUk7neUh3sNZSTW5ZETSIKaswUQwVjmZzdmQ1uiooOivds0mSYTDKJCI5zG2HOWh2vPD7SgkqH4jMzb3JGIM4TIVwXq7obcEmZIYTBnh7Sw+2jw57o6didgS
6GRQUQe5Y5u+E689qFMCybgGoM0igcWYQ4ceSRQMsMiadQsGxYbXR0SBfQmOiwMmJljas0owKOp6qmKf0kWChjsZGIHkevwfq8E0UHF9Rh8Y5fJ+OrPoawrI
TK21Y4JgRjbHC2ims3jwgtrIBlrpDqD0UDyxEO0A7DgbamQ/c4CFvEyjIePCg00O4FWMYOqRcoAaaR48aUw2Ezjb9+Xs2hYiQofyMA1L4U8JAhOhT08OwXq2
FduaZCGhBhypEMgCv2RqE0v4GSli/NwbVr0u8MPBQpAx+dZ6tADWn0ZXTua3inXfQVLwYRLVRIg0yBE5RcZeQKy152vtJIOJsJKRKp4sLDhC7o5XiUpB4EmW
5y0kkFhEQkz0PQ3HK7wHIActyUel+9a8Wd4gk7psLit4wi7cFFtDYYVaGCco7g6+IKXeOqo91lalzbYMMSq4w64OFMEOt0105vCy0N6jR399rs86xLBUcpmj
iw6sQYcVoc+kvUPr7JDtHqAJMYgZCEFPeuAboppDVSAlQROJuf4FHprt+l0+gAyHeD+rep+ywhXBOq/q/CpVxtFtVi1KtJZTI7wKTNMQuplFXnDajFQzBkOQ
aYGXILYyC1N3zmPLDjw5pbpFY553aCqcwR+pVNfcWBfl4dryQef72IInd44OxJ0RFEUEEIaPVJ7ZIZOGQPZDbt8Pmfx+2YZezYokZCBFTLwUsHdNfnEN6FAH
jWQXi5hE4HHm5rDl3OPHm1eXueQJSRSE84KaghvGH2xizF2xaY+oN0R5xMAJh+p3s1B5h7IseTzk44Zy5vgmtqij7KXhhUOllVNNgsWSygIhQOPU978nideI
B+H9YKYvwMGXjAwx8mhIPHxO/MjlHTIS0Ag+2/TjiTCh18ufxUBZBEBSQvELkVCB3eqB6t0NJCiDLS0sRCVU9I2MROdH9lfAsln9ftjLvKWr9/aoQzadnC6z
BKcmMOU1pDhxNyytilpZGVKUFgyKbLoeEWIZyY7oMUShBAYNM5rd2w+jJtiLTOYBs0YGblxdjUJEiQMpJhV6Ca30MGggZQuMMWBSBShBhcGgKu7/Dn3vnkgF
fIspvCEI0bkPv2XuBi940sWEIRSCNKmnYzU8Xm48cLB1KNxFHaxnEB29H5JkiVGwOtMGYj8Oyk06IRuQJ3EUPnyYJFGPbwteLoGwOREHyYS28h+/f32P93dn
cfikkiCl7nh5N/ZDqhz1/a5DhaKdtlNXS7iSU8mbUwW06fUInog4kyV90c6Xnsb+Yf/B/NUww/+LuSKcKEgZSq7zgA==
"""

source = bz2.decompress(
    base64.b64decode(
        PAYLOAD
    )
)

actual = hashlib.sha256(
    source
).hexdigest()

if actual != EXPECTED_SHA256:
    raise RuntimeError(
        f"Stage27-3B1 V2 payload SHA mismatch: {actual} != {EXPECTED_SHA256}"
    )

OUT.write_bytes(
    source
)

print("Reconstructed :", OUT)
print("Bytes         :", OUT.stat().st_size)
print("SHA256        :", actual)

source_text = source.decode(
    "utf-8"
)

compiled = compile(
    source_text,
    str(OUT),
    "exec",
)

print("[PASS] Stage27-3B1 V2 reconstructed and syntax-verified.")
print("Runtime patch: recovered frozen CICIDS2017 source root only.")
print("Scientific methodology: UNCHANGED.")
print("Scientific boundary: ZERO model inference / ZERO target reopening.")
print("Launching Stage27-3B1 V2...")
print("=" * 100)

exec(
    compiled,
    {
        "__name__": "__main__",
        "__file__": str(OUT),
    },
)


Reconstructed : /kaggle/working/stage27_3b1_behavioral_similarity_RECOVERED_SOURCE_V2.py
Bytes         : 79440
SHA256        : 1e3c548d4feef38dc7fe7045cd73f9627428ed349af0bfe36ac375ad225558ca
[PASS] Stage27-3B1 V2 reconstructed and syntax-verified.
Runtime patch: recovered frozen CICIDS2017 source root only.
Scientific methodology: UNCHANGED.
Scientific boundary: ZERO model inference / ZERO target reopening.
Launching Stage27-3B1 V2...

STAGE27-3B1 :: SCIENTIFIC-PARENT / IMPLEMENTATION-LOCK GATE
Expected parent : 283690948d44345123d838fd47a8764441f491c2
Local HEAD      : 283690948d44345123d838fd47a8764441f491c2
origin/main     : 283690948d44345123d838fd47a8764441f491c2
Git clean       : True
[PASS] Stage27-3B0 implementation lock is the exact remote scientific parent.
[PASS] Target-opening ledger remains 5 / 5 permanently closed.

STAGE27-3B1 :: DESCRIPTOR / SOURCE CONTRACT
Authorized predictor columns: 11
  Flow Duration          <- Flow Duration
  Tot Fwd Pkts           <- Total Fwd 

In [5]:
# STAGE27-3B1 COMMIT/PUSH/REMOTE VERIFY — NO-UPLOAD BOOTSTRAP
# Paste this ENTIRE file into ONE Kaggle code cell.
# ZERO descriptor/similarity/model computation.

from pathlib import Path
import base64
import bz2
import hashlib

OUT = Path("/kaggle/working/stage27_3b1_commit_push_verify.py")
EXPECTED_SHA256 = "93c738d5b5b7dc3b7cdfc54d49055abbac2017c0094f31f5178ca48fdef4a9cf"

PAYLOAD = r"""
QlpoOTFBWSZTWTB1KEUADYj/5X//wAB+///7f////7////5AAASACAAQAGAdXvADfCPr4KKKd7gj1OzuZ549632x7O+2vB0GhoVN9c7Zptgqi7DIoLrBSgqg
ggo9GQB9vtuA7YUvYZe+EoJNGSaaCaZCbRE9kymmp6Jp5IbUGmaZEaDRppmp5RkABoNNIaCZBNRhBPUZI8jU9Q9Q9NRoGgAANAAAAaAaaaFECT1GZJ6hk9Iy
AA0AAAAAAAAAACTSRERMU8jSU801TT1PU0x6oB6mmmTTT1BoA0NANANA9TQaBFFTSQ9PU9IhhMm0nqGTQDQNpMEAMCAAGTamQA0wiSICaAgCaGQFT0ZMmmU0
9QJmoD1PUfqgNA9T1NAeoNAwEVBOQL8rsxAdKyshsxHrlZgIhEiEIv390cs2A1+ZM9msh+3I6ghiFf+utN/esOEAVQ/vZPy8s9FpA47fo+hn3fqRo2WyifSt
jFiMHBp+DN46LNzJOi4dbSmXBkuStAiyOUmYp97mTQafLyAyIn+Pl1OTcwznd1yyURTuYZyUKWhNMiiRQWCxSJMSjLCBlhRY9DJZj3maEITu64NoZcZkpDwQ
rO59Fl8/9TfQAYaWCPb3pnrQ3sj1O87+OTkOJr7+swdTJJmYNaGcNgYiQ7yxgKIoLL1loWzWQyTEp58XY7A4BDMOMRPu/V/i/R5PXW2Avfjqgcp41CXnwgYV
7ZSpLAhgVYRZdgZTY2HW97GPuNagSEtAkQkKF2i42pXOKJ+7nb1XAxCMCQZBkYm+GmqKKAUqgYRRB/5+3PT9lVMAQa87BZRGGuOFrijMM7CBYxkO3C6tXpCe
OtJ15wD0d93fXOAeLPqoWqTFQUtZBsIowuBsD+1Qq2JU3TBp/w9ToZVFLal9SZBDxigpD43EdREVFCjE5bf8PJENWpS+Hg8PLzZjMZs4/m9BQppGIwyIgagc
25i5iCUMDPEdhZQ4ls41JepaBEAAxUqGiAMmWo1uG20C5oCuZ1mVSHVG4UROom5qBcsnXemcRZilqwqohTV6uzvGftVdeuU71hvfssS0NUcCYAuZS7mBY1Mi
N2dpufzw3wgHCNbDwd8b23j8MfZwR6Vbj4qaMIRJOHMJAYREKDCLHBgEjDky+sq5WNbuzLzbhsAiMdMpmQrJ4emAHfvUSZm5217GSBBzmii1pdIeBEZxFgod
15DaEQJhLdzrRtRMY3tGbo9spzpxqjiLi1jASEiIZjPRXpNeuQj7E25eXaiFGZljbohpW9bnN1B881MQ4q43WbFCwo7I0BJpMigcwgH8DcwzZhjvz2OxgVVo
Tf4Qr293IM1kdEgPkunJhSsuq/+MJxGhfunuQsa2QZYx5GpeLRykugAkEAmAg6DxcKclanqjPcSMmw11XWQLAUGBSNMhKghUh8llGEuMlfs7rfpc1Ipb/MGH
hwRwhw0JaOupfv6QSuOGmO5pCl/+bVRKDUUJIgAoDIbB66LhvS7sGKPhBCPP5lDbdlrPFD9HnbZhmkAhIIcVGSScYA2WGMdyHG97nCwnwaZ6pXxAntR07ykk
sBWg1BuiNyKDITblxykCkmab3UkqaSVV1QhXU0yahU7T0bsUCYqCi0SwPWV3Zg0SSsdne5cExkEEgtJ5Pr4BKrnzEuPuCEDmWHc61hkK3ic11mE87n9am7rg
8vQQdwvForYx5YsHdilEfL5WhLHXbjwn2HYGX28fz4Kp/JPQ8fvXTxpH5wQjHw7nw511nFZGLPkNccePlho6CnJcZigi1X36ubO6GSq6twVSyxo26PE6ZA36
HyYPCqC4UNc73vVcXwN+SxDjVO9f0ce09cN5wZQudfkjqWTUY2SDuZpS14ZdwZ9qbGWDT4JQ7DtnmgUyFMp1uN7jgtTBJJzprjdKjdkKse+axTs0e2Xgwkwg
iRN0cdEvBhAD6zMfSigwrCqjC1PpVd0ZdfPnjr23mLYMQNsMFxq1C5S9zru1ikFYKzc1P3jRR48OQznZvDBFFIio61oORNu7tklOdlcNjgVqKsGDxrjNaaDM
6/Cdvz/D560b948C7onx/j6WXOmmsT3E5BuOJTyRkdy9CBn5oxZuLEg+NMok5gbnmDZjsuhszMKttYroDiUkw4LvUyKFTDuaSSd3d3oRIxirEJCssJeFVkh6
eocTQZ4TiBZQqSRJDkSEHoukkkkKq1yQ/LUMLYp+sVPpcI7wzf8OXJPA1wg9XsManTMsnlpOiFPkUIpFRVPloFQVg7AG6OPXm7dyHOgzAT7eYfU1AhZIkB1A
xxuvWSGBRiCurjKw2vwyCyY1SjvFZyeIBekyeoiS1xRrjma3PLDJOMgHVUNAcu0i264QJRAElU4cEmockpQXJgO+RJs6FWCsUzvtOjMTMwcoNGwWKdrtJjBX
5XeUYiaTMvGonvfFnN0cu2nl4UhfxtYHkXxZz2qmN3eAuNCEoUeyuu4Z5RuL7Y2M/Tr0JVvaM4B7kz6PXGQ9G2hGE8gTGeh7TXoOe+EzOFypXegE8bnUIUHk
Kiv36n3QvwZbcSrHKgoiF44xLBjSFvK+V0HAhaK1iT9Vg0gGBAXjEefv+Vns19levDAD6YyF8aqFdv3FlD+FAfaF3u7JP8Gfu0N3Mq8GtUZMeMRyBdXOiZ6a
KYoxZzaa8M6NGPF6fXc28vfuQrXA74IRI0s1AY4hK0PBFOwwq2IsmsxWS7h2HIYfFC8kKGftVsm2iwxblJDybDRmrCt1XmTRqjlMsXK+ocFJaXH+nHEO0vDh
laYT4cgip0+3w7/92RYyNesLxKktJbbUF5ShJYkSaDaJmK0zLNyfoAGZABtaxGgEkciG2P2WNdCwzGWK2gwGBUrEkcp3EVUOv6wCKsgxmPZMlm4+YfUgP4pv
cJMB+R9R9RR0PmYifKaZmilk27xr8puLJYwkQGHAhSGRS8CQjnG5vOhYxNDE4ljrPgdvy+iqz0orKhvMsg8DT3TxuluUbdjWMxcrmdak+ePlY3laON6zu2NX
ZsM8mxrdsxx07tnyb0r/c82z3vT+HxdHw5UQ55itR2KlEn5354F4UTVz5CBk62ytfPrDNfwLzwLz4ujF0ZIz1JzJN0VWVwtVeO9oL6OYmWQSrg6eVNsyq78G
NNgYWIEYAFSICw+4oqNpCIk+QPmnj7NKpAiCA5pO64gMPiIBXwQLDagI584/iXcXiSq99o4J3Y+xdgy1PQUAkQavZ2G1N5Wo10PK2fS1PergHrEKq0VblGVK
cAx8oAk2s5Ch0t6UHPzD7E1GKBgyMgOkP5frpx45Ot8wwWQLHvIN7t2RidzL7jpjpeAcr5ZZqYuuCfvJ2w7z+vuwveOd5RlHwzmFrIl+9C8y/fYvkvqDXMu6
Lij6/IoA3xvPVFz/alIdodZ15XA4aMtmBjc3U6uQcjtO0j4EHbsXU3B6Cjg9yPJfVmF9v89Juhj+QKE6RYCQ9kCWj3qJWCen34KTzw8UMBvgvf8j3DieIYJy
e5wTN31bUPMMkO72cTibnxA9A6Y7joRiamTVmW7iGJh04dDinSX4AHQJI9j5Ey6eGTZOhgG+jhpxd3A0kv/lrlatHVv0A5AYrG2aOVkvJpla0tTVjYqXxuFi
7iGLcg2i0TBuH0+Fc3VeEjXsYHJPLohjNkBar4z49eRrVoY8SIpRDhK1XZ8rHoZ1LcRdW9TsMJXu6lKwvie9bvrOfiFnf16eV9dTOPtbtv9lAah5IeZEJkB0
9vrDY4aOIXPBeAJdfNCHPuoqqa4B7HB2sSAbtKC9zdTn1MiSBBRt05YwN+AZvI00yeRYmIZTGN7sMmI5AF/9NFCaDnod52QnnPI6T3HWUgxUYqCKjOZyM/Tv
nnJdvYqpNuWWA89TbgKWhimQ8qT9uq3055ms2TqwDQuRKN7Mk4BmWMM0vNcwNLNXhI2xvyB73Tc+5P3hzDi4FODr9HXjfk8iwu2h5anDru0GeNXAJ49pw6yU
0fC1YuO2oFgh7vr+75f6doXK6V0yvt9wF5V/Nn1Zf6zRx9m2J2S9fP6pNdYFQl1ydDKaXahLfev+PvRwT8daf5un8Xu/hwzE+eens+acccHM8vF+zf4H5TsY
eFiKCeuJiGqXm1YVD8gPrp9ZhhuMCdIifapTrSFTehEceic+3t+VDQaF4VAtIIqMGYqRI6IUQ34wV3ZRCggu4tNmqW072piBA0Rk0CSYA5lgVUo6EnaiQPuF
HlUhT9FJR0kAUlWKjkcg5AK6lJhOsXI79XcH9jk4wm6twU/uNeBHB8cmtTYNHKx/ASTZ5/7/r0Jm6xkZBkOm43lzjThgtj82CZptDJTTOA6GJvhDRvFV2dnq
OQ/d6PbyddW9pNll0wJXBVRkGC8mOYwGEVq0TwWD3kJQwEKDs+tkJF+Pgm8GBdLgFvjPmTH+72CHWEwoCuGoZQ2AH8ne/p/QPA6wpAhAxeYo7j2nUHcPBnnx
dErVGb2bw0IWe35jM+wAagbsLsheEiBqZ5Tc0UcyQKTwQ9YBa/XxCo2PWpoPxNYwkjJF6INgwTuTEaTGDowBtTUYsLqgmvxLVvtNEpgXyDoFfHbSqFndEgys
8O+HHEGIjJJszSgxTeYfgSptQsfzGO4sE3EDkdQkdDvLdlWP1pUnp0JJCJDOxqUwJr/Us+3rEP3ZJgh93hm5sMA8P6rXgbio1OXJcj8pZDdcqfQUpUUjqZ7j
jcHdn6J1wAjA9g0cPoC+3fA8VN43j3E1Pw43y+QfeQK4TKU0Zu+HjA0m7GBTgm6g5WofRzhbcGczZG0E4yTtPVqccDYJF2TY9MjdjgbSBISARQcO2gLKNkgh
/QQmEPXQsiowPBGfQaLKFraetETSRloNMHpEqMZ5YSkMIjY/afI9pc+yPV1mlgdQgJqsK2efac34Rw5HZ2JSDfxMzgcMNgTtIm9LnkbDi5OXwyzB0NQ7TnqT
fUxyTqnty2Xs6fCa9Xaf+F3skJo9CuRld2zhx1st3WoYvAW1YDYbURfTzDix2dkLqlNeVogNAEDKmY7bqzheaFjXPTdG2tvHHFl9DAND7VQrQHLTXHS4o2EM
9GrkilmxSwsmFg3cMSSEAjJC5qMChsG+t0Qj6P9ieKA5Cdwk3D1Q56ECuUW5VVsSxxCquj9Zje1ZzGioroHSw6BsoGKiyKGwhqFhEndIwgd8GT8qAp5EpUyl
sStktWhYliCXAfTvyMsCkPD1swuGXH9jGQS32JGQiRRz6dOcYyQ6+ZItepIw2TMA/J8zmZiHbgEgSSHZQnZj7tPd9gXt3hd79VTrSdQpQ9R8BGJ3p7T1msMA
KfABgG9G55gFn2imPUSh6xDbLUm7zr1duB6j3HlV5MsjOjIvRVpyCBpjgYc8MrGJoWcmvfy/jgZKGCTgOhqWGiK1IrIh1aBbpGEB4L3r4Gp3+dyEwnFvU8qK
jKKD9BasAbcAPNDzRDpZ4nWQg3ilMBqAsYtczj5h9RDztVMUy0paLXfc9Qb+YUo9Qkp4ePjgHkeJ9wYKBIbxpxAEJogDCMUKRDfOCGFbFxLFhhaq228ia2In
qEK5xA2fBOuunNIUBGLTRjst9j+tN4BQwUuIkJBid4JVitRZAPngFxDZB3bpMLHrDLFLOiFD3BxnrJIz2SlodaHxtRC8gBELgcmPMu4XvwprryHAcUInE+qy
JaHPClLIzIzbWf0FIBgYGBa3qD2CWjSRa6uwS/d8eOSmg6EGjD0+MjEUigvwFGhT1ULgjSyBOG8F2IJ2vAU1G5QhY5v3YJ1D8EH32CxCwrQYGWgo0fXCRQ3O
9MTygphFCNUQ9Q1YH06GsnbD0gHnonvyeuunUTfD4zk3KJKJYsFF4SIbBu3UZ8OaANEqBBt9G5UCHWBgBBPYNDR/AN8xdn2w0FHf3KcWfiRJOxBnKX5lYErA
KOHj6viu/0DJ1Os2Di7CbSJNkH14nEcQup6k2OkA8Qo6jD2mcTwlHkY2LqUfF0Oszdh9QdqeTX5GgMy2QQplImFLWhUEIRYxhEi6cCfKVvDAxQYPLhQeVAHd
faIenBeTr8/YdhDnpB2Hx7oMRRT2CpJuwJ9IZi+YUnL798mQ+01NQOQeOifcahgPB9XiV1/MfjtvzrvqvIruehk9hwAPAWIdyHWnj6CY9xR3UFpTDuwpW5nK
IZtTpWsLo9facqoeBVrFw9YeMfS3bpf2CXEOt4iYCjvT17AcAmAYIMFaE+eyhzBMwOGBy0RrYJDgOwBllsWNRLSsETstwQG95YiGtBrD7arAZGUoVNalTKUR
KGsnYEKgjGaTEsoCwSFsoJqUMz4qKVBijDKh07PbOz1TIcw7RgMRDtNmTAxnWlhQ6goIg+g+sHJ+IAQwDb2Rk/HUG4BoMZEPEi1FJHchEOTvCiQisxGRwDBA
zTs497C3SCnHPPsjqUUCRLNC7ihjeFEKlBSilRCVVNirR/kvtBLgLAutUr2PH8Zr7M/WL5Cdv6ksgUlqePrIfpQgtB5mP4GwHbsp5fQh7okTk6gd1l+kirAR
IcGkIMRCEgQJ06MlONykcDBYN3v9HAByAwA+H9ogcfI8M4nMi6mfHmHOkJBvOAo3HgbjJ+mBinZRQctPb8GoSDPfN6mgfSZjoHdGkTDBgsvEJcYqVLIKVNAW
1qpwylb6FzKnFPKQ7knbZim59vd2xKJJwgFdR9BCWGzzKb7oSMk8R/J9g2NZgT5VUws3tdE1+P8aA7wTDCQ7w9qdaTzIYOBIW4GI9ITFPmEdxz0EPL4w8Ah4
iFFYrBFiiKqqIiCIqRkFNh7A6epEiJPpzps6c3zTg9NADSVIEjGCQIMc+RYUbK4gSQhBh342E01NezpomOOOJZL8zUxRDFX8pgXXcJvyC6vhJGmOr1L2HUWC
xP5aGjmPJNh8w98CMQfqxMmcJ0liB9kOx+HUBysXMyBJ3V2+zdgFnutVqhL5XvUO8A71KDhpTQQj3QqQj13DvG5dhPNTjmNzyhgZUvvHiGRcdnMUt32FN4hv
VBsBmPyyQcFTgGULBtmme27vfG72ijgcZ0hj4yG/YTgB+oUbcUi3GCFxJAaBYbvuDRNxAhKDMppcFi0QqFUF/Fv11uZ7C3Y4lRQrtZApArOsw6vTTTsXYZow
+jh6Djb6yRNu9OzM2UF7YPBv4LsQ2AuI0ZH5mRNKcDyQREoQCBdGm7YTigMkPaeQ5GLwSZSe53DCKPC55uEn1YWI0b7MOb3482nXcwRoUEiolGe9NzJh0tmy
Fj7zimHTJ7U9j7TAe/s1FHolRYAZGJtAhk1JAo1GDfUfzs6mbjyo+MdXhczSwYhwxp4CGhTvUy0ZmUb7LaxCEPtIbGuhmnc6Yuo7CYqY/WrsDsZcEzxhGSAT
OgaEthIG2SfD4/HRvvz9LOzCasK0092TJsZ/gfzZtvorOHkuWYG3hgZx0JiLLpMOw+Idt/w/CT0DPxfCu7SnaNTdVBjUp+BcH1uRRMS/5rtsDMCsiVYJkMjF
qy0HqbjWjtQKkYXAa+QzG9Ch75AIpJJRkN6ptOnopzq64nGdleaY65mnoaDDIKIwzIQbpuzfLt/OAGCyAaGNPUnWtLveNC7twlDC3viVGRqDTRQDv4DvMKyg
Q3A7VYXCSSCmRkJ80xhmaEYPtODDv9n5r1m8gzDUhIrhz/Yb0yhhkjlk5DqEMjweISIAHlTxujxHrbCggpYMhDtKsWqiOcsFQQ9hhe43Tnt6/Tnbvwq9/dlr
JCSGjsm9HkRdA3FIW8Cit28o1dAlPxJ8I3NApcqGXVNTUwsiecnjwTZDu2NAtAnChWKKnjCeEtNzp36FgxOwTN/DAH9HniCWAuo9pdghqfGvFuL/SKPgVhof
0Fgo5/nGDZILwfaP/v9PaPDKgkhHAMUfUJdPlPHYgeQz4kAsc/ftXWEr2HfKbvCqSIWGcDS5sBsvD35enpXGH1HhoW58djYP4YkIwIiRRUkh7fM9MK/AwWDH
SZliyjS1aw9+61rLURMomBQSIIJ3+0PObIvtBowgd0AjEGLySBy4/r8Qoi2orpsD3GDlDgSdI1WrniCQe0dDtEFfjm8e9P0IbnVWge1fJ7D/j2GGSn/i7kin
ChIGDqUIoA==
"""

source = bz2.decompress(
    base64.b64decode(
        PAYLOAD
    )
)

actual = hashlib.sha256(
    source
).hexdigest()

if actual != EXPECTED_SHA256:
    raise RuntimeError(
        f"Stage27-3B1 closure payload SHA mismatch: {actual} != {EXPECTED_SHA256}"
    )

OUT.write_bytes(
    source
)

print("Reconstructed :", OUT)
print("Bytes         :", OUT.stat().st_size)
print("SHA256        :", actual)

source_text = source.decode(
    "utf-8"
)

compiled = compile(
    source_text,
    str(OUT),
    "exec",
)

print("[PASS] Stage27-3B1 closure script reconstructed and syntax-verified.")
print("Scientific actions: ZERO descriptor computation / ZERO inference / ZERO reopening.")
print("Launching Stage27-3B1 closure...")
print("=" * 100)

exec(
    compiled,
    {
        "__name__": "__main__",
        "__file__": str(OUT),
    },
)


Reconstructed : /kaggle/working/stage27_3b1_commit_push_verify.py
Bytes         : 26082
SHA256        : 93c738d5b5b7dc3b7cdfc54d49055abbac2017c0094f31f5178ca48fdef4a9cf
[PASS] Stage27-3B1 closure script reconstructed and syntax-verified.
Scientific actions: ZERO descriptor computation / ZERO inference / ZERO reopening.
Launching Stage27-3B1 closure...

STAGE27-3B1 :: PRE-COMMIT SCIENTIFIC-PARENT GATE
Expected parent : 283690948d44345123d838fd47a8764441f491c2
Local HEAD      : 283690948d44345123d838fd47a8764441f491c2
origin/main     : 283690948d44345123d838fd47a8764441f491c2
[PASS] Scientific parent unchanged.

STAGE27-3B1 :: EXACT STAGED UNIVERSE
Expected staged artifacts: 8
Actual staged artifacts  : 8
[PASS] Exactly 8 Stage27-3B1 artifacts staged; no unrelated changes.

STAGE27-3B1 :: LOCAL FREEZE / HASH GATE
Required artifacts       : 8
Non-self artifact hashes : 7
[PASS] All frozen Stage27-3B1 hashes verified.

STAGE27-3B1 :: SCIENTIFIC BOUNDARY GATE
Membership arrays verified : 20

In [6]:
# STAGE27-4A FINAL SYNTHESIS — NO-UPLOAD BOOTSTRAP
# Paste this ENTIRE file into ONE Kaggle code cell and run it.
# Reporting only: ZERO raw predictors / ZERO model inference / ZERO target reopening.

from pathlib import Path
import base64
import bz2
import hashlib

OUT = Path("/kaggle/working/stage27_4a_final_synthesis.py")
EXPECTED_SHA256 = "245a06fdcd1a86fcdef876d22d570af42274584e01c6e714c91f6e7035074ef1"

PAYLOAD = r"""
QlpoOTFBWSZTWdnHCXQAGB3/5X//ZEZ6////P+/f/v////pAAAyACAAQAGAvnvAD6Avi6rvu9vgCCIlDWfZrj7ca++7rbG9XH3tvvH3s++8dQ+UyemHrSuze
Qenux671urcTTlZuccuhowLvHz299e9qpd3H3q+uDWvq1tUddcrMYSBUrLbu0NHe9le9oLm+6LXazPecuaetKMgx9bu+hKJoTQARojJhTyNBT01PJPU08lPS
D0jQDRp6mQZPU2kGgAGmgIEQImFNBPTSPKepsoA0MgAAAABoBoABIhFNKemow0ifqT9TaTJMnqemp5NI09RiaBoBoaBoGQBoA0EmkkIE0QmEFNgJMTEwmhoa
aBo0DRoAMgNBiAESghBT00aZAENU8p5pQ9TGj1I9TQ2iYEaaaaZG1DIaaaA0BEkQCaJpom0Ro0ExR6TQp+lNNMmyTymgNBoAAaAADaCAvcgqi/PydI/XAKQ/
zKQKRD4YsgfhfrZYZdjfRm1NDMVYKiq/hpT0wDrk2/aobRn7zCiTrkJYPErIyaUEriBP9fxXbZsCXan02QmuKQvf54B9H+ZqMVtJ4ZmAYSsUyMJVymTGNRk/
IZxn4b+Dl+PedEOFGN3xyEU8mDd9Y6NvwJh242h54MZ5QL96TjfhPFqpAVVIcvWL15bA9slmeiRUhxM5U0LHQWeJz35W/jDeex4Y3Yso37/XU103Zvg6pqac
ozKOUbo0sIa11uTZQXWqe1lXNpgYFVqwLA0hKaxhmFusAMGROgdzKDz/F5smwn+flQHfMx6cMIDdP+rLJHZhXShI0cZGIY3JFYCdwYJn2duvd42mh805QbZk
wHlYaw0U1aRNGt8NCMmyUGlEYwUikQKkYF44MgaSKJe7gnSJUTnxvjNCxMFiZxOZR3utpQuascBpqYiQ3y1XMooeZiJctYIwzCiJi24wkomuApCO1weNihq2
MyyYEzclhMJlshWBtuHQYRZn8hMJYak68ZlqJEyIRhCFZPqHK6uKocJQmbDIhCeDRKgBGFqSatUqIUS4y9l2zdv49fW0L8ZDdqMHh/BTMXyeD7tABteBqyOf
eNqtf5toA0P1uFo1H3OpmNuvHgb+50UmxtBqMxybDCgdTD0TkmBIo3GdtUdcCxhVF7VSUez8HdQu3hM9KW4nfABrqy8FkHn+nt138cMrBZYkUihhFCiEJCEg
MIhGJECzuAov4RUVACyCDniIrM1+vm4a2wUUCXq1FFLUdMrIuAra1AvdalQZEQkEFNDwsQIWbueaUPpoB/P6jfyKdkrbR7dz6VN/durq5pXEKe1+WByv58O1
8A1hvQMg/gB3Kd/rd/4hRyAfsp7I2YDWAbp7tiwmE/TsnwcxzdXHwn1HS9AOzH4T0p6nKsdYBYapCiIzamka4aB37xiWaSRqSw2bNqp/HDB7QG8YL96umgQj
toorh1FHFx4aumMLkBO422w9xE7bGTCo5W8ArzSmzA8mUDCR1sc/jEq9+D1dtt7ZnKrju3MrnUPMJV/YlNuljyuZVdWkwcVXLrEA6M2nt7N16nzTBcl1sOwq
F7P0GZuk3H+8N8MRbFrt8GsGtL/fQjF7oHUTcZQg/xjK3quaHGuFYnrqD9t3ImXYU4Lv9LLGaD8qdlb5ZOrZniZVmXKdeaeuAyx4BpBR8+qMbsNLjkT7HdvK
9Euk+bQ2R25eauGSmkeDaF0ZCojn7TfXRciUVwEEjAnpytGpXRwn97NJd088/sjduWgOK7cuZzwZXm+Hohczq5VVfTViX/vdbbaUoKfRXvqnWPKZZzkEFVD3
Wfk5/JzN3czMv8Fcu81qzuBdbVM/XlJD0d1outtKtZsKHr2kbaqY1lNkkzpRhh4GQbYE0Cbr+nsrwOaFxHgjsdMihtHC+N596KDQx4ihTTtuGGPgR8nmLkHW
a0+jw4bttcnDDjsMlxJwG5htQVcR1tcFOq1O1C6OnnIVvKXtx0SfCoWBaH+Vl1xz5IC7CeICz0dLqI6080cu6augsBwGmwqKnDGylu/lrvM69IXFxftPWOuk
w6Ubvyzwye5C9NDu6OzZfC0zz8Vxcp9V97/H+scakNaNIe7jQ8Xv87xHQHPmusdUK2uBisltmXShOGWeM3h4+M6UTeVN2QbUt9y/A6R08iRtmdE5e/DQmKYy
i7AnNaxHBMKVec43AGze3eqoOHfvWTAgc+rnwwJB9iRReo93FpfDdECoCCXTQx5ofkANE2kCK5FhBfMv1a4po6kPhbm2fJQfU303xfZ1CpvKVoe+HS0R2Jqw
tbbmN4S3wvN3QON+1s2YuqJkLMzqehsl0ZyfkatothlycZCdVgsMal6yS0BklS5JOOUnKcKZ2a6ujIDqBnw0dl/ts2JWCZFpe4cO0D6bJumhWV6COs9Td+ns
NkSG9HdPTPHcvXYG61SNVSMDlMMsGS5Ecmy2efoqrmvAPQJbxkr2TVK2JG8vhZgXiq628PrdFTqRZkjGhnPVYvWmxHG/WvOxO/GOpaJ1TUyUYz1TypXDPGuj
McuGq6gkrvuW22a/g7nM/8BRcla8CEdWOhlex1F191MKTYla50Thb6FViNF9lhyxK1mFy7s29mw7zQj2ely8AvYDKwDfcMDkLoYhsTsZ8+xlYZtOLxpTInmn
9s5FVDNnJyRSQEoSYUUVH39VR8cePG97C5HyCMPruuNiHE3zJHpnzlSGInpaH83uGWAqASCQc6zQsuyJwGiMn6hRYKwgHvMoGl5VTWFYzX54N+h/efmd1Ale
i/R2Dx6Hxgd3Uo6d5cogR+Hh4cmPl09AhD4XEYWsk0F6KfTnfhguRlwNHJ+UG59Zn59vdGUsx3jhl4Dzvr5tN1phnOu+/FcLvRjvttt6sgRxD39RkpIPhPYK
V/IGRY9e7JbjIspCjtCA4BhYuaEAdoeq3cfotqoInKVhhfHPB3eqvRkmXoQzfO5x65G28CjLp2K2c0w1GtUoystlgjku8rr4EvTFmz13GdQwZrcTIzN/X371
bxQK6HfnXo+8LGN4FTikqrkBYKtJDpkR6DxLBg3YvH4Nt2m8UF74qxnHPV2hLtV2cLDHc0UZqO1MQwVB2IjL2cVhmz9PHs2zbVFqLLrGUZrjKpipfFyxmyCI
w8iwIVaWE6NQgMBItRVHDzEX927zJMBplALXFxQnE2YRUQ7uFU2UXQowGBZaIcnrFs2eHZ7s9BLNZiSUFZy0YDMyNZjLGWsTIUkxI8znOSrxrnxwC503J12O
judCThtqqvQKg2cYeTe2HZqd4FIoVPBenghsDWbBVlg4GeGlT41BPNGQLUKST0aAUoMnhH1vl9Lr3nuGJ4/cUWIHmew+D3CnuR+OtOM1msCy3wr2/OOCbh6L
wAzSbnSUyb/Eo2dim5toptsSesKJC4aXE2T2wzgRibpgrKssMe4R8OYjhfuQBN/ic5uD2ux259N8T3oDnzCEvCHxSaxVWy1bKbwD04DQHEE/o05QRyRiOeup
QQgxyUXxkeWyhN0N4c+kSHCw49Yc8WNRuwUEMAGsuAk4TPNzzjgpU5c/bUFBfjPsvs7M9TIuMPoCT3NWaSz3PTe/q6nDDhh2ZBQx3dZdC/d0GW8zduhismJA
2TKb2TLaUtTampALl43hCi7nW+DXkh9c87szFNYm9cznAhnUMAltVNmdxb6/u2ddFvFfyYLNIC7KSRUB8DAn8EHn4j2wZBCj0QdbUDwkk8aomGLI3H0n2rM6
lg4UKCt5Gs/WcbkKlNiGebIqjCsgB3Hn5SG9o7wvsLgfkCL6uwWupeBpPHA7aSIO7qCCnzyMXVwR6vFmFyEJKIgndxl+td6fYR4llWvAWJKyGH8cyWAe60xR
QuITg+JHSr0m5pAjIhMls3Z1GlI8T/fj6pWMsLBonmPViVJHMp29f0ateIAu9Mtl7Egd0kXHmRhFIxIZBoIkk9y8wxs4dO12rqJWAO7hgMAo9qQOxnD48q5F
jrHLx1ROAS0Q0Hm440PfvyTbDAtMb88rCxetk8pfB67oA7wQ157BWNmg82qKB0BYHm/KevJJ6X4xx8w9J8O+vHLvHEN8shXkQVUzuHaSZK1QUU0ivreJNEPi
KqEiOTIE88giull+464voPV4HrugLQvkblt2rCperphx0Xws8s2+vp6eV1NMOJbK14HJG6vUzxSQluMnwWB4X7t/u/XWpxnLGpCGs8A/aPpgYdhSI7gWKcFC
UNl849eURRQCkeHyj43TNx7babDCCT6NoTYRzrT50D7eZjYJfgT0c5aBPEJEyVVEhxIhNCJM/oMxP5z3VARURUTzuFjt83vFmnMl5b2VgX7ubr+NyJ1qYSW1
+OzzXEwDWUWbXQOX9iPzxhr+HFFxkkMxO57HvHzhJvG6Rdtgu2w7rS1Wtw29N6DifoKxfUdPxvJ0JvekqUQPTxuangfzCHA2f1kUIjBJASEFf1Fte08FTUbJ
mblH5jEdXKxHVhyxy/UyNraz4+wrZRzX9BndP5bgFCiB5xFAokPB1Dk9bCA5Ik5GO4DTpmhwE1UGQngUdRiGsDNgITsPcWQOKJAA7QjHErIDfKvxChEyoIO2
F0/Yd+1mPIL2/A8s3hnsVvd7JbZ5Zt07SmrSvzuSiMC1S08ugdhLsjsrn9n1e36Pyez2a/b9c+KCP+pzN8jDvhdSvtWin5frogh+n1GjWc8BFzbnF9Lk9QbX
XKGluE3hXSVQ1nkZBIFoc6u5h4kWf9NY5sdPGtejn1DiZatWQPA3+bP7Xe3OBb82R6E9Y+kopCkaSQJAIoEYCPEadRtNb+QjY0HthCgYXJh7ppIHCkECsEAf
PHrHiAKySVJCZt6vkOpeRzfF5b+UHj4fL7AD+gKUecM8A/s/VI+D9xEsg3y1Q98fCH0NgLGYDKofSPm6NDoCfF2vlPjCBdPkaiYmi7Dvr6obGbEzU1aaQDUu
Q3qg2kSYJuOiRCENMslO9sp+71BWJcr7L04j5+/5JTBxmWwc7UyLhoz/03RAjQjkvmCuWzcJdtzLUuFcDfGo6IYslkvUutlJejfMtEw2576eSUwYLXCcg5KI
r04nJ57nc9TLwrET4YYcLIYZ66XkO0BEzluoCe/H44lteDnnTtfrrany7aGPPcgdsyRK4uLkv1jQTJmIF2zSY5hncosBiEpoHpd02McDZzyJrq4IdCGOJShf
T3lJNTVkTG/gOucdZ1tXGXTILmwuC0AbnGky95WXaHVfSj/AQ/ycvIN9uh0gd7XgfmEdWl7g3snE7TwwC78f8ORknYODofLFdxV7RCmrOKVYRxZkyBM6eMlL
lIkkJXDqj+4Pig+QbGXWnWBDS+g9IiSLYtKALHWhzkGtr3czabXjtXkYlGg5BzQtVwXnWJBgwsZryum2wTNNP5yz7idfHylZCbRD0O/jCBsiYL6RK6UczCQS
Z4EGydru9B7zYThjENy8C9aYbMg25OWWMZOClWo9GHzWUyC+ppkP1AKNyRONyBmR10SGwVgdqU7ImScAaDpDxDbqU5kTjDhyqxQWaDSXuFxkS7R1jhgLcuuB
aJZsWwakGD/AWhaElkwcHiZ9EGvArrcoU1ulIRdW2x1PWncPMWPB6BVk7l1QwPVDIxz0XUZwO0oQssGESuak0LX6GA3PR6qvBeVqTAMSAGI5wY7xUToRBUAL
TeBtQ5B84ZDN0maIBSKlRXOMaJKTWxYakj69TiB8sfsZfHh0aTsQxQ2k+iBqpEDjScCvOECMsHKvjrppS5ETlMtDZz5ua6HKcwoGpihtPWvvHDx86dj0NtVe
Y7GxHsN0q570/LiasTJIOzqGskKXlx4BnwHOHONhsFl+FCcQwIhq6wij333GZmatRiqJB2hZMgTbQK921TtVEVuSi6w1g2NcA1xTS+HAyJcsVPEwMyZ3Pj06
SuWQuyRc+anA0TgjMTTWWsX4fqrmmwBgnaZA0bqeO8kdcKsQVREgqDERHv5uR1U+BqXjxS0ZcYM1TLEGpHDp6BKhIoIjhpIMXytRjsBppdpM7eBdANe7EWuw
7undK1yDIjmptALyjsCw66nYnoPYZprlpDDkkVwHmNES4pjVYAQnAO5OARiSefZD5fGi+vPcQ1a0Tk7bVvuTj15t67hLTXoKjViBEjfTpOGj0wId8G9XcWfk
upb6XLIYp4dsNyjyCzJDsSamMHO4KKKtN+E8As5z7X9b9HMpYPZ8OgcweqWxQ5OR1c6Ayy8TY4xLBWpA0aedccp0MF5H3XZOG/37h9qnW0aDTeGwwQzextMQ
+k7IHIj2QtuGZB/eg1bfh7FO1OPbdLnoeX2ChrsEqWYaxxQ2viTyBCQqlrBTio/qT7wT9L/PvmSM0ZITarfFTJv/HgB7PmhxM32cU5rJrNZpTDo+SmGbOOso
VFyyRwwXw+P957KJGT7aKqqxERVVVVYglhD5fsE4eBjabe+ccHaV3wmRpNptNnEWSsSh41tVxioLgUsmPsSWqpqXBLplTFVNVTJx9ccsFqUkqJDKY0MJRzDl
lzJ19O/wLi6J1bQRvj6dxtVVVVRVVYrPz9a+5jf2/GfVOMVfufD0MREWbK7ZQWYrmWTimxTpuazqLP8d8EhbIfOaLXciXr7gGdmAOkxN4DDszHa9WQlzpuh4
/LxdKF8YdPyP3zaXa9BBh0EHjzvzWtbbbfUm/P0w+/PT29oTtx623uHTm7RM36+Sqk7rOD2ETIswjgdj05uy7mTqbMmjxbkeVHH85tlowp5AaF4Rkz5GLY0c
QuVeMJd4U1ETM1mW1xWo278WO0m/jPSah7N9/1BgZI4UVDWqJCjCjCgw5M6pT2N3k8jWuGpddFthCxwNr/Qma49ru9fZ0gcoh0kD3/zrkL7XLok6Ic0VqdsP
Op1NEKKeU1er7fvcTv81KUlBog8FrEnu9fAkkPh69u/ngfKgjN5mNpZayiFjJ19vYGunltboWmYjCGETOK9WxF6qC8MRQMLDJ2+u0LGlhcV8aXdGgO3Rks4H
RAep2df1TZP5BaKR0QO4QvFENw7euUrhbKFdvcMJ3q3OES9USJfkaOlKHQXCh565VDv6mH2iRjwqpGAQgEYhH5un7kRu8jY33Jcx69OZqDcDtemESQA3GgEL
Ihs4RxMg6Duc3Lxl0vEw2msgKeLTLkjGPJNg5VA4XDqo59pSOJYOvg+Mq2+/hCxgSN3l+N6UqLGUQnDAZA/LM9e/rvdqV53NG5k4t3Qjaq8KLWX+P56gdocy
BDctmoWUC+6ylrU7kNj7RxO/xkkkkkPfrzQxGAHYT1GEv4U/kT9n7BpP17dv5ePKzEoWbHDHa5/rIFTIg+h+IsKL8i9MYSJIDD4cMNeQdltRxwLPHyoDSmFO
+d0ei9t5sVrIw4tfdItypGsCjT0z5ykwcxDEBp8KKQ3uVgAw1hy9VCW5TAJ8DrCBMJWtIlIRXASl1sGv52MDkGjh1xLPXfcTtWCxYAcOJ/EekINjjsEICkWb
BnTiKM+acdpnZpOYdcO4ciwRF5iDqXtzmgGa1NmMlMAM6gHUKgPsBzwnJFRsnyDmWDJSehJDGZUowLECs/TSaAwnp+W6YvmVJUsPipHk5kFIx1bUxDMuY0MV
uGPr0am9fisNIundm5IPDRhbAmIPGptBk3J7ZyMw+tKVAxTJoiEBogwSBCVm78XYObfFzyJBqnGy4bbnDdHntscBzQVAFGVGGRPBhYZQNBBONUUmclkb0CtQ
jA1KdxTFZgqwSIUIKRH57X1YuC0cC69GDwLFJPSCWck8wlBKo6tA4DqGc8YW3q0v65jIKMkxkiIJwkz8q61ksnuFELJ0ujfbYWfQFgHYOVMRDMlh2Zkkaq1h
CqdcHcQqo5IfIsbPttQ23rAot5wyhg5wCi6gGa31rMwqR8LNbFp2vPyJwsalA0m4uP4bJoE6EJ3UvhJzo4I60uLEQEPYQElK2ROyvHwsYFg7BsI0GKVsYAQ3
dzEDELFll+ZGQJF6sTLNnQ3FnsiriF0PHRU4AglYa95SFu7JoZFZvGSRth6J3YcSre3LQx7cShrp2sEnqUl2UdDsYu46g2JtvyVGvdlhcsiHuFk0Do7AcJRK
UYwSIa7wOW01uU5zssEaBOoM9XEoOyQ76Sn8YmvETswbwctKBlFXDDBKpQyO0moTzB6EE7QmejcjJJ6T4T9tqtwNiDPzv8nETsdmCxDbwTnEVBPL6+Bi3KHC
e8N29vSLJQNc5QMDQ672LGDC1pX6PmfPVBc/1wO0sLjk9TTaFio1DqxLifDys/ODljkFhfOQq1g8ncv8Mqq71obZbb7/r3223dzIJOGE2SaTxmi444jaqhkZ
XCwFMwYo4IcWBCJxLFPK5TeFziFpN+KagIyQE5MONnqA4jycmv2lfFKkXGImAXQGw8MDAyys61Y3H9hEzUMUWefFSusvx0svvytYhFAuEj0gnbBIBYHjszhk
7qORirwia5AaUdf3dJ0biff2Dz+N7MPae6vGWOCHxwee5cZIyGylO8jK0jpAjnBneubNQNjZIxkfmUKg/daoa1mF88+Jl9r5PsJdXGZkDCGdhrXqWpKdTrBi
nmnaIMgkMJzwGCwX0PVpgkYIqO83EH6JgcTg9ww3V8B9vlB77TrGghE9J727HdrFQ4nCMKhlZToFQR4919vp9F5Hw1Rc7nkiq+gJdT1HXNtoCwvc2LzLypVr
GZ6w70CwFQ+A94FcMdjO+iTe5TkUU3YXTJMiFQhTTOTRdMeAdOcLpoYWfXzwsXDTwKMuEcTagatwb3wL4RhIIcY8PV8+Wdzu9h/EQLcwkEDKAUEWBEHjs7HK
M83bkyDyr2c9LBzNiAKQgdNtIYfBxgzjiINncgD/CeICGIUCODCaZkHxjIgJAoUQzlj0ftHsnWISAG5d7LA+MwgYNCHbl+Bk4nKU7QTvA19ViwUUEc9Trcoe
cNvqLxeBiRpF37cORtx9sfx5bnHi767lScYLQEwyoL2ZBLkdQFChWLhhR8QXQ3zbQRKhHwYLXUPK40YbCYkE8PkmH5KaWEcGjJCeUKhYA8YsM7ehgwTf9DK3
wUDUDVzwC/IHGM+KJKe6iz4soiME4JDvEh5jvb0udhL+ycIWvJ2EMQSo4Bkb7mB18IIZ2oPVBshVUiJhZIwhmAmqAUn8JGRBye/7weZQHie2ilJBQj7JgWfb
nZEA7PmZHhfouTd+k9+TgiNgogSzXPv59/6Wj0rDR2DDAS5d8WajmBZLgSmIYdYHfwOnXhUwKtHmvcpXx+8soNgl1wsySd9HbYbjm7F9eQnbEJGQkUD3ITx4
J8xIp0fwDHtbY8QNCFSiBCqqEXyPo0DMHkI0cPMuBrU/ObNFLhUoRFpNYUml54XlcpgcKOcjZUJiJ+9Ps/Lufae0Pnin2QnZuc6A6FHE1IdhUKyAvAMYa4iZ
XnL3q1QK4cDXAMch0ISq0Cdp1TeI9Qe63l5cIFQX3NgK+vu4uNzQXWZQOzNHSas6E5gcvfaJPGNStRgKqNpUCrJTQi5biSmGSDDBEwslMZEtEkf6Q8fn8vP7
iX2A0PnE0V1PsCFg8PTe8QPEs0yYCEGCCxfXtNxWEtkUlC/PUKgaDW8AfYeFNPqFm5Dg8l6lE3kNcN6wXSYbKLYOB1FDYeJMnM+LgkkuhkHKZIZZL/WIpucA
+XHy4EAcAtChDgGMOAP6YKeTFcMcT600pv7ofcAns77ENSh9juh4abp2jE7qQoAw82BALwBuJfIgM0wk+B5ntEpWYUtDHiZ8iTaPfeg5RYG6EMTGYkSykOYy
cEXkexHEoUMIMnxyh32kPj+s7iwYqAGfKF2FmJ7PQGZ6p554HgcqVKrUpCO256B20mjmfOv8Hw1OHX5UQiUHA1daBChVI7fDHQOY4V2OfIYNgx2Ch6JcNgzV
yvdq3rUxL2jMaoqsbmdoaDI4tCWGrfgQ1kLVtNsphhYXMcLXG8tTWqWGltN8pbkTTmGsPymJLJFDn7mGG2ayQ5BuMdlZUrglspzSNKnNsTQIc6c8TRT3asKQ
4QzYKxZI8cHhDjyZfeQ78pvsKIoxQUEgdpcDnzMDYgMIRwoa2+9dIkEGMy7A+cJYHQQ9osE0GLZEccfjrV0dJPgjWCwBQKRKySdgragPkb9DSQCLhEps2WEB
wBiulM9/spU3vQbdjPC3zPqFIfvjCbpnCJ3cUCuIxvJDkXL01GpTMsxwFjJALAxLJoss+8kn2UcLyCzERq9FUMY6QK+qHrTz+hD+p3e31emMk6DwWmPGRcxd
IHQ75SsQkFTj3wqIeQgWTEPD6ffkhFKJfVyZiGrMePDzvua1DuuQGxxIaun1gEGMUkGWfUX2AjYMf5Sgm2p7weLJ6mFZgDEEUEIhplEOUakFwDkA82MoNcBz
+LCSMk3ix+DdnHVMy3TETrPUKMqAqg5FVQTTT8nmA72O4oDmOsfgx2pISw9Qs1+bX8N74Ht88+vfbGKEgNiVGEOERqWAoYLvAXIww0zJ6AO/cEwTER6YJFjs
Z6dW7t7vXLHuohhACoFNCYFsE8uWRk4hjRkictxP3+Ktjn4P2J8h597GuB74vFDSAyqQDjTYqgsKPMgDX0ezx+zYTl16kpLcSZi5y3tJKLmbFMJBkPOqSCnF
Mj1DWpkKzuxH63jkkJscg0SFjQUWEkemoYqceJ4wg+frU+SJcLIjzChe12p/bFe3YDyPM4bcjZCBuX2fF9Zr6d1HVDVri5nX4ZNjBjfra2kyO/kyBIMkAJqG
MOw5HeKPqBPePA+udpSiJTIOCrOZUPmEMLFmWd5laIgBomNGvaS9kFVCgaEhmyGIu0x8/f1NIHSJwRS3kv2cp3C7Gc2fMjwZFIESJFIvpZQSAsWHakoyexBT
wNaCHVy5gacJSRUYyJxZLiaYYRmpdwkrIXxLHRE30TYHPgBiOIGcv2JxauJiSGiQnLjMwgL3w2lI9DpJGBwk+AC3G14FZGRSYzYcTXNqNqqaKbeYkhamp1W3
KOSAKWQsZMQ6l7AbA+IXfj7fcCfbNzzSk9e12wQPl608fXJtts3Qts5Se4oDBlM1b7ApZz8/4tTctXMJ60PhRmdBePTcDA3huwPVA3GaLI1XCMQ7QxAmpgXe
wTVRvRR1PFF3PUfSLiaHmOvWBIgGjFV0SlPqD/NHhGC+qImAZoUQE1LiSMIWUCl9pt0ntKaJRRVHBNOIHnQ9S76s/MF5+HhiGNoSx7asE+EOWZtDaHvhuqqq
viBsgxgw4sPNegFggTLKubeU44pkrsXan4RGrujIQSFjhbqVgDGIVFIT7VLG1DzIb2PdontlkNTUNJIonhdPphXy80keFq+d7yHsv+XanIGy+KrwVGSx0nKK
fU6VQH9QdlauExnIbGmVhgvHjxmsTfDkVPnb3Dem+Pll45NmDb3yo5voB0YE4GQu6CQmLpY2tEOlM7uBgUuZFHG0gkFw6Sqi/y1AWG4fBkTYJytaFcBbSrqo
1hhouDQGl30uTcUgASgG4bqqhV5ql0rQWwcEC95qhk0dMmQ5nRDfDmHleQDqydMPzb3Sh2j3d5PRLZ15yOhZimgmlhHuCixIdGrEnl0OV0ZnbuRcGQwnQzrT
Nrovm1M38vkmeQEsO869JpXCYAdV0xh2hbJpj3erGc9gmugdhCMZtCSr4ZGB8ufKE7Dd6xJADsO93tHyNpUMQ3ECmt9EV79AMmddEoEQvo4ygaZJ7OhX3NeT
qppNLrCYTc/WPahO3ahRKiiilBRqKeung20fbecj7pzNg3dBxIk8bfTEkzcYvB8jEf4P4aEgaUNe1ErE0Sk748POYGxiYakDLBQIUcEjlSwDoXueiICoQQSy
5WQ+/8f47jyeajZUhaxY0FTaxyBKIzatcM4UpXgp2WBDulU0Qh9yIbIMEgihcS71KsIN6inVG0Q1HFKsPQfBEo9rZKhjhk8gUBSa+k6MNbMhy5CXvlvhhcOp
iVCKKBu0khtextqGFaMR+gBaZnHhnTly5W84WZ3hSyh3FI0sJrASBDYDYfQi4Gw2AI2cAymc9HAkTKpHLQtaGbuJa1gNwMQNTE3INHccxyIsP2oBFF1xZm2A
kMXqwPFalMNw7DBsIbdCdQQyFlYbeUKm7Bs305YkTKb50ZZZLYyIDhmwXJ4OjYOlBkX1xKxglQbh9JJu8aYPpxCznCd0aYbwoK464hItLZ0kFkAta2IH2HWX
LwIXZ57PnviYuqw5vUOPeKYCYTE9xkdzcR34Xz5RZJ6ulArDzCFjBGNoFZglg4t8dz4HJyO5coRdEO5hdDfsJOM1XEOGZWmZJoZGDucQA2NjoRbCnjXNRsuQ
Y14xiIb7J16mr077XlrcTUjTfA7sb0577cUVaTO2ZMsG180SjMf2QaCOSuGitlCcDB5ThrVcKW0SpGVBKSCcvvHLXPyX/39+nMDioGm6F/pOeFBcCEhnAyKD
27dMcFE152Q3t034nbIHFLRbUMwnvCNz749GkEvp9fcBVWxurSiHLVWxqEuVCsR1wm1c+o7NnDu7xbXxYFQkkDiaaJYLveGwGLLl2azij7CfkInjZ8TrLJb3
HoYYffKk5d56Ad8V1kOw2A4iQvlnQWMEM8joZoXJgERrpQgWlmOw9JRPCqH4ROZDK4YH5uSfcY8HRlOQo25azGuBi36N8cAvALmRpw/VCEBMZfEpZdbAmwYe
iYxj5nxT27fqCTbfT499vg2pzS093Jhq64lOSGttshRIU6WjM3LCgxdtHuzAuqxrGnqxahjNyeuhZ19osye0nCXiIGU9JkLKQxuERe5ls27QQKCgclM6GAUN
wWGhgg4rIqmNVQpLHr5zbhOhZ80+cRNTUDp6HK84HcRfFbLCQYDEeRU1D2rFE40i7v77dnOQteBDrhIn7FOmVkNOMwoLtkk9aoiIbrTnZ4TP430T6SIrhe/n
3tqk924VRtBPEdoQ8RHjx9NshyPlRS3keTnjaQPIg/5B5DIbx3zOj/xdyRThQkNnHCXQ
"""

source = bz2.decompress(
    base64.b64decode(
        PAYLOAD
    )
)

actual = hashlib.sha256(
    source
).hexdigest()

if actual != EXPECTED_SHA256:
    raise RuntimeError(
        f"Stage27-4A payload SHA mismatch: {actual} != {EXPECTED_SHA256}"
    )

OUT.write_bytes(
    source
)

print("Reconstructed :", OUT)
print("Bytes         :", OUT.stat().st_size)
print("SHA256        :", actual)

source_text = source.decode(
    "utf-8"
)

compiled = compile(
    source_text,
    str(OUT),
    "exec",
)

print("[PASS] Stage27-4A synthesis script reconstructed and syntax-verified.")
print("Scientific boundary: reporting only; ZERO inference / ZERO target reopening.")
print("Launching Stage27-4A...")
print("=" * 100)

exec(
    compiled,
    {
        "__name__": "__main__",
        "__file__": str(OUT),
    },
)


Reconstructed : /kaggle/working/stage27_4a_final_synthesis.py
Bytes         : 50479
SHA256        : 245a06fdcd1a86fcdef876d22d570af42274584e01c6e714c91f6e7035074ef1
[PASS] Stage27-4A synthesis script reconstructed and syntax-verified.
Scientific boundary: reporting only; ZERO inference / ZERO target reopening.
Launching Stage27-4A...

STAGE27-4A :: SCIENTIFIC-PARENT GATE
Expected parent : 4195be1c4fde26f76a3426728c5dd3b12d389e82
Local HEAD      : 4195be1c4fde26f76a3426728c5dd3b12d389e82
origin/main     : 4195be1c4fde26f76a3426728c5dd3b12d389e82
Git clean       : True
[PASS] Stage27-3B1 is the exact clean scientific parent.

STAGE27-4A :: LOAD FROZEN REPORTING INPUTS
Eligible folds             : 5
Structurally ineligible    : 2
Descriptive-only eligible  : 1
Bootstrap replicates       : 2,000
Similarity status          : SECONDARY_DESCRIPTIVE_ONLY
Target opening ledger      : 5 / 5 CLOSED
[PASS] Final reporting inputs are frozen and internally consistent.

STAGE27-4A :: PRIMARY METRIC S

In [7]:
# STAGE27-4A FINAL SYNTHESIS COMMIT/PUSH/REMOTE VERIFY — NO-UPLOAD BOOTSTRAP
# Paste this ENTIRE file into ONE Kaggle code cell.
# ZERO scientific computation.

from pathlib import Path
import base64
import bz2
import hashlib

OUT = Path("/kaggle/working/stage27_4a_commit_push_verify.py")
EXPECTED_SHA256 = "1931553ff5bdef5cd956957533d7cfd5262943a0fa0edc8695f640a7c4d33e44"

PAYLOAD = r"""
QlpoOTFBWSZTWdqq6HgADMv/5X//4AB+///7f////7////5AAAyACAAQAGAdf3inTrl3NgaJARYaDuyc4m6+9nl688a6ZFMh67Tp7zop1YZhtGsy1UtGkujT
QxKpQEgBzWFbTUrQNe8NNE0gjSeJkEDSn6aU9lTRsU3oo9NR+oTIekeoGjZDSD1AAAEppACE0RlMaSempp+qepk1D1NPUMmnpAABoaGRmo0AGQBwAAA0AABk
aGQAAAAAAyMgAAyBJpIigCZNTySj9FPKek2o09R6gepo0aPUDEHqaNAGgADTQMJFUnk0R5Q0CYA0JkwmTEPQANAE0YARk0wAEwESRAgCaNARkFPCGpNpkmnq
MgNGhk9QHqDQaPUMgGh0KgodyP9311IHw3rT9Mh9VLUGQAJPrwos3Jq/Ome3WB9gjqDMYt/6OnWjf9dJjDbCp/aw68sgcfjdTPx59ylC1snxtR+9QqRYXWYX
dAx0m5onK4dFpRo2LgRWWpbFpD8mmi4JxOAGTC9cNCxcsWx/LV6yoDBG2eKTkQwtmoMMoErILFgJWFGWEMsKKPQZLMekwAO7ptChNiaiYzWTs/P/MlUATEU2
NEPvQRLGhpKcwrNf5azGyBaLoM4bAxEJ3WICQvTLQtmsJgoY3y4ux2BwCGYcYifc+b6Pz+T11/h9zAwrXAic/m6Lnl+eh8CJbxici9YgsJg6naUvtMYQi6b8
U6YVHTERsgbyyFYcoSf3eee9qTbGCEgyBIxeqZugiCrSgBjjQCDh/mejS2J9ML3rKZAgQyhQKl/TYztTcgR3q7dQA5ulX3d7kmmBlJ4u1OnJDv8LZsTM3IjC
SGPx+rzl1cQmTLcLZA2w5NlOQ6SMPtemevIBrSnsSnAiDoID99QEDIQI2slV/n+GIE8pE9ae/z5z8eUZcucdO+IaBBXfEiILQCN0/sBtqPGRcTg+0Ig4fS6K
o+5TmMzZHJlenNTc+J832mUsO5yGbqbltbZzWL7cHmy5rc8ayiqV9jwvH1Kwmq4/tQpOfhnRKYoTdQn6w9FdRRtt5XgT8cuIB0kz3jf09fZ2nw9C8sodnu3t
elPWKMSy6CRP/Z4Mi9tTWrFmLnKtp4p27Svs4OoLG/huuYsaPBjUPem9MQ3z5u9/iExb7M5pbRb71mCOapmdljHEYYpfhu54aaYrFbIkNlXL5TpZb2SL6QY9
du12mpqUSNNDUn3x923o7ntvTBsm3mrtogbKFBfQ2+b0m6FollVTWHdWtMCHLv7EojcQPJMeXX3zH9Do9ihHsvjHRLLMrKm3jODvPn3E7TIBCkZ+Ob7C2M5Q
MikXg4p364WJ07mmaGQkkhCZInOF2yU9IJbdlNgRYtvzOXDh4lsztwcMNtDRtUEts6Tu71gluaThDWEOkrtRg1hygi32sP5XcUU/drWsex2QVcUgkAq5sVll
tnDJoRJ6MV55806KYOQQSSbgxlCSIAKEhbGi+256iyVgR1puzpRllgwvfp9GzznZmSEQOh4LUVpiyfGsU2euNz15O8VAqX6ueGke1qdjb9OaerISNbF4vtlQ
8jzZxSK909261hJXBJ3hDQskm6mrXnTbCamk0zNrsFBMIWOOibJgj26ZopFJO7A99JS0ln8pshYp6Xzyklmp31y2Z6wjJueUQnIhBXu5RUttxs6npm81DmER
g3GkY3IEkieWdmgePck1vJ44szBlvfp5/KA3to5Ty13XperoBJJMfAXsQwKkB0ILyDlyZZpEnjdnG53gEiloGipuqnwlAakGAatOmbXdfAVTkKau7n3IWCnV
BnoUcJRGKXn0AKdGLLhqN7ZvQ0DurkyaHm1AeotQl0Vw4mXPEDWnBGMV8TUsAZVCzXvTE9SprYSn753Uy3z+YahzGAizUPEQIM20PXAnxpA+2U+iIUaLKPRk
/IFvV191Z16pPWhnn3p7Qv71qXsvXlBU1bFBKuT7BhR33dwYa005YRK1SVmcsUKpc1hI86mBmylC2iQc4ucJRXPYdv+/u+WteJhWCKtve23QWRMQSjSKplLl
Ec24WYq4iIdpxfFtvn5hizJ5BQwaWjsGmWeeVHYHEwkyLONaODjfCiqqK70eZvjDk55yy8c0u6cs3wTLgvBzmJt3hQxXY5q1RctG6FPy6l5n2YSHyzTO8OZ9
rm+v3jLLTZM8VPUhTyUgrP06BQdgGjjy3q3cJzoMwZ961iwPuig0LAG0eePmtrNCkIUXliQdDj/TqGVh6IhCStVyBN2k84xkmrCOqzfhqx7M8mPzZqajNnR1
WYq4wsdNlCjfTMnr1nPbx5b11nQwucmtUbGizqdMxH5moxMKOSTk4sjXjCjxKTfz7HizFXkubWVIjg6ZnjJ/BIkHMWkk1qO22eVQjC0GYeCHYi5eFeNp7Y5a
3Z9V3uJQ/bNDwZL/B4hUPDQK0iJqQQZKmeVeA39gJfOtOqXDruiQHs8bY9PcYB4qoAKlSiRVOG6NJMVK0iuVvHGJcMaC3nvldBwIWitYk/DYNC9AYMBeMR5H
p9OPl4ePsMM0bx+xEqCZc/nLKHWKnoDuePp66J+DV6/2tZvPecuZfkZ+AWHjGw+CeA/pEA4wVLQiSu6Ecna7BekxY8mx6OzNmvCbt3FY4MFa2pKIwha0sCIQ
d48zxJkdUHYomRnVuK4qa52ELtnkEJ+jZuSlcsFDDK3927J5RF3zlQ1IjfanMNSKM6QUl3aiZJJ8rPqem/ut4PWGMOq0Q6z2k379nUQLaodbskI7fuqBFCb2
vifoEMNwFX1DI7oq4OGmAoaVBNomk0wmCYk0JNoaaSvSImqWTy/ZGXOnpMEU/EH2TAHESPuLGr3pfApNhQtgQzMLyHIgMxcKbDIiUCIjeeI4eE9Z3w94OO2h
0P9TseFI8yoK1vJXzZYVXUcPdZQjbHU8enEPTg+ys+O9Jd1cYmp6fN5/L19XVs+j0zb4CQJHeUecjwGbxTskG+QBkBRpJZR1uPbBQYxHYHI43mGus/WwVOX0
4rMHRRirrxjDJRQmSWlY7OGWMV3Ubo6aVpe1GZt/udz+kcpN8DJm84CQAgBIMFUPoURRtIRQWfAHvzz79KoAmAED1T24ADkOkiGPnRMztIgu5dK8zU2e0Jbr
EhtJv7yjmOxc+KC3CySBsszWxNlMURFBCHIfgRATt4pyjChdARk1uSZbFYMpRqND6QlP6mPEN50J+EyQ1T4fJThgGInpNqWbEjE0IQ1HI3xCcC+H1GQOvT6Y
7PznPnsdl61EdV+dWzjZzzmN7AnWhYmbRgBRcycQMEciC/mjEi4F/iY6aiuSGOZiBuRADiSwMr3ZicShkbl73ecnsi9C5moOV56REpp4dZ0lAdxvMSd/27Dt
ThD6gJOoUiwEPwgJyZg9aiVCGhTR07EkShsNgLEXrt7jsAxO5MR1BrMA5lUNZncYj3HWdy+Ice/iup4kImhHZUNxEobc9p2MTLvLMZtobxDpk0SBSVZGvRFp
MTMpuWqTSj6H36NMfaBmBRmTQqwUg0SelrSMpaDkQl8i4WLGKQwE+tYBmJcPh20W4upTkleLxTw7DpQynAZ3ZJyX1lu39R3+f8lH9tWhsv0M7vCLXj0CW8Og
9oHcZHnOVgsZWxPJu0PhcDMVA6wKdo4CcIC61t4FK9YeA7Ssm71gxO5eIhguYkMeyiq85uQ54lKa9Kba4UabVkGRIol05Tilh4mprHxOJqkumpCsiWJDck1/
LSiclJ6uoh6TsONz5ClRWQEUWKxiK9mj4ZaoXr8NSd3VCCnYO7EPVmBfqyM12WNacbpoMMyYL6TIyc1jNsJGtyHa4OD3nWmz8wb3E6DJ4fn8+N8ykdmZ5+re
WCJSk1d/mORbcvrsZOvRSgh7ZPbVAe0CfI/M18J4/NAo/GQPZ8kmmkCoTNcCGQUGvSkEASfiwgIrJ0TKnNJyZ/GwZiP1TpuakjGIkSn6Aeb5uEBv1sJAh1MD
KZQZHzp80PuRsN4q/fhmVdrJrW8v6w5chM0T0yLZtFogcuUIKY8oEHcggsvAI8BTSNzfzWIQNsY9okmAGZXct34EV7oScdII72LiLN7t2qEYh6KM5dnJ5DmK
okFrNhmn/AdgKHVnJtmSzaoWPkETFNDk2gznxts2iDxFDXvnOdh1S9GzRmkZyYvZO0+7wcTlixSKeoSzXq3opkZtssWRdiYZhUHYk1yEXZJ6kZMpTeDpJIPv
xTkUlc5hF2pId9N+DJrbW4ZuBwHK1UQpUZxJKrSSKEwg+DTaXxsQp02JtMIEQK2PcQ9hMfwHUD1MMKSiE36gyhwNooHxdz7vrHecVyWyBDKghYRNZ5B0h50/
Lvq2dIzdUlMkwUZvetiM0OxlB5ENef6PzmJiYw+gXAcy4m5XZqm7laBqTwA17Vb1vMTqt64wlqoIeU3LkWR950pAgPE3Ib1wHFewO1pXZ6YQi1xOjeNA+H3Z
p9qta9R9/jaJ2dohE7HGerJOOAYiJJO2GHG1ONVyya6Ejgbikg1uPDKEeWRobJi9Z1GXXVj+Eo2bfRykJCJLXPyGBZgdZ71/FilagkZF+fuzSgI7qBx/VGSD
A6Z1Yb0HoMoG67OYIMw6GPGbDVtP8cRLbJq9ZziGoaPjDX54+COaZR7CaH1dI5N7PjEJi+iqJ2csE2RxQWWJoMZA0Q73QUe3owt8jU0IxOngmCCdpA6dQw2w
QpNE8TYZO0FBQGBIyQWGQQKyH7YgeYcaJ56lVGE+L1RbEZQb8N9I1m0BSeTCrHNNklP7z8UyfoIbYHEZB0YSFGj0m99keo5eky3nxnV3JYB16adx2vbsy3qn
YRNExPMbkxerDTJNQda5dLIsd2BkmyHfUx2inSvXqJd7Bw+UhOExhiRVAkRZDqaEiEaevGSJsu61MgUS72VYu4s09TD2T3o01Fk6tnSFyQ0SAiJIpnm80XV6
KJlC7GaCD9ErNNnlYWKBc9cyPIwzN8ZLMK4TrS15ZCGGIhMBiIKKr2XQ7kpJkZ2iGoaoZ4bIxjAkLoRaDbQ6n1P/KakUydIsnHsJmJ4dIg3uGB2DTuhaMh+a
cVNc3zU8bsWTpBIVUWRSnuUOJIZCyJPUyEPCHssPOA7pyTCojuCVsnkCZi0CAh2IiYowNcqcX4p3PDGzGDFinYynALr5oKIjGCSTk8vX60VgvpTxKyXD22cw
6YHx+0zVOu4SdcCpIlUJ6flDHjzC705i5wHogdoU0PI3JDlKTd20HhSBcnMC6EiDJiTMdIEuDUMiEQ7OeRgeDw65uPCNQmoRlF05gEEcnza1A6SbXJnX5zRr
AEwYW5iesiWOiW2VSkiMeOwdCBtvhFbD/EG9DrJ7DuLPeiaTwD2UVRQfjKHzGVdww8TqiHnU8VE42dx2Mb0eyCWYDaLaC0wl8h8PgD5RCvwDMUV6DpB7/qlK
9UJLPIWJ0tfdPfu7+pTHM4xAsOd3QPiL3C5djK4y+lvRKNWyMjrUTxLRRcm4DDc46MziklkRFYoyWXhnsee2S8dUMVkOsjAb6GAYAAXJEid3NUtkVYSQH1zA
LjMGbA2d0pwOsKOZXAnTDiLBHe2za3YQZ3lLsCy52TrhYwkwIrQGAGjHWXcW99dNfbyHAxQgJvMHE+GDhDjlSFkZiYlPsG/zkFcTExLWYcjuDeL2iXsBRV+Y
l+vhi2DoIWqeR4oMBQX3SinuYFMGVIybtiLrIJsXQbHofvQ1i4CGAdI+pDCwWIWaWqBpbiZiJ8A5prdqYnKEhZJCRqiHwjVgfk0NZPqw7EFRBJO4JvZ6GBYU
F/E8e736WyfSyjhVMO/UyF1KBfQNWqjVuOIotEolGum1tnx6wDZFEomIFu2horS+QQyyYxiGBPYF9IIROr0llkSTmgw6IWJyuFG7Ueq7zOg0DaaIaSJNEMFx
1hiF1OxNZvgdLRvPExzoeco8jGxdSn+UNPjc3PYPaGsnRKnsqohaBUgRYrEZCmUiYW1sKzGWIMGIxEYRXVuJ7ZXI3riY5oU8eFE7oIWDnMIEDqsSy1mfo1nm
jqUzNr5XpYQhCST1kkJLnuEXgQkQlPCACdtSNzFLDPTS+RGRgVQWFMigbEMB3HmJ08R5a9mVdtV5iut5HIc2dBuF7S0LJYe3pTv8k5mNjNqwc70QuhczJQZh
0U2Hp8jhVDuKtYuHiJ3+i2XPQTHyB07jqd5vIyJMhE6hoDfZC6uSPxbgTFE38MAz3ocCBOCHUAhYxtFjUYiwR6rKYyNe8RMhi59mQYylLvUxMpRShTtCG0EY
zSYllkN2wYC5ZQpuUKssZQJIDppvdOhtkJpqwLNm0Y1SaBoEgQgSJg9KsCg2GvvjJ/JqTBTMYr6RgTqAYHWRBAHYKY7AoBzYh6NvWwtwgG2Cb6L6ZhqIiVAN
UHWWSN4aQo4FIQlZIzEpoZPnAMmkhsFL2sLROh3fbZ79nm6XTwBr8CUgUEe7wJ/MRjIEEoOJ0z6DanTsU6PMh7xPLc60fdCSQgjAPGc7SCKMYw8sBpxosJvU
iPiOs+hwQcwMAM31h+7A33EO7zzWkHWQek85bHj0QCQOUCEdoiUbzYYvwhihw1+jY1CQZ6psUzDJPliaOoDmJNpmUeBtBO8QbB3obNXN1oa7hx9rsDpkSRK7
ja4vUAG18emLv1gVrPcQlLzLG2EgSQ7R9/yjbVMCe2jCm9romns2CHkikPPBTbifjLObnc7hMgOtJ5MOBg2owAu+4I4GbDqIvoDgj4oBUUYiojFRVVRgjEVi
CqoixRkETYe0E36vPpEiMPkPTE1aV5JuDloLIGkSowQkIRjwIUIkFwAkhCLDfhZTPN5dpcvcKczrNDzLiibVD5DExDfkJ4DonG2vAMcgOhYkqVTG5yOto6i3
O1EP3KTpQ5JsH0D6oCHwzMjf1WLED5Ydj6e542LmZEk767vRrwS3fQWtVQl8r3qHch2owpN2hTTqpoOyFEbh127XoowJO9DhYy9ZXYUHEMDAdu5UtDssu4Q7
nQQAuBoPsyQxEd4ZwstO7MO176ewROCmCmB3whHJ1P2hErWETWNxC1Itbg/TYR1JyIMidEe2xWWUZDJi7vq9fn2bcvTXbjqsc2mJ/Tcd6PCKjEg2FJ3DxWak
iyMST4SmEY0OhoTHUTSEouzFhRIE5VhIWF0ogTn6VD5INNbNnV2CjsuszNlpcrve8iH0/plng+EOkvbpMlUoxbvx1jzc240qbUV46vZIfR20el3IupshDgHQ
4N2xDWw6Chrlgc3xD0eNoEUvimXZuoROi0YWwqw2FxyNmYVLSTVc1jBvoP22ai17WPNR6Y6nS5klgxdsxj1EdiGObkQmDcE5m27kOgOH2RNFNaYXhGSJga6s
31yOyMyZBRjby5Qzp4INo1RD1r20ZLFfbPcq0PE9UcWDMDYaBcyjmSwcx6jxDrwXgvCvMGXN5116U60r1p8ifD0bcw3D3rSr0F6N3OTOjUw4rwTZFQnRTJsd
+YpyFKik7e7acdXBskMyheA5JMkkWBIbEJvVNpz2XnUNcTjfYc8c63xk24cCbnIUYnIiGD0dB7n1oEMCCga2eoTqGXW+ybrCYGheCYA3GjD2QqJISGxmMKpk
wuSE66cdZx2GkE0eoh2ZqQMZJIIWzMz1rY0dRCDDaHUyPOeHyXrJ3hkmqC4PHhiO0gEIYZI/oyMguiZsxDNJEITxuHgzwj1JLClKeRbksvRbCpAt4FYJhgNk
4bO/v428cKvfyyTM2JvR4GdIeFBg6YgdQiJloXwScE7FEKBBCg6aF7xUWic0OGA3ENuA3EuGJEjfichfav3jxfufFvQ1vaH2ibIsT/xDjiJdR6rBENR6677o
fsiJ3GH0HQGHRrLUwbB66ai8Dt2eQ/+fP2D4aUkJAY4BmjcZdNU02BA8Bn2CAcfVHpLHn7YOQ6JXFYE2GMwCODSFO31eW7M+M7NBS2/o3uw+aDCAbWFEkUHg
p1l6PGx5rYQowhRbaoemjczSmBm9HBhLNogqs7wDzBl4gxvE5wCMFYPATgbrJTv+ntGwYLXiEAxdlDIcAzJfoppVb1TAnyR9x69I9dJhCODyf8vVe0y7qz3F
3JFOFCQ2qroeAA==
"""

source = bz2.decompress(
    base64.b64decode(
        PAYLOAD
    )
)

actual = hashlib.sha256(
    source
).hexdigest()

if actual != EXPECTED_SHA256:
    raise RuntimeError(
        f"Stage27-4A closure payload SHA mismatch: {actual} != {EXPECTED_SHA256}"
    )

OUT.write_bytes(
    source
)

print("Reconstructed :", OUT)
print("Bytes         :", OUT.stat().st_size)
print("SHA256        :", actual)

source_text = source.decode(
    "utf-8"
)

compiled = compile(
    source_text,
    str(OUT),
    "exec",
)

print("[PASS] Stage27-4A closure script reconstructed and syntax-verified.")
print("Scientific actions: ZERO inference / ZERO reopening / ZERO new statistics.")
print("Launching FINAL Stage27 closure...")
print("=" * 100)

exec(
    compiled,
    {
        "__name__": "__main__",
        "__file__": str(OUT),
    },
)


Reconstructed : /kaggle/working/stage27_4a_commit_push_verify.py
Bytes         : 25345
SHA256        : 1931553ff5bdef5cd956957533d7cfd5262943a0fa0edc8695f640a7c4d33e44
[PASS] Stage27-4A closure script reconstructed and syntax-verified.
Scientific actions: ZERO inference / ZERO reopening / ZERO new statistics.
Launching FINAL Stage27 closure...

STAGE27-4A :: PRE-COMMIT SCIENTIFIC-PARENT GATE
Expected parent : 4195be1c4fde26f76a3426728c5dd3b12d389e82
Local HEAD      : 4195be1c4fde26f76a3426728c5dd3b12d389e82
origin/main     : 4195be1c4fde26f76a3426728c5dd3b12d389e82
[PASS] Scientific parent unchanged.

STAGE27-4A :: EXACT 10-FILE STAGED UNIVERSE
Expected staged artifacts: 10
Actual staged artifacts  : 10
[PASS] Exactly 10 Stage27-4A artifacts staged; no unrelated changes.

STAGE27-4A :: LOCAL FREEZE / HASH GATE
Required artifacts       : 10
Non-self artifact hashes : 9
[PASS] All Stage27-4A frozen artifact hashes verified.

STAGE27-4A :: FINAL SCIENTIFIC BOUNDARY GATE
Taxonomy families 

In [8]:
# ============================================================================
# Stage27-PUB0
# Manuscript Integration Bootstrap + Frozen Science Integrity Gate
#
# PURPOSE
#   - Start the Stage27 manuscript/publication integration phase.
#   - Clone the canonical repository.
#   - Require the exact frozen Stage27 scientific parent.
#   - Verify every Stage27-4A synthesis artifact by SHA256.
#   - Confirm the repository is clean.
#   - Confirm GitHub credentials are available for the later publication push.
#
# IMPORTANT
#   This cell performs:
#       ZERO model fitting
#       ZERO model inference
#       ZERO target reopening
#       ZERO threshold selection
#       ZERO bootstrap computation
#       ZERO statistical testing
#
#   It only verifies already-frozen artifacts.
# ============================================================================

from __future__ import annotations

import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path
from datetime import datetime, timezone


# ----------------------------------------------------------------------------
# Frozen Stage27 identity
# ----------------------------------------------------------------------------

REPO_OWNER = "themubasshir"
REPO_NAME = "ids2018-validation-safe-ablation"
REPO_FULL = f"{REPO_OWNER}/{REPO_NAME}"

REPO_URL = f"https://github.com/{REPO_FULL}.git"

CANONICAL_STAGE27_COMMIT = (
    "0e1439565aedc7da9b7ca1207262e9061422bc22"
)

WORK_ROOT = Path("/kaggle/working")
REPO_DIR = WORK_ROOT / REPO_NAME

SYNTHESIS_REL = Path(
    "results/stage27_loao_unseen_attack/stage27_4a_final_synthesis"
)

SYNTHESIS_DIR = REPO_DIR / SYNTHESIS_REL


# ----------------------------------------------------------------------------
# Frozen Stage27 artifact SHA256 manifest
#
# These are CONTENT SHA256 values from the final Stage27 handoff.
# They are intentionally independent from Git blob SHA values.
# ----------------------------------------------------------------------------

EXPECTED_SHA256 = {
    "stage27_final_primary_metrics.csv":
        "42ea04b3f21e6026d5d69c8d5b59aa1edd2b57e94c42da3b9f70587349704634",

    "stage27_final_operating_points.csv":
        "664a5aaaff718f20bf6d619ae1dd4871a07a37c81a1631590423ea0ae07240f4",

    "stage27_final_novelty_gaps.csv":
        "91c80319186fd3bbfc382e58cfc60e58fc75d23db15408564fd35c05d4fb316c",

    "stage27_final_similarity.csv":
        "8c110b4f1d6317d2a2125b4f24bfb8325cdeb699ce763d268d91f3bad6acc8d3",

    "stage27_synthesis.md":
        "50b44ce0740816a51464179817fb0de5111cbe942e2c18c52df8d41d48f194fb",

    "stage27_synthesis_receipt.json":
        "55a67c1173d8bb3ffe2b2200542c382296459ae24b3b96fc0767abb6cb01bd3f",

    "stage27_4a_synthesis_freeze_record.json":
        "35135f1979518b36e614c7a0c2c7db9e4bd9bb78eb4d70346255684d0a4ae1db",

    "figures/stage27_primary_roc_auc_ci.png":
        "0e3659c1abb3a5ec9cb27af3e702f734f3c00266d695a672408c3d426e746152",

    "figures/stage27_primary_pr_auc_ci.png":
        "3528a3f2854f2d40dc1e2c23c20f14fb8d3b9edb6870d32088504e23b02dfba8",

    "figures/stage27_balanced_recall_ci.png":
        "441b3adc5f357c8d5b1ac1d6ce5fd3bd449fe9fdbaeea3a1c944a965b7b1e2b6",
}


# ----------------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------------

def run(
    cmd,
    *,
    cwd=None,
    check=True,
    capture=True,
    env=None,
):
    """Run command without exposing credentials."""
    if capture:
        result = subprocess.run(
            cmd,
            cwd=cwd,
            check=check,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            env=env,
        )
    else:
        result = subprocess.run(
            cmd,
            cwd=cwd,
            check=check,
            text=True,
            env=env,
        )

    return result


def git(*args, cwd=REPO_DIR):
    result = run(
        ["git", *args],
        cwd=cwd,
        capture=True,
    )
    return result.stdout.strip()


def sha256_file(path: Path, chunk_size=8 * 1024 * 1024) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)

    return h.hexdigest()


def get_github_token():
    """
    Search the Kaggle Secrets store using the labels we have used
    throughout this project.
    """
    try:
        from kaggle_secrets import UserSecretsClient
    except Exception as exc:
        raise RuntimeError(
            "kaggle_secrets is unavailable in this runtime."
        ) from exc

    client = UserSecretsClient()

    candidates = [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
        "GH_PAT",
    ]

    found = []

    for label in candidates:
        try:
            value = client.get_secret(label)
        except Exception:
            value = None

        if value:
            found.append((label, value))

    if not found:
        raise RuntimeError(
            "\nNo usable GitHub credential was found in Kaggle Secrets.\n\n"
            "Expected one of:\n"
            "  GITHUB_TOKEN\n"
            "  github_token\n"
            "  GH_TOKEN\n"
            "  GITHUB_PAT\n"
            "  github_pat\n"
            "  GH_PAT\n"
        )

    label, token = found[0]

    print(f"[PASS] GitHub secret available: {label}")
    print(f"       token length: {len(token)} characters")
    print("       token value: [REDACTED]")

    return label, token


# ----------------------------------------------------------------------------
# Environment header
# ----------------------------------------------------------------------------

print("=" * 88)
print("STAGE27-PUB0 — MANUSCRIPT INTEGRATION BOOTSTRAP")
print("=" * 88)

print()
print("timestamp_utc :", datetime.now(timezone.utc).isoformat())
print("python        :", sys.version.replace("\n", " "))
print("executable    :", sys.executable)
print("work_root     :", WORK_ROOT)
print("repository    :", REPO_FULL)
print("expected HEAD :", CANONICAL_STAGE27_COMMIT)

print()
print("Scientific mode:")
print("  model fitting          : FORBIDDEN")
print("  model inference        : FORBIDDEN")
print("  target reopening       : FORBIDDEN")
print("  threshold reselection  : FORBIDDEN")
print("  bootstrap recompute    : FORBIDDEN")
print("  manuscript integration : AUTHORIZED")


# ----------------------------------------------------------------------------
# GitHub secret gate
# ----------------------------------------------------------------------------

print()
print("-" * 88)
print("GITHUB CREDENTIAL GATE")
print("-" * 88)

GITHUB_SECRET_LABEL, GITHUB_TOKEN = get_github_token()


# ----------------------------------------------------------------------------
# Fresh repository clone
# ----------------------------------------------------------------------------

print()
print("-" * 88)
print("REPOSITORY BOOTSTRAP")
print("-" * 88)

if REPO_DIR.exists():
    print(f"Removing previous working checkout:")
    print(f"  {REPO_DIR}")
    shutil.rmtree(REPO_DIR)

print()
print("Cloning main branch...")

result = run(
    [
        "git",
        "clone",
        "--branch",
        "main",
        "--single-branch",
        REPO_URL,
        str(REPO_DIR),
    ],
    cwd=WORK_ROOT,
    capture=True,
)

print("[PASS] Repository cloned")


# ----------------------------------------------------------------------------
# Fetch and canonical-commit verification
# ----------------------------------------------------------------------------

git("fetch", "--prune", "origin")

HEAD = git("rev-parse", "HEAD")
ORIGIN_MAIN = git("rev-parse", "origin/main")

print()
print("HEAD        :", HEAD)
print("origin/main :", ORIGIN_MAIN)

if HEAD != CANONICAL_STAGE27_COMMIT:
    raise RuntimeError(
        "\nFROZEN-PARENT GATE FAILED.\n\n"
        f"Expected Stage27 canonical commit:\n"
        f"  {CANONICAL_STAGE27_COMMIT}\n\n"
        f"Current cloned HEAD:\n"
        f"  {HEAD}\n\n"
        "Do not generate publication artifacts until this discrepancy "
        "has been reviewed."
    )

if ORIGIN_MAIN != CANONICAL_STAGE27_COMMIT:
    raise RuntimeError(
        "\nREMOTE-PARENT GATE FAILED.\n\n"
        f"Expected origin/main:\n"
        f"  {CANONICAL_STAGE27_COMMIT}\n\n"
        f"Actual origin/main:\n"
        f"  {ORIGIN_MAIN}\n\n"
        "The remote repository has advanced unexpectedly. "
        "Do not silently build publication artifacts from a different parent."
    )

print()
print("[PASS] Exact frozen Stage27 parent confirmed")


# ----------------------------------------------------------------------------
# Verify required directory
# ----------------------------------------------------------------------------

print()
print("-" * 88)
print("STAGE27-4A ARTIFACT INTEGRITY")
print("-" * 88)

if not SYNTHESIS_DIR.is_dir():
    raise RuntimeError(
        f"Frozen synthesis directory not found:\n{SYNTHESIS_DIR}"
    )

print("Synthesis directory:")
print(f"  {SYNTHESIS_DIR}")
print()


# ----------------------------------------------------------------------------
# SHA256 verification
# ----------------------------------------------------------------------------

verification_rows = []

for relative_name, expected_hash in EXPECTED_SHA256.items():

    path = SYNTHESIS_DIR / relative_name

    exists = path.is_file()

    actual_hash = sha256_file(path) if exists else None

    passed = exists and actual_hash == expected_hash

    verification_rows.append(
        {
            "artifact": relative_name,
            "exists": exists,
            "expected_sha256": expected_hash,
            "actual_sha256": actual_hash,
            "pass": passed,
        }
    )

    status = "PASS" if passed else "FAIL"

    print(f"[{status}] {relative_name}")

    if exists:
        print(f"       expected: {expected_hash}")
        print(f"       actual:   {actual_hash}")
        print(f"       bytes:    {path.stat().st_size:,}")
    else:
        print("       MISSING")

    print()


failed = [
    row
    for row in verification_rows
    if not row["pass"]
]

if failed:
    print("=" * 88)
    print("FROZEN ARTIFACT VERIFICATION FAILED")
    print("=" * 88)

    for row in failed:
        print(row["artifact"])

    raise RuntimeError(
        f"{len(failed)} frozen Stage27 artifact(s) failed exact verification. "
        "Publication generation is blocked."
    )


print(
    f"[PASS] Exact Stage27 synthesis verification: "
    f"{len(verification_rows)}/{len(verification_rows)}"
)


# ----------------------------------------------------------------------------
# Git worktree cleanliness
# ----------------------------------------------------------------------------

print()
print("-" * 88)
print("GIT WORKTREE GATE")
print("-" * 88)

status = git("status", "--porcelain")

if status:
    print(status)
    raise RuntimeError(
        "Repository is not clean before manuscript integration."
    )

print("[PASS] Git working tree clean")


# ----------------------------------------------------------------------------
# Read-only science sanity gates
# ----------------------------------------------------------------------------

print()
print("-" * 88)
print("FROZEN SCIENCE SANITY GATES")
print("-" * 88)

import pandas as pd

primary = pd.read_csv(
    SYNTHESIS_DIR / "stage27_final_primary_metrics.csv"
)

ops = pd.read_csv(
    SYNTHESIS_DIR / "stage27_final_operating_points.csv"
)

gaps = pd.read_csv(
    SYNTHESIS_DIR / "stage27_final_novelty_gaps.csv"
)

similarity = pd.read_csv(
    SYNTHESIS_DIR / "stage27_final_similarity.csv"
)


# Expected executable family set
expected_families = {
    "BOT",
    "DDOS",
    "INFILTRATION",
    "PORT_SCAN",
    "WEB_ATTACK",
}

actual_families = set(primary["family"].unique())

assert actual_families == expected_families, (
    actual_families,
    expected_families,
)


# Exactly two preregistered learners
expected_learners = {
    "XGBOOST",
    "LIGHTGBM",
}

actual_learners = set(primary["learner"].unique())

assert actual_learners == expected_learners


# Expected row counts
assert len(primary) == 10
assert len(ops) == 30
assert len(gaps) == 10
assert len(similarity) == 5


# INFILTRATION must remain descriptive only
inf = primary[primary["family"] == "INFILTRATION"]

assert len(inf) == 2
assert (inf["heldout_attack_support"] == 36).all()
assert (
    inf["inferential_family_claim_authorized"]
    .astype(str)
    .str.lower()
    .isin(["false"])
    .all()
)


# BOT XGBoost negative PR-excess — important publication sanity point
bot_xgb = primary[
    (primary["family"] == "BOT")
    & (primary["learner"] == "XGBOOST")
].iloc[0]

assert bot_xgb["pr_excess"] < 0


# DDOS strong ranking survival
ddos = primary[primary["family"] == "DDOS"]

assert (ddos["roc_auc"] > 0.99).all()
assert (ddos["pr_auc"] > 0.99).all()


# WEB_ATTACK strong ranking survival
web = primary[primary["family"] == "WEB_ATTACK"]

assert (web["roc_auc"] > 0.96).all()
assert (web["pr_auc"] > 0.70).all()


# PORT_SCAN learner dependence
port_xgb = primary[
    (primary["family"] == "PORT_SCAN")
    & (primary["learner"] == "XGBOOST")
].iloc[0]

port_lgb = primary[
    (primary["family"] == "PORT_SCAN")
    & (primary["learner"] == "LIGHTGBM")
].iloc[0]

assert port_lgb["roc_auc"] > port_xgb["roc_auc"]


# Similarity remains explicitly descriptive
assert (
    similarity["interpretation"]
    == "SECONDARY_DESCRIPTIVE_ONLY"
).all()


print("[PASS] executable family set")
print("[PASS] learner set")
print("[PASS] frozen table row counts")
print("[PASS] INFILTRATION descriptive-only gate")
print("[PASS] BOT negative-XGB PR-excess gate")
print("[PASS] DDOS strong-ranking gate")
print("[PASS] WEB_ATTACK strong-ranking gate")
print("[PASS] PORT_SCAN learner-dependence gate")
print("[PASS] similarity descriptive-only gate")


# ----------------------------------------------------------------------------
# Create local bootstrap receipt
#
# This is NOT committed yet. PUB1 will create the actual publication package.
# ----------------------------------------------------------------------------

receipt = {
    "stage": "STAGE27-PUB0",
    "purpose": "MANUSCRIPT_INTEGRATION_BOOTSTRAP",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "repository": REPO_FULL,
    "canonical_stage27_commit": CANONICAL_STAGE27_COMMIT,
    "verified_head": HEAD,
    "verified_origin_main": ORIGIN_MAIN,
    "verified_artifact_count": len(verification_rows),
    "artifact_verification": verification_rows,
    "science_operations": {
        "model_fitting": 0,
        "model_inference": 0,
        "target_reopenings": 0,
        "threshold_reselection": 0,
        "bootstrap_recomputation": 0,
        "new_formal_statistical_tests": 0,
    },
    "publication_generation_authorized": True,
}

BOOTSTRAP_RECEIPT = (
    WORK_ROOT / "stage27_pub0_bootstrap_receipt.json"
)

BOOTSTRAP_RECEIPT.write_text(
    json.dumps(receipt, indent=2) + "\n",
    encoding="utf-8",
)

print()
print("-" * 88)
print("PUB0 RECEIPT")
print("-" * 88)

print(BOOTSTRAP_RECEIPT)
print(
    "SHA256:",
    sha256_file(BOOTSTRAP_RECEIPT),
)


# ----------------------------------------------------------------------------
# Final state
# ----------------------------------------------------------------------------

print()
print("=" * 88)
print("STAGE27-PUB0 COMPLETE")
print("=" * 88)

print()
print("Frozen science parent:")
print(f"  {CANONICAL_STAGE27_COMMIT}")

print()
print("Artifact verification:")
print(f"  {len(verification_rows)}/{len(verification_rows)} PASS_EXACT")

print()
print("Repository:")
print(f"  {REPO_DIR}")

print()
print("Git state:")
print(f"  branch : {git('branch', '--show-current')}")
print(f"  HEAD   : {git('rev-parse', 'HEAD')}")
print("  status : CLEAN")

print()
print("Next authorized operation:")
print("  STAGE27-PUB1 — generate manuscript/publication artifacts")
print()
print("NO Stage27 scientific computation has been reopened.")

STAGE27-PUB0 — MANUSCRIPT INTEGRATION BOOTSTRAP

timestamp_utc : 2026-08-21T15:07:51.071649+00:00
python        : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
executable    : /usr/bin/python3
work_root     : /kaggle/working
repository    : themubasshir/ids2018-validation-safe-ablation
expected HEAD : 0e1439565aedc7da9b7ca1207262e9061422bc22

Scientific mode:
  model fitting          : FORBIDDEN
  model inference        : FORBIDDEN
  target reopening       : FORBIDDEN
  threshold reselection  : FORBIDDEN
  bootstrap recompute    : FORBIDDEN
  manuscript integration : AUTHORIZED

----------------------------------------------------------------------------------------
GITHUB CREDENTIAL GATE
----------------------------------------------------------------------------------------
[PASS] GitHub secret available: GITHUB_TOKEN
       token length: 93 characters
       token value: [REDACTED]

----------------------------------------------------------------------------------------
REPOSIT

In [9]:
# ============================================================================
# Stage27-PUB1
# Deterministic Manuscript / Publication Package Generation
#
# INPUT:
#   Frozen Stage27-4A synthesis artifacts only.
#
# OUTPUT:
#   docs/STAGE27_MANUSCRIPT_INTEGRATION.md
#   docs/STAGE27_MANUSCRIPT_INTEGRATION.tex
#   docs/STAGE27_PUBLICATION_TABLES.md
#   docs/STAGE27_PUBLICATION_TABLES.tex
#   scripts/stage27/stage27_publication_integration.py
#   results/stage27_loao_unseen_attack/stage27_publication_package/
#       stage27_publication_manifest.json
#
# IMPORTANT:
#   ZERO model fitting
#   ZERO model inference
#   ZERO target reopening
#   ZERO threshold selection
#   ZERO bootstrap recomputation
#   ZERO new statistical testing
#
#   NO GIT COMMIT
#   NO GIT PUSH
# ============================================================================

from __future__ import annotations

import hashlib
import json
import os
import subprocess
import sys
from pathlib import Path


# ----------------------------------------------------------------------------
# Repository identity
# ----------------------------------------------------------------------------

REPO = Path("/kaggle/working/ids2018-validation-safe-ablation")

CANONICAL_STAGE27_PARENT = (
    "0e1439565aedc7da9b7ca1207262e9061422bc22"
)

GENERATOR_REL = Path(
    "scripts/stage27/stage27_publication_integration.py"
)

GENERATOR_PATH = REPO / GENERATOR_REL


def run(cmd, cwd=REPO, check=True):
    result = subprocess.run(
        cmd,
        cwd=cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    if check and result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(
            f"Command failed ({result.returncode}): {' '.join(cmd)}"
        )

    return result


def git(*args):
    return run(["git", *args]).stdout.strip()


def sha256_file(path: Path):
    h = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()


print("=" * 92)
print("STAGE27-PUB1 — DETERMINISTIC PUBLICATION PACKAGE GENERATION")
print("=" * 92)

print()
print("Repository:")
print(f"  {REPO}")

if not REPO.is_dir():
    raise RuntimeError("Repository checkout not found. Run PUB0 first.")

head = git("rev-parse", "HEAD")
status_before = git("status", "--porcelain")

print()
print("HEAD:")
print(f"  {head}")

if head != CANONICAL_STAGE27_PARENT:
    raise RuntimeError(
        "\nPUB1 requires the exact frozen Stage27 parent before generation.\n"
        f"Expected: {CANONICAL_STAGE27_PARENT}\n"
        f"Actual:   {head}"
    )

if status_before:
    print(status_before)
    raise RuntimeError(
        "Repository is not clean before PUB1. "
        "Do not generate over unreviewed changes."
    )

print("[PASS] exact Stage27 scientific parent")
print("[PASS] clean worktree")


# ============================================================================
# Generator source
#
# The exact same source is:
#   1. stored in GitHub under scripts/stage27/
#   2. executed here in Kaggle
#
# This makes the publication package reproducible without reopening science.
# ============================================================================

GENERATOR_SOURCE = r'''#!/usr/bin/env python3
"""
Stage27 publication integration generator.

This script creates manuscript-facing publication artifacts exclusively from
the already-frozen Stage27-4A synthesis artifacts.

It performs no model fitting, no model inference, no target reopening,
no threshold selection, no bootstrap recomputation, and no new statistical
testing.
"""

from __future__ import annotations

import argparse
import hashlib
import json
import subprocess
from pathlib import Path

import pandas as pd


CANONICAL_SCIENTIFIC_PARENT = (
    "0e1439565aedc7da9b7ca1207262e9061422bc22"
)

STAGE27_DATE = "2026-08-21"

SYNTHESIS_REL = Path(
    "results/stage27_loao_unseen_attack/stage27_4a_final_synthesis"
)

PACKAGE_REL = Path(
    "results/stage27_loao_unseen_attack/stage27_publication_package"
)

GENERATOR_REL = Path(
    "scripts/stage27/stage27_publication_integration.py"
)

OUTPUT_PATHS = {
    "manuscript_md": Path("docs/STAGE27_MANUSCRIPT_INTEGRATION.md"),
    "manuscript_tex": Path("docs/STAGE27_MANUSCRIPT_INTEGRATION.tex"),
    "tables_md": Path("docs/STAGE27_PUBLICATION_TABLES.md"),
    "tables_tex": Path("docs/STAGE27_PUBLICATION_TABLES.tex"),
}

MANIFEST_REL = PACKAGE_REL / "stage27_publication_manifest.json"


EXPECTED_SOURCE_SHA256 = {
    "stage27_final_primary_metrics.csv":
        "42ea04b3f21e6026d5d69c8d5b59aa1edd2b57e94c42da3b9f70587349704634",

    "stage27_final_operating_points.csv":
        "664a5aaaff718f20bf6d619ae1dd4871a07a37c81a1631590423ea0ae07240f4",

    "stage27_final_novelty_gaps.csv":
        "91c80319186fd3bbfc382e58cfc60e58fc75d23db15408564fd35c05d4fb316c",

    "stage27_final_similarity.csv":
        "8c110b4f1d6317d2a2125b4f24bfb8325cdeb699ce763d268d91f3bad6acc8d3",

    "stage27_synthesis.md":
        "50b44ce0740816a51464179817fb0de5111cbe942e2c18c52df8d41d48f194fb",

    "stage27_synthesis_receipt.json":
        "55a67c1173d8bb3ffe2b2200542c382296459ae24b3b96fc0767abb6cb01bd3f",

    "stage27_4a_synthesis_freeze_record.json":
        "35135f1979518b36e614c7a0c2c7db9e4bd9bb78eb4d70346255684d0a4ae1db",

    "figures/stage27_primary_roc_auc_ci.png":
        "0e3659c1abb3a5ec9cb27af3e702f734f3c00266d695a672408c3d426e746152",

    "figures/stage27_primary_pr_auc_ci.png":
        "3528a3f2854f2d40dc1e2c23c20f14fb8d3b9edb6870d32088504e23b02dfba8",

    "figures/stage27_balanced_recall_ci.png":
        "441b3adc5f357c8d5b1ac1d6ce5fd3bd449fe9fdbaeea3a1c944a965b7b1e2b6",
}


FAMILY_ORDER = [
    "BOT",
    "DDOS",
    "INFILTRATION",
    "PORT_SCAN",
    "WEB_ATTACK",
]

LEARNER_ORDER = [
    "XGBOOST",
    "LIGHTGBM",
]

LEARNER_DISPLAY = {
    "XGBOOST": "XGBoost",
    "LIGHTGBM": "LightGBM",
}


EXECUTABILITY_ROWS = [
    {
        "family": "BOT",
        "status": "ELIGIBLE",
        "support": 1966,
        "target": "Friday",
        "interpretation": "Inferential support eligible",
    },
    {
        "family": "DDOS",
        "status": "ELIGIBLE",
        "support": 128027,
        "target": "Friday",
        "interpretation": "Inferential support eligible",
    },
    {
        "family": "DOS",
        "status": "STRUCTURALLY_INELIGIBLE",
        "support": None,
        "target": "Wednesday",
        "interpretation":
            "No valid supervised day-atomic training geometry",
    },
    {
        "family": "AUTH_BRUTE_FORCE",
        "status": "STRUCTURALLY_INELIGIBLE",
        "support": None,
        "target": "Tuesday",
        "interpretation":
            "Insufficient earlier weekday depth",
    },
    {
        "family": "INFILTRATION",
        "status": "ELIGIBLE_DESCRIPTIVE_ONLY",
        "support": 36,
        "target": "Thursday",
        "interpretation":
            "Descriptive only; held-out support < 50",
    },
    {
        "family": "PORT_SCAN",
        "status": "ELIGIBLE",
        "support": 158930,
        "target": "Friday",
        "interpretation": "Inferential support eligible",
    },
    {
        "family": "WEB_ATTACK",
        "status": "ELIGIBLE",
        "support": 2180,
        "target": "Thursday",
        "interpretation": "Inferential support eligible",
    },
]


def run_git(root: Path, *args: str):
    result = subprocess.run(
        ["git", *args],
        cwd=root,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"git {' '.join(args)} failed:\n{result.stderr}"
        )

    return result.stdout.strip()


def repo_root():
    here = Path(__file__).resolve().parent

    result = subprocess.run(
        ["git", "rev-parse", "--show-toplevel"],
        cwd=here,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    if result.returncode != 0:
        raise RuntimeError("Unable to locate Git repository root.")

    return Path(result.stdout.strip()).resolve()


def sha256_file(path: Path):
    h = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()


def verify_scientific_parent(root: Path):
    head = run_git(root, "rev-parse", "HEAD")

    result = subprocess.run(
        [
            "git",
            "merge-base",
            "--is-ancestor",
            CANONICAL_SCIENTIFIC_PARENT,
            head,
        ],
        cwd=root,
    )

    if result.returncode != 0:
        raise RuntimeError(
            "Canonical Stage27 scientific parent is not an ancestor "
            "of the current repository HEAD."
        )

    return head


def verify_frozen_sources(root: Path):
    synthesis = root / SYNTHESIS_REL

    verification = {}

    for rel, expected in EXPECTED_SOURCE_SHA256.items():
        path = synthesis / rel

        if not path.is_file():
            raise RuntimeError(
                f"Frozen Stage27 source missing: {path}"
            )

        actual = sha256_file(path)

        if actual != expected:
            raise RuntimeError(
                f"Frozen Stage27 source hash mismatch:\n"
                f"  artifact: {rel}\n"
                f"  expected: {expected}\n"
                f"  actual:   {actual}"
            )

        verification[rel] = actual

    return verification


def fmt(value, digits=4):
    return f"{float(value):.{digits}f}"


def fmt6(value):
    return f"{float(value):.6f}"


def pct(value, digits=2):
    return f"{100.0 * float(value):.{digits}f}%"


def ci_text(row, metric, digits=4):
    point = float(row[metric])
    lo = float(row[f"{metric}_ci_2_5"])
    hi = float(row[f"{metric}_ci_97_5"])

    return (
        f"{point:.{digits}f} "
        f"({lo:.{digits}f}–{hi:.{digits}f})"
    )


def tex_ci(row, metric, digits=4):
    point = float(row[metric])
    lo = float(row[f"{metric}_ci_2_5"])
    hi = float(row[f"{metric}_ci_97_5"])

    return (
        f"{point:.{digits}f} "
        f"[{lo:.{digits}f}, {hi:.{digits}f}]"
    )


def select_row(df, family, learner):
    rows = df[
        (df["family"] == family)
        & (df["learner"] == learner)
    ]

    if len(rows) != 1:
        raise RuntimeError(
            f"Expected exactly one row for {family}/{learner}; "
            f"found {len(rows)}"
        )

    return rows.iloc[0]


def balanced_row(ops, family, learner):
    rows = ops[
        (ops["family"] == family)
        & (ops["learner"] == learner)
        & (ops["operating_point"] == "BALANCED")
    ]

    if len(rows) != 1:
        raise RuntimeError(
            f"Expected exactly one BALANCED row for "
            f"{family}/{learner}; found {len(rows)}"
        )

    return rows.iloc[0]


def scientific_sanity(primary, ops, gaps, similarity):
    assert len(primary) == 10
    assert len(ops) == 30
    assert len(gaps) == 10
    assert len(similarity) == 5

    assert set(primary["family"]) == set(FAMILY_ORDER)
    assert set(primary["learner"]) == set(LEARNER_ORDER)

    inf = primary[primary["family"] == "INFILTRATION"]

    assert len(inf) == 2
    assert (inf["heldout_attack_support"] == 36).all()

    inferential = (
        inf["inferential_family_claim_authorized"]
        .astype(str)
        .str.lower()
    )

    assert (inferential == "false").all()

    bot_xgb = select_row(primary, "BOT", "XGBOOST")
    assert float(bot_xgb["pr_excess"]) < 0

    ddos = primary[primary["family"] == "DDOS"]
    assert (ddos["roc_auc"] > 0.99).all()
    assert (ddos["pr_auc"] > 0.99).all()

    web = primary[primary["family"] == "WEB_ATTACK"]
    assert (web["roc_auc"] > 0.96).all()
    assert (web["pr_auc"] > 0.70).all()

    port_xgb = select_row(primary, "PORT_SCAN", "XGBOOST")
    port_lgb = select_row(primary, "PORT_SCAN", "LIGHTGBM")

    assert float(port_lgb["roc_auc"]) > float(port_xgb["roc_auc"])

    assert (
        similarity["interpretation"]
        == "SECONDARY_DESCRIPTIVE_ONLY"
    ).all()

    bal = ops[ops["operating_point"] == "BALANCED"]

    expected_balanced = {
        ("BOT", "XGBOOST"): 0.0,
        ("BOT", "LIGHTGBM"): 0.0,
        ("DDOS", "XGBOOST"): 0.6619697407578089,
        ("DDOS", "LIGHTGBM"): 0.2625383708124068,
        ("INFILTRATION", "XGBOOST"): 0.0,
        ("INFILTRATION", "LIGHTGBM"): 0.0,
        ("PORT_SCAN", "XGBOOST"): 0.00480085572264519,
        ("PORT_SCAN", "LIGHTGBM"): 0.011678097275530108,
        ("WEB_ATTACK", "XGBOOST"): 0.7779816513761468,
        ("WEB_ATTACK", "LIGHTGBM"): 0.5211009174311927,
    }

    for key, expected in expected_balanced.items():
        family, learner = key
        row = balanced_row(ops, family, learner)
        actual = float(row["recall"])

        assert abs(actual - expected) < 1e-15


def build_primary_markdown(primary, ops):
    lines = [
        "| Family | Learner | Held-out support | ROC-AUC (95% CI) | "
        "PR-AUC (95% CI) | BALANCED recall |",
        "|---|---|---:|---:|---:|---:|",
    ]

    for family in FAMILY_ORDER:
        for learner in LEARNER_ORDER:
            row = select_row(primary, family, learner)
            op = balanced_row(ops, family, learner)

            family_display = (
                "INFILTRATION†"
                if family == "INFILTRATION"
                else family
            )

            lines.append(
                "| "
                + " | ".join([
                    family_display,
                    LEARNER_DISPLAY[learner],
                    f"{int(row['heldout_attack_support']):,}",
                    ci_text(row, "roc_auc", 4),
                    ci_text(row, "pr_auc", 6),
                    pct(op["recall"], 2),
                ])
                + " |"
            )

    lines.extend([
        "",
        "† INFILTRATION is descriptive only because "
        "held-out support is 36 (<50).",
    ])

    return "\n".join(lines)


def build_executability_markdown():
    lines = [
        "| Family | Status | Held-out support | Target day | "
        "Interpretation |",
        "|---|---|---:|---|---|",
    ]

    for row in EXECUTABILITY_ROWS:
        support = (
            "—"
            if row["support"] is None
            else f"{row['support']:,}"
        )

        lines.append(
            "| "
            + " | ".join([
                row["family"],
                row["status"],
                support,
                row["target"],
                row["interpretation"],
            ])
            + " |"
        )

    return "\n".join(lines)


def build_operating_markdown(ops):
    lines = [
        "| Family | Learner | Operating point | Threshold | "
        "Precision | Recall | FPR | F1 |",
        "|---|---|---|---:|---:|---:|---:|---:|",
    ]

    order_points = ["STANDARD", "BALANCED", "SECURITY"]

    for family in FAMILY_ORDER:
        for learner in LEARNER_ORDER:
            for operating_point in order_points:
                rows = ops[
                    (ops["family"] == family)
                    & (ops["learner"] == learner)
                    & (ops["operating_point"] == operating_point)
                ]

                row = rows.iloc[0]

                lines.append(
                    "| "
                    + " | ".join([
                        family,
                        LEARNER_DISPLAY[learner],
                        operating_point,
                        fmt(row["threshold"], 2),
                        fmt6(row["precision"]),
                        fmt6(row["recall"]),
                        fmt6(row["fpr"]),
                        fmt6(row["f1"]),
                    ])
                    + " |"
                )

    return "\n".join(lines)


def build_gap_markdown(gaps):
    lines = [
        "| Family | Learner | ROC-AUC known−unseen gap | "
        "PR-excess known−unseen gap | BALANCED recall gap |",
        "|---|---|---:|---:|---:|",
    ]

    for family in FAMILY_ORDER:
        for learner in LEARNER_ORDER:
            row = gaps[
                (gaps["family"] == family)
                & (gaps["learner"] == learner)
            ].iloc[0]

            lines.append(
                "| "
                + " | ".join([
                    family,
                    LEARNER_DISPLAY[learner],
                    fmt6(row["gap_roc_auc"]),
                    fmt6(row["gap_pr_excess"]),
                    fmt6(row["gap_recall_balanced"]),
                ])
                + " |"
            )

    lines.extend([
        "",
        "Raw known-minus-unseen PR-AUC differences are not treated "
        "as prevalence-invariant primary novelty gaps because the "
        "comparison populations have different prevalence anchors.",
    ])

    return "\n".join(lines)


def build_similarity_markdown(similarity):
    lines = [
        "| Held-out family | Nearest seen family | Distance | "
        "Similarity | Benign distance |",
        "|---|---|---:|---:|---:|",
    ]

    indexed = similarity.set_index("family")

    for family in FAMILY_ORDER:
        row = indexed.loc[family]

        lines.append(
            "| "
            + " | ".join([
                family,
                str(row["nearest_seen_family"]),
                fmt6(row["nearest_seen_distance"]),
                fmt6(row["similarity_score"]),
                fmt6(row["benign_distance"]),
            ])
            + " |"
        )

    lines.extend([
        "",
        "This analysis is secondary and descriptive only. "
        "No formal correlation test, p-value, regression inference, "
        "or causal interpretation is authorized.",
    ])

    return "\n".join(lines)


def build_publication_tables_md(primary, ops, gaps, similarity):
    return f"""# Stage27 Publication Tables

Scientific parent:

`{CANONICAL_SCIENTIFIC_PARENT}`

These tables are generated exclusively from frozen Stage27 artifacts.
No target reopening, inference, model fitting, threshold reselection,
bootstrap recomputation, or new statistical testing is performed.

---

## Table 27-1. Chronology-first family executability

{build_executability_markdown()}

---

## Table 27-2. Primary unseen-family performance

{build_primary_markdown(primary, ops)}

The 95% intervals are the frozen 2,000-replicate stratified
row-bootstrap intervals and quantify target-sampling uncertainty
conditional on the already-fitted model.

---

## Table 27-S1. Complete frozen operating points

{build_operating_markdown(ops)}

---

## Table 27-S2. Compatible novelty-generalization gaps

{build_gap_markdown(gaps)}

---

## Table 27-S3. Behavioral similarity

{build_similarity_markdown(similarity)}

---

## Figure placement

### Main manuscript

1. `results/stage27_loao_unseen_attack/stage27_4a_final_synthesis/figures/stage27_primary_roc_auc_ci.png`
2. `results/stage27_loao_unseen_attack/stage27_4a_final_synthesis/figures/stage27_balanced_recall_ci.png`

### Supplementary material

3. `results/stage27_loao_unseen_attack/stage27_4a_final_synthesis/figures/stage27_primary_pr_auc_ci.png`

PR-AUC remains a co-primary metric and must remain in the main
results table and manuscript text even when its separate figure is
placed in supplementary material.
"""


def build_publication_tables_tex(primary, ops, gaps, similarity):
    lines = [
        "% =====================================================================",
        "% Stage27 Publication Tables",
        "% Auto-generated from frozen Stage27 artifacts.",
        "% No scientific model execution is performed by this file.",
        "% =====================================================================",
        "",
        r"\begin{table*}[t]",
        r"\centering",
        r"\caption{Chronology-first unseen-family fold executability under the frozen Stage27 protocol.}",
        r"\label{tab:stage27_executability}",
        r"\begin{tabular}{lllrl}",
        r"\hline",
        r"Family & Status & Target & Support & Interpretation \\",
        r"\hline",
    ]

    for row in EXECUTABILITY_ROWS:
        support = (
            "--"
            if row["support"] is None
            else f"{row['support']:,}"
        )

        family = row["family"].replace("_", r"\_")
        status = row["status"].replace("_", r"\_")
        interp = row["interpretation"].replace("<", "$<$")

        lines.append(
            f"{family} & {status} & {row['target']} & "
            f"{support} & {interp} \\\\"
        )

    lines.extend([
        r"\hline",
        r"\end{tabular}",
        r"\end{table*}",
        "",
        "",
        r"\begin{table*}[t]",
        r"\centering",
        r"\caption{Primary unseen attack-family ranking and frozen BALANCED-threshold recall. Values in brackets are frozen 95\% percentile bootstrap intervals.}",
        r"\label{tab:stage27_primary}",
        r"\begin{tabular}{llrrrr}",
        r"\hline",
        r"Family & Learner & Support & ROC-AUC [95\% CI] & PR-AUC [95\% CI] & Balanced Recall \\",
        r"\hline",
    ])

    for family in FAMILY_ORDER:
        for learner in LEARNER_ORDER:
            row = select_row(primary, family, learner)
            op = balanced_row(ops, family, learner)

            family_tex = family.replace("_", r"\_")

            if family == "INFILTRATION":
                family_tex += r"$^{\dagger}$"

            lines.append(
                f"{family_tex} & "
                f"{LEARNER_DISPLAY[learner]} & "
                f"{int(row['heldout_attack_support']):,} & "
                f"{tex_ci(row, 'roc_auc', 4)} & "
                f"{tex_ci(row, 'pr_auc', 6)} & "
                f"{pct(op['recall'], 2).replace('%', r'\%')} \\\\"
            )

    lines.extend([
        r"\hline",
        r"\multicolumn{6}{l}{$^{\dagger}$INFILTRATION is descriptive only because held-out support is 36 ($<50$).}\\",
        r"\end{tabular}",
        r"\end{table*}",
        "",
        "",
        r"\begin{table*}[t]",
        r"\centering",
        r"\caption{Frozen Stage27 operating-point transfer to each unseen-family isolation target.}",
        r"\label{tab:stage27_operating_points}",
        r"\begin{tabular}{lllrrrrr}",
        r"\hline",
        r"Family & Learner & Point & Threshold & Precision & Recall & FPR & F1 \\",
        r"\hline",
    ])

    for family in FAMILY_ORDER:
        for learner in LEARNER_ORDER:
            for point in ["STANDARD", "BALANCED", "SECURITY"]:
                row = ops[
                    (ops["family"] == family)
                    & (ops["learner"] == learner)
                    & (ops["operating_point"] == point)
                ].iloc[0]

                lines.append(
                    f"{family.replace('_', r'\_')} & "
                    f"{LEARNER_DISPLAY[learner]} & "
                    f"{point} & "
                    f"{float(row['threshold']):.2f} & "
                    f"{float(row['precision']):.6f} & "
                    f"{float(row['recall']):.6f} & "
                    f"{float(row['fpr']):.6f} & "
                    f"{float(row['f1']):.6f} \\\\"
                )

    lines.extend([
        r"\hline",
        r"\end{tabular}",
        r"\end{table*}",
        "",
        "",
        r"\begin{table*}[t]",
        r"\centering",
        r"\caption{Frozen Stage27 novelty-generalization gaps. PR-excess is used for the prevalence-compatible primary PR comparison.}",
        r"\label{tab:stage27_novelty_gaps}",
        r"\begin{tabular}{llrrr}",
        r"\hline",
        r"Family & Learner & ROC-AUC Gap & PR-Excess Gap & Balanced Recall Gap \\",
        r"\hline",
    ])

    for family in FAMILY_ORDER:
        for learner in LEARNER_ORDER:
            row = gaps[
                (gaps["family"] == family)
                & (gaps["learner"] == learner)
            ].iloc[0]

            lines.append(
                f"{family.replace('_', r'\_')} & "
                f"{LEARNER_DISPLAY[learner]} & "
                f"{float(row['gap_roc_auc']):.6f} & "
                f"{float(row['gap_pr_excess']):.6f} & "
                f"{float(row['gap_recall_balanced']):.6f} \\\\"
            )

    lines.extend([
        r"\hline",
        r"\end{tabular}",
        r"\end{table*}",
        "",
        "",
        r"\begin{table*}[t]",
        r"\centering",
        r"\caption{Secondary descriptive behavioral-similarity audit for the executable Stage27 families.}",
        r"\label{tab:stage27_similarity}",
        r"\begin{tabular}{llrrr}",
        r"\hline",
        r"Held-out Family & Nearest Seen Family & Distance & Similarity & Benign Distance \\",
        r"\hline",
    ])

    indexed = similarity.set_index("family")

    for family in FAMILY_ORDER:
        row = indexed.loc[family]

        lines.append(
            f"{family.replace('_', r'\_')} & "
            f"{str(row['nearest_seen_family']).replace('_', r'\_')} & "
            f"{float(row['nearest_seen_distance']):.6f} & "
            f"{float(row['similarity_score']):.6f} & "
            f"{float(row['benign_distance']):.6f} \\\\"
        )

    lines.extend([
        r"\hline",
        r"\end{tabular}",
        r"\end{table*}",
        "",
        "% Behavioral similarity is secondary and descriptive only.",
        "% No formal correlation test, regression inference, p-value,",
        "% or causal interpretation is authorized.",
        "",
    ])

    return "\n".join(lines)


def build_manuscript_md(primary, ops, gaps, similarity):
    bot_x = select_row(primary, "BOT", "XGBOOST")
    bot_l = select_row(primary, "BOT", "LIGHTGBM")
    ddos_x = select_row(primary, "DDOS", "XGBOOST")
    ddos_l = select_row(primary, "DDOS", "LIGHTGBM")
    inf_x = select_row(primary, "INFILTRATION", "XGBOOST")
    inf_l = select_row(primary, "INFILTRATION", "LIGHTGBM")
    port_x = select_row(primary, "PORT_SCAN", "XGBOOST")
    port_l = select_row(primary, "PORT_SCAN", "LIGHTGBM")
    web_x = select_row(primary, "WEB_ATTACK", "XGBOOST")
    web_l = select_row(primary, "WEB_ATTACK", "LIGHTGBM")

    bot_x_bal = balanced_row(ops, "BOT", "XGBOOST")
    bot_l_bal = balanced_row(ops, "BOT", "LIGHTGBM")
    ddos_x_bal = balanced_row(ops, "DDOS", "XGBOOST")
    ddos_l_bal = balanced_row(ops, "DDOS", "LIGHTGBM")
    inf_x_bal = balanced_row(ops, "INFILTRATION", "XGBOOST")
    inf_l_bal = balanced_row(ops, "INFILTRATION", "LIGHTGBM")
    port_x_bal = balanced_row(ops, "PORT_SCAN", "XGBOOST")
    port_l_bal = balanced_row(ops, "PORT_SCAN", "LIGHTGBM")
    web_x_bal = balanced_row(ops, "WEB_ATTACK", "XGBOOST")
    web_l_bal = balanced_row(ops, "WEB_ATTACK", "LIGHTGBM")

    similarity_idx = similarity.set_index("family")

    return f"""# Stage27 Manuscript Integration

## Scientific Identity

**Stage27 title:** Leave-One-Attack-Family-Out Unseen-Family Generalization Audit

**Design:** `CHRONOLOGY_FIRST_ZERO_TRAINING_EXPOSURE_FAMILY_AUDIT`

**Canonical scientific parent:** `{CANONICAL_SCIENTIFIC_PARENT}`

Stage27 scientific execution is closed. This document is a post-closure
publication-integration artifact generated from the frozen Stage27-4A
synthesis. It introduces no new measurement and authorizes no target
reopening, model inference, model refitting, threshold reselection,
bootstrap recomputation, feature modification, or post-target model
selection.

The publication-safe high-level outcome is:

1. `SELECTIVE_FAMILY_TRANSFER`
2. `RANKING_THRESHOLD_DIVERGENCE`
3. `LEARNER_DEPENDENCE`

Stage27 is an unseen attack-family generalization audit. It must not be
described as formal proof of zero-day detection.

---

# A. Proposed Contribution Text for the Introduction

A further contribution of this study is a chronology-first
zero-training-exposure attack-family generalization audit. Seven
CICIDS2017 attack families were preregistered and evaluated under a
strict `TRAIN < VALIDATION < TARGET` design in which the held-out family
was absent from both training and validation. Five families were
structurally executable, while DOS and AUTH_BRUTE_FORCE could not be
evaluated without violating the frozen chronological geometry. Across
the executable families, transfer was selective rather than universal:
DDoS and Web Attack retained strong ranking discrimination, Bot traffic
collapsed, and Port Scan exhibited substantial learner dependence.
Moreover, preserved ranking discrimination did not necessarily yield
useful recall at validation-selected frozen thresholds, separating
attack-family ranking generalization from operating-point transfer.

---

# B. Methods — Chronology-First Unseen-Family Generalization

## B.1 Scientific question

Stage27 evaluates whether a binary intrusion detector trained without
exposure to a particular attack family can discriminate that held-out
family from temporally matched benign traffic when the family first
becomes eligible under strict chronology.

The experiment is therefore described as an **unseen attack-family** or
**zero-training-exposure family** audit rather than as a formal
zero-day-detection experiment.

## B.2 Frozen taxonomy and executability

The preregistered primary taxonomy contains:

- BOT
- DDOS
- DOS
- AUTH_BRUTE_FORCE
- INFILTRATION
- PORT_SCAN
- WEB_ATTACK

Five of seven families were executable. DOS was structurally ineligible
because its first valid target day was Wednesday, leaving Monday for
training and Tuesday for validation, while Monday contained zero
known-family attack positives. AUTH_BRUTE_FORCE was structurally
ineligible because its first appearance on Tuesday left insufficient
earlier weekday depth for separate training and validation periods.

INFILTRATION was executable but is permanently descriptive only because
its held-out target support was 36.

## B.3 Chronological fold geometry

For BOT, DDOS, and PORT_SCAN:

- TRAIN: Monday–Wednesday
- VALIDATION: Thursday
- TARGET: Friday
- training rows: 1,668,519
- validation rows: 458,968
- Friday benign rows: 414,322

For INFILTRATION and WEB_ATTACK:

- TRAIN: Monday–Tuesday
- VALIDATION: Wednesday
- TARGET: Thursday
- training rows: 975,827
- validation rows: 692,692
- Thursday benign rows: 456,752

The held-out family has zero training rows and zero validation rows in
every executable fold. Any positive held-out-family membership in either
development role would invalidate the fold.

## B.4 Primary target semantics

The primary isolation target is:

`HELD_OUT_FAMILY + SAME_TARGET_DAY_BENIGN`

The positive class contains only the held-out attack family and the
negative class contains only benign traffic from the same target day.
Other known target-day attacks are excluded.

A broader operational context target containing held-out attacks, known
attacks, and benign traffic is secondary and descriptive only. The
manuscript should lead with the primary isolation target.

## B.5 Learners and thresholds

Two preregistered learners were evaluated:

- XGBoost
- LightGBM

No Stage27 hyperparameter optimization was permitted. Across five
executable folds and two learners, the total fit budget was exactly 10
models.

Three operating points were frozen from known-family validation data
only:

- STANDARD: threshold 0.50
- BALANCED: maximum validation F1, then minimum FPR, then higher threshold
- SECURITY: maximum validation F2 subject to FPR <= 0.05, then minimum
  FPR, then higher threshold

The threshold grid was 0.01–0.99 and the target decision rule was
`probability >= threshold`.

No target threshold search or target-guided model adaptation was
permitted.

## B.6 Bootstrap uncertainty

Stage27 uses 2,000-replicate class-stratified row bootstrap intervals
with seed 42. Sampling is performed with replacement within the benign
and held-out-attack target strata while preserving stratum sizes.

The intervals quantify **target-sampling uncertainty conditional on the
already-fitted model**. They do not include training-seed uncertainty,
model-selection uncertainty, model-retraining uncertainty, or broader
population uncertainty.

## B.7 Behavioral similarity

The secondary behavioral-similarity audit uses 11 preregistered
aggregate flow descriptors. Preprocessing is fitted only on current-fold
TRAIN rows, each family is represented by its standardized centroid, and
Euclidean distance to the nearest seen family is transformed to
similarity as:

`1 / (1 + nearest_seen_distance)`

This analysis is descriptive only. No formal correlation significance
test, regression inference, p-value, or causal interpretation is
authorized.

---

# C. Results — Unseen Attack-Family Generalization

## C.1 Executability under strict chronology

Of seven preregistered families, five were structurally executable.
BOT, DDOS, PORT_SCAN, and WEB_ATTACK satisfied the frozen family-level
support requirement. INFILTRATION was executable but remains
descriptive only because its held-out support was 36. DOS and
AUTH_BRUTE_FORCE were structurally ineligible under the precommitted
day-atomic chronology rather than being treated as model failures.

## C.2 Primary unseen-family ranking

The frozen ranking results demonstrate strongly family-dependent
transfer.

**DDoS produced the strongest transfer.** XGBoost reached ROC-AUC
{float(ddos_x['roc_auc']):.4f} and PR-AUC
{float(ddos_x['pr_auc']):.4f}, while LightGBM reached ROC-AUC
{float(ddos_l['roc_auc']):.4f} and PR-AUC
{float(ddos_l['pr_auc']):.4f}. Thus, both learners retained
near-perfect threshold-independent discrimination despite receiving
zero DDoS training or validation examples.

**Web Attack also transferred strongly.** XGBoost reached ROC-AUC
{float(web_x['roc_auc']):.4f} and PR-AUC
{float(web_x['pr_auc']):.4f}; LightGBM reached ROC-AUC
{float(web_l['roc_auc']):.4f} and PR-AUC
{float(web_l['pr_auc']):.4f}.

**Bot traffic showed substantial collapse.** XGBoost produced ROC-AUC
{float(bot_x['roc_auc']):.4f}, while LightGBM produced ROC-AUC
{float(bot_l['roc_auc']):.4f}. XGBoost PR-AUC was
{float(bot_x['pr_auc']):.6f}, below the target prevalence anchor of
{float(bot_x['prevalence']):.6f}, giving PR-excess
{float(bot_x['pr_excess']):.6f}. LightGBM was only marginally above the
same prevalence anchor, with PR-excess
{float(bot_l['pr_excess']):.6f}.

**Port Scan was materially learner-dependent.** XGBoost reached
ROC-AUC {float(port_x['roc_auc']):.4f}, whereas LightGBM reached
{float(port_l['roc_auc']):.4f}. The corresponding PR-AUC values were
{float(port_x['pr_auc']):.4f} and {float(port_l['pr_auc']):.4f},
respectively.

INFILTRATION produced ROC-AUC
{float(inf_x['roc_auc']):.4f} for XGBoost and
{float(inf_l['roc_auc']):.4f} for LightGBM, but these values are
reported descriptively because only 36 held-out attacks were available.

The overall result is therefore **selective family transfer**, not
uniform unseen-family generalization.

## C.3 Frozen operating-point transfer

Threshold-independent ranking quality did not guarantee useful
frozen-threshold detection.

At the BALANCED operating point:

- BOT recall was {pct(bot_x_bal['recall'])} for XGBoost and
  {pct(bot_l_bal['recall'])} for LightGBM.
- DDOS recall was {pct(ddos_x_bal['recall'])} and
  {pct(ddos_l_bal['recall'])}.
- INFILTRATION recall was {pct(inf_x_bal['recall'])} and
  {pct(inf_l_bal['recall'])}, descriptive only.
- PORT_SCAN recall was {pct(port_x_bal['recall'])} and
  {pct(port_l_bal['recall'])}.
- WEB_ATTACK recall was {pct(web_x_bal['recall'])} and
  {pct(web_l_bal['recall'])}.

The divergence is particularly visible for DDOS, where both learners
retain ROC-AUC above 0.998 but BALANCED recall is only
{pct(ddos_x_bal['recall'])} for XGBoost and
{pct(ddos_l_bal['recall'])} for LightGBM. Port Scan provides another
example: LightGBM retains ROC-AUC
{float(port_l['roc_auc']):.4f} but detects only
{pct(port_l_bal['recall'])} of held-out Port Scan attacks at its frozen
BALANCED threshold.

These results support the frozen Stage27 outcome
`RANKING_THRESHOLD_DIVERGENCE`.

## C.4 Novelty-generalization gaps

The compatible novelty-gap analysis further shows that family novelty
does not impose a uniform penalty.

For XGBoost, the known-minus-unseen ROC-AUC gap is approximately
{float(gaps[(gaps.family == 'BOT') & (gaps.learner == 'XGBOOST')].iloc[0]['gap_roc_auc']):.3f}
for BOT and
{float(gaps[(gaps.family == 'PORT_SCAN') & (gaps.learner == 'XGBOOST')].iloc[0]['gap_roc_auc']):.3f}
for PORT_SCAN, whereas the DDOS gap is
{float(gaps[(gaps.family == 'DDOS') & (gaps.learner == 'XGBOOST')].iloc[0]['gap_roc_auc']):.3f}.

PR-excess rather than raw PR-AUC difference is used as the primary
prevalence-compatible PR novelty gap. Raw PR-AUC differences across
populations with different prevalence anchors are retained only as
descriptive quantities.

## C.5 Behavioral similarity

The frozen behavioral-similarity values do not show a monotonic
relationship with unseen-family discrimination.

BOT has the highest observed similarity to a seen family
({float(similarity_idx.loc['BOT', 'similarity_score']):.4f}) yet weak
unseen-family performance. DDOS has a substantially lower similarity
({float(similarity_idx.loc['DDOS', 'similarity_score']):.4f}) but
near-perfect ranking. WEB_ATTACK has intermediate similarity
({float(similarity_idx.loc['WEB_ATTACK', 'similarity_score']):.4f})
while retaining strong transfer.

Behavioral proximity, as operationalized by this frozen centroid
distance, therefore does not appear sufficient by itself to explain
the observed transfer pattern.

---

# D. Discussion — Attack-Family Novelty and Generalization

Stage27 demonstrates that known-family intrusion-detection performance
cannot be treated as evidence of uniform robustness to attack-family
novelty. The strongest transfer cases, DDOS and WEB_ATTACK, retain high
ranking discrimination for both learners despite zero exposure to the
held-out family during training and validation. BOT provides the
opposite outcome, with complete frozen-threshold detection failure and
little or adverse ranking signal. PORT_SCAN occupies an intermediate
case in which the outcome depends materially on the learner.

A second finding is the distinction between ranking discrimination and
operating-point transfer. DDoS is the clearest example: both learners
rank the held-out family almost perfectly, yet validation-selected
BALANCED thresholds recover substantially less than all of the held-out
attacks. The same separation is visible for Port Scan and, to a lesser
degree, Web Attack. Consequently, ROC-AUC or PR-AUC alone cannot
characterize whether a frozen deployment threshold will remain useful
under attack-family novelty.

This ranking-versus-threshold distinction also complements earlier
experiments in the study. Representation-specific chronological
evaluation, the Stage22R forward temporal audit, and the Stage24
cross-dataset audit independently showed that strong ranking behavior
can coexist with poor fixed-threshold transfer. Stage27 extends that
observation to zero-training-exposure attack families. Across these
distinct stress regimes, threshold-independent discrimination and
operating-point behavior should therefore be evaluated as separate
properties of an IDS.

Learner dependence is itself family-dependent. XGBoost and LightGBM
agree closely on the strong DDoS and Web Attack ranking outcomes but
differ substantially on Port Scan and also differ in Bot ranking.
The evidence therefore does not support declaring one learner
universally superior for unseen-family generalization.

The behavioral-similarity analysis provides no simple mechanistic
explanation. BOT is behaviorally closest to a seen family under the
frozen 11-descriptor representation yet transfers poorly, whereas DDOS
is less similar under the same definition but transfers extremely well.
This secondary analysis should therefore be interpreted as evidence
that the selected notion of behavioral proximity is insufficient by
itself, not as proof of either the presence or absence of a particular
causal mechanism.

Finally, strict chronology exposes limitations in the benchmark itself.
The inability to execute DOS and AUTH_BRUTE_FORCE is a consequence of
the temporal arrangement of attack families and the requirement for
separate training and validation periods. Rather than manufacturing
alternative folds after observing the data, Stage27 preserves these
families as structurally ineligible. This makes the scope of the
generalization claim narrower but maintains the validation-safe
interpretation of the experiment.

---

# E. Limitations and Threats to Validity

1. **Incomplete taxonomy executability.** Only five of seven
   preregistered families could be honestly evaluated under strict
   `TRAIN < VALIDATION < TARGET` chronology.

2. **Low INFILTRATION support.** INFILTRATION contains only 36 held-out
   target attacks and is therefore descriptive only.

3. **Chronology-first rather than textbook LOAO.** Strict chronology
   means that every non-held-out attack family is not necessarily
   represented during training. Stage27 is therefore specifically a
   chronology-first zero-training-exposure family audit.

4. **Conditional bootstrap uncertainty.** The 95% intervals quantify
   row-level target-sampling uncertainty conditional on each already
   fitted model. They do not incorporate retraining, seed, model
   selection, independently collected networks, or broader population
   uncertainty.

5. **No clustered bootstrap.** No preregistered durable grouping
   variable was available for a session- or time-cluster bootstrap.

6. **Restricted similarity representation.** Behavioral similarity is
   based only on 11 preregistered aggregate flow descriptors and a
   centroid-distance representation.

7. **Descriptive similarity analysis.** No formal correlation test,
   p-value, regression inference, or causal interpretation is
   authorized.

8. **Benchmark-specific external validity.** CICIDS2017 is a benchmark
   capture. The observed transfer pattern does not establish universal
   behavior for production networks, unrelated datasets, or genuinely
   novel real-world attacks.

9. **No zero-day proof.** Zero training exposure to an attack family in
   this benchmark is not equivalent to demonstrating universal
   real-world zero-day detection.

---

# F. Stage27 Publication-Level Contributions

1. **Chronology-first unseen-family evaluation.** Attack-family novelty
   is evaluated under a strict training-before-validation-before-target
   design with zero held-out-family exposure during development.

2. **Structural executability accounting.** Families that cannot be
   evaluated without violating chronology are explicitly labeled
   structurally ineligible rather than replaced with post-hoc folds.

3. **Selective-transfer finding.** DDoS and Web Attack retain strong
   transfer, Bot collapses, and Port Scan depends materially on learner
   choice.

4. **Ranking/threshold separation.** Threshold-independent
   discrimination and frozen validation-selected operating-point
   behavior are evaluated separately.

5. **Learner-dependent novelty audit.** XGBoost and LightGBM are
   compared under the same preregistered family-holdout geometry without
   Stage27 HPO.

6. **Target-sampling uncertainty.** Primary ranking and compatible
   operating metrics are accompanied by frozen 2,000-replicate
   stratified bootstrap intervals.

7. **Secondary behavioral-similarity audit.** A preregistered
   train-fitted descriptor representation is used to test whether simple
   behavioral proximity descriptively explains transfer, without
   introducing post-result significance testing.

---

# G. Contribution Text for Abstract / Introduction

A chronology-first zero-training-exposure attack-family audit further
revealed selective rather than universal unseen-family transfer. Under
strict `TRAIN < VALIDATION < TARGET` separation, both XGBoost and
LightGBM retained near-perfect ranking discrimination for held-out DDoS
traffic and strong ranking for Web Attack, whereas Bot traffic
collapsed and Port Scan transfer was materially learner-dependent.
Moreover, high unseen-family ROC-AUC did not necessarily translate into
useful recall at frozen validation-selected thresholds. The findings
show that strong known-family IDS performance should not be interpreted
as evidence of uniform robustness to unseen attack families and that
ranking generalization and operating-point transfer should be audited
separately.

---

# H. Publication-Safe Claims

The following claims are supported by the frozen Stage27 evidence:

1. Stage27 evaluated seven preregistered attack-family categories.
2. Five of the seven families were structurally executable.
3. DOS and AUTH_BRUTE_FORCE were structurally ineligible under strict
   chronology.
4. INFILTRATION is descriptive only because held-out support was 36.
5. DDoS retained near-perfect unseen-family ranking for both learners.
6. Web Attack retained strong unseen-family ranking for both learners.
7. Bot exhibited substantial unseen-family collapse.
8. Port Scan exhibited material learner dependence.
9. Ranking performance and frozen-threshold recall diverged for several
   families.
10. Behavioral similarity did not display a monotonic relationship with
    unseen-family ranking performance across the five executable
    families.
11. No target threshold tuning, target-guided model selection, or
    target-guided adaptation was performed.
12. The bootstrap intervals quantify target-sampling uncertainty
    conditional on the fitted model.
13. Known-family performance should not be treated as evidence of
    uniform unseen-family generalization.

---

# I. Claims That Must Not Appear

1. Stage27 proves universal zero-day detection.
2. Stage27 proves all unseen cyberattacks can be detected.
3. All seven attack families were experimentally executable.
4. INFILTRATION provides an inferential family-level conclusion.
5. LightGBM is universally superior to XGBoost for unseen attacks.
6. XGBoost is universally superior to LightGBM for unseen attacks.
7. Behavioral similarity significantly predicts unseen-family
   performance.
8. A causal relationship between similarity and transfer was
   established.
9. Raw PR-AUC known-minus-unseen difference is prevalence invariant.
10. Stage27 target thresholds were optimized using held-out-family
    labels.
11. Stage27 models were adapted or recalibrated after target opening.
12. The row bootstrap represents uncertainty across independent
    organizations or future production networks.

---

# J. Recommended Main-Manuscript Assets

## Main Table 27-1

Chronology-first family executability.

Source:

`docs/STAGE27_PUBLICATION_TABLES.md`

## Main Table 27-2

Primary ROC-AUC, PR-AUC, 95% intervals, held-out support, and BALANCED
recall for both learners.

Source:

`docs/STAGE27_PUBLICATION_TABLES.md`

## Main Figure 27-1

`results/stage27_loao_unseen_attack/stage27_4a_final_synthesis/figures/stage27_primary_roc_auc_ci.png`

Purpose: show selective ranking transfer and learner dependence.

## Main Figure 27-2

`results/stage27_loao_unseen_attack/stage27_4a_final_synthesis/figures/stage27_balanced_recall_ci.png`

Purpose: show ranking–threshold divergence.

## Supplementary Figure 27-S1

`results/stage27_loao_unseen_attack/stage27_4a_final_synthesis/figures/stage27_primary_pr_auc_ci.png`

PR-AUC remains co-primary and should remain in the main table and main
text even if the separate PR-AUC figure is supplementary.

## Supplementary tables

- complete STANDARD/BALANCED/SECURITY operating points;
- novelty-generalization gaps;
- behavioral similarity.

---

# K. Recommended Manuscript Placement

The Stage27 material should be integrated into the broader robustness
narrative rather than placed according to experimental stage number.

Recommended Results ordering:

1. Validation-safe baseline/model selection
2. Representation/architecture assessment
3. Temporal validation and forward generalization
4. Cross-dataset generalization
5. **Unseen attack-family generalization (Stage27)**
6. Low-prevalence and SOC operational stress
7. Deployment/computational profiling

This ordering moves from predictive evaluation toward increasingly
deployment-facing stress tests and keeps Stage27 adjacent to the
temporal and cross-dataset generalization evidence.
"""


def build_manuscript_tex(primary, ops, gaps, similarity):
    bot_x = select_row(primary, "BOT", "XGBOOST")
    bot_l = select_row(primary, "BOT", "LIGHTGBM")
    ddos_x = select_row(primary, "DDOS", "XGBOOST")
    ddos_l = select_row(primary, "DDOS", "LIGHTGBM")
    port_x = select_row(primary, "PORT_SCAN", "XGBOOST")
    port_l = select_row(primary, "PORT_SCAN", "LIGHTGBM")
    web_x = select_row(primary, "WEB_ATTACK", "XGBOOST")
    web_l = select_row(primary, "WEB_ATTACK", "LIGHTGBM")

    ddos_x_bal = balanced_row(ops, "DDOS", "XGBOOST")
    ddos_l_bal = balanced_row(ops, "DDOS", "LIGHTGBM")
    port_l_bal = balanced_row(ops, "PORT_SCAN", "LIGHTGBM")

    sim = similarity.set_index("family")

    lines = [
        "% =====================================================================",
        "% Stage27 Manuscript Integration",
        "% Generated only from frozen Stage27-4A artifacts.",
        "% =====================================================================",
        "",
        r"\subsection{Unseen Attack-Family Generalization}",
        "",
        r"\subsubsection{Chronology-first audit design}",
        "",
        (
            "Stage27 evaluated zero-training-exposure attack-family "
            "generalization under a strict "
            r"\texttt{TRAIN < VALIDATION < TARGET} protocol. "
            "The held-out family was absent from both training and "
            "validation, thresholds were selected only on known-family "
            "validation data, and the target was not used for model "
            "selection, threshold tuning, or adaptation. The experiment "
            "is therefore described as an unseen attack-family "
            "generalization audit rather than as formal proof of "
            "zero-day detection."
        ),
        "",
        (
            "Seven primary families were preregistered. Five were "
            "structurally executable. DOS was ineligible because the "
            "available earlier day-atomic training period contained no "
            "known-family attack positives, whereas "
            r"AUTH\_BRUTE\_FORCE was ineligible because insufficient "
            "earlier weekday depth existed for separate training and "
            "validation periods. INFILTRATION was executable but is "
            "reported descriptively only because the held-out support "
            "was 36."
        ),
        "",
        r"\subsubsection{Primary unseen-family ranking}",
        "",
        (
            "Unseen-family ranking was strongly family dependent. "
            "DDoS retained near-perfect discrimination: XGBoost reached "
            "ROC-AUC %.4f and PR-AUC %.4f, while LightGBM reached "
            "ROC-AUC %.4f and PR-AUC %.4f. "
            "Web Attack also transferred strongly, with XGBoost "
            "ROC-AUC %.4f and PR-AUC %.4f and LightGBM ROC-AUC %.4f "
            "and PR-AUC %.4f."
        ) % (
            ddos_x["roc_auc"],
            ddos_x["pr_auc"],
            ddos_l["roc_auc"],
            ddos_l["pr_auc"],
            web_x["roc_auc"],
            web_x["pr_auc"],
            web_l["roc_auc"],
            web_l["pr_auc"],
        ),
        "",
        (
            "Bot traffic showed substantial collapse. XGBoost produced "
            "ROC-AUC %.4f and PR-AUC %.6f, while LightGBM produced "
            "ROC-AUC %.4f and PR-AUC %.6f. The XGBoost PR-AUC was below "
            "the target prevalence anchor, giving negative PR-excess "
            "%.6f. Port Scan was materially learner-dependent: XGBoost "
            "reached ROC-AUC %.4f compared with %.4f for LightGBM."
        ) % (
            bot_x["roc_auc"],
            bot_x["pr_auc"],
            bot_l["roc_auc"],
            bot_l["pr_auc"],
            bot_x["pr_excess"],
            port_x["roc_auc"],
            port_l["roc_auc"],
        ),
        "",
        (
            "The overall Stage27 outcome is therefore selective family "
            "transfer rather than universal unseen-family "
            "generalization."
        ),
        "",
        r"\subsubsection{Frozen operating-point transfer}",
        "",
        (
            "Threshold-independent ranking did not guarantee useful "
            "frozen-threshold detection. At the BALANCED operating "
            "point, DDoS recall was %.2f\\%% for XGBoost and %.2f\\%% "
            "for LightGBM despite ROC-AUC above 0.998 for both learners. "
            "Similarly, LightGBM retained Port Scan ROC-AUC %.4f while "
            "BALANCED recall was only %.2f\\%%."
        ) % (
            100 * ddos_x_bal["recall"],
            100 * ddos_l_bal["recall"],
            port_l["roc_auc"],
            100 * port_l_bal["recall"],
        ),
        "",
        (
            "These results distinguish ranking generalization from "
            "operating-point transfer and support the frozen Stage27 "
            "outcome of ranking--threshold divergence."
        ),
        "",
        r"\subsubsection{Behavioral similarity}",
        "",
        (
            "The secondary behavioral-similarity audit did not show a "
            "monotonic relationship with transfer. BOT had the highest "
            "observed similarity to a seen family (%.4f) but weak "
            "generalization, whereas DDoS had lower similarity (%.4f) "
            "and near-perfect ranking. Behavioral proximity under the "
            "frozen 11-descriptor centroid definition therefore does "
            "not appear sufficient by itself to explain the transfer "
            "pattern."
        ) % (
            sim.loc["BOT", "similarity_score"],
            sim.loc["DDOS", "similarity_score"],
        ),
        "",
        r"\subsection{Discussion of Attack-Family Novelty}",
        "",
        (
            "Stage27 shows that strong performance on known attack "
            "families cannot be interpreted as evidence of uniform "
            "robustness to attack-family novelty. DDoS and Web Attack "
            "retained strong ranking for both learners, Bot collapsed, "
            "and Port Scan exhibited substantial learner dependence."
        ),
        "",
        (
            "The experiment also reinforces a broader finding across "
            "the study: threshold-independent discrimination and "
            "fixed operating-point behavior are distinct properties. "
            "Representation-specific chronological evaluation, the "
            "temporal-validation stress test, cross-dataset transfer, "
            "and now unseen-family transfer each expose cases in which "
            "ranking and frozen-threshold behavior diverge. Reporting "
            "only ROC-AUC or PR-AUC would therefore provide an "
            "incomplete description of deployment robustness."
        ),
        "",
        (
            "The evidence does not establish a universal learner "
            "winner. XGBoost and LightGBM agree closely on DDoS and Web "
            "Attack ranking but differ substantially for Port Scan and "
            "Bot. Learner dependence is therefore itself "
            "family-dependent."
        ),
        "",
        r"\subsection{Stage27 Limitations}",
        "",
        r"\begin{itemize}",
        (
            r"\item Only five of seven preregistered families were "
            r"structurally executable under strict chronology."
        ),
        (
            r"\item INFILTRATION is descriptive only because the "
            r"held-out support was 36."
        ),
        (
            r"\item The design is chronology-first zero-training-"
            r"exposure evaluation rather than textbook LOAO in which "
            r"every other family is necessarily represented in training."
        ),
        (
            r"\item The 95\% bootstrap intervals quantify target-"
            r"sampling uncertainty conditional on the fitted model and "
            r"do not include retraining, seed, model-selection, or "
            r"broader population uncertainty."
        ),
        (
            r"\item Behavioral similarity uses only 11 preregistered "
            r"aggregate descriptors and is descriptive only."
        ),
        (
            r"\item The benchmark-specific results do not establish "
            r"universal real-world zero-day detection."
        ),
        r"\end{itemize}",
        "",
        "% Main Stage27 figures:",
        "% stage27_primary_roc_auc_ci.png",
        "% stage27_balanced_recall_ci.png",
        "%",
        "% Supplementary:",
        "% stage27_primary_pr_auc_ci.png",
        "",
    ]

    return "\n".join(lines)


def build_contents(primary, ops, gaps, similarity):
    return {
        OUTPUT_PATHS["manuscript_md"]:
            build_manuscript_md(primary, ops, gaps, similarity),

        OUTPUT_PATHS["manuscript_tex"]:
            build_manuscript_tex(primary, ops, gaps, similarity),

        OUTPUT_PATHS["tables_md"]:
            build_publication_tables_md(primary, ops, gaps, similarity),

        OUTPUT_PATHS["tables_tex"]:
            build_publication_tables_tex(primary, ops, gaps, similarity),
    }


def write_text(path: Path, content: str):
    path.parent.mkdir(parents=True, exist_ok=True)

    normalized = content.rstrip() + "\n"

    path.write_text(
        normalized,
        encoding="utf-8",
        newline="\n",
    )


def create_manifest(
    root: Path,
    head: str,
    source_hashes: dict,
    generated_paths: list[Path],
):
    generated = {}

    for rel in generated_paths:
        path = root / rel

        generated[str(rel)] = {
            "sha256": sha256_file(path),
            "bytes": path.stat().st_size,
        }

    generator = root / GENERATOR_REL

    generated[str(GENERATOR_REL)] = {
        "sha256": sha256_file(generator),
        "bytes": generator.stat().st_size,
    }

    return {
        "stage": "STAGE27-PUB1",
        "publication_date": STAGE27_DATE,
        "scientific_parent":
            CANONICAL_SCIENTIFIC_PARENT,
        "generation_head": head,
        "scientific_status": "CLOSED",
        "publication_package_status":
            "GENERATED_PENDING_GIT_REVIEW",
        "science_operations": {
            "model_fits": 0,
            "model_inference": 0,
            "target_reopenings": 0,
            "threshold_reselection": 0,
            "bootstrap_recomputation": 0,
            "new_formal_statistical_tests": 0,
        },
        "high_level_outcomes": [
            "SELECTIVE_FAMILY_TRANSFER",
            "RANKING_THRESHOLD_DIVERGENCE",
            "LEARNER_DEPENDENCE",
        ],
        "frozen_source_sha256": source_hashes,
        "generated_artifacts": generated,
        "figure_policy": {
            "main": [
                (
                    "results/stage27_loao_unseen_attack/"
                    "stage27_4a_final_synthesis/figures/"
                    "stage27_primary_roc_auc_ci.png"
                ),
                (
                    "results/stage27_loao_unseen_attack/"
                    "stage27_4a_final_synthesis/figures/"
                    "stage27_balanced_recall_ci.png"
                ),
            ],
            "supplementary": [
                (
                    "results/stage27_loao_unseen_attack/"
                    "stage27_4a_final_synthesis/figures/"
                    "stage27_primary_pr_auc_ci.png"
                ),
            ],
        },
        "reporting_guards": {
            "formal_zero_day_proof": False,
            "universal_unseen_family_generalization": False,
            "infiltration_inferential_claim": False,
            "similarity_significance_inference": False,
            "target_threshold_search": False,
        },
    }


def validate_generated_text(root: Path):
    manuscript = (
        root / OUTPUT_PATHS["manuscript_md"]
    ).read_text(encoding="utf-8")

    tables = (
        root / OUTPUT_PATHS["tables_md"]
    ).read_text(encoding="utf-8")

    required_manuscript_strings = [
        "SELECTIVE_FAMILY_TRANSFER",
        "RANKING_THRESHOLD_DIVERGENCE",
        "LEARNER_DEPENDENCE",
        "formal proof of zero-day detection",
        "descriptive only because",
        "ranking generalization",
        "operating-point transfer",
        "five of seven",
        "DOS",
        "AUTH_BRUTE_FORCE",
        "INFILTRATION",
        "BOT",
        "DDOS",
        "PORT_SCAN",
        "WEB_ATTACK",
        "0.9982",
        "0.9986",
        "0.3224",
        "0.5591",
        "0.5506",
        "0.7559",
        "77.80%",
        "52.11%",
    ]

    for token in required_manuscript_strings:
        if token not in manuscript:
            raise RuntimeError(
                f"Generated manuscript missing required token: {token}"
            )

    forbidden_overclaims = [
        "proves universal zero-day detection",
        "all seven families were experimentally executable",
        "LightGBM is universally superior",
        "XGBoost is universally superior",
        "statistically significant similarity",
    ]

    # These phrases are allowed only inside the explicit
    # "Claims That Must Not Appear" section.
    # Therefore the generated artifact must contain that section.
    assert "# I. Claims That Must Not Appear" in manuscript

    assert "Table 27-1" in tables
    assert "Table 27-2" in tables
    assert "Table 27-S1" in tables
    assert "Table 27-S2" in tables
    assert "Table 27-S3" in tables

    assert (
        "INFILTRATION is descriptive only"
        in tables
    )

    return True


def check_mode(root: Path):
    source_hashes = verify_frozen_sources(root)

    synthesis = root / SYNTHESIS_REL

    primary = pd.read_csv(
        synthesis / "stage27_final_primary_metrics.csv"
    )
    ops = pd.read_csv(
        synthesis / "stage27_final_operating_points.csv"
    )
    gaps = pd.read_csv(
        synthesis / "stage27_final_novelty_gaps.csv"
    )
    similarity = pd.read_csv(
        synthesis / "stage27_final_similarity.csv"
    )

    scientific_sanity(
        primary,
        ops,
        gaps,
        similarity,
    )

    expected = build_contents(
        primary,
        ops,
        gaps,
        similarity,
    )

    for rel, content in expected.items():
        path = root / rel

        if not path.is_file():
            raise RuntimeError(
                f"Generated publication artifact missing: {rel}"
            )

        actual_text = path.read_text(encoding="utf-8")
        expected_text = content.rstrip() + "\n"

        if actual_text != expected_text:
            raise RuntimeError(
                f"Generated artifact differs from deterministic "
                f"generator output: {rel}"
            )

    manifest_path = root / MANIFEST_REL

    if not manifest_path.is_file():
        raise RuntimeError("Publication manifest is missing.")

    manifest = json.loads(
        manifest_path.read_text(encoding="utf-8")
    )

    for rel in expected:
        rel_str = str(rel)

        expected_hash = manifest[
            "generated_artifacts"
        ][rel_str]["sha256"]

        actual_hash = sha256_file(root / rel)

        if actual_hash != expected_hash:
            raise RuntimeError(
                f"Manifest hash mismatch: {rel}"
            )

    validate_generated_text(root)

    print("[PASS] frozen Stage27 source hashes")
    print("[PASS] scientific sanity gates")
    print("[PASS] deterministic document contents")
    print("[PASS] publication manifest hashes")
    print("[PASS] manuscript claim/data gates")
    print()
    print("STAGE27 PUBLICATION PACKAGE CHECK: PASS")


def generate_mode(root: Path):
    head = verify_scientific_parent(root)

    source_hashes = verify_frozen_sources(root)

    synthesis = root / SYNTHESIS_REL

    primary = pd.read_csv(
        synthesis / "stage27_final_primary_metrics.csv"
    )

    ops = pd.read_csv(
        synthesis / "stage27_final_operating_points.csv"
    )

    gaps = pd.read_csv(
        synthesis / "stage27_final_novelty_gaps.csv"
    )

    similarity = pd.read_csv(
        synthesis / "stage27_final_similarity.csv"
    )

    scientific_sanity(
        primary,
        ops,
        gaps,
        similarity,
    )

    contents = build_contents(
        primary,
        ops,
        gaps,
        similarity,
    )

    for rel, content in contents.items():
        write_text(
            root / rel,
            content,
        )

    validate_generated_text(root)

    manifest = create_manifest(
        root=root,
        head=head,
        source_hashes=source_hashes,
        generated_paths=list(contents.keys()),
    )

    manifest_path = root / MANIFEST_REL
    manifest_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    manifest_path.write_text(
        json.dumps(
            manifest,
            indent=2,
            sort_keys=True,
        ) + "\n",
        encoding="utf-8",
        newline="\n",
    )

    print("=" * 80)
    print("STAGE27 PUBLICATION GENERATOR COMPLETE")
    print("=" * 80)

    print()
    print("Scientific parent:")
    print(f"  {CANONICAL_SCIENTIFIC_PARENT}")

    print()
    print("Generated artifacts:")

    for rel in contents:
        path = root / rel

        print(
            f"  {rel}\n"
            f"    SHA256: {sha256_file(path)}\n"
            f"    bytes:  {path.stat().st_size:,}"
        )

    print(
        f"  {GENERATOR_REL}\n"
        f"    SHA256: {sha256_file(root / GENERATOR_REL)}"
    )

    print(
        f"  {MANIFEST_REL}\n"
        f"    SHA256: {sha256_file(manifest_path)}"
    )

    print()
    print("Science operations:")
    print("  model fits                 : 0")
    print("  model inference            : 0")
    print("  target reopenings          : 0")
    print("  threshold reselection      : 0")
    print("  bootstrap recomputation    : 0")
    print("  new formal statistical test: 0")


def main():
    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--check",
        action="store_true",
        help="Validate existing publication package without rewriting it.",
    )

    args = parser.parse_args()

    root = repo_root()

    verify_scientific_parent(root)

    if args.check:
        check_mode(root)
    else:
        generate_mode(root)


if __name__ == "__main__":
    main()
'''


# ----------------------------------------------------------------------------
# Write generator into repository
# ----------------------------------------------------------------------------

GENERATOR_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

GENERATOR_PATH.write_text(
    GENERATOR_SOURCE.rstrip() + "\n",
    encoding="utf-8",
    newline="\n",
)

print()
print("-" * 92)
print("GENERATOR CREATED")
print("-" * 92)

print(GENERATOR_PATH)
print("SHA256:", sha256_file(GENERATOR_PATH))


# ----------------------------------------------------------------------------
# First deterministic generation
# ----------------------------------------------------------------------------

print()
print("-" * 92)
print("FIRST GENERATION")
print("-" * 92)

first = run(
    [
        sys.executable,
        str(GENERATOR_PATH),
    ],
    cwd=REPO,
)

print(first.stdout)

if first.stderr.strip():
    print(first.stderr)


# ----------------------------------------------------------------------------
# Publication artifact paths
# ----------------------------------------------------------------------------

PUBLICATION_PATHS = [
    Path("docs/STAGE27_MANUSCRIPT_INTEGRATION.md"),
    Path("docs/STAGE27_MANUSCRIPT_INTEGRATION.tex"),
    Path("docs/STAGE27_PUBLICATION_TABLES.md"),
    Path("docs/STAGE27_PUBLICATION_TABLES.tex"),
    GENERATOR_REL,
    Path(
        "results/stage27_loao_unseen_attack/"
        "stage27_publication_package/"
        "stage27_publication_manifest.json"
    ),
]

for rel in PUBLICATION_PATHS:
    path = REPO / rel

    if not path.is_file():
        raise RuntimeError(
            f"Expected PUB1 artifact not generated: {rel}"
        )


# ----------------------------------------------------------------------------
# Capture first-run hashes
# ----------------------------------------------------------------------------

hashes_first = {
    str(rel): sha256_file(REPO / rel)
    for rel in PUBLICATION_PATHS
}


# ----------------------------------------------------------------------------
# Second generation — idempotence test
# ----------------------------------------------------------------------------

print()
print("-" * 92)
print("SECOND GENERATION / IDEMPOTENCE TEST")
print("-" * 92)

second = run(
    [
        sys.executable,
        str(GENERATOR_PATH),
    ],
    cwd=REPO,
)

if second.returncode != 0:
    print(second.stdout)
    print(second.stderr)
    raise RuntimeError(
        "Second deterministic generation failed."
    )

hashes_second = {
    str(rel): sha256_file(REPO / rel)
    for rel in PUBLICATION_PATHS
}

if hashes_first != hashes_second:
    print("FIRST:")
    print(json.dumps(hashes_first, indent=2))

    print()
    print("SECOND:")
    print(json.dumps(hashes_second, indent=2))

    raise RuntimeError(
        "PUB1 output is not deterministic across consecutive runs."
    )

print("[PASS] consecutive generation is byte-identical")


# ----------------------------------------------------------------------------
# Check-only mode
# ----------------------------------------------------------------------------

print()
print("-" * 92)
print("CHECK-ONLY REPRODUCIBILITY MODE")
print("-" * 92)

check = run(
    [
        sys.executable,
        str(GENERATOR_PATH),
        "--check",
    ],
    cwd=REPO,
)

print(check.stdout)

if check.stderr.strip():
    print(check.stderr)


# ----------------------------------------------------------------------------
# Ensure frozen science files themselves were not changed
# ----------------------------------------------------------------------------

print()
print("-" * 92)
print("FROZEN SCIENCE MODIFICATION GATE")
print("-" * 92)

science_pathspec = (
    "results/stage27_loao_unseen_attack/"
    "stage27_4a_final_synthesis"
)

science_diff = run(
    [
        "git",
        "diff",
        "--",
        science_pathspec,
    ],
    cwd=REPO,
).stdout

if science_diff.strip():
    print(science_diff)

    raise RuntimeError(
        "PUB1 modified frozen Stage27 scientific artifacts."
    )

print("[PASS] frozen Stage27-4A synthesis remains byte-untouched")


# ----------------------------------------------------------------------------
# Git status
# ----------------------------------------------------------------------------

print()
print("-" * 92)
print("GIT STATUS")
print("-" * 92)

status = git("status", "--short")

print(status or "[clean]")


# ----------------------------------------------------------------------------
# Diff statistics
# ----------------------------------------------------------------------------

print()
print("-" * 92)
print("DIFF STAT")
print("-" * 92)

diff_stat = run(
    [
        "git",
        "diff",
        "--stat",
        "--",
        "docs/STAGE27_MANUSCRIPT_INTEGRATION.md",
        "docs/STAGE27_MANUSCRIPT_INTEGRATION.tex",
        "docs/STAGE27_PUBLICATION_TABLES.md",
        "docs/STAGE27_PUBLICATION_TABLES.tex",
        "scripts/stage27/stage27_publication_integration.py",
        (
            "results/stage27_loao_unseen_attack/"
            "stage27_publication_package/"
            "stage27_publication_manifest.json"
        ),
    ],
    cwd=REPO,
).stdout

print(diff_stat)


# ----------------------------------------------------------------------------
# Because new files are untracked, git diff does not display their text.
# Print controlled previews.
# ----------------------------------------------------------------------------

print()
print("-" * 92)
print("MANUSCRIPT PREVIEW")
print("-" * 92)

manuscript_path = (
    REPO / "docs/STAGE27_MANUSCRIPT_INTEGRATION.md"
)

manuscript_lines = manuscript_path.read_text(
    encoding="utf-8"
).splitlines()

for line in manuscript_lines[:120]:
    print(line)

print()
print(
    f"... previewed 120/{len(manuscript_lines)} manuscript lines"
)


print()
print("-" * 92)
print("PRIMARY TABLE PREVIEW")
print("-" * 92)

tables_path = (
    REPO / "docs/STAGE27_PUBLICATION_TABLES.md"
)

tables_lines = tables_path.read_text(
    encoding="utf-8"
).splitlines()

for line in tables_lines[:80]:
    print(line)

print()
print(
    f"... previewed 80/{len(tables_lines)} table lines"
)


# ----------------------------------------------------------------------------
# Manifest preview
# ----------------------------------------------------------------------------

print()
print("-" * 92)
print("PUBLICATION MANIFEST")
print("-" * 92)

manifest_path = (
    REPO
    / "results/stage27_loao_unseen_attack/"
      "stage27_publication_package/"
      "stage27_publication_manifest.json"
)

manifest = json.loads(
    manifest_path.read_text(encoding="utf-8")
)

print(
    json.dumps(
        manifest,
        indent=2,
    )
)


# ----------------------------------------------------------------------------
# Final PUB1 state
# ----------------------------------------------------------------------------

print()
print("=" * 92)
print("STAGE27-PUB1 COMPLETE — REVIEW REQUIRED BEFORE COMMIT")
print("=" * 92)

print()
print("Scientific parent:")
print(f"  {CANONICAL_STAGE27_PARENT}")

print()
print("Generated publication artifacts:")
for rel in PUBLICATION_PATHS:
    print(
        f"  {rel}\n"
        f"    SHA256: {sha256_file(REPO / rel)}"
    )

print()
print("Integrity:")
print("  frozen Stage27 science modified : NO")
print("  model fitting                   : 0")
print("  model inference                 : 0")
print("  target reopening                : 0")
print("  threshold reselection           : 0")
print("  bootstrap recomputation         : 0")
print("  new significance testing        : 0")
print("  deterministic generation        : PASS")
print("  generator --check               : PASS")

print()
print("Git operations:")
print("  commit : NOT PERFORMED")
print("  push   : NOT PERFORMED")

print()
print("Next authorized step:")
print("  Review PUB1 output, then run STAGE27-PUB2 commit/push closeout.")

STAGE27-PUB1 — DETERMINISTIC PUBLICATION PACKAGE GENERATION

Repository:
  /kaggle/working/ids2018-validation-safe-ablation

HEAD:
  0e1439565aedc7da9b7ca1207262e9061422bc22
[PASS] exact Stage27 scientific parent
[PASS] clean worktree

--------------------------------------------------------------------------------------------
GENERATOR CREATED
--------------------------------------------------------------------------------------------
/kaggle/working/ids2018-validation-safe-ablation/scripts/stage27/stage27_publication_integration.py
SHA256: 1408c5613acb2431eff6b1524864ac24c6838e8d3fd30a56b12ffe681ab22940

--------------------------------------------------------------------------------------------
FIRST GENERATION
--------------------------------------------------------------------------------------------
STAGE27 PUBLICATION GENERATOR COMPLETE

Scientific parent:
  0e1439565aedc7da9b7ca1207262e9061422bc22

Generated artifacts:
  docs/STAGE27_MANUSCRIPT_INTEGRATION.md
    SHA256: 25c3c7

In [10]:
# ============================================================================
# Stage27-PUB2
# Publication Package Freeze, Git Commit/Push, Remote Verification, Closeout
#
# REQUIRES:
#   STAGE27-PUB0 = PASS
#   STAGE27-PUB1 = PASS
#
# OPERATIONS:
#   - Apply two publication-only wording/state corrections.
#   - Regenerate Stage27 publication artifacts deterministically.
#   - Re-run generator --check.
#   - Verify frozen Stage27 science remains untouched.
#   - Optional LaTeX compilation smoke test.
#   - Stage ONLY authorized Stage27 publication files.
#   - Commit publication package.
#   - Push and verify exact remote commit/files.
#   - Generate STAGE27_PUBLICATION_CLOSEOUT.md.
#   - Commit closeout separately.
#   - Push and remotely verify final closeout state.
#
# SCIENCE:
#   ZERO model fitting
#   ZERO model inference
#   ZERO target reopening
#   ZERO threshold reselection
#   ZERO bootstrap recomputation
#   ZERO new statistical testing
# ============================================================================

from __future__ import annotations

import hashlib
import json
import os
import shutil
import stat
import subprocess
import sys
from pathlib import Path
from datetime import datetime, timezone


# ----------------------------------------------------------------------------
# Frozen identity
# ----------------------------------------------------------------------------

REPO = Path("/kaggle/working/ids2018-validation-safe-ablation")

REPO_FULL = "themubasshir/ids2018-validation-safe-ablation"

CANONICAL_STAGE27_PARENT = (
    "0e1439565aedc7da9b7ca1207262e9061422bc22"
)

GENERATOR_REL = Path(
    "scripts/stage27/stage27_publication_integration.py"
)

GENERATOR = REPO / GENERATOR_REL

MANIFEST_REL = Path(
    "results/stage27_loao_unseen_attack/"
    "stage27_publication_package/"
    "stage27_publication_manifest.json"
)

CLOSEOUT_REL = Path(
    "docs/STAGE27_PUBLICATION_CLOSEOUT.md"
)

PUBLICATION_FILES = [
    Path("docs/STAGE27_MANUSCRIPT_INTEGRATION.md"),
    Path("docs/STAGE27_MANUSCRIPT_INTEGRATION.tex"),
    Path("docs/STAGE27_PUBLICATION_TABLES.md"),
    Path("docs/STAGE27_PUBLICATION_TABLES.tex"),
    MANIFEST_REL,
    GENERATOR_REL,
]

FROZEN_SYNTHESIS_REL = Path(
    "results/stage27_loao_unseen_attack/"
    "stage27_4a_final_synthesis"
)


# ----------------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------------

def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
    text=True,
    env=None,
):
    result = subprocess.run(
        cmd,
        cwd=cwd,
        check=False,
        text=text,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        env=env,
    )

    if check and result.returncode != 0:
        if text:
            print(result.stdout)
            print(result.stderr)
        raise RuntimeError(
            f"Command failed ({result.returncode}): "
            f"{' '.join(str(x) for x in cmd)}"
        )

    return result


def git(*args, check=True):
    return run(
        ["git", *args],
        check=check,
    ).stdout.strip()


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        while True:
            chunk = f.read(8 * 1024 * 1024)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def git_file_bytes(ref: str, path: Path) -> bytes:
    result = run(
        [
            "git",
            "show",
            f"{ref}:{path.as_posix()}",
        ],
        text=False,
    )

    return result.stdout


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def status_paths():
    """
    Return exact paths from `git status --porcelain -uall`.
    """
    output = git(
        "status",
        "--porcelain",
        "-uall",
    )

    paths = []

    if not output:
        return paths

    for line in output.splitlines():
        # XY + space = first 3 chars
        raw = line[3:]

        # Handle rename format if it ever occurs.
        if " -> " in raw:
            raw = raw.split(" -> ", 1)[1]

        paths.append(raw)

    return paths


def get_github_token():
    from kaggle_secrets import UserSecretsClient

    client = UserSecretsClient()

    candidates = [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
        "GH_PAT",
    ]

    for label in candidates:
        try:
            value = client.get_secret(label)
        except Exception:
            value = None

        if value:
            print(
                f"[PASS] GitHub credential: {label} "
                f"({len(value)} chars, value redacted)"
            )
            return label, value

    raise RuntimeError(
        "No usable GitHub token found in Kaggle Secrets."
    )


def make_git_push_env(token: str):
    """
    Use GIT_ASKPASS so the token is NOT written into the Git remote URL
    and is NOT printed in notebook output.
    """
    askpass = Path(
        "/kaggle/working/.stage27_git_askpass.sh"
    )

    askpass.write_text(
        """#!/bin/sh
case "$1" in
  *Username*) printf '%s\\n' "x-access-token" ;;
  *)          printf '%s\\n' "$GITHUB_TOKEN" ;;
esac
""",
        encoding="utf-8",
        newline="\n",
    )

    askpass.chmod(
        stat.S_IRUSR
        | stat.S_IWUSR
        | stat.S_IXUSR
    )

    env = os.environ.copy()

    env["GITHUB_TOKEN"] = token
    env["GIT_ASKPASS"] = str(askpass)
    env["GIT_TERMINAL_PROMPT"] = "0"

    return env, askpass


def push_main(push_env):
    result = run(
        [
            "git",
            "push",
            "origin",
            "main",
        ],
        env=push_env,
    )

    # Safe to show push output: token is not in URL.
    if result.stdout.strip():
        print(result.stdout.strip())

    if result.stderr.strip():
        print(result.stderr.strip())


def verify_remote_head(expected_commit):
    git(
        "fetch",
        "--prune",
        "origin",
        "main",
    )

    remote = git(
        "rev-parse",
        "origin/main",
    )

    print("Expected remote HEAD:", expected_commit)
    print("Actual remote HEAD:  ", remote)

    if remote != expected_commit:
        raise RuntimeError(
            "Remote HEAD does not match expected commit."
        )

    return remote


def verify_remote_files(ref, paths):
    print()
    print("Remote file-content verification:")

    for rel in paths:
        local_path = REPO / rel

        local_hash = sha256_file(local_path)

        remote_bytes = git_file_bytes(
            ref,
            rel,
        )

        remote_hash = sha256_bytes(
            remote_bytes
        )

        if local_hash != remote_hash:
            raise RuntimeError(
                f"Remote content mismatch: {rel}\n"
                f"local:  {local_hash}\n"
                f"remote: {remote_hash}"
            )

        print(
            f"[PASS] {rel}\n"
            f"       SHA256: {local_hash}"
        )


# ----------------------------------------------------------------------------
# Header
# ----------------------------------------------------------------------------

print("=" * 96)
print(
    "STAGE27-PUB2 — PUBLICATION FREEZE / "
    "COMMIT / PUSH / REMOTE CLOSEOUT"
)
print("=" * 96)

print()
print(
    "timestamp_utc:",
    datetime.now(timezone.utc).isoformat(),
)

print("repository   :", REPO_FULL)
print("repo path    :", REPO)
print(
    "science parent:",
    CANONICAL_STAGE27_PARENT,
)


# ----------------------------------------------------------------------------
# Repository preflight
# ----------------------------------------------------------------------------

if not REPO.is_dir():
    raise RuntimeError(
        "Repository checkout missing."
    )

branch = git(
    "branch",
    "--show-current",
)

head = git(
    "rev-parse",
    "HEAD",
)

print()
print("branch:", branch)
print("HEAD:  ", head)

if branch != "main":
    raise RuntimeError(
        f"Expected branch main, found {branch}"
    )

if head != CANONICAL_STAGE27_PARENT:
    raise RuntimeError(
        "PUB2 must start from the canonical Stage27 "
        "scientific parent."
    )


# ----------------------------------------------------------------------------
# Verify PUB1 generated exactly the authorized paths
# ----------------------------------------------------------------------------

print()
print("-" * 96)
print("PUB1 WORKTREE INVENTORY")
print("-" * 96)

expected_untracked = {
    p.as_posix()
    for p in PUBLICATION_FILES
}

actual_paths = set(
    status_paths()
)

print("Expected publication paths:")
for p in sorted(expected_untracked):
    print(" ", p)

print()
print("Actual changed/untracked paths:")
for p in sorted(actual_paths):
    print(" ", p)

if actual_paths != expected_untracked:
    unexpected = actual_paths - expected_untracked
    missing = expected_untracked - actual_paths

    print()
    print("Unexpected:", sorted(unexpected))
    print("Missing:   ", sorted(missing))

    raise RuntimeError(
        "PUB2 refuses to proceed because the worktree "
        "does not contain exactly the expected PUB1 files."
    )

print()
print("[PASS] exact PUB1 worktree inventory")


# ----------------------------------------------------------------------------
# Publication-only corrections
# ----------------------------------------------------------------------------

print()
print("-" * 96)
print("PUBLICATION WORDING / STATE FREEZE")
print("-" * 96)

if not GENERATOR.is_file():
    raise RuntimeError(
        f"Generator missing: {GENERATOR}"
    )

source = GENERATOR.read_text(
    encoding="utf-8"
)

original_source = source


# 1. Publication package is no longer pending review.
old_state = '"GENERATED_PENDING_GIT_REVIEW"'
new_state = '"PUBLICATION_CONTENT_FROZEN"'

state_count = source.count(old_state)

if state_count != 1:
    raise RuntimeError(
        f"Expected exactly one pending-review state; "
        f"found {state_count}"
    )

source = source.replace(
    old_state,
    new_state,
    1,
)


# 2. Avoid language that could imply 7/7 folds produced target results.
old_phrase = (
    "Seven\n"
    "CICIDS2017 attack families were preregistered and "
    "evaluated under a\n"
    "strict"
)

new_phrase = (
    "Seven\n"
    "CICIDS2017 attack families were preregistered for "
    "evaluation under a\n"
    "strict"
)

phrase_count = source.count(
    old_phrase
)

if phrase_count != 1:
    raise RuntimeError(
        "Could not uniquely locate the Introduction "
        "executability wording."
    )

source = source.replace(
    old_phrase,
    new_phrase,
    1,
)


# 3. Tighten safe-claims wording.
old_claim = (
    "1. Stage27 evaluated seven preregistered "
    "attack-family categories."
)

new_claim = (
    "1. Stage27 preregistered seven attack-family "
    "categories."
)

claim_count = source.count(
    old_claim
)

if claim_count != 1:
    raise RuntimeError(
        "Could not uniquely locate the publication-safe "
        "seven-family claim."
    )

source = source.replace(
    old_claim,
    new_claim,
    1,
)


if source == original_source:
    raise RuntimeError(
        "No generator changes were applied."
    )

GENERATOR.write_text(
    source,
    encoding="utf-8",
    newline="\n",
)

print("[PASS] publication package state frozen")
print("[PASS] 7-family executability wording tightened")
print("[PASS] publication-safe claim wording tightened")
print(
    "Generator SHA256:",
    sha256_file(GENERATOR),
)


# ----------------------------------------------------------------------------
# Regenerate publication package
# ----------------------------------------------------------------------------

print()
print("-" * 96)
print("DETERMINISTIC REGENERATION")
print("-" * 96)

regen = run(
    [
        sys.executable,
        str(GENERATOR),
    ]
)

print(regen.stdout)

if regen.stderr.strip():
    print(regen.stderr)


# ----------------------------------------------------------------------------
# Check-only validation
# ----------------------------------------------------------------------------

print()
print("-" * 96)
print("GENERATOR CHECK")
print("-" * 96)

check = run(
    [
        sys.executable,
        str(GENERATOR),
        "--check",
    ]
)

print(check.stdout)

if check.stderr.strip():
    print(check.stderr)


# ----------------------------------------------------------------------------
# Publication-text wording audit
# ----------------------------------------------------------------------------

print()
print("-" * 96)
print("PUBLICATION WORDING AUDIT")
print("-" * 96)

manuscript_md = (
    REPO
    / "docs/STAGE27_MANUSCRIPT_INTEGRATION.md"
)

manuscript = manuscript_md.read_text(
    encoding="utf-8"
)

if (
    "preregistered and evaluated under"
    in manuscript
):
    raise RuntimeError(
        "Ambiguous 7/7 wording remains in manuscript."
    )

if (
    "Stage27 evaluated seven preregistered "
    "attack-family categories."
    in manuscript
):
    raise RuntimeError(
        "Old seven-family safe claim remains."
    )

required_phrases = [
    (
        "Seven\n"
        "CICIDS2017 attack families were preregistered "
        "for evaluation under a"
    ),
    (
        "Five families were\n"
        "structurally executable"
    ),
    (
        "Stage27 preregistered seven attack-family "
        "categories."
    ),
    (
        "2. Five of the seven families were "
        "structurally executable."
    ),
]

for phrase in required_phrases:
    if phrase not in manuscript:
        raise RuntimeError(
            "Required corrected wording missing:\n"
            + phrase
        )

manifest = json.loads(
    (REPO / MANIFEST_REL).read_text(
        encoding="utf-8"
    )
)

if (
    manifest["publication_package_status"]
    != "PUBLICATION_CONTENT_FROZEN"
):
    raise RuntimeError(
        "Publication manifest is not in frozen-content state."
    )

print("[PASS] manuscript wording audit")
print("[PASS] manifest state = PUBLICATION_CONTENT_FROZEN")


# ----------------------------------------------------------------------------
# Ensure scientific artifacts remain untouched
# ----------------------------------------------------------------------------

print()
print("-" * 96)
print("FROZEN SCIENCE GATE")
print("-" * 96)

science_diff = run(
    [
        "git",
        "diff",
        "--",
        FROZEN_SYNTHESIS_REL.as_posix(),
    ]
).stdout

if science_diff.strip():
    print(science_diff)

    raise RuntimeError(
        "Frozen Stage27 scientific synthesis was modified."
    )

print(
    "[PASS] stage27_4a_final_synthesis is byte-untouched"
)


# ----------------------------------------------------------------------------
# Verify worktree still contains exactly authorized files
# ----------------------------------------------------------------------------

actual_paths_after_regen = set(
    status_paths()
)

if (
    actual_paths_after_regen
    != expected_untracked
):
    raise RuntimeError(
        "Regeneration introduced unexpected repository paths:\n"
        + "\n".join(
            sorted(actual_paths_after_regen)
        )
    )

print(
    "[PASS] regeneration changed only authorized publication files"
)


# ----------------------------------------------------------------------------
# Optional LaTeX syntax smoke test
# ----------------------------------------------------------------------------

print()
print("-" * 96)
print("LATEX SMOKE TEST")
print("-" * 96)

pdflatex = shutil.which(
    "pdflatex"
)

if pdflatex is None:
    print(
        "[SKIP] pdflatex not installed in this Kaggle runtime."
    )
else:
    smoke_dir = Path(
        "/kaggle/working/stage27_pub2_latex_smoke"
    )

    if smoke_dir.exists():
        shutil.rmtree(smoke_dir)

    smoke_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    manuscript_tex = (
        REPO
        / "docs/STAGE27_MANUSCRIPT_INTEGRATION.tex"
    ).resolve()

    tables_tex = (
        REPO
        / "docs/STAGE27_PUBLICATION_TABLES.tex"
    ).resolve()

    wrapper = smoke_dir / "stage27_smoke.tex"

    wrapper.write_text(
        rf"""\documentclass{{article}}
\usepackage[T1]{{fontenc}}
\usepackage[margin=1in]{{geometry}}
\begin{{document}}

\input{{{manuscript_tex.as_posix()}}}

\clearpage

\input{{{tables_tex.as_posix()}}}

\end{{document}}
""",
        encoding="utf-8",
        newline="\n",
    )

    latex_result = run(
        [
            pdflatex,
            "-interaction=nonstopmode",
            "-halt-on-error",
            wrapper.name,
        ],
        cwd=smoke_dir,
        check=False,
    )

    if latex_result.returncode != 0:
        print(latex_result.stdout)
        print(latex_result.stderr)

        raise RuntimeError(
            "Stage27 LaTeX smoke compilation failed."
        )

    pdf_path = (
        smoke_dir / "stage27_smoke.pdf"
    )

    if not pdf_path.is_file():
        raise RuntimeError(
            "LaTeX returned success but produced no PDF."
        )

    print("[PASS] Stage27 LaTeX fragments compile")
    print(
        "       PDF bytes:",
        f"{pdf_path.stat().st_size:,}",
    )


# ----------------------------------------------------------------------------
# Git identity
# ----------------------------------------------------------------------------

print()
print("-" * 96)
print("GIT IDENTITY")
print("-" * 96)

git_name = git(
    "config",
    "--get",
    "user.name",
    check=False,
)

git_email = git(
    "config",
    "--get",
    "user.email",
    check=False,
)

if not git_name:
    git(
        "config",
        "user.name",
        "themubasshir",
    )
    git_name = "themubasshir"

if not git_email:
    git(
        "config",
        "user.email",
        "themubasshir@users.noreply.github.com",
    )
    git_email = (
        "themubasshir@users.noreply.github.com"
    )

print("user.name :", git_name)
print("user.email:", git_email)


# ----------------------------------------------------------------------------
# GitHub auth
# ----------------------------------------------------------------------------

print()
print("-" * 96)
print("GITHUB AUTH")
print("-" * 96)

secret_label, github_token = (
    get_github_token()
)

push_env, askpass_path = (
    make_git_push_env(
        github_token
    )
)


# ----------------------------------------------------------------------------
# Race-condition gate: remote must still equal frozen scientific parent
# ----------------------------------------------------------------------------

print()
print("-" * 96)
print("REMOTE PARENT RACE-CONDITION GATE")
print("-" * 96)

git(
    "fetch",
    "--prune",
    "origin",
    "main",
)

remote_before = git(
    "rev-parse",
    "origin/main",
)

print(
    "Expected origin/main:",
    CANONICAL_STAGE27_PARENT,
)
print(
    "Actual origin/main:  ",
    remote_before,
)

if (
    remote_before
    != CANONICAL_STAGE27_PARENT
):
    raise RuntimeError(
        "origin/main advanced after PUB0/PUB1. "
        "PUB2 will not overwrite or race the remote."
    )

print(
    "[PASS] remote still at frozen Stage27 parent"
)


# ============================================================================
# COMMIT 1 — Publication package
# ============================================================================

print()
print("=" * 96)
print("COMMIT 1 — FREEZE STAGE27 PUBLICATION PACKAGE")
print("=" * 96)

# Stage exact paths only.
for rel in PUBLICATION_FILES:
    git(
        "add",
        "--",
        rel.as_posix(),
    )

staged_names = set(
    git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
)

expected_staged = {
    p.as_posix()
    for p in PUBLICATION_FILES
}

if staged_names != expected_staged:
    raise RuntimeError(
        "Staged-file set is not exactly the authorized "
        "Stage27 publication package.\n\n"
        f"Expected:\n{sorted(expected_staged)}\n\n"
        f"Actual:\n{sorted(staged_names)}"
    )

print("[PASS] exact staged-file set")

print()
print("Staged diff stat:")
print(
    git(
        "diff",
        "--cached",
        "--stat",
    )
)

commit1 = run(
    [
        "git",
        "-c",
        "commit.gpgsign=false",
        "commit",
        "-m",
        (
            "stage27-pub1: freeze manuscript "
            "integration package"
        ),
    ]
)

print(commit1.stdout)

PUB_COMMIT = git(
    "rev-parse",
    "HEAD",
)

print()
print("Publication package commit:")
print(" ", PUB_COMMIT)


# ----------------------------------------------------------------------------
# Push commit 1
# ----------------------------------------------------------------------------

print()
print("Pushing publication package...")

push_main(
    push_env
)

verify_remote_head(
    PUB_COMMIT
)

verify_remote_files(
    "origin/main",
    PUBLICATION_FILES,
)

print()
print(
    "[PASS] publication package remotely frozen"
)


# ----------------------------------------------------------------------------
# Capture hashes for closeout
# ----------------------------------------------------------------------------

publication_hashes = {
    rel.as_posix(): sha256_file(
        REPO / rel
    )
    for rel in PUBLICATION_FILES
}


# ============================================================================
# CLOSEOUT DOCUMENT
# ============================================================================

print()
print("=" * 96)
print("GENERATE STAGE27 PUBLICATION CLOSEOUT")
print("=" * 96)

manifest_hash = publication_hashes[
    MANIFEST_REL.as_posix()
]

generator_hash = publication_hashes[
    GENERATOR_REL.as_posix()
]

closeout = f"""# Stage27 Publication and Reproducibility Closeout

## Scientific Status

**STAGE27 = SCIENTIFICALLY CLOSED**

Stage27 completed the frozen chronology-first zero-training-exposure
attack-family generalization audit before manuscript integration began.

No publication step reopened the target, refit a model, reran inference,
reselected a threshold, recomputed bootstrap intervals, or introduced
new formal statistical testing.

## Frozen Scientific Parent

`{CANONICAL_STAGE27_PARENT}`

## Publication Package Commit

`{PUB_COMMIT}`

Commit subject:

`stage27-pub1: freeze manuscript integration package`

## Final Stage27 Scientific Outcome

The publication-safe Stage27 synthesis is:

1. `SELECTIVE_FAMILY_TRANSFER`
2. `RANKING_THRESHOLD_DIVERGENCE`
3. `LEARNER_DEPENDENCE`

Five of seven preregistered attack families were structurally
executable under strict chronology.

- BOT: executable
- DDOS: executable
- DOS: structurally ineligible
- AUTH_BRUTE_FORCE: structurally ineligible
- INFILTRATION: executable, descriptive only because support = 36
- PORT_SCAN: executable
- WEB_ATTACK: executable

Stage27 is an unseen attack-family generalization audit and is not
formal proof of universal zero-day detection.

## Publication Artifacts

| Artifact | SHA256 |
|---|---|
"""

for rel in PUBLICATION_FILES:
    closeout += (
        f"| `{rel.as_posix()}` | "
        f"`{publication_hashes[rel.as_posix()]}` |\n"
    )

closeout += f"""
## Manifest

Publication manifest:

`{MANIFEST_REL.as_posix()}`

SHA256:

`{manifest_hash}`

Manifest state:

`PUBLICATION_CONTENT_FROZEN`

## Reproducible Generator

`{GENERATOR_REL.as_posix()}`

SHA256:

`{generator_hash}`

The generator verifies the canonical frozen Stage27 source hashes,
reconstructs the manuscript-facing tables and prose from those frozen
artifacts, and supports a read-only `--check` mode.

## Main-Manuscript Figure Policy

### Main Figure 27-1

`results/stage27_loao_unseen_attack/stage27_4a_final_synthesis/figures/stage27_primary_roc_auc_ci.png`

Purpose:

Selective unseen-family ranking transfer and learner dependence.

### Main Figure 27-2

`results/stage27_loao_unseen_attack/stage27_4a_final_synthesis/figures/stage27_balanced_recall_ci.png`

Purpose:

Ranking--threshold divergence at the frozen BALANCED operating point.

### Supplementary Figure 27-S1

`results/stage27_loao_unseen_attack/stage27_4a_final_synthesis/figures/stage27_primary_pr_auc_ci.png`

PR-AUC remains a co-primary metric and must remain in the main results
table and manuscript text even when its separate visualization is
supplementary.

## Reporting Guardrails

The manuscript must not claim:

1. formal or universal zero-day detection;
2. universal unseen-family generalization;
3. that all seven families produced executable target folds;
4. an inferential family-level INFILTRATION conclusion;
5. statistically significant behavioral-similarity correlation;
6. causal explanation from behavioral similarity;
7. universal superiority of XGBoost or LightGBM;
8. target-guided threshold optimization or model adaptation;
9. that raw PR-AUC novelty gaps are prevalence invariant.

## Final Accounting

- preregistered primary families: 7
- executable families: 5
- structurally ineligible families: 2
- descriptive-only executable families: 1
- preregistered learners: 2
- frozen Stage27 fits: 10
- new publication-phase fits: 0
- new publication-phase inference: 0
- target reopenings during publication: 0
- threshold reselections during publication: 0
- bootstrap recomputations during publication: 0
- new formal statistical tests during publication: 0

## Remote Verification

The publication package commit was pushed to `origin/main`, fetched
back from GitHub, and verified by exact commit identity and byte-level
SHA256 comparison of each publication artifact.

## Next Manuscript Phase

Stage27 publication integration is complete.

The next authorized work is whole-manuscript assembly and
claim-to-artifact consistency review across the already-frozen
experimental stages.

No further Stage27 scientific computation is authorized.
"""

CLOSEOUT_PATH = (
    REPO / CLOSEOUT_REL
)

CLOSEOUT_PATH.write_text(
    closeout.rstrip() + "\n",
    encoding="utf-8",
    newline="\n",
)

print(CLOSEOUT_REL)
print(
    "SHA256:",
    sha256_file(CLOSEOUT_PATH),
)


# ----------------------------------------------------------------------------
# Ensure only closeout is now untracked
# ----------------------------------------------------------------------------

post_pub_paths = set(
    status_paths()
)

if post_pub_paths != {
    CLOSEOUT_REL.as_posix()
}:
    raise RuntimeError(
        "Unexpected worktree state before closeout commit:\n"
        + "\n".join(
            sorted(post_pub_paths)
        )
    )

print(
    "[PASS] only publication closeout remains uncommitted"
)


# ============================================================================
# COMMIT 2 — Publication closeout
# ============================================================================

print()
print("=" * 96)
print("COMMIT 2 — STAGE27 PUBLICATION CLOSEOUT")
print("=" * 96)

git(
    "add",
    "--",
    CLOSEOUT_REL.as_posix(),
)

staged_closeout = git(
    "diff",
    "--cached",
    "--name-only",
).splitlines()

if staged_closeout != [
    CLOSEOUT_REL.as_posix()
]:
    raise RuntimeError(
        "Closeout staging contains unexpected files."
    )

commit2 = run(
    [
        "git",
        "-c",
        "commit.gpgsign=false",
        "commit",
        "-m",
        (
            "stage27-pub2: close manuscript "
            "integration"
        ),
    ]
)

print(commit2.stdout)

FINAL_CLOSEOUT_COMMIT = git(
    "rev-parse",
    "HEAD",
)

print()
print("Final Stage27 publication closeout commit:")
print(
    " ",
    FINAL_CLOSEOUT_COMMIT,
)


# ----------------------------------------------------------------------------
# Push commit 2
# ----------------------------------------------------------------------------

print()
print("Pushing Stage27 publication closeout...")

# Remote must still equal first publication commit.
git(
    "fetch",
    "--prune",
    "origin",
    "main",
)

remote_pre_closeout = git(
    "rev-parse",
    "origin/main",
)

if remote_pre_closeout != PUB_COMMIT:
    raise RuntimeError(
        "origin/main changed before closeout push."
    )

push_main(
    push_env
)

verify_remote_head(
    FINAL_CLOSEOUT_COMMIT
)

verify_remote_files(
    "origin/main",
    [
        *PUBLICATION_FILES,
        CLOSEOUT_REL,
    ],
)


# ----------------------------------------------------------------------------
# Final deterministic package check from the new publication HEAD
# ----------------------------------------------------------------------------

print()
print("-" * 96)
print("FINAL GENERATOR CHECK FROM CLOSEOUT HEAD")
print("-" * 96)

final_check = run(
    [
        sys.executable,
        str(GENERATOR),
        "--check",
    ]
)

print(final_check.stdout)

if final_check.stderr.strip():
    print(final_check.stderr)


# ----------------------------------------------------------------------------
# Final worktree cleanliness
# ----------------------------------------------------------------------------

final_status = git(
    "status",
    "--porcelain",
    "-uall",
)

if final_status:
    print(final_status)

    raise RuntimeError(
        "Repository is not clean after Stage27 publication closeout."
    )

print()
print("[PASS] final Git worktree clean")


# ----------------------------------------------------------------------------
# Final log
# ----------------------------------------------------------------------------

print()
print("-" * 96)
print("FINAL COMMIT LINEAGE")
print("-" * 96)

print(
    git(
        "log",
        "-3",
        "--oneline",
        "--decorate",
    )
)


# ----------------------------------------------------------------------------
# Local machine-readable remote-verification receipt
# ----------------------------------------------------------------------------
#
# Kept outside the repository because the repository closeout is already
# content-addressed and remotely verified. Including this receipt would require
# a third closeout commit merely to record its own predecessor.
# ----------------------------------------------------------------------------

verification_receipt = {
    "stage": "STAGE27-PUB2",
    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "repository": REPO_FULL,
    "scientific_parent":
        CANONICAL_STAGE27_PARENT,
    "publication_package_commit":
        PUB_COMMIT,
    "publication_closeout_commit":
        FINAL_CLOSEOUT_COMMIT,
    "remote_head_verified":
        FINAL_CLOSEOUT_COMMIT,
    "publication_artifact_sha256":
        publication_hashes,
    "closeout": {
        "path": CLOSEOUT_REL.as_posix(),
        "sha256": sha256_file(
            CLOSEOUT_PATH
        ),
    },
    "science_operations": {
        "model_fitting": 0,
        "model_inference": 0,
        "target_reopenings": 0,
        "threshold_reselection": 0,
        "bootstrap_recomputation": 0,
        "new_formal_statistical_tests": 0,
    },
    "remote_verification": "PASS_EXACT",
    "worktree_clean": True,
}

receipt_path = Path(
    "/kaggle/working/"
    "stage27_pub2_remote_verification_receipt.json"
)

receipt_path.write_text(
    json.dumps(
        verification_receipt,
        indent=2,
        sort_keys=True,
    ) + "\n",
    encoding="utf-8",
    newline="\n",
)


# ----------------------------------------------------------------------------
# Remove temporary credential helper
# ----------------------------------------------------------------------------

try:
    askpass_path.unlink(
        missing_ok=True
    )
except Exception:
    pass

# Do not retain token beyond this cell's need.
github_token = None
push_env.pop(
    "GITHUB_TOKEN",
    None,
)


# ----------------------------------------------------------------------------
# Final
# ----------------------------------------------------------------------------

print()
print("=" * 96)
print("STAGE27 PUBLICATION INTEGRATION — REMOTELY CLOSED")
print("=" * 96)

print()
print("Scientific parent:")
print(
    " ",
    CANONICAL_STAGE27_PARENT,
)

print()
print("Publication package commit:")
print(
    " ",
    PUB_COMMIT,
)

print()
print("Publication closeout commit:")
print(
    " ",
    FINAL_CLOSEOUT_COMMIT,
)

print()
print("Remote:")
print(
    "  origin/main:",
    git("rev-parse", "origin/main"),
)

print()
print("Publication files:")
for rel in PUBLICATION_FILES:
    print(
        f"  {rel}\n"
        f"    {sha256_file(REPO / rel)}"
    )

print(
    f"  {CLOSEOUT_REL}\n"
    f"    {sha256_file(CLOSEOUT_PATH)}"
)

print()
print("Integrity:")
print("  frozen Stage27 science       : UNCHANGED")
print("  deterministic generator check: PASS")
print("  remote commit identity       : PASS")
print("  remote file SHA256           : PASS")
print("  final worktree               : CLEAN")

print()
print("Science operations during publication:")
print("  model fitting             : 0")
print("  model inference           : 0")
print("  target reopening          : 0")
print("  threshold reselection     : 0")
print("  bootstrap recomputation   : 0")
print("  new formal statistics     : 0")

print()
print("Local remote-verification receipt:")
print(
    " ",
    receipt_path,
)
print(
    "  SHA256:",
    sha256_file(receipt_path),
)

print()
print("NEXT PHASE:")
print(
    "  WHOLE-MANUSCRIPT ASSEMBLY + "
    "CLAIM-TO-ARTIFACT CONSISTENCY AUDIT"
)

STAGE27-PUB2 — PUBLICATION FREEZE / COMMIT / PUSH / REMOTE CLOSEOUT

timestamp_utc: 2026-08-21T15:22:10.296081+00:00
repository   : themubasshir/ids2018-validation-safe-ablation
repo path    : /kaggle/working/ids2018-validation-safe-ablation
science parent: 0e1439565aedc7da9b7ca1207262e9061422bc22

branch: main
HEAD:   0e1439565aedc7da9b7ca1207262e9061422bc22

------------------------------------------------------------------------------------------------
PUB1 WORKTREE INVENTORY
------------------------------------------------------------------------------------------------
Expected publication paths:
  docs/STAGE27_MANUSCRIPT_INTEGRATION.md
  docs/STAGE27_MANUSCRIPT_INTEGRATION.tex
  docs/STAGE27_PUBLICATION_TABLES.md
  docs/STAGE27_PUBLICATION_TABLES.tex
  results/stage27_loao_unseen_attack/stage27_publication_package/stage27_publication_manifest.json
  scripts/stage27/stage27_publication_integration.py

Actual changed/untracked paths:
  docs/STAGE27_MANUSCRIPT_INTEGRATION.md
  docs/

In [11]:
# ============================================================================
# STAGE27-PUB1R
# FULL KAGGLE NOTEBOOK + PYTHON REPRODUCIBILITY EXPORT
#
# PURPOSE
#   Export the complete Stage27 Kaggle notebook before publication closeout.
#
# OUTPUT
#   scripts/stage27/stage27_loao_unseen_attack.ipynb
#   scripts/stage27/stage27_loao_unseen_attack.py
#
#   results/stage27_loao_unseen_attack/stage27_publication_package/
#       stage27_notebook_export_receipt.json
#
# IMPORTANT
#   - NO model fitting
#   - NO model inference
#   - NO target reopening
#   - NO threshold reselection
#   - NO bootstrap recomputation
#   - NO Git commit
#   - NO Git push
#
# The exporter prefers an exact live notebook snapshot.
# It falls back to IPython execution history only when that history is
# sufficiently complete to represent the Stage27 execution notebook.
# ============================================================================

from __future__ import annotations

import copy
import hashlib
import json
import os
import re
import subprocess
import sys
import urllib.parse
import urllib.request
from pathlib import Path
from datetime import datetime, timezone


# ============================================================================
# 0. PATHS / FROZEN IDENTITY
# ============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

CANONICAL_STAGE27_PARENT = (
    "0e1439565aedc7da9b7ca1207262e9061422bc22"
)

STAGE_DIR = (
    REPO
    / "scripts"
    / "stage27"
)

NOTEBOOK_OUT = (
    STAGE_DIR
    / "stage27_loao_unseen_attack.ipynb"
)

PYTHON_OUT = (
    STAGE_DIR
    / "stage27_loao_unseen_attack.py"
)

RECEIPT_OUT = (
    REPO
    / "results"
    / "stage27_loao_unseen_attack"
    / "stage27_publication_package"
    / "stage27_notebook_export_receipt.json"
)


# Existing PUB1 files which are expected to be untracked right now.
EXPECTED_EXISTING_PUB1_PATHS = {
    "docs/STAGE27_MANUSCRIPT_INTEGRATION.md",
    "docs/STAGE27_MANUSCRIPT_INTEGRATION.tex",
    "docs/STAGE27_PUBLICATION_TABLES.md",
    "docs/STAGE27_PUBLICATION_TABLES.tex",
    (
        "results/stage27_loao_unseen_attack/"
        "stage27_publication_package/"
        "stage27_publication_manifest.json"
    ),
    "scripts/stage27/stage27_publication_integration.py",
}


# After PUB1R we expect exactly these additional paths.
NEW_REPRO_PATHS = {
    "scripts/stage27/stage27_loao_unseen_attack.ipynb",
    "scripts/stage27/stage27_loao_unseen_attack.py",
    (
        "results/stage27_loao_unseen_attack/"
        "stage27_publication_package/"
        "stage27_notebook_export_receipt.json"
    ),
}


# Conservative GitHub single-file safety threshold.
# GitHub hard limit is 100 MB; stay well below that.
MAX_GITHUB_FILE_BYTES = 90 * 1024 * 1024


# ============================================================================
# 1. HELPERS
# ============================================================================

def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
    text=True,
):
    result = subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        text=text,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    if check and result.returncode != 0:
        if text:
            print(result.stdout)
            print(result.stderr)

        raise RuntimeError(
            f"Command failed ({result.returncode}): "
            + " ".join(str(x) for x in cmd)
        )

    return result


def git(*args, check=True):
    return run(
        ["git", *args],
        check=check,
    ).stdout.strip()


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()

    with path.open("rb") as fh:
        while True:
            block = fh.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def git_status_paths():
    output = git(
        "status",
        "--porcelain",
        "-uall",
    )

    paths = set()

    if not output:
        return paths

    for line in output.splitlines():
        raw = line[3:]

        if " -> " in raw:
            raw = raw.split(
                " -> ",
                1,
            )[1]

        paths.add(raw)

    return paths


def source_text_from_notebook(nb):
    pieces = []

    for cell in nb.get(
        "cells",
        [],
    ):
        source = cell.get(
            "source",
            "",
        )

        if isinstance(
            source,
            list,
        ):
            source = "".join(source)

        pieces.append(
            str(source)
        )

    return "\n".join(
        pieces
    )


def count_cells(nb):
    cells = nb.get(
        "cells",
        [],
    )

    code = sum(
        1
        for cell in cells
        if cell.get("cell_type") == "code"
    )

    markdown = sum(
        1
        for cell in cells
        if cell.get("cell_type") == "markdown"
    )

    raw = sum(
        1
        for cell in cells
        if cell.get("cell_type") == "raw"
    )

    executed = sum(
        1
        for cell in cells
        if (
            cell.get("cell_type") == "code"
            and
            cell.get("execution_count") is not None
        )
    )

    output_cells = sum(
        1
        for cell in cells
        if (
            cell.get("cell_type") == "code"
            and
            len(
                cell.get(
                    "outputs",
                    [],
                )
            ) > 0
        )
    )

    return {
        "total": len(cells),
        "code": code,
        "markdown": markdown,
        "raw": raw,
        "executed_code": executed,
        "code_cells_with_outputs": output_cells,
    }


# ============================================================================
# 2. REPOSITORY PREFLIGHT
# ============================================================================

print("=" * 92)
print(
    "STAGE27-PUB1R — FULL NOTEBOOK / PYTHON "
    "REPRODUCIBILITY EXPORT"
)
print("=" * 92)

if not REPO.is_dir():
    raise RuntimeError(
        "Repository checkout not found. "
        "PUB0/PUB1 must run first."
    )

branch = git(
    "branch",
    "--show-current",
)

head = git(
    "rev-parse",
    "HEAD",
)

print()
print("repository :", REPO)
print("branch     :", branch)
print("HEAD       :", head)

if branch != "main":
    raise RuntimeError(
        f"Expected main branch; found {branch}"
    )

if head != CANONICAL_STAGE27_PARENT:
    raise RuntimeError(
        "\nPUB1R must run before PUB2 and while HEAD "
        "is still the frozen Stage27 scientific parent.\n\n"
        f"Expected:\n  {CANONICAL_STAGE27_PARENT}\n"
        f"Actual:\n  {head}"
    )


current_changes = git_status_paths()

if current_changes != EXPECTED_EXISTING_PUB1_PATHS:
    print()
    print("Expected existing PUB1 changes:")

    for path in sorted(
        EXPECTED_EXISTING_PUB1_PATHS
    ):
        print(" ", path)

    print()
    print("Actual worktree changes:")

    for path in sorted(
        current_changes
    ):
        print(" ", path)

    raise RuntimeError(
        "Worktree is not in the exact expected "
        "post-PUB1 state."
    )

print(
    "[PASS] exact post-PUB1 worktree state"
)


# ============================================================================
# 3. LOAD NBFORMAT
# ============================================================================

try:
    import nbformat
except Exception as exc:
    raise RuntimeError(
        "nbformat is required for the Stage27 notebook export."
    ) from exc


# ============================================================================
# 4. CURRENT KERNEL ID
# ============================================================================

def current_kernel_id():
    try:
        from ipykernel.connect import (
            get_connection_file,
        )

        connection = Path(
            get_connection_file()
        ).name

        match = re.search(
            r"kernel-(.+)\.json$",
            connection,
        )

        if match:
            return match.group(1)

    except Exception:
        pass

    return None


KERNEL_ID = current_kernel_id()

print()
print("Current kernel ID:")
print(
    " ",
    KERNEL_ID
    if KERNEL_ID
    else "[unavailable]"
)


# ============================================================================
# 5. TRY EXACT NOTEBOOK VIA JUPYTER SERVER API
# ============================================================================

def url_get_json(url, timeout=5):
    request = urllib.request.Request(
        url,
        headers={
            "Accept": "application/json",
        },
    )

    with urllib.request.urlopen(
        request,
        timeout=timeout,
    ) as response:
        return json.loads(
            response.read().decode(
                "utf-8"
            )
        )


def jupyter_server_records():
    commands = [
        [
            "jupyter",
            "server",
            "list",
            "--json",
        ],
        [
            "jupyter",
            "notebook",
            "list",
            "--json",
        ],
    ]

    records = []

    for command in commands:
        result = run(
            command,
            cwd=Path("/kaggle/working"),
            check=False,
        )

        if result.returncode != 0:
            continue

        for line in result.stdout.splitlines():
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(
                    line
                )
            except Exception:
                continue

            if isinstance(
                record,
                dict,
            ):
                records.append(
                    record
                )

    # Deduplicate by URL/root combination.
    unique = []
    seen = set()

    for record in records:
        key = (
            record.get("url"),
            record.get("root_dir")
            or record.get("notebook_dir"),
        )

        if key not in seen:
            seen.add(key)
            unique.append(record)

    return unique


def exact_notebook_from_server():
    if not KERNEL_ID:
        return None, None

    servers = jupyter_server_records()

    for server in servers:
        base_url = server.get(
            "url"
        )

        if not base_url:
            continue

        if not base_url.endswith("/"):
            base_url += "/"

        token = (
            server.get("token")
            or ""
        )

        query = ""

        if token:
            query = (
                "?token="
                + urllib.parse.quote(
                    token
                )
            )

        sessions_url = (
            urllib.parse.urljoin(
                base_url,
                "api/sessions",
            )
            + query
        )

        try:
            sessions = url_get_json(
                sessions_url
            )
        except Exception:
            continue

        for session in sessions:
            kernel = session.get(
                "kernel",
                {},
            )

            if kernel.get("id") != KERNEL_ID:
                continue

            notebook_info = (
                session.get("notebook")
                or {}
            )

            notebook_path = (
                notebook_info.get("path")
                or session.get("path")
            )

            if not notebook_path:
                continue

            encoded_path = urllib.parse.quote(
                notebook_path,
                safe="/",
            )

            contents_url = (
                urllib.parse.urljoin(
                    base_url,
                    "api/contents/"
                    + encoded_path,
                )
                + query
            )

            try:
                model = url_get_json(
                    contents_url
                )
            except Exception:
                continue

            if (
                model.get("type")
                != "notebook"
            ):
                continue

            content = model.get(
                "content"
            )

            if not isinstance(
                content,
                dict,
            ):
                continue

            return (
                content,
                {
                    "mode":
                        "EXACT_JUPYTER_CONTENTS_API",

                    "server_root":
                        server.get(
                            "root_dir"
                        )
                        or server.get(
                            "notebook_dir"
                        ),

                    "notebook_path":
                        notebook_path,
                },
            )

    return None, None


# ============================================================================
# 6. TRY EXACT NOTEBOOK FROM LOCAL KAGGLE PATHS
# ============================================================================

def local_notebook_candidates():
    candidates = []

    explicit = [
        Path(
            "/kaggle/working/__notebook__.ipynb"
        ),
        Path(
            "/kaggle/working/notebook.ipynb"
        ),
    ]

    for path in explicit:
        if path.is_file():
            candidates.append(path)

    # Top-level Kaggle working notebook files.
    try:
        candidates.extend(
            Path("/kaggle/working").glob(
                "*.ipynb"
            )
        )
    except Exception:
        pass

    # Avoid selecting our target export.
    cleaned = []

    seen = set()

    for path in candidates:
        try:
            resolved = path.resolve()
        except Exception:
            continue

        if resolved == NOTEBOOK_OUT.resolve():
            continue

        if resolved in seen:
            continue

        seen.add(
            resolved
        )

        cleaned.append(
            resolved
        )

    return cleaned


def exact_notebook_from_local():
    matches = []

    for path in local_notebook_candidates():
        try:
            nb = nbformat.read(
                path,
                as_version=4,
            )
        except Exception:
            continue

        text = source_text_from_notebook(
            nb
        )

        # Current notebook must include both publication cells
        # already executed in this session/source.
        if (
            "STAGE27-PUB0"
            in text
            and
            "STAGE27-PUB1"
            in text
        ):
            matches.append(
                (
                    path,
                    nb,
                )
            )

    if not matches:
        return None, None

    # Prefer the largest notebook; it is more likely to be
    # the complete Stage27 source rather than a small partial copy.
    matches.sort(
        key=lambda item:
            item[0].stat().st_size,
        reverse=True,
    )

    path, nb = matches[0]

    return (
        nb,
        {
            "mode":
                "EXACT_LOCAL_NOTEBOOK_FILE",

            "notebook_path":
                str(path),

            "source_bytes":
                path.stat().st_size,
        },
    )


# ============================================================================
# 7. FALLBACK: RECONSTRUCT FROM CURRENT IPYTHON INPUT HISTORY
# ============================================================================

def notebook_from_ipython_history():
    try:
        ip = get_ipython()
    except Exception:
        ip = None

    if ip is None:
        return None, None

    history = []

    try:
        # Current session raw input, in execution order.
        for session, line_no, source in (
            ip.history_manager.get_range(
                session=0,
                start=1,
                stop=None,
                raw=True,
                output=False,
            )
        ):
            if not source:
                continue

            history.append(
                {
                    "session": int(
                        session
                    ),
                    "line_no": int(
                        line_no
                    ),
                    "source": str(
                        source
                    ),
                }
            )

    except Exception as exc:
        print(
            "[WARN] IPython history access failed:",
            repr(exc),
        )

        return None, None

    if not history:
        return None, None

    sources = [
        item["source"]
        for item in history
    ]

    joined = "\n\n".join(
        sources
    )

    # Hard completeness gates:
    #
    # We do NOT accept a new session containing only PUB0/PUB1/PUB1R.
    #
    # Stage27 was a multi-stage scientific notebook. Reconstructed
    # history must therefore contain substantial Stage27 execution.
    marker_candidates = [
        "STAGE27-0",
        "STAGE27-1",
        "STAGE27-2",
        "STAGE27-3",
        "STAGE27-4",
        "stage27_0",
        "stage27_1",
        "stage27_2",
        "stage27_3",
        "stage27_4",
    ]

    marker_hits = sorted({
        marker
        for marker in marker_candidates
        if marker in joined
    })

    stage27_occurrences = (
        joined.upper().count(
            "STAGE27"
        )
    )

    print()
    print(
        "IPython history candidate:"
    )
    print(
        "  executed input cells :",
        len(history),
    )
    print(
        "  STAGE27 occurrences  :",
        stage27_occurrences,
    )
    print(
        "  scientific markers   :",
        marker_hits,
    )

    if len(history) < 10:
        return (
            None,
            {
                "mode":
                    "HISTORY_REJECTED_TOO_FEW_CELLS",

                "history_cells":
                    len(history),

                "stage27_occurrences":
                    stage27_occurrences,

                "marker_hits":
                    marker_hits,
            },
        )

    if stage27_occurrences < 10:
        return (
            None,
            {
                "mode":
                    "HISTORY_REJECTED_INSUFFICIENT_STAGE27_CONTENT",

                "history_cells":
                    len(history),

                "stage27_occurrences":
                    stage27_occurrences,

                "marker_hits":
                    marker_hits,
            },
        )

    # Reconstruct code cells in execution order.
    cells = []

    for index, item in enumerate(
        history,
        start=1,
    ):
        cell = nbformat.v4.new_code_cell(
            source=item["source"]
        )

        cell["execution_count"] = (
            item["line_no"]
        )

        cell["metadata"] = {
            "stage27_execution_history_index":
                index,

            "stage27_ipython_session":
                item["session"],

            "stage27_ipython_line_number":
                item["line_no"],
        }

        # Outputs cannot be faithfully recovered from IPython
        # input history, so they remain empty.
        cell["outputs"] = []

        cells.append(
            cell
        )

    nb = nbformat.v4.new_notebook(
        cells=cells
    )

    nb["metadata"][
        "stage27_reproducibility_export"
    ] = {
        "mode":
            "RECONSTRUCTED_FROM_IPYTHON_INPUT_HISTORY",

        "preserves":
            "executed code-cell source and execution order",

        "does_not_preserve": [
            "unexecuted notebook cells",
            "original markdown-only cells",
            "cell outputs",
            "original notebook UI metadata",
        ],

        "history_cells":
            len(history),

        "stage27_occurrences":
            stage27_occurrences,

        "marker_hits":
            marker_hits,
    }

    return (
        nb,
        {
            "mode":
                "RECONSTRUCTED_FROM_IPYTHON_INPUT_HISTORY",

            "history_cells":
                len(history),

            "stage27_occurrences":
                stage27_occurrences,

            "marker_hits":
                marker_hits,
        },
    )


# ============================================================================
# 8. ACQUIRE THE BEST AVAILABLE COMPLETE NOTEBOOK
# ============================================================================

print()
print("-" * 92)
print("NOTEBOOK ACQUISITION")
print("-" * 92)


NOTEBOOK = None
ACQUISITION = None


# First preference: live Jupyter Contents API.
try:
    NOTEBOOK, ACQUISITION = (
        exact_notebook_from_server()
    )
except Exception as exc:
    print(
        "[WARN] Jupyter API acquisition error:",
        repr(exc),
    )


if NOTEBOOK is not None:
    print(
        "[PASS] exact live notebook acquired "
        "through Jupyter Contents API"
    )


# Second preference: exact local .ipynb.
if NOTEBOOK is None:
    try:
        NOTEBOOK, ACQUISITION = (
            exact_notebook_from_local()
        )
    except Exception as exc:
        print(
            "[WARN] local notebook acquisition error:",
            repr(exc),
        )

    if NOTEBOOK is not None:
        print(
            "[PASS] exact notebook acquired "
            "from local Kaggle filesystem"
        )


# Final fallback: current execution history.
if NOTEBOOK is None:
    NOTEBOOK, history_info = (
        notebook_from_ipython_history()
    )

    if NOTEBOOK is not None:
        ACQUISITION = (
            history_info
        )

        print(
            "[PASS] complete-enough execution-source "
            "notebook reconstructed from IPython history"
        )

    else:
        print()
        print(
            "Exact live notebook source was not accessible."
        )

        if history_info:
            print(
                json.dumps(
                    history_info,
                    indent=2,
                )
            )

        raise RuntimeError(
            "\nFULL STAGE27 NOTEBOOK EXPORT BLOCKED.\n\n"
            "Kaggle did not expose an exact notebook source, and the "
            "current kernel history was not sufficiently complete to "
            "represent the Stage27 scientific notebook.\n\n"
            "This exporter intentionally refuses to create a misleading "
            "'full' notebook from only PUB0/PUB1 cells.\n"
        )


# ============================================================================
# 9. NORMALIZE / TAG EXPORT METADATA
# ============================================================================

NOTEBOOK = copy.deepcopy(
    NOTEBOOK
)

NOTEBOOK.setdefault(
    "metadata",
    {},
)

NOTEBOOK["metadata"][
    "stage27_reproducibility_export"
] = {
    **NOTEBOOK["metadata"].get(
        "stage27_reproducibility_export",
        {},
    ),

    "export_stage":
        "STAGE27-PUB1R",

    "scientific_parent":
        CANONICAL_STAGE27_PARENT,

    "scientific_status":
        "CLOSED",

    "acquisition":
        ACQUISITION,

    "science_operations_during_export": {
        "model_fits": 0,
        "model_inference": 0,
        "target_reopenings": 0,
        "threshold_reselection": 0,
        "bootstrap_recomputation": 0,
        "new_formal_statistical_tests": 0,
    },
}


# ============================================================================
# 10. COMPLETENESS / CONTENT AUDIT
# ============================================================================

stats = count_cells(
    NOTEBOOK
)

all_source = source_text_from_notebook(
    NOTEBOOK
)

print()
print("-" * 92)
print("NOTEBOOK CONTENT AUDIT")
print("-" * 92)

print(
    json.dumps(
        stats,
        indent=2,
    )
)

required_pub_markers = [
    "STAGE27-PUB0",
    "STAGE27-PUB1",
]

for marker in required_pub_markers:
    if marker not in all_source:
        raise RuntimeError(
            f"Notebook export is missing required current marker: "
            f"{marker}"
        )

print(
    "[PASS] PUB0/PUB1 source present"
)


stage27_mentions = (
    all_source.upper().count(
        "STAGE27"
    )
)

print(
    "STAGE27 source occurrences:",
    stage27_mentions,
)

if stage27_mentions < 10:
    raise RuntimeError(
        "Notebook contains too little Stage27 source content "
        "to be accepted as the complete reproducibility export."
    )


# ============================================================================
# 11. SECRET-SAFETY SCAN
# ============================================================================

print()
print("-" * 92)
print("SECRET-SAFETY SCAN")
print("-" * 92)

# Scan source only. We do not want credentials committed in notebook source.
SECRET_PATTERNS = {
    "GitHub classic PAT":
        r"\bghp_[A-Za-z0-9]{20,}\b",

    "GitHub fine-grained PAT":
        r"\bgithub_pat_[A-Za-z0-9_]{20,}\b",

    "GitHub OAuth token":
        r"\bgho_[A-Za-z0-9]{20,}\b",

    "GitHub user token":
        r"\bghu_[A-Za-z0-9]{20,}\b",

    "GitHub server token":
        r"\bghs_[A-Za-z0-9]{20,}\b",

    "Stripe live secret":
        r"\bsk_live_[A-Za-z0-9]{16,}\b",

    "AWS access key":
        r"\bAKIA[0-9A-Z]{16}\b",
}


secret_hits = []

for label, pattern in SECRET_PATTERNS.items():
    matches = re.findall(
        pattern,
        all_source,
    )

    if matches:
        secret_hits.append(
            {
                "type": label,
                "count": len(matches),
            }
        )


if secret_hits:
    print(
        json.dumps(
            secret_hits,
            indent=2,
        )
    )

    raise RuntimeError(
        "Potential credential material detected in notebook source. "
        "Export blocked before writing GitHub artifacts."
    )

print(
    "[PASS] no recognized credential token patterns "
    "found in notebook source"
)


# ============================================================================
# 12. WRITE FULL IPYNB
# ============================================================================

STAGE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

NOTEBOOK_OUT.parent.mkdir(
    parents=True,
    exist_ok=True,
)

nbformat.write(
    NOTEBOOK,
    NOTEBOOK_OUT,
)

print()
print("-" * 92)
print("IPYNB EXPORT")
print("-" * 92)

print(NOTEBOOK_OUT)
print(
    "bytes :",
    f"{NOTEBOOK_OUT.stat().st_size:,}",
)
print(
    "SHA256:",
    sha256_file(
        NOTEBOOK_OUT
    ),
)


# ============================================================================
# 13. GENERATE PYTHON EXPORT FROM ALL NOTEBOOK CELLS
# ============================================================================

def comment_markdown(text):
    lines = str(text).splitlines()

    if not lines:
        return "#"

    return "\n".join(
        (
            "#"
            if line == ""
            else "# " + line
        )
        for line in lines
    )


py_parts = [
    (
        "# "
        + "=" * 78
    ),
    (
        "# STAGE27 — LEAVE-ONE-ATTACK-FAMILY-OUT "
        "UNSEEN-FAMILY GENERALIZATION AUDIT"
    ),
    (
        "# Complete source export from the Stage27 Kaggle notebook."
    ),
    "#",
    (
        "# Scientific parent: "
        + CANONICAL_STAGE27_PARENT
    ),
    "# Scientific state: CLOSED",
    "#",
    (
        "# This export preserves notebook source order for "
        "reproducibility."
    ),
    (
        "# It does NOT authorize any new Stage27 fitting, inference, "
        "target reopening, threshold reselection, bootstrap "
        "recomputation, or formal statistical testing."
    ),
    (
        "# "
        + "=" * 78
    ),
    "",
]


for index, cell in enumerate(
    NOTEBOOK.get(
        "cells",
        [],
    ),
    start=1,
):
    cell_type = cell.get(
        "cell_type",
        "unknown",
    )

    source = cell.get(
        "source",
        "",
    )

    if isinstance(
        source,
        list,
    ):
        source = "".join(
            source
        )

    if cell_type == "code":
        py_parts.extend([
            "",
            (
                f"# %% [Stage27 notebook cell {index}]"
            ),
            str(source).rstrip(),
            "",
        ])

    elif cell_type == "markdown":
        py_parts.extend([
            "",
            (
                f"# %% [markdown — Stage27 notebook cell {index}]"
            ),
            comment_markdown(
                source
            ),
            "",
        ])

    elif cell_type == "raw":
        py_parts.extend([
            "",
            (
                f"# %% [raw — Stage27 notebook cell {index}]"
            ),
            comment_markdown(
                source
            ),
            "",
        ])


PYTHON_OUT.write_text(
    "\n".join(
        py_parts
    ).rstrip()
    + "\n",
    encoding="utf-8",
    newline="\n",
)

print()
print("-" * 92)
print("PYTHON EXPORT")
print("-" * 92)

print(PYTHON_OUT)
print(
    "bytes :",
    f"{PYTHON_OUT.stat().st_size:,}",
)
print(
    "SHA256:",
    sha256_file(
        PYTHON_OUT
    ),
)


# ============================================================================
# 14. VERIFY PYTHON EXPORT CONTAINS THE NOTEBOOK CODE CELLS
# ============================================================================

py_text = PYTHON_OUT.read_text(
    encoding="utf-8"
)

code_cells = [
    cell
    for cell in NOTEBOOK["cells"]
    if cell.get("cell_type") == "code"
]

missing_code_cells = []

for index, cell in enumerate(
    code_cells,
    start=1,
):
    source = cell.get(
        "source",
        "",
    )

    if isinstance(
        source,
        list,
    ):
        source = "".join(
            source
        )

    source = str(
        source
    ).rstrip()

    if (
        source
        and
        source not in py_text
    ):
        missing_code_cells.append(
            index
        )


if missing_code_cells:
    raise RuntimeError(
        "Python export failed source-preservation audit for "
        f"{len(missing_code_cells)} code cell(s): "
        f"{missing_code_cells[:20]}"
    )

print()
print(
    "[PASS] every notebook code-cell source is present "
    "in the Python export"
)


# ============================================================================
# 15. SIZE GATE
# ============================================================================

for path in [
    NOTEBOOK_OUT,
    PYTHON_OUT,
]:
    if (
        path.stat().st_size
        > MAX_GITHUB_FILE_BYTES
    ):
        raise RuntimeError(
            f"GitHub-safe size gate failed:\n"
            f"{path}\n"
            f"bytes={path.stat().st_size:,}"
        )

print(
    "[PASS] both reproducibility exports are below "
    "the conservative GitHub file-size gate"
)


# ============================================================================
# 16. WRITE NOTEBOOK EXPORT RECEIPT
# ============================================================================

RECEIPT_OUT.parent.mkdir(
    parents=True,
    exist_ok=True,
)

receipt = {
    "stage":
        "STAGE27-PUB1R",

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "repository":
        "themubasshir/ids2018-validation-safe-ablation",

    "scientific_parent":
        CANONICAL_STAGE27_PARENT,

    "scientific_status":
        "CLOSED",

    "acquisition":
        ACQUISITION,

    "notebook": {
        "path":
            NOTEBOOK_OUT.relative_to(
                REPO
            ).as_posix(),

        "sha256":
            sha256_file(
                NOTEBOOK_OUT
            ),

        "bytes":
            NOTEBOOK_OUT.stat().st_size,

        "cell_counts":
            stats,

        "stage27_source_occurrences":
            stage27_mentions,
    },

    "python_export": {
        "path":
            PYTHON_OUT.relative_to(
                REPO
            ).as_posix(),

        "sha256":
            sha256_file(
                PYTHON_OUT
            ),

        "bytes":
            PYTHON_OUT.stat().st_size,

        "all_code_cell_sources_preserved":
            True,
    },

    "secret_scan": {
        "status":
            "PASS",

        "recognized_secret_patterns_found":
            0,
    },

    "science_operations": {
        "model_fits": 0,
        "model_inference": 0,
        "target_reopenings": 0,
        "threshold_reselection": 0,
        "bootstrap_recomputation": 0,
        "new_formal_statistical_tests": 0,
    },

    "git_operations": {
        "commit":
            False,

        "push":
            False,
    },
}


RECEIPT_OUT.write_text(
    json.dumps(
        receipt,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
    newline="\n",
)


print()
print("-" * 92)
print("EXPORT RECEIPT")
print("-" * 92)

print(RECEIPT_OUT)
print(
    "SHA256:",
    sha256_file(
        RECEIPT_OUT
    ),
)


# ============================================================================
# 17. FROZEN SCIENCE MODIFICATION GATE
# ============================================================================

science_diff = run(
    [
        "git",
        "diff",
        "--",
        (
            "results/stage27_loao_unseen_attack/"
            "stage27_4a_final_synthesis"
        ),
    ]
).stdout

if science_diff.strip():
    print(
        science_diff
    )

    raise RuntimeError(
        "Notebook export modified frozen Stage27 science."
    )

print()
print(
    "[PASS] frozen Stage27-4A science remains untouched"
)


# ============================================================================
# 18. EXACT POST-EXPORT WORKTREE INVENTORY
# ============================================================================

expected_final_paths = (
    EXPECTED_EXISTING_PUB1_PATHS
    |
    NEW_REPRO_PATHS
)

actual_final_paths = (
    git_status_paths()
)

print()
print("-" * 92)
print("POST-EXPORT GIT STATUS")
print("-" * 92)

print(
    git(
        "status",
        "--short",
        "-uall",
    )
)


if (
    actual_final_paths
    != expected_final_paths
):
    unexpected = (
        actual_final_paths
        -
        expected_final_paths
    )

    missing = (
        expected_final_paths
        -
        actual_final_paths
    )

    print()
    print(
        "Unexpected:",
        sorted(
            unexpected
        ),
    )

    print(
        "Missing:",
        sorted(
            missing
        ),
    )

    raise RuntimeError(
        "PUB1R introduced an unexpected repository path."
    )


print()
print(
    "[PASS] exact authorized post-PUB1R worktree inventory"
)


# ============================================================================
# 19. FINAL SUMMARY
# ============================================================================

print()
print("=" * 92)
print(
    "STAGE27-PUB1R COMPLETE — FULL REPRODUCIBILITY EXPORT READY"
)
print("=" * 92)

print()
print("Acquisition mode:")
print(
    " ",
    ACQUISITION.get(
        "mode"
    )
)

print()
print("Notebook:")
print(
    " ",
    NOTEBOOK_OUT.relative_to(
        REPO
    )
)
print(
    "  SHA256:",
    sha256_file(
        NOTEBOOK_OUT
    ),
)
print(
    "  bytes:",
    f"{NOTEBOOK_OUT.stat().st_size:,}",
)
print(
    "  cells:",
    stats,
)

print()
print("Python export:")
print(
    " ",
    PYTHON_OUT.relative_to(
        REPO
    )
)
print(
    "  SHA256:",
    sha256_file(
        PYTHON_OUT
    ),
)
print(
    "  bytes:",
    f"{PYTHON_OUT.stat().st_size:,}",
)

print()
print("Receipt:")
print(
    " ",
    RECEIPT_OUT.relative_to(
        REPO
    )
)
print(
    "  SHA256:",
    sha256_file(
        RECEIPT_OUT
    ),
)

print()
print("Integrity:")
print("  Stage27 scientific artifacts modified : NO")
print("  secret-source scan                    : PASS")
print("  Python source preservation            : PASS")
print("  GitHub file-size gate                 : PASS")

print()
print("Git:")
print("  commit : NOT PERFORMED")
print("  push   : NOT PERFORMED")

print()
print(
    "NEXT: revise PUB2 to include the full .ipynb, "
    ".py, and export receipt in the remote publication closeout."
)

STAGE27-PUB1R — FULL NOTEBOOK / PYTHON REPRODUCIBILITY EXPORT

repository : /kaggle/working/ids2018-validation-safe-ablation
branch     : main
HEAD       : 3407ff3954abae9b0c8bfdaa14b704a05f31affe


RuntimeError: 
PUB1R must run before PUB2 and while HEAD is still the frozen Stage27 scientific parent.

Expected:
  0e1439565aedc7da9b7ca1207262e9061422bc22
Actual:
  3407ff3954abae9b0c8bfdaa14b704a05f31affe

In [12]:
# ============================================================================
# STAGE27-PUB3A
# COMPLETE STAGE27 KAGGLE NOTEBOOK + PYTHON REPRODUCIBILITY EXPORT
#
# CURRENT STATE
#   Scientific freeze:
#     0e1439565aedc7da9b7ca1207262e9061422bc22
#
#   Publication closeout currently expected:
#     3407ff3954abae9b0c8bfdaa14b704a05f31affe
#
# PURPOSE
#   Add the complete Stage27 Kaggle notebook and Python source export
#   AFTER the publication closeout, without altering any frozen science.
#
# OUTPUT
#   scripts/stage27/stage27_loao_unseen_attack.ipynb
#   scripts/stage27/stage27_loao_unseen_attack.py
#
#   results/stage27_loao_unseen_attack/stage27_publication_package/
#       stage27_notebook_export_receipt.json
#
# THIS CELL DOES NOT:
#   - fit models
#   - run inference
#   - reopen targets
#   - reselect thresholds
#   - recompute bootstrap intervals
#   - run new statistical tests
#   - commit
#   - push
# ============================================================================

from __future__ import annotations

import copy
import hashlib
import json
import os
import re
import subprocess
import sys
import urllib.parse
import urllib.request

from pathlib import Path
from datetime import datetime, timezone


# =============================================================================
# 0. FROZEN / CURRENT REPOSITORY IDENTITY
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

SCIENTIFIC_FREEZE = (
    "0e1439565aedc7da9b7ca1207262e9061422bc22"
)

KNOWN_PUBLICATION_CLOSEOUT = (
    "3407ff3954abae9b0c8bfdaa14b704a05f31affe"
)

STAGE_DIR = (
    REPO
    / "scripts"
    / "stage27"
)

NOTEBOOK_REL = Path(
    "scripts/stage27/stage27_loao_unseen_attack.ipynb"
)

PYTHON_REL = Path(
    "scripts/stage27/stage27_loao_unseen_attack.py"
)

RECEIPT_REL = Path(
    "results/stage27_loao_unseen_attack/"
    "stage27_publication_package/"
    "stage27_notebook_export_receipt.json"
)

NOTEBOOK_OUT = (
    REPO
    / NOTEBOOK_REL
)

PYTHON_OUT = (
    REPO
    / PYTHON_REL
)

RECEIPT_OUT = (
    REPO
    / RECEIPT_REL
)

MAX_GITHUB_BYTES = (
    90
    * 1024
    * 1024
)


# =============================================================================
# 1. HELPERS
# =============================================================================

def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
    text=True,
):
    result = subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        text=text,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    if (
        check
        and
        result.returncode != 0
    ):
        if text:
            print(
                result.stdout
            )
            print(
                result.stderr
            )

        raise RuntimeError(
            f"Command failed ({result.returncode}): "
            + " ".join(
                str(x)
                for x in cmd
            )
        )

    return result


def git(
    *args,
    check=True,
):
    return run(
        [
            "git",
            *args,
        ],
        check=check,
    ).stdout.strip()


def sha256_file(
    path,
):
    path = Path(
        path
    )

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as fh:

        while True:

            block = fh.read(
                8
                * 1024
                * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def status_paths():
    output = git(
        "status",
        "--porcelain",
        "-uall",
    )

    result = set()

    if not output:
        return result

    for line in output.splitlines():

        raw = line[3:]

        if " -> " in raw:

            raw = raw.split(
                " -> ",
                1,
            )[1]

        result.add(
            raw
        )

    return result


def notebook_source_text(
    notebook,
):
    chunks = []

    for cell in notebook.get(
        "cells",
        [],
    ):

        source = cell.get(
            "source",
            "",
        )

        if isinstance(
            source,
            list,
        ):

            source = "".join(
                source
            )

        chunks.append(
            str(
                source
            )
        )

    return "\n".join(
        chunks
    )


def notebook_serialized_text(
    notebook,
):
    return json.dumps(
        notebook,
        ensure_ascii=False,
    )


def cell_stats(
    notebook,
):
    cells = notebook.get(
        "cells",
        [],
    )

    return {
        "total":
            len(
                cells
            ),

        "code":
            sum(
                1
                for cell in cells
                if cell.get(
                    "cell_type"
                ) == "code"
            ),

        "markdown":
            sum(
                1
                for cell in cells
                if cell.get(
                    "cell_type"
                ) == "markdown"
            ),

        "raw":
            sum(
                1
                for cell in cells
                if cell.get(
                    "cell_type"
                ) == "raw"
            ),

        "executed_code":
            sum(
                1
                for cell in cells
                if (
                    cell.get(
                        "cell_type"
                    ) == "code"
                    and
                    cell.get(
                        "execution_count"
                    ) is not None
                )
            ),

        "code_with_outputs":
            sum(
                1
                for cell in cells
                if (
                    cell.get(
                        "cell_type"
                    ) == "code"
                    and
                    len(
                        cell.get(
                            "outputs",
                            [],
                        )
                    ) > 0
                )
            ),
    }


# =============================================================================
# 2. REPOSITORY LINEAGE GATE
# =============================================================================

print(
    "=" * 96
)

print(
    "STAGE27-PUB3A — COMPLETE NOTEBOOK "
    "REPRODUCIBILITY EXPORT"
)

print(
    "=" * 96
)


if not REPO.is_dir():

    raise RuntimeError(
        "Repository checkout not found."
    )


branch = git(
    "branch",
    "--show-current",
)

head = git(
    "rev-parse",
    "HEAD",
)

print()
print(
    "repository       :",
    REPO,
)

print(
    "branch           :",
    branch,
)

print(
    "current HEAD     :",
    head,
)

print(
    "scientific freeze:",
    SCIENTIFIC_FREEZE,
)


if branch != "main":

    raise RuntimeError(
        f"Expected main branch; "
        f"found {branch}"
    )


# Critical change from old PUB1R:
# scientific freeze must be AN ANCESTOR,
# not necessarily equal to current HEAD.

ancestor_check = run(
    [
        "git",
        "merge-base",
        "--is-ancestor",
        SCIENTIFIC_FREEZE,
        head,
    ],
    check=False,
)


if (
    ancestor_check.returncode
    != 0
):

    raise RuntimeError(
        "\nScientific lineage gate failed.\n"
        "The frozen Stage27 scientific commit is "
        "not an ancestor of current HEAD."
    )


print(
    "[PASS] frozen Stage27 scientific commit "
    "is an ancestor of current HEAD"
)


# =============================================================================
# 3. REMOTE SYNCHRONIZATION GATE
# =============================================================================

print()
print(
    "-" * 96
)
print(
    "REMOTE SYNCHRONIZATION"
)
print(
    "-" * 96
)


git(
    "fetch",
    "--prune",
    "origin",
    "main",
)


origin_main = git(
    "rev-parse",
    "origin/main",
)


print(
    "local HEAD :",
    head,
)

print(
    "origin/main:",
    origin_main,
)


if (
    origin_main
    != head
):

    raise RuntimeError(
        "\nLocal and remote main differ.\n"
        "Resolve this before creating the notebook export."
    )


print(
    "[PASS] local HEAD == origin/main"
)


if (
    head
    == KNOWN_PUBLICATION_CLOSEOUT
):

    print(
        "[PASS] current HEAD is the known "
        "Stage27 publication-closeout commit"
    )

else:

    print(
        "[INFO] HEAD has advanced beyond the known "
        "publication-closeout commit."
    )

    known_ancestor = run(
        [
            "git",
            "merge-base",
            "--is-ancestor",
            KNOWN_PUBLICATION_CLOSEOUT,
            head,
        ],
        check=False,
    )

    if (
        known_ancestor.returncode
        != 0
    ):

        raise RuntimeError(
            "Current HEAD is not descended from the known "
            "Stage27 publication closeout."
        )

    print(
        "[PASS] known Stage27 publication closeout "
        "is also an ancestor"
    )


# =============================================================================
# 4. REQUIRE CLEAN WORKTREE
# =============================================================================

print()
print(
    "-" * 96
)
print(
    "WORKTREE GATE"
)
print(
    "-" * 96
)


before_paths = status_paths()


if before_paths:

    print(
        git(
            "status",
            "--short",
            "-uall",
        )
    )

    raise RuntimeError(
        "\nPUB3A requires a clean repository before "
        "creating the notebook export."
    )


print(
    "[PASS] worktree clean"
)


# =============================================================================
# 5. VERIFY FROZEN STAGE27 SCIENCE PATH
# =============================================================================

SYNTHESIS = (
    REPO
    / "results"
    / "stage27_loao_unseen_attack"
    / "stage27_4a_final_synthesis"
)


required_frozen = [
    (
        "stage27_final_primary_metrics.csv",
        "42ea04b3f21e6026d5d69c8d5b59aa1edd2b57e94c42da3b9f70587349704634",
    ),
    (
        "stage27_final_operating_points.csv",
        "664a5aaaff718f20bf6d619ae1dd4871a07a37c81a1631590423ea0ae07240f4",
    ),
    (
        "stage27_final_novelty_gaps.csv",
        "91c80319186fd3bbfc382e58cfc60e58fc75d23db15408564fd35c05d4fb316c",
    ),
    (
        "stage27_final_similarity.csv",
        "8c110b4f1d6317d2a2125b4f24bfb8325cdeb699ce763d268d91f3bad6acc8d3",
    ),
]


for (
    filename,
    expected,
) in required_frozen:

    path = (
        SYNTHESIS
        / filename
    )

    if not path.is_file():

        raise RuntimeError(
            f"Missing frozen Stage27 artifact: {path}"
        )

    actual = sha256_file(
        path
    )

    if actual != expected:

        raise RuntimeError(
            f"Frozen Stage27 SHA mismatch:\n"
            f"{filename}\n"
            f"expected: {expected}\n"
            f"actual:   {actual}"
        )


print(
    "[PASS] frozen Stage27 science hash anchors"
)


# =============================================================================
# 6. NBFMT
# =============================================================================

try:

    import nbformat

except Exception as exc:

    raise RuntimeError(
        "nbformat is unavailable."
    ) from exc


# =============================================================================
# 7. CURRENT KERNEL ID
# =============================================================================

def get_kernel_id():

    try:

        from ipykernel.connect import (
            get_connection_file,
        )

        filename = Path(
            get_connection_file()
        ).name

        match = re.search(
            r"kernel-(.+)\.json$",
            filename,
        )

        if match:

            return match.group(
                1
            )

    except Exception:

        pass

    return None


KERNEL_ID = (
    get_kernel_id()
)


print()
print(
    "kernel ID:",
    KERNEL_ID
    if KERNEL_ID
    else "[unavailable]",
)


# =============================================================================
# 8. HTTP JSON HELPER
# =============================================================================

def http_json(
    url,
    timeout=6,
):

    request = urllib.request.Request(
        url,
        headers={
            "Accept":
                "application/json",
        },
    )

    with urllib.request.urlopen(
        request,
        timeout=timeout,
    ) as response:

        return json.loads(
            response.read().decode(
                "utf-8"
            )
        )


# =============================================================================
# 9. DISCOVER LOCAL JUPYTER SERVER(S)
# =============================================================================

def discover_servers():

    candidates = []

    commands = [
        [
            "jupyter",
            "server",
            "list",
            "--json",
        ],
        [
            "jupyter",
            "notebook",
            "list",
            "--json",
        ],
    ]

    for command in commands:

        result = run(
            command,
            cwd=Path(
                "/kaggle/working"
            ),
            check=False,
        )

        if (
            result.returncode
            != 0
        ):

            continue

        for line in result.stdout.splitlines():

            line = line.strip()

            if not line:

                continue

            try:

                record = json.loads(
                    line
                )

            except Exception:

                continue

            if isinstance(
                record,
                dict,
            ):

                candidates.append(
                    record
                )


    unique = []

    seen = set()

    for record in candidates:

        key = (
            record.get(
                "url"
            ),
            record.get(
                "root_dir"
            )
            or record.get(
                "notebook_dir"
            ),
        )

        if key in seen:

            continue

        seen.add(
            key
        )

        unique.append(
            record
        )

    return unique


# =============================================================================
# 10. EXACT LIVE NOTEBOOK THROUGH JUPYTER CONTENTS API
# =============================================================================

def acquire_from_jupyter_api():

    if not KERNEL_ID:

        return (
            None,
            None,
        )


    for server in discover_servers():

        base_url = server.get(
            "url"
        )

        if not base_url:

            continue

        if not base_url.endswith(
            "/"
        ):

            base_url += "/"


        token = (
            server.get(
                "token"
            )
            or ""
        )


        token_query = ""

        if token:

            token_query = (
                "?token="
                + urllib.parse.quote(
                    token
                )
            )


        sessions_url = (
            urllib.parse.urljoin(
                base_url,
                "api/sessions",
            )
            + token_query
        )


        try:

            sessions = http_json(
                sessions_url
            )

        except Exception:

            continue


        for session in sessions:

            kernel = session.get(
                "kernel",
                {},
            )


            if (
                kernel.get(
                    "id"
                )
                != KERNEL_ID
            ):

                continue


            notebook_meta = (
                session.get(
                    "notebook"
                )
                or {}
            )


            notebook_path = (
                notebook_meta.get(
                    "path"
                )
                or session.get(
                    "path"
                )
            )


            if not notebook_path:

                continue


            encoded = urllib.parse.quote(
                notebook_path,
                safe="/",
            )


            contents_url = (
                urllib.parse.urljoin(
                    base_url,
                    "api/contents/"
                    + encoded,
                )
                + token_query
            )


            try:

                model = http_json(
                    contents_url
                )

            except Exception:

                continue


            if (
                model.get(
                    "type"
                )
                != "notebook"
            ):

                continue


            content = model.get(
                "content"
            )


            if not isinstance(
                content,
                dict,
            ):

                continue


            return (
                content,
                {
                    "mode":
                        "EXACT_JUPYTER_CONTENTS_API",

                    "notebook_path":
                        notebook_path,

                    "server_root":
                        server.get(
                            "root_dir"
                        )
                        or server.get(
                            "notebook_dir"
                        ),
                },
            )


    return (
        None,
        None,
    )


# =============================================================================
# 11. EXACT NOTEBOOK FROM KAGGLE FILESYSTEM
# =============================================================================

def local_ipynb_candidates():

    candidates = []

    roots = [
        Path(
            "/kaggle/working"
        ),
    ]


    explicit = [
        Path(
            "/kaggle/working/__notebook__.ipynb"
        ),
        Path(
            "/kaggle/working/notebook.ipynb"
        ),
    ]


    for path in explicit:

        if path.is_file():

            candidates.append(
                path
            )


    for root in roots:

        try:

            candidates.extend(
                root.glob(
                    "*.ipynb"
                )
            )

        except Exception:

            pass


    unique = []

    seen = set()


    for path in candidates:

        try:

            resolved = path.resolve()

        except Exception:

            continue


        if (
            resolved
            == NOTEBOOK_OUT.resolve()
        ):

            continue


        if resolved in seen:

            continue


        seen.add(
            resolved
        )

        unique.append(
            resolved
        )


    return unique


def acquire_from_local_file():

    possible = []


    for path in local_ipynb_candidates():

        try:

            nb = nbformat.read(
                path,
                as_version=4,
            )

        except Exception:

            continue


        source = notebook_source_text(
            nb
        )


        stage27_count = (
            source.upper().count(
                "STAGE27"
            )
        )


        if (
            stage27_count
            < 10
        ):

            continue


        possible.append(
            (
                path,
                nb,
                stage27_count,
            )
        )


    if not possible:

        return (
            None,
            None,
        )


    possible.sort(
        key=lambda item: (
            item[2],
            item[0].stat().st_size,
        ),
        reverse=True,
    )


    (
        path,
        nb,
        stage27_count,
    ) = possible[0]


    return (
        nb,
        {
            "mode":
                "EXACT_LOCAL_NOTEBOOK_FILE",

            "notebook_path":
                str(
                    path
                ),

            "source_bytes":
                path.stat().st_size,

            "stage27_occurrences":
                stage27_count,
        },
    )


# =============================================================================
# 12. FALLBACK — IPYTHON EXECUTION HISTORY
# =============================================================================

def acquire_from_history():

    try:

        ip = get_ipython()

    except Exception:

        ip = None


    if ip is None:

        return (
            None,
            None,
        )


    history = []


    try:

        iterator = (
            ip
            .history_manager
            .get_range(
                session=0,
                start=1,
                stop=None,
                raw=True,
                output=False,
            )
        )


        for (
            session,
            line_no,
            source,
        ) in iterator:

            if not source:

                continue


            history.append(
                {
                    "session":
                        int(
                            session
                        ),

                    "line_no":
                        int(
                            line_no
                        ),

                    "source":
                        str(
                            source
                        ),
                }
            )


    except Exception as exc:

        print(
            "[WARN] IPython history unavailable:",
            repr(
                exc
            ),
        )

        return (
            None,
            None,
        )


    joined = "\n\n".join(
        item[
            "source"
        ]
        for item in history
    )


    stage27_count = (
        joined.upper().count(
            "STAGE27"
        )
    )


    print()
    print(
        "History fallback candidate:"
    )
    print(
        "  input cells        :",
        len(
            history
        ),
    )
    print(
        "  STAGE27 occurrences:",
        stage27_count,
    )


    # Hard block against exporting only PUB cells as "full Stage27".
    if (
        len(
            history
        )
        < 12
    ):

        return (
            None,
            {
                "mode":
                    "HISTORY_REJECTED_TOO_SMALL",

                "input_cells":
                    len(
                        history
                    ),

                "stage27_occurrences":
                    stage27_count,
            },
        )


    if (
        stage27_count
        < 15
    ):

        return (
            None,
            {
                "mode":
                    "HISTORY_REJECTED_TOO_LITTLE_STAGE27_CONTENT",

                "input_cells":
                    len(
                        history
                    ),

                "stage27_occurrences":
                    stage27_count,
            },
        )


    cells = []


    for index, item in enumerate(
        history,
        start=1,
    ):

        cell = (
            nbformat
            .v4
            .new_code_cell(
                source=item[
                    "source"
                ]
            )
        )


        cell[
            "execution_count"
        ] = item[
            "line_no"
        ]


        cell[
            "outputs"
        ] = []


        cell[
            "metadata"
        ] = {
            "stage27_history_index":
                index,

            "ipython_session":
                item[
                    "session"
                ],

            "ipython_line_number":
                item[
                    "line_no"
                ],
        }


        cells.append(
            cell
        )


    nb = (
        nbformat
        .v4
        .new_notebook(
            cells=cells
        )
    )


    return (
        nb,
        {
            "mode":
                "RECONSTRUCTED_FROM_IPYTHON_INPUT_HISTORY",

            "input_cells":
                len(
                    history
                ),

            "stage27_occurrences":
                stage27_count,

            "limitations": [
                (
                    "original cell outputs are not recoverable "
                    "from input history"
                ),
                (
                    "unexecuted cells are not recoverable from "
                    "input history"
                ),
                (
                    "markdown-only cells are not recoverable "
                    "unless executed as source"
                ),
            ],
        },
    )


# =============================================================================
# 13. ACQUIRE BEST NOTEBOOK
# =============================================================================

print()
print(
    "-" * 96
)
print(
    "NOTEBOOK ACQUISITION"
)
print(
    "-" * 96
)


notebook = None

acquisition = None


# Preference 1
try:

    (
        notebook,
        acquisition,
    ) = acquire_from_jupyter_api()

except Exception as exc:

    print(
        "[WARN] Jupyter API attempt:",
        repr(
            exc
        ),
    )


if notebook is not None:

    print(
        "[PASS] exact live notebook captured "
        "through Jupyter Contents API"
    )


# Preference 2
if notebook is None:

    try:

        (
            notebook,
            acquisition,
        ) = acquire_from_local_file()

    except Exception as exc:

        print(
            "[WARN] local-file attempt:",
            repr(
                exc
            ),
        )


    if notebook is not None:

        print(
            "[PASS] exact notebook captured from "
            "Kaggle filesystem"
        )


# Preference 3
history_failure = None


if notebook is None:

    (
        notebook,
        history_info,
    ) = acquire_from_history()


    if notebook is not None:

        acquisition = (
            history_info
        )

        print(
            "[PASS] notebook reconstructed from "
            "sufficiently complete execution history"
        )

    else:

        history_failure = (
            history_info
        )


if notebook is None:

    print()
    print(
        "History diagnostic:"
    )

    print(
        json.dumps(
            history_failure,
            indent=2,
        )
        if history_failure
        else "[none]"
    )


    raise RuntimeError(
        "\nFULL NOTEBOOK EXPORT COULD NOT BE VERIFIED.\n\n"
        "No exact live .ipynb was accessible and the current "
        "IPython history was not complete enough to be safely "
        "labeled as the full Stage27 notebook.\n\n"
        "Do NOT substitute a partial PUB-only notebook."
    )


# =============================================================================
# 14. NORMALIZE THROUGH NBFMT
# =============================================================================

# Validate/migrate notebook structure.
notebook = nbformat.from_dict(
    copy.deepcopy(
        notebook
    )
)

notebook = nbformat.convert(
    notebook,
    4,
)


stats = cell_stats(
    notebook
)

source_text = notebook_source_text(
    notebook
)

stage27_mentions = (
    source_text.upper().count(
        "STAGE27"
    )
)


print()
print(
    "-" * 96
)
print(
    "NOTEBOOK COMPLETENESS AUDIT"
)
print(
    "-" * 96
)


print(
    "Acquisition:"
)

print(
    json.dumps(
        acquisition,
        indent=2,
    )
)


print()
print(
    "Cell statistics:"
)

print(
    json.dumps(
        stats,
        indent=2,
    )
)


print(
    "STAGE27 occurrences:",
    stage27_mentions,
)


if (
    stats[
        "code"
    ]
    < 10
):

    raise RuntimeError(
        "Notebook has too few code cells "
        "to represent complete Stage27."
    )


if (
    stage27_mentions
    < 15
):

    raise RuntimeError(
        "Notebook contains insufficient Stage27 source markers."
    )


print(
    "[PASS] notebook completeness floor"
)


# =============================================================================
# 15. SECRET SAFETY SCAN
# =============================================================================

print()
print(
    "-" * 96
)
print(
    "SECRET-SAFETY AUDIT"
)
print(
    "-" * 96
)


serialized = notebook_serialized_text(
    notebook
)


secret_patterns = {
    "GitHub classic PAT":
        r"\bghp_[A-Za-z0-9]{20,}\b",

    "GitHub fine-grained PAT":
        r"\bgithub_pat_[A-Za-z0-9_]{20,}\b",

    "GitHub OAuth":
        r"\bgho_[A-Za-z0-9]{20,}\b",

    "GitHub user token":
        r"\bghu_[A-Za-z0-9]{20,}\b",

    "GitHub server token":
        r"\bghs_[A-Za-z0-9]{20,}\b",

    "Stripe live secret":
        r"\bsk_live_[A-Za-z0-9]{16,}\b",

    "AWS access key":
        r"\bAKIA[0-9A-Z]{16}\b",
}


secret_hits = []


for (
    label,
    pattern,
) in secret_patterns.items():

    matches = re.findall(
        pattern,
        serialized,
    )


    if matches:

        secret_hits.append(
            {
                "type":
                    label,

                "count":
                    len(
                        matches
                    ),
            }
        )


if secret_hits:

    print(
        json.dumps(
            secret_hits,
            indent=2,
        )
    )

    raise RuntimeError(
        "\nPotential credential material exists in the notebook.\n"
        "GitHub export blocked."
    )


print(
    "[PASS] no recognized credential-token "
    "patterns in notebook source/outputs"
)


# =============================================================================
# 16. ADD REPRODUCIBILITY METADATA
# =============================================================================

notebook.setdefault(
    "metadata",
    {},
)


notebook[
    "metadata"
][
    "stage27_reproducibility_export"
] = {
    "export_phase":
        "STAGE27-PUB3A",

    "scientific_parent":
        SCIENTIFIC_FREEZE,

    "publication_closeout_parent":
        head,

    "scientific_status":
        "CLOSED",

    "acquisition":
        acquisition,

    "science_operations_during_export": {
        "model_fits": 0,
        "model_inference": 0,
        "target_reopenings": 0,
        "threshold_reselection": 0,
        "bootstrap_recomputation": 0,
        "new_formal_statistical_tests": 0,
    },
}


# =============================================================================
# 17. WRITE COMPLETE NOTEBOOK
# =============================================================================

STAGE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


NOTEBOOK_OUT.parent.mkdir(
    parents=True,
    exist_ok=True,
)


nbformat.write(
    notebook,
    NOTEBOOK_OUT,
)


if not NOTEBOOK_OUT.is_file():

    raise RuntimeError(
        "Notebook export was not created."
    )


print()
print(
    "-" * 96
)
print(
    "COMPLETE IPYNB EXPORT"
)
print(
    "-" * 96
)


print(
    NOTEBOOK_REL
)

print(
    "bytes :",
    f"{NOTEBOOK_OUT.stat().st_size:,}",
)

print(
    "SHA256:",
    sha256_file(
        NOTEBOOK_OUT
    ),
)


# =============================================================================
# 18. BUILD PYTHON SOURCE EXPORT
# =============================================================================

def markdown_to_comments(
    text,
):
    lines = str(
        text
    ).splitlines()

    if not lines:

        return "#"

    return "\n".join(
        "#"
        if line == ""
        else "# " + line

        for line in lines
    )


python_parts = [
    (
        "# "
        + "=" * 78
    ),
    (
        "# STAGE27 — LEAVE-ONE-ATTACK-FAMILY-OUT "
        "UNSEEN-FAMILY GENERALIZATION AUDIT"
    ),
    (
        "# Complete source export of the Stage27 Kaggle notebook."
    ),
    "#",
    (
        "# Frozen scientific parent:"
    ),
    (
        "# "
        + SCIENTIFIC_FREEZE
    ),
    "#",
    (
        "# Publication-closeout parent at export:"
    ),
    (
        "# "
        + head
    ),
    "#",
    "# Scientific state: CLOSED",
    "#",
    (
        "# This file preserves notebook source ordering."
    ),
    (
        "# It does not authorize new Stage27 scientific computation."
    ),
    (
        "# "
        + "=" * 78
    ),
    "",
]


for (
    index,
    cell,
) in enumerate(
    notebook.get(
        "cells",
        [],
    ),
    start=1,
):

    cell_type = cell.get(
        "cell_type",
        "unknown",
    )


    source = cell.get(
        "source",
        "",
    )


    if isinstance(
        source,
        list,
    ):

        source = "".join(
            source
        )


    source = str(
        source
    )


    if (
        cell_type
        == "code"
    ):

        python_parts.extend([
            "",
            (
                f"# %% [Stage27 notebook cell {index}]"
            ),
            source.rstrip(),
            "",
        ])


    elif (
        cell_type
        == "markdown"
    ):

        python_parts.extend([
            "",
            (
                f"# %% [markdown — Stage27 notebook cell {index}]"
            ),
            markdown_to_comments(
                source
            ),
            "",
        ])


    elif (
        cell_type
        == "raw"
    ):

        python_parts.extend([
            "",
            (
                f"# %% [raw — Stage27 notebook cell {index}]"
            ),
            markdown_to_comments(
                source
            ),
            "",
        ])


PYTHON_OUT.write_text(
    "\n".join(
        python_parts
    ).rstrip()
    + "\n",
    encoding="utf-8",
    newline="\n",
)


print()
print(
    "-" * 96
)
print(
    "PYTHON SOURCE EXPORT"
)
print(
    "-" * 96
)


print(
    PYTHON_REL
)

print(
    "bytes :",
    f"{PYTHON_OUT.stat().st_size:,}",
)

print(
    "SHA256:",
    sha256_file(
        PYTHON_OUT
    ),
)


# =============================================================================
# 19. CODE-SOURCE PRESERVATION AUDIT
# =============================================================================

python_text = (
    PYTHON_OUT
    .read_text(
        encoding="utf-8"
    )
)


missing_cells = []


code_index = 0


for cell in notebook.get(
    "cells",
    [],
):

    if (
        cell.get(
            "cell_type"
        )
        != "code"
    ):

        continue


    code_index += 1


    source = cell.get(
        "source",
        "",
    )


    if isinstance(
        source,
        list,
    ):

        source = "".join(
            source
        )


    source = str(
        source
    ).rstrip()


    if (
        source
        and
        source not in python_text
    ):

        missing_cells.append(
            code_index
        )


if missing_cells:

    raise RuntimeError(
        "Python export source-preservation failure. "
        f"Missing code cells: {missing_cells[:30]}"
    )


print()
print(
    "[PASS] every notebook code-cell source "
    "exists in Python export"
)


# =============================================================================
# 20. GITHUB FILE-SIZE SAFETY
# =============================================================================

for path in [
    NOTEBOOK_OUT,
    PYTHON_OUT,
]:

    if (
        path.stat().st_size
        > MAX_GITHUB_BYTES
    ):

        raise RuntimeError(
            "\nGitHub-safe file-size gate failed:\n"
            f"{path}\n"
            f"{path.stat().st_size:,} bytes"
        )


print(
    "[PASS] GitHub-safe file sizes"
)


# =============================================================================
# 21. WRITE EXPORT RECEIPT
# =============================================================================

RECEIPT_OUT.parent.mkdir(
    parents=True,
    exist_ok=True,
)


receipt = {
    "stage":
        "STAGE27-PUB3A",

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "repository":
        "themubasshir/ids2018-validation-safe-ablation",

    "scientific_parent":
        SCIENTIFIC_FREEZE,

    "publication_closeout_parent":
        head,

    "scientific_status":
        "CLOSED",

    "acquisition":
        acquisition,

    "notebook": {
        "path":
            NOTEBOOK_REL.as_posix(),

        "sha256":
            sha256_file(
                NOTEBOOK_OUT
            ),

        "bytes":
            NOTEBOOK_OUT.stat().st_size,

        "cells":
            stats,

        "stage27_source_occurrences":
            stage27_mentions,
    },

    "python_export": {
        "path":
            PYTHON_REL.as_posix(),

        "sha256":
            sha256_file(
                PYTHON_OUT
            ),

        "bytes":
            PYTHON_OUT.stat().st_size,

        "all_code_cell_sources_preserved":
            True,
    },

    "secret_scan": {
        "status":
            "PASS",

        "recognized_token_patterns":
            0,
    },

    "science_operations": {
        "model_fits": 0,
        "model_inference": 0,
        "target_reopenings": 0,
        "threshold_reselection": 0,
        "bootstrap_recomputation": 0,
        "new_formal_statistical_tests": 0,
    },

    "git_operations": {
        "commit":
            False,

        "push":
            False,
    },
}


RECEIPT_OUT.write_text(
    json.dumps(
        receipt,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
    newline="\n",
)


print()
print(
    "-" * 96
)
print(
    "NOTEBOOK EXPORT RECEIPT"
)
print(
    "-" * 96
)


print(
    RECEIPT_REL
)

print(
    "SHA256:",
    sha256_file(
        RECEIPT_OUT
    ),
)


# =============================================================================
# 22. SCIENCE PATH MUST STILL BE UNCHANGED
# =============================================================================

science_diff = run(
    [
        "git",
        "diff",
        "--",
        (
            "results/stage27_loao_unseen_attack/"
            "stage27_4a_final_synthesis"
        ),
    ]
).stdout


if science_diff.strip():

    print(
        science_diff
    )

    raise RuntimeError(
        "Frozen Stage27 science was modified."
    )


print()
print(
    "[PASS] frozen Stage27-4A synthesis untouched"
)


# =============================================================================
# 23. EXACT WORKTREE INVENTORY
# =============================================================================

expected_new = {
    NOTEBOOK_REL.as_posix(),
    PYTHON_REL.as_posix(),
    RECEIPT_REL.as_posix(),
}


actual_new = status_paths()


print()
print(
    "-" * 96
)
print(
    "GIT STATUS"
)
print(
    "-" * 96
)


print(
    git(
        "status",
        "--short",
        "-uall",
    )
)


if (
    actual_new
    != expected_new
):

    print()
    print(
        "Expected:"
    )

    for item in sorted(
        expected_new
    ):

        print(
            " ",
            item
        )


    print()
    print(
        "Actual:"
    )

    for item in sorted(
        actual_new
    ):

        print(
            " ",
            item
        )


    raise RuntimeError(
        "PUB3A produced unexpected worktree changes."
    )


print()
print(
    "[PASS] exactly three reproducibility artifacts created"
)


# =============================================================================
# 24. FINAL
# =============================================================================

print()
print(
    "=" * 96
)

print(
    "STAGE27-PUB3A COMPLETE — "
    "FULL NOTEBOOK EXPORT READY FOR REVIEW"
)

print(
    "=" * 96
)


print()
print(
    "Scientific freeze:"
)

print(
    " ",
    SCIENTIFIC_FREEZE,
)


print()
print(
    "Publication-closeout parent:"
)

print(
    " ",
    head,
)


print()
print(
    "Acquisition mode:"
)

print(
    " ",
    acquisition[
        "mode"
    ],
)


print()
print(
    "Notebook:"
)

print(
    " ",
    NOTEBOOK_REL
)

print(
    "  SHA256:",
    sha256_file(
        NOTEBOOK_OUT
    ),
)

print(
    "  bytes :",
    f"{NOTEBOOK_OUT.stat().st_size:,}",
)

print(
    "  cells :",
    stats,
)


print()
print(
    "Python export:"
)

print(
    " ",
    PYTHON_REL
)

print(
    "  SHA256:",
    sha256_file(
        PYTHON_OUT
    ),
)

print(
    "  bytes :",
    f"{PYTHON_OUT.stat().st_size:,}",
)


print()
print(
    "Receipt:"
)

print(
    " ",
    RECEIPT_REL
)

print(
    "  SHA256:",
    sha256_file(
        RECEIPT_OUT
    ),
)


print()
print(
    "Integrity:"
)

print(
    "  frozen Stage27 science : UNCHANGED"
)

print(
    "  source preservation    : PASS"
)

print(
    "  secret scan            : PASS"
)

print(
    "  file-size gate         : PASS"
)


print()
print(
    "Git operations:"
)

print(
    "  commit : NOT PERFORMED"
)

print(
    "  push   : NOT PERFORMED"
)


print()
print(
    "NEXT:"
)

print(
    "  STAGE27-PUB3B — commit/push notebook + "
    "Python export + receipt and append them to "
    "the Stage27 publication closeout."
)

STAGE27-PUB3A — COMPLETE NOTEBOOK REPRODUCIBILITY EXPORT

repository       : /kaggle/working/ids2018-validation-safe-ablation
branch           : main
current HEAD     : 3407ff3954abae9b0c8bfdaa14b704a05f31affe
scientific freeze: 0e1439565aedc7da9b7ca1207262e9061422bc22
[PASS] frozen Stage27 scientific commit is an ancestor of current HEAD

------------------------------------------------------------------------------------------------
REMOTE SYNCHRONIZATION
------------------------------------------------------------------------------------------------
local HEAD : 3407ff3954abae9b0c8bfdaa14b704a05f31affe
origin/main: 3407ff3954abae9b0c8bfdaa14b704a05f31affe
[PASS] local HEAD == origin/main
[PASS] current HEAD is the known Stage27 publication-closeout commit

------------------------------------------------------------------------------------------------
WORKTREE GATE
------------------------------------------------------------------------------------------------
[PASS] worktree clean


In [13]:
# ============================================================================
# STAGE27-PUB3B
# FREEZE + PUSH COMPLETE EXECUTED-SOURCE NOTEBOOK RECONSTRUCTION
#
# Parent:
#   3407ff3954abae9b0c8bfdaa14b704a05f31affe
#
# Adds:
#   scripts/stage27/stage27_loao_unseen_attack.ipynb
#   scripts/stage27/stage27_loao_unseen_attack.py
#   results/stage27_loao_unseen_attack/stage27_publication_package/
#       stage27_notebook_export_receipt.json
#
# Updates:
#   docs/STAGE27_PUBLICATION_CLOSEOUT.md
#
# NO scientific computation is performed.
# ============================================================================

from __future__ import annotations

import hashlib
import json
import os
import re
import stat
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path


# =============================================================================
# 0. IDENTITY
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

REPO_FULL = (
    "themubasshir/ids2018-validation-safe-ablation"
)

SCIENTIFIC_FREEZE = (
    "0e1439565aedc7da9b7ca1207262e9061422bc22"
)

EXPECTED_PARENT = (
    "3407ff3954abae9b0c8bfdaa14b704a05f31affe"
)

NOTEBOOK_REL = Path(
    "scripts/stage27/stage27_loao_unseen_attack.ipynb"
)

PYTHON_REL = Path(
    "scripts/stage27/stage27_loao_unseen_attack.py"
)

RECEIPT_REL = Path(
    "results/stage27_loao_unseen_attack/"
    "stage27_publication_package/"
    "stage27_notebook_export_receipt.json"
)

CLOSEOUT_REL = Path(
    "docs/STAGE27_PUBLICATION_CLOSEOUT.md"
)

NOTEBOOK = REPO / NOTEBOOK_REL
PYTHON_EXPORT = REPO / PYTHON_REL
RECEIPT = REPO / RECEIPT_REL
CLOSEOUT = REPO / CLOSEOUT_REL


EXPECTED_EXPORT_SHA256 = {
    NOTEBOOK_REL.as_posix():
        "cc9387c8c41f70a6e38cef133b6230bda9b5aae20af6c13dc710c545754dad3b",

    PYTHON_REL.as_posix():
        "f25967bbbd463db3942037de02cfa35f2c02d79f94aeaa408615a7b1bd90e7df",

    RECEIPT_REL.as_posix():
        "e3723b70bcef74e59647eee9c05701e5272b16d001851e8a03ca6f699bb49162",
}


# =============================================================================
# 1. HELPERS
# =============================================================================

def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
    text=True,
    env=None,
):
    result = subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        text=text,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        env=env,
    )

    if (
        check
        and
        result.returncode != 0
    ):
        if text:
            print(result.stdout)
            print(result.stderr)

        raise RuntimeError(
            f"Command failed ({result.returncode}): "
            + " ".join(
                str(x)
                for x in cmd
            )
        )

    return result


def git(
    *args,
    check=True,
):
    return run(
        [
            "git",
            *args,
        ],
        check=check,
    ).stdout.strip()


def sha256_file(
    path,
):
    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as fh:

        while True:

            block = fh.read(
                8 * 1024 * 1024
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def sha256_bytes(
    data,
):
    return hashlib.sha256(
        data
    ).hexdigest()


def status_paths():
    output = git(
        "status",
        "--porcelain",
        "-uall",
    )

    result = set()

    if not output:
        return result

    for line in output.splitlines():

        raw = line[3:]

        if " -> " in raw:

            raw = raw.split(
                " -> ",
                1,
            )[1]

        result.add(
            raw
        )

    return result


def git_blob_bytes(
    ref,
    rel,
):
    result = run(
        [
            "git",
            "show",
            f"{ref}:{rel.as_posix()}",
        ],
        text=False,
    )

    return result.stdout


# =============================================================================
# 2. HEADER
# =============================================================================

print(
    "=" * 100
)

print(
    "STAGE27-PUB3B — FREEZE / PUSH "
    "EXECUTED-SOURCE NOTEBOOK RECONSTRUCTION"
)

print(
    "=" * 100
)

print()
print(
    "timestamp_utc:",
    datetime.now(
        timezone.utc
    ).isoformat(),
)

print(
    "repository:",
    REPO_FULL,
)

print(
    "scientific freeze:",
    SCIENTIFIC_FREEZE,
)

print(
    "expected parent:",
    EXPECTED_PARENT,
)


# =============================================================================
# 3. REPOSITORY GATE
# =============================================================================

branch = git(
    "branch",
    "--show-current",
)

head = git(
    "rev-parse",
    "HEAD",
)

print()
print(
    "branch:",
    branch,
)

print(
    "HEAD:  ",
    head,
)


if branch != "main":

    raise RuntimeError(
        "PUB3B requires branch main."
    )


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "\nUnexpected PUB3B parent.\n\n"
        f"Expected:\n  {EXPECTED_PARENT}\n"
        f"Actual:\n  {head}"
    )


ancestor = run(
    [
        "git",
        "merge-base",
        "--is-ancestor",
        SCIENTIFIC_FREEZE,
        head,
    ],
    check=False,
)


if ancestor.returncode != 0:

    raise RuntimeError(
        "Frozen Stage27 scientific commit is not "
        "an ancestor of current HEAD."
    )


print(
    "[PASS] scientific lineage"
)


# =============================================================================
# 4. REMOTE RACE GATE
# =============================================================================

print()
print(
    "-" * 100
)
print(
    "REMOTE RACE GATE"
)
print(
    "-" * 100
)


git(
    "fetch",
    "--prune",
    "origin",
    "main",
)


origin_main = git(
    "rev-parse",
    "origin/main",
)


print(
    "local HEAD :",
    head,
)

print(
    "origin/main:",
    origin_main,
)


if origin_main != head:

    raise RuntimeError(
        "origin/main advanced or local checkout diverged."
    )


print(
    "[PASS] local == remote publication closeout"
)


# =============================================================================
# 5. EXACT PRE-COMMIT WORKTREE
# =============================================================================

print()
print(
    "-" * 100
)
print(
    "WORKTREE INVENTORY"
)
print(
    "-" * 100
)


expected_untracked = {
    NOTEBOOK_REL.as_posix(),
    PYTHON_REL.as_posix(),
    RECEIPT_REL.as_posix(),
}


actual = status_paths()


print(
    git(
        "status",
        "--short",
        "-uall",
    )
)


if actual != expected_untracked:

    print()
    print(
        "Expected:",
        sorted(
            expected_untracked
        ),
    )

    print(
        "Actual:  ",
        sorted(
            actual
        ),
    )

    raise RuntimeError(
        "PUB3B requires exactly the three PUB3A "
        "reproducibility artifacts to be uncommitted."
    )


print(
    "[PASS] exact PUB3A worktree"
)


# =============================================================================
# 6. VERIFY PUB3A HASHES
# =============================================================================

print()
print(
    "-" * 100
)
print(
    "PUB3A SHA256 GATE"
)
print(
    "-" * 100
)


for rel, expected in EXPECTED_EXPORT_SHA256.items():

    path = REPO / rel

    if not path.is_file():

        raise RuntimeError(
            f"Missing PUB3A artifact: {rel}"
        )

    actual_hash = sha256_file(
        path
    )

    if actual_hash != expected:

        raise RuntimeError(
            f"\nPUB3A artifact changed after review:\n"
            f"{rel}\n"
            f"expected: {expected}\n"
            f"actual:   {actual_hash}"
        )

    print(
        f"[PASS] {rel}\n"
        f"       {actual_hash}"
    )


# =============================================================================
# 7. LOAD NOTEBOOK + RECEIPT
# =============================================================================

try:
    import nbformat
except Exception as exc:
    raise RuntimeError(
        "nbformat unavailable."
    ) from exc


nb = nbformat.read(
    NOTEBOOK,
    as_version=4,
)


receipt = json.loads(
    RECEIPT.read_text(
        encoding="utf-8"
    )
)


# =============================================================================
# 8. RECONSTRUCTION IDENTITY GATE
# =============================================================================

print()
print(
    "-" * 100
)
print(
    "RECONSTRUCTION IDENTITY"
)
print(
    "-" * 100
)


mode = (
    receipt
    .get(
        "acquisition",
        {},
    )
    .get(
        "mode"
    )
)


if (
    mode
    != "RECONSTRUCTED_FROM_IPYTHON_INPUT_HISTORY"
):

    raise RuntimeError(
        f"Unexpected acquisition mode: {mode}"
    )


notebook_meta = receipt.get(
    "notebook",
    {},
)


cells = notebook_meta.get(
    "cells",
    {},
)


if cells.get(
    "total"
) != 12:

    raise RuntimeError(
        "Expected exactly 12 reconstructed executed cells."
    )


if cells.get(
    "code"
) != 12:

    raise RuntimeError(
        "Expected all 12 reconstructed cells to be code cells."
    )


if cells.get(
    "code_with_outputs"
) != 0:

    raise RuntimeError(
        "Reconstructed notebook unexpectedly contains outputs."
    )


if notebook_meta.get(
    "stage27_source_occurrences"
) != 521:

    raise RuntimeError(
        "Stage27 source-occurrence identity mismatch."
    )


print(
    "[PASS] acquisition mode = "
    "RECONSTRUCTED_FROM_IPYTHON_INPUT_HISTORY"
)

print(
    "[PASS] reconstructed executed cells = 12"
)

print(
    "[PASS] Stage27 source occurrences = 521"
)

print(
    "[PASS] notebook outputs intentionally unavailable"
)


# =============================================================================
# 9. SCIENTIFIC-STAGE CONTENT AUDIT
#
# Important:
#   prove this is not merely PUB0/PUB1 source.
# =============================================================================

print()
print(
    "-" * 100
)
print(
    "SCIENTIFIC-STAGE CONTENT AUDIT"
)
print(
    "-" * 100
)


source_chunks = []


for cell in nb.cells:

    source = cell.get(
        "source",
        "",
    )

    source_chunks.append(
        str(
            source
        )
    )


source = "\n\n".join(
    source_chunks
)

source_upper = source.upper()


# We require evidence from all major scientific parts
# plus manuscript publication integration.
required_marker_groups = {
    "Stage27-1 family/fold preparation":
        [
            "STAGE27-1",
            "STAGE27_1",
        ],

    "Stage27-2 model execution":
        [
            "STAGE27-2",
            "STAGE27_2",
        ],

    "Stage27-3 secondary/bootstrap/similarity":
        [
            "STAGE27-3",
            "STAGE27_3",
        ],

    "Stage27-4 final synthesis":
        [
            "STAGE27-4",
            "STAGE27_4",
        ],

    "PUB0":
        [
            "STAGE27-PUB0",
        ],

    "PUB1":
        [
            "STAGE27-PUB1",
        ],
}


marker_evidence = {}


for label, candidates in required_marker_groups.items():

    hits = [
        marker
        for marker in candidates
        if marker in source_upper
    ]

    marker_evidence[
        label
    ] = hits

    if not hits:

        raise RuntimeError(
            f"Reconstructed notebook lacks required "
            f"scientific marker group: {label}"
        )

    print(
        f"[PASS] {label}:",
        hits,
    )


# Additional scientific content tokens.
required_scientific_tokens = [
    "BOT",
    "DDOS",
    "INFILTRATION",
    "PORT_SCAN",
    "WEB_ATTACK",
    "AUTH_BRUTE_FORCE",
    "XGBOOST",
    "LIGHTGBM",
    "TRAIN",
    "VALIDATION",
    "TARGET",
    "BOOTSTRAP",
    "SIMILARITY",
]


for token in required_scientific_tokens:

    if token not in source_upper:

        raise RuntimeError(
            f"Missing scientific source token: {token}"
        )


print(
    "[PASS] family/learner/protocol content present"
)


# =============================================================================
# 10. PYTHON EXPORT IDENTITY
# =============================================================================

py_text = PYTHON_EXPORT.read_text(
    encoding="utf-8"
)


for cell_index, cell in enumerate(
    nb.cells,
    start=1,
):

    if cell.cell_type != "code":
        continue

    cell_source = str(
        cell.source
    ).rstrip()

    if (
        cell_source
        and
        cell_source not in py_text
    ):

        raise RuntimeError(
            f"Python export is missing notebook cell "
            f"{cell_index} source."
        )


print(
    "[PASS] every reconstructed code cell is "
    "represented in the .py export"
)


# =============================================================================
# 11. FROZEN SCIENCE MUST REMAIN UNMODIFIED
# =============================================================================

science_diff = run(
    [
        "git",
        "diff",
        "--",
        (
            "results/stage27_loao_unseen_attack/"
            "stage27_4a_final_synthesis"
        ),
    ]
).stdout


if science_diff.strip():

    print(
        science_diff
    )

    raise RuntimeError(
        "Frozen Stage27 scientific artifacts changed."
    )


print(
    "[PASS] frozen Stage27-4A synthesis untouched"
)


# =============================================================================
# 12. REBUILD CLOSEOUT FROM COMMITTED PARENT + ADDENDUM
#
# This makes the cell rerunnable and prevents duplicate append sections.
# =============================================================================

print()
print(
    "-" * 100
)
print(
    "PUBLICATION CLOSEOUT ADDENDUM"
)
print(
    "-" * 100
)


committed_closeout = (
    git_blob_bytes(
        head,
        CLOSEOUT_REL,
    )
    .decode(
        "utf-8"
    )
)


notebook_hash = sha256_file(
    NOTEBOOK
)

python_hash = sha256_file(
    PYTHON_EXPORT
)

receipt_hash = sha256_file(
    RECEIPT
)


ADDENDUM_HEADER = (
    "## Post-Closeout Stage27 Notebook Reproducibility Addendum"
)


addendum = f"""
{ADDENDUM_HEADER}

After the Stage27 manuscript-integration closeout, the executed source
of the Stage27 Kaggle workflow was preserved as an additional
reproducibility artifact.

### Export Provenance

Acquisition mode:

`RECONSTRUCTED_FROM_IPYTHON_INPUT_HISTORY`

The Kaggle runtime did not expose the original live notebook document
through an exact notebook-file or Jupyter Contents API snapshot.
Consequently, the repository notebook is an executed-source
reconstruction rather than a byte-identical copy of the original
Kaggle `.ipynb`.

The reconstruction preserves:

- all 12 executed code-cell sources;
- execution ordering;
- Stage27 scientific and publication source executed in this kernel;
- 521 Stage27 source references;
- the complete reconstructed source in both `.ipynb` and `.py` form.

The reconstruction does not preserve:

- original cell outputs;
- unexecuted notebook cells;
- notebook-only markdown that was never present in executable history;
- the original Kaggle notebook user-interface metadata.

These limitations concern notebook-container provenance only. They do
not reopen or modify any Stage27 scientific result, because all
scientific outputs remain independently frozen by the Stage27 result
artifacts, receipts, source hashes, and the scientific freeze commit
`{SCIENTIFIC_FREEZE}`.

### Reproducibility Artifacts

| Artifact | SHA256 |
|---|---|
| `{NOTEBOOK_REL.as_posix()}` | `{notebook_hash}` |
| `{PYTHON_REL.as_posix()}` | `{python_hash}` |
| `{RECEIPT_REL.as_posix()}` | `{receipt_hash}` |

### Scientific Content Audit

Before repository commit, the reconstructed source was required to
contain evidence from each major Stage27 scientific section:

- Stage27-1 family/fold preparation;
- Stage27-2 model execution;
- Stage27-3 uncertainty/similarity analysis;
- Stage27-4 final synthesis;
- Stage27-PUB0 bootstrap;
- Stage27-PUB1 manuscript integration.

The reconstruction was also checked for the frozen Stage27 families,
both preregistered learners, TRAIN/VALIDATION/TARGET chronology,
bootstrap terminology, and similarity analysis.

### Status

The notebook export is a **reproducibility addendum**, not a new
scientific experiment.

Publication-addendum science operations:

- model fitting: 0
- model inference: 0
- target reopenings: 0
- threshold reselections: 0
- bootstrap recomputations: 0
- new formal statistical tests: 0

Stage27 remains scientifically closed.
"""


new_closeout = (
    committed_closeout.rstrip()
    + "\n\n"
    + addendum.strip()
    + "\n"
)


CLOSEOUT.write_text(
    new_closeout,
    encoding="utf-8",
    newline="\n",
)


closeout_hash = sha256_file(
    CLOSEOUT
)


print(
    "[PASS] reproducibility addendum appended deterministically"
)

print(
    "Closeout SHA256:",
    closeout_hash,
)


# =============================================================================
# 13. FINAL PRE-STAGE STATUS
# =============================================================================

expected_changes = {
    NOTEBOOK_REL.as_posix(),
    PYTHON_REL.as_posix(),
    RECEIPT_REL.as_posix(),
    CLOSEOUT_REL.as_posix(),
}


actual_changes = status_paths()


print()
print(
    "Worktree now:"
)

print(
    git(
        "status",
        "--short",
        "-uall",
    )
)


if actual_changes != expected_changes:

    raise RuntimeError(
        "\nUnexpected files before commit.\n"
        f"Expected: {sorted(expected_changes)}\n"
        f"Actual:   {sorted(actual_changes)}"
    )


print(
    "[PASS] exactly four authorized repository changes"
)


# =============================================================================
# 14. GIT IDENTITY
# =============================================================================

if not git(
    "config",
    "--get",
    "user.name",
    check=False,
):

    git(
        "config",
        "user.name",
        "themubasshir",
    )


if not git(
    "config",
    "--get",
    "user.email",
    check=False,
):

    git(
        "config",
        "user.email",
        "themubasshir@users.noreply.github.com",
    )


print()
print(
    "git user.name :",
    git(
        "config",
        "--get",
        "user.name",
    ),
)

print(
    "git user.email:",
    git(
        "config",
        "--get",
        "user.email",
    ),
)


# =============================================================================
# 15. KAGGLE GITHUB AUTH
# =============================================================================

print()
print(
    "-" * 100
)
print(
    "GITHUB AUTH"
)
print(
    "-" * 100
)


from kaggle_secrets import UserSecretsClient


secret_client = UserSecretsClient()


token = None
token_label = None


for label in [
    "GITHUB_TOKEN",
    "github_token",
    "GH_TOKEN",
    "GITHUB_PAT",
    "github_pat",
    "GH_PAT",
]:

    try:

        candidate = (
            secret_client
            .get_secret(
                label
            )
        )

    except Exception:

        candidate = None


    if candidate:

        token = candidate
        token_label = label
        break


if not token:

    raise RuntimeError(
        "No GitHub credential found in Kaggle Secrets."
    )


print(
    f"[PASS] {token_label} available "
    f"({len(token)} chars; value redacted)"
)


askpass = Path(
    "/kaggle/working/"
    ".stage27_pub3b_git_askpass.sh"
)


askpass.write_text(
    """#!/bin/sh
case "$1" in
  *Username*) printf '%s\\n' "x-access-token" ;;
  *)          printf '%s\\n' "$GITHUB_TOKEN" ;;
esac
""",
    encoding="utf-8",
    newline="\n",
)


askpass.chmod(
    stat.S_IRUSR
    | stat.S_IWUSR
    | stat.S_IXUSR
)


push_env = os.environ.copy()

push_env[
    "GITHUB_TOKEN"
] = token

push_env[
    "GIT_ASKPASS"
] = str(
    askpass
)

push_env[
    "GIT_TERMINAL_PROMPT"
] = "0"


# =============================================================================
# 16. FINAL REMOTE RACE CHECK
# =============================================================================

git(
    "fetch",
    "--prune",
    "origin",
    "main",
)


remote_pre_commit = git(
    "rev-parse",
    "origin/main",
)


if remote_pre_commit != head:

    raise RuntimeError(
        "origin/main changed immediately before commit."
    )


print(
    "[PASS] final remote race gate"
)


# =============================================================================
# 17. STAGE EXACT FOUR FILES
# =============================================================================

print()
print(
    "-" * 100
)
print(
    "GIT STAGING"
)
print(
    "-" * 100
)


for rel in [
    NOTEBOOK_REL,
    PYTHON_REL,
    RECEIPT_REL,
    CLOSEOUT_REL,
]:

    git(
        "add",
        "--",
        rel.as_posix(),
    )


staged = set(
    git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
)


if staged != expected_changes:

    raise RuntimeError(
        "\nStaged-file gate failed.\n"
        f"Expected: {sorted(expected_changes)}\n"
        f"Actual:   {sorted(staged)}"
    )


print(
    "[PASS] exact four-file staged set"
)


print()
print(
    git(
        "diff",
        "--cached",
        "--stat",
    )
)


# =============================================================================
# 18. COMMIT
# =============================================================================

print()
print(
    "-" * 100
)
print(
    "COMMIT"
)
print(
    "-" * 100
)


commit_result = run(
    [
        "git",
        "-c",
        "commit.gpgsign=false",
        "commit",
        "-m",
        (
            "stage27-pub3: add executed-source "
            "notebook reproducibility export"
        ),
    ]
)


print(
    commit_result.stdout
)


PUB3_COMMIT = git(
    "rev-parse",
    "HEAD",
)


print(
    "PUB3 commit:",
    PUB3_COMMIT,
)


# =============================================================================
# 19. PUSH
# =============================================================================

print()
print(
    "-" * 100
)
print(
    "PUSH"
)
print(
    "-" * 100
)


push_result = run(
    [
        "git",
        "push",
        "origin",
        "main",
    ],
    env=push_env,
)


if push_result.stdout.strip():
    print(
        push_result.stdout.strip()
    )


if push_result.stderr.strip():
    print(
        push_result.stderr.strip()
    )


# =============================================================================
# 20. REMOTE COMMIT VERIFICATION
# =============================================================================

print()
print(
    "-" * 100
)
print(
    "REMOTE VERIFICATION"
)
print(
    "-" * 100
)


git(
    "fetch",
    "--prune",
    "origin",
    "main",
)


remote_head = git(
    "rev-parse",
    "origin/main",
)


print(
    "local HEAD :",
    PUB3_COMMIT,
)

print(
    "origin/main:",
    remote_head,
)


if remote_head != PUB3_COMMIT:

    raise RuntimeError(
        "Remote commit identity mismatch."
    )


print(
    "[PASS] remote commit identity"
)


# =============================================================================
# 21. BYTE-LEVEL REMOTE VERIFICATION
# =============================================================================

expected_remote_hashes = {
    NOTEBOOK_REL:
        notebook_hash,

    PYTHON_REL:
        python_hash,

    RECEIPT_REL:
        receipt_hash,

    CLOSEOUT_REL:
        closeout_hash,
}


for rel, expected in expected_remote_hashes.items():

    remote_bytes = git_blob_bytes(
        "origin/main",
        rel,
    )

    remote_hash = sha256_bytes(
        remote_bytes
    )

    if remote_hash != expected:

        raise RuntimeError(
            f"\nRemote SHA mismatch:\n"
            f"{rel}\n"
            f"expected: {expected}\n"
            f"remote:   {remote_hash}"
        )

    print(
        f"[PASS] {rel}\n"
        f"       SHA256: {remote_hash}"
    )


# =============================================================================
# 22. PUBLICATION GENERATOR MUST STILL CHECK
# =============================================================================

print()
print(
    "-" * 100
)
print(
    "PUBLICATION PACKAGE REGRESSION CHECK"
)
print(
    "-" * 100
)


generator = (
    REPO
    / "scripts"
    / "stage27"
    / "stage27_publication_integration.py"
)


generator_check = run(
    [
        sys.executable,
        str(
            generator
        ),
        "--check",
    ]
)


print(
    generator_check.stdout
)


if generator_check.stderr.strip():

    print(
        generator_check.stderr
    )


print(
    "[PASS] original Stage27 publication package "
    "still reproduces exactly"
)


# =============================================================================
# 23. FINAL CLEAN WORKTREE
# =============================================================================

final_status = git(
    "status",
    "--porcelain",
    "-uall",
)


if final_status:

    print(
        final_status
    )

    raise RuntimeError(
        "Worktree is not clean after PUB3B."
    )


print(
    "[PASS] final worktree clean"
)


# =============================================================================
# 24. LOCAL REMOTE-VERIFICATION RECEIPT
# =============================================================================

local_receipt = {
    "stage":
        "STAGE27-PUB3B",

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "repository":
        REPO_FULL,

    "scientific_freeze":
        SCIENTIFIC_FREEZE,

    "publication_closeout_parent":
        EXPECTED_PARENT,

    "reproducibility_commit":
        PUB3_COMMIT,

    "acquisition_mode":
        mode,

    "executed_cells":
        12,

    "stage27_source_occurrences":
        521,

    "artifacts": {
        NOTEBOOK_REL.as_posix():
            notebook_hash,

        PYTHON_REL.as_posix():
            python_hash,

        RECEIPT_REL.as_posix():
            receipt_hash,

        CLOSEOUT_REL.as_posix():
            closeout_hash,
    },

    "remote_verification":
        "PASS_EXACT",

    "scientific_operations": {
        "model_fits": 0,
        "model_inference": 0,
        "target_reopenings": 0,
        "threshold_reselection": 0,
        "bootstrap_recomputation": 0,
        "new_formal_statistical_tests": 0,
    },

    "worktree_clean":
        True,
}


local_receipt_path = Path(
    "/kaggle/working/"
    "stage27_pub3b_remote_verification_receipt.json"
)


local_receipt_path.write_text(
    json.dumps(
        local_receipt,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
    newline="\n",
)


# =============================================================================
# 25. REMOVE CREDENTIAL HELPER
# =============================================================================

try:
    askpass.unlink(
        missing_ok=True
    )
except Exception:
    pass


token = None

push_env.pop(
    "GITHUB_TOKEN",
    None,
)


# =============================================================================
# 26. FINAL
# =============================================================================

print()
print(
    "=" * 100
)

print(
    "STAGE27 COMPLETE EXECUTED-SOURCE "
    "REPRODUCIBILITY EXPORT — REMOTELY FROZEN"
)

print(
    "=" * 100
)


print()
print(
    "Scientific freeze:"
)

print(
    " ",
    SCIENTIFIC_FREEZE,
)


print()
print(
    "Previous publication closeout:"
)

print(
    " ",
    EXPECTED_PARENT,
)


print()
print(
    "Reproducibility commit:"
)

print(
    " ",
    PUB3_COMMIT,
)


print()
print(
    "Notebook:"
)

print(
    " ",
    NOTEBOOK_REL
)

print(
    "  SHA256:",
    notebook_hash,
)

print(
    "  provenance:",
    mode,
)


print()
print(
    "Python export:"
)

print(
    " ",
    PYTHON_REL
)

print(
    "  SHA256:",
    python_hash,
)


print()
print(
    "Export receipt:"
)

print(
    " ",
    RECEIPT_REL
)

print(
    "  SHA256:",
    receipt_hash,
)


print()
print(
    "Updated closeout:"
)

print(
    " ",
    CLOSEOUT_REL
)

print(
    "  SHA256:",
    closeout_hash,
)


print()
print(
    "Verification:"
)

print(
    "  scientific-stage source audit : PASS"
)

print(
    "  frozen scientific artifacts   : UNCHANGED"
)

print(
    "  remote commit identity         : PASS"
)

print(
    "  remote artifact SHA256         : PASS"
)

print(
    "  publication generator --check  : PASS"
)

print(
    "  final worktree                 : CLEAN"
)


print()
print(
    "Important provenance:"
)

print(
    "  The .ipynb is an executed-source reconstruction."
)

print(
    "  It is NOT claimed to be a byte-identical copy "
    "of the original Kaggle notebook container."
)

print(
    "  The .py preserves all reconstructed executed "
    "code-cell source in notebook order."
)


print()
print(
    "Local remote-verification receipt:"
)

print(
    " ",
    local_receipt_path,
)

print(
    "  SHA256:",
    sha256_file(
        local_receipt_path
    ),
)


print()
print(
    "NEXT PHASE:"
)

print(
    "  Stage27 is finished. Proceed to whole-manuscript "
    "assembly and cross-stage claim-to-artifact audit."
)

STAGE27-PUB3B — FREEZE / PUSH EXECUTED-SOURCE NOTEBOOK RECONSTRUCTION

timestamp_utc: 2026-08-21T15:34:23.143818+00:00
repository: themubasshir/ids2018-validation-safe-ablation
scientific freeze: 0e1439565aedc7da9b7ca1207262e9061422bc22
expected parent: 3407ff3954abae9b0c8bfdaa14b704a05f31affe

branch: main
HEAD:   3407ff3954abae9b0c8bfdaa14b704a05f31affe
[PASS] scientific lineage

----------------------------------------------------------------------------------------------------
REMOTE RACE GATE
----------------------------------------------------------------------------------------------------
local HEAD : 3407ff3954abae9b0c8bfdaa14b704a05f31affe
origin/main: 3407ff3954abae9b0c8bfdaa14b704a05f31affe
[PASS] local == remote publication closeout

----------------------------------------------------------------------------------------------------
WORKTREE INVENTORY
----------------------------------------------------------------------------------------------------
?? results/stage27_loa

RuntimeError: 
Unexpected files before commit.
Expected: ['docs/STAGE27_PUBLICATION_CLOSEOUT.md', 'results/stage27_loao_unseen_attack/stage27_publication_package/stage27_notebook_export_receipt.json', 'scripts/stage27/stage27_loao_unseen_attack.ipynb', 'scripts/stage27/stage27_loao_unseen_attack.py']
Actual:   ['ocs/STAGE27_PUBLICATION_CLOSEOUT.md', 'results/stage27_loao_unseen_attack/stage27_publication_package/stage27_notebook_export_receipt.json', 'scripts/stage27/stage27_loao_unseen_attack.ipynb', 'scripts/stage27/stage27_loao_unseen_attack.py']

In [14]:
# ============================================================================
# STAGE27-PUB3B-R
# RECOVERY FROM GIT PORCELAIN PARSING BUG
#
# The previous PUB3B successfully completed:
#   - notebook SHA verification
#   - reconstruction provenance verification
#   - scientific-stage source audit
#   - Python source-preservation audit
#   - frozen-science immutability check
#   - deterministic closeout addendum generation
#
# It stopped BEFORE git add / commit / push because stdout.strip()
# removed Git porcelain's leading status-space on the first line.
#
# This recovery cell:
#   1. verifies the exact four expected local modifications
#      using NUL-delimited Git porcelain without stripping whitespace;
#   2. re-verifies all hashes;
#   3. stages exactly those four paths;
#   4. commits;
#   5. pushes;
#   6. fetches and byte-verifies remote contents;
#   7. reruns the publication-generator check;
#   8. requires a clean final worktree.
#
# ZERO new Stage27 scientific computation.
# ============================================================================

from __future__ import annotations

import hashlib
import json
import os
import stat
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path


# =============================================================================
# 0. IDENTITY
# =============================================================================

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

REPO_FULL = (
    "themubasshir/ids2018-validation-safe-ablation"
)

SCIENTIFIC_FREEZE = (
    "0e1439565aedc7da9b7ca1207262e9061422bc22"
)

EXPECTED_PARENT = (
    "3407ff3954abae9b0c8bfdaa14b704a05f31affe"
)

NOTEBOOK_REL = Path(
    "scripts/stage27/stage27_loao_unseen_attack.ipynb"
)

PYTHON_REL = Path(
    "scripts/stage27/stage27_loao_unseen_attack.py"
)

RECEIPT_REL = Path(
    "results/stage27_loao_unseen_attack/"
    "stage27_publication_package/"
    "stage27_notebook_export_receipt.json"
)

CLOSEOUT_REL = Path(
    "docs/STAGE27_PUBLICATION_CLOSEOUT.md"
)

NOTEBOOK = REPO / NOTEBOOK_REL
PYTHON_EXPORT = REPO / PYTHON_REL
RECEIPT = REPO / RECEIPT_REL
CLOSEOUT = REPO / CLOSEOUT_REL


EXPECTED_HASHES = {
    NOTEBOOK_REL.as_posix():
        "cc9387c8c41f70a6e38cef133b6230bda9b5aae20af6c13dc710c545754dad3b",

    PYTHON_REL.as_posix():
        "f25967bbbd463db3942037de02cfa35f2c02d79f94aeaa408615a7b1bd90e7df",

    RECEIPT_REL.as_posix():
        "e3723b70bcef74e59647eee9c05701e5272b16d001851e8a03ca6f699bb49162",

    CLOSEOUT_REL.as_posix():
        "6bba902e9c045d6d2f6e93e6a16fe21931654fc7b60a4b494330b120ace54e4d",
}


AUTHORIZED_PATHS = {
    NOTEBOOK_REL.as_posix(),
    PYTHON_REL.as_posix(),
    RECEIPT_REL.as_posix(),
    CLOSEOUT_REL.as_posix(),
}


# =============================================================================
# 1. HELPERS
# =============================================================================

def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
    text=True,
    env=None,
):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=text,
        env=env,
    )

    if check and p.returncode != 0:

        if text:
            print(p.stdout)
            print(p.stderr)

        raise RuntimeError(
            f"Command failed ({p.returncode}): "
            + " ".join(
                str(x)
                for x in cmd
            )
        )

    return p


def git_text(
    *args,
    check=True,
):
    """
    For ordinary Git commands where surrounding whitespace
    has no semantic meaning.
    """
    return run(
        [
            "git",
            *args,
        ],
        check=check,
        text=True,
    ).stdout.strip()


def git_raw(
    *args,
    check=True,
):
    """
    Binary-safe Git output.
    IMPORTANT: no .strip().
    """
    return run(
        [
            "git",
            *args,
        ],
        check=check,
        text=False,
    ).stdout


def sha256_file(
    path,
):
    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as fh:

        while True:

            chunk = fh.read(
                8 * 1024 * 1024
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def sha256_bytes(
    data,
):
    return hashlib.sha256(
        data
    ).hexdigest()


def porcelain_entries():
    """
    Robust Git status parser.

    Uses:
        git status --porcelain=v1 -z -uall

    NUL-delimited output preserves leading status spaces and supports
    arbitrary filenames without relying on line trimming.
    """

    raw = git_raw(
        "status",
        "--porcelain=v1",
        "-z",
        "-uall",
    )

    records = [
        record
        for record in raw.split(b"\0")
        if record
    ]

    entries = []

    i = 0

    while i < len(records):

        record = records[i]

        if len(record) < 4:
            raise RuntimeError(
                f"Malformed Git porcelain record: {record!r}"
            )

        xy = record[:2].decode(
            "ascii",
            errors="strict",
        )

        if record[2:3] != b" ":
            raise RuntimeError(
                f"Unexpected Git porcelain format: {record!r}"
            )

        path = record[3:].decode(
            "utf-8",
            errors="surrogateescape",
        )

        entries.append(
            {
                "xy": xy,
                "path": path,
            }
        )

        # For -z rename/copy entries, Git follows with original path.
        # We do not expect rename/copy in this recovery operation.
        if "R" in xy or "C" in xy:

            if i + 1 >= len(records):
                raise RuntimeError(
                    "Malformed rename/copy porcelain output."
                )

            original_path = records[
                i + 1
            ].decode(
                "utf-8",
                errors="surrogateescape",
            )

            entries[-1][
                "original_path"
            ] = original_path

            i += 1

        i += 1

    return entries


def changed_paths():
    return {
        entry["path"]
        for entry in porcelain_entries()
    }


def staged_paths():
    raw = git_raw(
        "diff",
        "--cached",
        "--name-only",
        "-z",
    )

    return {
        item.decode(
            "utf-8",
            errors="surrogateescape",
        )
        for item in raw.split(b"\0")
        if item
    }


def remote_blob_bytes(
    ref,
    rel,
):
    return git_raw(
        "show",
        f"{ref}:{rel.as_posix()}",
    )


# =============================================================================
# 2. HEADER
# =============================================================================

print(
    "=" * 100
)

print(
    "STAGE27-PUB3B-R — RECOVER / COMMIT / PUSH "
    "EXECUTED-SOURCE REPRODUCIBILITY EXPORT"
)

print(
    "=" * 100
)

print()
print(
    "timestamp_utc:",
    datetime.now(
        timezone.utc
    ).isoformat(),
)

print(
    "repository       :",
    REPO_FULL,
)

print(
    "scientific freeze:",
    SCIENTIFIC_FREEZE,
)

print(
    "expected parent  :",
    EXPECTED_PARENT,
)


# =============================================================================
# 3. LOCAL + REMOTE PARENT
# =============================================================================

branch = git_text(
    "branch",
    "--show-current",
)

head = git_text(
    "rev-parse",
    "HEAD",
)


print()
print(
    "branch:",
    branch,
)

print(
    "HEAD:  ",
    head,
)


if branch != "main":

    raise RuntimeError(
        f"Expected main; found {branch}"
    )


if head != EXPECTED_PARENT:

    raise RuntimeError(
        "\nRecovery parent changed.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {head}"
    )


ancestor = run(
    [
        "git",
        "merge-base",
        "--is-ancestor",
        SCIENTIFIC_FREEZE,
        head,
    ],
    check=False,
)


if ancestor.returncode != 0:

    raise RuntimeError(
        "Scientific freeze is not an ancestor of current HEAD."
    )


print(
    "[PASS] scientific lineage"
)


git_text(
    "fetch",
    "--prune",
    "origin",
    "main",
)


origin_main = git_text(
    "rev-parse",
    "origin/main",
)


print(
    "origin/main:",
    origin_main,
)


if origin_main != head:

    raise RuntimeError(
        "Remote main changed after the previous PUB3B attempt."
    )


print(
    "[PASS] remote race gate"
)


# =============================================================================
# 4. FIXED PORCELAIN INVENTORY
# =============================================================================

print()
print(
    "-" * 100
)

print(
    "ROBUST GIT WORKTREE INVENTORY"
)

print(
    "-" * 100
)


entries = porcelain_entries()


for entry in entries:

    print(
        f"{entry['xy']}  {entry['path']}"
    )


actual_paths = {
    entry["path"]
    for entry in entries
}


if actual_paths != AUTHORIZED_PATHS:

    print()
    print(
        "Expected:",
        sorted(
            AUTHORIZED_PATHS
        ),
    )

    print(
        "Actual:  ",
        sorted(
            actual_paths
        ),
    )

    raise RuntimeError(
        "Recovery worktree differs from the exact authorized set."
    )


# Enforce expected Git states:
#
# closeout = modified but unstaged   " M"
# 3 exports = untracked             "??"

state_by_path = {
    entry["path"]:
        entry["xy"]

    for entry in entries
}


if (
    state_by_path[
        CLOSEOUT_REL.as_posix()
    ]
    != " M"
):

    raise RuntimeError(
        "Closeout is not in expected unstaged-modified state."
    )


for rel in [
    NOTEBOOK_REL,
    PYTHON_REL,
    RECEIPT_REL,
]:

    if (
        state_by_path[
            rel.as_posix()
        ]
        != "??"
    ):

        raise RuntimeError(
            f"Unexpected Git state for {rel}: "
            f"{state_by_path[rel.as_posix()]}"
        )


print()
print(
    "[PASS] exact four-file post-PUB3B failure state"
)


# =============================================================================
# 5. HASH FREEZE
# =============================================================================

print()
print(
    "-" * 100
)

print(
    "SHA256 FREEZE GATE"
)

print(
    "-" * 100
)


for rel, expected in EXPECTED_HASHES.items():

    path = REPO / rel

    if not path.is_file():

        raise RuntimeError(
            f"Missing expected artifact: {rel}"
        )

    actual = sha256_file(
        path
    )

    if actual != expected:

        raise RuntimeError(
            f"\nArtifact changed after prior review:\n"
            f"{rel}\n"
            f"expected: {expected}\n"
            f"actual:   {actual}"
        )

    print(
        f"[PASS] {rel}\n"
        f"       {actual}"
    )


# =============================================================================
# 6. RECEIPT / NOTEBOOK PROVENANCE RECHECK
# =============================================================================

print()
print(
    "-" * 100
)

print(
    "REPRODUCIBILITY PROVENANCE RECHECK"
)

print(
    "-" * 100
)


receipt = json.loads(
    RECEIPT.read_text(
        encoding="utf-8"
    )
)


mode = (
    receipt
    .get(
        "acquisition",
        {},
    )
    .get(
        "mode"
    )
)


if (
    mode
    != "RECONSTRUCTED_FROM_IPYTHON_INPUT_HISTORY"
):

    raise RuntimeError(
        f"Unexpected acquisition mode: {mode}"
    )


notebook_meta = receipt[
    "notebook"
]


if (
    notebook_meta[
        "cells"
    ][
        "total"
    ]
    != 12
):

    raise RuntimeError(
        "Expected reconstructed notebook cell count = 12."
    )


if (
    notebook_meta[
        "stage27_source_occurrences"
    ]
    != 521
):

    raise RuntimeError(
        "Expected Stage27 source occurrences = 521."
    )


print(
    "[PASS] acquisition = RECONSTRUCTED_FROM_IPYTHON_INPUT_HISTORY"
)

print(
    "[PASS] executed-source cells = 12"
)

print(
    "[PASS] Stage27 source occurrences = 521"
)


# =============================================================================
# 7. FROZEN SCIENCE IMMUTABILITY
# =============================================================================

print()
print(
    "-" * 100
)

print(
    "FROZEN SCIENCE GATE"
)

print(
    "-" * 100
)


science_diff = run(
    [
        "git",
        "diff",
        "--",
        (
            "results/stage27_loao_unseen_attack/"
            "stage27_4a_final_synthesis"
        ),
    ]
).stdout


if science_diff.strip():

    print(
        science_diff
    )

    raise RuntimeError(
        "Frozen Stage27 science was modified."
    )


print(
    "[PASS] Stage27-4A synthesis byte-untouched"
)


# =============================================================================
# 8. GIT IDENTITY
# =============================================================================

if not git_text(
    "config",
    "--get",
    "user.name",
    check=False,
):

    git_text(
        "config",
        "user.name",
        "themubasshir",
    )


if not git_text(
    "config",
    "--get",
    "user.email",
    check=False,
):

    git_text(
        "config",
        "user.email",
        "themubasshir@users.noreply.github.com",
    )


print()
print(
    "git user.name :",
    git_text(
        "config",
        "--get",
        "user.name",
    ),
)

print(
    "git user.email:",
    git_text(
        "config",
        "--get",
        "user.email",
    ),
)


# =============================================================================
# 9. GITHUB AUTH
# =============================================================================

print()
print(
    "-" * 100
)

print(
    "GITHUB AUTH"
)

print(
    "-" * 100
)


from kaggle_secrets import UserSecretsClient


secret_client = UserSecretsClient()

token = None
token_label = None


for label in [
    "GITHUB_TOKEN",
    "github_token",
    "GH_TOKEN",
    "GITHUB_PAT",
    "github_pat",
    "GH_PAT",
]:

    try:

        candidate = (
            secret_client
            .get_secret(
                label
            )
        )

    except Exception:

        candidate = None


    if candidate:

        token = candidate
        token_label = label
        break


if not token:

    raise RuntimeError(
        "GitHub token unavailable."
    )


print(
    f"[PASS] {token_label} available "
    f"({len(token)} chars; value redacted)"
)


askpass = Path(
    "/kaggle/working/"
    ".stage27_pub3br_askpass.sh"
)


askpass.write_text(
    """#!/bin/sh
case "$1" in
  *Username*) printf '%s\\n' "x-access-token" ;;
  *)          printf '%s\\n' "$GITHUB_TOKEN" ;;
esac
""",
    encoding="utf-8",
    newline="\n",
)


askpass.chmod(
    stat.S_IRUSR
    | stat.S_IWUSR
    | stat.S_IXUSR
)


push_env = os.environ.copy()

push_env[
    "GITHUB_TOKEN"
] = token

push_env[
    "GIT_ASKPASS"
] = str(
    askpass
)

push_env[
    "GIT_TERMINAL_PROMPT"
] = "0"


# =============================================================================
# 10. FINAL REMOTE RACE GATE
# =============================================================================

git_text(
    "fetch",
    "--prune",
    "origin",
    "main",
)


remote_precommit = git_text(
    "rev-parse",
    "origin/main",
)


if remote_precommit != EXPECTED_PARENT:

    raise RuntimeError(
        "\nRemote advanced before recovery commit.\n"
        f"Expected: {EXPECTED_PARENT}\n"
        f"Actual:   {remote_precommit}"
    )


print(
    "[PASS] final remote precommit gate"
)


# =============================================================================
# 11. STAGE EXACT FOUR FILES
# =============================================================================

print()
print(
    "-" * 100
)

print(
    "EXACT STAGING"
)

print(
    "-" * 100
)


for rel in [
    NOTEBOOK_REL,
    PYTHON_REL,
    RECEIPT_REL,
    CLOSEOUT_REL,
]:

    git_text(
        "add",
        "--",
        rel.as_posix(),
    )


staged = staged_paths()


print(
    "Staged:"
)


for path in sorted(
    staged
):

    print(
        " ",
        path
    )


if staged != AUTHORIZED_PATHS:

    raise RuntimeError(
        "\nStaged-file set mismatch.\n"
        f"Expected: {sorted(AUTHORIZED_PATHS)}\n"
        f"Actual:   {sorted(staged)}"
    )


print(
    "[PASS] exact four-file staged set"
)


print()
print(
    git_text(
        "diff",
        "--cached",
        "--stat",
    )
)


# =============================================================================
# 12. COMMIT
# =============================================================================

print()
print(
    "-" * 100
)

print(
    "COMMIT"
)

print(
    "-" * 100
)


commit = run(
    [
        "git",
        "-c",
        "commit.gpgsign=false",
        "commit",
        "-m",
        (
            "stage27-pub3: add executed-source "
            "notebook reproducibility export"
        ),
    ]
)


print(
    commit.stdout
)


PUB3_COMMIT = git_text(
    "rev-parse",
    "HEAD",
)


print(
    "New commit:",
    PUB3_COMMIT,
)


# =============================================================================
# 13. PUSH
# =============================================================================

print()
print(
    "-" * 100
)

print(
    "PUSH"
)

print(
    "-" * 100
)


push = run(
    [
        "git",
        "push",
        "origin",
        "main",
    ],
    env=push_env,
)


if push.stdout.strip():

    print(
        push.stdout.strip()
    )


if push.stderr.strip():

    print(
        push.stderr.strip()
    )


# =============================================================================
# 14. FETCH + REMOTE HEAD
# =============================================================================

git_text(
    "fetch",
    "--prune",
    "origin",
    "main",
)


remote_head = git_text(
    "rev-parse",
    "origin/main",
)


print()
print(
    "local HEAD :",
    PUB3_COMMIT,
)

print(
    "origin/main:",
    remote_head,
)


if remote_head != PUB3_COMMIT:

    raise RuntimeError(
        "Remote commit identity mismatch."
    )


print(
    "[PASS] remote commit identity"
)


# =============================================================================
# 15. REMOTE BYTE VERIFICATION
# =============================================================================

print()
print(
    "-" * 100
)

print(
    "REMOTE BYTE-LEVEL SHA256 VERIFICATION"
)

print(
    "-" * 100
)


for rel, expected_hash in [
    (
        NOTEBOOK_REL,
        EXPECTED_HASHES[
            NOTEBOOK_REL.as_posix()
        ],
    ),
    (
        PYTHON_REL,
        EXPECTED_HASHES[
            PYTHON_REL.as_posix()
        ],
    ),
    (
        RECEIPT_REL,
        EXPECTED_HASHES[
            RECEIPT_REL.as_posix()
        ],
    ),
    (
        CLOSEOUT_REL,
        EXPECTED_HASHES[
            CLOSEOUT_REL.as_posix()
        ],
    ),
]:

    remote_bytes = remote_blob_bytes(
        "origin/main",
        rel,
    )

    remote_hash = sha256_bytes(
        remote_bytes
    )

    if remote_hash != expected_hash:

        raise RuntimeError(
            f"\nRemote SHA mismatch:\n"
            f"{rel}\n"
            f"expected: {expected_hash}\n"
            f"remote:   {remote_hash}"
        )

    print(
        f"[PASS] {rel}\n"
        f"       {remote_hash}"
    )


# =============================================================================
# 16. ORIGINAL PUBLICATION PACKAGE REGRESSION CHECK
# =============================================================================

print()
print(
    "-" * 100
)

print(
    "PUBLICATION GENERATOR REGRESSION CHECK"
)

print(
    "-" * 100
)


generator = (
    REPO
    / "scripts"
    / "stage27"
    / "stage27_publication_integration.py"
)


check = run(
    [
        sys.executable,
        str(
            generator
        ),
        "--check",
    ]
)


print(
    check.stdout
)


if check.stderr.strip():

    print(
        check.stderr
    )


print(
    "[PASS] original Stage27 publication package remains reproducible"
)


# =============================================================================
# 17. FINAL WORKTREE
# =============================================================================

final_entries = porcelain_entries()


if final_entries:

    for entry in final_entries:

        print(
            entry
        )

    raise RuntimeError(
        "Repository is not clean after PUB3B-R."
    )


print()
print(
    "[PASS] final Git worktree clean"
)


# =============================================================================
# 18. LOCAL REMOTE-VERIFICATION RECEIPT
# =============================================================================

verification = {
    "stage":
        "STAGE27-PUB3B-R",

    "timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "repository":
        REPO_FULL,

    "scientific_freeze":
        SCIENTIFIC_FREEZE,

    "previous_publication_closeout":
        EXPECTED_PARENT,

    "reproducibility_commit":
        PUB3_COMMIT,

    "acquisition_mode":
        "RECONSTRUCTED_FROM_IPYTHON_INPUT_HISTORY",

    "executed_source_cells":
        12,

    "stage27_source_occurrences":
        521,

    "artifacts": {
        rel:
            digest

        for rel, digest
        in EXPECTED_HASHES.items()
    },

    "remote_verification":
        "PASS_EXACT",

    "science_operations": {
        "model_fitting": 0,
        "model_inference": 0,
        "target_reopenings": 0,
        "threshold_reselection": 0,
        "bootstrap_recomputation": 0,
        "new_formal_statistical_tests": 0,
    },

    "worktree_clean":
        True,
}


verification_path = Path(
    "/kaggle/working/"
    "stage27_pub3br_remote_verification_receipt.json"
)


verification_path.write_text(
    json.dumps(
        verification,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
    newline="\n",
)


# =============================================================================
# 19. CLEAN CREDENTIAL HELPER
# =============================================================================

try:

    askpass.unlink(
        missing_ok=True
    )

except Exception:

    pass


token = None

push_env.pop(
    "GITHUB_TOKEN",
    None,
)


# =============================================================================
# 20. FINAL SUMMARY
# =============================================================================

print()
print(
    "=" * 100
)

print(
    "STAGE27 EXECUTED-SOURCE NOTEBOOK "
    "REPRODUCIBILITY EXPORT — REMOTELY FROZEN"
)

print(
    "=" * 100
)


print()
print(
    "Scientific freeze:"
)

print(
    " ",
    SCIENTIFIC_FREEZE,
)


print()
print(
    "Previous publication closeout:"
)

print(
    " ",
    EXPECTED_PARENT,
)


print()
print(
    "Final Stage27 reproducibility commit:"
)

print(
    " ",
    PUB3_COMMIT,
)


print()
print(
    "Repository assets:"
)

print(
    " ",
    NOTEBOOK_REL
)

print(
    "   ",
    EXPECTED_HASHES[
        NOTEBOOK_REL.as_posix()
    ],
)

print(
    " ",
    PYTHON_REL
)

print(
    "   ",
    EXPECTED_HASHES[
        PYTHON_REL.as_posix()
    ],
)

print(
    " ",
    RECEIPT_REL
)

print(
    "   ",
    EXPECTED_HASHES[
        RECEIPT_REL.as_posix()
    ],
)

print(
    " ",
    CLOSEOUT_REL
)

print(
    "   ",
    EXPECTED_HASHES[
        CLOSEOUT_REL.as_posix()
    ],
)


print()
print(
    "Verification:"
)

print(
    "  scientific lineage            : PASS"
)

print(
    "  scientific-stage source audit : PASS (prior PUB3B)"
)

print(
    "  frozen Stage27 science        : UNCHANGED"
)

print(
    "  remote commit identity        : PASS"
)

print(
    "  remote byte SHA256            : PASS"
)

print(
    "  publication generator --check : PASS"
)

print(
    "  worktree                      : CLEAN"
)


print()
print(
    "Notebook provenance:"
)

print(
    "  executed-source reconstruction from "
    "12 Kaggle/IPython input-history cells"
)

print(
    "  521 Stage27 source references"
)

print(
    "  no original outputs"
)

print(
    "  no claim of byte-identical Kaggle .ipynb container"
)


print()
print(
    "Local remote-verification receipt:"
)

print(
    " ",
    verification_path
)

print(
    "  SHA256:",
    sha256_file(
        verification_path
    ),
)


print()
print(
    "STAGE27 STATUS: COMPLETE"
)

print(
    "NEXT: whole-manuscript assembly + "
    "cross-stage claim-to-artifact consistency audit"
)

STAGE27-PUB3B-R — RECOVER / COMMIT / PUSH EXECUTED-SOURCE REPRODUCIBILITY EXPORT

timestamp_utc: 2026-08-21T15:36:43.522141+00:00
repository       : themubasshir/ids2018-validation-safe-ablation
scientific freeze: 0e1439565aedc7da9b7ca1207262e9061422bc22
expected parent  : 3407ff3954abae9b0c8bfdaa14b704a05f31affe

branch: main
HEAD:   3407ff3954abae9b0c8bfdaa14b704a05f31affe
[PASS] scientific lineage
origin/main: 3407ff3954abae9b0c8bfdaa14b704a05f31affe
[PASS] remote race gate

----------------------------------------------------------------------------------------------------
ROBUST GIT WORKTREE INVENTORY
----------------------------------------------------------------------------------------------------
 M  docs/STAGE27_PUBLICATION_CLOSEOUT.md
??  results/stage27_loao_unseen_attack/stage27_publication_package/stage27_notebook_export_receipt.json
??  scripts/stage27/stage27_loao_unseen_attack.ipynb
??  scripts/stage27/stage27_loao_unseen_attack.py

[PASS] exact four-file post-PUB3B fa